# SODA-RFF fixed-FAR notebook

This notebook keeps the minimal experimental pipeline for **SODA-RFF** and the comparison baselines, and adds a complete **fixed-FAR operating-point experiment** that only sweeps the final accept/reject threshold.
**SourceOnly**, **PCPD**, **OVANet**, **COMET**, and **JRFFP-SC**.

The SODA-RFF mainline is locked to the protocol-specific clean-win settings selected from the v36.6 search results.
For each protocol, the notebook now fixes both:
1. the SODA-RFF adaptation parameters; and
2. the final rejector head used by the clean-win configuration.

The retained protocol-specific settings are:
- **RX9-3_TX4-2**: fixed-win SODA-RFF with default `energy` rejector.
- **RX9-3_TX2-4**: pure-v19 local optimum corresponding to `pv004`, with `energy_class` rejector.
- **RX6-6_TX3-3**: fixed-win SODA-RFF with default `energy` rejector.
- **RX3-9_TX2-4**: pure-v19 local optimum corresponding to `pv008`, with `proto` rejector.

The notebook preserves a minimal smoke-test entry and the compact evaluation pipeline only.


# SODA direct core-buffer threshold-first final comparison\n
\n
- Derived from `wisig_SODA_RFF_Final_direct_corebuffer.ipynb`; previous top-fraction selection is not modified in place.\n
- Keeps method name `SODA_direct_corebuffer`, but default subset selection is now threshold-first.\n
- Ambiguous samples enter buffer and are skipped during adaptation.\n

# Final strict RFF-OSDA refactor changelog

- Adds a canonical method-native score bundle used by all comparison protocols.
- Separates paper-native threshold policy from score extraction.
- Adds target-calibrated `fixed_far_unknown` and `fixed_frr_drift_all` policies on the same native scores.
- Adds a unified operating-point artifact exporter for raw rows, summaries, curves, workbooks, manifests, plots, and best-method tables.
- Leaves the shared RFF-OSDA benchmark, backbone, source/target splits, adaptation budget, and strict protocol best-param loading unchanged.


In [1]:
import os, json, time, math
import itertools
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
import openpyxl
import time

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc, confusion_matrix
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pickle
import warnings
warnings.filterwarnings("ignore", message="Tight layout not applied")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


def _win_safe_path(path):
    if path is None:
        return path
    p = os.path.abspath(os.path.normpath(str(path)))
    if os.name == "nt":
        if p.startswith('\\\\?\\'):
            return p
        if len(p) >= 240:
            if p.startswith('\\'):
                return '\\\\?\\UNC\\' + p.lstrip('\\')
            return '\\\\?\\' + p
    return p

def _safe_makedirs(path):
    if path is None or path == '':
        return
    os.makedirs(_win_safe_path(path), exist_ok=True)

def _safe_open(path, mode='r', **kwargs):
    return open(_win_safe_path(path), mode, **kwargs)


def log_banner(title, char="="):
    line = char * 18
    print(f"\n{line} {title} {line}")

def log_kv(prefix, **kwargs):
    msg = " | ".join([f"{k}={v}" for k, v in kwargs.items()])
    print(f"[{prefix}] {msg}")


dataset_name = "ManySig"
dataset_path = "../ManySig.pkl/"

def load_compact_pkl_dataset(dataset_path,dataset_name):
    with _safe_open(dataset_path+dataset_name+'.pkl','rb') as f:
        dataset = pickle.load(f)
    return dataset

# from <your_loader_module> import load_compact_pkl_dataset
compact_dataset = load_compact_pkl_dataset(dataset_path, dataset_name)


TX_TOTAL_USE = 6
RX_TOTAL_USE = 12
DAY_TOTAL_USE = 4

# legacy defaults kept for backward compatibility in some helper/config dumps
KNOWN_TX_NUM = 4
SOURCE_RX_NUM = 3

# experiment / run-control
EXPERIMENT_MODE = "ext_compare"
CODE_FILE_NAME = "wisig__SODA_RFF_fixed_cleanwin.ipynb"
RUN_VERSION_TAG = "SODA_RFF_fixed_cleanwin"
RUN_NOTES = (
    "Minimal SODA-RFF notebook with protocol-specific clean-win parameters locked for the "
    "four protocols, plus SourceOnly / PCPD / OVANet / COMET / JRFFP-SC baselines."
)
RUN_HOST_TAG = "pcA"
RUN_TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")

# legacy flag kept defined for compatibility with helper/config code
V34_PROTOCOL_TUNING_ENABLED = False


TX_SPLIT_REPEATS = 3

# protocol libraries
RX_PROTOCOL_LIBRARY = {
    "3-9": dict(source_rx_num=3, drift_rx_num=9),
    "6-6": dict(source_rx_num=6, drift_rx_num=6),
    "9-3": dict(source_rx_num=9, drift_rx_num=3),
}
TX_PROTOCOL_LIBRARY = {
    "2-4": dict(known_tx_num=2, unknown_tx_num=4),
    "3-3": dict(known_tx_num=3, unknown_tx_num=3),
    "4-2": dict(known_tx_num=4, unknown_tx_num=2),
}

# development subset to cut runtime while keeping representative coverage
# rationale:
#   ("9-3","4-2"): main realistic target / typical challenge
#   ("9-3","2-4"): typical RX with unknown-heavy split; sensitive to routing quality
#   ("6-6","3-3"): balanced mid-difficulty setting
#   ("3-9","2-4"): hard but still responsive stress setting
SELECTED_RX_PROTOCOLS = ["3-9", "6-6", "9-3"]
SELECTED_TX_PROTOCOLS = ["2-4", "3-3", "4-2"]

FAST_DEV_PROTOCOL_COMBOS = [
    ("9-3", "4-2"),
    ("9-3", "2-4"),
    ("6-6", "3-3"),
    ("3-9", "2-4")]
# tuning defaults: use the 4 representative protocols for the small-range parameter-set comparison
V22_TUNING_PROTOCOL_COMBOS = [
    ("9-3", "4-2"),
    ("9-3", "2-4"),
    ("6-6", "3-3"),
    ("3-9", "2-4")]
# set to None to restore the full 3x3 protocol suite
EXT_COMPARE_PROTOCOL_COMBOS = [("9-3", "4-2"), ("9-3", "2-4"), ("6-6", "3-3"), ("3-9", "2-4")]
SELECTED_PROTOCOL_COMBOS = EXT_COMPARE_PROTOCOL_COMBOS if EXPERIMENT_MODE == "ext_compare" else (V22_TUNING_PROTOCOL_COMBOS if EXPERIMENT_MODE == "v22_tuning" else FAST_DEV_PROTOCOL_COMBOS)

TRAIN_DATES = ["2021_03_15"]
TEST_DATES_RX = ["2021_03_15"]
TEST_DATES_DAY = ["2021_03_01"]

Z_FROM_EQ = 1
D_FROM = "raw"
MAX_SIG_PER_TRIPLE = None

BATCH_SIZE = 128
EPOCHS = 100
LR = 1e-4
WEIGHT_DECAY = 1e-3
PATIENCE = 8
IN_PLANES = 64
DROPOUT = 0.3
D_DIM = 32

# fixed Module 2
FUSION_LAM = 0.5
FRR_TARGET = 0.05
FALSE_DRIFT_TARGET = 0.05

# stream split
ADAPT_FRAC = 0.5
MAX_ADAPT_PER_SET = 2000
MAX_EVAL_PER_SET = 3000

# adapter
ADAPTER_BOTTLENECK = 64
ADAPTER_SCALE = 0.3
ADAPT_EPOCHS = 30
ADAPT_LR = 1e-3
ADAPT_WEIGHT_DECAY = 1e-4
ADAPT_BATCH = 128

# replay
REPLAY_PER_CLASS = 200
LAMBDA_REPLAY_CE = 1.0
LAMBDA_REPLAY_ANCHOR = 1.0

# pseudo-label suite params
PSEUDO_MIN_KEEP = 64
SOFT_T = 2.0
PROTO_T = 0.10
KNN_TOPK = 15
KNN_MEM_PER_CLASS = 300

# fusion weights
W_LOGIT = 1.0
W_PROTO = 1.0
W_KNN = 1.0

# confidence threshold from stable source-like set A
CONF_STABLE_Q = 0.10

# new suite hyper-parameters
CONF_DRIFT_Q = 0.40
REFINE_ITERS = 2
UNKNOWN_SCORE_Q = 0.35
EMB_AUG_NOISE = 0.02
MMD_SIGMAS = [1.0, 2.0, 4.0, 8.0, 16.0]
LAMBDA_ALIGN = 0.35
LAMBDA_ENT_MIN = 0.02
LAMBDA_ENT_MAX = 0.05
LAMBDA_CONS = 0.05
LAMBDA_PROTO = 0.05
GCODWFA_ALIGN_MAX = 0.80
RAIN_STAGE1_EPOCHS = 8
RAIN_STAGE2_EPOCHS = 12


# v5 three-bucket parameters
RELIABLE_KEEP_Q = 0.12
AMBIG_KEEP_Q = 0.30
UNKNOWN_KEEP_Q = 0.08

AMBIG_TOPK = 2
RELIABLE_WEIGHT_FLOOR = 0.65
AMBIG_WEIGHT_FLOOR = 0.30
AMBIG_WEIGHT_SCALE = 0.70

EMA_MOMENTUM = 0.995
EMA_TEACHER_BLEND = 0.60
TEACHER_TEMP = 1.0

THREE_BUCKET_EPOCHS = 24
LAMBDA_AMB_SUP = 0.60
LAMBDA_SRC_PROTO = 0.08
LAMBDA_SRC_LOGIT = 0.30
LAMBDA_UNKNOWN_REPEL = 0.02
UNKNOWN_REPEL_MARGIN = 0.25
UNKNOWN_WARMUP_EPOCHS = 6
UNKNOWN_RAMP_EPOCHS = 8

# v6 curriculum three-bucket parameters
V6_RELIABLE_KEEP_Q = 0.22
V6_AMBIG_KEEP_Q = 0.30
V6_UNKNOWN_KEEP_Q = 0.05
V6_UNKNOWN_KEEP_Q_HARD = 0.08
V6_WEAK_MIN_KEEP = 64
V6_WEAK_TOPK = 3
V6_WEAK_SHARPEN = 0.95
V6_REL_WEIGHT_FLOOR = 0.72
V6_AMB_WEIGHT_FLOOR = 0.36
V6_AMB_WEIGHT_SCALE = 0.44
V6_WEAK_WEIGHT_FLOOR = 0.10
V6_WEAK_WEIGHT_SCALE = 0.20
V6_WEAK_START_EPOCH = 4
V6_WEAK_TEACHER_BLEND = 0.35
V6_LAMBDA_WEAK_SUP = 0.22
V6_LAMBDA_AMB_SUP = 0.50
V6_LAMBDA_ALIGN = 0.30
V6_LAMBDA_ENT_MIN = 0.01
V6_LAMBDA_ENT_MAX = 0.025
V6_LAMBDA_CONS = 0.04
V6_LAMBDA_PROTO = 0.04
V6_LAMBDA_UNKNOWN_REPEL = 0.008
V6_UNKNOWN_WARMUP_EPOCHS = 10
V6_UNKNOWN_RAMP_EPOCHS = 10

# v7 two-stage promotion three-bucket parameters
V7_RELIABLE_KEEP_Q = 0.16
V7_AMBIG_KEEP_Q = 0.28
V7_UNKNOWN_KEEP_Q = 0.04
V7_UNKNOWN_KEEP_Q_HARD = 0.06
V7_WEAK_MIN_KEEP = 96
V7_WEAK_TOPK = 3
V7_WEAK_SHARPEN = 1.00
V7_REL_WEIGHT_FLOOR = 0.80
V7_AMB_WEIGHT_FLOOR = 0.28
V7_AMB_WEIGHT_SCALE = 0.32
V7_WEAK_WEIGHT_FLOOR = 0.08
V7_WEAK_WEIGHT_SCALE = 0.12
V7_STAGE1_EPOCHS = 8
V7_PROMOTE_BLEND = 0.65
V7_PROMOTE_REL_FRACTION = 0.18
V7_PROMOTE_AMB_FRACTION = 0.32
V7_PROMOTE_MIN_REL = 24
V7_PROMOTE_MIN_AMB = 48
V7_PROMOTE_CONF = 0.72
V7_PROMOTE_MARGIN = 0.12
V7_AMB_CONF = 0.48
V7_AMB_MARGIN = 0.06
V7_LAMBDA_WEAK_SUP = 0.0
V7_LAMBDA_AMB_SUP = 0.42
V7_LAMBDA_ALIGN = 0.22
V7_LAMBDA_ENT_MIN = 0.008
V7_LAMBDA_ENT_MAX = 0.015
V7_LAMBDA_CONS = 0.06
V7_LAMBDA_PROTO = 0.05
V7_LAMBDA_UNKNOWN_REPEL = 0.003
V7_UNKNOWN_WARMUP_EPOCHS = 16
V7_UNKNOWN_RAMP_EPOCHS = 12



# DG-RAINCOAT (diagnosis-guided RAINCOAT-lite) parameters
DG_REL_KEEP_Q = 0.14
DG_AMB_KEEP_Q = 0.18
DG_UNK_KEEP_Q = 0.08
DG_STAGE1_EPOCHS = 8
DG_STAGE2_EPOCHS = 14
DG_WARM_SUP_SCALE = 0.75
DG_FINAL_SUP_SCALE = 1.00
DG_LAMBDA_ALIGN_STAGE1 = 0.26
DG_LAMBDA_ALIGN_STAGE2 = 0.16
DG_LAMBDA_ENT_MIN = 0.006
DG_LAMBDA_ENT_MAX = 0.020
DG_LAMBDA_CONS = 0.05
DG_LAMBDA_PROTO = 0.05
DG_LAMBDA_UNKNOWN_REPEL = 0.006
DG_MOVE_BLEND = 0.28
DG_STABLE_BLEND = 0.16

# v8 adaptive multi-refresh promotion parameters
V8_RELIABLE_KEEP_Q = 0.18
V8_AMBIG_KEEP_Q = 0.24
V8_UNKNOWN_KEEP_Q = 0.06
V8_UNKNOWN_KEEP_Q_HARD = 0.08
V8_REL_WEIGHT_FLOOR = 0.84
V8_AMB_WEIGHT_FLOOR = 0.26
V8_AMB_WEIGHT_SCALE = 0.30
V8_WEAK_WEIGHT_FLOOR = 0.06
V8_WEAK_WEIGHT_SCALE = 0.10
V8_STAGE0_EPOCHS = 6
V8_REFRESH_EPOCHS = 6
V8_REFRESH_ROUNDS = 3
V8_STABLE_K_REL = 2
V8_STABLE_K_AMB = 1
V8_PROMOTE_CONF_REL = 0.72
V8_PROMOTE_CONF_AMB = 0.50
V8_PROMOTE_MARGIN_REL = 0.12
V8_PROMOTE_MARGIN_AMB = 0.04
V8_PROMOTE_SCORE_REL_Q = 0.72
V8_PROMOTE_SCORE_AMB_Q = 0.52
V8_LAMBDA_ALIGN = 0.18
V8_LAMBDA_ENT_MIN = 0.006
V8_LAMBDA_ENT_MAX = 0.016
V8_LAMBDA_CONS = 0.05
V8_LAMBDA_PROTO = 0.05
V8_LAMBDA_UNKNOWN_REPEL = 0.004
V8_STABILITY_BLEND = 0.16
DG_RESCUE_SDOM_Q = 0.62
DG_RESCUE_CONF = 0.56
DG_RESCUE_MARGIN = 0.05
DG_SUPPORT_WEIGHT_FLOOR = 0.08
DG_SUPPORT_WEIGHT_SCALE = 0.72
DG_UNKNOWN_EXTREME_Q = 0.94

ADAPT_GRAD_CLIP = 5.0
SAFE_REPLAY_ACC_DROP = 0.10
SAFE_REPLAY_CE_SCALE = 1.40
SAFE_MIN_UNIQUE_CLASS_FRAC = 0.20
SAFE_MAX_DOM_CLASS_SHARE = 0.92
SAFE_MAX_MEAN_CONF = 0.995

V81_COLD_WEAK_Q = 0.42
V81_COLD_WEIGHT_FLOOR = 0.02
V81_COLD_WEIGHT_SCALE = 0.04
V81_CLASS_REL_FRAC = 0.12
V81_CLASS_AMB_FRAC = 0.20
V81_SEMIWEAK_TOPK = 3


# explicit historical version tracking for DG_RAINCOAT and TBV8
DGR_VERSION_CFG = {
    "v9": dict(
        rel_keep_q=0.14, amb_keep_q=0.18, unk_keep_q=0.08,
        warm_rel_scale=DG_WARM_SUP_SCALE, warm_amb_scale=0.55, warm_amb_init=0.35,
        mix_seed_blend=0.75,
        rescue_mode="low_sdom", rescue_sid_q=0.52, rescue_sdom_q=0.48,
        rescue_conf=0.50, rescue_margin=0.03, rescue_proto_q=0.35,
        rescue_frac=0.10, unk_divisor=4,
        common_score_blend=(1.00, 0.85, 0.00), rescue_rank_blend=(0.60, 0.40, 0.00),
        support_mode="nonunk", support_common_q=0.46, support_conf_q=0.45, support_unknown_q=0.72,
        support_fallback_div=6, support_refine_mix=0.80,
        support_unknown_penalty=0.30, support_weight_floor=0.10, support_weight_scale=0.90,
        rescue_min_weight=0.32, reliable_min_weight=0.78,
        ambiguous_min_weight=0.24, ambiguous_topk=max(2, AMBIG_TOPK), ambiguous_sharpen=1.00,
        low_weight_cut=0.18, align_mode="nonunk", align_common_q=0.38, align_unknown_q=0.80,
        final_ent_max_scale=0.50),
    "v10": dict(
        rel_keep_q=0.14, amb_keep_q=0.18, unk_keep_q=0.08,
        warm_rel_scale=0.95, warm_amb_scale=0.55, warm_amb_init=0.22,
        mix_seed_blend=0.70,
        rescue_mode="high_sdom", rescue_sid_q=0.48, rescue_sdom_q=0.62,
        rescue_conf=0.56, rescue_margin=0.05, rescue_proto_q=0.32,
        rescue_frac=0.08, unk_divisor=5,
        common_score_blend=(1.00, 0.95, 0.00), rescue_rank_blend=(0.56, 0.32, 0.12),
        support_mode="filtered", support_common_q=0.46, support_conf_q=0.45, support_unknown_q=0.72,
        support_fallback_div=6, support_refine_mix=0.76,
        support_unknown_penalty=0.18, support_weight_floor=0.08, support_weight_scale=0.72,
        rescue_min_weight=0.48, reliable_min_weight=0.88,
        ambiguous_min_weight=0.12, ambiguous_topk=max(2, AMBIG_TOPK), ambiguous_sharpen=0.97,
        low_weight_cut=0.20, align_mode="nonunk", align_common_q=0.38, align_unknown_q=0.80,
        final_ent_max_scale=0.50),
    "v11": dict(
        rel_keep_q=0.14, amb_keep_q=0.16, unk_keep_q=0.08,
        warm_rel_scale=0.95, warm_amb_scale=0.45, warm_amb_init=0.16,
        mix_seed_blend=0.66,
        rescue_mode="high_sdom", rescue_sid_q=0.52, rescue_sdom_q=0.68,
        rescue_conf=0.60, rescue_margin=0.06, rescue_proto_q=0.40,
        rescue_frac=0.07, unk_divisor=5,
        common_score_blend=(1.00, 0.92, 0.00), rescue_rank_blend=(0.60, 0.24, 0.16),
        support_mode="filtered", support_common_q=0.54, support_conf_q=0.54, support_unknown_q=0.66,
        support_fallback_div=7, support_refine_mix=0.78,
        support_unknown_penalty=0.24, support_weight_floor=0.12, support_weight_scale=0.58,
        rescue_min_weight=0.58, reliable_min_weight=0.90,
        ambiguous_min_weight=0.08, ambiguous_topk=2, ambiguous_sharpen=0.99,
        low_weight_cut=0.16, align_mode="safe_nonunk", align_common_q=0.42, align_unknown_q=0.82,
        final_ent_max_scale=0.45),
    "v12": dict(
        rel_keep_q=0.14, amb_keep_q=0.17, unk_keep_q=0.08,
        warm_rel_scale=0.95, warm_amb_scale=0.42, warm_amb_init=0.14,
        mix_seed_blend=0.68,
        rescue_mode="high_sdom", rescue_sid_q=0.50, rescue_sdom_q=0.66,
        rescue_conf=0.58, rescue_margin=0.05, rescue_proto_q=0.38,
        rescue_frac=0.07, unk_divisor=5,
        common_score_blend=(1.00, 0.90, 0.00), rescue_rank_blend=(0.60, 0.24, 0.16),
        support_mode="filtered", support_common_q=0.50, support_conf_q=0.50, support_unknown_q=0.68,
        support_fallback_div=7, support_refine_mix=0.79,
        support_unknown_penalty=0.22, support_weight_floor=0.10, support_weight_scale=0.60,
        rescue_min_weight=0.60, reliable_min_weight=0.90,
        ambiguous_min_weight=0.06, ambiguous_topk=2, ambiguous_sharpen=1.00,
        low_weight_cut=0.12, align_mode="safe_nonunk", align_common_q=0.38, align_unknown_q=0.84,
        final_ent_max_scale=0.42,
        stable_hard_thr=0.67, support_stable_min=0.67,
        amb_near_q=0.58, amb_near_min_weight=0.18, amb_far_min_weight=0.04, amb_far_max_weight=0.10, amb_far_sharpen=1.03),
}

TBV8_VERSION_CFG = {
    "v9": dict(
        rel_keep_q=0.18, amb_keep_q=0.24, unknown_keep_q=0.06, unknown_keep_q_hard=0.08,
        amb_quant_floor=0.36, amb_conf_scale=0.90, amb_conf_floor=0.40,
        cold_weak_q=0.55, class_rel_frac=0.10, class_amb_frac=0.14,
        semiweak_topk=2, promote_conf_rel=0.72, promote_conf_amb=0.55,
        promote_margin_rel=0.12, promote_margin_amb=0.06,
        amb_weight_bonus=0.00, weak_weight_bonus=0.02),
    "v10": dict(
        rel_keep_q=0.18, amb_keep_q=0.24, unknown_keep_q=0.06, unknown_keep_q_hard=0.08,
        amb_quant_floor=0.32, amb_conf_scale=0.85, amb_conf_floor=0.36,
        cold_weak_q=0.42, class_rel_frac=0.12, class_amb_frac=0.20,
        semiweak_topk=3, promote_conf_rel=0.72, promote_conf_amb=0.50,
        promote_margin_rel=0.12, promote_margin_amb=0.04,
        amb_weight_bonus=0.06, weak_weight_bonus=0.08),
    "v11": dict(
        rel_keep_q=0.18, amb_keep_q=0.26, unknown_keep_q=0.06, unknown_keep_q_hard=0.08,
        amb_quant_floor=0.30, amb_conf_scale=0.82, amb_conf_floor=0.34,
        cold_weak_q=0.34, class_rel_frac=0.12, class_amb_frac=0.24,
        semiweak_topk=4, promote_conf_rel=0.72, promote_conf_amb=0.47,
        promote_margin_rel=0.12, promote_margin_amb=0.03,
        amb_weight_bonus=0.10, weak_weight_bonus=0.10),
    "v12": dict(
        rel_keep_q=0.18, amb_keep_q=0.24, unknown_keep_q=0.06, unknown_keep_q_hard=0.08,
        amb_quant_floor=0.36, amb_conf_scale=0.90, amb_conf_floor=0.40,
        cold_weak_q=0.52, class_rel_frac=0.10, class_amb_frac=0.16,
        semiweak_topk=2, promote_conf_rel=0.72, promote_conf_amb=0.54,
        promote_margin_rel=0.12, promote_margin_amb=0.05,
        amb_weight_bonus=0.02, weak_weight_bonus=0.03,
        rarity_boost=0.22, extra_amb_cap=2),
}

LQ_METHOD_CFG = {
    "dgr": dict(
        weights=[1.30, 0.95, 1.00, 0.85, 0.70],
        refine_iters=3,
        src_mix=0.72,
        class_balance_strength=0.18,
        temp_low=0.60,
        temp_high=0.95,
        low_rel_thr=0.42,
        mid_rel_thr=0.62,
        topk_low=3,
        topk_mid=2,
        low_weight_floor=0.34,
        weight_gain=0.55,
        lambda_align=DG_LAMBDA_ALIGN_STAGE2 * 0.55,
        lambda_ent_min=DG_LAMBDA_ENT_MIN * 0.45,
        lambda_ent_max=DG_LAMBDA_ENT_MAX * 0.28,
        lambda_cons=DG_LAMBDA_CONS * 0.95,
        lambda_proto=DG_LAMBDA_PROTO * 0.90),
    "tbv8": dict(
        weights=[1.20, 1.00, 1.05, 0.90, 0.75],
        refine_iters=3,
        src_mix=0.70,
        class_balance_strength=0.16,
        temp_low=0.58,
        temp_high=0.92,
        low_rel_thr=0.40,
        mid_rel_thr=0.60,
        topk_low=3,
        topk_mid=2,
        low_weight_floor=0.32,
        weight_gain=0.50,
        lambda_align=V8_LAMBDA_ALIGN * 0.55,
        lambda_ent_min=V8_LAMBDA_ENT_MIN * 0.40,
        lambda_ent_max=V8_LAMBDA_ENT_MAX * 0.30,
        lambda_cons=V8_LAMBDA_CONS * 1.05,
        lambda_proto=V8_LAMBDA_PROTO * 0.95),
}
LQ_AUG_VIEWS = 2
LQ_AUG_NOISE_SCALE = 0.85


# v14 stable label-quality guard / conservative refinement
V14_LQ_MIN_TOTAL = 48
V14_LQ_MIN_PER_CLASS = 2
V14_LQ_REQUIRE_TWO_CLASSES = True
V14_LQ_HIGH_CONF = 0.82
V14_LQ_LOW_CONF = 0.58
V14_LQ_FREEZE_REL = 0.72
V14_LQ_SEED_BLEND_HIGH = 0.82
V14_LQ_SEED_BLEND_MID = 0.62
V14_LQ_SEED_BLEND_LOW = 0.80
V14_LQ_MAX_WEIGHT = 1.15
V14_LQ_AUG_VIEWS = 1
V14_LQ_AUG_NOISE_SCALE = 0.55
V14_SAFE_STD_FALLBACK = 1.0

# v15 quality-aware pseudo-labeling
V15_QA_MIN_TOTAL = 32
V15_QA_MIN_PER_CLASS = 1
V15_QA_REQUIRE_TWO_CLASSES = False
V15_QA_AUG_VIEWS = 2
V15_QA_AUG_NOISE_SCALE = 0.70
V15_QA_MAX_WEIGHT = 1.20


V15_QA_CFG = {
    "dgr": dict(
        weights=[1.20, 1.00, 1.00, 0.92, 0.72],
        refine_iters=2,
        src_mix=0.72,
        class_balance_strength=0.10,
        temp_low=0.62,
        temp_high=0.98,
        low_weight_floor=0.10,
        weight_gain=0.95,
        gamma=1.35,
        mid_rel_thr=0.55,
        high_rel_thr=0.80,
        seed_min=0.10,
        seed_max=0.24,
        ref_min=0.38,
        ref_max=0.58,
        lambda_align=DG_LAMBDA_ALIGN_STAGE2 * 0.90,
        lambda_ent_min=DG_LAMBDA_ENT_MIN * 0.80,
        lambda_ent_max=DG_LAMBDA_ENT_MAX * 0.70,
        lambda_cons=DG_LAMBDA_CONS * 1.00,
        lambda_proto=DG_LAMBDA_PROTO * 1.00),
    "tbv8": dict(
        weights=[1.15, 1.00, 1.05, 0.95, 0.75],
        refine_iters=2,
        src_mix=0.70,
        class_balance_strength=0.10,
        temp_low=0.60,
        temp_high=0.96,
        low_weight_floor=0.08,
        weight_gain=0.92,
        gamma=1.30,
        mid_rel_thr=0.54,
        high_rel_thr=0.78,
        seed_min=0.10,
        seed_max=0.22,
        ref_min=0.40,
        ref_max=0.60,
        lambda_align=V8_LAMBDA_ALIGN * 0.90,
        lambda_ent_min=V8_LAMBDA_ENT_MIN * 0.85,
        lambda_ent_max=V8_LAMBDA_ENT_MAX * 0.75,
        lambda_cons=V8_LAMBDA_CONS * 1.05,
        lambda_proto=V8_LAMBDA_PROTO * 1.00),
}

V16_QA_MIN_TOTAL = 32
V16_QA_MIN_PER_CLASS = 1
V16_QA_REQUIRE_TWO_CLASSES = False
V16_QA_AUG_VIEWS = 2
V16_QA_AUG_NOISE_SCALE = 0.65
V16_QA_MAX_WEIGHT = 1.20


V16_QA_CFG = {
    "dgr": dict(
        weights=[1.18, 1.00, 1.00, 0.92, 0.72],
        refine_iters=2,
        src_mix=0.72,
        class_balance_strength=0.08,
        temp_low=0.58,
        temp_high=1.05,
        low_weight_floor=0.08,
        gamma_known=1.25,
        gamma_class=1.12,
        gamma_overall=1.18,
        known_floor=0.24,
        known_gain=0.86,
        class_floor=0.34,
        class_gain=0.72,
        seed_min=0.08,
        seed_max=0.22,
        ref_min=0.34,
        ref_max=0.56,
        prior_min=0.18,
        prior_max=0.40,
        uniform_max=0.22,
        mid_rel_thr=0.54,
        high_rel_thr=0.80,
        align_known_thr=0.48,
        lambda_align=DG_LAMBDA_ALIGN_STAGE2 * 0.90,
        lambda_ent_min=DG_LAMBDA_ENT_MIN * 0.82,
        lambda_ent_max=DG_LAMBDA_ENT_MAX * 0.72,
        lambda_cons=DG_LAMBDA_CONS * 1.00,
        lambda_proto=DG_LAMBDA_PROTO * 1.00),
    "tbv8": dict(
        weights=[1.14, 1.00, 1.05, 0.95, 0.75],
        refine_iters=2,
        src_mix=0.70,
        class_balance_strength=0.08,
        temp_low=0.56,
        temp_high=1.02,
        low_weight_floor=0.07,
        gamma_known=1.22,
        gamma_class=1.10,
        gamma_overall=1.15,
        known_floor=0.24,
        known_gain=0.84,
        class_floor=0.34,
        class_gain=0.70,
        seed_min=0.08,
        seed_max=0.20,
        ref_min=0.36,
        ref_max=0.58,
        prior_min=0.18,
        prior_max=0.42,
        uniform_max=0.22,
        mid_rel_thr=0.53,
        high_rel_thr=0.78,
        align_known_thr=0.46,
        lambda_align=V8_LAMBDA_ALIGN * 0.92,
        lambda_ent_min=V8_LAMBDA_ENT_MIN * 0.86,
        lambda_ent_max=V8_LAMBDA_ENT_MAX * 0.76,
        lambda_cons=V8_LAMBDA_CONS * 1.05,
        lambda_proto=V8_LAMBDA_PROTO * 1.00),
}

V17_QA_MIN_TOTAL = V16_QA_MIN_TOTAL
V17_QA_MIN_PER_CLASS = V16_QA_MIN_PER_CLASS
V17_QA_REQUIRE_TWO_CLASSES = V16_QA_REQUIRE_TWO_CLASSES
V17_QA_AUG_VIEWS = V16_QA_AUG_VIEWS
V17_QA_AUG_NOISE_SCALE = V16_QA_AUG_NOISE_SCALE
V17_QA_MAX_WEIGHT = V16_QA_MAX_WEIGHT
V17_LOCAL_BLOCK = 256

V17_QA_CFG = {
    "dgr": dict(
        **V16_QA_CFG["dgr"],
        local_k=24,
        local_block=V17_LOCAL_BLOCK,
        local_sim_floor=0.10,
        local_blend_known=0.32,
        local_blend_class=0.36,
        local_align_thr=0.50),
    "tbv8": dict(
        **V16_QA_CFG["tbv8"],
        local_k=20,
        local_block=V17_LOCAL_BLOCK,
        local_sim_floor=0.10,
        local_blend_known=0.30,
        local_blend_class=0.34,
        local_align_thr=0.48),
}



# module2 v4 integration (used as soft scorer for module3)
M2V4_KNOWN_Q = 0.35
M2V4_REL_Q = 0.70
M2V4_AMB_Q = 0.45
M2V4_UNK_Q = 0.70
M2V4_MIN_KNOWN_ABS = 0.50
M2V4_MIN_DRIFT_ABS = 0.45
M2V4_ALIGN_KNOWN_Q = 0.25

# v19: use Module2-v4 scores only as mild weighting / curriculum guidance
# instead of hard route takeover
V19_M2W_CFG = {
    "dgr": dict(
        lambda_align=V17_QA_CFG["dgr"]["lambda_align"] * 0.95,
        lambda_ent_min=V17_QA_CFG["dgr"]["lambda_ent_min"] * 0.90,
        lambda_ent_max=V17_QA_CFG["dgr"]["lambda_ent_max"] * 0.90,
        lambda_cons=V17_QA_CFG["dgr"]["lambda_cons"] * 1.00,
        lambda_proto=V17_QA_CFG["dgr"]["lambda_proto"] * 1.00,
        align_q=0.25,
        extra_unknown_thr=0.34),
    "tbv8": dict(
        lambda_align=V17_QA_CFG["tbv8"]["lambda_align"] * 0.90,
        lambda_ent_min=V17_QA_CFG["tbv8"]["lambda_ent_min"] * 0.90,
        lambda_ent_max=V17_QA_CFG["tbv8"]["lambda_ent_max"] * 0.90,
        lambda_cons=V17_QA_CFG["tbv8"]["lambda_cons"] * 1.00,
        lambda_proto=V17_QA_CFG["tbv8"]["lambda_proto"] * 1.00,
        align_q=0.30,
        extra_unknown_thr=0.32),
}



# ablation controls
UNKNOWN_LOSS_DEFAULT = "uniform_kl"
UNKNOWN_ENERGY_MARGIN = 0.10

# v22 = WeightedEntry-Minimal (base defaults)
V22_GAMMA_DRIFT = 1.0
V22_GAMMA_NOTUNK = 1.0
V22_WMAX = 1.0
V22_CONF_SMOOTH = 0.03
V22_SUP_EVAL_THR = 0.20
V22_UNK_EVAL_THR = 0.50
V22_LAMBDA_SUP = 1.0
V22_LAMBDA_PROTO = 0.25
V22_LAMBDA_STAB = 0.02
V22_LAMBDA_UNKE = 0.045
V22_ENERGY_BG_Q = 0.60
V22_ENERGY_BG_EMA = 0.90
V22_UNKNOWN_ENERGY_MARGIN = UNKNOWN_ENERGY_MARGIN

# v22 tuning design
# Recommendation:
# 1) run single-factor sweeps first on the hard protocols
# 2) keep the top 2-3 values
# 3) then run a small joint grid with those shortlisted values
V22_SWEEP_PLAN = "manual"   # "lambda_unkE", "gamma_notunk", "lambda_stab", "joint_small", "manual"
V22_INCLUDE_BASELINE_VARIANT = False

V22_JOINT_SMALL_VARIANTS = [
    dict(tag="joint_a", gamma_notunk=0.75, lambda_unkE=0.05, lambda_stab=0.02),
    dict(tag="joint_b", gamma_notunk=0.75, lambda_unkE=0.05, lambda_stab=0.05),
    dict(tag="joint_c", gamma_notunk=1.00, lambda_unkE=0.05, lambda_stab=0.02),
    dict(tag="joint_d", gamma_notunk=1.00, lambda_unkE=0.06, lambda_stab=0.02)]

# Three operating-point profiles for module2_v8 methods.
# highsec  : strict security point. In v30.4 this inherits the former Balanced profile.
# balanced : mid-point between strict security and business continuity; tuned from the former BizCont profile.
# bizcont  : more permissive business-continuity point; further lowers FRR / raises known-pass coverage at the cost of a modest FAR increase.
V22_MANUAL_VARIANTS = [
    dict(
        tag="highsec",
        profile_display="HighSec",
        gamma_notunk=1.0,
        lambda_unkE=0.038,
        lambda_stab=0.010,
        v19_lambda_align=0.065,
        v19_lambda_ent_min=0.015,
        v19_lambda_ent_max=0.035,
        v19_lambda_proto=0.21,
        v19_unknown_energy_margin=0.08,
    ),
    dict(
        tag="balanced",
        profile_display="Balanced",
        gamma_notunk=1.0,
        lambda_unkE=0.026,
        lambda_stab=0.000,
        v19_lambda_align=0.045,
        v19_lambda_ent_min=0.008,
        v19_lambda_ent_max=0.018,
        v19_lambda_proto=0.16,
        v19_unknown_energy_margin=0.055,
    ),
    dict(
        tag="bizcont",
        profile_display="BizCont",
        gamma_notunk=1.0,
        lambda_unkE=0.022,
        lambda_stab=0.000,
        v19_lambda_align=0.035,
        v19_lambda_ent_min=0.006,
        v19_lambda_ent_max=0.014,
        v19_lambda_proto=0.14,
        v19_unknown_energy_margin=0.045,
    ),
]

def make_v19_profile_name(tag):
    return f"DG_RAINCOAT_v19M2W_EnergyU__{tag}"

# manual-run selector
# Use "all" to run all manual parameter sets.
# Or pass a single tag like "base"/"joint_a", or a list like ["base", "joint_a", "joint_d"].
V22_MANUAL_SELECTION = "all"

# tau-conf sweep controls (kept for backward compatibility with run_one_split)
# In the small-grid v24 runs we disable this by default, but leave the variables
# defined so the old sweep block never crashes due to missing globals.
RUN_TAU_CONF_SWEEP = False
TAU_CONF_SWEEP_QUANTILES = []


def _v22_slug(value):
    s = f"{value:.4f}".rstrip("0").rstrip(".")
    return s.replace("-", "m").replace(".", "p")

def make_v22_variant_name(tag):
    if tag == "base":
        return "DG_RAINCOAT_v22WeightedEntryMinimal"
    return f"DG_RAINCOAT_v22WeightedEntryMinimal__{tag}"


def make_v35_variant_name(tag):
    if tag == "base":
        return "DG_RAINCOAT_v35HybridEntry"
    return f"DG_RAINCOAT_v35HybridEntry__{tag}"


def normalize_v22_manual_selection():
    if V22_MANUAL_SELECTION == "all":
        return "all"
    if isinstance(V22_MANUAL_SELECTION, str):
        selected = [V22_MANUAL_SELECTION]
    elif isinstance(V22_MANUAL_SELECTION, (list, tuple, set)):
        selected = list(V22_MANUAL_SELECTION)
    else:
        raise ValueError("V22_MANUAL_SELECTION must be 'all', a string tag, or a list/tuple/set of tags.")
    selected = [str(x) for x in selected]
    if len(selected) == 0:
        raise ValueError("V22_MANUAL_SELECTION cannot be empty.")
    return selected

def build_v22_active_variants():
    base = dict(
        gamma_drift=V22_GAMMA_DRIFT,
        gamma_notunk=V22_GAMMA_NOTUNK,
        wmax=V22_WMAX,
        conf_smooth=V22_CONF_SMOOTH,
        sup_eval_thr=V22_SUP_EVAL_THR,
        unk_eval_thr=V22_UNK_EVAL_THR,
        lambda_sup=V22_LAMBDA_SUP,
        lambda_proto=V22_LAMBDA_PROTO,
        lambda_stab=V22_LAMBDA_STAB,
        lambda_unkE=V22_LAMBDA_UNKE,
        energy_bg_q=V22_ENERGY_BG_Q,
        energy_bg_ema=V22_ENERGY_BG_EMA,
        unknown_energy_margin=V22_UNKNOWN_ENERGY_MARGIN)

    variants = []
    if V22_INCLUDE_BASELINE_VARIANT:
        variants.append(dict(name=make_v22_variant_name("base"), tag="base", **base))

    if V22_SWEEP_PLAN == "lambda_unkE":
        for val in [0.03, 0.05, 0.06, 0.08, 0.10]:
            if float(val) == float(base["lambda_unkE"]):
                continue
            variants.append(dict(name=make_v22_variant_name(f"lu{_v22_slug(val)}"), tag=f"lu{_v22_slug(val)}", **{**base, "lambda_unkE": float(val)}))
    elif V22_SWEEP_PLAN == "gamma_notunk":
        for val in [0.50, 0.75, 1.00, 1.25, 1.50]:
            if float(val) == float(base["gamma_notunk"]):
                continue
            variants.append(dict(name=make_v22_variant_name(f"gu{_v22_slug(val)}"), tag=f"gu{_v22_slug(val)}", **{**base, "gamma_notunk": float(val)}))
    elif V22_SWEEP_PLAN == "lambda_stab":
        for val in [0.00, 0.02, 0.05, 0.08, 0.10]:
            if float(val) == float(base["lambda_stab"]):
                continue
            variants.append(dict(name=make_v22_variant_name(f"ls{_v22_slug(val)}"), tag=f"ls{_v22_slug(val)}", **{**base, "lambda_stab": float(val)}))
    elif V22_SWEEP_PLAN == "joint_small":
        for item in V22_JOINT_SMALL_VARIANTS:
            variants.append(dict(name=make_v22_variant_name(item["tag"]), **{**base, **item}))
    elif V22_SWEEP_PLAN == "manual":
        selection = normalize_v22_manual_selection()
        manual_items = list(V22_MANUAL_VARIANTS)
        manual_tag_set = {item.get("tag", "") for item in manual_items}
        if selection != "all":
            selected_tags = set(selection)
            unknown_tags = sorted([tag for tag in selected_tags if tag != "base" and tag not in manual_tag_set])
            if len(unknown_tags) > 0:
                raise ValueError(f"Unknown tags in V22_MANUAL_SELECTION: {unknown_tags}")
            if "base" not in selected_tags:
                variants = [v for v in variants if v.get("tag") != "base"]
            manual_items = [item for item in manual_items if item.get("tag", "") in selected_tags]
        for item in manual_items:
            tag = item.get("tag", f"manual_{len(variants)}")
            variants.append(dict(name=make_v22_variant_name(tag), **{**base, **item}))
    else:
        raise ValueError(f"Unknown V22_SWEEP_PLAN: {V22_SWEEP_PLAN}")

    dedup = []
    seen = set()
    for item in variants:
        if item["name"] in seen:
            continue
        seen.add(item["name"])
        dedup.append(item)
    return dedup


def get_v22_selection_suffix():
    if V22_SWEEP_PLAN != "manual":
        return V22_SWEEP_PLAN
    selection = normalize_v22_manual_selection()
    if selection == "all":
        return "allsets"
    safe = "_".join([str(x).replace("-", "m").replace(".", "p") for x in selection])
    return f"sel_{safe}"

def refresh_v22_runtime_config():
    global V22_ACTIVE_VARIANTS, V22_ACTIVE_METHOD_NAMES
    global TAU_CONF_SWEEP_METHODS
    global FULL_METHOD_ORDER, FAST_DEV_METHOD_ORDER, V22_TUNING_METHOD_ORDER, METHOD_ORDER
    global RUN_BASENAME, RUN_DIR

    V22_ACTIVE_VARIANTS = build_v22_active_variants()
    V22_ACTIVE_METHOD_NAMES = [v["name"] for v in V22_ACTIVE_VARIANTS]
    PROFILED_V19_METHOD_NAMES = [make_v19_profile_name(v["tag"]) for v in V22_ACTIVE_VARIANTS]

    TAU_CONF_SWEEP_METHODS = PROFILED_V19_METHOD_NAMES + [nm for nm in V22_ACTIVE_METHOD_NAMES if "WeightedEntryMinimal" in nm]

    # In v30.4 ext-compare we keep:
    #   (1) three Module2-v8 single-configuration reference methods with no module3 profile selection,
    #   (2) module2_v8 methods under three module3 operating-point profiles, and
    #   (3) four independent external baselines.
    SOURCE_ONLY_METHOD = "SourceOnly"

    MODULE2_ONLY_REFERENCE_METHOD_ORDER = [
        SOURCE_ONLY_METHOD,
        "ProtoRefineSoft",
        "M2V8RouteSplit"]

    PROFILED_OWN_METHOD_ORDER = PROFILED_V19_METHOD_NAMES + V22_ACTIVE_METHOD_NAMES

    EXT_COMPARE_METHOD_ORDER = MODULE2_ONLY_REFERENCE_METHOD_ORDER + PROFILED_OWN_METHOD_ORDER + [
        "PCPD_recon",
        "OVANet_recon",
        "COMET_recon",
        "JRFFP_SC_recon"]

    # Keep the compact order in all modes. The three restored reference methods remain
    # single-configuration baselines and are not part of the module3 profile-selection logic.
    FULL_METHOD_ORDER = list(EXT_COMPARE_METHOD_ORDER)
    FAST_DEV_METHOD_ORDER = list(EXT_COMPARE_METHOD_ORDER)
    V22_TUNING_METHOD_ORDER = list(EXT_COMPARE_METHOD_ORDER)

    if EXPERIMENT_MODE == "ext_compare":
        METHOD_ORDER = EXT_COMPARE_METHOD_ORDER
    elif EXPERIMENT_MODE == "v22_tuning":
        METHOD_ORDER = V22_TUNING_METHOD_ORDER
    elif EXPERIMENT_MODE == "fast_dev":
        METHOD_ORDER = FAST_DEV_METHOD_ORDER
    else:
        METHOD_ORDER = FULL_METHOD_ORDER

    RUN_BASENAME = f"{RUN_VERSION_TAG}__{RUN_HOST_TAG}__{RUN_TIMESTAMP}"
    RUN_DIR = f"../owdc_results/results_SODA_RFF/run_{RUN_BASENAME}"
    _safe_makedirs(RUN_DIR)

refresh_v22_runtime_config()


STAGEA_CACHE_ENABLED = True
STAGEA_CACHE_FORCE_REBUILD = False
STAGEA_CACHE_FILENAME = "sA_soda_rff.pkl"
STAGEA_CACHE_VERSION = "soda_rff_stageA_cache_v1"

RUN_METADATA = dict(
    code_file_name=CODE_FILE_NAME,
    run_version_tag=RUN_VERSION_TAG,
    run_notes=RUN_NOTES,
    run_host_tag=RUN_HOST_TAG,
    experiment_mode=EXPERIMENT_MODE,
    run_timestamp=RUN_TIMESTAMP,
    run_basename=RUN_BASENAME,
    run_dir=RUN_DIR,
    v22_sweep_plan=V22_SWEEP_PLAN,
    v22_active_variants=V22_ACTIVE_VARIANTS,
    selected_protocol_combos=SELECTED_PROTOCOL_COMBOS,
    tx_split_repeats=TX_SPLIT_REPEATS)


cfg = dict(
    SEED=SEED, TX_TOTAL_USE=TX_TOTAL_USE, RX_TOTAL_USE=RX_TOTAL_USE, DAY_TOTAL_USE=DAY_TOTAL_USE,
    KNOWN_TX_NUM=KNOWN_TX_NUM, SOURCE_RX_NUM=SOURCE_RX_NUM, TX_SPLIT_REPEATS=TX_SPLIT_REPEATS,
    RX_PROTOCOL_LIBRARY=RX_PROTOCOL_LIBRARY, TX_PROTOCOL_LIBRARY=TX_PROTOCOL_LIBRARY,
    SELECTED_RX_PROTOCOLS=SELECTED_RX_PROTOCOLS, SELECTED_TX_PROTOCOLS=SELECTED_TX_PROTOCOLS,
    SELECTED_PROTOCOL_COMBOS=SELECTED_PROTOCOL_COMBOS,
    TRAIN_DATES=TRAIN_DATES, TEST_DATES_RX=TEST_DATES_RX, TEST_DATES_DAY=TEST_DATES_DAY,
    Z_FROM_EQ=Z_FROM_EQ, D_FROM=D_FROM, MAX_SIG_PER_TRIPLE=MAX_SIG_PER_TRIPLE,
    BATCH_SIZE=BATCH_SIZE, EPOCHS=EPOCHS, LR=LR, WEIGHT_DECAY=WEIGHT_DECAY, PATIENCE=PATIENCE,
    IN_PLANES=IN_PLANES, DROPOUT=DROPOUT, D_DIM=D_DIM,
    FUSION_LAM=FUSION_LAM, FRR_TARGET=FRR_TARGET, FALSE_DRIFT_TARGET=FALSE_DRIFT_TARGET,
    ADAPT_FRAC=ADAPT_FRAC, MAX_ADAPT_PER_SET=MAX_ADAPT_PER_SET, MAX_EVAL_PER_SET=MAX_EVAL_PER_SET,
    ADAPTER_BOTTLENECK=ADAPTER_BOTTLENECK, ADAPTER_SCALE=ADAPTER_SCALE,
    ADAPT_EPOCHS=ADAPT_EPOCHS, ADAPT_LR=ADAPT_LR, ADAPT_WEIGHT_DECAY=ADAPT_WEIGHT_DECAY, ADAPT_BATCH=ADAPT_BATCH,
    REPLAY_PER_CLASS=REPLAY_PER_CLASS, LAMBDA_REPLAY_CE=LAMBDA_REPLAY_CE, LAMBDA_REPLAY_ANCHOR=LAMBDA_REPLAY_ANCHOR,
    PSEUDO_MIN_KEEP=PSEUDO_MIN_KEEP, SOFT_T=SOFT_T, PROTO_T=PROTO_T, KNN_TOPK=KNN_TOPK, KNN_MEM_PER_CLASS=KNN_MEM_PER_CLASS,
    W_LOGIT=W_LOGIT, W_PROTO=W_PROTO, W_KNN=W_KNN, CONF_STABLE_Q=CONF_STABLE_Q,
    M2V4_KNOWN_Q=M2V4_KNOWN_Q, M2V4_REL_Q=M2V4_REL_Q, M2V4_AMB_Q=M2V4_AMB_Q, M2V4_UNK_Q=M2V4_UNK_Q,
    M2V4_MIN_KNOWN_ABS=M2V4_MIN_KNOWN_ABS, M2V4_MIN_DRIFT_ABS=M2V4_MIN_DRIFT_ABS, M2V4_ALIGN_KNOWN_Q=M2V4_ALIGN_KNOWN_Q,
    CONF_DRIFT_Q=CONF_DRIFT_Q, REFINE_ITERS=REFINE_ITERS, UNKNOWN_SCORE_Q=UNKNOWN_SCORE_Q,
    EMB_AUG_NOISE=EMB_AUG_NOISE, MMD_SIGMAS=MMD_SIGMAS,
    LAMBDA_ALIGN=LAMBDA_ALIGN, LAMBDA_ENT_MIN=LAMBDA_ENT_MIN, LAMBDA_ENT_MAX=LAMBDA_ENT_MAX,
    LAMBDA_CONS=LAMBDA_CONS, LAMBDA_PROTO=LAMBDA_PROTO,
GCODWFA_ALIGN_MAX=GCODWFA_ALIGN_MAX, RAIN_STAGE1_EPOCHS=RAIN_STAGE1_EPOCHS, RAIN_STAGE2_EPOCHS=RAIN_STAGE2_EPOCHS,
RELIABLE_KEEP_Q=RELIABLE_KEEP_Q, AMBIG_KEEP_Q=AMBIG_KEEP_Q, UNKNOWN_KEEP_Q=UNKNOWN_KEEP_Q,
AMBIG_TOPK=AMBIG_TOPK, AMBIG_WEIGHT_SCALE=AMBIG_WEIGHT_SCALE,
RELIABLE_WEIGHT_FLOOR=RELIABLE_WEIGHT_FLOOR, AMBIG_WEIGHT_FLOOR=AMBIG_WEIGHT_FLOOR,
EMA_MOMENTUM=EMA_MOMENTUM, EMA_TEACHER_BLEND=EMA_TEACHER_BLEND, TEACHER_TEMP=TEACHER_TEMP,
THREE_BUCKET_EPOCHS=THREE_BUCKET_EPOCHS, LAMBDA_AMB_SUP=LAMBDA_AMB_SUP,
LAMBDA_SRC_PROTO=LAMBDA_SRC_PROTO, LAMBDA_SRC_LOGIT=LAMBDA_SRC_LOGIT,
LAMBDA_UNKNOWN_REPEL=LAMBDA_UNKNOWN_REPEL, UNKNOWN_REPEL_MARGIN=UNKNOWN_REPEL_MARGIN,
UNKNOWN_WARMUP_EPOCHS=UNKNOWN_WARMUP_EPOCHS, UNKNOWN_RAMP_EPOCHS=UNKNOWN_RAMP_EPOCHS,
V7_RELIABLE_KEEP_Q=V7_RELIABLE_KEEP_Q, V7_AMBIG_KEEP_Q=V7_AMBIG_KEEP_Q,
V7_UNKNOWN_KEEP_Q=V7_UNKNOWN_KEEP_Q, V7_UNKNOWN_KEEP_Q_HARD=V7_UNKNOWN_KEEP_Q_HARD,
V7_WEAK_MIN_KEEP=V7_WEAK_MIN_KEEP, V7_WEAK_TOPK=V7_WEAK_TOPK, V7_WEAK_SHARPEN=V7_WEAK_SHARPEN,
V7_REL_WEIGHT_FLOOR=V7_REL_WEIGHT_FLOOR, V7_AMB_WEIGHT_FLOOR=V7_AMB_WEIGHT_FLOOR,
V7_AMB_WEIGHT_SCALE=V7_AMB_WEIGHT_SCALE, V7_WEAK_WEIGHT_FLOOR=V7_WEAK_WEIGHT_FLOOR,
V7_WEAK_WEIGHT_SCALE=V7_WEAK_WEIGHT_SCALE, V7_STAGE1_EPOCHS=V7_STAGE1_EPOCHS,
V7_PROMOTE_BLEND=V7_PROMOTE_BLEND, V7_PROMOTE_REL_FRACTION=V7_PROMOTE_REL_FRACTION,
V7_PROMOTE_AMB_FRACTION=V7_PROMOTE_AMB_FRACTION, V7_PROMOTE_MIN_REL=V7_PROMOTE_MIN_REL,
V7_PROMOTE_MIN_AMB=V7_PROMOTE_MIN_AMB, V7_PROMOTE_CONF=V7_PROMOTE_CONF, V7_PROMOTE_MARGIN=V7_PROMOTE_MARGIN,
V7_AMB_CONF=V7_AMB_CONF, V7_AMB_MARGIN=V7_AMB_MARGIN,
V7_LAMBDA_WEAK_SUP=V7_LAMBDA_WEAK_SUP, V7_LAMBDA_AMB_SUP=V7_LAMBDA_AMB_SUP,
V7_LAMBDA_ALIGN=V7_LAMBDA_ALIGN, V7_LAMBDA_ENT_MIN=V7_LAMBDA_ENT_MIN, V7_LAMBDA_ENT_MAX=V7_LAMBDA_ENT_MAX,
V7_LAMBDA_CONS=V7_LAMBDA_CONS, V7_LAMBDA_PROTO=V7_LAMBDA_PROTO,
V7_LAMBDA_UNKNOWN_REPEL=V7_LAMBDA_UNKNOWN_REPEL, V7_UNKNOWN_WARMUP_EPOCHS=V7_UNKNOWN_WARMUP_EPOCHS,
V7_UNKNOWN_RAMP_EPOCHS=V7_UNKNOWN_RAMP_EPOCHS,
    V14_LQ_MIN_TOTAL=V14_LQ_MIN_TOTAL, V14_LQ_MIN_PER_CLASS=V14_LQ_MIN_PER_CLASS,
    V14_LQ_REQUIRE_TWO_CLASSES=V14_LQ_REQUIRE_TWO_CLASSES,
    V14_LQ_HIGH_CONF=V14_LQ_HIGH_CONF, V14_LQ_LOW_CONF=V14_LQ_LOW_CONF, V14_LQ_FREEZE_REL=V14_LQ_FREEZE_REL,
    V14_LQ_SEED_BLEND_HIGH=V14_LQ_SEED_BLEND_HIGH, V14_LQ_SEED_BLEND_MID=V14_LQ_SEED_BLEND_MID, V14_LQ_SEED_BLEND_LOW=V14_LQ_SEED_BLEND_LOW,
    V14_LQ_MAX_WEIGHT=V14_LQ_MAX_WEIGHT, V14_LQ_AUG_VIEWS=V14_LQ_AUG_VIEWS, V14_LQ_AUG_NOISE_SCALE=V14_LQ_AUG_NOISE_SCALE,
    V14_SAFE_STD_FALLBACK=V14_SAFE_STD_FALLBACK,
    METHOD_ORDER=METHOD_ORDER,
    STAGEA_CACHE_ENABLED=STAGEA_CACHE_ENABLED,
    STAGEA_CACHE_FORCE_REBUILD=STAGEA_CACHE_FORCE_REBUILD,
    STAGEA_CACHE_FILENAME=STAGEA_CACHE_FILENAME,
    STAGEA_CACHE_VERSION=STAGEA_CACHE_VERSION)
with _safe_open(os.path.join(RUN_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)
print("RUN_DIR:", RUN_DIR)


# ===== v32 native-style final rejector sweep =====
DEFAULT_FINAL_REJECTOR_HEAD = "energy"
GENERIC_FINAL_REJECTOR_HEADS = [
    "energy",
    "energy_proto",
    "energy_state",
    "energy_class",
    "module2",
    "msp",
    "entropy",
    "margin",
    "proto",
]
BASELINE_NATIVE_REJECTOR_FAMILY = {
    "PCPD_recon": "pcpd",
    "OVANet_recon": "ovanet",
    "COMET_recon": "comet",
    "JRFFP_SC_recon": "jrffp_sc",
}
BASELINE_EXTRA_REJECTOR_HEADS = ["native"]
COMET_NATIVE_ENTROPY_THRESHOLD = 0.50
FINAL_REJECTOR_HEADS = list(GENERIC_FINAL_REJECTOR_HEADS)  # kept for backward-compatible config dumps
FINAL_REJECTOR_FRR_TARGET = FRR_TARGET
FINAL_REJECTOR_SWEEP_BASE_METHODS = "all"   # or provide an explicit list of base method names

# v33 rejector upgrades
ENERGY_CASCADE_BAND_STD = 0.20
ENERGY_CASCADE_BAND_MINABS = 0.02
ENERGY_STATE_SHIFT_STD = 0.20
ENERGY_STATE_SHIFT_MINABS = 0.02
ENERGY_STATE_UNKNOWN_SCALE = 0.50


def get_default_rejector_head(base_name):
    base_name = str(base_name)
    return str(DEFAULT_FINAL_REJECTOR_HEAD)


def get_supported_eval_heads_for_base_method(base_name):
    base_name = str(base_name)
    default_head = get_default_rejector_head(base_name)
    heads = [default_head] + [h for h in GENERIC_FINAL_REJECTOR_HEADS if h != default_head]
    if base_name in BASELINE_NATIVE_REJECTOR_FAMILY and "native" not in heads:
        heads.append("native")
    return heads


def make_eval_method_name(base_name, rejector_head):
    base_name = str(base_name)
    rejector_head = str(rejector_head).lower()
    if rejector_head == get_default_rejector_head(base_name):
        return base_name
    return f"{base_name}|RJ_{rejector_head}"


def parse_eval_method_name(eval_name):
    eval_name = str(eval_name)
    tag = "|RJ_"
    if tag not in eval_name:
        return eval_name, get_default_rejector_head(eval_name)
    base_name, rejector_head = eval_name.rsplit(tag, 1)
    return base_name, rejector_head.lower()


def get_eval_heads_for_base_method(base_name):
    base_name = str(base_name)
    default_head = get_default_rejector_head(base_name)
    supported = get_supported_eval_heads_for_base_method(base_name)
    if FINAL_REJECTOR_SWEEP_BASE_METHODS == "all":
        return list(supported)
    allow = set([str(v) for v in FINAL_REJECTOR_SWEEP_BASE_METHODS])
    if base_name in allow:
        return list(supported)
    return [default_head]


def resolve_eval_method_order(base_method_order):
    out = []
    for _base in base_method_order:
        for _head in get_eval_heads_for_base_method(_base):
            _name = make_eval_method_name(_base, _head)
            if _name not in out:
                out.append(_name)
    return out


EVAL_METHOD_ORDER = resolve_eval_method_order(METHOD_ORDER)
EVAL_METHOD_BASES = [parse_eval_method_name(v)[0] for v in EVAL_METHOD_ORDER]

cfg["GENERIC_FINAL_REJECTOR_HEADS"] = list(GENERIC_FINAL_REJECTOR_HEADS)
cfg["BASELINE_NATIVE_REJECTOR_FAMILY"] = dict(BASELINE_NATIVE_REJECTOR_FAMILY)
cfg["BASELINE_EXTRA_REJECTOR_HEADS"] = list(BASELINE_EXTRA_REJECTOR_HEADS)
cfg["COMET_NATIVE_ENTROPY_THRESHOLD"] = float(COMET_NATIVE_ENTROPY_THRESHOLD)
cfg["FINAL_REJECTOR_HEADS"] = list(FINAL_REJECTOR_HEADS)
cfg["FINAL_REJECTOR_FRR_TARGET"] = float(FINAL_REJECTOR_FRR_TARGET)
cfg["FINAL_REJECTOR_SWEEP_BASE_METHODS"] = FINAL_REJECTOR_SWEEP_BASE_METHODS
cfg["DEFAULT_FINAL_REJECTOR_HEAD"] = str(DEFAULT_FINAL_REJECTOR_HEAD)
cfg["ENERGY_CASCADE_BAND_STD"] = float(ENERGY_CASCADE_BAND_STD)
cfg["ENERGY_CASCADE_BAND_MINABS"] = float(ENERGY_CASCADE_BAND_MINABS)
cfg["ENERGY_STATE_SHIFT_STD"] = float(ENERGY_STATE_SHIFT_STD)
cfg["ENERGY_STATE_SHIFT_MINABS"] = float(ENERGY_STATE_SHIFT_MINABS)
cfg["ENERGY_STATE_UNKNOWN_SCALE"] = float(ENERGY_STATE_UNKNOWN_SCALE)
cfg["EVAL_METHOD_ORDER"] = list(EVAL_METHOD_ORDER)
RUN_METADATA["generic_final_rejector_heads"] = list(GENERIC_FINAL_REJECTOR_HEADS)
RUN_METADATA["baseline_native_rejector_family"] = dict(BASELINE_NATIVE_REJECTOR_FAMILY)
RUN_METADATA["baseline_extra_rejector_heads"] = list(BASELINE_EXTRA_REJECTOR_HEADS)
RUN_METADATA["comet_native_entropy_threshold"] = float(COMET_NATIVE_ENTROPY_THRESHOLD)
RUN_METADATA["final_rejector_heads"] = list(FINAL_REJECTOR_HEADS)
RUN_METADATA["final_rejector_frr_target"] = float(FINAL_REJECTOR_FRR_TARGET)
RUN_METADATA["final_rejector_sweep_base_methods"] = FINAL_REJECTOR_SWEEP_BASE_METHODS
RUN_METADATA["default_final_rejector_head"] = str(DEFAULT_FINAL_REJECTOR_HEAD)
RUN_METADATA["energy_cascade_band_std"] = float(ENERGY_CASCADE_BAND_STD)
RUN_METADATA["energy_cascade_band_minabs"] = float(ENERGY_CASCADE_BAND_MINABS)
RUN_METADATA["energy_state_shift_std"] = float(ENERGY_STATE_SHIFT_STD)
RUN_METADATA["energy_state_shift_minabs"] = float(ENERGY_STATE_SHIFT_MINABS)
RUN_METADATA["energy_state_unknown_scale"] = float(ENERGY_STATE_UNKNOWN_SCALE)
RUN_METADATA["eval_method_order"] = list(EVAL_METHOD_ORDER)
with _safe_open(os.path.join(RUN_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)
print("GENERIC_FINAL_REJECTOR_HEADS:", GENERIC_FINAL_REJECTOR_HEADS)
print("BASELINE_NATIVE_REJECTOR_FAMILY:", BASELINE_NATIVE_REJECTOR_FAMILY)
print("#base methods / #eval methods:", len(METHOD_ORDER), len(EVAL_METHOD_ORDER))


Device: cuda
RUN_DIR: ../owdc_results/results_SODA_RFF/run_SODA_RFF_fixed_cleanwin__pcA__20260426_183400
GENERIC_FINAL_REJECTOR_HEADS: ['energy', 'energy_proto', 'energy_state', 'energy_class', 'module2', 'msp', 'entropy', 'margin', 'proto']
BASELINE_NATIVE_REJECTOR_FAMILY: {'PCPD_recon': 'pcpd', 'OVANet_recon': 'ovanet', 'COMET_recon': 'comet', 'JRFFP_SC_recon': 'jrffp_sc'}
#base methods / #eval methods: 13 121


In [2]:

# ===== Fixed clean-win SODA-RFF patch =====
CODE_FILE_NAME = "wisig_SODA_RFF_Final_direct_corebuffer_thresholdfirst_thresholdfirst.ipynb"
RUN_VERSION_TAG = "wisig_SODA_RFF_Final_direct_corebuffer_thresholdfirst"
RUN_NOTES = (
    "SODA-RFF final strict comparison derived notebook; adds SODA_direct_corebuffer "
    "as a direct soft-route core-buffer diagnostic variant while preserving SODA-RFF."
)
RUN_TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")
RUN_BASENAME = f"{RUN_VERSION_TAG}__{RUN_HOST_TAG}__{RUN_TIMESTAMP}"
RUN_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "owdc_results", "wisig_soda_rff_final_direct_corebuffer_thresholdfirst", f"run_{RUN_BASENAME}"))
_safe_makedirs(RUN_DIR)

# ===== Selected search-snapshot restoration =====
# Default path expected from the search notebook export. If missing, the notebook
# falls back to the manual fixed protocol configs below.
USE_SELECTED_VARIANT_JSON = True
SELECTED_VARIANT_JSON_PATH = "./resolved_selected_variants.json"

SODA_SELECTED_VARIANT_SNAPSHOTS = None
SODA_SELECTED_VARIANT_SNAPSHOTS_PATH = None
SODA_LAST_APPLIED_CONFIG_SOURCE = "manual_fixed_config"
SODA_LAST_APPLIED_PROFILE_TAG = None
SODA_LAST_APPLIED_REJECTOR_HEAD = None
SODA_LAST_APPLIED_SNAPSHOT_PATH = None
SODA_LAST_APPLIED_PROTOCOL_TAG = None
SODA_LAST_APPLIED_RUNTIME = {}


# ===== Execution entry switch =====
# Set to "smoke" to run the minimal smoke test only.
# Set to "full" to run the full four-protocol suite.
EXECUTION_MODE = "full"
SMOKE_PROTOCOL_COMBO = ("9-3", "4-2")
SMOKE_MAX_SPLITS = 1
SMOKE_MAX_FOLDS = 1


SODA_METHOD_NAME = "SODA-RFF"
DIRECT_COREBUFFER_METHOD_NAME = "SODA_direct_corebuffer"
DIRECT_COREBUFFER_ENABLE = True
RUN_DIRECT_COREBUFFER_SMOKE = bool(globals().get("RUN_DIRECT_COREBUFFER_SMOKE", False))
DIRECT_COREBUFFER_SELECTION_MODE = "threshold_first"
DIRECT_COREBUFFER_CONFLICT_POLICY = "priority_unknown_shift_stable"
DIRECT_COREBUFFER_P_SOURCE = "base_soda"

# Legacy top-k knobs are retained only for explicit DIRECT_COREBUFFER_SELECTION_MODE="topk_legacy".
DIRECT_COREBUFFER_STABLE_KEEP_FRAC = 0.25
DIRECT_COREBUFFER_SHIFT_KEEP_FRAC  = 0.25
DIRECT_COREBUFFER_UNK_KEEP_FRAC    = 0.25

DIRECT_COREBUFFER_STABLE_SCORE_MIN = 0.05
DIRECT_COREBUFFER_SHIFT_SCORE_MIN = 0.05
DIRECT_COREBUFFER_UNK_SCORE_MIN = 0.05
DIRECT_COREBUFFER_STABLE_MARGIN_MIN = 0.05
DIRECT_COREBUFFER_SHIFT_MARGIN_MIN  = 0.05
DIRECT_COREBUFFER_UNK_MARGIN_MIN    = 0.05
DIRECT_COREBUFFER_STABLE_CONF_MIN = 0.60
DIRECT_COREBUFFER_SHIFT_CONF_MIN = 0.55
DIRECT_COREBUFFER_UNK_CONF_MAX = 0.50
DIRECT_COREBUFFER_STABLE_ENTROPY_MAX = 0.60
DIRECT_COREBUFFER_SHIFT_ENTROPY_MAX = 0.70
DIRECT_COREBUFFER_UNK_ENTROPY_MIN = 0.50
DIRECT_COREBUFFER_SHIFT_KNOWN_MIN = 0.50
DIRECT_COREBUFFER_UNK_KNOWN_MAX = 0.50
DIRECT_COREBUFFER_USE_AGREEMENT_MIN = False
DIRECT_COREBUFFER_STABLE_AGREE_MIN = 0.50
DIRECT_COREBUFFER_SHIFT_AGREE_MIN = 0.50
DIRECT_COREBUFFER_USE_MAX_CAP = False
DIRECT_COREBUFFER_STABLE_MAX_FRAC = 0.40
DIRECT_COREBUFFER_SHIFT_MAX_FRAC = 0.40
DIRECT_COREBUFFER_UNK_MAX_FRAC = 0.40
DIRECT_COREBUFFER_SMOKE_RELAX_THRESHOLDS = True
DIRECT_COREBUFFER_CLASS_BALANCED_KNOWN = True
DIRECT_COREBUFFER_BUFFER_MODE = "skip"
DIRECT_COREBUFFER_SUP_WEIGHT_MODE = "score"
DIRECT_COREBUFFER_MIN_CORE_SIZE = 8
if RUN_DIRECT_COREBUFFER_SMOKE:
    EXECUTION_MODE = "smoke"
    SMOKE_MAX_SPLITS = 1
    SMOKE_MAX_FOLDS = 1
    EPOCHS = 1
    ADAPT_EPOCHS = 1
    PATIENCE = 1
MAX_FOLDS_PER_PROTOCOL = None

# Protocol-specific fixed clean-win settings distilled from the v36.6 search results.
# Each entry fixes:
#   (1) the protocol-tuned v19-style SODA base parameters,
#   (2) the v36 unknown-proxy controls, and
#   (3) the final rejector head actually used by the clean-win selection.
SODA_PROTOCOL_FIXED_CONFIGS = {
    "RX9-3_TX4-2": dict(
        profile_tag="u9342_fixedwin",
        gamma_notunk=0.98,
        lambda_unkE=0.021,
        lambda_stab=0.000,
        v19_lambda_align=0.037,
        v19_lambda_ent_min=0.006,
        v19_lambda_ent_max=0.013,
        v19_lambda_proto=0.145,
        v19_unknown_energy_margin=0.044,
        v36_proxy_cap=0.06,
        v36_proxy_q=0.94,
        v36_proxy_pu_min=0.45,
        v36_proxy_conf_max=0.75,
        v36_proxy_conf_smooth=0.08,
        v36_proxy_entropy_mix=0.50,
        v36_lambda_ent_max_scale=1.10,
        v36_proxy_min_keep=24,
        default_rejector_head="energy",
    ),
    "RX9-3_TX2-4": dict(
        profile_tag="u93_24_pv004",
        # anchor = p9324_bal_relax_b
        # selected local preset = reject_tight_b
        gamma_notunk=0.8232,
        lambda_unkE=0.01888,
        lambda_stab=0.000,
        v19_lambda_align=0.030,
        v19_lambda_ent_min=0.004,
        v19_lambda_ent_max=0.009,
        v19_lambda_proto=0.1144,
        v19_unknown_energy_margin=0.0368,
        v36_proxy_cap=0.0,
        v36_proxy_q=0.95,
        v36_proxy_pu_min=0.44,
        v36_proxy_conf_max=0.80,
        v36_proxy_conf_smooth=0.08,
        v36_proxy_entropy_mix=0.50,
        v36_lambda_ent_max_scale=1.0,
        v36_proxy_min_keep=0,
        default_rejector_head="energy_class",
    ),
    "RX6-6_TX3-3": dict(
        profile_tag="u6633_fixedwin",
        gamma_notunk=1.0,
        lambda_unkE=0.038,
        lambda_stab=0.010,
        v19_lambda_align=0.065,
        v19_lambda_ent_min=0.015,
        v19_lambda_ent_max=0.035,
        v19_lambda_proto=0.21,
        v19_unknown_energy_margin=0.08,
        v36_proxy_cap=0.10,
        v36_proxy_q=0.90,
        v36_proxy_pu_min=0.38,
        v36_proxy_conf_max=0.82,
        v36_proxy_conf_smooth=0.08,
        v36_proxy_entropy_mix=0.50,
        v36_lambda_ent_max_scale=1.25,
        v36_proxy_min_keep=24,
        default_rejector_head="energy",
    ),
    "RX3-9_TX2-4": dict(
        profile_tag="u39_24_pv008",
        # anchor = p3924_bc_relax
        # selected local preset = align_dn_ent_up
        gamma_notunk=0.84,
        lambda_unkE=0.015,
        lambda_stab=0.000,
        v19_lambda_align=0.02668,
        v19_lambda_ent_min=0.004,
        v19_lambda_ent_max=0.0099,
        v19_lambda_proto=0.144,
        v19_unknown_energy_margin=0.032,
        v36_proxy_cap=0.0,
        v36_proxy_q=0.985,
        v36_proxy_pu_min=0.56,
        v36_proxy_conf_max=0.78,
        v36_proxy_conf_smooth=0.08,
        v36_proxy_entropy_mix=0.45,
        v36_lambda_ent_max_scale=1.0,
        v36_proxy_min_keep=0,
        default_rejector_head="proto",
    ),
}

SODA_DEFAULT_PROTOCOL = "RX9-3_TX4-2"
SODA_ACTIVE_PROTOCOL_TAG = SODA_DEFAULT_PROTOCOL

METHOD_ORDER = [
    SODA_METHOD_NAME,
    "SourceOnly",
    "PCPD_recon",
    "OVANet_recon",
    "COMET_recon",
    "JRFFP_SC_recon",
]
EVAL_METHOD_ORDER = list(METHOD_ORDER)
EVAL_METHOD_BASES = list(METHOD_ORDER)

def load_selected_variant_snapshots(path=None, force_reload=False):
    global SODA_SELECTED_VARIANT_SNAPSHOTS, SODA_SELECTED_VARIANT_SNAPSHOTS_PATH
    path = str(path or SELECTED_VARIANT_JSON_PATH)
    if (not force_reload) and (SODA_SELECTED_VARIANT_SNAPSHOTS is not None) and (SODA_SELECTED_VARIANT_SNAPSHOTS_PATH == path):
        return SODA_SELECTED_VARIANT_SNAPSHOTS

    snapshots = {}
    if not bool(USE_SELECTED_VARIANT_JSON):
        SODA_SELECTED_VARIANT_SNAPSHOTS = snapshots
        SODA_SELECTED_VARIANT_SNAPSHOTS_PATH = path
        return snapshots
    if not path or (not os.path.exists(path)):
        SODA_SELECTED_VARIANT_SNAPSHOTS = snapshots
        SODA_SELECTED_VARIANT_SNAPSHOTS_PATH = path
        return snapshots

    with _safe_open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    def _add_snapshot(protocol_tag, entry):
        if protocol_tag is None or not isinstance(entry, dict):
            return
        protocol_tag = str(protocol_tag)
        item = deepcopy(entry)
        item["protocol_tag"] = protocol_tag
        snapshots[protocol_tag] = item

    def _consume_list(seq):
        if not isinstance(seq, list):
            return False
        used = False
        for item in seq:
            if isinstance(item, dict):
                ptag = item.get("protocol_tag") or item.get("protocol") or item.get("tag")
                if ptag is not None:
                    _add_snapshot(ptag, item)
                    used = True
        return used

    if isinstance(raw, dict):
        if any(str(k).startswith("RX") and isinstance(v, dict) for k, v in raw.items()):
            for _ptag, _entry in raw.items():
                _add_snapshot(_ptag, _entry)
        else:
            for _key in ["selected_by_protocol", "protocols", "snapshots", "resolved_selected_variants", "by_protocol"]:
                _obj = raw.get(_key)
                if isinstance(_obj, dict):
                    for _ptag, _entry in _obj.items():
                        _add_snapshot(_ptag, _entry)
                elif isinstance(_obj, list):
                    _consume_list(_obj)
            for _key in ["entries", "items", "rows", "variants"]:
                _consume_list(raw.get(_key))
    elif isinstance(raw, list):
        _consume_list(raw)

    SODA_SELECTED_VARIANT_SNAPSHOTS = snapshots
    SODA_SELECTED_VARIANT_SNAPSHOTS_PATH = path
    return snapshots

def get_selected_snapshot_for_protocol(protocol_tag, snapshots=None):
    protocol_tag = str(protocol_tag)
    if snapshots is None:
        snapshots = load_selected_variant_snapshots()
    if not isinstance(snapshots, dict):
        return None
    return snapshots.get(protocol_tag)

def _snapshot_required_field_report(snapshot):
    merged = {}
    if isinstance(snapshot.get("resolved_variant_dict"), dict):
        merged.update(snapshot.get("resolved_variant_dict", {}))
    if isinstance(snapshot.get("runtime_snapshot"), dict):
        merged.update(snapshot.get("runtime_snapshot", {}))
    has_profile = (
        (snapshot.get("selected_profile_tag") is not None) or
        (snapshot.get("profile_tag") is not None) or
        (merged.get("profile_tag") is not None)
    )
    has_rejector = (
        (snapshot.get("selected_rejector_head") is not None) or
        (snapshot.get("default_rejector_head") is not None) or
        (merged.get("default_rejector_head") is not None)
    )
    has_runtime = isinstance(snapshot.get("runtime_snapshot"), dict) or isinstance(snapshot.get("resolved_variant_dict"), dict)
    return {
        "has_profile": bool(has_profile),
        "has_rejector": bool(has_rejector),
        "has_runtime": bool(has_runtime),
    }

def build_runtime_from_selected_snapshot(snapshot):
    snapshot = dict(snapshot or {})
    merged = {}
    if isinstance(snapshot.get("resolved_variant_dict"), dict):
        merged.update(deepcopy(snapshot.get("resolved_variant_dict")))
    if isinstance(snapshot.get("runtime_snapshot"), dict):
        merged.update(deepcopy(snapshot.get("runtime_snapshot")))

    def _first_nonnull(*keys):
        for _k in keys:
            if _k in merged and merged[_k] is not None:
                return merged[_k]
            if _k in snapshot and snapshot[_k] is not None:
                return snapshot[_k]
        return None

    def _first_dict(*keys):
        for _k in keys:
            _v = _first_nonnull(_k)
            if isinstance(_v, dict):
                return deepcopy(_v)
        return None

    def _coerce_float_dict(obj):
        if not isinstance(obj, dict):
            return None
        out = {}
        for _k, _v in obj.items():
            try:
                out[str(_k)] = float(_v)
            except Exception:
                continue
        return out if len(out) > 0 else None

    alias_map = {
        "profile_tag": ["profile_tag", "selected_profile_tag"],
        "default_rejector_head": ["default_rejector_head", "selected_rejector_head"],
        "selected_rejector_family": ["selected_rejector_family", "rejector_family"],
        "selected_score_name": ["selected_score_name", "score_name"],
        "base_tag": ["base_tag"],
        "gamma_notunk": ["gamma_notunk", "v22_gamma_notunk", "V22_GAMMA_NOTUNK"],
        "lambda_unkE": ["lambda_unkE", "lambda_unke", "v22_lambda_unkE", "v22_lambda_unke", "V22_LAMBDA_UNKE"],
        "lambda_stab": ["lambda_stab", "v22_lambda_stab", "V22_LAMBDA_STAB"],
        "lambda_sup": ["lambda_sup", "v22_lambda_sup", "V22_LAMBDA_SUP"],
        "lambda_proto": ["lambda_proto", "v22_lambda_proto", "V22_LAMBDA_PROTO"],
        "lambda_cons": ["lambda_cons"],
        "v19_lambda_align": ["v19_lambda_align", "lambda_align"],
        "v19_lambda_ent_min": ["v19_lambda_ent_min", "lambda_ent_min"],
        "v19_lambda_ent_max": ["v19_lambda_ent_max", "lambda_ent_max"],
        "v19_lambda_proto": ["v19_lambda_proto", "lambda_proto"],
        "v19_lambda_cons": ["v19_lambda_cons", "lambda_cons"],
        "v19_unknown_energy_margin": ["v19_unknown_energy_margin", "unknown_energy_margin", "V22_UNKNOWN_ENERGY_MARGIN"],
        "v36_proxy_cap": ["v36_proxy_cap"],
        "v36_proxy_q": ["v36_proxy_q"],
        "v36_proxy_pu_min": ["v36_proxy_pu_min"],
        "v36_proxy_conf_max": ["v36_proxy_conf_max"],
        "v36_proxy_conf_smooth": ["v36_proxy_conf_smooth"],
        "v36_proxy_entropy_mix": ["v36_proxy_entropy_mix"],
        "v36_lambda_ent_max_scale": ["v36_lambda_ent_max_scale"],
        "v36_proxy_min_keep": ["v36_proxy_min_keep"],
    }
    for _target, _keys in alias_map.items():
        _val = _first_nonnull(*_keys)
        if _val is not None:
            merged[_target] = _val

    if "selected_profile_tag" not in merged and snapshot.get("selected_profile_tag") is not None:
        merged["selected_profile_tag"] = snapshot.get("selected_profile_tag")
    if "selected_rejector_head" not in merged and snapshot.get("selected_rejector_head") is not None:
        merged["selected_rejector_head"] = snapshot.get("selected_rejector_head")
    if "selected_rejector_family" not in merged and snapshot.get("selected_rejector_family") is not None:
        merged["selected_rejector_family"] = snapshot.get("selected_rejector_family")
    if "selected_score_name" not in merged and snapshot.get("selected_score_name") is not None:
        merged["selected_score_name"] = snapshot.get("selected_score_name")
    if "selected_metrics" not in merged and snapshot.get("selected_metrics") is not None:
        merged["selected_metrics"] = deepcopy(snapshot.get("selected_metrics"))

    # Recover the final effective atomic loss weights exported by the search notebook.
    # These may appear either directly in runtime_snapshot/effective_atomic_weights or
    # in resolved_variant_dict/soda_effective_atomic_weights. If only default atomics and
    # group weights are present, reconstruct the effective atomics here.
    default_atomic = _coerce_float_dict(
        _first_dict("soda_default_atomic_weights", "default_atomic_weights")
    )
    effective_atomic = _coerce_float_dict(
        _first_dict("soda_effective_atomic_weights", "effective_atomic_weights")
    )

    dynamic_groups = _coerce_float_dict(_first_dict("dynamic_group_weights")) or {}
    for _gname, _field in [
        ("preserve", "soda_group_weight_preserve"),
        ("recover", "soda_group_weight_recover"),
        ("repel", "soda_group_weight_repel"),
    ]:
        _val = _first_nonnull(_field)
        if _val is not None:
            try:
                dynamic_groups[_gname] = float(_val)
            except Exception:
                pass
    if len(dynamic_groups) == 0:
        dynamic_groups = None

    if (effective_atomic is None) and (default_atomic is not None) and (dynamic_groups is not None):
        preserve = float(dynamic_groups.get("preserve", 1.0))
        recover = float(dynamic_groups.get("recover", 1.0))
        repel = float(dynamic_groups.get("repel", 1.0))
        effective_atomic = {
            "lambda_sup": float(default_atomic.get("lambda_sup", 0.0)) * preserve,
            "lambda_align": float(default_atomic.get("lambda_align", 0.0)) * recover,
            "lambda_ent_min": float(default_atomic.get("lambda_ent_min", 0.0)) * recover,
            "lambda_ent_max": float(default_atomic.get("lambda_ent_max", 0.0)) * repel,
            "lambda_cons": float(default_atomic.get("lambda_cons", 0.0)) * preserve,
            "lambda_proto": float(default_atomic.get("lambda_proto", 0.0)) * recover,
        }

    if default_atomic is not None:
        merged["soda_default_atomic_weights"] = deepcopy(default_atomic)
    if dynamic_groups is not None:
        merged["dynamic_group_weights"] = deepcopy(dynamic_groups)
        merged["soda_group_weight_preserve"] = float(dynamic_groups.get("preserve", 1.0))
        merged["soda_group_weight_recover"] = float(dynamic_groups.get("recover", 1.0))
        merged["soda_group_weight_repel"] = float(dynamic_groups.get("repel", 1.0))
    if effective_atomic is not None:
        merged["soda_effective_atomic_weights"] = deepcopy(effective_atomic)
        merged["effective_atomic_weights"] = deepcopy(effective_atomic)
        # Only the atomics actually scaled by the dynamic group scheduler should be
        # overridden here. lambda_unkE / lambda_stab / gamma_notunk are *not* group-scaled.
        if "lambda_sup" in effective_atomic:
            merged["lambda_sup"] = float(effective_atomic["lambda_sup"])
        if "lambda_cons" in effective_atomic:
            merged["lambda_cons"] = float(effective_atomic["lambda_cons"])
            merged["v19_lambda_cons"] = float(effective_atomic["lambda_cons"])
        if "lambda_align" in effective_atomic:
            merged["v19_lambda_align"] = float(effective_atomic["lambda_align"])
        if "lambda_ent_min" in effective_atomic:
            merged["v19_lambda_ent_min"] = float(effective_atomic["lambda_ent_min"])
        if "lambda_ent_max" in effective_atomic:
            merged["v19_lambda_ent_max"] = float(effective_atomic["lambda_ent_max"])
            merged["v36_lambda_ent_max_scale"] = 1.0
        if "lambda_proto" in effective_atomic:
            merged["lambda_proto"] = float(effective_atomic["lambda_proto"])
            merged["v19_lambda_proto"] = float(effective_atomic["lambda_proto"])

    return merged


def apply_selected_snapshot_runtime(snapshot, protocol_tag=None, manual_base_cfg=None):
    cfg_local = dict(manual_base_cfg or {})
    cfg_local.update(build_runtime_from_selected_snapshot(snapshot))
    if protocol_tag is not None:
        cfg_local["protocol_tag"] = str(protocol_tag)
    return cfg_local

def _write_applied_runtime_config(protocol_tag, cfg_local):
    out_dir = os.path.join(RUN_DIR, "applied_runtime_configs")
    _safe_makedirs(out_dir)
    out_path = os.path.join(out_dir, f"applied_runtime_config_{str(protocol_tag)}.json")
    payload = deepcopy(dict(cfg_local or {}))
    if os.path.exists(out_path):
        try:
            with _safe_open(out_path, "r", encoding="utf-8") as f:
                old_payload = json.load(f)
            if old_payload == payload:
                return out_path
        except Exception:
            pass
    with _safe_open(out_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
    return out_path

def get_soda_protocol_fixed_config(protocol_tag):
    protocol_tag = str(protocol_tag)
    manual_cfg = dict(SODA_PROTOCOL_FIXED_CONFIGS.get(protocol_tag, SODA_PROTOCOL_FIXED_CONFIGS[SODA_DEFAULT_PROTOCOL]))
    manual_cfg["config_source"] = "manual_fixed_config"
    manual_cfg["selected_profile_tag"] = manual_cfg.get("profile_tag")
    manual_cfg["selected_rejector_head"] = manual_cfg.get("default_rejector_head")
    manual_cfg["selected_snapshot_path"] = str(SELECTED_VARIANT_JSON_PATH)
    manual_cfg["protocol_tag"] = protocol_tag

    if not bool(USE_SELECTED_VARIANT_JSON):
        return manual_cfg
    snapshots = load_selected_variant_snapshots(SELECTED_VARIANT_JSON_PATH)
    snapshot = get_selected_snapshot_for_protocol(protocol_tag, snapshots)
    if not isinstance(snapshot, dict):
        return manual_cfg

    required = _snapshot_required_field_report(snapshot)
    if not (required["has_profile"] and required["has_rejector"] and required["has_runtime"]):
        return manual_cfg

    cfg_local = apply_selected_snapshot_runtime(snapshot, protocol_tag=protocol_tag, manual_base_cfg=manual_cfg)
    cfg_local["config_source"] = "selected_snapshot_json"
    cfg_local["selected_profile_tag"] = cfg_local.get("selected_profile_tag", cfg_local.get("profile_tag"))
    cfg_local["selected_rejector_head"] = cfg_local.get("selected_rejector_head", cfg_local.get("default_rejector_head"))
    cfg_local["selected_snapshot_path"] = str(SELECTED_VARIANT_JSON_PATH)
    return cfg_local

def apply_soda_protocol_runtime(protocol_tag):
    global SODA_ACTIVE_PROTOCOL_TAG
    global V22_GAMMA_NOTUNK, V22_LAMBDA_UNKE, V22_LAMBDA_STAB
    global V22_LAMBDA_SUP, V22_LAMBDA_PROTO
    global UNKNOWN_ENERGY_MARGIN, V22_UNKNOWN_ENERGY_MARGIN
    global SODA_LAST_APPLIED_CONFIG_SOURCE, SODA_LAST_APPLIED_PROFILE_TAG
    global SODA_LAST_APPLIED_REJECTOR_HEAD, SODA_LAST_APPLIED_SNAPSHOT_PATH
    global SODA_LAST_APPLIED_PROTOCOL_TAG, SODA_LAST_APPLIED_RUNTIME

    protocol_tag = str(protocol_tag)
    cfg_local = get_soda_protocol_fixed_config(protocol_tag)
    SODA_ACTIVE_PROTOCOL_TAG = protocol_tag

    V22_GAMMA_NOTUNK = float(cfg_local["gamma_notunk"])
    V22_LAMBDA_UNKE = float(cfg_local["lambda_unkE"])
    V22_LAMBDA_STAB = float(cfg_local["lambda_stab"])
    if "lambda_sup" in cfg_local and cfg_local["lambda_sup"] is not None:
        V22_LAMBDA_SUP = float(cfg_local["lambda_sup"])
    if "lambda_proto" in cfg_local and cfg_local["lambda_proto"] is not None:
        V22_LAMBDA_PROTO = float(cfg_local["lambda_proto"])
    UNKNOWN_ENERGY_MARGIN = float(cfg_local["v19_unknown_energy_margin"])
    V22_UNKNOWN_ENERGY_MARGIN = float(cfg_local["v19_unknown_energy_margin"])

    if "V19_M2W_CFG" in globals():
        for _mode in ["dgr", "tbv8"]:
            if _mode in V19_M2W_CFG:
                if "v19_lambda_align" in cfg_local:
                    V19_M2W_CFG[_mode]["lambda_align"] = float(cfg_local["v19_lambda_align"])
                if "v19_lambda_ent_min" in cfg_local:
                    V19_M2W_CFG[_mode]["lambda_ent_min"] = float(cfg_local["v19_lambda_ent_min"])
                if "v19_lambda_ent_max" in cfg_local:
                    V19_M2W_CFG[_mode]["lambda_ent_max"] = float(cfg_local["v19_lambda_ent_max"])
                if "v19_lambda_proto" in cfg_local:
                    V19_M2W_CFG[_mode]["lambda_proto"] = float(cfg_local["v19_lambda_proto"])
                if "v19_lambda_cons" in cfg_local:
                    V19_M2W_CFG[_mode]["lambda_cons"] = float(cfg_local["v19_lambda_cons"])
                elif "lambda_cons" in cfg_local:
                    V19_M2W_CFG[_mode]["lambda_cons"] = float(cfg_local["lambda_cons"])

    SODA_LAST_APPLIED_CONFIG_SOURCE = str(cfg_local.get("config_source", "manual_fixed_config"))
    SODA_LAST_APPLIED_PROFILE_TAG = cfg_local.get("selected_profile_tag", cfg_local.get("profile_tag"))
    SODA_LAST_APPLIED_REJECTOR_HEAD = cfg_local.get("selected_rejector_head", cfg_local.get("default_rejector_head"))
    SODA_LAST_APPLIED_SNAPSHOT_PATH = str(cfg_local.get("selected_snapshot_path", SELECTED_VARIANT_JSON_PATH))
    SODA_LAST_APPLIED_PROTOCOL_TAG = protocol_tag
    SODA_LAST_APPLIED_RUNTIME = deepcopy(cfg_local)
    _applied_path = _write_applied_runtime_config(protocol_tag, cfg_local)

    print(f"[SODA-CONFIG] protocol={protocol_tag} | source={SODA_LAST_APPLIED_CONFIG_SOURCE}")
    if SODA_LAST_APPLIED_CONFIG_SOURCE == "selected_snapshot_json":
        print(f"[SODA-CONFIG] loaded selected snapshot for {protocol_tag}")
    else:
        print(f"[SODA-CONFIG] fallback to manual fixed config for {protocol_tag}")
    print(
        f"[SODA-CONFIG] selected_profile_tag={SODA_LAST_APPLIED_PROFILE_TAG} | "
        f"selected_rejector_head={SODA_LAST_APPLIED_REJECTOR_HEAD} | "
        f"json_path={SODA_LAST_APPLIED_SNAPSHOT_PATH}"
    )
    _debug_keys = [
        "gamma_notunk", "lambda_unkE", "lambda_stab", "lambda_sup", "lambda_proto", "lambda_cons",
        "v19_lambda_align", "v19_lambda_ent_min", "v19_lambda_ent_max",
        "v19_lambda_proto", "v19_lambda_cons", "v19_unknown_energy_margin",
        "v36_proxy_cap", "v36_proxy_q", "v36_proxy_pu_min",
        "v36_proxy_conf_max", "v36_proxy_conf_smooth",
        "v36_proxy_entropy_mix", "v36_lambda_ent_max_scale", "v36_proxy_min_keep",
        "dynamic_group_weights", "soda_default_atomic_weights", "soda_effective_atomic_weights",
    ]
    _debug_loaded = {k: cfg_local.get(k) for k in _debug_keys if k in cfg_local}
    _debug_applied = {
        "V22_GAMMA_NOTUNK": V22_GAMMA_NOTUNK,
        "V22_LAMBDA_UNKE": V22_LAMBDA_UNKE,
        "V22_LAMBDA_STAB": V22_LAMBDA_STAB,
        "V22_LAMBDA_SUP": V22_LAMBDA_SUP,
        "V22_LAMBDA_PROTO": V22_LAMBDA_PROTO,
        "V22_UNKNOWN_ENERGY_MARGIN": V22_UNKNOWN_ENERGY_MARGIN,
    }
    print(f"[SODA-CONFIG] snapshot/runtime keys = {_debug_loaded}")
    print(f"[SODA-CONFIG] applied globals = {_debug_applied}")
    print(f"[SODA-CONFIG] applied_runtime_path = {_applied_path}")

    return cfg_local



def get_default_rejector_head(base_name):
    base_name = str(base_name)
    if base_name == SODA_METHOD_NAME:
        return str(get_soda_protocol_fixed_config(globals().get("SODA_ACTIVE_PROTOCOL_TAG", SODA_DEFAULT_PROTOCOL)).get("default_rejector_head", "energy"))
    return str(DEFAULT_FINAL_REJECTOR_HEAD)

def get_supported_eval_heads_for_base_method(base_name):
    return [get_default_rejector_head(base_name)]

def get_eval_heads_for_base_method(base_name):
    return [get_default_rejector_head(base_name)]

def make_eval_method_name(base_name, rejector_head):
    base_name = str(base_name)
    rejector_head = str(rejector_head).lower()
    if rejector_head == get_default_rejector_head(base_name):
        return base_name
    return f"{base_name}|RJ_{rejector_head}"

def parse_eval_method_name(eval_name):
    eval_name = str(eval_name)
    tag = "|RJ_"
    if tag not in eval_name:
        return eval_name, get_default_rejector_head(eval_name)
    base_name, rejector_head = eval_name.rsplit(tag, 1)
    return base_name, rejector_head.lower()

RUN_METADATA["code_file_name"] = CODE_FILE_NAME
RUN_METADATA["run_version_tag"] = RUN_VERSION_TAG
RUN_METADATA["run_notes"] = RUN_NOTES
RUN_METADATA["run_basename"] = RUN_BASENAME
RUN_METADATA["run_dir"] = RUN_DIR
RUN_METADATA["method_order"] = list(METHOD_ORDER)
RUN_METADATA["eval_method_order"] = list(EVAL_METHOD_ORDER)
RUN_METADATA["soda_protocol_fixed_configs"] = deepcopy(SODA_PROTOCOL_FIXED_CONFIGS)
RUN_METADATA["use_selected_variant_json"] = bool(USE_SELECTED_VARIANT_JSON)
RUN_METADATA["selected_variant_json_path"] = SELECTED_VARIANT_JSON_PATH

cfg["code_file_name"] = CODE_FILE_NAME
cfg["run_version_tag"] = RUN_VERSION_TAG
cfg["run_notes"] = RUN_NOTES
cfg["run_basename"] = RUN_BASENAME
cfg["run_dir"] = RUN_DIR
cfg["METHOD_ORDER"] = list(METHOD_ORDER)
cfg["EVAL_METHOD_ORDER"] = list(EVAL_METHOD_ORDER)
cfg["SODA_PROTOCOL_FIXED_CONFIGS"] = deepcopy(SODA_PROTOCOL_FIXED_CONFIGS)
cfg["USE_SELECTED_VARIANT_JSON"] = bool(USE_SELECTED_VARIANT_JSON)
cfg["SELECTED_VARIANT_JSON_PATH"] = SELECTED_VARIANT_JSON_PATH

with _safe_open(os.path.join(RUN_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)

print("Patched METHOD_ORDER =", METHOD_ORDER)
print("Patched RUN_DIR =", RUN_DIR)
for _ptag, _pcfg in SODA_PROTOCOL_FIXED_CONFIGS.items():
    print(_ptag, "->", _pcfg["profile_tag"], "| RJ =", _pcfg["default_rejector_head"])


# Compatibility aliases for legacy v36 builder paths that may still read these names.
# They mirror the fixed clean-win protocol configuration and prevent NameError in mixed code paths.
SODA_DEFAULT_OVERRIDES = {
    k: v for k, v in get_soda_protocol_fixed_config(SODA_DEFAULT_PROTOCOL).items()
    if k.startswith("v36_")
}
SODA_PROTOCOL_OVERRIDES = {
    str(_ptag): {k: v for k, v in _cfg.items() if k.startswith("v36_")}
    for _ptag, _cfg in SODA_PROTOCOL_FIXED_CONFIGS.items()
}


Patched METHOD_ORDER = ['SODA-RFF', 'SourceOnly', 'PCPD_recon', 'OVANet_recon', 'COMET_recon', 'JRFFP_SC_recon']
Patched RUN_DIR = d:\Program\MW-RFF\owdc_results\wisig_soda_rff_final_direct_corebuffer_thresholdfirst\run_wisig_SODA_RFF_Final_direct_corebuffer_thresholdfirst__pcA__20260426_183400
RX9-3_TX4-2 -> u9342_fixedwin | RJ = energy
RX9-3_TX2-4 -> u93_24_pv004 | RJ = energy_class
RX6-6_TX3-3 -> u6633_fixedwin | RJ = energy
RX3-9_TX2-4 -> u39_24_pv008 | RJ = proto


In [3]:

def harmonic_mean_np(a, b, eps=1e-12):
    a = float(a)
    b = float(b)
    if (a <= 0.0) or (b <= 0.0):
        return 0.0
    return float((2.0 * a * b) / max(a + b, eps))

def get_idx(lst, val):
    return lst.index(val)

def subsample(sigs, max_sig):
    if max_sig is None:
        return sigs
    return sigs[:min(len(sigs), max_sig)]

def get_signals(compact_dataset, tx, rx, date, equalized):
    tx_i = get_idx(compact_dataset["tx_list"], tx)
    rx_i = get_idx(compact_dataset["rx_list"], rx)
    date_i = get_idx(compact_dataset["capture_date_list"], date)
    eq_i = get_idx(compact_dataset["equalized_list"], equalized)
    sigs = compact_dataset["data"][tx_i][rx_i][date_i][eq_i]
    return np.array(sigs, dtype=np.float32)

def iq_to_complex(x):
    return (x[...,0] + 1j*x[...,1]).astype(np.complex64)

def domain_feat_256_complex(x256_c, d_dim=32):
    x = x256_c / (np.sqrt(np.mean(np.abs(x256_c)**2)) + 1e-12)
    X = np.fft.fft(x, n=256)
    mag = np.abs(X) + 1e-12
    logm = np.log(mag)
    D = np.fft.rfft(logm, n=512)
    return np.real(D[:d_dim]).astype(np.float32)

def compute_domain_feats_batch(X, d_dim=32):
    Xc = iq_to_complex(X)
    return np.stack([domain_feat_256_complex(Xc[i], d_dim=d_dim) for i in range(Xc.shape[0])], axis=0).astype(np.float32)

tx_list = compact_dataset["tx_list"]
rx_list = compact_dataset["rx_list"]
date_list = compact_dataset["capture_date_list"]


TX_USE = tx_list[:TX_TOTAL_USE]
RX_USE = rx_list[:RX_TOTAL_USE]
DATE_USE = date_list[:DAY_TOTAL_USE]

print("TX_USE:", TX_USE)
print("RX_USE:", RX_USE)
print("DATE_USE:", DATE_USE)

def build_unique_tx_splits(tx_use, known_tx_num, n_splits, seed=42):
    all_known = [tuple(c) for c in itertools.combinations(list(tx_use), known_tx_num)]
    if n_splits > len(all_known):
        raise ValueError(
            f"Requested TX_SPLIT_REPEATS={n_splits}, but only {len(all_known)} unique TX splits "
            f"exist for C({len(tx_use)},{known_tx_num}). Reduce TX_SPLIT_REPEATS or increase TX_TOTAL_USE."
        )
    rng = np.random.RandomState(seed)
    order = rng.permutation(len(all_known))
    tx_splits = []
    for idx in order[:n_splits]:
        known = list(all_known[idx])
        unknown = [tx for tx in tx_use if tx not in known]
        tx_splits.append((known, unknown))
    return tx_splits

def build_protocol_specs(tx_use, rx_use, selected_rx_protocols, selected_tx_protocols,
                         rx_protocol_library, tx_protocol_library,
                         tx_split_repeats, seed=42, explicit_protocol_combos=None):
    if explicit_protocol_combos is None:
        combo_pairs = [(rx_tag, tx_tag) for rx_tag in selected_rx_protocols for tx_tag in selected_tx_protocols]
    else:
        combo_pairs = list(explicit_protocol_combos)

    protocol_specs = []
    for combo_idx, (rx_tag, tx_tag) in enumerate(combo_pairs, start=1):
        if rx_tag not in rx_protocol_library:
            raise KeyError(f"Unknown RX protocol tag: {rx_tag}")
        if tx_tag not in tx_protocol_library:
            raise KeyError(f"Unknown TX protocol tag: {tx_tag}")

        rx_cfg = dict(rx_protocol_library[rx_tag])
        tx_cfg = dict(tx_protocol_library[tx_tag])

        src_n = int(rx_cfg["source_rx_num"])
        drift_n = int(rx_cfg["drift_rx_num"])
        known_n = int(tx_cfg["known_tx_num"])
        unknown_n = int(tx_cfg["unknown_tx_num"])

        if src_n + drift_n != len(rx_use):
            raise ValueError(
                f"RX protocol {rx_tag} expects source+drift={src_n + drift_n}, but len(RX_USE)={len(rx_use)}."
            )
        if known_n + unknown_n != len(tx_use):
            raise ValueError(
                f"TX protocol {tx_tag} expects known+unknown={known_n + unknown_n}, but len(TX_USE)={len(tx_use)}."
            )

        rng_rx = np.random.RandomState(seed + 1000 * combo_idx + 17 * src_n + drift_n)
        source_rxs = list(rng_rx.choice(list(rx_use), size=src_n, replace=False))
        source_rxs.sort()
        drift_rx = [r for r in rx_use if r not in source_rxs]
        if len(drift_rx) != drift_n:
            raise RuntimeError(f"RX protocol {rx_tag} produced drift count mismatch: {len(drift_rx)} vs expected {drift_n}")

        tx_splits = build_unique_tx_splits(tx_use, known_n, tx_split_repeats, seed=seed + 5000 + 97 * combo_idx)

        protocol_specs.append(dict(
            protocol_tag=f"RX{rx_tag}_TX{tx_tag}",
            rx_protocol=rx_tag,
            tx_protocol=tx_tag,
            source_rx_num=src_n,
            drift_rx_num=drift_n,
            known_tx_num=known_n,
            unknown_tx_num=unknown_n,
            source_rxs=source_rxs,
            drift_rx=drift_rx,
            tx_splits=tx_splits))
    return protocol_specs

PROTOCOL_SPECS = build_protocol_specs(
    TX_USE, RX_USE,
    SELECTED_RX_PROTOCOLS, SELECTED_TX_PROTOCOLS,
    RX_PROTOCOL_LIBRARY, TX_PROTOCOL_LIBRARY,
    tx_split_repeats=TX_SPLIT_REPEATS,
    seed=SEED + 777,
    explicit_protocol_combos=SELECTED_PROTOCOL_COMBOS)

log_banner("Selected protocol combinations")
print("Selected protocol combinations:", len(PROTOCOL_SPECS))
for spec in PROTOCOL_SPECS:
    print(f"[{spec['protocol_tag']}] SOURCE_RXS={spec['source_rxs']} | DRIFT_RX={spec['drift_rx']}")
    print(f"  TX protocol={spec['tx_protocol']} -> unique TX splits={len(spec['tx_splits'])}")
    for i, (known, unknown) in enumerate(spec["tx_splits"], start=1):
        print(f"  TX_SPLIT_{i}: KNOWN={known} | UNKNOWN={unknown}")

# Legacy protocol-tuning entrypoints were removed in the minimal SODA-RFF notebook.
# Keep an empty placeholder only so downstream code and editor diagnostics remain clean.
PROTOCOL_SPECS_TO_RUN = []


TX_USE: ['14-10', '14-7', '20-15', '20-19', '6-15', '8-20']
RX_USE: ['1-1', '1-19', '14-7', '18-2', '19-2', '2-1', '2-19', '20-1', '3-19', '7-14', '7-7', '8-8']
DATE_USE: ['2021_03_01', '2021_03_08', '2021_03_15', '2021_03_23']

================== Selected protocol combinations ==================
Selected protocol combinations: 4
[RX9-3_TX4-2] SOURCE_RXS=[np.str_('14-7'), np.str_('18-2'), np.str_('2-1'), np.str_('2-19'), np.str_('20-1'), np.str_('3-19'), np.str_('7-14'), np.str_('7-7'), np.str_('8-8')] | DRIFT_RX=['1-1', '1-19', '19-2']
  TX protocol=4-2 -> unique TX splits=3
  TX_SPLIT_1: KNOWN=['14-7', '20-15', '20-19', '8-20'] | UNKNOWN=['14-10', '6-15']
  TX_SPLIT_2: KNOWN=['14-10', '20-15', '20-19', '8-20'] | UNKNOWN=['14-7', '6-15']
  TX_SPLIT_3: KNOWN=['14-10', '20-15', '20-19', '6-15'] | UNKNOWN=['14-7', '8-20']
[RX9-3_TX2-4] SOURCE_RXS=[np.str_('1-1'), np.str_('1-19'), np.str_('14-7'), np.str_('18-2'), np.str_('2-1'), np.str_('3-19'), np.str_('7-14'), np.str_('7-7'), np.str_('

In [4]:
class BasicBlock1D(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1, downsample=None, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv1d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(planes)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)

class ResNet18_1D(nn.Module):
    def __init__(self, num_classes, in_planes=64, dropout=0.0):
        super().__init__()
        self.in_planes = in_planes
        self.conv1 = nn.Conv1d(2, in_planes, 7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(in_planes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(3, stride=2, padding=1)
        self.layer1 = self._make_layer(64, 2, stride=1, dropout=dropout)
        self.layer2 = self._make_layer(128, 2, stride=2, dropout=dropout)
        self.layer3 = self._make_layer(256, 2, stride=2, dropout=dropout)
        self.layer4 = self._make_layer(512, 2, stride=2, dropout=dropout)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, planes, blocks, stride, dropout):
        downsample = None
        if stride != 1 or self.in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv1d(self.in_planes, planes, 1, stride=stride, bias=False),
                nn.BatchNorm1d(planes)
            )
        layers = [BasicBlock1D(self.in_planes, planes, stride, downsample, dropout)]
        self.in_planes = planes
        for _ in range(1, blocks):
            layers.append(BasicBlock1D(self.in_planes, planes, dropout=dropout))
        return nn.Sequential(*layers)

    def forward(self, x, return_embed=False):
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).squeeze(-1)
        logits = self.fc(x)
        if return_embed:
            return logits, x
        return logits


In [5]:
class ArrayDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, i):
        return self.X[i], self.y[i]

def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    crit = nn.CrossEntropyLoss()
    total_loss, total_correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        if train:
            optimizer.zero_grad()
        logits = model(xb)
        loss = crit(logits, yb)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += float(loss.item()) * yb.size(0)
        total_correct += int((logits.argmax(1) == yb).sum().item())
        total += int(yb.size(0))
    return total_loss/max(1,total), total_correct/max(1,total)

def infer_logits_embed(model, X_np, batch=512):
    model.eval()
    ds = ArrayDataset(X_np, np.zeros((len(X_np)), dtype=np.int64))
    dl = DataLoader(ds, batch_size=batch, shuffle=False)
    logits_all, Z_all = [], []
    with torch.no_grad():
        for xb,_ in dl:
            xb = xb.to(device)
            logits, emb = model(xb, return_embed=True)
            logits_all.append(logits.cpu().numpy().astype(np.float32))
            Z_all.append(emb.cpu().numpy().astype(np.float32))
    return np.concatenate(logits_all,0), np.concatenate(Z_all,0)

def closed_set_acc_from_logits(logits, y_true):
    return float((np.argmax(logits,1) == y_true).mean())

def roc_auc(y_true, score):
    y_true = np.asarray(y_true).reshape(-1)
    score = np.asarray(score, dtype=np.float64).reshape(-1)
    finite = np.isfinite(score)
    if finite.sum() < 2:
        return 0.5
    y_true = y_true[finite]
    score = score[finite]
    if np.unique(y_true).size < 2:
        return 0.5
    try:
        fpr, tpr, _ = roc_curve(y_true, score)
        return float(auc(fpr, tpr))
    except Exception:
        return 0.5


def compute_known_unknown_auc_metrics(score_A, score_B, score_C, score_U):
    """Return AUC_SU / AUC_DU / AUC_KU, keeping auc_unknown_eval as a legacy AUC_SU alias."""
    score_A = np.asarray(score_A, dtype=np.float32).reshape(-1)
    score_B = np.asarray(score_B, dtype=np.float32).reshape(-1)
    score_C = np.asarray(score_C, dtype=np.float32).reshape(-1)
    score_U = np.asarray(score_U, dtype=np.float32).reshape(-1)

    drift_known = np.concatenate([score_B, score_C], axis=0).astype(np.float32)
    all_known = np.concatenate([score_A, drift_known], axis=0).astype(np.float32)

    auc_su = roc_auc(
        np.concatenate([np.zeros((len(score_A)), dtype=np.int64), np.ones((len(score_U)), dtype=np.int64)]),
        np.concatenate([score_A, score_U], axis=0)
    )
    auc_du = roc_auc(
        np.concatenate([np.zeros((len(drift_known)), dtype=np.int64), np.ones((len(score_U)), dtype=np.int64)]),
        np.concatenate([drift_known, score_U], axis=0)
    )
    auc_ku = roc_auc(
        np.concatenate([np.zeros((len(all_known)), dtype=np.int64), np.ones((len(score_U)), dtype=np.int64)]),
        np.concatenate([all_known, score_U], axis=0)
    )
    return dict(
        auc_unknown_eval=float(auc_su),  # legacy alias kept for backward compatibility
        auc_su=float(auc_su),
        auc_du=float(auc_du),
        auc_ku=float(auc_ku),
    )


In [6]:
def fit_emb_maha_diag(Z_train, y_train, K):
    mu = np.zeros((K, Z_train.shape[1]), dtype=np.float32)
    var = np.zeros((K, Z_train.shape[1]), dtype=np.float32)
    for k in range(K):
        Zk = Z_train[y_train==k]
        mu[k] = Zk.mean(axis=0)
        var[k] = Zk.var(axis=0) + 1e-3
    return mu, var

def sid_embmaha(Z, mu_z, var_z):
    dif = Z[:,None,:] - mu_z[None,:,:]
    dist = np.sum((dif*dif)/(var_z[None,:,:] + 1e-6), axis=2)
    return (-np.min(dist, axis=1)).astype(np.float32)

def sid_energy(logits):
    logits = np.asarray(logits, dtype=np.float32)
    logits = np.nan_to_num(logits, nan=-50.0, posinf=50.0, neginf=-50.0)
    m = logits.max(axis=1, keepdims=True)
    out = m + np.log(np.exp(np.clip(logits - m, -60.0, 60.0)).sum(axis=1, keepdims=True) + 1e-12)
    return np.nan_to_num(out.squeeze(1), nan=-1e6, posinf=1e6, neginf=-1e6).astype(np.float32)

def zscore_fixed(x, ref):
    x = np.asarray(x, dtype=np.float32)
    ref = np.asarray(ref, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=1e6, neginf=-1e6)
    ref = np.nan_to_num(ref, nan=0.0, posinf=1e6, neginf=-1e6)
    mu = float(np.mean(ref))
    std = float(np.std(ref))
    if not np.isfinite(std) or std < 1e-12:
        std = 1.0
    out = (x - mu) / std
    return np.nan_to_num(out, nan=0.0, posinf=1e6, neginf=-1e6).astype(np.float32)

def sid_fusion_fixed(Z, logits, mu_z, var_z, ref_sid_emb_A, ref_sid_en_A, lam=0.5):
    out = (zscore_fixed(sid_embmaha(Z, mu_z, var_z), ref_sid_emb_A) +
           lam * zscore_fixed(sid_energy(logits), ref_sid_en_A))
    return np.nan_to_num(out, nan=-1e6, posinf=1e6, neginf=-1e6).astype(np.float32)

def mahalanobis_batch(D, mu, Sinv):
    X = D - mu.reshape(1,-1)
    return np.einsum("nd,dd,nd->n", X, Sinv, X).astype(np.float32)

def fit_gaussian(D, reg=1e-3):
    mu = D.mean(axis=0).astype(np.float32)
    C = np.cov(D.T, bias=False).astype(np.float32)
    C = C + reg*np.eye(C.shape[0], dtype=np.float32)
    Sinv = np.linalg.inv(C).astype(np.float32)
    sign, logdet = np.linalg.slogdet(C)
    if sign <= 0:
        logdet = float(np.log(np.maximum(np.linalg.det(C), 1e-12)))
    return mu, Sinv, float(logdet)

def logpdf_gaussian(D, mu, Sinv, logdet):
    d = D.shape[1]
    maha = mahalanobis_batch(D, mu, Sinv)
    return (-0.5*(maha + logdet + d*np.log(2*np.pi))).astype(np.float32)

def fit_device_rx_models(D_train, y_train, rx_train, K, source_rx_ids, reg=1e-3, min_n=20):
    models = {}
    fallback = {}
    for k in range(K):
        fallback[k] = fit_gaussian(D_train[y_train==k], reg=reg)
        for rxid in source_rx_ids:
            idx = np.where((y_train==k) & (rx_train==rxid))[0]
            if idx.size >= min_n:
                models[(k,rxid)] = fit_gaussian(D_train[idx], reg=reg)
    return models, fallback

def sdom_V1_mixNLL(D_eval, k_hat, models_kr, fallback_k, source_rx_ids):
    N = D_eval.shape[0]
    out = np.zeros((N), dtype=np.float32)
    R = len(source_rx_ids)
    for i in range(N):
        k = int(k_hat[i])
        d = D_eval[i:i+1]
        logps = []
        for rxid in source_rx_ids:
            mu,Sinv,logdet = models_kr.get((k,rxid), fallback_k[k])
            logps.append(float(logpdf_gaussian(d, mu, Sinv, logdet)[0]))
        logps = np.array(logps, dtype=np.float32)
        m = logps.max()
        loglik = m + np.log(np.exp(logps-m).sum()+1e-12) - np.log(R)
        out[i] = float(-loglik)
    return out

def calibrate_module2_thresholds(Sid_A, Sdom_A, frr_target=0.05, false_drift_target=0.05):
    tau_id = float(np.quantile(Sid_A, frr_target))
    tau_dom = float(np.quantile(Sdom_A, 1.0 - false_drift_target))
    return tau_id, tau_dom

def gate_predict(Sid, Sdom, tau_id, tau_dom):
    pred = np.zeros_like(Sid, dtype=np.int64)
    pred[Sid < tau_id] = 2
    pred[(Sid >= tau_id) & (Sdom > tau_dom)] = 1
    return pred

def compute_module2_scores_from_cached(Z, logits, D, mu_z_src, var_z_src,
                                       ref_sid_emb_A, ref_sid_en_A,
                                       models_kr, fallback_k, source_rx_ids,
                                       tau_id=None, tau_dom=None):
    logits = np.nan_to_num(np.asarray(logits, dtype=np.float32), nan=0.0, posinf=50.0, neginf=-50.0)
    Z = np.nan_to_num(np.asarray(Z, dtype=np.float32), nan=0.0, posinf=1e4, neginf=-1e4)
    D = np.nan_to_num(np.asarray(D, dtype=np.float32), nan=0.0, posinf=1e4, neginf=-1e4)
    k_hat = np.argmax(logits, axis=1).astype(np.int64)
    Sid = sid_fusion_fixed(Z, logits, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, lam=FUSION_LAM)
    Sdom = sdom_V1_mixNLL(D, k_hat, models_kr, fallback_k, source_rx_ids)
    Sid = np.nan_to_num(Sid, nan=-1e6, posinf=1e6, neginf=-1e6).astype(np.float32)
    Sdom = np.nan_to_num(Sdom, nan=1e6, posinf=1e6, neginf=-1e6).astype(np.float32)
    out = dict(Sid=Sid, Sdom=Sdom, k_hat=k_hat)
    if tau_id is not None and tau_dom is not None:
        out["gate_pred"] = gate_predict(Sid, Sdom, tau_id, tau_dom)
    return out


def robust_scale(x, min_scale=0.10):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=1e6, neginf=-1e6)
    if x.size == 0:
        return float(max(min_scale, 1.0))
    q75, q25 = np.quantile(x, [0.75, 0.25])
    iqr = float(q75 - q25)
    std = float(np.std(x))
    scale = max(iqr / 1.349 if iqr > 1e-8 else 0.0, std, min_scale)
    return float(scale)

def fit_classwise_dom_stats(Sdom_train, y_train, K, min_std=0.10):
    Sdom_train = np.asarray(Sdom_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.int64)
    global_mu = float(np.mean(Sdom_train))
    global_std = float(max(np.std(Sdom_train), min_std))
    dom_mu = np.full((K), global_mu, dtype=np.float32)
    dom_std = np.full((K), global_std, dtype=np.float32)
    for k in range(K):
        vals = Sdom_train[y_train == k]
        if vals.size >= 8:
            dom_mu[k] = float(np.mean(vals))
            dom_std[k] = float(max(np.std(vals), min_std))
    return dom_mu, dom_std, global_mu, global_std

def normalize_dom_by_class(Sdom, k_hat, dom_mu, dom_std, global_mu, global_std):
    Sdom = np.asarray(Sdom, dtype=np.float32)
    k_hat = np.asarray(k_hat, dtype=np.int64)
    out = np.zeros_like(Sdom, dtype=np.float32)
    K = len(dom_mu)
    for i, k in enumerate(k_hat):
        if 0 <= int(k) < K:
            mu = float(dom_mu[int(k)])
            sd = float(dom_std[int(k)])
        else:
            mu = float(global_mu)
            sd = float(global_std)
        out[i] = (float(Sdom[i]) - mu) / max(sd, 1e-6)
    return out.astype(np.float32)

def calibrate_module2_v4_thresholds(Sid_stable, Sdrift_stable, frr_target=0.05, false_drift_target=0.05):
    tau_id = float(np.quantile(Sid_stable, frr_target))
    tau_drift = float(np.quantile(Sdrift_stable, 1.0 - false_drift_target))
    temp_id = robust_scale(Sid_stable, min_scale=0.10)
    temp_drift = robust_scale(Sdrift_stable, min_scale=0.10)
    return tau_id, tau_drift, temp_id, temp_drift

def gate_predict_v4(Sid, Sdrift, tau_id, tau_drift):
    pred = np.zeros_like(Sid, dtype=np.int64)
    pred[Sid < tau_id] = 2
    pred[(Sid >= tau_id) & (Sdrift > tau_drift)] = 1
    return pred

def route_probs_v4(Sid, Sdrift, tau_id, tau_drift, temp_id, temp_drift, unknown_prior=1.0, drift_prior=1.0):
    p_known = sigmoid_np((Sid - tau_id) / max(temp_id, 1e-6))
    p_drift_given_known = sigmoid_np((Sdrift - tau_drift) / max(temp_drift, 1e-6))
    p_unknown = np.clip((1.0 - p_known) * unknown_prior, 1e-8, None)
    p_drift = np.clip(p_known * p_drift_given_known * drift_prior, 1e-8, None)
    p_stable = np.clip(p_known * (1.0 - p_drift_given_known), 1e-8, None)
    probs = np.stack([p_stable, p_drift, p_unknown], axis=1).astype(np.float32)
    probs /= np.clip(probs.sum(axis=1, keepdims=True), 1e-8, None)
    aux = dict(
        p_known=p_known.astype(np.float32),
        p_drift_given_known=p_drift_given_known.astype(np.float32),
        p_stable=p_stable.astype(np.float32),
        p_drift=p_drift.astype(np.float32),
        p_unknown=p_unknown.astype(np.float32))
    return probs, aux

def compute_module2_v4_scores_from_cached(
    Z, logits, D, mu_z_src, var_z_src,
    ref_sid_emb_A, ref_sid_en_A,
    models_kr, fallback_k, source_rx_ids,
    dom_mu, dom_std, dom_global_mu, dom_global_std,
    tau_id=None, tau_drift=None, temp_id=None, temp_drift=None):
    base = compute_module2_scores_from_cached(
        Z, logits, D, mu_z_src, var_z_src,
        ref_sid_emb_A, ref_sid_en_A,
        models_kr, fallback_k, source_rx_ids,
        tau_id=None, tau_dom=None
    )
    Sid = base["Sid"]
    Sdom = base["Sdom"]
    k_hat = base["k_hat"]
    Sdrift = normalize_dom_by_class(Sdom, k_hat, dom_mu, dom_std, dom_global_mu, dom_global_std)
    Sdrift = np.nan_to_num(Sdrift, nan=0.0, posinf=1e6, neginf=-1e6).astype(np.float32)
    out = dict(Sid=Sid, Sdom=Sdom, Sdrift=Sdrift, k_hat=k_hat)
    if tau_id is not None and tau_drift is not None:
        out["gate_pred"] = gate_predict_v4(Sid, Sdrift, tau_id, tau_drift)
        if temp_id is None:
            temp_id = robust_scale(Sid, min_scale=0.10)
        if temp_drift is None:
            temp_drift = robust_scale(Sdrift, min_scale=0.10)
        probs, aux = route_probs_v4(Sid, Sdrift, tau_id, tau_drift, temp_id, temp_drift)
        out["route_probs"] = probs
        out.update(aux)
    return out


In [7]:
import warnings

# =========================
# Module2-v8 joint router helpers (ported into Module3)
# Source-calibrated synthetic-drift tri-state router (module2_v8 / v7_C path)
# =========================
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings(
    "ignore",
    message=r".*'multi_class' was deprecated.*",
    category=FutureWarning)

# ---- module2_v8 router config ----
M2V8_ROUTE_NAME = "v8_C"
M2V8_ROUTER_FEATURE_MODE = "hardk"   # module2_v8 keeps only the hard-k path
M2V8_SYNTH_DRIFT_COPIES = 1
M2V8_SYNTH_SHIFT_MAX = 4
M2V8_SYNTH_PHASE_SLOPE_MAX = 0.08
M2V8_SYNTH_GAIN_MIN = 0.85
M2V8_SYNTH_GAIN_MAX = 1.15
M2V8_SYNTH_IQ_LEAK_MAX = 0.08
M2V8_SYNTH_NOISE_STD_MIN = 0.005
M2V8_SYNTH_NOISE_STD_MAX = 0.03
M2V8_SYNTH_MIN_CLASS_CAL = 24

def cosine_to_proto(Z, protos):
    Zx = normalize_vec_rows_np(Z)
    Px = normalize_vec_rows_np(protos)
    return (Zx @ Px.T).astype(np.float32)

def sdom_V1_mixNLL_allk(D_eval, K, models_kr, fallback_k, source_rx_ids):
    D_eval = np.asarray(D_eval, dtype=np.float32)
    N = D_eval.shape[0]
    out = np.zeros((N, K), dtype=np.float32)
    R = max(1, len(source_rx_ids))
    for k in range(K):
        logps = []
        for rxid in source_rx_ids:
            mu, Sinv, logdet = models_kr.get((k, rxid), fallback_k[k])
            logps.append(logpdf_gaussian(D_eval, mu, Sinv, logdet).astype(np.float32))
        logps = np.stack(logps, axis=1)
        m = np.max(logps, axis=1, keepdims=True)
        loglik = m[:, 0] + np.log(np.exp(logps - m).sum(axis=1) + 1e-12) - np.log(R)
        out[:, k] = (-loglik).astype(np.float32)
    return out

def normalize_dom_matrix_by_class(Sdom_allk, dom_mu, dom_std, dom_global_mu, dom_global_std):
    Sdom_allk = np.asarray(Sdom_allk, dtype=np.float32)
    K = Sdom_allk.shape[1]
    out = np.zeros_like(Sdom_allk, dtype=np.float32)
    for k in range(K):
        mu = float(dom_mu[k]) if k < len(dom_mu) else float(dom_global_mu)
        sd = float(dom_std[k]) if k < len(dom_std) else float(dom_global_std)
        out[:, k] = (Sdom_allk[:, k] - mu) / max(sd, 1e-6)
    return out.astype(np.float32)

def gather_hard_class_scores(score_allk, k_hat):
    score_allk = np.asarray(score_allk, dtype=np.float32)
    k_hat = np.asarray(k_hat, dtype=np.int64)
    return score_allk[np.arange(len(k_hat)), k_hat].astype(np.float32)

def complex_to_iq(x_c):
    return np.stack([np.real(x_c), np.imag(x_c)], axis=-1).astype(np.float32)

def synthetic_drift_augment_batch(X_256_2, rng, copies=1):
    X = np.asarray(X_256_2, dtype=np.float32)
    if X.ndim != 3 or X.shape[-1] != 2:
        raise ValueError(f"Expected [N, L, 2], got {X.shape}")
    Xc = iq_to_complex(X)
    N, L = Xc.shape
    n = np.arange(L, dtype=np.float32).reshape(1, L)

    outs = []
    for _ in range(int(max(1, copies))):
        xc = Xc.copy()

        shifts = rng.randint(-M2V8_SYNTH_SHIFT_MAX, M2V8_SYNTH_SHIFT_MAX + 1, size=N)
        for i, s in enumerate(shifts):
            if s != 0:
                xc[i] = np.roll(xc[i], int(s))

        phase0 = rng.uniform(-np.pi, np.pi, size=(N, 1)).astype(np.float32)
        slope = rng.uniform(-M2V8_SYNTH_PHASE_SLOPE_MAX, M2V8_SYNTH_PHASE_SLOPE_MAX, size=(N, 1)).astype(np.float32)
        xc = xc * np.exp(1j * (phase0 + slope * n)).astype(np.complex64)

        gain = rng.uniform(M2V8_SYNTH_GAIN_MIN, M2V8_SYNTH_GAIN_MAX, size=(N, 1)).astype(np.float32)
        xc = xc * gain

        leak = rng.uniform(-M2V8_SYNTH_IQ_LEAK_MAX, M2V8_SYNTH_IQ_LEAK_MAX, size=(N, 1)).astype(np.float32)
        i_gain = rng.uniform(0.95, 1.05, size=(N, 1)).astype(np.float32)
        q_gain = rng.uniform(0.95, 1.05, size=(N, 1)).astype(np.float32)
        I = np.real(xc).astype(np.float32)
        Q = np.imag(xc).astype(np.float32)
        I2 = i_gain * I + leak * Q
        Q2 = q_gain * Q + leak * I
        xc = (I2 + 1j * Q2).astype(np.complex64)

        noise_std = rng.uniform(M2V8_SYNTH_NOISE_STD_MIN, M2V8_SYNTH_NOISE_STD_MAX, size=(N, 1)).astype(np.float32)
        noise = (rng.randn(N, L) + 1j * rng.randn(N, L)).astype(np.complex64)
        xc = xc + noise_std * noise

        rms = np.sqrt(np.mean(np.abs(xc) ** 2, axis=1, keepdims=True) + 1e-12).astype(np.float32)
        xc = (xc / rms).astype(np.complex64)

        outs.append(complex_to_iq(xc))

    return np.concatenate(outs, axis=0).astype(np.float32)

def fit_binary_score_calibrator(neg_scores, pos_scores):
    neg_scores = np.asarray(neg_scores, dtype=np.float32).reshape(-1, 1)
    pos_scores = np.asarray(pos_scores, dtype=np.float32).reshape(-1, 1)

    if neg_scores.shape[0] < 4 or pos_scores.shape[0] < 4:
        thr = float(np.quantile(neg_scores[:, 0], 0.95)) if neg_scores.shape[0] > 0 else 0.0
        return dict(kind="threshold", tau=thr, temp=max(float(np.std(neg_scores[:, 0])), 1e-3))

    X = np.concatenate([neg_scores, pos_scores], axis=0)
    y = np.concatenate([
        np.zeros((neg_scores.shape[0]), dtype=np.int64),
        np.ones((pos_scores.shape[0]), dtype=np.int64)])
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    clf = LogisticRegression(max_iter=1000, class_weight="balanced", solver="lbfgs")
    clf.fit(Xs, y)
    return dict(kind="logreg", scaler=scaler, clf=clf)

def predict_binary_score_calibrator(scores, bundle):
    scores = np.asarray(scores, dtype=np.float32).reshape(-1, 1)
    if bundle["kind"] == "threshold":
        tau = float(bundle["tau"])
        temp = max(float(bundle["temp"]), 1e-6)
        return sigmoid_np((scores[:, 0] - tau) / temp).astype(np.float32)
    Xs = bundle["scaler"].transform(scores)
    return bundle["clf"].predict_proba(Xs)[:, 1].astype(np.float32)

def build_module2_v8_router_config(Sid_cal, Sdrift_router_cal, Sdrift_true_cal, y_true_cal,
                                   Sdrift_syn_true, y_syn_true, K):
    cfg = dict(
        name=M2V8_ROUTE_NAME,
        sequential_unknown_gate=False,     # v7_C: use argmax over tri-state probs
        classwise_drift=False,             # v7_C: global drift calibrator
        synthetic_drift_cal=True,          # v7_C: source-only synthetic drift calibration
        tau_id=float(np.quantile(np.asarray(Sid_cal, dtype=np.float32), FRR_TARGET)),
        temp_id=float(robust_scale(Sid_cal)),
        drift_mode="global_synth_calibrator",
        drift_calibrator=fit_binary_score_calibrator(Sdrift_true_cal, Sdrift_syn_true))
    return cfg

def predict_module2_v8_router(Sid, Sdrift_router, Sdrift_hard, k_hat, cfg):
    Sid = np.asarray(Sid, dtype=np.float32)
    Sdrift_router = np.asarray(Sdrift_router, dtype=np.float32)

    tau_id = float(cfg["tau_id"])
    temp_id = max(float(cfg["temp_id"]), 1e-6)

    p_unknown = sigmoid_np((tau_id - Sid) / temp_id).astype(np.float32)
    p_known = 1.0 - p_unknown
    p_drift_cond = predict_binary_score_calibrator(Sdrift_router, cfg["drift_calibrator"])
    p_stable = (p_known * (1.0 - p_drift_cond)).astype(np.float32)
    p_drift = (p_known * p_drift_cond).astype(np.float32)

    probs = np.stack([p_stable, p_drift, p_unknown], axis=1).astype(np.float32)
    probs = probs / np.clip(probs.sum(axis=1, keepdims=True), 1e-12, None)
    pred = np.argmax(probs, axis=1).astype(np.int64)

    aux = dict(
        p_unknown=p_unknown.astype(np.float32),
        p_drift_cond=p_drift_cond.astype(np.float32))
    return pred, probs, aux

def summarize_module2_v8_cfg(cfg):
    return dict(
        name=str(cfg.get("name", M2V8_ROUTE_NAME)),
        sequential_unknown_gate=bool(cfg["sequential_unknown_gate"]),
        classwise_drift=bool(cfg["classwise_drift"]),
        synthetic_drift_cal=bool(cfg["synthetic_drift_cal"]),
        tau_id=float(cfg["tau_id"]),
        temp_id=float(cfg["temp_id"]),
        drift_mode=str(cfg["drift_mode"]))

def fit_module2_v8_router_from_source_calibration(
    X_stable,
    y_stable,
    Z_stable,
    logits_stable,
    D_stable,
    protos,
    mu_z_src, var_z_src,
    ref_sid_emb_A, ref_sid_en_A,
    models_kr, fallback_k, source_rx_ids,
    dom_mu, dom_std, dom_global_mu, dom_global_std,
    seed=SEED):
    X_stable = np.asarray(X_stable, dtype=np.float32)
    y_stable = np.asarray(y_stable, dtype=np.int64)
    Z_stable = np.asarray(Z_stable, dtype=np.float32)
    logits_stable = np.asarray(logits_stable, dtype=np.float32)
    D_stable = np.asarray(D_stable, dtype=np.float32)
    K = protos.shape[0]

    Sid_cal = sid_fusion_fixed(
        Z_stable, logits_stable,
        mu_z_src, var_z_src,
        ref_sid_emb_A, ref_sid_en_A,
        lam=FUSION_LAM)
    Cos = cosine_to_proto(Z_stable, protos)
    k_hat = np.argmax(Cos, axis=1).astype(np.int64)
    Sdom_allk = sdom_V1_mixNLL_allk(D_stable, K, models_kr, fallback_k, source_rx_ids)
    Sdrift_allk = normalize_dom_matrix_by_class(
        Sdom_allk, dom_mu, dom_std, dom_global_mu, dom_global_std
    )
    Sdom_hard = gather_hard_class_scores(Sdom_allk, k_hat)
    Sdrift_hard = gather_hard_class_scores(Sdrift_allk, y_stable)
    Sdrift_router_cal = gather_hard_class_scores(Sdrift_allk, k_hat)

    rng_syn = np.random.RandomState(seed)
    X_syn = synthetic_drift_augment_batch(X_stable, rng_syn, copies=M2V8_SYNTH_DRIFT_COPIES)
    y_syn = np.repeat(y_stable, int(max(1, M2V8_SYNTH_DRIFT_COPIES)))
    D_syn = compute_domain_feats_batch(X_syn, d_dim=D_DIM)
    Sdom_allk_syn = sdom_V1_mixNLL_allk(D_syn, K, models_kr, fallback_k, source_rx_ids)
    Sdrift_allk_syn = normalize_dom_matrix_by_class(
        Sdom_allk_syn, dom_mu, dom_std, dom_global_mu, dom_global_std
    )
    Sdrift_syn_true = gather_hard_class_scores(Sdrift_allk_syn, y_syn)

    cfg = build_module2_v8_router_config(
        Sid_cal=Sid_cal,
        Sdrift_router_cal=Sdrift_router_cal,
        Sdrift_true_cal=Sdrift_hard,
        y_true_cal=y_stable,
        Sdrift_syn_true=Sdrift_syn_true,
        y_syn_true=y_syn,
        K=K)

    return dict(
        route_name=M2V8_ROUTE_NAME,
        router_feature_mode=M2V8_ROUTER_FEATURE_MODE,
        cfg=cfg,
        protos=np.asarray(protos, dtype=np.float32),
        mu_z_src=np.asarray(mu_z_src, dtype=np.float32),
        var_z_src=np.asarray(var_z_src, dtype=np.float32),
        ref_sid_emb_A=np.asarray(ref_sid_emb_A, dtype=np.float32),
        ref_sid_en_A=np.asarray(ref_sid_en_A, dtype=np.float32),
        models_kr=models_kr,
        fallback_k=fallback_k,
        source_rx_ids=source_rx_ids,
        dom_mu=np.asarray(dom_mu, dtype=np.float32),
        dom_std=np.asarray(dom_std, dtype=np.float32),
        dom_global_mu=float(dom_global_mu),
        dom_global_std=float(dom_global_std),
        cfg_summary=summarize_module2_v8_cfg(cfg),
        debug=dict(
            Sid_cal=Sid_cal.astype(np.float32),
            Sdom_hard=Sdom_hard.astype(np.float32),
            Sdrift_true=Sdrift_hard.astype(np.float32),
            Sdrift_router=Sdrift_router_cal.astype(np.float32),
            Sdrift_syn_true=Sdrift_syn_true.astype(np.float32),
            y_stable=y_stable.astype(np.int64),
            y_syn=y_syn.astype(np.int64)))

def compute_module2_v8_scores_from_cached(Z, logits, D, router_bundle):
    Z = np.asarray(Z, dtype=np.float32)
    logits = np.asarray(logits, dtype=np.float32)
    D = np.asarray(D, dtype=np.float32)
    protos = router_bundle["protos"]
    K = protos.shape[0]

    Sid = sid_fusion_fixed(
        Z, logits,
        router_bundle["mu_z_src"], router_bundle["var_z_src"],
        router_bundle["ref_sid_emb_A"], router_bundle["ref_sid_en_A"],
        lam=FUSION_LAM)
    Cos = cosine_to_proto(Z, protos)
    k_hat = np.argmax(Cos, axis=1).astype(np.int64)
    Sdom_allk = sdom_V1_mixNLL_allk(
        D, K, router_bundle["models_kr"], router_bundle["fallback_k"], router_bundle["source_rx_ids"]
    )
    Sdrift_allk = normalize_dom_matrix_by_class(
        Sdom_allk,
        router_bundle["dom_mu"], router_bundle["dom_std"],
        router_bundle["dom_global_mu"], router_bundle["dom_global_std"])
    Sdom = gather_hard_class_scores(Sdom_allk, k_hat)
    Sdrift_hard = gather_hard_class_scores(Sdrift_allk, k_hat)
    Sdrift_router = Sdrift_hard.copy()  # module2_v8 notebook keeps hard-k path only

    gate_pred, probs, aux = predict_module2_v8_router(
        Sid=Sid,
        Sdrift_router=Sdrift_router,
        Sdrift_hard=Sdrift_hard,
        k_hat=k_hat,
        cfg=router_bundle["cfg"])

    p_stable = probs[:, 0]
    p_drift = probs[:, 1]
    p_unknown = probs[:, 2]
    p_known = np.clip(p_stable + p_drift, 1e-8, 1.0)
    p_drift_given_known = np.clip(p_drift / p_known, 0.0, 1.0)

    return dict(
        Sid=Sid.astype(np.float32),
        Sdom=Sdom.astype(np.float32),
        Sdrift=Sdrift_router.astype(np.float32),
        Sdom_allk=Sdom_allk.astype(np.float32),
        Sdrift_allk=Sdrift_allk.astype(np.float32),
        Cos=Cos.astype(np.float32),
        k_hat=k_hat.astype(np.int64),
        gate_pred=gate_pred.astype(np.int64),
        route_probs=probs.astype(np.float32),
        p_stable=p_stable.astype(np.float32),
        p_drift=p_drift.astype(np.float32),
        p_unknown=p_unknown.astype(np.float32),
        p_known=p_known.astype(np.float32),
        p_drift_given_known=p_drift_given_known.astype(np.float32),
        route_aux=dict(
            p_unknown=aux["p_unknown"].astype(np.float32),
            p_drift_cond=aux["p_drift_cond"].astype(np.float32)))


In [8]:
def softmax_np(logits):
    z = logits - logits.max(axis=1, keepdims=True)
    ez = np.exp(z)
    return ez / (ez.sum(axis=1, keepdims=True) + 1e-12)

def normalize_rows(P):
    P = np.asarray(P, dtype=np.float32)
    P = np.maximum(P, 1e-12)
    P = P / P.sum(axis=1, keepdims=True)
    return P.astype(np.float32)

def msp_from_probs(P):
    return P.max(axis=1).astype(np.float32)

def msp_from_logits(logits):
    return msp_from_probs(softmax_np(logits))

def onehot_from_labels(y, K):
    y = np.asarray(y, dtype=np.int64)
    out = np.zeros((len(y), K), dtype=np.float32)
    out[np.arange(len(y)), y] = 1.0
    return out

def pseudo_acc_from_probs(P, y_true):
    pred = np.argmax(P, axis=1)
    mask = y_true >= 0
    return float((pred[mask] == y_true[mask]).mean()) if np.any(mask) else float("nan")

def fit_class_prototypes(Z_train, y_train, K, l2norm=True):
    Z = Z_train.copy().astype(np.float32)
    if l2norm:
        Z = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12)
    protos = np.zeros((K, Z.shape[1]), dtype=np.float32)
    for k in range(K):
        z = Z[y_train == k]
        protos[k] = z.mean(axis=0)
    if l2norm:
        protos = protos / (np.linalg.norm(protos, axis=1, keepdims=True) + 1e-12)
    return protos

def proto_probs_cosine(Z, protos, temp=0.10, l2norm=True):
    Zx = Z.astype(np.float32)
    if l2norm:
        Zx = Zx / (np.linalg.norm(Zx, axis=1, keepdims=True) + 1e-12)
    scores = (Zx @ protos.T) / max(temp, 1e-6)
    return softmax_np(scores)

def build_knn_memory(Z_train, y_train, per_class=300, seed=0):
    rng = np.random.RandomState(seed)
    keep = []
    for k in np.unique(y_train):
        idx = np.where(y_train == k)[0]
        rng.shuffle(idx)
        keep.append(idx[:min(len(idx), per_class)])
    keep = np.concatenate(keep, axis=0)
    Zm = Z_train[keep].astype(np.float32)
    ym = y_train[keep].astype(np.int64)
    Zm = Zm / (np.linalg.norm(Zm, axis=1, keepdims=True) + 1e-12)
    return Zm, ym

def knn_class_probs(Z, Zm, ym, K, topk=15):
    Zx = Z.astype(np.float32)
    Zx = Zx / (np.linalg.norm(Zx, axis=1, keepdims=True) + 1e-12)
    sims = Zx @ Zm.T
    topk = min(topk, Zm.shape[0])
    idx = np.argpartition(-sims, kth=topk-1, axis=1)[:, :topk]
    probs = np.zeros((Zx.shape[0], K), dtype=np.float32)
    for i in range(Zx.shape[0]):
        ii = idx[i]
        ss = sims[i, ii]
        ww = np.exp(ss - ss.max())
        for j, w in zip(ii, ww):
            probs[i, ym[j]] += float(w)
    return normalize_rows(probs)

def weighted_prob_fusion(prob_list, weights):
    out = np.zeros_like(prob_list[0], dtype=np.float32)
    for P, w in zip(prob_list, weights):
        out += float(w) * P.astype(np.float32)
    return normalize_rows(out)

def select_idx_by_mask_with_fallback(base_idx, mask_keep, conf, min_keep=64):
    base_idx = np.asarray(base_idx, dtype=np.int64)
    if len(base_idx) == 0:
        return base_idx
    idx_keep = base_idx[mask_keep]
    if len(idx_keep) >= min_keep:
        return idx_keep
    order = np.argsort(-conf[base_idx])
    k = min(len(base_idx), min_keep)
    return base_idx[order[:k]]

def normalize_conf_weights(conf):
    conf = np.asarray(conf, dtype=np.float32)
    if len(conf) == 0:
        return conf
    cmin, cmax = float(conf.min()), float(conf.max())
    if cmax - cmin < 1e-12:
        return np.ones_like(conf, dtype=np.float32)
    return (0.5 + 0.5 * (conf - cmin) / (cmax - cmin)).astype(np.float32)

def summarize_method_on_selected(idx_sel, y_true, P):
    if len(idx_sel) == 0:
        return dict(sel_size=0, pseudo_acc=float("nan"))
    mask = y_true[idx_sel] >= 0
    if np.any(mask):
        pa = float((np.argmax(P[idx_sel], axis=1)[mask] == y_true[idx_sel][mask]).mean())
    else:
        pa = float("nan")
    return dict(sel_size=int(len(idx_sel)), pseudo_acc=pa)


In [9]:
def build_source_train(compact_dataset, KNOWN_TX, source_rxs=None):
    if source_rxs is None:
        source_rxs = globals().get("SOURCE_RXS", None)
        if source_rxs is None:
            raise NameError("source_rxs is required for build_source_train() in protocol-config mode.")
    X_list, y_list, D_list, rx_id_list = [], [], [], []
    for tx in KNOWN_TX:
        for rx in source_rxs:
            for day in TRAIN_DATES:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0 if D_FROM=="raw" else 1), MAX_SIG_PER_TRIPLE)
                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]; Xd = Xd[:n]
                y = np.full((n), KNOWN_TX.index(tx), dtype=np.int64)
                Df = compute_domain_feats_batch(Xd, d_dim=D_DIM)
                X_list.append(Xz); y_list.append(y); D_list.append(Df)
                rx_id_list.append(np.full((n), RX_USE.index(rx), dtype=np.int64))
    return (
        np.concatenate(X_list,0).astype(np.float32),
        np.concatenate(y_list,0).astype(np.int64),
        np.concatenate(D_list,0).astype(np.float32),
        np.concatenate(rx_id_list,0).astype(np.int64))

def build_eval_set(compact_dataset, KNOWN_TX, txs, rxs, days):
    X_list, y_list, D_list = [], [], []
    for tx in txs:
        for rx in rxs:
            for day in days:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0 if D_FROM=="raw" else 1), MAX_SIG_PER_TRIPLE)
                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]; Xd = Xd[:n]
                y = np.full((n), KNOWN_TX.index(tx), dtype=np.int64) if tx in KNOWN_TX else np.full((n), -1, dtype=np.int64)
                Df = compute_domain_feats_batch(Xd, d_dim=D_DIM)
                X_list.append(Xz); y_list.append(y); D_list.append(Df)
    return (
        np.concatenate(X_list,0).astype(np.float32),
        np.concatenate(y_list,0).astype(np.int64),
        np.concatenate(D_list,0).astype(np.float32))

def split_adapt_eval(X, y, D, frac=0.5, max_adapt=None, max_eval=None, seed=0):
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(X))
    cut = int(len(idx) * frac)
    idx_ad = idx[:cut]
    idx_ev = idx[cut:]
    if max_adapt is not None and len(idx_ad) > max_adapt:
        idx_ad = idx_ad[:max_adapt]
    if max_eval is not None and len(idx_ev) > max_eval:
        idx_ev = idx_ev[:max_eval]
    return X[idx_ad], y[idx_ad], D[idx_ad], X[idx_ev], y[idx_ev], D[idx_ev]

def concat_sets(items):
    X = np.concatenate([it[0] for it in items], axis=0)
    y = np.concatenate([it[1] for it in items], axis=0)
    D = np.concatenate([it[2] for it in items], axis=0)
    return X, y, D


In [10]:
def ensure_parent_dir(path):
    if path is None:
        return
    p = _win_safe_path(path) if "_win_safe_path" in globals() else os.path.abspath(os.path.normpath(str(path)))
    parent = os.path.dirname(p)
    if parent:
        os.makedirs(parent, exist_ok=True)


class EmbDatasetWeightedHard(Dataset):
    def __init__(self, Z, y, w=None):
        self.Z = torch.tensor(Z, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        if w is None:
            w = np.ones((len(Z)), dtype=np.float32)
        self.w = torch.tensor(w, dtype=torch.float32)
    def __len__(self):
        return self.Z.shape[0]
    def __getitem__(self, i):
        return self.Z[i], self.y[i], self.w[i]

class EmbDatasetWeightedSoft(Dataset):
    def __init__(self, Z, q, w=None):
        self.Z = torch.tensor(Z, dtype=torch.float32)
        self.q = torch.tensor(q, dtype=torch.float32)
        if w is None:
            w = np.ones((len(Z)), dtype=np.float32)
        self.w = torch.tensor(w, dtype=torch.float32)
    def __len__(self):
        return self.Z.shape[0]
    def __getitem__(self, i):
        return self.Z[i], self.q[i], self.w[i]

class EmbOnlyDataset(Dataset):
    def __init__(self, Z):
        self.Z = torch.tensor(Z, dtype=torch.float32)
    def __len__(self):
        return self.Z.shape[0]
    def __getitem__(self, i):
        return self.Z[i]

class EmbOnlyDatasetWeighted(Dataset):
    def __init__(self, Z, w=None):
        self.Z = torch.tensor(Z, dtype=torch.float32)
        if w is None:
            w = np.ones((len(Z)), dtype=np.float32)
        self.w = torch.tensor(w, dtype=torch.float32)
    def __len__(self):
        return self.Z.shape[0]
    def __getitem__(self, i):
        return self.Z[i], self.w[i]

class ResidualAdapter(nn.Module):
    def __init__(self, dim=512, bottleneck=64, scale=0.3):
        super().__init__()
        self.fc1 = nn.Linear(dim, bottleneck)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(bottleneck, dim)
        self.scale = scale
        nn.init.zeros_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)
    def forward(self, z):
        return z + self.scale * self.fc2(self.relu(self.fc1(z)))


def clone_state_dict_cpu(module):
    return {k: v.detach().cpu().clone() for k, v in module.state_dict().items()}



def apply_adapter_and_fc(adapter, fc_layer, Z_np, batch=512):
    fc_default = nn.Linear(fc_layer.in_features, fc_layer.out_features, bias=True)
    fc_default.load_state_dict(fc_layer.state_dict())
    fc_default.eval()

    custom_fc = None
    adapter_module = adapter
    if isinstance(adapter, dict):
        adapter_module = adapter.get("adapter", None)
        custom_fc = adapter.get("fc", None)

    if custom_fc is not None:
        fc = custom_fc
    else:
        fc = fc_default

    if adapter_module is None:
        with torch.no_grad():
            z = torch.tensor(Z_np, dtype=torch.float32)
            logits = fc(z).numpy().astype(np.float32)
        z_np = np.nan_to_num(np.asarray(Z_np, dtype=np.float32), nan=0.0, posinf=1e4, neginf=-1e4)
        logits = np.nan_to_num(logits, nan=0.0, posinf=50.0, neginf=-50.0).astype(np.float32)
        return z_np, logits

    fc = fc.to(device)
    fc.eval()
    for p in fc.parameters():
        p.requires_grad = False
    adapter_module = adapter_module.to(device)
    adapter_module.eval()

    ds = EmbDatasetWeightedHard(Z_np, np.zeros((len(Z_np)), dtype=np.int64))
    dl = DataLoader(ds, batch_size=batch, shuffle=False)
    Z2_all, logits_all = [], []
    with torch.no_grad():
        for zb,_,_ in dl:
            zb = zb.to(device)
            z2 = adapter_module(zb)
            logits = fc(z2)
            Z2_all.append(z2.cpu().numpy().astype(np.float32))
            logits_all.append(logits.cpu().numpy().astype(np.float32))
    Z2 = np.concatenate(Z2_all,0)
    logits = np.concatenate(logits_all,0)
    Z2 = np.nan_to_num(Z2, nan=0.0, posinf=1e4, neginf=-1e4).astype(np.float32)
    logits = np.nan_to_num(logits, nan=0.0, posinf=50.0, neginf=-50.0).astype(np.float32)
    return Z2, logits



def make_replay_subset(Z_src, y_src, per_class=200, seed=0):
    rng = np.random.RandomState(seed)
    keep = []
    for k in np.unique(y_src):
        idx = np.where(y_src == k)[0]
        rng.shuffle(idx)
        keep.append(idx[:min(len(idx), per_class)])
    keep = np.concatenate(keep, axis=0)
    return Z_src[keep], y_src[keep]


def normalize_vec_rows_np(X):
    X = np.asarray(X, dtype=np.float32)
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)


def sigmoid_np(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.clip(x, -30.0, 30.0)
    return 1.0 / (1.0 + np.exp(-x))


def robust_unit_interval(x, idx_ref=None):
    x = np.asarray(x, dtype=np.float32)
    if idx_ref is None or len(idx_ref) == 0:
        ref = x
    else:
        ref = x[np.asarray(idx_ref, dtype=np.int64)]
    lo = float(np.quantile(ref, 0.05))
    hi = float(np.quantile(ref, 0.95))
    if hi - lo < 1e-8:
        return np.zeros_like(x, dtype=np.float32)
    y = (x - lo) / (hi - lo)
    return np.clip(y, 0.0, 1.0).astype(np.float32)


def top2_margin(P):
    P = np.asarray(P, dtype=np.float32)
    if P.shape[1] < 2:
        return np.ones((P.shape[0]), dtype=np.float32)
    s = np.sort(P, axis=1)
    return (s[:, -1] - s[:, -2]).astype(np.float32)


def prototype_margin_dist(Z, protos):
    Zx = normalize_vec_rows_np(Z)
    Px = normalize_vec_rows_np(protos)
    sims = Zx @ Px.T
    order = np.argsort(-sims, axis=1)
    y1 = order[:, 0]
    s1 = sims[np.arange(len(Zx)), y1]
    if sims.shape[1] > 1:
        s2 = sims[np.arange(len(Zx)), order[:, 1]]
    else:
        s2 = np.zeros_like(s1)
    margin = (s1 - s2).astype(np.float32)
    dist = (1.0 - s1).astype(np.float32)
    return y1.astype(np.int64), margin, dist


def refine_probs_multi_stage(Z, P_seed, protos, idx_support=None, iters=2, src_mix=0.75):
    Zx = normalize_vec_rows_np(Z)
    P = normalize_rows(P_seed)
    Psrc = normalize_vec_rows_np(protos)
    if idx_support is None or len(idx_support) == 0:
        idx_support = np.arange(len(Zx))
    idx_support = np.asarray(idx_support, dtype=np.int64)
    for _ in range(max(1, int(iters))):
        Ps = normalize_rows(P[idx_support])
        Zs = Zx[idx_support]
        mass = Ps.sum(axis=0, keepdims=False)[:, None]
        proto_t = Psrc.copy()
        valid = mass.squeeze(1) > 1e-6
        if np.any(valid):
            proto_est = (Ps.T @ Zs) / np.maximum(mass, 1e-6)
            proto_t[valid] = normalize_vec_rows_np(src_mix * Psrc[valid] + (1.0 - src_mix) * proto_est[valid])
        P_proto = proto_probs_cosine(Zx, proto_t, temp=PROTO_T, l2norm=False)
        P = normalize_rows(0.60 * P + 0.40 * P_proto)
    return P.astype(np.float32)


def split_common_unknown_candidates(Z, P_logit, P_proto, P_knn, gate_pred, protos, tau_conf, min_keep=64):
    idx_gate = np.where(gate_pred == 1)[0]
    common_score = np.zeros((len(gate_pred)), dtype=np.float32)
    if len(idx_gate) == 0:
        return idx_gate, idx_gate, common_score, dict(threshold=float("nan"), gate_size=0)

    P_tri = weighted_prob_fusion([P_logit, P_proto, P_knn], [W_LOGIT, W_PROTO, W_KNN])
    y_logit = np.argmax(P_logit, axis=1)
    y_proto = np.argmax(P_proto, axis=1)
    y_knn = np.argmax(P_knn, axis=1)
    conf_tri = msp_from_probs(P_tri)
    margin_tri = top2_margin(P_tri)
    _, proto_margin, proto_dist = prototype_margin_dist(Z, protos)

    agree = ((y_logit == y_proto).astype(np.float32) + (y_logit == y_knn).astype(np.float32) + (y_proto == y_knn).astype(np.float32)) / 3.0
    margin_tri_n = robust_unit_interval(margin_tri, idx_gate)
    proto_margin_n = robust_unit_interval(proto_margin, idx_gate)
    proto_dist_n = robust_unit_interval(proto_dist, idx_gate)
    conf_n = robust_unit_interval(conf_tri, idx_gate)

    common_score = (
        0.40 * conf_n +
        0.25 * agree +
        0.20 * margin_tri_n +
        0.15 * proto_margin_n -
        0.15 * proto_dist_n
    ).astype(np.float32)

    thr_q = float(np.quantile(common_score[idx_gate], UNKNOWN_SCORE_Q))
    mask_common = (common_score >= thr_q) & (conf_tri >= min(float(tau_conf), float(np.max(conf_tri[idx_gate]))))
    idx_common = select_idx_by_mask_with_fallback(idx_gate, mask_common[idx_gate], common_score, min_keep=min_keep)
    idx_unknown = np.setdiff1d(idx_gate, idx_common, assume_unique=False)
    info = dict(threshold=thr_q, gate_size=int(len(idx_gate)), common_size=int(len(idx_common)), unknown_size=int(len(idx_unknown)))
    return idx_common.astype(np.int64), idx_unknown.astype(np.int64), common_score, info


def soft_ce_from_probs(logits, q):
    return -(q * F.log_softmax(logits, dim=1)).sum(dim=1)


def entropy_from_logits_torch(logits):
    p = torch.softmax(logits, dim=1)
    return -(p * torch.log(p + 1e-12)).sum(dim=1)


def symmetric_kl_from_logits(logits_a, logits_b):
    pa = torch.softmax(logits_a, dim=1)
    pb = torch.softmax(logits_b, dim=1)
    log_pa = torch.log(pa + 1e-12)
    log_pb = torch.log(pb + 1e-12)
    kl_ab = (pa * (log_pa - log_pb)).sum(dim=1)
    kl_ba = (pb * (log_pb - log_pa)).sum(dim=1)
    return 0.5 * (kl_ab + kl_ba)


def unknown_regularization_from_logits(logits_u, mode="uniform_kl", energy_margin=0.10):
    mode = str(mode).lower()
    if mode in {"uniform_kl", "uniform-kl", "kl"}:
        uniform = torch.full_like(logits_u, 1.0 / logits_u.shape[1])
        return F.kl_div(F.log_softmax(logits_u, dim=1), uniform, reduction="batchmean")
    if mode in {"energy_margin", "energy-margin", "energy"}:
        target_energy = float(-np.log(max(2, logits_u.shape[1])) + float(energy_margin))
        energy_u = -torch.logsumexp(logits_u, dim=1)
        target_t = torch.tensor(target_energy, dtype=energy_u.dtype, device=energy_u.device)
        return F.relu(target_t - energy_u).mean()
    raise ValueError(f"Unsupported unknown loss mode: {mode}")


def pairwise_sq_dists(x, y):
    x2 = (x * x).sum(dim=1, keepdim=True)
    y2 = (y * y).sum(dim=1, keepdim=True).T
    return torch.clamp(x2 + y2 - 2.0 * (x @ y.T), min=0.0)


def mmd_rbf(x, y, sigmas=None):
    if sigmas is None:
        sigmas = MMD_SIGMAS
    d_xx = pairwise_sq_dists(x, x)
    d_yy = pairwise_sq_dists(y, y)
    d_xy = pairwise_sq_dists(x, y)
    K_xx = 0.0
    K_yy = 0.0
    K_xy = 0.0
    for s in sigmas:
        gamma = 1.0 / max(2.0 * float(s) * float(s), 1e-12)
        K_xx = K_xx + torch.exp(-gamma * d_xx)
        K_yy = K_yy + torch.exp(-gamma * d_yy)
        K_xy = K_xy + torch.exp(-gamma * d_xy)
    return K_xx.mean() + K_yy.mean() - 2.0 * K_xy.mean()



def embedding_jitter(z, noise_std=EMB_AUG_NOISE):
    if noise_std <= 0:
        return z
    if z is None or z.numel() == 0:
        return z
    if z.ndim == 1:
        z = z.unsqueeze(0)
    if z.size(0) <= 1:
        scale = z.new_tensor(float(V14_SAFE_STD_FALLBACK))
    else:
        scale = z.detach().std(dim=0, keepdim=True, unbiased=False).mean().clamp(min=1e-4)
    return z + noise_std * scale * torch.randn_like(z)



def proto_targets_torch(protos, y=None, q=None, device=device):
    P = torch.tensor(protos, dtype=torch.float32, device=device)
    P = F.normalize(P, dim=1)
    if y is not None:
        return P[y]
    if q is not None:
        q_t = torch.as_tensor(q, dtype=torch.float32, device=device)
        return q_t @ P
    raise ValueError("Either y or q must be provided for proto_targets_torch.")


def cycle_loader(dl):
    while True:
        for batch in dl:
            yield batch


def normalized_weighted_mean_torch(v, w=None):
    if w is None:
        return v.mean()
    w = torch.clamp(w, min=0.0)
    ws = w.sum()
    if float(ws.detach().cpu()) <= 1e-12:
        return v.mean() * 0.0
    return (v * (w / (ws + 1e-12))).sum()


def energy_from_logits_torch(logits):
    return -torch.logsumexp(logits, dim=1)


def weighted_quantile_np(x, q=0.5):
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    if len(x) == 0:
        return float("nan")
    q = float(np.clip(q, 0.0, 1.0))
    return float(np.quantile(x, q))


def adapt_on_embeddings_generic(
    fc_layer,
    Z_replay,
    y_replay,
    Z_sup=None,
    y_sup=None,
    q_sup=None,
    w_sup=None,
    Z_align=None,
    Z_unrel=None,
    proto_bank=None,
    bottleneck=64,
    scale=0.3,
    epochs=20,
    lr=1e-3,
    wd=1e-4,
    batch=128,
    lambda_replay_ce=1.0,
    lambda_replay_anchor=1.0,
    lambda_sup=1.0,
    lambda_align=0.0,
    lambda_ent_min=0.0,
    lambda_ent_max=0.0,
    lambda_cons=0.0,
    lambda_proto=0.0,
    dynamic_align=False,
    init_state=None,
    unknown_loss_mode=UNKNOWN_LOSS_DEFAULT,
    unknown_energy_margin=UNKNOWN_ENERGY_MARGIN):
    has_sup = (Z_sup is not None) and (len(Z_sup) > 0)
    has_align = (Z_align is not None) and (len(Z_align) > 0)
    has_unrel = (Z_unrel is not None) and (len(Z_unrel) > 0)
    if (not has_sup) and (not has_align) and (not has_unrel):
        return None

    adapter = ResidualAdapter(dim=Z_replay.shape[1], bottleneck=bottleneck, scale=scale).to(device)
    if init_state is not None:
        adapter.load_state_dict(init_state)

    fc = nn.Linear(fc_layer.in_features, fc_layer.out_features, bias=True).to(device)
    fc.load_state_dict(fc_layer.state_dict())
    fc.eval()
    for p in fc.parameters():
        p.requires_grad = False

    ds_s = EmbDatasetWeightedHard(Z_replay, y_replay, None)
    dl_s = DataLoader(ds_s, batch_size=batch, shuffle=True, drop_last=False)
    it_s = cycle_loader(dl_s)

    dl_sup = None
    if has_sup:
        if y_sup is not None:
            ds_sup = EmbDatasetWeightedHard(Z_sup, y_sup, w_sup)
        else:
            ds_sup = EmbDatasetWeightedSoft(Z_sup, q_sup, w_sup)
        dl_sup = DataLoader(ds_sup, batch_size=batch, shuffle=True, drop_last=False)
        it_sup = cycle_loader(dl_sup)

    dl_align = None
    if has_align:
        ds_align = EmbOnlyDataset(Z_align)
        dl_align = DataLoader(ds_align, batch_size=batch, shuffle=True, drop_last=False)
        it_align = cycle_loader(dl_align)

    dl_unrel = None
    if has_unrel:
        ds_unrel = EmbOnlyDataset(Z_unrel)
        dl_unrel = DataLoader(ds_unrel, batch_size=batch, shuffle=True, drop_last=False)
        it_unrel = cycle_loader(dl_unrel)

    opt = optim.Adam(adapter.parameters(), lr=lr, weight_decay=wd)
    ce_none = nn.CrossEntropyLoss(reduction="none")
    ce = nn.CrossEntropyLoss()
    mse = nn.MSELoss()

    steps_per_epoch = max(
        len(dl_s),
        len(dl_sup) if dl_sup is not None else 0,
        len(dl_align) if dl_align is not None else 0,
        len(dl_unrel) if dl_unrel is not None else 0,
        1)

    for ep in range(max(1, int(epochs))):
        adapter.train()
        lam_align_ep = float(lambda_align) * ((ep + 1) / max(1, int(epochs))) if dynamic_align else float(lambda_align)
        for _ in range(steps_per_epoch):
            zs, ys, _ = next(it_s)
            zs, ys = zs.to(device), ys.to(device)

            opt.zero_grad()
            zs2 = adapter(zs)
            logits_s = fc(zs2)
            loss = lambda_replay_ce * ce(logits_s, ys) + lambda_replay_anchor * mse(zs2, zs)

            if has_sup:
                batch_sup = next(it_sup)
                if y_sup is not None:
                    zsup, yb, wb = batch_sup
                    zsup, yb, wb = zsup.to(device), yb.to(device), wb.to(device)
                    zsup2 = adapter(zsup)
                    logits_sup = fc(zsup2)
                    loss = loss + lambda_sup * (ce_none(logits_sup, yb) * wb).mean()
                    if lambda_proto > 0 and proto_bank is not None:
                        tgt = proto_targets_torch(proto_bank, y=yb, device=device)
                        loss = loss + lambda_proto * ((F.normalize(zsup2, dim=1) - tgt) ** 2).mean()
                else:
                    zsup, qb, wb = batch_sup
                    zsup, qb, wb = zsup.to(device), qb.to(device), wb.to(device)
                    zsup2 = adapter(zsup)
                    logits_sup = fc(zsup2)
                    loss = loss + lambda_sup * (soft_ce_from_probs(logits_sup, qb) * wb).mean()
                    if lambda_proto > 0 and proto_bank is not None:
                        tgt = proto_targets_torch(proto_bank, q=qb, device=device)
                        loss = loss + lambda_proto * ((F.normalize(zsup2, dim=1) - tgt) ** 2).mean()

            if has_align:
                zt = next(it_align).to(device)
                zt2 = adapter(zt)
                logits_t = fc(zt2)
                if lam_align_ep > 0:
                    loss = loss + lam_align_ep * mmd_rbf(zs2, zt2)
                if lambda_ent_min > 0:
                    loss = loss + lambda_ent_min * entropy_from_logits_torch(logits_t).mean()
                if lambda_cons > 0:
                    logits_aug = fc(adapter(embedding_jitter(zt)))
                    loss = loss + lambda_cons * symmetric_kl_from_logits(logits_t, logits_aug).mean()

            if has_unrel and lambda_ent_max > 0:
                zu = next(it_unrel).to(device)
                logits_u = fc(adapter(zu))
                loss = loss + lambda_ent_max * unknown_regularization_from_logits(
                    logits_u,
                    mode=unknown_loss_mode,
                    energy_margin=unknown_energy_margin)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(adapter.parameters(), ADAPT_GRAD_CLIP)
            opt.step()
    return adapter


def _adapter_eval_stats(adapter, fc_layer, Z_eval, y_eval=None, batch=512):
    if adapter is None or Z_eval is None or len(Z_eval) == 0:
        return dict(valid=False, replay_acc=float("nan"), replay_ce=float("nan"), unique_frac=float("nan"), dom_share=float("nan"), mean_conf=float("nan"))
    Z2, logits = apply_adapter_and_fc(adapter, fc_layer, Z_eval, batch=batch)
    probs = softmax_np(logits)
    pred = np.argmax(probs, axis=1)
    dom_share = float(np.max(np.bincount(pred, minlength=probs.shape[1])) / max(1, len(pred)))
    unique_frac = float(len(np.unique(pred)) / max(1, probs.shape[1]))
    mean_conf = float(msp_from_probs(probs).mean()) if len(probs) > 0 else float("nan")
    replay_acc = float(np.mean(pred == np.asarray(y_eval, dtype=np.int64))) if y_eval is not None else float("nan")
    if y_eval is not None:
        logits_t = torch.tensor(logits, dtype=torch.float32)
        y_t = torch.tensor(np.asarray(y_eval, dtype=np.int64), dtype=torch.long)
        replay_ce = float(F.cross_entropy(logits_t, y_t).item())
    else:
        replay_ce = float("nan")
    valid = np.isfinite(mean_conf) and np.isfinite(dom_share) and np.isfinite(unique_frac)
    if y_eval is not None:
        valid = valid and np.isfinite(replay_acc) and np.isfinite(replay_ce)
    return dict(valid=bool(valid), replay_acc=replay_acc, replay_ce=replay_ce, unique_frac=unique_frac, dom_share=dom_share, mean_conf=mean_conf)


def guard_adapter_with_fallback(adapter, fallback_adapter, fc_layer, Z_replay, y_replay, Z_probe=None, probe_name="adapter"):
    cand = _adapter_eval_stats(adapter, fc_layer, Z_replay, y_replay)
    ref = _adapter_eval_stats(fallback_adapter, fc_layer, Z_replay, y_replay) if fallback_adapter is not None else dict(valid=False, replay_acc=float("nan"), replay_ce=float("nan"))
    probe = _adapter_eval_stats(adapter, fc_layer, Z_probe, None) if (Z_probe is not None and len(Z_probe) > 0) else dict(valid=True, unique_frac=1.0, dom_share=0.0, mean_conf=0.0)

    collapse = (not cand.get("valid", False))
    if not collapse and ref.get("valid", False):
        collapse = collapse or (cand["replay_acc"] < ref["replay_acc"] - SAFE_REPLAY_ACC_DROP)
        collapse = collapse or (cand["replay_ce"] > ref["replay_ce"] * SAFE_REPLAY_CE_SCALE)
    collapse = collapse or (probe.get("unique_frac", 1.0) < SAFE_MIN_UNIQUE_CLASS_FRAC and probe.get("dom_share", 0.0) > SAFE_MAX_DOM_CLASS_SHARE)
    collapse = collapse or (probe.get("mean_conf", 0.0) > SAFE_MAX_MEAN_CONF and probe.get("dom_share", 0.0) > SAFE_MAX_DOM_CLASS_SHARE)

    info = dict(
        guard_reverted=bool(collapse and (fallback_adapter is not None)),
        guard_reason=(f"{probe_name}_collapse" if collapse else "ok"),
        guard_replay_acc=float(cand.get("replay_acc", float("nan"))),
        guard_replay_ce=float(cand.get("replay_ce", float("nan"))),
        guard_probe_unique_frac=float(probe.get("unique_frac", float("nan"))),
        guard_probe_dom_share=float(probe.get("dom_share", float("nan"))),
        guard_probe_mean_conf=float(probe.get("mean_conf", float("nan"))))
    if collapse and (fallback_adapter is not None):
        return fallback_adapter, info
    return adapter, info


def adapt_on_embeddings_hard(fc_layer, Z_target, y_target, Z_replay, y_replay,
                             target_weights=None,
                             bottleneck=64, scale=0.3, epochs=20, lr=1e-3, wd=1e-4, batch=128,
                             lambda_replay_ce=1.0, lambda_replay_anchor=1.0):
    if len(Z_target) == 0:
        return None
    return adapt_on_embeddings_generic(
        fc_layer=fc_layer,
        Z_replay=Z_replay, y_replay=y_replay,
        Z_sup=Z_target, y_sup=y_target, w_sup=target_weights,
        bottleneck=bottleneck, scale=scale, epochs=epochs, lr=lr, wd=wd, batch=batch,
        lambda_replay_ce=lambda_replay_ce, lambda_replay_anchor=lambda_replay_anchor)


def adapt_on_embeddings_soft(fc_layer, Z_target, q_target, Z_replay, y_replay,
                             target_weights=None,
                             bottleneck=64, scale=0.3, epochs=20, lr=1e-3, wd=1e-4, batch=128,
                             lambda_replay_ce=1.0, lambda_replay_anchor=1.0):
    if len(Z_target) == 0:
        return None
    return adapt_on_embeddings_generic(
        fc_layer=fc_layer,
        Z_replay=Z_replay, y_replay=y_replay,
        Z_sup=Z_target, q_sup=q_target, w_sup=target_weights,
        bottleneck=bottleneck, scale=scale, epochs=epochs, lr=lr, wd=wd, batch=batch,
        lambda_replay_ce=lambda_replay_ce, lambda_replay_anchor=lambda_replay_anchor)

def adapt_on_embeddings_weighted_open_minimal(
    fc_layer,
    Z_replay,
    y_replay,
    Z_sup=None,
    q_sup=None,
    w_sup=None,
    Z_stab=None,
    w_stab=None,
    Z_unk=None,
    w_unk=None,
    proto_bank=None,
    bottleneck=64,
    scale=0.3,
    epochs=20,
    lr=1e-3,
    wd=1e-4,
    batch=128,
    lambda_replay_ce=1.0,
    lambda_replay_anchor=1.0,
    lambda_sup=1.0,
    lambda_proto=0.0,
    lambda_stab=0.0,
    lambda_unkE=0.0,
    unknown_energy_margin=UNKNOWN_ENERGY_MARGIN,
    energy_bg_q=V22_ENERGY_BG_Q,
    energy_bg_ema=V22_ENERGY_BG_EMA):
    has_sup = (Z_sup is not None) and (len(Z_sup) > 0)
    has_stab = (Z_stab is not None) and (len(Z_stab) > 0)
    has_unk = (Z_unk is not None) and (len(Z_unk) > 0)
    if (not has_sup) and (not has_stab) and (not has_unk):
        return None

    adapter = ResidualAdapter(dim=Z_replay.shape[1], bottleneck=bottleneck, scale=scale).to(device)
    fc = nn.Linear(fc_layer.in_features, fc_layer.out_features, bias=True).to(device)
    fc.load_state_dict(fc_layer.state_dict())
    fc.eval()
    for p in fc.parameters():
        p.requires_grad = False

    ds_s = EmbDatasetWeightedHard(Z_replay, y_replay, None)
    dl_s = DataLoader(ds_s, batch_size=batch, shuffle=True, drop_last=False)
    it_s = cycle_loader(dl_s)

    dl_sup = None
    if has_sup:
        ds_sup = EmbDatasetWeightedSoft(Z_sup, q_sup, w_sup)
        dl_sup = DataLoader(ds_sup, batch_size=batch, shuffle=True, drop_last=False)
        it_sup = cycle_loader(dl_sup)

    dl_stab = None
    if has_stab:
        ds_stab = EmbOnlyDatasetWeighted(Z_stab, w_stab)
        dl_stab = DataLoader(ds_stab, batch_size=batch, shuffle=True, drop_last=False)
        it_stab = cycle_loader(dl_stab)

    dl_unk = None
    if has_unk:
        ds_unk = EmbOnlyDatasetWeighted(Z_unk, w_unk)
        dl_unk = DataLoader(ds_unk, batch_size=batch, shuffle=True, drop_last=False)
        it_unk = cycle_loader(dl_unk)

    opt = optim.Adam(adapter.parameters(), lr=lr, weight_decay=wd)
    ce = nn.CrossEntropyLoss()
    mse = nn.MSELoss()

    steps_per_epoch = max(
        len(dl_s),
        len(dl_sup) if dl_sup is not None else 0,
        len(dl_stab) if dl_stab is not None else 0,
        len(dl_unk) if dl_unk is not None else 0,
        1)

    with torch.no_grad():
        z0 = torch.tensor(Z_replay[:min(len(Z_replay), 1024)], dtype=torch.float32, device=device)
        e0 = energy_from_logits_torch(fc(z0))
        ema_bg_energy = float(torch.quantile(e0, float(np.clip(energy_bg_q, 0.0, 1.0))).item()) if e0.numel() > 0 else float(-np.log(max(2, fc.out_features)))

    for ep in range(max(1, int(epochs))):
        adapter.train()
        for _ in range(steps_per_epoch):
            zs, ys, _ = next(it_s)
            zs, ys = zs.to(device), ys.to(device)

            opt.zero_grad()
            zs2 = adapter(zs)
            logits_s = fc(zs2)
            loss = lambda_replay_ce * ce(logits_s, ys) + lambda_replay_anchor * mse(zs2, zs)

            bg_energy_chunks = [energy_from_logits_torch(logits_s).detach()]

            if has_sup:
                zsup, qb, wb = next(it_sup)
                zsup, qb, wb = zsup.to(device), qb.to(device), wb.to(device)
                zsup2 = adapter(zsup)
                logits_sup = fc(zsup2)
                loss = loss + lambda_sup * normalized_weighted_mean_torch(soft_ce_from_probs(logits_sup, qb), wb)
                if lambda_proto > 0 and proto_bank is not None:
                    tgt = proto_targets_torch(proto_bank, q=qb, device=device)
                    proto_err = ((F.normalize(zsup2, dim=1) - tgt) ** 2).mean(dim=1)
                    loss = loss + lambda_proto * normalized_weighted_mean_torch(proto_err, wb)

            if has_stab and lambda_stab > 0:
                zstab, wsb = next(it_stab)
                zstab, wsb = zstab.to(device), wsb.to(device)
                zstab2 = adapter(zstab)
                logits_stab = fc(zstab2)
                resid_sq = ((zstab2 - zstab) ** 2).mean(dim=1)
                loss = loss + lambda_stab * normalized_weighted_mean_torch(resid_sq, wsb)
                bg_energy_chunks.append(energy_from_logits_torch(logits_stab).detach())

            with torch.no_grad():
                bg_vec = torch.cat([v.reshape(-1) for v in bg_energy_chunks if v is not None and v.numel() > 0], dim=0)
                if bg_vec.numel() > 0:
                    bg_now = float(torch.quantile(bg_vec, float(np.clip(energy_bg_q, 0.0, 1.0))).item())
                    ema_bg_energy = float(energy_bg_ema * ema_bg_energy + (1.0 - energy_bg_ema) * bg_now)

            if has_unk and lambda_unkE > 0:
                zunk, wub = next(it_unk)
                zunk, wub = zunk.to(device), wub.to(device)
                logits_u = fc(adapter(zunk))
                energy_u = energy_from_logits_torch(logits_u)
                target_t = torch.tensor(float(ema_bg_energy + float(unknown_energy_margin)), dtype=energy_u.dtype, device=energy_u.device)
                loss = loss + lambda_unkE * normalized_weighted_mean_torch(F.relu(target_t - energy_u), wub)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(adapter.parameters(), ADAPT_GRAD_CLIP)
            opt.step()
    return adapter


def fit_single_adapter_from_spec(
    spec,
    fc_layer,
    Z_replay,
    y_replay,
    Z_adapt,
    protos,
    sc_adapt=None):
    extra_info = {}
    train_mode = spec["train"]

    if train_mode == "hard":
        idx = spec["idx"]
        adapter = adapt_on_embeddings_hard(
            fc_layer, Z_adapt[idx], spec["y"], Z_replay, y_replay, target_weights=spec["w"],
            bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE, epochs=ADAPT_EPOCHS,
            lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
            lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR
        )
        return adapter, extra_info

    if train_mode == "soft":
        idx = spec["idx"]
        adapter = adapt_on_embeddings_soft(
            fc_layer, Z_adapt[idx], spec["q"], Z_replay, y_replay, target_weights=spec["w"],
            bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE, epochs=ADAPT_EPOCHS,
            lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
            lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR
        )
        return adapter, extra_info

    if train_mode == "generic":
        sup_idx = spec["sup_idx"]
        align_idx = spec.get("align_idx", np.zeros((0), dtype=np.int64))
        unrel_idx = spec.get("unrel_idx", np.zeros((0), dtype=np.int64))
        adapter = adapt_on_embeddings_generic(
            fc_layer=fc_layer,
            Z_replay=Z_replay, y_replay=y_replay,
            Z_sup=Z_adapt[sup_idx] if len(sup_idx) > 0 else None,
            y_sup=spec.get("y", None),
            q_sup=spec.get("q", None),
            w_sup=spec.get("w", None),
            Z_align=Z_adapt[align_idx] if len(align_idx) > 0 else None,
            Z_unrel=Z_adapt[unrel_idx] if len(unrel_idx) > 0 else None,
            proto_bank=protos,
            bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE, epochs=ADAPT_EPOCHS,
            lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
            lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR,
            lambda_sup=spec.get("lambda_sup", 1.0),
            lambda_align=spec.get("lambda_align", 0.0),
            lambda_ent_min=spec.get("lambda_ent_min", 0.0),
            lambda_ent_max=spec.get("lambda_ent_max", 0.0),
            lambda_cons=spec.get("lambda_cons", 0.0),
            lambda_proto=spec.get("lambda_proto", 0.0),
            dynamic_align=spec.get("dynamic_align", False),
            unknown_loss_mode=spec.get("unknown_loss_mode", UNKNOWN_LOSS_DEFAULT),
            unknown_energy_margin=spec.get("unknown_energy_margin", UNKNOWN_ENERGY_MARGIN))
        return adapter, extra_info

    if train_mode == "lq_soft":
        sup_idx = np.asarray(spec.get("sup_idx", np.zeros((0), dtype=np.int64)), dtype=np.int64)
        align_idx = np.asarray(spec.get("align_idx", np.zeros((0), dtype=np.int64)), dtype=np.int64)
        unrel_idx = np.asarray(spec.get("unrel_idx", np.zeros((0), dtype=np.int64)), dtype=np.int64)
        adapter = adapt_on_embeddings_generic(
            fc_layer=fc_layer,
            Z_replay=Z_replay, y_replay=y_replay,
            Z_sup=Z_adapt[sup_idx] if len(sup_idx) > 0 else None,
            q_sup=spec["q"] if len(sup_idx) > 0 else None,
            w_sup=spec.get("w", None),
            Z_align=Z_adapt[align_idx] if len(align_idx) > 0 else None,
            Z_unrel=Z_adapt[unrel_idx] if len(unrel_idx) > 0 else None,
            proto_bank=protos,
            bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE, epochs=ADAPT_EPOCHS,
            lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
            lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR,
            lambda_sup=spec.get("lambda_sup", 1.0),
            lambda_align=spec.get("lambda_align", 0.0),
            lambda_ent_min=spec.get("lambda_ent_min", 0.0),
            lambda_ent_max=spec.get("lambda_ent_max", 0.0),
            lambda_cons=spec.get("lambda_cons", 0.0),
            lambda_proto=spec.get("lambda_proto", 0.0),
            dynamic_align=spec.get("dynamic_align", True),
            unknown_loss_mode=spec.get("unknown_loss_mode", UNKNOWN_LOSS_DEFAULT),
            unknown_energy_margin=spec.get("unknown_energy_margin", UNKNOWN_ENERGY_MARGIN))
        return adapter, extra_info

    if train_mode == "weighted_open_minimal":
        sup_idx = np.asarray(spec.get("sup_idx", np.zeros((0), dtype=np.int64)), dtype=np.int64)
        stab_idx = np.asarray(spec.get("stab_idx", np.zeros((0), dtype=np.int64)), dtype=np.int64)
        unk_idx = np.asarray(spec.get("unk_idx", np.zeros((0), dtype=np.int64)), dtype=np.int64)
        adapter = adapt_on_embeddings_weighted_open_minimal(
            fc_layer=fc_layer,
            Z_replay=Z_replay, y_replay=y_replay,
            Z_sup=Z_adapt[sup_idx] if len(sup_idx) > 0 else None,
            q_sup=spec.get("q", None),
            w_sup=spec.get("w_sup", None),
            Z_stab=Z_adapt[stab_idx] if len(stab_idx) > 0 else None,
            w_stab=spec.get("w_stab", None),
            Z_unk=Z_adapt[unk_idx] if len(unk_idx) > 0 else None,
            w_unk=spec.get("w_unk", None),
            proto_bank=protos,
            bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE, epochs=ADAPT_EPOCHS,
            lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
            lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR,
            lambda_sup=spec.get("lambda_sup", V22_LAMBDA_SUP),
            lambda_proto=spec.get("lambda_proto", V22_LAMBDA_PROTO),
            lambda_stab=spec.get("lambda_stab", V22_LAMBDA_STAB),
            lambda_unkE=spec.get("lambda_unkE", V22_LAMBDA_UNKE),
            unknown_energy_margin=spec.get("unknown_energy_margin", V22_UNKNOWN_ENERGY_MARGIN),
            energy_bg_q=spec.get("energy_bg_q", V22_ENERGY_BG_Q),
            energy_bg_ema=spec.get("energy_bg_ema", V22_ENERGY_BG_EMA))
        return adapter, extra_info

    if train_mode == "dg_raincoat_versioned":
        adapter, dg_info = adapt_dg_raincoat_lite_versioned(
            fc_layer=fc_layer,
            Z_replay=Z_replay, y_replay=y_replay,
            Z_target=Z_adapt,
            idx_gate=spec["idx_gate"], idx_rel=spec["idx_rel"], idx_amb=spec["idx_amb"], idx_unk=spec["idx_unk"],
            q_seed=spec["q_seed"], w_rel=spec["w_rel"], w_amb=spec["w_amb"],
            protos=protos,
            Sid_adapt=None if sc_adapt is None else sc_adapt.get("Sid", None),
            Sdom_adapt=None if sc_adapt is None else sc_adapt.get("Sdom", None),
            bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE,
            epochs_stage1=DG_STAGE1_EPOCHS, epochs_stage2=DG_STAGE2_EPOCHS,
            lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
            lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR,
            lambda_align_stage1=spec.get("lambda_align_stage1", DG_LAMBDA_ALIGN_STAGE1),
            lambda_align_stage2=spec.get("lambda_align_stage2", DG_LAMBDA_ALIGN_STAGE2),
            lambda_ent_min=spec.get("lambda_ent_min", DG_LAMBDA_ENT_MIN),
            lambda_ent_max=spec.get("lambda_ent_max", DG_LAMBDA_ENT_MAX),
            lambda_cons=spec.get("lambda_cons", DG_LAMBDA_CONS),
            lambda_proto=spec.get("lambda_proto", DG_LAMBDA_PROTO),
            lambda_u_repel=spec.get("lambda_u_repel", DG_LAMBDA_UNKNOWN_REPEL),
            version=spec.get("version", "v10"),
            unknown_loss_mode=spec.get("unknown_loss_mode", UNKNOWN_LOSS_DEFAULT),
            unknown_energy_margin=spec.get("unknown_energy_margin", UNKNOWN_ENERGY_MARGIN))
        return adapter, {**spec.get("bucket_info", {}), **(dg_info or {})}

    if train_mode == "three_bucket_v8_versioned":
        adapter, v8_info = adapt_three_bucket_v8_versioned(
            fc_layer=fc_layer,
            Z_replay=Z_replay, y_replay=y_replay,
            Z_target=Z_adapt,
            idx_gate=spec["idx_gate"], idx_rel=spec["idx_rel"], idx_amb=spec["idx_amb"],
            idx_weak=spec["idx_weak"], idx_unk=spec["idx_unk"],
            q_seed=spec["q_seed"], w_rel=spec["w_rel"], w_amb=spec["w_amb"], w_weak=spec["w_weak"],
            proto_bank=protos,
            idx_weak_cold=spec.get("idx_weak_cold", None), w_weak_cold=spec.get("w_weak_cold", None),
            bucket_score=spec.get("bucket_score", None), unknown_score=spec.get("unknown_score", None),
            Sid_adapt=None if sc_adapt is None else sc_adapt.get("Sid", None),
            Sdom_adapt=None if sc_adapt is None else sc_adapt.get("Sdom", None),
            bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE,
            stage0_epochs=V8_STAGE0_EPOCHS, refresh_epochs=V8_REFRESH_EPOCHS, refresh_rounds=V8_REFRESH_ROUNDS,
            lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
            lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR,
            lambda_align=spec.get("lambda_align", V8_LAMBDA_ALIGN),
            lambda_ent_min=spec.get("lambda_ent_min", V8_LAMBDA_ENT_MIN),
            lambda_ent_max=spec.get("lambda_ent_max", V8_LAMBDA_ENT_MAX),
            lambda_cons=spec.get("lambda_cons", V8_LAMBDA_CONS),
            lambda_proto=spec.get("lambda_proto", V8_LAMBDA_PROTO),
            lambda_u_repel=spec.get("lambda_u_repel", V8_LAMBDA_UNKNOWN_REPEL),
            version=spec.get("version", "v10"),
            unknown_loss_mode=spec.get("unknown_loss_mode", UNKNOWN_LOSS_DEFAULT),
            unknown_energy_margin=spec.get("unknown_energy_margin", UNKNOWN_ENERGY_MARGIN))
        return adapter, {**spec.get("bucket_info", {}), **(v8_info or {})}

    raise ValueError(f"Unsupported train mode for tau sweep helper: {train_mode}")


def source_head_probs_from_embeddings(fc_layer, Z, noise_std=0.0, views=1, batch=512):
    if Z is None or len(Z) == 0:
        return np.zeros((0, fc_layer.out_features), dtype=np.float32)
    fc = nn.Linear(fc_layer.in_features, fc_layer.out_features, bias=True).to(device)
    fc.load_state_dict(fc_layer.state_dict())
    fc.eval()
    for p in fc.parameters():
        p.requires_grad = False
    outs = []
    with torch.no_grad():
        for _ in range(max(1, int(views))):
            chunks = []
            for i in range(0, len(Z), batch):
                zb = torch.tensor(Z[i:i+batch], dtype=torch.float32, device=device)
                if noise_std > 0:
                    zb = embedding_jitter(zb, noise_std=noise_std)
                logits = fc(zb)
                chunks.append(softmax_np(logits.detach().cpu().numpy()))
            outs.append(np.concatenate(chunks, axis=0))
    return normalize_rows(np.mean(outs, axis=0).astype(np.float32))


def sharpen_rows_by_temperature(P, temp):
    P = normalize_rows(P.astype(np.float32))
    temp = np.asarray(temp, dtype=np.float32)
    if temp.ndim == 0:
        temp = np.full((len(P)), float(temp), dtype=np.float32)
    temp = np.clip(temp, 1e-3, 10.0)[:, None]
    Q = np.power(np.clip(P, 1e-8, 1.0), 1.0 / temp)
    return normalize_rows(Q.astype(np.float32))


def mild_class_balance_calibration(P, strength=0.15):
    P = normalize_rows(P.astype(np.float32))
    strength = float(np.clip(strength, 0.0, 1.0))
    if strength <= 0:
        return P
    mass = np.clip(P.mean(axis=0), 1e-6, None)
    inv = 1.0 / np.sqrt(mass)
    inv = inv / np.mean(inv)
    P_bal = normalize_rows(P * inv[None, :])
    return normalize_rows((1.0 - strength) * P + strength * P_bal).astype(np.float32)


def _vote_fraction(*arrays):
    preds = [np.argmax(a, axis=1) for a in arrays if a is not None and len(a) > 0]
    if len(preds) == 0:
        return np.zeros((0), dtype=np.float32)
    M = np.stack(preds, axis=1)
    out = np.zeros((M.shape[0]), dtype=np.float32)
    for i in range(M.shape[0]):
        _, cnts = np.unique(M[i], return_counts=True)
        out[i] = float(cnts.max() / max(1, len(M[i])))
    return out


def gather_base_weights_from_spec(spec, n_target):
    w_full = np.ones((n_target), dtype=np.float32) * 0.25
    keys = [
        ("idx_rel", "w_rel", 1.00),
        ("idx_amb", "w_amb", 0.70),
        ("idx_weak", "w_weak", 0.55),
        ("idx_weak_cold", "w_weak_cold", 0.35)]
    for idx_key, w_key, default_val in keys:
        idx = np.asarray(spec.get(idx_key, np.zeros((0), dtype=np.int64)), dtype=np.int64)
        if len(idx) == 0:
            continue
        w = spec.get(w_key, None)
        if w is None or len(w) == 0:
            vals = np.ones((len(idx)), dtype=np.float32) * float(default_val)
        else:
            vals = np.asarray(w, dtype=np.float32)
            if len(vals) != len(idx):
                vals = np.resize(vals, len(idx)).astype(np.float32)
        w_full[idx] = vals
    return np.maximum(w_full, 1e-3).astype(np.float32)



def _v14_hard_counts_from_probs(P):
    if P is None or len(P) == 0:
        return np.zeros((0), dtype=np.int64), np.zeros((0), dtype=np.int64)
    y = np.argmax(P, axis=1).astype(np.int64)
    cnt = np.bincount(y, minlength=P.shape[1]).astype(np.int64)
    return y, cnt

def _v14_lq_support_ok(P_seed_sup, min_total=V14_LQ_MIN_TOTAL, min_per_class=V14_LQ_MIN_PER_CLASS):
    if P_seed_sup is None or len(P_seed_sup) == 0:
        return False, dict(v14_lq_reason="empty_support", v14_lq_support_size=0)
    y_seed, cnt = _v14_hard_counts_from_probs(P_seed_sup)
    represented = int(np.sum(cnt > 0))
    too_small = len(P_seed_sup) < int(min_total)
    too_thin = np.any((cnt > 0) & (cnt < int(min_per_class)))
    too_few_classes = bool(V14_LQ_REQUIRE_TWO_CLASSES and P_seed_sup.shape[1] >= 2 and represented < 2)
    ok = (not too_small) and (not too_thin) and (not too_few_classes)
    return ok, dict(
        v14_lq_support_size=int(len(P_seed_sup)),
        v14_lq_repr_classes=represented,
        v14_lq_class_counts=cnt.tolist(),
        v14_lq_reason=("ok" if ok else ("few_total" if too_small else ("thin_class" if too_thin else "single_class"))))

def build_label_quality_soft_targets_v14(fc_layer, Z_sup, q_seed_sup, protos, P_src_sup, P_proto_sup, P_knn_sup, base_weights, mode="dgr"):
    if Z_sup is None or len(Z_sup) == 0:
        return np.zeros((0, protos.shape[0]), dtype=np.float32), np.zeros((0), dtype=np.float32), dict(v14_lq_active=False, v14_lq_reason="empty_support")

    cfg = dict(LQ_METHOD_CFG[mode])
    P_seed = normalize_rows(q_seed_sup.astype(np.float32))
    P_src = normalize_rows(P_src_sup.astype(np.float32))
    P_proto = normalize_rows(P_proto_sup.astype(np.float32))
    P_knn = normalize_rows(P_knn_sup.astype(np.float32))

    support_ok, support_info = _v14_lq_support_ok(P_seed)
    if not support_ok:
        Q_fb = normalize_rows(0.72 * P_seed + 0.28 * weighted_prob_fusion([P_src, P_proto, P_knn], [1.0, 0.9, 0.8]))
        Q_fb = restrict_topk_probs(Q_fb, k=min(2, Q_fb.shape[1]), sharpen=1.00)
        w_fb = np.clip(np.asarray(base_weights, dtype=np.float32), cfg["low_weight_floor"], V14_LQ_MAX_WEIGHT)
        info = {**support_info, "v14_lq_active": False, "v14_lq_fallback": True}
        return Q_fb.astype(np.float32), w_fb.astype(np.float32), info

    aug_views = V14_LQ_AUG_VIEWS if len(Z_sup) > 1 else 1
    aug_noise = EMB_AUG_NOISE * V14_LQ_AUG_NOISE_SCALE if len(Z_sup) > 1 else 0.0
    P_aug = source_head_probs_from_embeddings(fc_layer, Z_sup, noise_std=aug_noise, views=aug_views)

    P_mix = weighted_prob_fusion([P_seed, P_src, P_proto, P_knn, P_aug], cfg["weights"])
    support_idx = np.arange(len(Z_sup), dtype=np.int64)
    P_ref = refine_probs_multi_stage(
        Z_sup, P_mix, protos, idx_support=support_idx,
        iters=max(1, min(2, int(cfg.get("refine_iters", 2)))),
        src_mix=min(0.82, max(0.65, float(cfg.get("src_mix", 0.72))))
    )
    P_cal = mild_class_balance_calibration(P_ref, strength=min(0.10, float(cfg.get("class_balance_strength", 0.15))))

    vote_frac = _vote_fraction(P_seed, P_src, P_proto, P_knn, P_aug, P_ref)
    seed_ref_agree = (np.argmax(P_seed, axis=1) == np.argmax(P_ref, axis=1)).astype(np.float32)
    src_proto_agree = (np.argmax(P_src, axis=1) == np.argmax(P_proto, axis=1)).astype(np.float32)
    conf_ref = msp_from_probs(P_ref)
    conf_seed = msp_from_probs(P_seed)
    rel = (0.34 * conf_ref + 0.22 * conf_seed + 0.22 * vote_frac + 0.14 * seed_ref_agree + 0.08 * src_proto_agree).astype(np.float32)
    rel = np.clip(rel, 0.0, 1.0)

    high_mask = (conf_seed >= V14_LQ_HIGH_CONF) & (rel >= V14_LQ_FREEZE_REL)
    mid_mask = (~high_mask) & (conf_seed >= V14_LQ_LOW_CONF)
    low_mask = ~(high_mask | mid_mask)

    Q = P_cal.copy()
    if np.any(high_mask):
        Q[high_mask] = restrict_topk_probs(P_seed[high_mask], k=1, sharpen=1.00)
    if np.any(mid_mask):
        Q[mid_mask] = normalize_rows(
            V14_LQ_SEED_BLEND_MID * P_seed[mid_mask] + (1.0 - V14_LQ_SEED_BLEND_MID) * P_cal[mid_mask]
        )
        Q[mid_mask] = restrict_topk_probs(Q[mid_mask], k=min(2, Q.shape[1]), sharpen=1.01)
    if np.any(low_mask):
        Q[low_mask] = normalize_rows(
            V14_LQ_SEED_BLEND_LOW * P_seed[low_mask] + (1.0 - V14_LQ_SEED_BLEND_LOW) * P_src[low_mask]
        )
        Q[low_mask] = restrict_topk_probs(Q[low_mask], k=min(3, Q.shape[1]), sharpen=1.00)

    w_ref = np.asarray(base_weights, dtype=np.float32) * (cfg["low_weight_floor"] + 0.35 * rel)
    w_ref = np.clip(w_ref, cfg["low_weight_floor"], V14_LQ_MAX_WEIGHT).astype(np.float32)

    info = {
        **support_info,
        "v14_lq_active": True,
        "v14_lq_fallback": False,
        "v14_freeze_ratio": float(np.mean(high_mask)),
        "v14_mid_ratio": float(np.mean(mid_mask)),
        "v14_low_ratio": float(np.mean(low_mask)),
        "v14_rel_mean": float(np.mean(rel)),
        "v14_conf_seed_mean": float(np.mean(conf_seed)),
        "v14_conf_ref_mean": float(np.mean(conf_ref)),
        "v14_vote_mean": float(np.mean(vote_frac)),
    }
    return Q.astype(np.float32), w_ref.astype(np.float32), info

def build_label_quality_method_from_base_v14(base_spec, mode, fc_layer, Z_adapt, P_logit_adapt, P_proto_adapt, P_knn_adapt, protos, base_name=None):
    if base_spec is None:
        return None, {}
    support_idx = np.asarray(base_spec.get("idx_sup_eval", base_spec.get("sup_idx", np.zeros((0), dtype=np.int64))), dtype=np.int64)
    fallback_name = base_name if base_name is not None else ("DG_RAINCOAT_v12" if mode == "dgr" else "ThreeBucketV8Adaptive_v12")
    if len(support_idx) == 0:
        spec = deepcopy(base_spec)
        spec.setdefault("bucket_info", {})
        spec["bucket_info"] = {**spec.get("bucket_info", {}), "v14_lq_active": False, "v14_lq_reason": "empty_support"}
        return spec, spec["bucket_info"]

    q_seed_full = normalize_rows(np.asarray(base_spec.get("q_seed", base_spec.get("P")), dtype=np.float32))
    w_full = gather_base_weights_from_spec(base_spec, len(Z_adapt))
    Q_ref, w_ref, info = build_label_quality_soft_targets_v14(
        fc_layer=fc_layer,
        Z_sup=Z_adapt[support_idx],
        q_seed_sup=q_seed_full[support_idx],
        protos=protos,
        P_src_sup=P_logit_adapt[support_idx],
        P_proto_sup=P_proto_adapt[support_idx],
        P_knn_sup=P_knn_adapt[support_idx],
        base_weights=w_full[support_idx],
        mode=mode)
    if not info.get("v14_lq_active", False):
        spec = deepcopy(base_spec)
        spec.setdefault("bucket_info", {})
        spec["bucket_info"] = {**spec.get("bucket_info", {}), **info}
        return spec, spec["bucket_info"]

    P_full = q_seed_full.copy()
    P_full[support_idx] = Q_ref
    idx_unk = np.asarray(base_spec.get("idx_unk", np.zeros((0), dtype=np.int64)), dtype=np.int64)
    idx_gate = np.asarray(base_spec.get("idx_gate", support_idx), dtype=np.int64)
    align_idx = np.setdiff1d(idx_gate, idx_unk, assume_unique=False)
    cfg = dict(LQ_METHOD_CFG[mode])
    spec = dict(
        train="lq_soft",
        sup_idx=support_idx,
        idx_sup_eval=support_idx,
        q=Q_ref,
        w=w_ref,
        align_idx=align_idx,
        unrel_idx=idx_unk,
        P=P_full,
        fallback_name=fallback_name,
        lambda_sup=1.0,
        lambda_align=cfg["lambda_align"] * 0.90,
        lambda_ent_min=cfg["lambda_ent_min"] * 0.80,
        lambda_ent_max=cfg["lambda_ent_max"] * 0.80,
        lambda_cons=cfg["lambda_cons"] * 0.95,
        lambda_proto=cfg["lambda_proto"] * 0.95,
        dynamic_align=True,
        bucket_info=info)
    return spec, info




def _v15_quality_support_ok(P_seed_sup, min_total=V15_QA_MIN_TOTAL, min_per_class=V15_QA_MIN_PER_CLASS):
    if P_seed_sup is None or len(P_seed_sup) == 0:
        return False, dict(v15_qw_reason="empty_support", v15_qw_support_size=0)
    _, cnt = _v14_hard_counts_from_probs(P_seed_sup)
    represented = int(np.sum(cnt > 0))
    too_small = len(P_seed_sup) < int(min_total)
    too_thin = np.any((cnt > 0) & (cnt < int(min_per_class)))
    too_few_classes = bool(V15_QA_REQUIRE_TWO_CLASSES and P_seed_sup.shape[1] >= 2 and represented < 2)
    ok = (not too_small) and (not too_thin) and (not too_few_classes)
    return ok, dict(
        v15_qw_support_size=int(len(P_seed_sup)),
        v15_qw_repr_classes=represented,
        v15_qw_class_counts=cnt.tolist(),
        v15_qw_reason=("ok" if ok else ("few_total" if too_small else ("thin_class" if too_thin else "single_class"))))


def _v15_row_margin(P):
    P = normalize_rows(P.astype(np.float32))
    if P.shape[1] <= 1:
        return np.ones((len(P)), dtype=np.float32)
    part = np.partition(P, kth=P.shape[1] - 2, axis=1)
    top1 = part[:, -1]
    top2 = part[:, -2]
    return np.clip(top1 - top2, 0.0, 1.0).astype(np.float32)


def build_quality_aware_soft_targets_v15(fc_layer, Z_sup, q_seed_sup, protos, P_src_sup, P_proto_sup, P_knn_sup, base_weights, mode="dgr"):
    if Z_sup is None or len(Z_sup) == 0:
        return np.zeros((0, protos.shape[0]), dtype=np.float32), np.zeros((0), dtype=np.float32), dict(v15_qw_active=False, v15_qw_reason="empty_support")

    cfg = dict(V15_QA_CFG[mode])
    P_seed = normalize_rows(q_seed_sup.astype(np.float32))
    P_src = normalize_rows(P_src_sup.astype(np.float32))
    P_proto = normalize_rows(P_proto_sup.astype(np.float32))
    P_knn = normalize_rows(P_knn_sup.astype(np.float32))

    support_ok, support_info = _v15_quality_support_ok(P_seed)
    if not support_ok:
        Q_fb = normalize_rows(0.70 * P_seed + 0.30 * weighted_prob_fusion([P_src, P_proto, P_knn], [0.9, 1.0, 0.9]))
        Q_fb = sharpen_rows_by_temperature(Q_fb, 0.95)
        Q_fb = restrict_topk_probs(Q_fb, k=min(2, Q_fb.shape[1]), sharpen=1.00)
        w_fb = np.clip(np.asarray(base_weights, dtype=np.float32), cfg["low_weight_floor"], V15_QA_MAX_WEIGHT)
        info = {**support_info, "v15_qw_active": False, "v15_qw_fallback": True}
        return Q_fb.astype(np.float32), w_fb.astype(np.float32), info

    aug_views = V15_QA_AUG_VIEWS if len(Z_sup) > 1 else 1
    aug_noise = EMB_AUG_NOISE * V15_QA_AUG_NOISE_SCALE if len(Z_sup) > 1 else 0.0
    P_aug = source_head_probs_from_embeddings(fc_layer, Z_sup, noise_std=aug_noise, views=aug_views)

    P_prior = weighted_prob_fusion([P_proto, P_knn, P_src], [1.0, 0.95, 0.85])
    P_mix = weighted_prob_fusion([P_seed, P_src, P_proto, P_knn, P_aug], cfg["weights"])
    support_idx = np.arange(len(Z_sup), dtype=np.int64)
    P_ref = refine_probs_multi_stage(
        Z_sup, P_mix, protos, idx_support=support_idx,
        iters=max(1, int(cfg.get("refine_iters", 2))),
        src_mix=float(cfg.get("src_mix", 0.72))
    )
    P_cal = mild_class_balance_calibration(P_ref, strength=float(cfg.get("class_balance_strength", 0.10)))

    conf_seed = msp_from_probs(P_seed)
    conf_ref = msp_from_probs(P_ref)
    margin_seed = _v15_row_margin(P_seed)
    margin_ref = _v15_row_margin(P_ref)
    vote_frac = _vote_fraction(P_seed, P_src, P_proto, P_knn, P_aug, P_ref)
    seed_ref_agree = (np.argmax(P_seed, axis=1) == np.argmax(P_ref, axis=1)).astype(np.float32)
    src_proto_agree = (np.argmax(P_src, axis=1) == np.argmax(P_proto, axis=1)).astype(np.float32)
    proto_peak = np.max(P_proto, axis=1).astype(np.float32)
    ent_ref = (-np.sum(P_ref * np.log(np.clip(P_ref, 1e-12, 1.0)), axis=1) / np.log(max(2, P_ref.shape[1]))).astype(np.float32)
    ent_score = np.clip(1.0 - ent_ref, 0.0, 1.0)

    quality = (
        0.24 * conf_ref +
        0.14 * conf_seed +
        0.14 * margin_ref +
        0.08 * margin_seed +
        0.14 * vote_frac +
        0.10 * ent_score +
        0.08 * seed_ref_agree +
        0.04 * src_proto_agree +
        0.04 * proto_peak
    ).astype(np.float32)
    quality = np.clip(quality, 0.0, 1.0)
    qpow = np.power(np.clip(quality, 1e-6, 1.0), float(cfg.get("gamma", 1.30))).astype(np.float32)

    seed_alpha = cfg["seed_min"] + (cfg["seed_max"] - cfg["seed_min"]) * quality
    ref_alpha = cfg["ref_min"] + (cfg["ref_max"] - cfg["ref_min"]) * quality
    prior_alpha = np.clip(1.0 - seed_alpha - ref_alpha, 0.08, 0.70)
    alpha_sum = np.clip(seed_alpha + ref_alpha + prior_alpha, 1e-6, None)
    seed_alpha = seed_alpha / alpha_sum
    ref_alpha = ref_alpha / alpha_sum
    prior_alpha = prior_alpha / alpha_sum

    Q = normalize_rows(seed_alpha[:, None] * P_seed + ref_alpha[:, None] * P_cal + prior_alpha[:, None] * P_prior)
    temp = cfg["temp_high"] - (cfg["temp_high"] - cfg["temp_low"]) * qpow
    Q = sharpen_rows_by_temperature(Q, temp)

    high_mask = quality >= float(cfg.get("high_rel_thr", 0.78))
    mid_mask = (quality >= float(cfg.get("mid_rel_thr", 0.55))) & (~high_mask)
    low_mask = ~(high_mask | mid_mask)
    if np.any(high_mask):
        Q[high_mask] = restrict_topk_probs(Q[high_mask], k=1, sharpen=1.00)
    if np.any(mid_mask):
        Q[mid_mask] = restrict_topk_probs(Q[mid_mask], k=min(2, Q.shape[1]), sharpen=1.02)
    if np.any(low_mask):
        Q[low_mask] = restrict_topk_probs(Q[low_mask], k=min(3, Q.shape[1]), sharpen=1.00)

    w_ref = np.asarray(base_weights, dtype=np.float32) * (cfg["low_weight_floor"] + cfg["weight_gain"] * qpow)
    w_ref = np.clip(w_ref, cfg["low_weight_floor"], V15_QA_MAX_WEIGHT).astype(np.float32)

    info = {
        **support_info,
        "v15_qw_active": True,
        "v15_qw_fallback": False,
        "v15_quality_mean": float(np.mean(quality)),
        "v15_quality_std": float(np.std(quality)),
        "v15_quality_p25": float(np.quantile(quality, 0.25)),
        "v15_quality_p75": float(np.quantile(quality, 0.75)),
        "v15_conf_seed_mean": float(np.mean(conf_seed)),
        "v15_conf_ref_mean": float(np.mean(conf_ref)),
        "v15_vote_mean": float(np.mean(vote_frac)),
        "v15_high_ratio": float(np.mean(high_mask)),
        "v15_mid_ratio": float(np.mean(mid_mask)),
        "v15_low_ratio": float(np.mean(low_mask)),
    }
    return Q.astype(np.float32), w_ref.astype(np.float32), info


def build_quality_aware_method_from_base_v15(base_spec, mode, fc_layer, Z_adapt, P_logit_adapt, P_proto_adapt, P_knn_adapt, protos, base_name=None):
    if base_spec is None:
        return None, {}
    support_idx = np.asarray(base_spec.get("idx_sup_eval", base_spec.get("sup_idx", np.zeros((0), dtype=np.int64))), dtype=np.int64)
    fallback_name = base_name if base_name is not None else ("DG_RAINCOAT_v14Stable" if mode == "dgr" else "ThreeBucketV8Adaptive_v14Stable")
    if len(support_idx) == 0:
        spec = deepcopy(base_spec)
        spec.setdefault("bucket_info", {})
        spec["bucket_info"] = {**spec.get("bucket_info", {}), "v15_qw_active": False, "v15_qw_reason": "empty_support"}
        return spec, spec["bucket_info"]

    q_seed_full = normalize_rows(np.asarray(base_spec.get("q_seed", base_spec.get("P")), dtype=np.float32))
    w_full = gather_base_weights_from_spec(base_spec, len(Z_adapt))
    Q_ref, w_ref, info = build_quality_aware_soft_targets_v15(
        fc_layer=fc_layer,
        Z_sup=Z_adapt[support_idx],
        q_seed_sup=q_seed_full[support_idx],
        protos=protos,
        P_src_sup=P_logit_adapt[support_idx],
        P_proto_sup=P_proto_adapt[support_idx],
        P_knn_sup=P_knn_adapt[support_idx],
        base_weights=w_full[support_idx],
        mode=mode)
    if not info.get("v15_qw_active", False):
        spec = deepcopy(base_spec)
        spec.setdefault("bucket_info", {})
        spec["bucket_info"] = {**spec.get("bucket_info", {}), **info}
        return spec, spec["bucket_info"]

    P_full = q_seed_full.copy()
    P_full[support_idx] = Q_ref
    idx_unk = np.asarray(base_spec.get("idx_unk", np.zeros((0), dtype=np.int64)), dtype=np.int64)
    idx_gate = np.asarray(base_spec.get("idx_gate", support_idx), dtype=np.int64)
    align_idx = np.setdiff1d(idx_gate, idx_unk, assume_unique=False)
    cfg = dict(V15_QA_CFG[mode])
    spec = dict(
        train="lq_soft",
        sup_idx=support_idx,
        idx_sup_eval=support_idx,
        q=Q_ref,
        w=w_ref,
        align_idx=align_idx,
        unrel_idx=idx_unk,
        P=P_full,
        fallback_name=fallback_name,
        lambda_sup=1.0,
        lambda_align=cfg["lambda_align"],
        lambda_ent_min=cfg["lambda_ent_min"],
        lambda_ent_max=cfg["lambda_ent_max"],
        lambda_cons=cfg["lambda_cons"],
        lambda_proto=cfg["lambda_proto"],
        dynamic_align=True,
        bucket_info=info)
    return spec, info



def _v16_safe_unit_scale(x, eps=1e-6):
    x = np.asarray(x, dtype=np.float32)
    if x.size == 0:
        return x.astype(np.float32)
    lo = float(np.min(x))
    hi = float(np.max(x))
    if (hi - lo) < eps:
        return np.full_like(x, 0.5, dtype=np.float32)
    return np.clip((x - lo) / (hi - lo + eps), 0.0, 1.0).astype(np.float32)


def _v16_support_ok(P_seed_sup, min_total=V16_QA_MIN_TOTAL, min_per_class=V16_QA_MIN_PER_CLASS):
    if P_seed_sup is None or len(P_seed_sup) == 0:
        return False, dict(v16_qw_reason="empty_support", v16_qw_support_size=0)
    _, cnt = _v14_hard_counts_from_probs(P_seed_sup)
    represented = int(np.sum(cnt > 0))
    too_small = len(P_seed_sup) < int(min_total)
    too_thin = np.any((cnt > 0) & (cnt < int(min_per_class)))
    too_few_classes = bool(V16_QA_REQUIRE_TWO_CLASSES and P_seed_sup.shape[1] >= 2 and represented < 2)
    ok = (not too_small) and (not too_thin) and (not too_few_classes)
    return ok, dict(
        v16_qw_support_size=int(len(P_seed_sup)),
        v16_qw_repr_classes=represented,
        v16_qw_class_counts=cnt.tolist(),
        v16_qw_reason=("ok" if ok else ("few_total" if too_small else ("thin_class" if too_thin else "single_class"))))


def build_quality_aware_soft_targets_v16(fc_layer, Z_sup, q_seed_sup, protos, P_src_sup, P_proto_sup, P_knn_sup, base_weights, mode="dgr"):
    if Z_sup is None or len(Z_sup) == 0:
        info = dict(v16_qw_active=False, v16_qw_reason="empty_support")
        aux = dict(knownness=np.zeros((0), dtype=np.float32), class_quality=np.zeros((0), dtype=np.float32), overall_quality=np.zeros((0), dtype=np.float32))
        return np.zeros((0, protos.shape[0]), dtype=np.float32), np.zeros((0), dtype=np.float32), info, aux

    cfg = dict(V16_QA_CFG[mode])
    P_seed = normalize_rows(q_seed_sup.astype(np.float32))
    P_src = normalize_rows(P_src_sup.astype(np.float32))
    P_proto = normalize_rows(P_proto_sup.astype(np.float32))
    P_knn = normalize_rows(P_knn_sup.astype(np.float32))

    support_ok, support_info = _v16_support_ok(P_seed)
    if not support_ok:
        Q_fb, w_fb, info_fb = build_quality_aware_soft_targets_v15(
            fc_layer=fc_layer,
            Z_sup=Z_sup,
            q_seed_sup=q_seed_sup,
            protos=protos,
            P_src_sup=P_src_sup,
            P_proto_sup=P_proto_sup,
            P_knn_sup=P_knn_sup,
            base_weights=base_weights,
            mode=mode)
        info = {**support_info, **{k: v for k, v in info_fb.items() if not k.startswith("v15_")}, "v16_qw_active": False, "v16_qw_fallback": True}
        aux = dict(
            knownness=np.zeros((len(Z_sup)), dtype=np.float32),
            class_quality=np.zeros((len(Z_sup)), dtype=np.float32),
            overall_quality=np.zeros((len(Z_sup)), dtype=np.float32))
        return Q_fb.astype(np.float32), w_fb.astype(np.float32), info, aux

    aug_views = V16_QA_AUG_VIEWS if len(Z_sup) > 1 else 1
    aug_noise = EMB_AUG_NOISE * V16_QA_AUG_NOISE_SCALE if len(Z_sup) > 1 else 0.0
    P_aug = source_head_probs_from_embeddings(fc_layer, Z_sup, noise_std=aug_noise, views=aug_views)

    P_prior = weighted_prob_fusion([P_proto, P_knn, P_src], [1.00, 0.95, 0.85])
    P_mix = weighted_prob_fusion([P_seed, P_src, P_proto, P_knn, P_aug], cfg["weights"])
    support_idx = np.arange(len(Z_sup), dtype=np.int64)
    P_ref = refine_probs_multi_stage(
        Z_sup, P_mix, protos, idx_support=support_idx,
        iters=max(1, int(cfg.get("refine_iters", 2))),
        src_mix=float(cfg.get("src_mix", 0.72))
    )
    P_cal = mild_class_balance_calibration(P_ref, strength=float(cfg.get("class_balance_strength", 0.08)))

    conf_seed = msp_from_probs(P_seed)
    conf_ref = msp_from_probs(P_ref)
    margin_seed = _v15_row_margin(P_seed)
    margin_ref = _v15_row_margin(P_ref)
    vote_frac = _vote_fraction(P_seed, P_src, P_proto, P_knn, P_aug, P_ref)
    seed_ref_agree = (np.argmax(P_seed, axis=1) == np.argmax(P_ref, axis=1)).astype(np.float32)
    src_proto_agree = (np.argmax(P_src, axis=1) == np.argmax(P_proto, axis=1)).astype(np.float32)
    src_knn_agree = (np.argmax(P_src, axis=1) == np.argmax(P_knn, axis=1)).astype(np.float32)
    proto_peak = np.max(P_proto, axis=1).astype(np.float32)
    prior_peak = np.max(P_prior, axis=1).astype(np.float32)
    ent_ref = (-np.sum(P_ref * np.log(np.clip(P_ref, 1e-12, 1.0)), axis=1) / np.log(max(2, P_ref.shape[1]))).astype(np.float32)
    ent_score = np.clip(1.0 - ent_ref, 0.0, 1.0)

    w_base = np.clip(np.asarray(base_weights, dtype=np.float32), 0.0, V16_QA_MAX_WEIGHT)
    w_base_unit = _v16_safe_unit_scale(w_base)

    knownness = (
        0.30 * w_base_unit +
        0.18 * conf_ref +
        0.12 * conf_seed +
        0.10 * margin_ref +
        0.10 * vote_frac +
        0.10 * ent_score +
        0.06 * seed_ref_agree +
        0.04 * prior_peak
    ).astype(np.float32)
    knownness = np.clip(knownness, 0.0, 1.0)

    class_quality = (
        0.26 * conf_ref +
        0.18 * margin_ref +
        0.16 * vote_frac +
        0.10 * seed_ref_agree +
        0.10 * src_proto_agree +
        0.08 * src_knn_agree +
        0.06 * proto_peak +
        0.06 * prior_peak
    ).astype(np.float32)
    class_quality = np.clip(class_quality, 0.0, 1.0)

    overall_quality = np.clip(0.55 * knownness + 0.45 * class_quality, 0.0, 1.0).astype(np.float32)
    known_pow = np.power(np.clip(knownness, 1e-6, 1.0), float(cfg.get("gamma_known", 1.20))).astype(np.float32)
    class_pow = np.power(np.clip(class_quality, 1e-6, 1.0), float(cfg.get("gamma_class", 1.10))).astype(np.float32)
    overall_pow = np.power(np.clip(overall_quality, 1e-6, 1.0), float(cfg.get("gamma_overall", 1.15))).astype(np.float32)

    seed_alpha = cfg["seed_min"] + (cfg["seed_max"] - cfg["seed_min"]) * class_pow
    ref_alpha = cfg["ref_min"] + (cfg["ref_max"] - cfg["ref_min"]) * overall_pow
    prior_alpha = cfg["prior_min"] + (cfg["prior_max"] - cfg["prior_min"]) * (1.0 - class_pow) * (0.35 + 0.65 * known_pow)
    uniform_alpha = cfg["uniform_max"] * np.power(1.0 - known_pow, 1.15)

    alpha_sum = np.clip(seed_alpha + ref_alpha + prior_alpha + uniform_alpha, 1e-6, None)
    seed_alpha = seed_alpha / alpha_sum
    ref_alpha = ref_alpha / alpha_sum
    prior_alpha = prior_alpha / alpha_sum
    uniform_alpha = uniform_alpha / alpha_sum

    U = np.full_like(P_seed, 1.0 / P_seed.shape[1], dtype=np.float32)
    Q = normalize_rows(
        seed_alpha[:, None] * P_seed +
        ref_alpha[:, None] * P_cal +
        prior_alpha[:, None] * P_prior +
        uniform_alpha[:, None] * U
    )

    temp = cfg["temp_high"] - (cfg["temp_high"] - cfg["temp_low"]) * (0.40 * known_pow + 0.60 * class_pow)
    Q = sharpen_rows_by_temperature(Q, temp)

    high_mask = (overall_quality >= float(cfg.get("high_rel_thr", 0.80))) & (knownness >= float(cfg.get("align_known_thr", 0.48)))
    mid_mask = (overall_quality >= float(cfg.get("mid_rel_thr", 0.54))) & (~high_mask)
    low_mask = ~(high_mask | mid_mask)

    if np.any(high_mask):
        Q[high_mask] = restrict_topk_probs(Q[high_mask], k=min(2, Q.shape[1]), sharpen=1.06)
    if np.any(mid_mask):
        Q[mid_mask] = restrict_topk_probs(Q[mid_mask], k=min(3, Q.shape[1]), sharpen=1.01)
    if np.any(low_mask):
        Q[low_mask] = normalize_rows(0.75 * Q[low_mask] + 0.25 * U[low_mask])

    w_scale_known = cfg["known_floor"] + cfg["known_gain"] * known_pow
    w_scale_class = cfg["class_floor"] + cfg["class_gain"] * class_pow
    w_ref = w_base * np.sqrt(np.clip(w_scale_known * w_scale_class, 1e-6, None))
    w_ref = np.maximum(w_ref, cfg["low_weight_floor"] + 0.5 * cfg["low_weight_floor"] * overall_pow)
    w_ref = np.clip(w_ref, cfg["low_weight_floor"], V16_QA_MAX_WEIGHT).astype(np.float32)

    info = {
        **support_info,
        "v16_qw_active": True,
        "v16_qw_fallback": False,
        "v16_knownness_mean": float(np.mean(knownness)),
        "v16_knownness_std": float(np.std(knownness)),
        "v16_class_quality_mean": float(np.mean(class_quality)),
        "v16_class_quality_std": float(np.std(class_quality)),
        "v16_overall_quality_mean": float(np.mean(overall_quality)),
        "v16_overall_quality_std": float(np.std(overall_quality)),
        "v16_high_ratio": float(np.mean(high_mask)),
        "v16_mid_ratio": float(np.mean(mid_mask)),
        "v16_low_ratio": float(np.mean(low_mask)),
        "v16_weight_mean": float(np.mean(w_ref)),
        "v16_weight_std": float(np.std(w_ref)),
        "v16_conf_ref_mean": float(np.mean(conf_ref)),
        "v16_vote_mean": float(np.mean(vote_frac)),
    }
    aux = dict(
        knownness=knownness.astype(np.float32),
        class_quality=class_quality.astype(np.float32),
        overall_quality=overall_quality.astype(np.float32))
    return Q.astype(np.float32), w_ref.astype(np.float32), info, aux




def _v17_local_consensus_stats(Z, P_ref, k=24, block=256, sim_floor=0.10):
    n = 0 if Z is None else int(len(Z))
    if n == 0:
        z = np.zeros((0), dtype=np.float32)
        return dict(local_agree=z, local_conf=z, local_label_prob=z, local_density=z)
    P = normalize_rows(np.asarray(P_ref, dtype=np.float32))
    conf = np.max(P, axis=1).astype(np.float32)
    if n == 1:
        one = np.ones((1), dtype=np.float32)
        return dict(local_agree=one, local_conf=conf.copy(), local_label_prob=conf.copy(), local_density=0.5 * one)

    Zx = normalize_vec_rows_np(np.asarray(Z, dtype=np.float32))
    y = np.argmax(P, axis=1).astype(np.int64)
    k_eff = int(min(max(1, int(k)), n - 1))
    block = int(max(32, block))
    sim_floor = float(np.clip(sim_floor, -0.5, 0.95))
    denom = max(1e-6, 1.0 - sim_floor)

    local_agree = np.zeros((n), dtype=np.float32)
    local_conf = np.zeros((n), dtype=np.float32)
    local_label_prob = np.zeros((n), dtype=np.float32)
    local_density = np.zeros((n), dtype=np.float32)

    for s in range(0, n, block):
        e = min(n, s + block)
        sims = (Zx[s:e] @ Zx.T).astype(np.float32)
        sims[np.arange(e - s), np.arange(s, e)] = -np.inf

        if k_eff < (n - 1):
            idx = np.argpartition(-sims, kth=k_eff - 1, axis=1)[:, :k_eff]
            sim_top = np.take_along_axis(sims, idx, axis=1)
        else:
            idx = np.argsort(-sims, axis=1)[:, :k_eff]
            sim_top = np.take_along_axis(sims, idx, axis=1)

        order = np.argsort(-sim_top, axis=1)
        idx = np.take_along_axis(idx, order, axis=1)
        sim_top = np.take_along_axis(sim_top, order, axis=1).astype(np.float32)

        w = np.clip((sim_top - sim_floor) / denom, 0.0, 1.0)
        wsum = np.sum(w, axis=1, keepdims=True)
        zero_rows = (wsum.squeeze(1) < 1e-6)
        if np.any(zero_rows):
            w[zero_rows] = 1.0
            wsum = np.sum(w, axis=1, keepdims=True)
        w = (w / np.clip(wsum, 1e-6, None)).astype(np.float32)

        y_self = y[s:e]
        local_agree[s:e] = np.sum(w * (y[idx] == y_self[:, None]).astype(np.float32), axis=1)
        local_conf[s:e] = np.sum(w * conf[idx], axis=1)

        flat_idx = idx.reshape(-1)
        flat_cls = np.repeat(y_self, k_eff)
        neigh_self_prob = P[flat_idx, flat_cls].reshape(e - s, k_eff).astype(np.float32)
        local_label_prob[s:e] = np.sum(w * neigh_self_prob, axis=1)
        local_density[s:e] = np.sum(w * np.clip(sim_top, -1.0, 1.0), axis=1)

    local_density = np.clip((local_density - sim_floor) / denom, 0.0, 1.0).astype(np.float32)
    return dict(
        local_agree=np.clip(local_agree, 0.0, 1.0).astype(np.float32),
        local_conf=np.clip(local_conf, 0.0, 1.0).astype(np.float32),
        local_label_prob=np.clip(local_label_prob, 0.0, 1.0).astype(np.float32),
        local_density=local_density)


def build_quality_aware_soft_targets_v17(fc_layer, Z_sup, q_seed_sup, protos, P_src_sup, P_proto_sup, P_knn_sup, base_weights, mode="dgr"):
    if Z_sup is None or len(Z_sup) == 0:
        info = dict(v17_lcq_active=False, v17_lcq_reason="empty_support")
        aux = dict(
            knownness=np.zeros((0), dtype=np.float32),
            class_quality=np.zeros((0), dtype=np.float32),
            overall_quality=np.zeros((0), dtype=np.float32),
            local_agree=np.zeros((0), dtype=np.float32),
            local_density=np.zeros((0), dtype=np.float32))
        return np.zeros((0, protos.shape[0]), dtype=np.float32), np.zeros((0), dtype=np.float32), info, aux

    cfg = dict(V17_QA_CFG[mode])
    P_seed = normalize_rows(q_seed_sup.astype(np.float32))
    P_src = normalize_rows(P_src_sup.astype(np.float32))
    P_proto = normalize_rows(P_proto_sup.astype(np.float32))
    P_knn = normalize_rows(P_knn_sup.astype(np.float32))

    support_ok, support_info = _v16_support_ok(P_seed)
    if not support_ok:
        Q_fb, w_fb, info_fb, aux_fb = build_quality_aware_soft_targets_v16(
            fc_layer=fc_layer,
            Z_sup=Z_sup,
            q_seed_sup=q_seed_sup,
            protos=protos,
            P_src_sup=P_src_sup,
            P_proto_sup=P_proto_sup,
            P_knn_sup=P_knn_sup,
            base_weights=base_weights,
            mode=mode)
        info = {
            **support_info,
            **{k: v for k, v in info_fb.items() if not k.startswith("v16_")},
            "v17_lcq_active": False,
            "v17_lcq_fallback": True,
            "v17_lcq_reason": "support_fallback",
        }
        aux = dict(
            knownness=np.asarray(aux_fb.get("knownness", np.zeros((len(Z_sup)), dtype=np.float32)), dtype=np.float32),
            class_quality=np.asarray(aux_fb.get("class_quality", np.zeros((len(Z_sup)), dtype=np.float32)), dtype=np.float32),
            overall_quality=np.asarray(aux_fb.get("overall_quality", np.zeros((len(Z_sup)), dtype=np.float32)), dtype=np.float32),
            local_agree=np.zeros((len(Z_sup)), dtype=np.float32),
            local_density=np.zeros((len(Z_sup)), dtype=np.float32))
        return Q_fb.astype(np.float32), w_fb.astype(np.float32), info, aux

    aug_views = V17_QA_AUG_VIEWS if len(Z_sup) > 1 else 1
    aug_noise = EMB_AUG_NOISE * V17_QA_AUG_NOISE_SCALE if len(Z_sup) > 1 else 0.0
    P_aug = source_head_probs_from_embeddings(fc_layer, Z_sup, noise_std=aug_noise, views=aug_views)

    P_prior = weighted_prob_fusion([P_proto, P_knn, P_src], [1.00, 0.95, 0.85])
    P_mix = weighted_prob_fusion([P_seed, P_src, P_proto, P_knn, P_aug], cfg["weights"])
    support_idx = np.arange(len(Z_sup), dtype=np.int64)
    P_ref = refine_probs_multi_stage(
        Z_sup, P_mix, protos, idx_support=support_idx,
        iters=max(1, int(cfg.get("refine_iters", 2))),
        src_mix=float(cfg.get("src_mix", 0.72))
    )
    P_cal = mild_class_balance_calibration(P_ref, strength=float(cfg.get("class_balance_strength", 0.08)))

    conf_seed = msp_from_probs(P_seed)
    conf_ref = msp_from_probs(P_ref)
    margin_seed = _v15_row_margin(P_seed)
    margin_ref = _v15_row_margin(P_ref)
    vote_frac = _vote_fraction(P_seed, P_src, P_proto, P_knn, P_aug, P_ref)
    seed_ref_agree = (np.argmax(P_seed, axis=1) == np.argmax(P_ref, axis=1)).astype(np.float32)
    src_proto_agree = (np.argmax(P_src, axis=1) == np.argmax(P_proto, axis=1)).astype(np.float32)
    src_knn_agree = (np.argmax(P_src, axis=1) == np.argmax(P_knn, axis=1)).astype(np.float32)
    proto_peak = np.max(P_proto, axis=1).astype(np.float32)
    prior_peak = np.max(P_prior, axis=1).astype(np.float32)
    ent_ref = (-np.sum(P_ref * np.log(np.clip(P_ref, 1e-12, 1.0)), axis=1) / np.log(max(2, P_ref.shape[1]))).astype(np.float32)
    ent_score = np.clip(1.0 - ent_ref, 0.0, 1.0)

    w_base = np.clip(np.asarray(base_weights, dtype=np.float32), 0.0, V17_QA_MAX_WEIGHT)
    w_base_unit = _v16_safe_unit_scale(w_base)

    knownness_base = (
        0.30 * w_base_unit +
        0.18 * conf_ref +
        0.12 * conf_seed +
        0.10 * margin_ref +
        0.10 * vote_frac +
        0.10 * ent_score +
        0.06 * seed_ref_agree +
        0.04 * prior_peak
    ).astype(np.float32)
    knownness_base = np.clip(knownness_base, 0.0, 1.0)

    class_quality_base = (
        0.26 * conf_ref +
        0.18 * margin_ref +
        0.16 * vote_frac +
        0.10 * seed_ref_agree +
        0.10 * src_proto_agree +
        0.08 * src_knn_agree +
        0.06 * proto_peak +
        0.06 * prior_peak
    ).astype(np.float32)
    class_quality_base = np.clip(class_quality_base, 0.0, 1.0)

    local = _v17_local_consensus_stats(
        Z_sup,
        P_ref,
        k=int(cfg.get("local_k", 24)),
        block=int(cfg.get("local_block", V17_LOCAL_BLOCK)),
        sim_floor=float(cfg.get("local_sim_floor", 0.10)))

    known_local = np.clip(
        0.40 * local["local_conf"] +
        0.35 * local["local_agree"] +
        0.25 * local["local_density"],
        0.0, 1.0
    ).astype(np.float32)
    class_local = np.clip(
        0.50 * local["local_label_prob"] +
        0.30 * local["local_agree"] +
        0.20 * local["local_conf"],
        0.0, 1.0
    ).astype(np.float32)

    knownness = np.clip(
        (1.0 - float(cfg.get("local_blend_known", 0.32))) * knownness_base +
        float(cfg.get("local_blend_known", 0.32)) * known_local,
        0.0, 1.0
    ).astype(np.float32)
    class_quality = np.clip(
        (1.0 - float(cfg.get("local_blend_class", 0.36))) * class_quality_base +
        float(cfg.get("local_blend_class", 0.36)) * class_local,
        0.0, 1.0
    ).astype(np.float32)

    knownness = np.clip(knownness * (0.92 + 0.08 * local["local_agree"]), 0.0, 1.0).astype(np.float32)
    class_quality = np.clip(class_quality * (0.90 + 0.10 * local["local_label_prob"]), 0.0, 1.0).astype(np.float32)

    overall_quality = np.clip(0.55 * knownness + 0.45 * class_quality, 0.0, 1.0).astype(np.float32)
    known_pow = np.power(np.clip(knownness, 1e-6, 1.0), float(cfg.get("gamma_known", 1.20))).astype(np.float32)
    class_pow = np.power(np.clip(class_quality, 1e-6, 1.0), float(cfg.get("gamma_class", 1.10))).astype(np.float32)
    overall_pow = np.power(np.clip(overall_quality, 1e-6, 1.0), float(cfg.get("gamma_overall", 1.15))).astype(np.float32)

    seed_alpha = cfg["seed_min"] + (cfg["seed_max"] - cfg["seed_min"]) * class_pow
    ref_alpha = cfg["ref_min"] + (cfg["ref_max"] - cfg["ref_min"]) * overall_pow
    prior_alpha = cfg["prior_min"] + (cfg["prior_max"] - cfg["prior_min"]) * (1.0 - class_pow) * (0.35 + 0.65 * known_pow)
    uniform_alpha = cfg["uniform_max"] * np.power(1.0 - known_pow, 1.15)

    alpha_sum = np.clip(seed_alpha + ref_alpha + prior_alpha + uniform_alpha, 1e-6, None)
    seed_alpha = seed_alpha / alpha_sum
    ref_alpha = ref_alpha / alpha_sum
    prior_alpha = prior_alpha / alpha_sum
    uniform_alpha = uniform_alpha / alpha_sum

    U = np.full_like(P_seed, 1.0 / P_seed.shape[1], dtype=np.float32)
    Q = normalize_rows(
        seed_alpha[:, None] * P_seed +
        ref_alpha[:, None] * P_cal +
        prior_alpha[:, None] * P_prior +
        uniform_alpha[:, None] * U
    )

    temp = cfg["temp_high"] - (cfg["temp_high"] - cfg["temp_low"]) * (0.40 * known_pow + 0.60 * class_pow)
    Q = sharpen_rows_by_temperature(Q, temp)

    high_mask = (overall_quality >= float(cfg.get("high_rel_thr", 0.80))) & (knownness >= float(cfg.get("align_known_thr", 0.48)))
    mid_mask = (overall_quality >= float(cfg.get("mid_rel_thr", 0.54))) & (~high_mask)
    low_mask = ~(high_mask | mid_mask)

    if np.any(high_mask):
        Q[high_mask] = restrict_topk_probs(Q[high_mask], k=min(2, Q.shape[1]), sharpen=1.06)
    if np.any(mid_mask):
        Q[mid_mask] = restrict_topk_probs(Q[mid_mask], k=min(3, Q.shape[1]), sharpen=1.01)
    if np.any(low_mask):
        Q[low_mask] = normalize_rows(0.75 * Q[low_mask] + 0.25 * U[low_mask])

    w_scale_known = cfg["known_floor"] + cfg["known_gain"] * known_pow
    w_scale_class = cfg["class_floor"] + cfg["class_gain"] * class_pow
    w_ref = w_base * np.sqrt(np.clip(w_scale_known * w_scale_class, 1e-6, None))
    w_ref = np.maximum(w_ref, cfg["low_weight_floor"] + 0.5 * cfg["low_weight_floor"] * overall_pow)
    w_ref = np.clip(w_ref, cfg["low_weight_floor"], V17_QA_MAX_WEIGHT).astype(np.float32)

    info = {
        **support_info,
        "v17_lcq_active": True,
        "v17_lcq_fallback": False,
        "v17_knownness_mean": float(np.mean(knownness)),
        "v17_knownness_std": float(np.std(knownness)),
        "v17_class_quality_mean": float(np.mean(class_quality)),
        "v17_class_quality_std": float(np.std(class_quality)),
        "v17_overall_quality_mean": float(np.mean(overall_quality)),
        "v17_overall_quality_std": float(np.std(overall_quality)),
        "v17_local_agree_mean": float(np.mean(local["local_agree"])),
        "v17_local_agree_std": float(np.std(local["local_agree"])),
        "v17_local_density_mean": float(np.mean(local["local_density"])),
        "v17_local_label_prob_mean": float(np.mean(local["local_label_prob"])),
        "v17_high_ratio": float(np.mean(high_mask)),
        "v17_mid_ratio": float(np.mean(mid_mask)),
        "v17_low_ratio": float(np.mean(low_mask)),
        "v17_weight_mean": float(np.mean(w_ref)),
        "v17_weight_std": float(np.std(w_ref)),
        "v17_conf_ref_mean": float(np.mean(conf_ref)),
        "v17_vote_mean": float(np.mean(vote_frac)),
    }
    aux = dict(
        knownness=knownness.astype(np.float32),
        class_quality=class_quality.astype(np.float32),
        overall_quality=overall_quality.astype(np.float32),
        local_agree=local["local_agree"].astype(np.float32),
        local_density=local["local_density"].astype(np.float32))
    return Q.astype(np.float32), w_ref.astype(np.float32), info, aux




def _safe_minmax01(x):
    x = np.asarray(x, dtype=np.float32)
    if len(x) == 0:
        return x
    xmin = float(np.min(x))
    xmax = float(np.max(x))
    if not np.isfinite(xmin) or not np.isfinite(xmax) or abs(xmax - xmin) < 1e-12:
        return np.ones_like(x, dtype=np.float32)
    return ((x - xmin) / (xmax - xmin + 1e-12)).astype(np.float32)


def build_m2v4_weighted_guided_method(base_spec, mode, Z_adapt, p_known_adapt, p_drift_given_known_adapt, base_name=None):
    if base_spec is None:
        return None, {"v19_m2w_active": False, "v19_m2w_reason": "base_spec_none"}

    n_target = len(Z_adapt)
    p_known = np.clip(np.asarray(p_known_adapt, dtype=np.float32), 0.0, 1.0)
    p_drift = np.clip(np.asarray(p_drift_given_known_adapt, dtype=np.float32), 0.0, 1.0)

    support_idx = np.asarray(base_spec.get("idx_sup_eval", base_spec.get("sup_idx", np.zeros((0), dtype=np.int64))), dtype=np.int64)
    if len(support_idx) == 0:
        spec = deepcopy(base_spec)
        spec.setdefault("bucket_info", {})
        spec["bucket_info"] = {**spec.get("bucket_info", {}), "v19_m2w_active": False, "v19_m2w_reason": "empty_support"}
        return spec, spec["bucket_info"]

    P_full = normalize_rows(np.asarray(base_spec.get("P", np.zeros((n_target, 1), dtype=np.float32)), dtype=np.float32))
    if "q" in base_spec and base_spec.get("q") is not None and len(base_spec["q"]) == len(support_idx):
        q_sup = normalize_rows(np.asarray(base_spec["q"], dtype=np.float32))
    else:
        q_sup = P_full[support_idx]

    if "w" in base_spec and base_spec.get("w") is not None and len(base_spec["w"]) == len(support_idx):
        w_base = np.asarray(base_spec["w"], dtype=np.float32)
    else:
        w_full = gather_base_weights_from_spec(base_spec, n_target)
        w_base = w_full[support_idx]

    cfg = dict(V19_M2W_CFG[mode])

    # Mild guidance only: prioritize high-known samples, use drift score as curriculum tie-breaker.
    support_raw = 0.75 * p_known[support_idx] + 0.25 * (p_known[support_idx] * p_drift[support_idx])
    support_gate = 0.60 + 0.40 * _safe_minmax01(support_raw)
    w_sup = np.clip(w_base * support_gate, 1e-3, None).astype(np.float32)

    align_base = np.asarray(base_spec.get("align_idx", base_spec.get("idx_gate", support_idx)), dtype=np.int64)
    if len(align_base) > 0:
        align_raw = 0.80 * p_known[align_base] + 0.20 * p_drift[align_base]
        thr = max(float(cfg.get("extra_unknown_thr", 0.32)), float(np.quantile(align_raw, cfg.get("align_q", 0.25))))
        keep = align_raw >= thr
        min_keep = min(len(align_base), max(16, int(0.35 * len(align_base))))
        if int(np.sum(keep)) < min_keep:
            order = np.argsort(-align_raw)
            keep = np.zeros((len(align_base)), dtype=bool)
            keep[order[:min_keep]] = True
        align_idx = align_base[keep]
    else:
        align_idx = align_base

    base_unrel = np.asarray(base_spec.get("unrel_idx", base_spec.get("idx_unk", np.zeros((0), dtype=np.int64))), dtype=np.int64)
    extra_unrel = np.where(p_known < float(cfg.get("extra_unknown_thr", 0.32)))[0]
    unrel_idx = np.unique(np.concatenate([base_unrel, extra_unrel])).astype(np.int64)

    spec = dict(
        train="lq_soft",
        sup_idx=support_idx,
        idx_sup_eval=support_idx,
        q=q_sup,
        w=w_sup,
        align_idx=align_idx,
        unrel_idx=unrel_idx,
        P=P_full,
        fallback_name=base_name,
        lambda_sup=1.0,
        lambda_align=cfg["lambda_align"],
        lambda_ent_min=cfg["lambda_ent_min"],
        lambda_ent_max=cfg["lambda_ent_max"],
        lambda_cons=cfg["lambda_cons"],
        lambda_proto=cfg["lambda_proto"],
        dynamic_align=True,
        bucket_info=dict(
            v19_m2w_active=True,
            v19_m2w_support_mean=float(np.mean(support_raw)) if len(support_raw) > 0 else float("nan"),
            v19_m2w_support_weight_mean=float(np.mean(w_sup)) if len(w_sup) > 0 else float("nan"),
            v19_m2w_align_keep=float(len(align_idx) / max(1, len(align_base))) if len(align_base) > 0 else 0.0,
            v19_m2w_unrel_ratio=float(len(unrel_idx) / max(1, n_target)),
            v19_m2w_known_mean=float(np.mean(p_known[support_idx])) if len(support_idx) > 0 else float("nan"),
            v19_m2w_drift_mean=float(np.mean(p_drift[support_idx])) if len(support_idx) > 0 else float("nan")))
    return spec, spec["bucket_info"]



def build_weighted_entry_minimal_from_base(
    base_spec,
    Z_adapt,
    p_known_adapt,
    p_drift_given_known_adapt,
    P_route_v4,
    P_target,
    tau_conf,
    base_name=None,
    overrides=None,
    method_name="DG_RAINCOAT_v22WeightedEntryMinimal"):
    if base_spec is None:
        return None, {"v22_active": False, "v22_reason": "base_spec_none", "v22_method_name": method_name}

    overrides = dict(overrides or {})
    gamma_drift = float(overrides.get("gamma_drift", V22_GAMMA_DRIFT))
    gamma_notunk = float(overrides.get("gamma_notunk", V22_GAMMA_NOTUNK))
    wmax = float(overrides.get("wmax", V22_WMAX))
    conf_smooth = float(overrides.get("conf_smooth", V22_CONF_SMOOTH))
    sup_eval_thr = float(overrides.get("sup_eval_thr", V22_SUP_EVAL_THR))
    unk_eval_thr = float(overrides.get("unk_eval_thr", V22_UNK_EVAL_THR))
    lambda_sup = float(overrides.get("lambda_sup", V22_LAMBDA_SUP))
    lambda_proto = float(overrides.get("lambda_proto", V22_LAMBDA_PROTO))
    lambda_stab = float(overrides.get("lambda_stab", V22_LAMBDA_STAB))
    lambda_unkE = float(overrides.get("lambda_unkE", V22_LAMBDA_UNKE))
    energy_bg_q = float(overrides.get("energy_bg_q", V22_ENERGY_BG_Q))
    energy_bg_ema = float(overrides.get("energy_bg_ema", V22_ENERGY_BG_EMA))
    unknown_energy_margin = float(overrides.get("unknown_energy_margin", V22_UNKNOWN_ENERGY_MARGIN))

    n_target = len(Z_adapt)
    idx_all = np.arange(n_target, dtype=np.int64)
    p_known = np.clip(np.asarray(p_known_adapt, dtype=np.float32), 0.0, 1.0)
    p_drift = np.clip(np.asarray(p_drift_given_known_adapt, dtype=np.float32), 0.0, 1.0)
    p_route = np.asarray(P_route_v4, dtype=np.float32)
    if p_route.ndim != 2 or p_route.shape[1] < 3:
        p_unknown = np.clip(1.0 - p_known, 0.0, 1.0)
        p_stable = np.clip(p_known * (1.0 - p_drift), 0.0, 1.0)
    else:
        p_unknown = np.clip(p_route[:, 2], 0.0, 1.0)
        p_stable = np.clip(p_route[:, 0], 0.0, 1.0)

    P_full = normalize_rows(np.asarray(P_target, dtype=np.float32))
    conf_full = msp_from_probs(P_full)
    conf_soft = 0.5 + 0.5 * sigmoid_np((conf_full - float(tau_conf)) / max(1e-6, conf_smooth))

    w_sup = np.clip(
        np.power(p_drift, gamma_drift) * np.power(np.clip(1.0 - p_unknown, 0.0, 1.0), gamma_notunk),
        0.0,
        wmax).astype(np.float32)
    w_sup = np.clip(w_sup * conf_soft, 0.0, wmax).astype(np.float32)
    w_stab = np.clip(p_stable * np.clip(1.0 - p_unknown, 0.0, 1.0), 0.0, 1.0).astype(np.float32)
    w_unk = np.clip(p_unknown, 0.0, 1.0).astype(np.float32)

    idx_sup_eval = np.where(w_sup > sup_eval_thr)[0].astype(np.int64)
    if len(idx_sup_eval) == 0 and len(idx_all) > 0:
        idx_sup_eval = np.argsort(-w_sup)[:min(len(idx_all), max(16, PSEUDO_MIN_KEEP))].astype(np.int64)
    idx_unk_eval = np.where(w_unk > unk_eval_thr)[0].astype(np.int64)
    if len(idx_unk_eval) == 0 and len(idx_all) > 0:
        idx_unk_eval = np.argsort(-w_unk)[:min(len(idx_all), max(16, PSEUDO_MIN_KEEP // 2))].astype(np.int64)

    spec = dict(
        train="weighted_open_minimal",
        method_name=method_name,
        sup_idx=idx_all,
        stab_idx=idx_all,
        unk_idx=idx_all,
        idx_sup_eval=idx_sup_eval,
        idx_unk_eval=idx_unk_eval,
        q=P_full,
        P=P_full,
        w_sup=w_sup.astype(np.float32),
        w_stab=w_stab.astype(np.float32),
        w_unk=w_unk.astype(np.float32),
        lambda_sup=lambda_sup,
        lambda_proto=lambda_proto,
        lambda_stab=lambda_stab,
        lambda_unkE=lambda_unkE,
        gamma_drift=gamma_drift,
        gamma_notunk=gamma_notunk,
        wmax=wmax,
        conf_smooth=conf_smooth,
        sup_eval_thr=sup_eval_thr,
        unk_eval_thr=unk_eval_thr,
        unknown_loss_mode="energy_margin",
        unknown_energy_margin=unknown_energy_margin,
        energy_bg_q=energy_bg_q,
        energy_bg_ema=energy_bg_ema,
        fallback_name=base_name,
        variant_overrides=overrides,
        bucket_info=dict(
            v22_active=True,
            v22_method_name=method_name,
            v22_support_weight_mean=float(np.mean(w_sup)) if len(w_sup) > 0 else float("nan"),
            v22_stable_weight_mean=float(np.mean(w_stab)) if len(w_stab) > 0 else float("nan"),
            v22_unknown_weight_mean=float(np.mean(w_unk)) if len(w_unk) > 0 else float("nan"),
            v22_conf_soft_mean=float(np.mean(conf_soft)) if len(conf_soft) > 0 else float("nan"),
            v22_known_mean=float(np.mean(p_known)) if len(p_known) > 0 else float("nan"),
            v22_drift_mean=float(np.mean(p_drift)) if len(p_drift) > 0 else float("nan"),
            v22_unknown_mean=float(np.mean(p_unknown)) if len(p_unknown) > 0 else float("nan"),
            v22_binary_support_keep=float(len(idx_sup_eval) / max(1, n_target)),
            v22_binary_unknown_keep=float(len(idx_unk_eval) / max(1, n_target)),
            v22_gamma_drift=gamma_drift,
            v22_gamma_notunk=gamma_notunk,
            v22_lambda_sup=lambda_sup,
            v22_lambda_proto=lambda_proto,
            v22_lambda_stab=lambda_stab,
            v22_lambda_unkE=lambda_unkE,
            v22_energy_bg_q=energy_bg_q,
            v22_energy_bg_ema=energy_bg_ema,
            v22_unknown_energy_margin=unknown_energy_margin))
    return spec, spec["bucket_info"]


def build_v22_method_info_from_spec(method_name, spec, y_adapt, extra_info=None, base_name=None, target_source=None):
    v22_sel_idx = np.asarray(
        spec.get("idx", spec.get("idx_sup_eval", spec.get("sup_idx", np.zeros((0), dtype=np.int64)))),
        dtype=np.int64)
    v22_stats = summarize_method_on_selected(v22_sel_idx, y_adapt, spec["P"])
    v22_total_known = max(1, int(np.sum(y_adapt >= 0)))
    v22_sel_precision = float((y_adapt[v22_sel_idx] >= 0).mean()) if len(v22_sel_idx) > 0 else float("nan")
    v22_sel_recall = float(np.sum(y_adapt[v22_sel_idx] >= 0) / v22_total_known) if len(v22_sel_idx) > 0 else 0.0
    v22_sel_keep = float(len(v22_sel_idx) / max(1, len(y_adapt)))

    v22_unk_idx = np.asarray(
        spec.get("idx_unk_eval", spec.get("unrel_idx", spec.get("idx_unk", np.zeros((0), dtype=np.int64)))),
        dtype=np.int64)
    v22_unknown_keep = float(len(v22_unk_idx) / max(1, len(y_adapt)))
    v22_unknown_precision = float((y_adapt[v22_unk_idx] < 0).mean()) if len(v22_unk_idx) > 0 else float("nan")

    info = dict(
        sel_precision=v22_sel_precision,
        sel_recall=v22_sel_recall,
        sel_keep_ratio=v22_sel_keep,
        sel_size=v22_stats["sel_size"],
        pseudo_acc_selected=v22_stats["pseudo_acc"],
        pseudo_acc=v22_stats["pseudo_acc"],
        unknown_cand_keep=v22_unknown_keep,
        unknown_cand_precision=v22_unknown_precision)

    w_sup_all = np.asarray(spec.get("w_sup", np.zeros((0), dtype=np.float32)), dtype=np.float32)
    if len(w_sup_all) == len(y_adapt):
        known_mask = (y_adapt >= 0)
        pred_all = np.argmax(spec["P"], axis=1)
        info.update(
            sum_w_sup=float(np.sum(w_sup_all)),
            num_w_sup_gt_02=int(np.sum(w_sup_all > 0.20)),
            num_w_sup_gt_05=int(np.sum(w_sup_all > 0.50)),
            effective_coverage=float(np.sum(w_sup_all[known_mask]) / max(1, np.sum(known_mask))),
            soft_pseudo_acc=float(np.sum(w_sup_all[known_mask] * (pred_all[known_mask] == y_adapt[known_mask])) / max(1e-6, np.sum(w_sup_all[known_mask]))))

    w_stab_all = np.asarray(spec.get("w_stab", np.zeros((0), dtype=np.float32)), dtype=np.float32)
    if len(w_stab_all) == len(y_adapt):
        stable_mask = (y_adapt >= 0)
        info.update(
            sum_w_stab=float(np.sum(w_stab_all)),
            effective_stable_pressure=float(np.sum(w_stab_all[stable_mask]) / max(1, np.sum(stable_mask))))

    w_unk_all = np.asarray(spec.get("w_unk", np.zeros((0), dtype=np.float32)), dtype=np.float32)
    if len(w_unk_all) == len(y_adapt):
        info.update(sum_w_unk=float(np.sum(w_unk_all)))

    info.update(spec.get("bucket_info", {}))
    info.update(
        v22_target_source=target_source,
        v22_base_name=base_name,
        v22_gamma_drift=float(spec.get("gamma_drift", np.nan)),
        v22_gamma_notunk=float(spec.get("gamma_notunk", np.nan)),
        v22_lambda_sup=float(spec.get("lambda_sup", np.nan)),
        v22_lambda_proto=float(spec.get("lambda_proto", np.nan)),
        v22_lambda_stab=float(spec.get("lambda_stab", np.nan)),
        v22_lambda_unkE=float(spec.get("lambda_unkE", np.nan)),
        v22_unknown_energy_margin=float(spec.get("unknown_energy_margin", np.nan)))
    if extra_info:
        info.update(extra_info)
    return info


def build_quality_aware_method_from_base_v17(base_spec, mode, fc_layer, Z_adapt, P_logit_adapt, P_proto_adapt, P_knn_adapt, protos, base_name=None):
    if base_spec is None:
        return None, {}
    support_idx = np.asarray(base_spec.get("idx_sup_eval", base_spec.get("sup_idx", np.zeros((0), dtype=np.int64))), dtype=np.int64)
    fallback_name = base_name if base_name is not None else ("DG_RAINCOAT_v16QW" if mode == "dgr" else "ThreeBucketV8Adaptive_v16QW")
    if len(support_idx) == 0:
        spec = deepcopy(base_spec)
        spec.setdefault("bucket_info", {})
        spec["bucket_info"] = {**spec.get("bucket_info", {}), "v17_lcq_active": False, "v17_lcq_reason": "empty_support"}
        return spec, spec["bucket_info"]

    q_seed_full = normalize_rows(np.asarray(base_spec.get("q_seed", base_spec.get("P")), dtype=np.float32))
    w_full = gather_base_weights_from_spec(base_spec, len(Z_adapt))
    Q_ref, w_ref, info, aux = build_quality_aware_soft_targets_v17(
        fc_layer=fc_layer,
        Z_sup=Z_adapt[support_idx],
        q_seed_sup=q_seed_full[support_idx],
        protos=protos,
        P_src_sup=P_logit_adapt[support_idx],
        P_proto_sup=P_proto_adapt[support_idx],
        P_knn_sup=P_knn_adapt[support_idx],
        base_weights=w_full[support_idx],
        mode=mode)
    if not info.get("v17_lcq_active", False):
        spec = deepcopy(base_spec)
        spec.setdefault("bucket_info", {})
        spec["bucket_info"] = {**spec.get("bucket_info", {}), **info}
        return spec, spec["bucket_info"]

    P_full = q_seed_full.copy()
    P_full[support_idx] = Q_ref
    idx_unk = np.asarray(base_spec.get("idx_unk", np.zeros((0), dtype=np.int64)), dtype=np.int64)
    cfg = dict(V17_QA_CFG[mode])

    knownness = aux["knownness"]
    overall_quality = aux["overall_quality"]
    local_agree = np.asarray(aux.get("local_agree", np.zeros_like(knownness)), dtype=np.float32)
    local_density = np.asarray(aux.get("local_density", np.zeros_like(knownness)), dtype=np.float32)
    local_align = 0.60 * local_agree + 0.40 * local_density

    align_mask = (
        (knownness >= float(cfg.get("align_known_thr", 0.48))) &
        (local_align >= float(cfg.get("local_align_thr", 0.50)))
    )
    min_keep = min(len(support_idx), max(16, int(0.25 * len(support_idx))))
    if np.sum(align_mask) < min_keep:
        score = 0.50 * knownness + 0.25 * overall_quality + 0.25 * local_align
        order = np.argsort(-score)
        align_mask = np.zeros((len(support_idx)), dtype=bool)
        align_mask[order[:min_keep]] = True
    align_idx = support_idx[align_mask]

    spec = dict(
        train="lq_soft",
        sup_idx=support_idx,
        idx_sup_eval=support_idx,
        q=Q_ref,
        w=w_ref,
        align_idx=align_idx,
        unrel_idx=idx_unk,
        P=P_full,
        fallback_name=fallback_name,
        lambda_sup=1.0,
        lambda_align=cfg["lambda_align"],
        lambda_ent_min=cfg["lambda_ent_min"],
        lambda_ent_max=cfg["lambda_ent_max"],
        lambda_cons=cfg["lambda_cons"],
        lambda_proto=cfg["lambda_proto"],
        dynamic_align=True,
        bucket_info={**info, "v17_align_ratio": float(np.mean(align_mask)), "v17_local_align_mean": float(np.mean(local_align))})
    return spec, spec["bucket_info"]


def build_quality_aware_method_from_base_v16(base_spec, mode, fc_layer, Z_adapt, P_logit_adapt, P_proto_adapt, P_knn_adapt, protos, base_name=None):
    if base_spec is None:
        return None, {}
    support_idx = np.asarray(base_spec.get("idx_sup_eval", base_spec.get("sup_idx", np.zeros((0), dtype=np.int64))), dtype=np.int64)
    fallback_name = base_name if base_name is not None else ("DG_RAINCOAT_v15QW" if mode == "dgr" else "ThreeBucketV8Adaptive_v15QW")
    if len(support_idx) == 0:
        spec = deepcopy(base_spec)
        spec.setdefault("bucket_info", {})
        spec["bucket_info"] = {**spec.get("bucket_info", {}), "v16_qw_active": False, "v16_qw_reason": "empty_support"}
        return spec, spec["bucket_info"]

    q_seed_full = normalize_rows(np.asarray(base_spec.get("q_seed", base_spec.get("P")), dtype=np.float32))
    w_full = gather_base_weights_from_spec(base_spec, len(Z_adapt))
    Q_ref, w_ref, info, aux = build_quality_aware_soft_targets_v16(
        fc_layer=fc_layer,
        Z_sup=Z_adapt[support_idx],
        q_seed_sup=q_seed_full[support_idx],
        protos=protos,
        P_src_sup=P_logit_adapt[support_idx],
        P_proto_sup=P_proto_adapt[support_idx],
        P_knn_sup=P_knn_adapt[support_idx],
        base_weights=w_full[support_idx],
        mode=mode)
    if not info.get("v16_qw_active", False):
        spec = deepcopy(base_spec)
        spec.setdefault("bucket_info", {})
        spec["bucket_info"] = {**spec.get("bucket_info", {}), **info}
        return spec, spec["bucket_info"]

    P_full = q_seed_full.copy()
    P_full[support_idx] = Q_ref
    idx_unk = np.asarray(base_spec.get("idx_unk", np.zeros((0), dtype=np.int64)), dtype=np.int64)
    cfg = dict(V16_QA_CFG[mode])

    knownness = aux["knownness"]
    overall_quality = aux["overall_quality"]
    align_mask = knownness >= float(cfg.get("align_known_thr", 0.48))
    min_keep = min(len(support_idx), max(16, int(0.25 * len(support_idx))))
    if np.sum(align_mask) < min_keep:
        score = 0.65 * knownness + 0.35 * overall_quality
        order = np.argsort(-score)
        align_mask = np.zeros((len(support_idx)), dtype=bool)
        align_mask[order[:min_keep]] = True
    align_idx = support_idx[align_mask]

    spec = dict(
        train="lq_soft",
        sup_idx=support_idx,
        idx_sup_eval=support_idx,
        q=Q_ref,
        w=w_ref,
        align_idx=align_idx,
        unrel_idx=idx_unk,
        P=P_full,
        fallback_name=fallback_name,
        lambda_sup=1.0,
        lambda_align=cfg["lambda_align"],
        lambda_ent_min=cfg["lambda_ent_min"],
        lambda_ent_max=cfg["lambda_ent_max"],
        lambda_cons=cfg["lambda_cons"],
        lambda_proto=cfg["lambda_proto"],
        dynamic_align=True,
        bucket_info={**info, "v16_align_ratio": float(np.mean(align_mask))})
    return spec, spec["bucket_info"]



def build_label_quality_soft_targets(fc_layer, Z_sup, q_seed_sup, protos, P_src_sup, P_proto_sup, P_knn_sup, base_weights, mode="dgr"):
    if Z_sup is None or len(Z_sup) == 0:
        return np.zeros((0, protos.shape[0]), dtype=np.float32), np.zeros((0), dtype=np.float32), {}
    cfg = dict(LQ_METHOD_CFG[mode])
    P_seed = normalize_rows(q_seed_sup.astype(np.float32))
    P_src = normalize_rows(P_src_sup.astype(np.float32))
    P_proto = normalize_rows(P_proto_sup.astype(np.float32))
    P_knn = normalize_rows(P_knn_sup.astype(np.float32))
    P_aug = source_head_probs_from_embeddings(fc_layer, Z_sup, noise_std=EMB_AUG_NOISE * LQ_AUG_NOISE_SCALE, views=LQ_AUG_VIEWS)
    P_mix = weighted_prob_fusion([P_seed, P_src, P_proto, P_knn, P_aug], cfg["weights"])
    P_ref = refine_probs_multi_stage(Z_sup, P_mix, protos, idx_support=np.arange(len(Z_sup)), iters=cfg["refine_iters"], src_mix=cfg["src_mix"])
    P_cal = mild_class_balance_calibration(P_ref, strength=cfg["class_balance_strength"])

    vote_frac = _vote_fraction(P_seed, P_src, P_proto, P_knn, P_aug, P_ref)
    seed_ref_agree = (np.argmax(P_seed, axis=1) == np.argmax(P_ref, axis=1)).astype(np.float32)
    src_proto_agree = (np.argmax(P_src, axis=1) == np.argmax(P_proto, axis=1)).astype(np.float32)
    src_aug_agree = (np.argmax(P_src, axis=1) == np.argmax(P_aug, axis=1)).astype(np.float32)
    conf_ref = msp_from_probs(P_ref)
    conf_seed = msp_from_probs(P_seed)
    rel = (0.36 * conf_ref + 0.18 * conf_seed + 0.22 * vote_frac + 0.12 * seed_ref_agree + 0.06 * src_proto_agree + 0.06 * src_aug_agree).astype(np.float32)
    rel = np.clip(rel, 0.0, 1.0)

    temp = cfg["temp_high"] - (cfg["temp_high"] - cfg["temp_low"]) * rel
    Q = sharpen_rows_by_temperature(P_cal, temp)
    low_mask = rel < cfg["low_rel_thr"]
    mid_mask = (rel >= cfg["low_rel_thr"]) & (rel < cfg["mid_rel_thr"])
    if np.any(low_mask):
        Q[low_mask] = restrict_topk_probs(Q[low_mask], k=cfg["topk_low"], sharpen=1.00)
    if np.any(mid_mask):
        Q[mid_mask] = restrict_topk_probs(Q[mid_mask], k=cfg["topk_mid"], sharpen=1.03)

    w_ref = np.asarray(base_weights, dtype=np.float32) * (cfg["low_weight_floor"] + cfg["weight_gain"] * rel)
    w_ref = np.clip(w_ref, cfg["low_weight_floor"], 1.35).astype(np.float32)
    info = dict(
        lq_support_size=int(len(Z_sup)),
        lq_rel_mean=float(np.mean(rel)),
        lq_rel_std=float(np.std(rel)),
        lq_vote_mean=float(np.mean(vote_frac)),
        lq_seed_ref_agree=float(np.mean(seed_ref_agree)),
        lq_src_proto_agree=float(np.mean(src_proto_agree)),
        lq_src_aug_agree=float(np.mean(src_aug_agree)),
        lq_conf_ref_mean=float(np.mean(conf_ref)))
    return Q.astype(np.float32), w_ref, info


def build_label_quality_method_from_base(base_spec, mode, fc_layer, Z_adapt, P_logit_adapt, P_proto_adapt, P_knn_adapt, protos):
    support_idx = np.asarray(base_spec.get("idx_sup_eval", base_spec.get("sup_idx", np.zeros((0), dtype=np.int64))), dtype=np.int64)
    if len(support_idx) == 0:
        return None, {}
    q_seed_full = normalize_rows(np.asarray(base_spec.get("q_seed", base_spec.get("P")), dtype=np.float32))
    w_full = gather_base_weights_from_spec(base_spec, len(Z_adapt))
    Q_ref, w_ref, info = build_label_quality_soft_targets(
        fc_layer=fc_layer,
        Z_sup=Z_adapt[support_idx],
        q_seed_sup=q_seed_full[support_idx],
        protos=protos,
        P_src_sup=P_logit_adapt[support_idx],
        P_proto_sup=P_proto_adapt[support_idx],
        P_knn_sup=P_knn_adapt[support_idx],
        base_weights=w_full[support_idx],
        mode=mode)
    P_full = q_seed_full.copy()
    P_full[support_idx] = Q_ref
    idx_unk = np.asarray(base_spec.get("idx_unk", np.zeros((0), dtype=np.int64)), dtype=np.int64)
    idx_gate = np.asarray(base_spec.get("idx_gate", support_idx), dtype=np.int64)
    align_idx = np.setdiff1d(idx_gate, idx_unk, assume_unique=False)
    cfg = dict(LQ_METHOD_CFG[mode])
    spec = dict(
        train="lq_soft",
        sup_idx=support_idx,
        idx_sup_eval=support_idx,
        q=Q_ref,
        w=w_ref,
        align_idx=align_idx,
        unrel_idx=idx_unk,
        P=P_full,
        fallback_name=("DG_RAINCOAT_v12" if mode == "dgr" else "ThreeBucketV8Adaptive_v12"),
        lambda_sup=1.0,
        lambda_align=cfg["lambda_align"],
        lambda_ent_min=cfg["lambda_ent_min"],
        lambda_ent_max=cfg["lambda_ent_max"],
        lambda_cons=cfg["lambda_cons"],
        lambda_proto=cfg["lambda_proto"],
        dynamic_align=True,
        bucket_info=info)
    return spec, info

def adapt_raincoat_lite(fc_layer, Z_target, idx_gate, idx_common_seed, idx_unknown_seed, Z_replay, y_replay,
                        protos, tau_conf, min_keep=64,
                        bottleneck=64, scale=0.3, epochs_stage1=8, epochs_stage2=12,
                        lr=1e-3, wd=1e-4, batch=128,
                        lambda_replay_ce=1.0, lambda_replay_anchor=1.0):
    if len(idx_gate) == 0:
        return None, dict(stage1_common_size=0, stage1_unknown_size=0)

    warm_adapter = adapt_on_embeddings_generic(
        fc_layer=fc_layer,
        Z_replay=Z_replay, y_replay=y_replay,
        Z_align=Z_target[idx_gate],
        proto_bank=protos,
        bottleneck=bottleneck, scale=scale, epochs=epochs_stage1,
        lr=lr, wd=wd, batch=batch,
        lambda_replay_ce=lambda_replay_ce, lambda_replay_anchor=lambda_replay_anchor,
        lambda_align=LAMBDA_ALIGN, lambda_ent_min=LAMBDA_ENT_MIN, lambda_cons=LAMBDA_CONS,
        dynamic_align=True)

    if warm_adapter is None:
        return None, dict(stage1_common_size=0, stage1_unknown_size=0)

    Z_gate_1, logits_gate_1 = apply_adapter_and_fc(warm_adapter, fc_layer, Z_target[idx_gate], batch=512)
    P_logit_1 = softmax_np(logits_gate_1)
    P_proto_1 = proto_probs_cosine(Z_gate_1, protos, temp=PROTO_T, l2norm=True)
    P_knn_1 = P_proto_1.copy()
    gate_local = np.ones((len(idx_gate)), dtype=np.int64)
    common_local, unknown_local, _, split_info = split_common_unknown_candidates(
        Z_gate_1, P_logit_1, P_proto_1, P_knn_1, gate_local, protos, tau_conf=tau_conf, min_keep=min_keep
    )
    idx_common = idx_gate[common_local]
    idx_unknown = idx_gate[unknown_local]

    support_idx = idx_common if len(idx_common) > 0 else idx_common_seed
    P_seed = weighted_prob_fusion([P_logit_1, P_proto_1], [1.0, 1.0])
    P_refined = refine_probs_multi_stage(Z_gate_1, P_seed, protos, idx_support=common_local if len(common_local) > 0 else np.arange(len(idx_gate)), iters=REFINE_ITERS)
    w_common = normalize_conf_weights(msp_from_probs(P_refined[common_local])) if len(common_local) > 0 else None

    final_adapter = adapt_on_embeddings_generic(
        fc_layer=fc_layer,
        Z_replay=Z_replay, y_replay=y_replay,
        Z_sup=Z_target[idx_common] if len(idx_common) > 0 else None,
        q_sup=P_refined[common_local] if len(common_local) > 0 else None,
        w_sup=w_common,
        Z_align=Z_target[idx_gate],
        Z_unrel=Z_target[idx_unknown] if len(idx_unknown) > 0 else None,
        proto_bank=protos,
        bottleneck=bottleneck, scale=scale, epochs=epochs_stage2,
        lr=lr, wd=wd, batch=batch,
        lambda_replay_ce=lambda_replay_ce, lambda_replay_anchor=lambda_replay_anchor,
        lambda_align=LAMBDA_ALIGN, lambda_ent_min=LAMBDA_ENT_MIN * 0.5,
        lambda_ent_max=LAMBDA_ENT_MAX, lambda_cons=LAMBDA_CONS, lambda_proto=LAMBDA_PROTO,
        init_state=clone_state_dict_cpu(warm_adapter))
    info = dict(
        stage1_common_size=int(len(idx_common)),
        stage1_unknown_size=int(len(idx_unknown)),
        seed_common_size=int(len(idx_common_seed)),
        seed_unknown_size=int(len(idx_unknown_seed)),
        split_threshold=float(split_info["threshold"]) if np.isfinite(split_info["threshold"]) else float("nan"))
    final_adapter, guard_info = guard_adapter_with_fallback(final_adapter, warm_adapter, fc_layer, Z_replay, y_replay, Z_probe=Z_target[idx_gate], probe_name="raincoat")
    info.update(guard_info)
    return final_adapter, info


def plot_bar_compare(metric_dict, out_png, title):
    keys = list(metric_dict.keys())
    vals = [metric_dict[k] for k in keys]
    plt.figure(figsize=(max(8, 0.55*len(keys)), 4.2))
    plt.bar(np.arange(len(keys)), vals)
    plt.xticks(np.arange(len(keys)), keys, rotation=55, ha="right")
    plt.ylabel("value")
    plt.title(title)
    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    ensure_parent_dir(out_png)
    plt.savefig(_win_safe_path(out_png), dpi=160)
    plt.close()



def select_bottom_idx_by_mask_with_fallback(base_idx, mask_keep, score, min_keep=64):
    base_idx = np.asarray(base_idx, dtype=np.int64)
    if len(base_idx) == 0:
        return base_idx
    idx_keep = base_idx[mask_keep]
    if len(idx_keep) >= min_keep:
        return idx_keep
    order = np.argsort(score[base_idx])
    k = min(len(base_idx), min_keep)
    return base_idx[order[:k]]


def restrict_topk_probs(P, k=2, sharpen=1.0):
    P = normalize_rows(P)
    if P.shape[1] <= max(1, int(k)):
        if abs(float(sharpen) - 1.0) < 1e-8:
            return P.astype(np.float32)
        return normalize_rows(P ** float(sharpen)).astype(np.float32)
    k = max(1, int(k))
    out = np.zeros_like(P, dtype=np.float32)
    order = np.argsort(-P, axis=1)[:, :k]
    rows = np.arange(P.shape[0])[:, None]
    out[rows, order] = P[rows, order]
    if abs(float(sharpen) - 1.0) > 1e-8:
        out = out ** float(sharpen)
    return normalize_rows(out).astype(np.float32)


def build_three_bucket_partition(
    Z_adapt,
    gate_pred,
    Sid_adapt,
    Sdom_adapt,
    P_logit_adapt,
    P_proto_adapt,
    P_knn_adapt,
    protos,
    tau_conf,
    min_keep=64,
    reliable_keep_q=RELIABLE_KEEP_Q,
    ambig_keep_q=AMBIG_KEEP_Q,
    unknown_keep_q=UNKNOWN_KEEP_Q):
    idx_gate = np.where(gate_pred == 1)[0]
    n = len(gate_pred)
    empty = np.zeros((0), dtype=np.int64)
    if len(idx_gate) == 0:
        aux = dict(
            bucket_score=np.zeros((n), dtype=np.float32),
            unknown_score=np.zeros((n), dtype=np.float32),
            P_seed=weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN]),
            threshold_reliable=float("nan"),
            threshold_unknown=float("nan"))
        info = dict(reliable_size=0, ambiguous_size=0, unknown_size=0)
        return empty, empty, empty, aux, info

    P_tri = weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN])
    y_logit = np.argmax(P_logit_adapt, axis=1)
    y_proto = np.argmax(P_proto_adapt, axis=1)
    y_knn = np.argmax(P_knn_adapt, axis=1)

    conf_tri = msp_from_probs(P_tri)
    margin_tri = top2_margin(P_tri)
    _, proto_margin, proto_dist = prototype_margin_dist(Z_adapt, protos)

    agree = ((y_logit == y_proto).astype(np.float32) + (y_logit == y_knn).astype(np.float32) + (y_proto == y_knn).astype(np.float32)) / 3.0
    sid_n = robust_unit_interval(Sid_adapt, idx_gate)
    sdom_src_n = 1.0 - robust_unit_interval(Sdom_adapt, idx_gate)
    conf_n = robust_unit_interval(conf_tri, idx_gate)
    margin_n = robust_unit_interval(margin_tri, idx_gate)
    proto_margin_n = robust_unit_interval(proto_margin, idx_gate)
    proto_dist_n = robust_unit_interval(proto_dist, idx_gate)

    bucket_score = (
        0.30 * sid_n +
        0.22 * conf_n +
        0.16 * agree +
        0.12 * margin_n +
        0.12 * proto_margin_n +
        0.10 * sdom_src_n -
        0.10 * proto_dist_n
    ).astype(np.float32)

    unknown_score = (
        0.48 * (1.0 - bucket_score) +
        0.22 * proto_dist_n +
        0.18 * (1.0 - conf_n) +
        0.12 * (1.0 - margin_n)
    ).astype(np.float32)

    ng = len(idx_gate)
    min_rel = min(ng, max(8, min_keep))
    min_amb = min(max(0, ng - min_rel), max(16, min_keep))
    min_unk = min(max(0, ng - min_rel), max(8, min_keep // 2))

    k_rel = min(ng, max(int(round(float(reliable_keep_q) * ng)), min_rel))
    k_unk = min(max(0, ng - k_rel), max(int(round(float(unknown_keep_q) * ng)), min_unk))
    k_amb = min(max(0, ng - k_rel - k_unk), max(int(round(float(ambig_keep_q) * ng)), min_amb))
    if k_amb <= 0 and ng - k_rel - k_unk > 0:
        k_amb = ng - k_rel - k_unk

    order_rel = idx_gate[np.argsort(-bucket_score[idx_gate])]
    idx_rel = order_rel[:k_rel]

    cand_after_rel = np.setdiff1d(idx_gate, idx_rel, assume_unique=False)
    order_unk = cand_after_rel[np.argsort(-unknown_score[cand_after_rel])]
    idx_unk = order_unk[:k_unk]

    rem = np.setdiff1d(cand_after_rel, idx_unk, assume_unique=False)
    order_amb = rem[np.argsort(-bucket_score[rem])]
    idx_amb = order_amb[:k_amb]

    thr_rel = float(np.min(bucket_score[idx_rel])) if len(idx_rel) > 0 else float("nan")
    thr_unk = float(np.min(unknown_score[idx_unk])) if len(idx_unk) > 0 else float("nan")

    support_idx = np.concatenate([idx_rel, idx_amb], axis=0) if (len(idx_rel) + len(idx_amb)) > 0 else idx_gate
    P_seed = refine_probs_multi_stage(Z_adapt, P_tri, protos, idx_support=support_idx, iters=max(REFINE_ITERS, 2), src_mix=0.78)

    if len(idx_amb) > 0:
        P_seed[idx_amb] = restrict_topk_probs(P_seed[idx_amb], k=AMBIG_TOPK, sharpen=1.10)

    w_rel = np.asarray([], dtype=np.float32)
    if len(idx_rel) > 0:
        rel_score = 0.60 * bucket_score[idx_rel] + 0.40 * conf_tri[idx_rel]
        w_rel = RELIABLE_WEIGHT_FLOOR + (1.0 - RELIABLE_WEIGHT_FLOOR) * normalize_conf_weights(rel_score)

    w_amb = np.asarray([], dtype=np.float32)
    if len(idx_amb) > 0:
        amb_score = 0.50 * bucket_score[idx_amb] + 0.50 * conf_tri[idx_amb]
        w_amb = AMBIG_WEIGHT_FLOOR + AMBIG_WEIGHT_SCALE * normalize_conf_weights(amb_score)

    aux = dict(
        bucket_score=bucket_score.astype(np.float32),
        unknown_score=unknown_score.astype(np.float32),
        P_seed=P_seed.astype(np.float32),
        threshold_reliable=thr_rel,
        threshold_unknown=thr_unk,
        w_rel=w_rel.astype(np.float32),
        w_amb=w_amb.astype(np.float32))
    info = dict(
        reliable_size=int(len(idx_rel)),
        ambiguous_size=int(len(idx_amb)),
        unknown_size=int(len(idx_unk)),
        reliable_keep=float(len(idx_rel) / max(1, len(gate_pred))),
        ambiguous_keep=float(len(idx_amb) / max(1, len(gate_pred))),
        unknown_keep=float(len(idx_unk) / max(1, len(gate_pred))),
        bucket_score_gate_mean=float(bucket_score[idx_gate].mean()),
        unknown_score_gate_mean=float(unknown_score[idx_gate].mean()),
        reliable_threshold=float(thr_rel),
        unknown_threshold=float(thr_unk))
    return idx_rel.astype(np.int64), idx_amb.astype(np.int64), idx_unk.astype(np.int64), aux, info



def normalize_rows_torch(q):
    q = torch.clamp(q, min=1e-12)
    return q / q.sum(dim=1, keepdim=True)


def ema_update_(teacher, student, momentum=0.995):
    with torch.no_grad():
        for pt, ps in zip(teacher.parameters(), student.parameters()):
            pt.data.mul_(momentum).add_(ps.data, alpha=1.0 - momentum)


def proto_repulsion_loss(z, proto_bank, margin=UNKNOWN_REPEL_MARGIN):
    if proto_bank is None or len(proto_bank) == 0 or z.numel() == 0:
        return torch.tensor(0.0, device=z.device)
    P = torch.tensor(proto_bank, dtype=torch.float32, device=z.device)
    P = F.normalize(P, dim=1)
    sims = F.normalize(z, dim=1) @ P.T
    max_sim = sims.max(dim=1).values
    return torch.relu(max_sim - float(margin)).mean()


def mixed_teacher_targets(q_seed, teacher_logits=None, blend=EMA_TEACHER_BLEND, temp=1.0):
    q_seed = normalize_rows_torch(q_seed)
    if teacher_logits is None:
        return q_seed
    q_teacher = torch.softmax(teacher_logits / max(float(temp), 1e-6), dim=1)
    q_mix = float(blend) * q_teacher + (1.0 - float(blend)) * q_seed
    return normalize_rows_torch(q_mix)


def adapt_three_bucket_v5(
    fc_layer,
    Z_replay,
    y_replay,
    Z_target,
    idx_gate,
    idx_rel,
    idx_amb,
    idx_unk,
    q_seed,
    w_rel=None,
    w_amb=None,
    proto_bank=None,
    bottleneck=64,
    scale=0.3,
    epochs=20,
    lr=1e-3,
    wd=1e-4,
    batch=128,
    lambda_replay_ce=1.0,
    lambda_replay_anchor=1.0,
    lambda_rel_sup=1.0,
    lambda_amb_sup=LAMBDA_AMB_SUP,
    lambda_align=LAMBDA_ALIGN,
    lambda_ent_min=LAMBDA_ENT_MIN,
    lambda_ent_max=LAMBDA_ENT_MAX,
    lambda_cons=LAMBDA_CONS,
    lambda_proto=LAMBDA_PROTO,
    lambda_src_proto=LAMBDA_SRC_PROTO,
    lambda_src_logit=LAMBDA_SRC_LOGIT,
    lambda_u_repel=LAMBDA_UNKNOWN_REPEL,
    use_ema=True,
    ema_momentum=EMA_MOMENTUM,
    teacher_blend=EMA_TEACHER_BLEND,
    unknown_warmup_epochs=UNKNOWN_WARMUP_EPOCHS,
    unknown_ramp_epochs=UNKNOWN_RAMP_EPOCHS,
    init_state=None):
    has_rel = len(idx_rel) > 0
    has_amb = len(idx_amb) > 0
    has_unk = len(idx_unk) > 0
    has_align = len(idx_gate) > 0
    if not (has_rel or has_amb or has_unk or has_align):
        return None

    adapter = ResidualAdapter(dim=Z_replay.shape[1], bottleneck=bottleneck, scale=scale).to(device)
    if init_state is not None:
        adapter.load_state_dict(init_state)

    teacher = None
    if use_ema:
        teacher = ResidualAdapter(dim=Z_replay.shape[1], bottleneck=bottleneck, scale=scale).to(device)
        teacher.load_state_dict(adapter.state_dict())
        teacher.eval()
        for p in teacher.parameters():
            p.requires_grad = False

    fc = nn.Linear(fc_layer.in_features, fc_layer.out_features, bias=True).to(device)
    fc.load_state_dict(fc_layer.state_dict())
    fc.eval()
    for p in fc.parameters():
        p.requires_grad = False

    ds_s = EmbDatasetWeightedHard(Z_replay, y_replay, None)
    dl_s = DataLoader(ds_s, batch_size=batch, shuffle=True, drop_last=False)
    it_s = cycle_loader(dl_s)

    dl_rel = it_rel = None
    if has_rel:
        if w_rel is None or len(w_rel) == 0:
            w_rel = np.ones((len(idx_rel)), dtype=np.float32)
        ds_rel = EmbDatasetWeightedSoft(Z_target[idx_rel], q_seed[idx_rel], w_rel)
        dl_rel = DataLoader(ds_rel, batch_size=batch, shuffle=True, drop_last=False)
        it_rel = cycle_loader(dl_rel)

    dl_amb = it_amb = None
    if has_amb:
        if w_amb is None or len(w_amb) == 0:
            w_amb = np.ones((len(idx_amb)), dtype=np.float32) * AMBIG_WEIGHT_FLOOR
        ds_amb = EmbDatasetWeightedSoft(Z_target[idx_amb], q_seed[idx_amb], w_amb)
        dl_amb = DataLoader(ds_amb, batch_size=batch, shuffle=True, drop_last=False)
        it_amb = cycle_loader(dl_amb)

    dl_align = it_align = None
    if has_align:
        ds_align = EmbOnlyDataset(Z_target[idx_gate])
        dl_align = DataLoader(ds_align, batch_size=batch, shuffle=True, drop_last=False)
        it_align = cycle_loader(dl_align)

    dl_unk = it_unk = None
    if has_unk:
        ds_unk = EmbOnlyDataset(Z_target[idx_unk])
        dl_unk = DataLoader(ds_unk, batch_size=batch, shuffle=True, drop_last=False)
        it_unk = cycle_loader(dl_unk)

    opt = optim.Adam(adapter.parameters(), lr=lr, weight_decay=wd)
    ce = nn.CrossEntropyLoss()
    mse = nn.MSELoss()

    steps_per_epoch = max(
        len(dl_s),
        len(dl_rel) if dl_rel is not None else 0,
        len(dl_amb) if dl_amb is not None else 0,
        len(dl_align) if dl_align is not None else 0,
        len(dl_unk) if dl_unk is not None else 0,
        1)

    src_proto_bank = None
    if proto_bank is not None:
        src_proto_bank = torch.tensor(proto_bank, dtype=torch.float32, device=device)
        src_proto_bank = F.normalize(src_proto_bank, dim=1)

    for ep in range(max(1, int(epochs))):
        adapter.train()
        lam_align_ep = float(lambda_align) * min(1.0, (ep + 1) / max(1, int(epochs)))
        if ep < int(unknown_warmup_epochs):
            unknown_scale = 0.0
        else:
            unknown_scale = min(1.0, float(ep - int(unknown_warmup_epochs) + 1) / max(1, int(unknown_ramp_epochs)))

        for _ in range(steps_per_epoch):
            zs, ys, _ = next(it_s)
            zs, ys = zs.to(device), ys.to(device)

            opt.zero_grad()
            zs2 = adapter(zs)
            logits_s = fc(zs2)
            with torch.no_grad():
                logits_s_ref = fc(zs)

            loss = lambda_replay_ce * ce(logits_s, ys)
            loss = loss + lambda_replay_anchor * mse(zs2, zs)

            if lambda_src_logit > 0:
                loss = loss + float(lambda_src_logit) * symmetric_kl_from_logits(logits_s, logits_s_ref).mean()

            if src_proto_bank is not None and lambda_src_proto > 0:
                tgt_src = src_proto_bank[ys]
                loss = loss + float(lambda_src_proto) * ((F.normalize(zs2, dim=1) - tgt_src) ** 2).mean()

            if has_rel:
                zrel, qrel_seed, wrel_t = next(it_rel)
                zrel = zrel.to(device); qrel_seed = qrel_seed.to(device); wrel_t = wrel_t.to(device)
                zrel2 = adapter(zrel)
                logits_rel = fc(zrel2)
                teacher_logits = None
                if teacher is not None:
                    with torch.no_grad():
                        teacher_logits = fc(teacher(zrel))
                qrel = mixed_teacher_targets(qrel_seed, teacher_logits, blend=teacher_blend)
                loss = loss + float(lambda_rel_sup) * (soft_ce_from_probs(logits_rel, qrel) * wrel_t).mean()
                if proto_bank is not None and lambda_proto > 0:
                    tgt_rel = proto_targets_torch(proto_bank, q=qrel, device=device)
                    loss = loss + float(lambda_proto) * ((F.normalize(zrel2, dim=1) - tgt_rel) ** 2).mean()

            if has_amb:
                zamb, qamb_seed, wamb_t = next(it_amb)
                zamb = zamb.to(device); qamb_seed = qamb_seed.to(device); wamb_t = wamb_t.to(device)
                zamb2 = adapter(zamb)
                logits_amb = fc(zamb2)
                teacher_logits = None
                if teacher is not None:
                    with torch.no_grad():
                        teacher_logits = fc(teacher(zamb))
                qamb = mixed_teacher_targets(qamb_seed, teacher_logits, blend=teacher_blend)
                loss = loss + float(lambda_amb_sup) * (soft_ce_from_probs(logits_amb, qamb) * wamb_t).mean()
                if lambda_cons > 0:
                    logits_amb_aug = fc(adapter(embedding_jitter(zamb)))
                    loss = loss + float(lambda_cons) * (symmetric_kl_from_logits(logits_amb, logits_amb_aug) * wamb_t).mean()
                if lambda_ent_min > 0:
                    loss = loss + float(lambda_ent_min) * (entropy_from_logits_torch(logits_amb) * wamb_t).mean()
                if proto_bank is not None and lambda_proto > 0:
                    tgt_amb = proto_targets_torch(proto_bank, q=qamb, device=device)
                    loss = loss + 0.5 * float(lambda_proto) * ((F.normalize(zamb2, dim=1) - tgt_amb) ** 2).mean()

            if has_align and lam_align_ep > 0:
                zt = next(it_align).to(device)
                zt2 = adapter(zt)
                loss = loss + lam_align_ep * mmd_rbf(zs2, zt2)

            if has_unk and unknown_scale > 0:
                zu = next(it_unk).to(device)
                zu2 = adapter(zu)
                logits_u = fc(zu2)
                if lambda_ent_max > 0:
                    uniform = torch.full_like(logits_u, 1.0 / logits_u.shape[1])
                    loss = loss + unknown_scale * float(lambda_ent_max) * F.kl_div(F.log_softmax(logits_u, dim=1), uniform, reduction="batchmean")
                if proto_bank is not None and lambda_u_repel > 0:
                    loss = loss + unknown_scale * float(lambda_u_repel) * proto_repulsion_loss(zu2, proto_bank, margin=UNKNOWN_REPEL_MARGIN)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(adapter.parameters(), ADAPT_GRAD_CLIP)
            opt.step()
            if teacher is not None:
                ema_update_(teacher, adapter, momentum=ema_momentum)

    return adapter



def adapt_three_bucket_v4(*args, **kwargs):
    return adapt_three_bucket_v5(*args, **kwargs)



def select_topk_class_balanced(indices, pseudo_y, scores, k_total, K, min_per_class=1):
    indices = np.asarray(indices, dtype=np.int64)
    if len(indices) == 0 or k_total <= 0:
        return np.zeros((0), dtype=np.int64)
    pseudo_y = np.asarray(pseudo_y, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float32)

    present = [k for k in range(int(K)) if np.any(pseudo_y[indices] == k)]
    if len(present) == 0:
        order = indices[np.argsort(-scores[indices])]
        return order[:min(len(order), int(k_total))].astype(np.int64)

    base = max(int(min_per_class), int(np.floor(float(k_total) / max(1, len(present)))))
    chosen = []
    for k in present:
        idx_k = indices[pseudo_y[indices] == k]
        if len(idx_k) == 0:
            continue
        order_k = idx_k[np.argsort(-scores[idx_k])]
        take = min(len(order_k), base)
        chosen.extend(order_k[:take].tolist())

    # fill the remaining budget globally by score
    chosen = np.array(sorted(set(chosen)), dtype=np.int64)
    if len(chosen) < int(k_total):
        rem = np.setdiff1d(indices, chosen, assume_unique=False)
        if len(rem) > 0:
            rem = rem[np.argsort(-scores[rem])]
            need = int(k_total) - len(chosen)
            chosen = np.concatenate([chosen, rem[:need]], axis=0)

    if len(chosen) > int(k_total):
        chosen = chosen[np.argsort(-scores[chosen])[:int(k_total)]]
    return chosen.astype(np.int64)


def build_three_bucket_partition_v6(
    K,
    Z_adapt,
    gate_pred,
    Sid_adapt,
    Sdom_adapt,
    P_logit_adapt,
    P_proto_adapt,
    P_knn_adapt,
    protos,
    tau_conf,
    min_keep=64):
    idx_gate = np.where(gate_pred == 1)[0]
    n = len(gate_pred)
    empty = np.zeros((0), dtype=np.int64)
    if len(idx_gate) == 0:
        aux = dict(
            bucket_score=np.zeros((n), dtype=np.float32),
            unknown_score=np.zeros((n), dtype=np.float32),
            P_seed=weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN]),
            threshold_reliable=float("nan"),
            threshold_unknown=float("nan"),
            w_rel=np.zeros((0), dtype=np.float32),
            w_amb=np.zeros((0), dtype=np.float32),
            w_weak=np.zeros((0), dtype=np.float32))
        info = dict(
            reliable_size=0, ambiguous_size=0, weak_size=0, unknown_size=0,
            reliable_keep=0.0, ambiguous_keep=0.0, weak_keep=0.0, unknown_keep=0.0,
            bucket_score_gate_mean=float("nan"))
        return empty, empty, empty, empty, aux, info

    P_tri = weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN])
    y_tri = np.argmax(P_tri, axis=1)
    y_logit = np.argmax(P_logit_adapt, axis=1)
    y_proto = np.argmax(P_proto_adapt, axis=1)
    y_knn = np.argmax(P_knn_adapt, axis=1)

    conf_tri = msp_from_probs(P_tri)
    margin_tri = top2_margin(P_tri)
    _, proto_margin, proto_dist = prototype_margin_dist(Z_adapt, protos)

    agree = ((y_logit == y_proto).astype(np.float32) + (y_logit == y_knn).astype(np.float32) + (y_proto == y_knn).astype(np.float32)) / 3.0
    sid_n = robust_unit_interval(Sid_adapt, idx_gate)
    sdom_src_n = 1.0 - robust_unit_interval(Sdom_adapt, idx_gate)
    conf_n = robust_unit_interval(conf_tri, idx_gate)
    margin_n = robust_unit_interval(margin_tri, idx_gate)
    proto_margin_n = robust_unit_interval(proto_margin, idx_gate)
    proto_dist_n = robust_unit_interval(proto_dist, idx_gate)

    bucket_score = (
        0.34 * sid_n +
        0.22 * conf_n +
        0.15 * agree +
        0.11 * margin_n +
        0.10 * proto_margin_n +
        0.08 * sdom_src_n -
        0.08 * proto_dist_n
    ).astype(np.float32)

    unknown_score = (
        0.42 * (1.0 - bucket_score) +
        0.24 * proto_dist_n +
        0.20 * (1.0 - conf_n) +
        0.14 * (1.0 - margin_n)
    ).astype(np.float32)

    ng = len(idx_gate)
    unk_gap = float(np.quantile(unknown_score[idx_gate], 0.90) - np.quantile(unknown_score[idx_gate], 0.60)) if ng >= 10 else 0.0
    unk_q = V6_UNKNOWN_KEEP_Q_HARD if unk_gap >= 0.08 else V6_UNKNOWN_KEEP_Q

    k_rel = min(ng, max(int(round(V6_RELIABLE_KEEP_Q * ng)), max(16, min_keep)))
    k_amb = min(max(0, ng - k_rel), max(int(round(V6_AMBIG_KEEP_Q * ng)), max(16, min_keep)))
    k_unk = min(max(0, ng - k_rel - k_amb), max(int(round(unk_q * ng)), max(8, min_keep // 3)))

    idx_rel = select_topk_class_balanced(idx_gate, y_tri, bucket_score, k_rel, K, min_per_class=1)

    rem1 = np.setdiff1d(idx_gate, idx_rel, assume_unique=False)
    idx_unk = rem1[np.argsort(-unknown_score[rem1])[:k_unk]] if len(rem1) > 0 and k_unk > 0 else np.zeros((0), dtype=np.int64)

    rem2 = np.setdiff1d(rem1, idx_unk, assume_unique=False)
    idx_amb = select_topk_class_balanced(rem2, y_tri, bucket_score, k_amb, K, min_per_class=1)

    idx_weak = np.setdiff1d(rem2, idx_amb, assume_unique=False)
    if len(idx_weak) < min(V6_WEAK_MIN_KEEP, len(rem2)):
        need = min(V6_WEAK_MIN_KEEP, len(rem2))
        order = rem2[np.argsort(-bucket_score[rem2])]
        keep = order[:need]
        idx_weak = np.setdiff1d(keep, idx_amb, assume_unique=False)

    thr_rel = float(np.min(bucket_score[idx_rel])) if len(idx_rel) > 0 else float("nan")
    thr_unk = float(np.min(unknown_score[idx_unk])) if len(idx_unk) > 0 else float("nan")

    support_idx = np.concatenate([idx_rel, idx_amb], axis=0) if (len(idx_rel) + len(idx_amb)) > 0 else idx_gate
    P_seed = refine_probs_multi_stage(Z_adapt, P_tri, protos, idx_support=support_idx, iters=max(REFINE_ITERS, 3), src_mix=0.80)

    if len(idx_amb) > 0:
        P_seed[idx_amb] = restrict_topk_probs(P_seed[idx_amb], k=max(AMBIG_TOPK, 2), sharpen=1.05)
    if len(idx_weak) > 0:
        P_seed[idx_weak] = restrict_topk_probs(P_seed[idx_weak], k=max(V6_WEAK_TOPK, 2), sharpen=V6_WEAK_SHARPEN)

    w_rel = np.asarray([], dtype=np.float32)
    if len(idx_rel) > 0:
        rel_score = 0.60 * bucket_score[idx_rel] + 0.40 * conf_tri[idx_rel]
        w_rel = V6_REL_WEIGHT_FLOOR + (1.0 - V6_REL_WEIGHT_FLOOR) * normalize_conf_weights(rel_score)

    w_amb = np.asarray([], dtype=np.float32)
    if len(idx_amb) > 0:
        amb_score = 0.55 * bucket_score[idx_amb] + 0.45 * conf_tri[idx_amb]
        w_amb = V6_AMB_WEIGHT_FLOOR + V6_AMB_WEIGHT_SCALE * normalize_conf_weights(amb_score)

    w_weak = np.asarray([], dtype=np.float32)
    if len(idx_weak) > 0:
        weak_score = 0.50 * bucket_score[idx_weak] + 0.50 * conf_tri[idx_weak]
        w_weak = V6_WEAK_WEIGHT_FLOOR + V6_WEAK_WEIGHT_SCALE * normalize_conf_weights(weak_score)

    aux = dict(
        bucket_score=bucket_score.astype(np.float32),
        unknown_score=unknown_score.astype(np.float32),
        P_seed=P_seed.astype(np.float32),
        threshold_reliable=thr_rel,
        threshold_unknown=thr_unk,
        w_rel=w_rel.astype(np.float32),
        w_amb=w_amb.astype(np.float32),
        w_weak=w_weak.astype(np.float32))
    info = dict(
        reliable_size=int(len(idx_rel)),
        ambiguous_size=int(len(idx_amb)),
        weak_size=int(len(idx_weak)),
        unknown_size=int(len(idx_unk)),
        reliable_keep=float(len(idx_rel) / max(1, len(gate_pred))),
        ambiguous_keep=float(len(idx_amb) / max(1, len(gate_pred))),
        weak_keep=float(len(idx_weak) / max(1, len(gate_pred))),
        unknown_keep=float(len(idx_unk) / max(1, len(gate_pred))),
        bucket_score_gate_mean=float(bucket_score[idx_gate].mean()),
        unknown_score_gate_mean=float(unknown_score[idx_gate].mean()),
        reliable_threshold=float(thr_rel),
        unknown_threshold=float(thr_unk),
        unknown_gap_q90_q60=float(unk_gap))
    return idx_rel.astype(np.int64), idx_amb.astype(np.int64), idx_weak.astype(np.int64), idx_unk.astype(np.int64), aux, info


def adapt_three_bucket_v6(
    fc_layer,
    Z_replay,
    y_replay,
    Z_target,
    idx_gate,
    idx_rel,
    idx_amb,
    idx_weak,
    idx_unk,
    q_seed,
    w_rel=None,
    w_amb=None,
    w_weak=None,
    proto_bank=None,
    idx_weak_cold=None,
    w_weak_cold=None,
    bottleneck=64,
    scale=0.3,
    epochs=24,
    lr=1e-3,
    wd=1e-4,
    batch=128,
    lambda_replay_ce=1.0,
    lambda_replay_anchor=1.0,
    lambda_rel_sup=1.0,
    lambda_amb_sup=V6_LAMBDA_AMB_SUP,
    lambda_weak_sup=V6_LAMBDA_WEAK_SUP,
    lambda_align=V6_LAMBDA_ALIGN,
    lambda_ent_min=V6_LAMBDA_ENT_MIN,
    lambda_ent_max=V6_LAMBDA_ENT_MAX,
    lambda_cons=V6_LAMBDA_CONS,
    lambda_proto=V6_LAMBDA_PROTO,
    lambda_src_proto=LAMBDA_SRC_PROTO,
    lambda_src_logit=LAMBDA_SRC_LOGIT,
    lambda_u_repel=V6_LAMBDA_UNKNOWN_REPEL,
    use_ema=True,
    ema_momentum=EMA_MOMENTUM,
    teacher_blend=EMA_TEACHER_BLEND,
    weak_teacher_blend=V6_WEAK_TEACHER_BLEND,
    weak_start_epoch=V6_WEAK_START_EPOCH,
    unknown_warmup_epochs=V6_UNKNOWN_WARMUP_EPOCHS,
    unknown_ramp_epochs=V6_UNKNOWN_RAMP_EPOCHS,
    init_state=None):
    has_rel = len(idx_rel) > 0
    has_amb = len(idx_amb) > 0
    has_weak = len(idx_weak) > 0
    has_unk = len(idx_unk) > 0
    has_align = len(idx_gate) > 0
    if not (has_rel or has_amb or has_weak or has_unk or has_align):
        return None

    adapter = ResidualAdapter(dim=Z_replay.shape[1], bottleneck=bottleneck, scale=scale).to(device)
    if init_state is not None:
        adapter.load_state_dict(init_state)

    teacher = None
    if use_ema:
        teacher = ResidualAdapter(dim=Z_replay.shape[1], bottleneck=bottleneck, scale=scale).to(device)
        teacher.load_state_dict(adapter.state_dict())
        teacher.eval()
        for p in teacher.parameters():
            p.requires_grad = False

    fc = nn.Linear(fc_layer.in_features, fc_layer.out_features, bias=True).to(device)
    fc.load_state_dict(fc_layer.state_dict())
    fc.eval()
    for p in fc.parameters():
        p.requires_grad = False

    ds_s = EmbDatasetWeightedHard(Z_replay, y_replay, None)
    dl_s = DataLoader(ds_s, batch_size=batch, shuffle=True, drop_last=False)
    it_s = cycle_loader(dl_s)

    dl_rel = it_rel = None
    if has_rel:
        if w_rel is None or len(w_rel) == 0:
            w_rel = np.ones((len(idx_rel)), dtype=np.float32)
        ds_rel = EmbDatasetWeightedSoft(Z_target[idx_rel], q_seed[idx_rel], w_rel)
        dl_rel = DataLoader(ds_rel, batch_size=batch, shuffle=True, drop_last=False)
        it_rel = cycle_loader(dl_rel)

    dl_amb = it_amb = None
    if has_amb:
        if w_amb is None or len(w_amb) == 0:
            w_amb = np.ones((len(idx_amb)), dtype=np.float32) * V6_AMB_WEIGHT_FLOOR
        ds_amb = EmbDatasetWeightedSoft(Z_target[idx_amb], q_seed[idx_amb], w_amb)
        dl_amb = DataLoader(ds_amb, batch_size=batch, shuffle=True, drop_last=False)
        it_amb = cycle_loader(dl_amb)

    dl_weak = it_weak = None
    if has_weak:
        if w_weak is None or len(w_weak) == 0:
            w_weak = np.ones((len(idx_weak)), dtype=np.float32) * V6_WEAK_WEIGHT_FLOOR
        ds_weak = EmbDatasetWeightedSoft(Z_target[idx_weak], q_seed[idx_weak], w_weak)
        dl_weak = DataLoader(ds_weak, batch_size=batch, shuffle=True, drop_last=False)
        it_weak = cycle_loader(dl_weak)

    dl_align = it_align = None
    if has_align:
        ds_align = EmbOnlyDataset(Z_target[idx_gate])
        dl_align = DataLoader(ds_align, batch_size=batch, shuffle=True, drop_last=False)
        it_align = cycle_loader(dl_align)

    dl_unk = it_unk = None
    if has_unk:
        ds_unk = EmbOnlyDataset(Z_target[idx_unk])
        dl_unk = DataLoader(ds_unk, batch_size=batch, shuffle=True, drop_last=False)
        it_unk = cycle_loader(dl_unk)

    opt = optim.Adam(adapter.parameters(), lr=lr, weight_decay=wd)
    ce = nn.CrossEntropyLoss()
    mse = nn.MSELoss()

    steps_per_epoch = max(
        len(dl_s),
        len(dl_rel) if dl_rel is not None else 0,
        len(dl_amb) if dl_amb is not None else 0,
        len(dl_weak) if dl_weak is not None else 0,
        len(dl_align) if dl_align is not None else 0,
        len(dl_unk) if dl_unk is not None else 0,
        1)

    src_proto_bank = None
    if proto_bank is not None:
        src_proto_bank = torch.tensor(proto_bank, dtype=torch.float32, device=device)
        src_proto_bank = F.normalize(src_proto_bank, dim=1)

    for ep in range(max(1, int(epochs))):
        adapter.train()
        lam_align_ep = float(lambda_align) * min(1.0, (ep + 1) / max(4, int(epochs) // 2))
        if ep < int(unknown_warmup_epochs):
            unknown_scale = 0.0
        else:
            unknown_scale = min(1.0, float(ep - int(unknown_warmup_epochs) + 1) / max(1, int(unknown_ramp_epochs)))

        for _ in range(steps_per_epoch):
            zs, ys, _ = next(it_s)
            zs, ys = zs.to(device), ys.to(device)

            opt.zero_grad()
            zs2 = adapter(zs)
            logits_s = fc(zs2)
            with torch.no_grad():
                logits_s_ref = fc(zs)

            loss = lambda_replay_ce * ce(logits_s, ys)
            loss = loss + lambda_replay_anchor * mse(zs2, zs)

            if lambda_src_logit > 0:
                loss = loss + float(lambda_src_logit) * symmetric_kl_from_logits(logits_s, logits_s_ref).mean()

            if src_proto_bank is not None and lambda_src_proto > 0:
                tgt_src = src_proto_bank[ys]
                loss = loss + float(lambda_src_proto) * ((F.normalize(zs2, dim=1) - tgt_src) ** 2).mean()

            if has_rel:
                zrel, qrel_seed, wrel_t = next(it_rel)
                zrel = zrel.to(device); qrel_seed = qrel_seed.to(device); wrel_t = wrel_t.to(device)
                zrel2 = adapter(zrel)
                logits_rel = fc(zrel2)
                teacher_logits = None
                if teacher is not None:
                    with torch.no_grad():
                        teacher_logits = fc(teacher(zrel))
                qrel = mixed_teacher_targets(qrel_seed, teacher_logits, blend=teacher_blend)
                loss = loss + float(lambda_rel_sup) * (soft_ce_from_probs(logits_rel, qrel) * wrel_t).mean()
                if proto_bank is not None and lambda_proto > 0:
                    tgt_rel = proto_targets_torch(proto_bank, q=qrel, device=device)
                    loss = loss + float(lambda_proto) * ((F.normalize(zrel2, dim=1) - tgt_rel) ** 2).mean()

            if has_amb:
                zamb, qamb_seed, wamb_t = next(it_amb)
                zamb = zamb.to(device); qamb_seed = qamb_seed.to(device); wamb_t = wamb_t.to(device)
                zamb2 = adapter(zamb)
                logits_amb = fc(zamb2)
                teacher_logits = None
                if teacher is not None:
                    with torch.no_grad():
                        teacher_logits = fc(teacher(zamb))
                qamb = mixed_teacher_targets(qamb_seed, teacher_logits, blend=teacher_blend)
                loss = loss + float(lambda_amb_sup) * (soft_ce_from_probs(logits_amb, qamb) * wamb_t).mean()
                if lambda_cons > 0:
                    logits_amb_aug = fc(adapter(embedding_jitter(zamb)))
                    loss = loss + float(lambda_cons) * (symmetric_kl_from_logits(logits_amb, logits_amb_aug) * wamb_t).mean()
                if lambda_ent_min > 0:
                    loss = loss + float(lambda_ent_min) * (entropy_from_logits_torch(logits_amb) * wamb_t).mean()

            if has_weak and ep >= int(weak_start_epoch):
                zweak, qweak_seed, wweak_t = next(it_weak)
                zweak = zweak.to(device); qweak_seed = qweak_seed.to(device); wweak_t = wweak_t.to(device)
                zweak2 = adapter(zweak)
                logits_weak = fc(zweak2)
                teacher_logits = None
                if teacher is not None:
                    with torch.no_grad():
                        teacher_logits = fc(teacher(zweak))
                qweak = mixed_teacher_targets(qweak_seed, teacher_logits, blend=weak_teacher_blend)
                loss = loss + float(lambda_weak_sup) * (soft_ce_from_probs(logits_weak, qweak) * wweak_t).mean()
                if lambda_cons > 0:
                    logits_weak_aug = fc(adapter(embedding_jitter(zweak)))
                    loss = loss + 0.5 * float(lambda_cons) * (symmetric_kl_from_logits(logits_weak, logits_weak_aug) * wweak_t).mean()

            if has_align and lam_align_ep > 0:
                zt = next(it_align).to(device)
                zt2 = adapter(zt)
                loss = loss + lam_align_ep * mmd_rbf(zs2, zt2)

            if has_unk and unknown_scale > 0:
                zu = next(it_unk).to(device)
                zu2 = adapter(zu)
                logits_u = fc(zu2)
                if lambda_ent_max > 0:
                    uniform = torch.full_like(logits_u, 1.0 / logits_u.shape[1])
                    loss = loss + unknown_scale * float(lambda_ent_max) * F.kl_div(F.log_softmax(logits_u, dim=1), uniform, reduction="batchmean")
                if proto_bank is not None and lambda_u_repel > 0:
                    loss = loss + unknown_scale * float(lambda_u_repel) * proto_repulsion_loss(zu2, proto_bank, margin=UNKNOWN_REPEL_MARGIN)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(adapter.parameters(), ADAPT_GRAD_CLIP)
            opt.step()
            if teacher is not None:
                ema_update_(teacher, adapter, momentum=ema_momentum)

    return teacher if (teacher is not None) else adapter



def build_three_bucket_partition_v7(
    K,
    Z_adapt,
    gate_pred,
    Sid_adapt,
    Sdom_adapt,
    P_logit_adapt,
    P_proto_adapt,
    P_knn_adapt,
    protos,
    tau_conf,
    min_keep=64):
    idx_gate = np.where(gate_pred == 1)[0]
    n = len(gate_pred)
    empty = np.zeros((0), dtype=np.int64)
    if len(idx_gate) == 0:
        aux = dict(
            bucket_score=np.zeros((n), dtype=np.float32),
            unknown_score=np.zeros((n), dtype=np.float32),
            P_seed=weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN]),
            threshold_reliable=float("nan"),
            threshold_unknown=float("nan"),
            w_rel=np.zeros((0), dtype=np.float32),
            w_amb=np.zeros((0), dtype=np.float32),
            w_weak=np.zeros((0), dtype=np.float32))
        info = dict(
            reliable_size=0, ambiguous_size=0, weak_size=0, unknown_size=0,
            reliable_keep=0.0, ambiguous_keep=0.0, weak_keep=0.0, unknown_keep=0.0,
            bucket_score_gate_mean=float("nan"))
        return empty, empty, empty, empty, aux, info

    P_tri = weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN])
    y_tri = np.argmax(P_tri, axis=1)
    y_logit = np.argmax(P_logit_adapt, axis=1)
    y_proto = np.argmax(P_proto_adapt, axis=1)
    y_knn = np.argmax(P_knn_adapt, axis=1)

    conf_tri = msp_from_probs(P_tri)
    margin_tri = top2_margin(P_tri)
    _, proto_margin, proto_dist = prototype_margin_dist(Z_adapt, protos)

    agree = ((y_logit == y_proto).astype(np.float32) + (y_logit == y_knn).astype(np.float32) + (y_proto == y_knn).astype(np.float32)) / 3.0
    sid_n = robust_unit_interval(Sid_adapt, idx_gate)
    sdom_src_n = 1.0 - robust_unit_interval(Sdom_adapt, idx_gate)
    conf_n = robust_unit_interval(conf_tri, idx_gate)
    margin_n = robust_unit_interval(margin_tri, idx_gate)
    proto_margin_n = robust_unit_interval(proto_margin, idx_gate)
    proto_dist_n = robust_unit_interval(proto_dist, idx_gate)

    bucket_score = (
        0.36 * sid_n +
        0.22 * conf_n +
        0.14 * agree +
        0.12 * margin_n +
        0.10 * proto_margin_n +
        0.08 * sdom_src_n -
        0.08 * proto_dist_n
    ).astype(np.float32)

    unknown_score = (
        0.40 * (1.0 - bucket_score) +
        0.24 * proto_dist_n +
        0.20 * (1.0 - conf_n) +
        0.16 * (1.0 - margin_n)
    ).astype(np.float32)

    ng = len(idx_gate)
    unk_gap = float(np.quantile(unknown_score[idx_gate], 0.90) - np.quantile(unknown_score[idx_gate], 0.60)) if ng >= 10 else 0.0
    unk_q = V7_UNKNOWN_KEEP_Q_HARD if unk_gap >= 0.08 else V7_UNKNOWN_KEEP_Q

    k_rel = min(ng, max(int(round(V7_RELIABLE_KEEP_Q * ng)), max(16, min_keep // 2)))
    k_amb = min(max(0, ng - k_rel), max(int(round(V7_AMBIG_KEEP_Q * ng)), max(16, min_keep // 2)))
    k_unk = min(max(0, ng - k_rel - k_amb), max(int(round(unk_q * ng)), max(8, min_keep // 4)))

    idx_rel = select_topk_class_balanced(idx_gate, y_tri, bucket_score, k_rel, K, min_per_class=1)
    rem1 = np.setdiff1d(idx_gate, idx_rel, assume_unique=False)
    idx_unk = rem1[np.argsort(-unknown_score[rem1])[:k_unk]] if len(rem1) > 0 and k_unk > 0 else np.zeros((0), dtype=np.int64)

    rem2 = np.setdiff1d(rem1, idx_unk, assume_unique=False)
    idx_amb = select_topk_class_balanced(rem2, y_tri, bucket_score, k_amb, K, min_per_class=1)

    idx_weak = np.setdiff1d(rem2, idx_amb, assume_unique=False)
    if len(idx_weak) < min(V7_WEAK_MIN_KEEP, len(rem2)):
        need = min(V7_WEAK_MIN_KEEP, len(rem2))
        order = rem2[np.argsort(-bucket_score[rem2])]
        keep = order[:need]
        idx_weak = np.setdiff1d(keep, idx_amb, assume_unique=False)

    support_idx = np.concatenate([idx_rel, idx_amb], axis=0) if (len(idx_rel) + len(idx_amb)) > 0 else idx_gate
    P_seed = refine_probs_multi_stage(Z_adapt, P_tri, protos, idx_support=support_idx, iters=max(REFINE_ITERS, 3), src_mix=0.82)
    if len(idx_amb) > 0:
        P_seed[idx_amb] = restrict_topk_probs(P_seed[idx_amb], k=max(AMBIG_TOPK, 2), sharpen=1.02)
    if len(idx_weak) > 0:
        P_seed[idx_weak] = restrict_topk_probs(P_seed[idx_weak], k=max(V7_WEAK_TOPK, 2), sharpen=V7_WEAK_SHARPEN)

    w_rel = np.asarray([], dtype=np.float32)
    if len(idx_rel) > 0:
        rel_score = 0.62 * bucket_score[idx_rel] + 0.38 * conf_tri[idx_rel]
        w_rel = V7_REL_WEIGHT_FLOOR + (1.0 - V7_REL_WEIGHT_FLOOR) * normalize_conf_weights(rel_score)

    w_amb = np.asarray([], dtype=np.float32)
    if len(idx_amb) > 0:
        amb_score = 0.58 * bucket_score[idx_amb] + 0.42 * conf_tri[idx_amb]
        w_amb = V7_AMB_WEIGHT_FLOOR + V7_AMB_WEIGHT_SCALE * normalize_conf_weights(amb_score)

    w_weak = np.asarray([], dtype=np.float32)
    if len(idx_weak) > 0:
        weak_score = 0.50 * bucket_score[idx_weak] + 0.50 * conf_tri[idx_weak]
        w_weak = V7_WEAK_WEIGHT_FLOOR + V7_WEAK_WEIGHT_SCALE * normalize_conf_weights(weak_score)

    aux = dict(
        bucket_score=bucket_score.astype(np.float32),
        unknown_score=unknown_score.astype(np.float32),
        P_seed=P_seed.astype(np.float32),
        threshold_reliable=float(np.min(bucket_score[idx_rel])) if len(idx_rel) > 0 else float("nan"),
        threshold_unknown=float(np.min(unknown_score[idx_unk])) if len(idx_unk) > 0 else float("nan"),
        w_rel=w_rel.astype(np.float32),
        w_amb=w_amb.astype(np.float32),
        w_weak=w_weak.astype(np.float32))
    info = dict(
        reliable_size=int(len(idx_rel)),
        ambiguous_size=int(len(idx_amb)),
        weak_size=int(len(idx_weak)),
        unknown_size=int(len(idx_unk)),
        reliable_keep=float(len(idx_rel) / max(1, len(gate_pred))),
        ambiguous_keep=float(len(idx_amb) / max(1, len(gate_pred))),
        weak_keep=float(len(idx_weak) / max(1, len(gate_pred))),
        unknown_keep=float(len(idx_unk) / max(1, len(gate_pred))),
        bucket_score_gate_mean=float(bucket_score[idx_gate].mean()),
        unknown_score_gate_mean=float(unknown_score[idx_gate].mean()))
    return idx_rel.astype(np.int64), idx_amb.astype(np.int64), idx_weak.astype(np.int64), idx_unk.astype(np.int64), aux, info


def adapt_three_bucket_v7(
    fc_layer,
    Z_replay,
    y_replay,
    Z_target,
    idx_gate,
    idx_rel,
    idx_amb,
    idx_weak,
    idx_unk,
    q_seed,
    w_rel=None,
    w_amb=None,
    w_weak=None,
    proto_bank=None,
    bucket_score=None,
    unknown_score=None,
    idx_weak_cold=None,
    w_weak_cold=None,
    bottleneck=64,
    scale=0.3,
    epochs=24,
    stage1_epochs=V7_STAGE1_EPOCHS,
    lr=1e-3,
    wd=1e-4,
    batch=128,
    lambda_replay_ce=1.0,
    lambda_replay_anchor=1.0,
    lambda_rel_sup=1.0,
    lambda_amb_sup=V7_LAMBDA_AMB_SUP,
    lambda_weak_sup=V7_LAMBDA_WEAK_SUP,
    lambda_align=V7_LAMBDA_ALIGN,
    lambda_ent_min=V7_LAMBDA_ENT_MIN,
    lambda_ent_max=V7_LAMBDA_ENT_MAX,
    lambda_cons=V7_LAMBDA_CONS,
    lambda_proto=V7_LAMBDA_PROTO,
    lambda_src_proto=LAMBDA_SRC_PROTO,
    lambda_src_logit=LAMBDA_SRC_LOGIT,
    lambda_u_repel=V7_LAMBDA_UNKNOWN_REPEL,
    use_ema=True,
    ema_momentum=EMA_MOMENTUM,
    teacher_blend=EMA_TEACHER_BLEND,
    promote_blend=V7_PROMOTE_BLEND,
    promote_rel_fraction=V7_PROMOTE_REL_FRACTION,
    promote_amb_fraction=V7_PROMOTE_AMB_FRACTION,
    promote_conf=V7_PROMOTE_CONF,
    promote_margin=V7_PROMOTE_MARGIN,
    unknown_warmup_epochs=V7_UNKNOWN_WARMUP_EPOCHS,
    unknown_ramp_epochs=V7_UNKNOWN_RAMP_EPOCHS,
    init_state=None):
    if (len(idx_gate) + len(idx_rel) + len(idx_amb) + len(idx_weak) + len(idx_unk)) == 0:
        return None, {}

    K = q_seed.shape[1]
    empty = np.zeros((0), dtype=np.int64)

    adapter = ResidualAdapter(dim=Z_replay.shape[1], bottleneck=bottleneck, scale=scale).to(device)
    if init_state is not None:
        adapter.load_state_dict(init_state)

    teacher = None
    if use_ema:
        teacher = ResidualAdapter(dim=Z_replay.shape[1], bottleneck=bottleneck, scale=scale).to(device)
        teacher.load_state_dict(adapter.state_dict())
        teacher.eval()
        for p in teacher.parameters():
            p.requires_grad = False

    fc = nn.Linear(fc_layer.in_features, fc_layer.out_features, bias=True).to(device)
    fc.load_state_dict(fc_layer.state_dict())
    fc.eval()
    for p in fc.parameters():
        p.requires_grad = False

    ds_s = EmbDatasetWeightedHard(Z_replay, y_replay, None)
    dl_s = DataLoader(ds_s, batch_size=batch, shuffle=True, drop_last=False)
    it_s = cycle_loader(dl_s)

    opt = optim.Adam(adapter.parameters(), lr=lr, weight_decay=wd)
    ce = nn.CrossEntropyLoss()
    mse = nn.MSELoss()

    src_proto_bank = normalize_vec_rows_np(proto_bank).astype(np.float32) if proto_bank is not None else None
    if bucket_score is None:
        bucket_score = np.zeros((len(Z_target)), dtype=np.float32)

    q_curr = normalize_rows(q_seed.astype(np.float32).copy())
    idx_rel_curr = np.asarray(idx_rel, dtype=np.int64)
    idx_amb_curr = np.asarray(idx_amb, dtype=np.int64)
    idx_weak_curr = np.asarray(idx_weak, dtype=np.int64)
    idx_unk_curr = np.asarray(idx_unk, dtype=np.int64)

    w_rel_curr = np.ones((len(idx_rel_curr)), dtype=np.float32) if (w_rel is None or len(w_rel) == 0) else np.asarray(w_rel, dtype=np.float32)
    w_amb_curr = np.ones((len(idx_amb_curr)), dtype=np.float32) * V7_AMB_WEIGHT_FLOOR if (w_amb is None or len(w_amb) == 0) else np.asarray(w_amb, dtype=np.float32)
    w_weak_curr = np.ones((len(idx_weak_curr)), dtype=np.float32) * V7_WEAK_WEIGHT_FLOOR if (w_weak is None or len(w_weak) == 0) else np.asarray(w_weak, dtype=np.float32)

    def _make_soft_loader(indices, weights):
        if len(indices) == 0:
            return None, None
        ds = EmbDatasetWeightedSoft(Z_target[indices], q_curr[indices], weights)
        dl = DataLoader(ds, batch_size=batch, shuffle=True, drop_last=False)
        return dl, cycle_loader(dl)

    def _make_align_loader(indices):
        if len(indices) == 0:
            return None, None
        ds = EmbOnlyDataset(Z_target[indices])
        dl = DataLoader(ds, batch_size=batch, shuffle=True, drop_last=False)
        return dl, cycle_loader(dl)

    def _target_proto_from_reliable(adapter_ref):
        if src_proto_bank is None:
            return None
        proto_np = src_proto_bank.copy()
        if len(idx_rel_curr) == 0:
            return proto_np
        Z_rel2, _ = apply_adapter_and_fc(adapter_ref, fc, Z_target[idx_rel_curr], batch=batch)
        Z_rel2 = normalize_vec_rows_np(Z_rel2)
        q_rel = normalize_rows(q_curr[idx_rel_curr])
        mass = q_rel.sum(axis=0, keepdims=False)[:, None]
        valid = mass.squeeze(1) > 1e-6
        if np.any(valid):
            proto_est = (q_rel.T @ Z_rel2) / np.maximum(mass, 1e-6)
            proto_est = normalize_vec_rows_np(proto_est)
            proto_np[valid] = proto_est[valid]
        return proto_np.astype(np.float32)

    def _train_range(ep_start, ep_end, use_unknown=False, use_weak=False):
        nonlocal adapter, teacher, idx_rel_curr, idx_amb_curr, idx_weak_curr, idx_unk_curr, w_rel_curr, w_amb_curr, w_weak_curr
        dl_rel, it_rel = _make_soft_loader(idx_rel_curr, w_rel_curr)
        dl_amb, it_amb = _make_soft_loader(idx_amb_curr, w_amb_curr)
        dl_weak, it_weak = _make_soft_loader(idx_weak_curr, w_weak_curr)
        dl_align, it_align = _make_align_loader(np.asarray(idx_gate, dtype=np.int64))
        dl_unk, it_unk = _make_align_loader(idx_unk_curr if use_unknown else empty)

        steps_per_epoch = max(
            len(dl_s),
            len(dl_rel) if dl_rel is not None else 0,
            len(dl_amb) if dl_amb is not None else 0,
            len(dl_weak) if (dl_weak is not None and use_weak) else 0,
            len(dl_align) if dl_align is not None else 0,
            len(dl_unk) if dl_unk is not None else 0,
            1)

        for ep in range(int(ep_start), int(ep_end)):
            adapter.train()
            lam_align_ep = float(lambda_align) * min(1.0, (ep + 1) / max(6, int(epochs) // 2))
            if ep < int(unknown_warmup_epochs):
                unknown_scale = 0.0
            else:
                unknown_scale = min(1.0, float(ep - int(unknown_warmup_epochs) + 1) / max(1, int(unknown_ramp_epochs)))

            proto_np_ep = _target_proto_from_reliable(teacher if teacher is not None else adapter)

            for _ in range(steps_per_epoch):
                zs, ys, _ = next(it_s)
                zs, ys = zs.to(device), ys.to(device)

                opt.zero_grad()
                zs2 = adapter(zs)
                logits_s = fc(zs2)
                with torch.no_grad():
                    logits_s_ref = fc(zs)

                loss = float(lambda_replay_ce) * ce(logits_s, ys)
                loss = loss + float(lambda_replay_anchor) * mse(zs2, zs)

                if lambda_src_logit > 0:
                    loss = loss + float(lambda_src_logit) * symmetric_kl_from_logits(logits_s, logits_s_ref).mean()

                if src_proto_bank is not None and lambda_src_proto > 0:
                    tgt_src = torch.tensor(src_proto_bank[ys.detach().cpu().numpy()], dtype=torch.float32, device=device)
                    loss = loss + float(lambda_src_proto) * ((F.normalize(zs2, dim=1) - tgt_src) ** 2).mean()

                if dl_rel is not None:
                    zrel, qrel_seed, wrel_t = next(it_rel)
                    zrel = zrel.to(device); qrel_seed = qrel_seed.to(device); wrel_t = wrel_t.to(device)
                    zrel2 = adapter(zrel)
                    logits_rel = fc(zrel2)
                    teacher_logits = None
                    if teacher is not None:
                        with torch.no_grad():
                            teacher_logits = fc(teacher(zrel))
                    qrel = mixed_teacher_targets(qrel_seed, teacher_logits, blend=teacher_blend)
                    loss = loss + float(lambda_rel_sup) * (soft_ce_from_probs(logits_rel, qrel) * wrel_t).mean()
                    if proto_np_ep is not None and lambda_proto > 0:
                        tgt_rel = proto_targets_torch(proto_np_ep, q=qrel.detach().cpu().numpy(), device=device)
                        loss = loss + float(lambda_proto) * ((F.normalize(zrel2, dim=1) - tgt_rel) ** 2).mean()

                if dl_amb is not None:
                    zamb, qamb_seed, wamb_t = next(it_amb)
                    zamb = zamb.to(device); qamb_seed = qamb_seed.to(device); wamb_t = wamb_t.to(device)
                    zamb2 = adapter(zamb)
                    logits_amb = fc(zamb2)
                    teacher_logits = None
                    if teacher is not None:
                        with torch.no_grad():
                            teacher_logits = fc(teacher(zamb))
                    qamb = mixed_teacher_targets(qamb_seed, teacher_logits, blend=teacher_blend)
                    loss = loss + float(lambda_amb_sup) * (soft_ce_from_probs(logits_amb, qamb) * wamb_t).mean()
                    if lambda_cons > 0:
                        logits_amb_aug = fc(adapter(embedding_jitter(zamb)))
                        loss = loss + 0.75 * float(lambda_cons) * (symmetric_kl_from_logits(logits_amb, logits_amb_aug) * wamb_t).mean()
                    if lambda_ent_min > 0:
                        loss = loss + float(lambda_ent_min) * (entropy_from_logits_torch(logits_amb) * wamb_t).mean()

                if use_weak and dl_weak is not None:
                    zweak, _, wweak_t = next(it_weak)
                    zweak = zweak.to(device); wweak_t = wweak_t.to(device)
                    logits_weak = fc(adapter(zweak))
                    weak_loss = 0.0
                    if teacher is not None:
                        with torch.no_grad():
                            logits_teacher = fc(teacher(zweak))
                        weak_loss = weak_loss + 0.5 * (symmetric_kl_from_logits(logits_weak, logits_teacher) * wweak_t).mean()
                    logits_weak_aug = fc(adapter(embedding_jitter(zweak)))
                    weak_loss = weak_loss + 0.5 * (symmetric_kl_from_logits(logits_weak, logits_weak_aug) * wweak_t).mean()
                    loss = loss + float(max(lambda_cons, 1e-8)) * weak_loss
                    if lambda_ent_min > 0:
                        loss = loss + 0.25 * float(lambda_ent_min) * (entropy_from_logits_torch(logits_weak) * wweak_t).mean()

                if dl_align is not None and lam_align_ep > 0:
                    zt = next(it_align).to(device)
                    zt2 = adapter(zt)
                    loss = loss + lam_align_ep * mmd_rbf(zs2, zt2)

                if use_unknown and dl_unk is not None and unknown_scale > 0:
                    zu = next(it_unk).to(device)
                    zu2 = adapter(zu)
                    logits_u = fc(zu2)
                    if lambda_ent_max > 0:
                        uniform = torch.full_like(logits_u, 1.0 / logits_u.shape[1])
                        loss = loss + unknown_scale * float(lambda_ent_max) * F.kl_div(F.log_softmax(logits_u, dim=1), uniform, reduction="batchmean")
                    if proto_np_ep is not None and lambda_u_repel > 0:
                        loss = loss + unknown_scale * float(lambda_u_repel) * proto_repulsion_loss(zu2, proto_np_ep, margin=UNKNOWN_REPEL_MARGIN)

                loss.backward()
                opt.step()
                if teacher is not None:
                    ema_update_(teacher, adapter, momentum=ema_momentum)

    def _promote():
        nonlocal idx_rel_curr, idx_amb_curr, idx_weak_curr, idx_unk_curr, q_curr, w_rel_curr, w_amb_curr, w_weak_curr
        adapter_eval = teacher if teacher is not None else adapter
        Z2_all, logits_all = apply_adapter_and_fc(adapter_eval, fc, Z_target, batch=batch)
        P_model = softmax_np(logits_all / max(1e-6, TEACHER_TEMP))
        P_mix = normalize_rows((1.0 - float(promote_blend)) * q_curr + float(promote_blend) * P_model)

        proto_np = _target_proto_from_reliable(adapter_eval)
        proto_for_score = src_proto_bank if proto_np is None else proto_np
        y_proto, proto_margin, proto_dist = prototype_margin_dist(Z2_all, proto_for_score)

        pred_seed = np.argmax(q_curr, axis=1)
        pred_mix = np.argmax(P_mix, axis=1)
        conf_mix = msp_from_probs(P_mix)
        margin_mix = top2_margin(P_mix)

        idx_eval = np.asarray(idx_gate, dtype=np.int64) if len(idx_gate) > 0 else np.arange(len(Z_target))
        conf_n = robust_unit_interval(conf_mix, idx_eval)
        margin_n = robust_unit_interval(margin_mix, idx_eval)
        proto_margin_n = robust_unit_interval(proto_margin, idx_eval)
        proto_dist_n = robust_unit_interval(proto_dist, idx_eval)
        bucket_n = robust_unit_interval(bucket_score, idx_eval) if np.any(np.isfinite(bucket_score)) else np.zeros_like(conf_n)

        stable = (pred_mix == pred_seed).astype(np.float32)
        proto_agree = (pred_mix == y_proto).astype(np.float32)
        promote_score = (
            0.30 * conf_n +
            0.20 * margin_n +
            0.16 * stable +
            0.16 * proto_agree +
            0.10 * proto_margin_n +
            0.08 * bucket_n -
            0.08 * proto_dist_n
        ).astype(np.float32)
        unknown_model = (
            0.36 * (1.0 - conf_n) +
            0.22 * proto_dist_n +
            0.18 * (1.0 - margin_n) +
            0.14 * (1.0 - stable) +
            0.10 * (1.0 - proto_agree)
        ).astype(np.float32)

        idx_known_pool = np.setdiff1d(np.asarray(idx_gate, dtype=np.int64), idx_unk_curr, assume_unique=False)
        idx_candidate = np.setdiff1d(idx_known_pool, idx_rel_curr, assume_unique=False)

        rel_mask = (
            (conf_mix >= float(promote_conf)) &
            (margin_mix >= float(promote_margin)) &
            (pred_mix == pred_seed) &
            (pred_mix == y_proto)
        )
        idx_rel_cand = idx_candidate[rel_mask[idx_candidate]]
        k_rel_extra = min(
            len(idx_rel_cand),
            max(int(round(float(promote_rel_fraction) * max(1, len(idx_candidate)))), int(V7_PROMOTE_MIN_REL))) if len(idx_rel_cand) > 0 else 0
        idx_rel_extra = select_topk_class_balanced(idx_rel_cand, pred_mix, promote_score, k_rel_extra, K, min_per_class=1) if k_rel_extra > 0 else empty
        idx_rel_new = np.unique(np.concatenate([idx_rel_curr, idx_rel_extra], axis=0)).astype(np.int64)

        rem = np.setdiff1d(idx_known_pool, idx_rel_new, assume_unique=False)
        amb_mask = (
            (conf_mix >= float(V7_AMB_CONF)) |
            ((margin_mix >= float(V7_AMB_MARGIN)) & ((pred_mix == pred_seed) | (pred_mix == y_proto)))
        )
        idx_amb_cand = rem[amb_mask[rem]]
        target_amb = max(
            len(idx_amb_curr),
            int(round(float(promote_amb_fraction) * max(1, len(idx_known_pool)))),
            int(V7_PROMOTE_MIN_AMB))
        idx_amb_new = select_topk_class_balanced(idx_amb_cand, pred_mix, promote_score, min(len(idx_amb_cand), target_amb), K, min_per_class=1) if len(idx_amb_cand) > 0 else empty

        rem2 = np.setdiff1d(rem, idx_amb_new, assume_unique=False)
        k_unk_new = min(
            max(0, len(rem2) // 8),
            max(0, int(round(V7_UNKNOWN_KEEP_Q * max(1, len(idx_gate)))))
        )
        idx_unk_extra = rem2[np.argsort(-unknown_model[rem2])[:k_unk_new]] if k_unk_new > 0 else empty
        idx_unk_new = np.unique(np.concatenate([idx_unk_curr, idx_unk_extra], axis=0)).astype(np.int64)

        idx_weak_new = np.setdiff1d(np.asarray(idx_gate, dtype=np.int64), np.unique(np.concatenate([idx_rel_new, idx_amb_new, idx_unk_new], axis=0)), assume_unique=False)
        if len(idx_weak_new) < min(V7_WEAK_MIN_KEEP, len(idx_gate)):
            keep_order = np.asarray(idx_gate, dtype=np.int64)[np.argsort(-promote_score[np.asarray(idx_gate, dtype=np.int64)])]
            keep_extra = keep_order[:min(V7_WEAK_MIN_KEEP, len(keep_order))]
            idx_weak_new = np.unique(np.concatenate([idx_weak_new, keep_extra], axis=0))
            idx_weak_new = np.setdiff1d(idx_weak_new, np.unique(np.concatenate([idx_rel_new, idx_amb_new, idx_unk_new], axis=0)), assume_unique=False)

        q_curr = normalize_rows(q_curr.copy())
        if len(idx_rel_new) > 0:
            q_curr[idx_rel_new] = normalize_rows(0.30 * q_curr[idx_rel_new] + 0.70 * P_mix[idx_rel_new])
        if len(idx_amb_new) > 0:
            q_curr[idx_amb_new] = restrict_topk_probs(P_mix[idx_amb_new], k=max(AMBIG_TOPK, 2), sharpen=1.02)
        if len(idx_weak_new) > 0:
            q_curr[idx_weak_new] = restrict_topk_probs(P_mix[idx_weak_new], k=max(V7_WEAK_TOPK, 2), sharpen=V7_WEAK_SHARPEN)

        if len(idx_rel_new) > 0:
            rel_score = 0.65 * promote_score[idx_rel_new] + 0.35 * conf_mix[idx_rel_new]
            w_rel_curr = V7_REL_WEIGHT_FLOOR + (1.0 - V7_REL_WEIGHT_FLOOR) * normalize_conf_weights(rel_score)
        else:
            w_rel_curr = np.zeros((0), dtype=np.float32)

        if len(idx_amb_new) > 0:
            amb_score = 0.60 * promote_score[idx_amb_new] + 0.40 * conf_mix[idx_amb_new]
            w_amb_curr = V7_AMB_WEIGHT_FLOOR + V7_AMB_WEIGHT_SCALE * normalize_conf_weights(amb_score)
        else:
            w_amb_curr = np.zeros((0), dtype=np.float32)

        if len(idx_weak_new) > 0:
            weak_score = 0.55 * promote_score[idx_weak_new] + 0.45 * conf_mix[idx_weak_new]
            w_weak_curr = V7_WEAK_WEIGHT_FLOOR + V7_WEAK_WEIGHT_SCALE * normalize_conf_weights(weak_score)
        else:
            w_weak_curr = np.zeros((0), dtype=np.float32)

        promo_info = dict(
            promoted_reliable=int(len(np.setdiff1d(idx_rel_new, idx_rel_curr, assume_unique=False))),
            promoted_ambiguous=int(len(np.setdiff1d(idx_amb_new, idx_amb_curr, assume_unique=False))),
            stage2_reliable_size=int(len(idx_rel_new)),
            stage2_ambiguous_size=int(len(idx_amb_new)),
            stage2_weak_size=int(len(idx_weak_new)),
            stage2_unknown_size=int(len(idx_unk_new)),
            promote_score_gate_mean=float(promote_score[idx_eval].mean()) if len(idx_eval) > 0 else float("nan"))

        idx_rel_curr = idx_rel_new.astype(np.int64)
        idx_amb_curr = idx_amb_new.astype(np.int64)
        idx_weak_curr = idx_weak_new.astype(np.int64)
        idx_unk_curr = idx_unk_new.astype(np.int64)
        return promo_info

    total_epochs = max(1, int(epochs))
    stage1_epochs = int(max(1, min(int(stage1_epochs), total_epochs - 1 if total_epochs > 1 else 1)))
    _train_range(0, stage1_epochs, use_unknown=False, use_weak=False)
    promo_info = _promote()
    _train_range(stage1_epochs, total_epochs, use_unknown=True, use_weak=True)

    info = dict(
        reliable_size=int(len(idx_rel)),
        ambiguous_size=int(len(idx_amb)),
        weak_size=int(len(idx_weak)),
        unknown_size=int(len(idx_unk)),
        reliable_keep=float(len(idx_rel) / max(1, len(idx_gate))),
        ambiguous_keep=float(len(idx_amb) / max(1, len(idx_gate))),
        weak_keep=float(len(idx_weak) / max(1, len(idx_gate))),
        unknown_keep=float(len(idx_unk) / max(1, len(idx_gate))),
        stage1_epochs=int(stage1_epochs))
    info.update(promo_info)
    return (teacher if (teacher is not None) else adapter), info




def _concat_unique_idx(*parts):
    arrs = [np.asarray(p, dtype=np.int64) for p in parts if p is not None and len(p) > 0]
    if len(arrs) == 0:
        return np.zeros((0), dtype=np.int64)
    return np.unique(np.concatenate(arrs, axis=0)).astype(np.int64)


def build_dg_raincoat_partition(
    K,
    Z_adapt,
    gate_pred,
    Sid_adapt,
    Sdom_adapt,
    P_logit_adapt,
    P_proto_adapt,
    P_knn_adapt,
    protos,
    tau_conf,
    min_keep=64):
    idx_gate = np.where(gate_pred == 1)[0]
    n = len(gate_pred)
    empty = np.zeros((0), dtype=np.int64)
    if len(idx_gate) == 0:
        aux = dict(
            common_score=np.zeros((n), dtype=np.float32),
            rescue_score=np.zeros((n), dtype=np.float32),
            unknown_score=np.zeros((n), dtype=np.float32),
            P_seed=weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN]),
            w_rel=np.zeros((0), dtype=np.float32),
            w_amb=np.zeros((0), dtype=np.float32),
            support_idx=np.zeros((0), dtype=np.int64),
            support_weight=np.zeros((0), dtype=np.float32),
            rescue_idx=np.zeros((0), dtype=np.int64))
        info = dict(
            reliable_size=0, ambiguous_size=0, unknown_size=0, rescue_size=0, support_size=0,
            reliable_keep=0.0, ambiguous_keep=0.0, unknown_keep=0.0,
            common_score_gate_mean=float("nan"), unknown_score_gate_mean=float("nan"))
        return empty, empty, empty, aux, info

    P_tri = weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN])
    y_tri = np.argmax(P_tri, axis=1)
    y_logit = np.argmax(P_logit_adapt, axis=1)
    y_proto = np.argmax(P_proto_adapt, axis=1)
    y_knn = np.argmax(P_knn_adapt, axis=1)
    conf_tri = msp_from_probs(P_tri)
    margin_tri = top2_margin(P_tri)
    _, proto_margin, proto_dist = prototype_margin_dist(Z_adapt, protos)

    agree = ((y_logit == y_proto).astype(np.float32) + (y_logit == y_knn).astype(np.float32) + (y_proto == y_knn).astype(np.float32)) / 3.0
    sid_n = robust_unit_interval(Sid_adapt, idx_gate)
    sdom_n = robust_unit_interval(Sdom_adapt, idx_gate)
    conf_n = robust_unit_interval(conf_tri, idx_gate)
    margin_n = robust_unit_interval(margin_tri, idx_gate)
    proto_margin_n = robust_unit_interval(proto_margin, idx_gate)
    proto_dist_n = robust_unit_interval(proto_dist, idx_gate)

    common_score = (
        0.32 * sid_n +
        0.18 * sdom_n +
        0.15 * conf_n +
        0.10 * margin_n +
        0.09 * agree +
        0.09 * proto_margin_n +
        0.07 * (1.0 - proto_dist_n)
    ).astype(np.float32)

    rescue_score = (
        0.24 * sid_n +
        0.22 * sdom_n +
        0.18 * conf_n +
        0.12 * margin_n +
        0.10 * agree +
        0.08 * proto_margin_n +
        0.06 * (1.0 - proto_dist_n)
    ).astype(np.float32)

    unknown_score = (
        0.42 * (1.0 - sid_n) +
        0.18 * proto_dist_n +
        0.14 * (1.0 - conf_n) +
        0.10 * (1.0 - margin_n) +
        0.08 * (1.0 - agree) +
        0.08 * (1.0 - sdom_n)
    ).astype(np.float32)

    ng = len(idx_gate)
    k_rel = min(ng, max(int(round(DG_REL_KEEP_Q * ng)), max(16, min_keep // 2)))
    k_amb = min(max(0, ng - k_rel), max(int(round(DG_AMB_KEEP_Q * ng)), max(16, min_keep // 3)))
    k_unk = min(max(0, ng - k_rel - k_amb), max(int(round(DG_UNK_KEEP_Q * ng)), max(8, min_keep // 5)))

    idx_rel = select_topk_class_balanced(idx_gate, y_tri, common_score, k_rel, K, min_per_class=1)
    rem1 = np.setdiff1d(idx_gate, idx_rel, assume_unique=False)

    high_sdom_thr = float(np.quantile(sdom_n[idx_gate], DG_RESCUE_SDOM_Q)) if ng >= 10 else 0.58
    rescue_mask = (
        (sid_n >= float(np.quantile(sid_n[idx_gate], 0.48)) if ng >= 10 else sid_n >= 0.55) &
        (conf_tri >= max(float(tau_conf) * 0.92, float(DG_RESCUE_CONF))) &
        (margin_tri >= float(DG_RESCUE_MARGIN)) &
        (proto_margin_n >= float(np.quantile(proto_margin_n[idx_gate], 0.32)) if ng >= 10 else proto_margin_n >= 0.35) &
        (sdom_n >= high_sdom_thr)
    )
    rescue_candidates = np.intersect1d(rem1, idx_gate[rescue_mask[idx_gate]], assume_unique=False)

    extreme_unk_thr = float(np.quantile(unknown_score[idx_gate], DG_UNKNOWN_EXTREME_Q)) if ng >= 10 else float(np.max(unknown_score[idx_gate]))
    hard_unk_candidates = np.setdiff1d(rem1[unknown_score[rem1] >= extreme_unk_thr], rescue_candidates, assume_unique=False)
    if len(hard_unk_candidates) < k_unk:
        ranked_unk = rem1[np.argsort(-unknown_score[rem1])]
        ranked_unk = np.setdiff1d(ranked_unk, rescue_candidates, assume_unique=False)
        idx_unk = ranked_unk[:k_unk]
    else:
        idx_unk = hard_unk_candidates[np.argsort(-unknown_score[hard_unk_candidates])[:k_unk]]

    rem2 = np.setdiff1d(rem1, idx_unk, assume_unique=False)
    rescue_rank = 0.56 * rescue_score + 0.32 * common_score + 0.12 * conf_tri
    k_rescue = min(len(rem2), max(8, int(round(0.08 * ng))))
    rescue_pool = np.intersect1d(rem2, rescue_candidates, assume_unique=False)
    idx_rescue = select_topk_class_balanced(
        rescue_pool,
        y_tri,
        rescue_rank,
        min(k_rescue, len(rescue_pool)),
        K,
        min_per_class=0) if len(rescue_pool) > 0 else np.zeros((0), dtype=np.int64)

    rem3 = np.setdiff1d(rem2, idx_rescue, assume_unique=False)
    amb_score = 0.70 * common_score + 0.20 * rescue_score + 0.10 * conf_tri
    idx_amb_main = select_topk_class_balanced(rem3, y_tri, amb_score, k_amb, K, min_per_class=1) if len(rem3) > 0 else np.zeros((0), dtype=np.int64)
    idx_amb = _concat_unique_idx(idx_rescue, idx_amb_main)

    support_idx = _concat_unique_idx(idx_rel, idx_amb)
    support_idx = support_idx if len(support_idx) > 0 else idx_rel if len(idx_rel) > 0 else idx_gate
    P_seed = refine_probs_multi_stage(Z_adapt, P_tri, protos, idx_support=support_idx, iters=max(REFINE_ITERS, 3), src_mix=0.76)
    if len(idx_amb) > 0:
        P_seed[idx_amb] = restrict_topk_probs(P_seed[idx_amb], k=max(2, AMBIG_TOPK), sharpen=1.00)

    support_score = np.maximum(common_score[support_idx], 0.95 * rescue_score[support_idx]) - 0.18 * unknown_score[support_idx]
    med = float(np.median(support_score)) if len(support_score) > 0 else 0.0
    std = float(np.std(support_score)) if len(support_score) > 0 else 1.0
    support_weight = DG_SUPPORT_WEIGHT_FLOOR + DG_SUPPORT_WEIGHT_SCALE * sigmoid_np((support_score - med) / max(1e-6, std + 1e-6))
    if len(idx_rescue) > 0:
        rescue_local = np.nonzero(np.isin(support_idx, idx_rescue))[0]
        support_weight[rescue_local] = np.maximum(support_weight[rescue_local], 0.48)
    if len(idx_rel) > 0:
        rel_local = np.nonzero(np.isin(support_idx, idx_rel))[0]
        support_weight[rel_local] = np.maximum(support_weight[rel_local], 0.88)

    w_rel = np.asarray([], dtype=np.float32)
    if len(idx_rel) > 0:
        rel_score = 0.72 * common_score[idx_rel] + 0.28 * conf_tri[idx_rel]
        w_rel = 0.88 + 0.12 * normalize_conf_weights(rel_score)
    w_amb = np.asarray([], dtype=np.float32)
    if len(idx_amb) > 0:
        amb_score_loc = 0.50 * np.maximum(common_score[idx_amb], rescue_score[idx_amb]) + 0.50 * conf_tri[idx_amb]
        w_amb = 0.12 + 0.18 * normalize_conf_weights(amb_score_loc)

    aux = dict(
        common_score=common_score.astype(np.float32),
        rescue_score=rescue_score.astype(np.float32),
        unknown_score=unknown_score.astype(np.float32),
        P_seed=P_seed.astype(np.float32),
        w_rel=w_rel.astype(np.float32),
        w_amb=w_amb.astype(np.float32),
        support_idx=support_idx.astype(np.int64),
        support_weight=support_weight.astype(np.float32),
        rescue_idx=idx_rescue.astype(np.int64))
    info = dict(
        reliable_size=int(len(idx_rel)),
        ambiguous_size=int(len(idx_amb)),
        unknown_size=int(len(idx_unk)),
        rescue_size=int(len(idx_rescue)),
        support_size=int(len(support_idx)),
        reliable_keep=float(len(idx_rel) / max(1, len(gate_pred))),
        ambiguous_keep=float(len(idx_amb) / max(1, len(gate_pred))),
        unknown_keep=float(len(idx_unk) / max(1, len(gate_pred))),
        common_score_gate_mean=float(common_score[idx_gate].mean()),
        unknown_score_gate_mean=float(unknown_score[idx_gate].mean()))
    return idx_rel.astype(np.int64), idx_amb.astype(np.int64), idx_unk.astype(np.int64), aux, info


def adapt_dg_raincoat_lite(
    fc_layer,
    Z_replay,
    y_replay,
    Z_target,
    idx_gate,
    idx_rel,
    idx_amb,
    idx_unk,
    q_seed,
    w_rel=None,
    w_amb=None,
    protos=None,
    Sid_adapt=None,
    Sdom_adapt=None,
    bottleneck=64,
    scale=0.3,
    epochs_stage1=DG_STAGE1_EPOCHS,
    epochs_stage2=DG_STAGE2_EPOCHS,
    lr=1e-3,
    wd=1e-4,
    batch=128,
    lambda_replay_ce=1.0,
    lambda_replay_anchor=1.0,
    lambda_align_stage1=DG_LAMBDA_ALIGN_STAGE1,
    lambda_align_stage2=DG_LAMBDA_ALIGN_STAGE2,
    lambda_ent_min=DG_LAMBDA_ENT_MIN,
    lambda_ent_max=DG_LAMBDA_ENT_MAX,
    lambda_cons=DG_LAMBDA_CONS,
    lambda_proto=DG_LAMBDA_PROTO,
    lambda_u_repel=DG_LAMBDA_UNKNOWN_REPEL):
    idx_gate = np.asarray(idx_gate, dtype=np.int64)
    idx_rel = np.asarray(idx_rel, dtype=np.int64)
    idx_amb = np.asarray(idx_amb, dtype=np.int64)
    idx_unk = np.asarray(idx_unk, dtype=np.int64)
    if len(idx_gate) == 0:
        return None, {}

    K = q_seed.shape[1]
    q_seed = normalize_rows(q_seed.astype(np.float32))
    w_rel = np.ones((len(idx_rel)), dtype=np.float32) if (w_rel is None or len(w_rel) == 0) else np.asarray(w_rel, dtype=np.float32)
    w_amb = np.ones((len(idx_amb)), dtype=np.float32) * 0.22 if (w_amb is None or len(w_amb) == 0) else np.asarray(w_amb, dtype=np.float32)

    idx_sup0 = _concat_unique_idx(idx_rel, idx_amb)
    w_sup0 = np.concatenate([0.95 * w_rel, 0.55 * w_amb], axis=0) if len(idx_sup0) > 0 else np.ones((0), dtype=np.float32)

    warm = adapt_on_embeddings_generic(
        fc_layer=fc_layer,
        Z_replay=Z_replay, y_replay=y_replay,
        Z_sup=Z_target[idx_sup0] if len(idx_sup0) > 0 else None,
        q_sup=q_seed[idx_sup0] if len(idx_sup0) > 0 else None,
        w_sup=w_sup0 if len(idx_sup0) > 0 else None,
        Z_align=Z_target[idx_gate],
        Z_unrel=Z_target[idx_unk] if len(idx_unk) > 0 else None,
        proto_bank=protos,
        bottleneck=bottleneck, scale=scale, epochs=epochs_stage1,
        lr=lr, wd=wd, batch=batch,
        lambda_replay_ce=lambda_replay_ce, lambda_replay_anchor=lambda_replay_anchor,
        lambda_sup=1.0,
        lambda_align=lambda_align_stage1,
        lambda_ent_min=lambda_ent_min * 0.8,
        lambda_ent_max=lambda_ent_max * 0.35,
        lambda_cons=lambda_cons * 0.8,
        lambda_proto=lambda_proto * 0.8,
        dynamic_align=True,
        unknown_loss_mode=unknown_loss_mode,
        unknown_energy_margin=unknown_energy_margin)
    if warm is None:
        return None, {}

    Z_gate1, logits_gate1 = apply_adapter_and_fc(warm, fc_layer, Z_target[idx_gate], batch=512)
    P_logit1 = softmax_np(logits_gate1)
    P_proto1 = proto_probs_cosine(Z_gate1, protos, temp=PROTO_T, l2norm=True)
    P_mix1 = weighted_prob_fusion([P_logit1, P_proto1, q_seed[idx_gate]], [1.0, 1.0, 0.70])

    conf1 = msp_from_probs(P_mix1)
    margin1 = top2_margin(P_mix1)
    y1 = np.argmax(P_mix1, axis=1)
    y0 = np.argmax(q_seed[idx_gate], axis=1)
    y_logit1 = np.argmax(P_logit1, axis=1)
    y_proto1 = np.argmax(P_proto1, axis=1)
    stable_vote = (((y1 == y0).astype(np.float32) + (y1 == y_logit1).astype(np.float32) + (y1 == y_proto1).astype(np.float32)) / 3.0).astype(np.float32)
    stable_hard = (stable_vote >= float(cfg.get("stable_hard_thr", 2.0 / 3.0))).astype(np.float32)
    stable_used = stable_vote if version == "v12" else stable_hard

    sid_n = robust_unit_interval(Sid_adapt, idx_gate)[idx_gate] if Sid_adapt is not None else np.ones((len(idx_gate)), dtype=np.float32) * 0.5
    sdom_n = robust_unit_interval(Sdom_adapt, idx_gate)[idx_gate] if Sdom_adapt is not None else np.ones((len(idx_gate)), dtype=np.float32) * 0.5
    conf_n = robust_unit_interval(conf1, np.arange(len(idx_gate)))
    margin_n = robust_unit_interval(margin1, np.arange(len(idx_gate)))
    _, proto_margin1, proto_dist1 = prototype_margin_dist(Z_gate1, protos)
    proto_margin_n = robust_unit_interval(proto_margin1, np.arange(len(idx_gate)))
    proto_dist_n = robust_unit_interval(proto_dist1, np.arange(len(idx_gate)))

    Z0n = normalize_vec_rows_np(Z_target[idx_gate])
    Z1n = normalize_vec_rows_np(Z_gate1)
    move = np.sqrt(np.maximum(1e-12, np.sum((Z1n - Z0n) ** 2, axis=1)))
    move_n = robust_unit_interval(move, np.arange(len(idx_gate)))
    move_low_n = 1.0 - move_n

    common_score = (
        0.30 * sid_n +
        0.18 * sdom_n +
        0.14 * conf_n +
        0.08 * margin_n +
        0.10 * stable +
        0.08 * proto_margin_n +
        0.08 * move_low_n +
        0.04 * (1.0 - proto_dist_n)
    ).astype(np.float32)

    rescue_score = (
        0.22 * sid_n +
        0.22 * sdom_n +
        0.18 * conf_n +
        0.10 * margin_n +
        0.12 * stable +
        0.10 * proto_margin_n +
        0.06 * move_low_n
    ).astype(np.float32)

    unknown_score = (
        0.38 * (1.0 - sid_n) +
        0.18 * proto_dist_n +
        0.12 * (1.0 - conf_n) +
        0.08 * (1.0 - margin_n) +
        0.08 * move_n +
        0.08 * (1.0 - stable) +
        0.08 * (1.0 - sdom_n)
    ).astype(np.float32)

    ng = len(idx_gate)
    high_sdom_thr = float(np.quantile(sdom_n, DG_RESCUE_SDOM_Q)) if ng >= 10 else 0.58
    rescue_mask_local = (
        (sid_n >= float(np.quantile(sid_n, 0.48)) if ng >= 10 else sid_n >= 0.55) &
        (conf1 >= max(float(np.quantile(conf1, 0.52)) if ng >= 10 else 0.56, DG_RESCUE_CONF)) &
        (margin1 >= float(DG_RESCUE_MARGIN)) &
        (proto_margin_n >= float(np.quantile(proto_margin_n, 0.35)) if ng >= 10 else 0.35) &
        (sdom_n >= high_sdom_thr) &
        (stable_hard >= 1.0)
    )

    extreme_unk_thr = float(np.quantile(unknown_score, DG_UNKNOWN_EXTREME_Q)) if ng >= 10 else float(np.max(unknown_score))
    local_all = np.arange(ng, dtype=np.int64)
    local_unk = local_all[(unknown_score >= extreme_unk_thr) & (~rescue_mask_local)]
    idx_unk1 = idx_gate[local_unk]

    candidate_mask = (
        rescue_mask_local |
        (common_score >= (float(np.quantile(common_score, 0.46)) if ng >= 10 else 0.50)) |
        ((stable_hard >= 1.0) & (conf1 >= (float(np.quantile(conf1, 0.45)) if ng >= 10 else 0.52)) & (unknown_score <= (float(np.quantile(unknown_score, 0.72)) if ng >= 10 else 0.60)))
    ) & (unknown_score < extreme_unk_thr)
    support_local = local_all[candidate_mask]
    if len(support_local) < max(16, ng // 8):
        support_score_fallback = np.maximum(common_score, 0.95 * rescue_score) - 0.20 * unknown_score
        k_support = min(ng, max(16, ng // 6))
        support_local = np.argsort(-support_score_fallback)[:k_support]
        support_local = np.setdiff1d(support_local, local_unk, assume_unique=False)
    if len(support_local) == 0:
        support_local = np.setdiff1d(local_all, local_unk, assume_unique=False)
        if len(support_local) == 0:
            support_local = local_all.copy()
            idx_unk1 = np.zeros((0), dtype=np.int64)
    support_idx = idx_gate[support_local]

    q_stage2_local = refine_probs_multi_stage(
        Z_gate1,
        P_mix1,
        protos,
        idx_support=support_local,
        iters=max(REFINE_ITERS, 3),
        src_mix=0.76)
    q_stage2 = q_seed.copy()
    q_stage2[idx_gate] = q_stage2_local.astype(np.float32)

    support_score = np.maximum(common_score[support_local], 0.96 * rescue_score[support_local]) - 0.18 * unknown_score[support_local]
    med = float(np.median(support_score)) if len(support_score) > 0 else 0.0
    std = float(np.std(support_score)) if len(support_score) > 0 else 1.0
    support_weight = DG_SUPPORT_WEIGHT_FLOOR + DG_SUPPORT_WEIGHT_SCALE * sigmoid_np((support_score - med) / max(1e-6, std + 1e-6))
    support_weight = support_weight.astype(np.float32)

    rescue_local = support_local[rescue_mask_local[support_local]]
    if len(rescue_local) > 0:
        rescue_pos = np.nonzero(np.isin(support_local, rescue_local))[0]
        support_weight[rescue_pos] = np.maximum(support_weight[rescue_pos], 0.48)

    reliable_mask_local = common_score[support_local] >= max(float(np.quantile(common_score[support_local], 0.80)) if len(support_local) >= 10 else 0.72, 0.72)
    reliable_pos = np.nonzero(reliable_mask_local)[0]
    if len(reliable_pos) > 0:
        support_weight[reliable_pos] = np.maximum(support_weight[reliable_pos], 0.88)

    ambiguous_pos = np.nonzero((~reliable_mask_local) & (support_weight < 0.88))[0]
    if len(ambiguous_pos) > 0:
        support_weight[ambiguous_pos] = np.maximum(support_weight[ambiguous_pos], 0.12)

    if len(support_local) > 0:
        low_weight_mask = support_weight < 0.20
        if np.any(low_weight_mask):
            q_stage2[support_idx[low_weight_mask]] = restrict_topk_probs(q_stage2[support_idx[low_weight_mask]], k=2, sharpen=0.97)

    align_idx = np.setdiff1d(idx_gate, idx_unk1, assume_unique=False)
    if len(align_idx) == 0:
        align_idx = support_idx

    final_adapter = adapt_on_embeddings_generic(
        fc_layer=fc_layer,
        Z_replay=Z_replay, y_replay=y_replay,
        Z_sup=Z_target[support_idx] if len(support_idx) > 0 else None,
        q_sup=q_stage2[support_idx] if len(support_idx) > 0 else None,
        w_sup=support_weight if len(support_idx) > 0 else None,
        Z_align=Z_target[align_idx],
        Z_unrel=Z_target[idx_unk1] if len(idx_unk1) > 0 else None,
        proto_bank=protos,
        bottleneck=bottleneck, scale=scale, epochs=epochs_stage2,
        lr=lr, wd=wd, batch=batch,
        lambda_replay_ce=lambda_replay_ce, lambda_replay_anchor=lambda_replay_anchor,
        lambda_sup=1.0,
        lambda_align=lambda_align_stage2,
        lambda_ent_min=lambda_ent_min,
        lambda_ent_max=lambda_ent_max * 0.50,
        lambda_cons=lambda_cons,
        lambda_proto=lambda_proto,
        dynamic_align=True,
        init_state=clone_state_dict_cpu(warm),
        unknown_loss_mode=unknown_loss_mode,
        unknown_energy_margin=unknown_energy_margin)

    info = dict(
        reliable_size=int(np.sum(support_weight >= 0.88)),
        ambiguous_size=int(np.sum((support_weight >= 0.12) & (support_weight < 0.88))),
        unknown_size=int(len(idx_unk1)),
        rescue_size=int(len(rescue_local)),
        support_size=int(len(support_idx)),
        reliable_keep=float(np.sum(support_weight >= 0.88) / max(1, len(Z_target))),
        ambiguous_keep=float(np.sum((support_weight >= 0.12) & (support_weight < 0.88)) / max(1, len(Z_target))),
        unknown_keep=float(len(idx_unk1) / max(1, len(Z_target))),
        common_score_gate_mean=float(common_score.mean()) if len(common_score) > 0 else float("nan"),
        unknown_score_gate_mean=float(unknown_score.mean()) if len(unknown_score) > 0 else float("nan"),
        move_gate_mean=float(move.mean()) if len(move) > 0 else float("nan"),
        stable_pred_ratio=float(stable.mean()) if len(stable) > 0 else float("nan"),
        stage1_reliable_size=int(len(idx_rel)),
        stage1_ambiguous_size=int(len(idx_amb)),
        stage1_unknown_size=int(len(idx_unk)),
        stage2_reliable_size=int(np.sum(support_weight >= 0.88)),
        stage2_ambiguous_size=int(np.sum((support_weight >= 0.12) & (support_weight < 0.88))),
        stage2_unknown_size=int(len(idx_unk1)),
        rescued_common_size=int(len(rescue_local)))
    final_adapter = final_adapter if final_adapter is not None else warm
    final_adapter, guard_info = guard_adapter_with_fallback(final_adapter, warm, fc_layer, Z_replay, y_replay, Z_probe=Z_target[idx_gate], probe_name=f"dg_{version}")
    info.update(guard_info)
    return final_adapter, info


def build_three_bucket_partition_v8(
    K,
    Z_adapt,
    gate_pred,
    Sid_adapt,
    Sdom_adapt,
    P_logit_adapt,
    P_proto_adapt,
    P_knn_adapt,
    protos,
    tau_conf,
    min_keep=64):
    idx_gate = np.where(gate_pred == 1)[0]
    n = len(gate_pred)
    empty = np.zeros((0), dtype=np.int64)
    if len(idx_gate) == 0:
        aux = dict(
            bucket_score=np.zeros((n), dtype=np.float32),
            unknown_score=np.zeros((n), dtype=np.float32),
            P_seed=weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN]),
            w_rel=np.zeros((0), dtype=np.float32),
            w_amb=np.zeros((0), dtype=np.float32),
            w_weak=np.zeros((0), dtype=np.float32),
            idx_weak_cold=np.zeros((0), dtype=np.int64),
            w_weak_cold=np.zeros((0), dtype=np.float32),
            support_idx=np.zeros((0), dtype=np.int64))
        info = dict(
            reliable_size=0, ambiguous_size=0, weak_size=0, cold_weak_size=0, unknown_size=0,
            reliable_keep=0.0, ambiguous_keep=0.0, weak_keep=0.0, unknown_keep=0.0,
            bucket_score_gate_mean=float("nan"), unknown_score_gate_mean=float("nan"))
        return empty, empty, empty, empty, aux, info

    P_tri = weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN])
    y_tri = np.argmax(P_tri, axis=1)
    y_logit = np.argmax(P_logit_adapt, axis=1)
    y_proto = np.argmax(P_proto_adapt, axis=1)
    y_knn = np.argmax(P_knn_adapt, axis=1)
    conf_tri = msp_from_probs(P_tri)
    margin_tri = top2_margin(P_tri)
    _, proto_margin, proto_dist = prototype_margin_dist(Z_adapt, protos)

    agree = ((y_logit == y_proto).astype(np.float32) + (y_logit == y_knn).astype(np.float32) + (y_proto == y_knn).astype(np.float32)) / 3.0
    sid_n = robust_unit_interval(Sid_adapt, idx_gate)
    sdom_n = robust_unit_interval(Sdom_adapt, idx_gate)
    conf_n = robust_unit_interval(conf_tri, idx_gate)
    margin_n = robust_unit_interval(margin_tri, idx_gate)
    proto_margin_n = robust_unit_interval(proto_margin, idx_gate)
    proto_dist_n = robust_unit_interval(proto_dist, idx_gate)

    bucket_score = (
        0.30 * sid_n +
        0.14 * sdom_n +
        0.18 * conf_n +
        0.12 * agree +
        0.12 * margin_n +
        0.10 * proto_margin_n +
        0.04 * (1.0 - proto_dist_n)
    ).astype(np.float32)
    unknown_score = (
        0.38 * (1.0 - sid_n) +
        0.18 * proto_dist_n +
        0.14 * (1.0 - conf_n) +
        0.12 * (1.0 - margin_n) +
        0.10 * (1.0 - agree) +
        0.08 * (1.0 - sdom_n)
    ).astype(np.float32)

    ng = len(idx_gate)
    unk_gap = float(np.quantile(unknown_score[idx_gate], 0.92) - np.quantile(unknown_score[idx_gate], 0.65)) if ng >= 10 else 0.0
    unk_q = V8_UNKNOWN_KEEP_Q_HARD if unk_gap >= 0.08 else V8_UNKNOWN_KEEP_Q

    rel_mask = (bucket_score >= float(np.quantile(bucket_score[idx_gate], max(0.50, 1.0 - V8_RELIABLE_KEEP_Q)))) & (conf_tri >= max(float(tau_conf), 0.55))
    idx_rel = select_idx_by_mask_with_fallback(idx_gate, rel_mask[idx_gate], bucket_score, min_keep=max(16, min_keep // 2))

    rem1 = np.setdiff1d(idx_gate, idx_rel, assume_unique=False)
    k_unk = min(len(rem1), max(int(round(unk_q * ng)), max(8, min_keep // 4)))
    idx_unk = rem1[np.argsort(-unknown_score[rem1])[:k_unk]] if len(rem1) > 0 and k_unk > 0 else np.zeros((0), dtype=np.int64)

    rem2 = np.setdiff1d(rem1, idx_unk, assume_unique=False)
    amb_mask = (bucket_score >= float(np.quantile(bucket_score[idx_gate], max(0.32, 1.0 - (V8_RELIABLE_KEEP_Q + V8_AMBIG_KEEP_Q))))) & (conf_tri >= max(float(tau_conf) * 0.85, 0.36))
    idx_amb = select_idx_by_mask_with_fallback(rem2, amb_mask[rem2] if len(rem2) > 0 else np.zeros((0), dtype=bool), bucket_score, min_keep=max(16, min_keep // 3))
    rem3 = np.setdiff1d(rem2, idx_amb, assume_unique=False)

    weak_strength = 0.60 * bucket_score[rem3] + 0.40 * conf_tri[rem3] - 0.25 * unknown_score[rem3] if len(rem3) > 0 else np.zeros((0), dtype=np.float32)
    if len(rem3) > 0:
        cold_thr = float(np.quantile(weak_strength, V81_COLD_WEAK_Q)) if len(rem3) >= 10 else float(np.median(weak_strength))
        idx_weak = rem3[weak_strength >= cold_thr]
        idx_weak_cold = rem3[weak_strength < cold_thr]
    else:
        idx_weak = np.zeros((0), dtype=np.int64)
        idx_weak_cold = np.zeros((0), dtype=np.int64)

    support_idx = _concat_unique_idx(idx_rel, idx_amb, idx_weak)
    support_idx = support_idx if len(support_idx) > 0 else idx_gate
    P_seed = refine_probs_multi_stage(Z_adapt, P_tri, protos, idx_support=support_idx, iters=max(REFINE_ITERS, 3), src_mix=0.82)
    if len(idx_amb) > 0:
        P_seed[idx_amb] = restrict_topk_probs(P_seed[idx_amb], k=max(2, AMBIG_TOPK), sharpen=1.01)
    if len(idx_weak) > 0:
        P_seed[idx_weak] = restrict_topk_probs(P_seed[idx_weak], k=max(2, V81_SEMIWEAK_TOPK), sharpen=0.99)
    if len(idx_weak_cold) > 0:
        P_seed[idx_weak_cold] = restrict_topk_probs(P_seed[idx_weak_cold], k=2, sharpen=0.96)

    w_rel = np.asarray([], dtype=np.float32)
    if len(idx_rel) > 0:
        rel_score = 0.65 * bucket_score[idx_rel] + 0.35 * conf_tri[idx_rel]
        w_rel = V8_REL_WEIGHT_FLOOR + (1.0 - V8_REL_WEIGHT_FLOOR) * normalize_conf_weights(rel_score)
    w_amb = np.asarray([], dtype=np.float32)
    if len(idx_amb) > 0:
        amb_score = 0.60 * bucket_score[idx_amb] + 0.40 * conf_tri[idx_amb]
        w_amb = V8_AMB_WEIGHT_FLOOR + V8_AMB_WEIGHT_SCALE * normalize_conf_weights(amb_score)
    w_weak = np.asarray([], dtype=np.float32)
    if len(idx_weak) > 0:
        weak_score = 0.60 * bucket_score[idx_weak] + 0.40 * conf_tri[idx_weak]
        w_weak = V8_WEAK_WEIGHT_FLOOR + (V8_WEAK_WEIGHT_SCALE + 0.04) * normalize_conf_weights(weak_score)
    w_weak_cold = np.asarray([], dtype=np.float32)
    if len(idx_weak_cold) > 0:
        cold_score = 0.60 * bucket_score[idx_weak_cold] + 0.40 * conf_tri[idx_weak_cold]
        w_weak_cold = V81_COLD_WEIGHT_FLOOR + V81_COLD_WEIGHT_SCALE * normalize_conf_weights(cold_score)

    aux = dict(
        bucket_score=bucket_score.astype(np.float32),
        unknown_score=unknown_score.astype(np.float32),
        P_seed=P_seed.astype(np.float32),
        w_rel=w_rel.astype(np.float32),
        w_amb=w_amb.astype(np.float32),
        w_weak=w_weak.astype(np.float32),
        idx_weak_cold=idx_weak_cold.astype(np.int64),
        w_weak_cold=w_weak_cold.astype(np.float32),
        support_idx=support_idx.astype(np.int64))
    info = dict(
        reliable_size=int(len(idx_rel)),
        ambiguous_size=int(len(idx_amb)),
        weak_size=int(len(idx_weak)),
        cold_weak_size=int(len(idx_weak_cold)),
        unknown_size=int(len(idx_unk)),
        reliable_keep=float(len(idx_rel) / max(1, len(gate_pred))),
        ambiguous_keep=float(len(idx_amb) / max(1, len(gate_pred))),
        weak_keep=float(len(idx_weak) / max(1, len(gate_pred))),
        unknown_keep=float(len(idx_unk) / max(1, len(gate_pred))),
        bucket_score_gate_mean=float(bucket_score[idx_gate].mean()),
        unknown_score_gate_mean=float(unknown_score[idx_gate].mean()))
    return idx_rel.astype(np.int64), idx_amb.astype(np.int64), idx_weak.astype(np.int64), idx_unk.astype(np.int64), aux, info


def adapt_three_bucket_v8(
    fc_layer,
    Z_replay,
    y_replay,
    Z_target,
    idx_gate,
    idx_rel,
    idx_amb,
    idx_weak,
    idx_unk,
    q_seed,
    w_rel=None,
    w_amb=None,
    w_weak=None,
    proto_bank=None,
    bucket_score=None,
    unknown_score=None,
    Sid_adapt=None,
    Sdom_adapt=None,
    idx_weak_cold=None,
    w_weak_cold=None,
    bottleneck=64,
    scale=0.3,
    stage0_epochs=V8_STAGE0_EPOCHS,
    refresh_epochs=V8_REFRESH_EPOCHS,
    refresh_rounds=V8_REFRESH_ROUNDS,
    lr=1e-3,
    wd=1e-4,
    batch=128,
    lambda_replay_ce=1.0,
    lambda_replay_anchor=1.0,
    lambda_align=V8_LAMBDA_ALIGN,
    lambda_ent_min=V8_LAMBDA_ENT_MIN,
    lambda_ent_max=V8_LAMBDA_ENT_MAX,
    lambda_cons=V8_LAMBDA_CONS,
    lambda_proto=V8_LAMBDA_PROTO,
    lambda_u_repel=V8_LAMBDA_UNKNOWN_REPEL):
    idx_gate = np.asarray(idx_gate, dtype=np.int64)
    if len(idx_gate) == 0:
        return None, {}

    K = q_seed.shape[1]
    q_curr = normalize_rows(q_seed.astype(np.float32).copy())
    idx_rel_curr = np.asarray(idx_rel, dtype=np.int64)
    idx_amb_curr = np.asarray(idx_amb, dtype=np.int64)
    idx_weak_curr = np.asarray(idx_weak, dtype=np.int64)
    idx_unk_curr = np.asarray(idx_unk, dtype=np.int64)
    idx_weak_cold_curr = np.asarray(idx_weak_cold if idx_weak_cold is not None else np.zeros((0), dtype=np.int64), dtype=np.int64)

    if bucket_score is None:
        bucket_score = np.zeros((len(Z_target)), dtype=np.float32)
    if unknown_score is None:
        unknown_score = np.zeros((len(Z_target)), dtype=np.float32)

    sid_n_full = robust_unit_interval(Sid_adapt, idx_gate)[idx_gate] if Sid_adapt is not None else np.ones((len(idx_gate)), dtype=np.float32) * 0.5
    sdom_n_full = robust_unit_interval(Sdom_adapt, idx_gate)[idx_gate] if Sdom_adapt is not None else np.ones((len(idx_gate)), dtype=np.float32) * 0.5

    w_rel_curr = np.ones((len(idx_rel_curr)), dtype=np.float32) if (w_rel is None or len(w_rel) == 0) else np.asarray(w_rel, dtype=np.float32)
    w_amb_curr = np.ones((len(idx_amb_curr)), dtype=np.float32) * V8_AMB_WEIGHT_FLOOR if (w_amb is None or len(w_amb) == 0) else np.asarray(w_amb, dtype=np.float32)
    w_weak_curr = np.ones((len(idx_weak_curr)), dtype=np.float32) * V8_WEAK_WEIGHT_FLOOR if (w_weak is None or len(w_weak) == 0) else np.asarray(w_weak, dtype=np.float32)
    w_weak_cold_curr = np.ones((len(idx_weak_cold_curr)), dtype=np.float32) * V81_COLD_WEIGHT_FLOOR if (w_weak_cold is None or len(w_weak_cold) == 0) else np.asarray(w_weak_cold, dtype=np.float32)

    state = None
    promo_total_rel = 0
    promo_total_amb = 0
    refresh_stats = []

    prev_pred = np.full((len(idx_gate)), -1, dtype=np.int64)
    stable_count = np.zeros((len(idx_gate)), dtype=np.int64)

    def _pack_sup(rel_idx, amb_idx, weak_idx):
        sup_idx = _concat_unique_idx(rel_idx, amb_idx, weak_idx)
        if len(sup_idx) == 0:
            return sup_idx, None, None
        q_sup = q_curr[sup_idx]
        parts = []
        if len(rel_idx) > 0:
            parts.append(w_rel_curr)
        if len(amb_idx) > 0:
            parts.append(w_amb_curr)
        if len(weak_idx) > 0:
            parts.append(w_weak_curr)
        w_sup = np.concatenate(parts, axis=0) if len(parts) > 0 else None
        return sup_idx, q_sup, w_sup

    sup0_idx, sup0_q, sup0_w = _pack_sup(idx_rel_curr, idx_amb_curr, np.zeros((0), dtype=np.int64))
    warm = adapt_on_embeddings_generic(
        fc_layer=fc_layer,
        Z_replay=Z_replay, y_replay=y_replay,
        Z_sup=Z_target[sup0_idx] if len(sup0_idx) > 0 else None,
        q_sup=sup0_q if sup0_q is not None else None,
        w_sup=sup0_w if sup0_w is not None else None,
        Z_align=Z_target[idx_gate],
        Z_unrel=Z_target[idx_unk_curr] if len(idx_unk_curr) > 0 else None,
        proto_bank=proto_bank,
        bottleneck=bottleneck, scale=scale, epochs=stage0_epochs,
        lr=lr, wd=wd, batch=batch,
        lambda_replay_ce=lambda_replay_ce, lambda_replay_anchor=lambda_replay_anchor,
        lambda_sup=1.0,
        lambda_align=lambda_align,
        lambda_ent_min=lambda_ent_min,
        lambda_ent_max=lambda_ent_max * 0.45,
        lambda_cons=lambda_cons * 0.75,
        lambda_proto=lambda_proto * 0.8,
        dynamic_align=True,
        init_state=state,
        unknown_loss_mode=unknown_loss_mode,
        unknown_energy_margin=unknown_energy_margin)
    if warm is None:
        return None, {}
    state = clone_state_dict_cpu(warm)

    final_adapter = warm
    for rr in range(int(max(1, refresh_rounds))):
        sup_idx_round, sup_q_round, sup_w_round = _pack_sup(idx_rel_curr, idx_amb_curr, idx_weak_curr)
        adapter_r = adapt_on_embeddings_generic(
            fc_layer=fc_layer,
            Z_replay=Z_replay, y_replay=y_replay,
            Z_sup=Z_target[sup_idx_round] if len(sup_idx_round) > 0 else None,
            q_sup=sup_q_round if len(sup_idx_round) > 0 else None,
            w_sup=sup_w_round if len(sup_idx_round) > 0 else None,
            Z_align=Z_target[np.setdiff1d(idx_gate, idx_unk_curr, assume_unique=False)],
            Z_unrel=Z_target[idx_unk_curr] if len(idx_unk_curr) > 0 else None,
            proto_bank=proto_bank,
            bottleneck=bottleneck, scale=scale, epochs=refresh_epochs,
            lr=lr, wd=wd, batch=batch,
            lambda_replay_ce=lambda_replay_ce, lambda_replay_anchor=lambda_replay_anchor,
            lambda_sup=1.0,
            lambda_align=lambda_align,
            lambda_ent_min=lambda_ent_min,
            lambda_ent_max=lambda_ent_max,
            lambda_cons=lambda_cons,
            lambda_proto=lambda_proto,
            dynamic_align=True,
            init_state=state,
            unknown_loss_mode=unknown_loss_mode,
            unknown_energy_margin=unknown_energy_margin)
        if adapter_r is None:
            break
        final_adapter = adapter_r
        state = clone_state_dict_cpu(adapter_r)

        Zg, logits_g = apply_adapter_and_fc(adapter_r, fc_layer, Z_target[idx_gate], batch=512)
        P_logit_g = softmax_np(logits_g)
        P_proto_g = proto_probs_cosine(Zg, proto_bank, temp=PROTO_T, l2norm=True)
        P_mix_g = weighted_prob_fusion([P_logit_g, P_proto_g, q_curr[idx_gate]], [1.0, 1.0, 0.80])

        conf_g = msp_from_probs(P_mix_g)
        margin_g = top2_margin(P_mix_g)
        pred_g = np.argmax(P_mix_g, axis=1)
        stable_now = (pred_g == prev_pred)
        stable_count = np.where(stable_now, stable_count + 1, 0)
        prev_pred = pred_g.copy()

        conf_n = robust_unit_interval(conf_g, np.arange(len(idx_gate)))
        margin_n = robust_unit_interval(margin_g, np.arange(len(idx_gate)))
        _, proto_margin_g, proto_dist_g = prototype_margin_dist(Zg, proto_bank)
        proto_margin_n = robust_unit_interval(proto_margin_g, np.arange(len(idx_gate)))
        proto_dist_n = robust_unit_interval(proto_dist_g, np.arange(len(idx_gate)))
        stability_n = np.clip(stable_count.astype(np.float32) / float(max(1, V8_STABLE_K_REL + 1)), 0.0, 1.0)

        class_counts = np.bincount(pred_g, minlength=K).astype(np.float32)
        class_counts[class_counts <= 0] = 1.0
        rarity = np.sqrt(class_counts.mean() / class_counts)
        rarity = rarity / np.maximum(1.0, rarity.max())

        promote_score = (
            0.24 * sid_n_full +
            0.12 * sdom_n_full +
            0.20 * conf_n +
            0.12 * margin_n +
            0.10 * proto_margin_n +
            0.14 * stability_n +
            0.08 * robust_unit_interval(bucket_score[idx_gate], np.arange(len(idx_gate)))
        ).astype(np.float32)
        rarity_boost = float(cfg.get("rarity_boost", 0.15))
        promote_score_pc = promote_score * ((1.0 - rarity_boost) + rarity_boost * rarity[pred_g])

        unknown_score_dyn = (
            0.34 * (1.0 - sid_n_full) +
            0.16 * proto_dist_n +
            0.14 * (1.0 - conf_n) +
            0.10 * (1.0 - margin_n) +
            0.14 * (1.0 - stability_n) +
            0.12 * (1.0 - sdom_n_full)
        ).astype(np.float32)

        unk_thr = float(np.quantile(unknown_score_dyn, max(0.82, 1.0 - V8_UNKNOWN_KEEP_Q))) if len(idx_gate) >= 10 else float(np.max(unknown_score_dyn))
        local_unk = np.where(unknown_score_dyn >= unk_thr)[0]
        idx_unk_curr = idx_gate[local_unk]

        local_all = np.arange(len(idx_gate), dtype=np.int64)
        local_nonunk = np.setdiff1d(local_all, local_unk, assume_unique=False)
        rel_budget = int(sum(max(1, int(np.ceil(V81_CLASS_REL_FRAC * c))) for c in class_counts if c > 0))
        amb_budget = int(sum(max(1, int(np.ceil(V81_CLASS_AMB_FRAC * c))) for c in class_counts if c > 0))

        cand_rel_local = np.intersect1d(
            local_nonunk,
            np.where(
                (stable_count >= int(V8_STABLE_K_REL)) &
                (conf_g >= float(V8_PROMOTE_CONF_REL)) &
                (margin_g >= float(V8_PROMOTE_MARGIN_REL))
            )[0],
            assume_unique=False)
        local_rel_sel = select_topk_class_balanced(
            cand_rel_local,
            pred_g,
            promote_score_pc,
            min(max(16, rel_budget), len(cand_rel_local)),
            K,
            min_per_class=1) if len(cand_rel_local) > 0 else np.zeros((0), dtype=np.int64)
        idx_rel_new = idx_gate[local_rel_sel]

        local_rem_after_rel = np.setdiff1d(local_nonunk, local_rel_sel, assume_unique=False)
        cand_amb_local = np.intersect1d(
            local_rem_after_rel,
            np.where(
                (stable_count >= int(V8_STABLE_K_AMB)) &
                (conf_g >= float(V8_PROMOTE_CONF_AMB)) &
                (margin_g >= float(V8_PROMOTE_MARGIN_AMB))
            )[0],
            assume_unique=False)
        local_amb_sel = select_topk_class_balanced(
            cand_amb_local,
            pred_g,
            promote_score_pc,
            min(max(16, amb_budget), len(cand_amb_local)),
            K,
            min_per_class=1) if len(cand_amb_local) > 0 else np.zeros((0), dtype=np.int64)
        idx_amb_new = idx_gate[local_amb_sel]

        local_rem_after_amb = np.setdiff1d(local_rem_after_rel, local_amb_sel, assume_unique=False)
        if len(local_rem_after_amb) > 0:
            semiweak_candidates = local_rem_after_amb[
                (conf_g[local_rem_after_amb] >= max(0.45, float(V8_PROMOTE_CONF_AMB) - 0.05)) &
                (margin_g[local_rem_after_amb] >= max(0.02, float(V8_PROMOTE_MARGIN_AMB) - 0.02))
            ]
            extra_amb = []
            current_amb_pred = pred_g[local_amb_sel] if len(local_amb_sel) > 0 else np.zeros((0), dtype=np.int64)
            for k in range(K):
                target_k = max(1, int(np.ceil(V81_CLASS_AMB_FRAC * class_counts[k]))) if class_counts[k] > 0 else 0
                cur_k = int(np.sum(current_amb_pred == k)) if len(current_amb_pred) > 0 else 0
                need_k = max(0, target_k - cur_k)
                if need_k <= 0:
                    continue
                cand_k = semiweak_candidates[pred_g[semiweak_candidates] == k]
                if len(cand_k) == 0:
                    continue
                score_k = promote_score_pc[cand_k]
                take_cap = int(cfg.get("extra_amb_cap", need_k))
                take_k = cand_k[np.argsort(-score_k)[:min(need_k, take_cap, len(cand_k))]]
                extra_amb.append(take_k)
            if len(extra_amb) > 0:
                extra_amb = np.concatenate(extra_amb, axis=0).astype(np.int64)
                local_amb_sel = np.unique(np.concatenate([local_amb_sel, extra_amb], axis=0)).astype(np.int64)
        local_rem_after_amb = np.setdiff1d(local_rem_after_rel, local_amb_sel, assume_unique=False)
        weak_strength = 0.56 * promote_score[local_rem_after_amb] + 0.26 * conf_n[local_rem_after_amb] + 0.18 * stability_n[local_rem_after_amb] - 0.16 * unknown_score_dyn[local_rem_after_amb] if len(local_rem_after_amb) > 0 else np.zeros((0), dtype=np.float32)
        if len(local_rem_after_amb) > 0:
            cold_thr = float(np.quantile(weak_strength, V81_COLD_WEAK_Q)) if len(local_rem_after_amb) >= 10 else float(np.median(weak_strength))
            local_weak_sel = local_rem_after_amb[weak_strength >= cold_thr]
            local_cold_sel = local_rem_after_amb[weak_strength < cold_thr]
        else:
            local_weak_sel = np.zeros((0), dtype=np.int64)
            local_cold_sel = np.zeros((0), dtype=np.int64)

        idx_rel_curr = idx_rel_new
        idx_amb_curr = idx_amb_new
        idx_weak_curr = idx_gate[local_weak_sel]
        idx_weak_cold_curr = idx_gate[local_cold_sel]

        support_curr = _concat_unique_idx(idx_rel_curr, idx_amb_curr, idx_weak_curr)
        local_support = np.nonzero(np.isin(idx_gate, support_curr))[0]
        if len(local_support) == 0:
            local_support = local_nonunk.copy()
            support_curr = idx_gate[local_support]

        q_local = refine_probs_multi_stage(Zg, P_mix_g, proto_bank, idx_support=local_support, iters=max(REFINE_ITERS, 3), src_mix=0.80)
        q_curr[idx_gate] = q_local.astype(np.float32)

        if len(idx_amb_curr) > 0:
            q_curr[idx_amb_curr] = restrict_topk_probs(q_curr[idx_amb_curr], k=max(2, AMBIG_TOPK), sharpen=1.01)
        if len(idx_weak_curr) > 0:
            q_curr[idx_weak_curr] = restrict_topk_probs(q_curr[idx_weak_curr], k=max(2, V81_SEMIWEAK_TOPK), sharpen=0.99)
        if len(idx_weak_cold_curr) > 0:
            q_curr[idx_weak_cold_curr] = restrict_topk_probs(q_curr[idx_weak_cold_curr], k=2, sharpen=0.96)

        if len(idx_rel_curr) > 0:
            rel_local = np.nonzero(np.isin(idx_gate, idx_rel_curr))[0]
            w_rel_curr = V8_REL_WEIGHT_FLOOR + (1.0 - V8_REL_WEIGHT_FLOOR) * normalize_conf_weights(0.60 * promote_score[rel_local] + 0.40 * conf_g[rel_local])
        else:
            w_rel_curr = np.zeros((0), dtype=np.float32)
        if len(idx_amb_curr) > 0:
            amb_local = np.nonzero(np.isin(idx_gate, idx_amb_curr))[0]
            w_amb_curr = V8_AMB_WEIGHT_FLOOR + (V8_AMB_WEIGHT_SCALE + 0.06) * normalize_conf_weights(0.55 * promote_score[amb_local] + 0.45 * conf_g[amb_local])
        else:
            w_amb_curr = np.zeros((0), dtype=np.float32)
        if len(idx_weak_curr) > 0:
            weak_local = np.nonzero(np.isin(idx_gate, idx_weak_curr))[0]
            w_weak_curr = V8_WEAK_WEIGHT_FLOOR + (V8_WEAK_WEIGHT_SCALE + 0.08) * normalize_conf_weights(0.55 * promote_score[weak_local] + 0.25 * conf_g[weak_local] + 0.20 * stability_n[weak_local])
        else:
            w_weak_curr = np.zeros((0), dtype=np.float32)
        if len(idx_weak_cold_curr) > 0:
            cold_local = np.nonzero(np.isin(idx_gate, idx_weak_cold_curr))[0]
            w_weak_cold_curr = V81_COLD_WEIGHT_FLOOR + V81_COLD_WEIGHT_SCALE * normalize_conf_weights(0.50 * promote_score[cold_local] + 0.50 * conf_g[cold_local])
        else:
            w_weak_cold_curr = np.zeros((0), dtype=np.float32)

        promo_total_rel += int(len(np.setdiff1d(idx_rel_curr, idx_rel, assume_unique=False)))
        promo_total_amb += int(len(np.setdiff1d(idx_amb_curr, idx_amb, assume_unique=False)))
        refresh_stats.append(dict(
            round=int(rr + 1),
            reliable_size=int(len(idx_rel_curr)),
            ambiguous_size=int(len(idx_amb_curr)),
            weak_size=int(len(idx_weak_curr)),
            cold_weak_size=int(len(idx_weak_cold_curr)),
            unknown_size=int(len(idx_unk_curr)),
            promote_score_gate_mean=float(promote_score.mean()),
            unknown_score_gate_mean=float(unknown_score_dyn.mean()),
            stability_mean=float(stability_n.mean())))

    info = dict(
        reliable_size=int(len(idx_rel_curr)),
        ambiguous_size=int(len(idx_amb_curr)),
        weak_size=int(len(idx_weak_curr)),
        cold_weak_size=int(len(idx_weak_cold_curr)),
        unknown_size=int(len(idx_unk_curr)),
        reliable_keep=float(len(idx_rel_curr) / max(1, len(Z_target))),
        ambiguous_keep=float(len(idx_amb_curr) / max(1, len(Z_target))),
        weak_keep=float(len(idx_weak_curr) / max(1, len(Z_target))),
        unknown_keep=float(len(idx_unk_curr) / max(1, len(Z_target))),
        promoted_reliable=float(promo_total_rel),
        promoted_ambiguous=float(promo_total_amb),
        stage2_reliable_size=int(len(idx_rel_curr)),
        stage2_ambiguous_size=int(len(idx_amb_curr)),
        stage2_weak_size=int(len(idx_weak_curr)),
        stage2_cold_weak_size=int(len(idx_weak_cold_curr)),
        stage2_unknown_size=int(len(idx_unk_curr)),
        promote_score_gate_mean=float(refresh_stats[-1]["promote_score_gate_mean"]) if len(refresh_stats) > 0 else float("nan"),
        stability_gate_mean=float(refresh_stats[-1]["stability_mean"]) if len(refresh_stats) > 0 else float("nan"),
        refresh_rounds=float(len(refresh_stats)))
    return final_adapter, info


def build_pseudo_method_bank(
    K,
    Z_adapt,
    gate_pred,
    y_true_adapt,
    P_logit_adapt,
    P_proto_adapt,
    P_knn_adapt,
    protos,
    tau_conf,
    Sid_adapt=None,
    Sdom_adapt=None,
    gate_pred_v4=None,
    p_known_adapt=None,
    p_drift_given_known_adapt=None,
    P_route_v4=None,
    min_keep=64):
    idx_gate = np.where(gate_pred == 1)[0]
    idx_all = np.arange(len(gate_pred))
    idx_oracle = np.where(y_true_adapt >= 0)[0]

    if Sid_adapt is None:
        Sid_adapt = np.zeros((len(gate_pred)), dtype=np.float32)
    if Sdom_adapt is None:
        Sdom_adapt = np.zeros((len(gate_pred)), dtype=np.float32)
    if gate_pred_v4 is None:
        gate_pred_v4 = gate_pred
    if p_known_adapt is None:
        p_known_adapt = np.where(gate_pred_v4 != 2, 0.80, 0.20).astype(np.float32)
    if p_drift_given_known_adapt is None:
        p_drift_given_known_adapt = np.where(gate_pred_v4 == 1, 0.80, 0.20).astype(np.float32)
    if P_route_v4 is None:
        _p_unknown = np.clip(1.0 - p_known_adapt, 1e-8, None)
        _p_drift = np.clip(p_known_adapt * p_drift_given_known_adapt, 1e-8, None)
        _p_stable = np.clip(p_known_adapt * (1.0 - p_drift_given_known_adapt), 1e-8, None)
        P_route_v4 = np.stack([_p_stable, _p_drift, _p_unknown], axis=1).astype(np.float32)
        P_route_v4 /= np.clip(P_route_v4.sum(axis=1, keepdims=True), 1e-8, None)

    P_tri = weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN])
    P_lp = weighted_prob_fusion([P_logit_adapt, P_proto_adapt], [W_LOGIT, W_PROTO])

    y_logit = np.argmax(P_logit_adapt, axis=1)
    y_proto = np.argmax(P_proto_adapt, axis=1)
    y_knn = np.argmax(P_knn_adapt, axis=1)
    y_tri = np.argmax(P_tri, axis=1)

    conf_logit = msp_from_probs(P_logit_adapt)
    conf_proto = msp_from_probs(P_proto_adapt)
    conf_knn = msp_from_probs(P_knn_adapt)
    conf_lp = msp_from_probs(P_lp)
    conf_tri = msp_from_probs(P_tri)

    mask_lp_agree = (y_logit == y_proto)
    mask_tri_agree = (y_logit == y_proto) & (y_proto == y_knn)

    idx_lp = select_idx_by_mask_with_fallback(idx_gate, mask_lp_agree[idx_gate], conf_lp, min_keep=min_keep)
    idx_tri = select_idx_by_mask_with_fallback(idx_gate, mask_tri_agree[idx_gate], conf_tri, min_keep=min_keep)

    gate_conf_thr = float(tau_conf)
    if len(idx_gate) > 0:
        gate_conf_thr = max(gate_conf_thr, float(np.quantile(conf_tri[idx_gate], CONF_DRIFT_Q)))

    idx_conf = select_idx_by_mask_with_fallback(idx_gate, conf_tri[idx_gate] >= gate_conf_thr, conf_tri, min_keep=min_keep)
    idx_agree_conf = select_idx_by_mask_with_fallback(
        idx_gate,
        ((mask_tri_agree & (conf_tri >= gate_conf_thr))[idx_gate]),
        conf_tri,
        min_keep=min_keep)

    support_idx = idx_agree_conf if len(idx_agree_conf) > 0 else idx_gate
    P_refine = refine_probs_multi_stage(Z_adapt, P_tri, protos, idx_support=support_idx, iters=REFINE_ITERS)
    conf_refine = msp_from_probs(P_refine)

    idx_rel = select_idx_by_mask_with_fallback(
        idx_gate,
        (((mask_tri_agree | mask_lp_agree) & (conf_tri >= gate_conf_thr))[idx_gate]),
        conf_tri,
        min_keep=min_keep)
    idx_unrel = np.setdiff1d(idx_gate, idx_rel, assume_unique=False)

    idx_common, idx_unknown, common_score, split_info = split_common_unknown_candidates(
        Z_adapt, P_logit_adapt, P_proto_adapt, P_knn_adapt, gate_pred, protos, tau_conf=tau_conf, min_keep=min_keep
    )

    idx_rel3, idx_amb3, idx_unk3, three_aux, three_info = build_three_bucket_partition(
        Z_adapt=Z_adapt,
        gate_pred=gate_pred,
        Sid_adapt=Sid_adapt,
        Sdom_adapt=Sdom_adapt,
        P_logit_adapt=P_logit_adapt,
        P_proto_adapt=P_proto_adapt,
        P_knn_adapt=P_knn_adapt,
        protos=protos,
        tau_conf=tau_conf,
        min_keep=max(16, min_keep // 2),
        reliable_keep_q=RELIABLE_KEEP_Q,
        ambig_keep_q=AMBIG_KEEP_Q,
        unknown_keep_q=UNKNOWN_KEEP_Q)
    P_three = np.asarray(three_aux.get("P_seed", P_tri.copy()), dtype=np.float32)
    w_rel3 = np.asarray(three_aux.get("w_rel", np.ones((len(idx_rel3)), dtype=np.float32)), dtype=np.float32)
    w_amb3 = np.asarray(three_aux.get("w_amb", np.ones((len(idx_amb3)), dtype=np.float32)), dtype=np.float32)
    idx_sup3 = np.concatenate([idx_rel3, idx_amb3], axis=0) if (len(idx_rel3) + len(idx_amb3)) > 0 else np.zeros((0), dtype=np.int64)

    idx_rel6, idx_amb6, idx_weak6, idx_unk6, six_aux, six_info = build_three_bucket_partition_v6(
        K=K,
        Z_adapt=Z_adapt,
        gate_pred=gate_pred,
        Sid_adapt=Sid_adapt,
        Sdom_adapt=Sdom_adapt,
        P_logit_adapt=P_logit_adapt,
        P_proto_adapt=P_proto_adapt,
        P_knn_adapt=P_knn_adapt,
        protos=protos,
        tau_conf=tau_conf,
        min_keep=max(16, min_keep // 2))
    P_six = np.asarray(six_aux.get("P_seed", P_tri.copy()), dtype=np.float32)
    w_rel6 = np.asarray(six_aux.get("w_rel", np.ones((len(idx_rel6)), dtype=np.float32)), dtype=np.float32)
    w_amb6 = np.asarray(six_aux.get("w_amb", np.ones((len(idx_amb6)), dtype=np.float32)), dtype=np.float32)
    w_weak6 = np.asarray(six_aux.get("w_weak", np.ones((len(idx_weak6)), dtype=np.float32)), dtype=np.float32)
    idx_sup6 = np.concatenate([idx_rel6, idx_amb6, idx_weak6], axis=0) if (len(idx_rel6) + len(idx_amb6) + len(idx_weak6)) > 0 else np.zeros((0), dtype=np.int64)

    idx_rel7, idx_amb7, idx_weak7, idx_unk7, seven_aux, seven_info = build_three_bucket_partition_v7(
        K=K,
        Z_adapt=Z_adapt,
        gate_pred=gate_pred,
        Sid_adapt=Sid_adapt,
        Sdom_adapt=Sdom_adapt,
        P_logit_adapt=P_logit_adapt,
        P_proto_adapt=P_proto_adapt,
        P_knn_adapt=P_knn_adapt,
        protos=protos,
        tau_conf=tau_conf,
        min_keep=max(16, min_keep // 2))
    P_seven = np.asarray(seven_aux.get("P_seed", P_tri.copy()), dtype=np.float32)
    w_rel7 = np.asarray(seven_aux.get("w_rel", np.ones((len(idx_rel7)), dtype=np.float32)), dtype=np.float32)
    w_amb7 = np.asarray(seven_aux.get("w_amb", np.ones((len(idx_amb7)), dtype=np.float32)), dtype=np.float32)
    w_weak7 = np.asarray(seven_aux.get("w_weak", np.ones((len(idx_weak7)), dtype=np.float32)), dtype=np.float32)
    idx_sup7 = np.concatenate([idx_rel7, idx_amb7], axis=0) if (len(idx_rel7) + len(idx_amb7)) > 0 else np.zeros((0), dtype=np.int64)

    
    dg_versions = {}
    for _dg_ver in ["v9", "v10", "v11", "v12"]:
        idx_rel_dg_v, idx_amb_dg_v, idx_unk_dg_v, dg_aux_v, dg_info_v = build_dg_raincoat_partition_versioned(
            K=K,
            Z_adapt=Z_adapt,
            gate_pred=gate_pred,
            Sid_adapt=Sid_adapt,
            Sdom_adapt=Sdom_adapt,
            P_logit_adapt=P_logit_adapt,
            P_proto_adapt=P_proto_adapt,
            P_knn_adapt=P_knn_adapt,
            protos=protos,
            tau_conf=tau_conf,
            min_keep=max(16, min_keep // 2),
            version=_dg_ver)
        dg_versions[_dg_ver] = dict(
            idx_rel=idx_rel_dg_v, idx_amb=idx_amb_dg_v, idx_unk=idx_unk_dg_v,
            aux=dg_aux_v, info=dg_info_v,
            q_seed=np.asarray(dg_aux_v.get("P_seed", P_tri.copy()), dtype=np.float32), w_rel=np.asarray(dg_aux_v.get("w_rel", np.ones((len(idx_rel_dg_v)), dtype=np.float32)), dtype=np.float32), w_amb=np.asarray(dg_aux_v.get("w_amb", np.ones((len(idx_amb_dg_v)), dtype=np.float32)), dtype=np.float32),
            idx_sup_eval=dg_aux_v.get("support_idx", np.concatenate([idx_rel_dg_v, idx_amb_dg_v], axis=0) if (len(idx_rel_dg_v) + len(idx_amb_dg_v)) > 0 else np.zeros((0), dtype=np.int64)))

    
    tbv8_versions = {}
    for _tb_ver in ["v9", "v10", "v11", "v12"]:
        idx_rel8_v, idx_amb8_v, idx_weak8_v, idx_unk8_v, eight_aux_v, eight_info_v = build_three_bucket_partition_v8_versioned(
            K=K,
            Z_adapt=Z_adapt,
            gate_pred=gate_pred,
            Sid_adapt=Sid_adapt,
            Sdom_adapt=Sdom_adapt,
            P_logit_adapt=P_logit_adapt,
            P_proto_adapt=P_proto_adapt,
            P_knn_adapt=P_knn_adapt,
            protos=protos,
            tau_conf=tau_conf,
            min_keep=max(16, min_keep // 2),
            version=_tb_ver)
        tbv8_versions[_tb_ver] = dict(
            idx_rel=idx_rel8_v, idx_amb=idx_amb8_v, idx_weak=idx_weak8_v, idx_unk=idx_unk8_v,
            aux=eight_aux_v, info=eight_info_v,
            q_seed=np.asarray(eight_aux_v.get("P_seed", P_tri.copy()), dtype=np.float32), w_rel=np.asarray(eight_aux_v.get("w_rel", np.ones((len(idx_rel8_v)), dtype=np.float32)), dtype=np.float32), w_amb=np.asarray(eight_aux_v.get("w_amb", np.ones((len(idx_amb8_v)), dtype=np.float32)), dtype=np.float32), w_weak=np.asarray(eight_aux_v.get("w_weak", np.ones((len(idx_weak8_v)), dtype=np.float32)), dtype=np.float32),
            idx_weak_cold=eight_aux_v.get("idx_weak_cold", np.zeros((0), dtype=np.int64)),
            w_weak_cold=eight_aux_v.get("w_weak_cold", np.zeros((0), dtype=np.float32)),
            idx_sup_eval=eight_aux_v.get("support_idx", np.concatenate([idx_rel8_v, idx_amb8_v], axis=0) if (len(idx_rel8_v) + len(idx_amb8_v)) > 0 else np.zeros((0), dtype=np.int64)))



    known_score_v4 = np.clip(np.asarray(p_known_adapt, dtype=np.float32), 1e-6, 1.0)
    drift_score_v4 = np.clip(np.asarray(P_route_v4[:, 1], dtype=np.float32), 1e-6, 1.0)
    unknown_score_v4 = np.clip(np.asarray(P_route_v4[:, 2], dtype=np.float32), 1e-6, 1.0)

    known_thr_v4 = max(float(np.quantile(known_score_v4, M2V4_KNOWN_Q)), float(M2V4_MIN_KNOWN_ABS))
    idx_known_v4 = np.where((known_score_v4 >= known_thr_v4) | (gate_pred_v4 != 2))[0]
    if len(idx_known_v4) == 0:
        idx_known_v4 = idx_gate if len(idx_gate) > 0 else idx_all

    drift_thr_rel_v4 = max(float(np.quantile(drift_score_v4[idx_known_v4], M2V4_REL_Q)), float(M2V4_MIN_DRIFT_ABS)) if len(idx_known_v4) > 0 else float(M2V4_MIN_DRIFT_ABS)
    drift_thr_amb_v4 = max(float(np.quantile(drift_score_v4[idx_known_v4], M2V4_AMB_Q)), float(M2V4_MIN_DRIFT_ABS * 0.80)) if len(idx_known_v4) > 0 else float(M2V4_MIN_DRIFT_ABS * 0.80)
    unk_thr_v4 = max(float(np.quantile(unknown_score_v4, M2V4_UNK_Q)), 0.50)
    align_known_thr_v4 = max(float(np.quantile(known_score_v4, M2V4_ALIGN_KNOWN_Q)), float(M2V4_MIN_KNOWN_ABS * 0.90))

    score_sup_v4 = drift_score_v4 * (0.40 + 0.40 * conf_refine + 0.10 * mask_lp_agree.astype(np.float32) + 0.10 * mask_tri_agree.astype(np.float32))
    score_amb_v4 = (0.50 * drift_score_v4 + 0.50 * known_score_v4) * (0.50 + 0.50 * conf_tri)

    rel_mask_v4 = ((drift_score_v4 >= drift_thr_rel_v4) & (known_score_v4 >= known_thr_v4) & (mask_lp_agree | mask_tri_agree))
    idx_rel_v4 = select_idx_by_mask_with_fallback(idx_known_v4, rel_mask_v4[idx_known_v4], score_sup_v4, min_keep=min_keep)

    rem_known_v4 = np.setdiff1d(idx_known_v4, idx_rel_v4, assume_unique=False)
    amb_mask_v4 = ((drift_score_v4 >= drift_thr_amb_v4) & (known_score_v4 >= align_known_thr_v4))
    idx_amb_v4 = select_idx_by_mask_with_fallback(rem_known_v4, amb_mask_v4[rem_known_v4] if len(rem_known_v4) > 0 else np.zeros((0), dtype=bool), score_amb_v4, min_keep=max(16, min_keep // 2))

    idx_sup_v4 = np.concatenate([idx_rel_v4, idx_amb_v4], axis=0) if (len(idx_rel_v4) + len(idx_amb_v4)) > 0 else np.zeros((0), dtype=np.int64)
    idx_align_v4 = select_idx_by_mask_with_fallback(idx_known_v4, (known_score_v4[idx_known_v4] >= align_known_thr_v4), known_score_v4, min_keep=max(min_keep, len(idx_sup_v4)))
    idx_unk_v4 = select_idx_by_mask_with_fallback(idx_all, unknown_score_v4 >= unk_thr_v4, unknown_score_v4, min_keep=max(16, min_keep // 2))


    methods = {
        "GatedAdapt": dict(train="hard", idx=idx_gate, y=y_logit[idx_gate], q=None, w=np.ones((len(idx_gate)), dtype=np.float32), P=P_logit_adapt),
        "GatedSelfSoft": dict(train="soft", idx=idx_gate, y=None, q=normalize_rows(P_logit_adapt[idx_gate] ** (1.0 / SOFT_T)), w=normalize_conf_weights(conf_logit[idx_gate]), P=P_logit_adapt),
        "GatedProtoAgree": dict(train="hard", idx=idx_lp, y=y_logit[idx_lp], q=None, w=normalize_conf_weights(conf_lp[idx_lp]), P=onehot_from_labels(y_logit, K)),
        "GatedProtoFusionSoft": dict(train="soft", idx=idx_gate, y=None, q=P_lp[idx_gate], w=normalize_conf_weights(conf_lp[idx_gate]), P=P_lp),
        "GatedTriAgree": dict(train="hard", idx=idx_tri, y=y_tri[idx_tri], q=None, w=normalize_conf_weights(conf_tri[idx_tri]), P=onehot_from_labels(y_tri, K)),
        "GatedTriFusionSoft": dict(train="soft", idx=idx_gate, y=None, q=P_tri[idx_gate], w=normalize_conf_weights(conf_tri[idx_gate]), P=P_tri),
        "ConfThreshSoft": dict(train="soft", idx=idx_conf, y=None, q=normalize_rows(P_logit_adapt[idx_conf] ** (1.0 / SOFT_T)), w=normalize_conf_weights(conf_tri[idx_conf]), P=P_logit_adapt),
        "AgreementConfSoft": dict(train="soft", idx=idx_agree_conf, y=None, q=P_tri[idx_agree_conf], w=normalize_conf_weights(conf_tri[idx_agree_conf]), P=P_tri),
        "ProtoRefineSoft": dict(train="soft", idx=idx_gate, y=None, q=P_refine[idx_gate], w=normalize_conf_weights(conf_refine[idx_gate]), P=P_refine),
        "ReliableEntropySplit": dict(
            train="generic", sup_mode="soft", sup_idx=idx_rel, q=P_tri[idx_rel], y=None,
            w=normalize_conf_weights(conf_tri[idx_rel]), align_idx=idx_gate, unrel_idx=idx_unrel,
            lambda_align=0.0, lambda_ent_min=0.0, lambda_ent_max=LAMBDA_ENT_MAX,
            lambda_cons=LAMBDA_CONS * 0.5, lambda_proto=LAMBDA_PROTO * 0.5, dynamic_align=False, P=P_tri),
        "TwoStageUnknown": dict(
            train="generic", sup_mode="soft", sup_idx=idx_common, q=P_refine[idx_common], y=None,
            w=normalize_conf_weights(common_score[idx_common]) if len(idx_common) > 0 else np.ones((0), dtype=np.float32),
            align_idx=idx_gate, unrel_idx=idx_unknown,
            lambda_align=LAMBDA_ALIGN * 0.5, lambda_ent_min=LAMBDA_ENT_MIN * 0.5, lambda_ent_max=LAMBDA_ENT_MAX,
            lambda_cons=LAMBDA_CONS, lambda_proto=LAMBDA_PROTO, dynamic_align=False, P=P_refine),
        "GCODWFA_Lite": dict(
            train="generic", sup_mode="soft", sup_idx=idx_common, q=P_refine[idx_common], y=None,
            w=normalize_conf_weights(common_score[idx_common]) if len(idx_common) > 0 else np.ones((0), dtype=np.float32),
            align_idx=idx_gate, unrel_idx=idx_unknown,
            lambda_align=GCODWFA_ALIGN_MAX, lambda_ent_min=LAMBDA_ENT_MIN, lambda_ent_max=0.0,
            lambda_cons=LAMBDA_CONS, lambda_proto=LAMBDA_PROTO * 0.5, dynamic_align=True, P=P_refine),
        "RAINCOAT_Lite": dict(
            train="raincoat", idx_gate=idx_gate, sup_idx=idx_common, unrel_idx=idx_unknown,
            q=P_refine[idx_common] if len(idx_common) > 0 else np.zeros((0, K), dtype=np.float32),
            w=normalize_conf_weights(common_score[idx_common]) if len(idx_common) > 0 else np.ones((0), dtype=np.float32),
            P=P_refine),
        
        "DG_RAINCOAT_v9": dict(
            train="dg_raincoat_versioned",
            version="v9",
            idx_gate=idx_gate, idx_rel=dg_versions["v9"]["idx_rel"], idx_amb=dg_versions["v9"]["idx_amb"], idx_unk=dg_versions["v9"]["idx_unk"],
            idx_sup_eval=dg_versions["v9"]["idx_sup_eval"],
            q_seed=dg_versions["v9"]["q_seed"], w_rel=dg_versions["v9"]["w_rel"], w_amb=dg_versions["v9"]["w_amb"],
            lambda_align_stage1=DG_LAMBDA_ALIGN_STAGE1, lambda_align_stage2=DG_LAMBDA_ALIGN_STAGE2,
            lambda_ent_min=DG_LAMBDA_ENT_MIN, lambda_ent_max=DG_LAMBDA_ENT_MAX,
            lambda_cons=DG_LAMBDA_CONS, lambda_proto=DG_LAMBDA_PROTO,
            lambda_u_repel=DG_LAMBDA_UNKNOWN_REPEL,
            P=dg_versions["v9"]["q_seed"], bucket_info=dg_versions["v9"]["info"]),
        "DG_RAINCOAT_v10": dict(
            train="dg_raincoat_versioned",
            version="v10",
            idx_gate=idx_gate, idx_rel=dg_versions["v10"]["idx_rel"], idx_amb=dg_versions["v10"]["idx_amb"], idx_unk=dg_versions["v10"]["idx_unk"],
            idx_sup_eval=dg_versions["v10"]["idx_sup_eval"],
            q_seed=dg_versions["v10"]["q_seed"], w_rel=dg_versions["v10"]["w_rel"], w_amb=dg_versions["v10"]["w_amb"],
            lambda_align_stage1=DG_LAMBDA_ALIGN_STAGE1, lambda_align_stage2=DG_LAMBDA_ALIGN_STAGE2,
            lambda_ent_min=DG_LAMBDA_ENT_MIN, lambda_ent_max=DG_LAMBDA_ENT_MAX,
            lambda_cons=DG_LAMBDA_CONS, lambda_proto=DG_LAMBDA_PROTO,
            lambda_u_repel=DG_LAMBDA_UNKNOWN_REPEL,
            P=dg_versions["v10"]["q_seed"], bucket_info=dg_versions["v10"]["info"]),
        "DG_RAINCOAT_v11": dict(
            train="dg_raincoat_versioned",
            version="v11",
            idx_gate=idx_gate, idx_rel=dg_versions["v11"]["idx_rel"], idx_amb=dg_versions["v11"]["idx_amb"], idx_unk=dg_versions["v11"]["idx_unk"],
            idx_sup_eval=dg_versions["v11"]["idx_sup_eval"],
            q_seed=dg_versions["v11"]["q_seed"], w_rel=dg_versions["v11"]["w_rel"], w_amb=dg_versions["v11"]["w_amb"],
            lambda_align_stage1=DG_LAMBDA_ALIGN_STAGE1, lambda_align_stage2=DG_LAMBDA_ALIGN_STAGE2,
            lambda_ent_min=DG_LAMBDA_ENT_MIN, lambda_ent_max=DG_LAMBDA_ENT_MAX,
            lambda_cons=DG_LAMBDA_CONS, lambda_proto=DG_LAMBDA_PROTO,
            lambda_u_repel=DG_LAMBDA_UNKNOWN_REPEL,
            P=dg_versions["v11"]["q_seed"], bucket_info=dg_versions["v11"]["info"]),
        "DG_RAINCOAT_v12": dict(
            train="dg_raincoat_versioned",
            version="v12",
            idx_gate=idx_gate, idx_rel=dg_versions["v12"]["idx_rel"], idx_amb=dg_versions["v12"]["idx_amb"], idx_unk=dg_versions["v12"]["idx_unk"],
            idx_sup_eval=dg_versions["v12"]["idx_sup_eval"],
            q_seed=dg_versions["v12"]["q_seed"], w_rel=dg_versions["v12"]["w_rel"], w_amb=dg_versions["v12"]["w_amb"],
            lambda_align_stage1=DG_LAMBDA_ALIGN_STAGE1, lambda_align_stage2=DG_LAMBDA_ALIGN_STAGE2,
            lambda_ent_min=DG_LAMBDA_ENT_MIN, lambda_ent_max=DG_LAMBDA_ENT_MAX,
            lambda_cons=DG_LAMBDA_CONS, lambda_proto=DG_LAMBDA_PROTO,
            lambda_u_repel=DG_LAMBDA_UNKNOWN_REPEL,
            P=dg_versions["v12"]["q_seed"], bucket_info=dg_versions["v12"]["info"]),
        "ThreeBucketV5Soft": dict(
            train="three_bucket",
            idx_gate=idx_gate, idx_rel=idx_rel3, idx_amb=idx_amb3, idx_unk=idx_unk3,
            idx_sup_eval=idx_sup3,
            q_seed=P_three, w_rel=w_rel3, w_amb=w_amb3,
            use_ema=False, teacher_blend=0.0,
            lambda_rel_sup=1.0, lambda_amb_sup=LAMBDA_AMB_SUP,
            lambda_align=LAMBDA_ALIGN * 0.70, lambda_ent_min=LAMBDA_ENT_MIN * 0.35,
            lambda_ent_max=LAMBDA_ENT_MAX * 0.60, lambda_cons=LAMBDA_CONS,
            lambda_proto=LAMBDA_PROTO * 0.80, lambda_src_proto=LAMBDA_SRC_PROTO,
            lambda_src_logit=LAMBDA_SRC_LOGIT, lambda_u_repel=LAMBDA_UNKNOWN_REPEL * 0.60,
            unknown_warmup_epochs=UNKNOWN_WARMUP_EPOCHS, unknown_ramp_epochs=UNKNOWN_RAMP_EPOCHS,
            P=P_three, bucket_info=three_info),
        "ThreeBucketV5EMA": dict(
            train="three_bucket",
            idx_gate=idx_gate, idx_rel=idx_rel3, idx_amb=idx_amb3, idx_unk=idx_unk3,
            idx_sup_eval=idx_sup3,
            q_seed=P_three, w_rel=w_rel3, w_amb=w_amb3,
            use_ema=True, teacher_blend=EMA_TEACHER_BLEND,
            lambda_rel_sup=1.0, lambda_amb_sup=LAMBDA_AMB_SUP,
            lambda_align=LAMBDA_ALIGN * 0.72, lambda_ent_min=LAMBDA_ENT_MIN * 0.30,
            lambda_ent_max=LAMBDA_ENT_MAX * 0.55, lambda_cons=LAMBDA_CONS,
            lambda_proto=LAMBDA_PROTO, lambda_src_proto=LAMBDA_SRC_PROTO,
            lambda_src_logit=LAMBDA_SRC_LOGIT, lambda_u_repel=LAMBDA_UNKNOWN_REPEL * 0.70,
            unknown_warmup_epochs=UNKNOWN_WARMUP_EPOCHS, unknown_ramp_epochs=UNKNOWN_RAMP_EPOCHS,
            P=P_three, bucket_info=three_info),
        "ThreeBucketV5EMAProto": dict(
            train="three_bucket",
            idx_gate=idx_gate, idx_rel=idx_rel3, idx_amb=idx_amb3, idx_unk=idx_unk3,
            idx_sup_eval=idx_sup3,
            q_seed=P_three, w_rel=w_rel3, w_amb=w_amb3,
            use_ema=True, teacher_blend=min(0.72, EMA_TEACHER_BLEND + 0.05),
            lambda_rel_sup=1.0, lambda_amb_sup=LAMBDA_AMB_SUP * 0.95,
            lambda_align=LAMBDA_ALIGN * 0.68, lambda_ent_min=LAMBDA_ENT_MIN * 0.25,
            lambda_ent_max=LAMBDA_ENT_MAX * 0.50, lambda_cons=LAMBDA_CONS,
            lambda_proto=LAMBDA_PROTO * 1.15, lambda_src_proto=LAMBDA_SRC_PROTO * 1.25,
            lambda_src_logit=LAMBDA_SRC_LOGIT * 1.20, lambda_u_repel=LAMBDA_UNKNOWN_REPEL * 0.85,
            unknown_warmup_epochs=UNKNOWN_WARMUP_EPOCHS, unknown_ramp_epochs=UNKNOWN_RAMP_EPOCHS,
            P=P_three, bucket_info=three_info),
        "ThreeBucketV6Curriculum": dict(
            train="three_bucket_v6",
            idx_gate=idx_gate, idx_rel=idx_rel6, idx_amb=idx_amb6, idx_weak=idx_weak6, idx_unk=idx_unk6,
            idx_sup_eval=idx_sup6,
            q_seed=P_six, w_rel=w_rel6, w_amb=w_amb6, w_weak=w_weak6,
            use_ema=True, teacher_blend=EMA_TEACHER_BLEND,
            weak_teacher_blend=V6_WEAK_TEACHER_BLEND, weak_start_epoch=V6_WEAK_START_EPOCH,
            lambda_rel_sup=1.0, lambda_amb_sup=V6_LAMBDA_AMB_SUP, lambda_weak_sup=V6_LAMBDA_WEAK_SUP,
            lambda_align=V6_LAMBDA_ALIGN, lambda_ent_min=V6_LAMBDA_ENT_MIN, lambda_ent_max=V6_LAMBDA_ENT_MAX,
            lambda_cons=V6_LAMBDA_CONS, lambda_proto=V6_LAMBDA_PROTO,
            lambda_src_proto=LAMBDA_SRC_PROTO, lambda_src_logit=LAMBDA_SRC_LOGIT,
            lambda_u_repel=V6_LAMBDA_UNKNOWN_REPEL,
            unknown_warmup_epochs=V6_UNKNOWN_WARMUP_EPOCHS, unknown_ramp_epochs=V6_UNKNOWN_RAMP_EPOCHS,
            P=P_six, bucket_info=six_info),
        "ThreeBucketV7Promotion": dict(
            train="three_bucket_v7",
            idx_gate=idx_gate, idx_rel=idx_rel7, idx_amb=idx_amb7, idx_weak=idx_weak7, idx_unk=idx_unk7,
            idx_sup_eval=idx_sup7,
            q_seed=P_seven, w_rel=w_rel7, w_amb=w_amb7, w_weak=w_weak7,
            bucket_score=seven_aux["bucket_score"], unknown_score=seven_aux["unknown_score"],
            use_ema=True, teacher_blend=EMA_TEACHER_BLEND,
            stage1_epochs=V7_STAGE1_EPOCHS, promote_blend=V7_PROMOTE_BLEND,
            promote_rel_fraction=V7_PROMOTE_REL_FRACTION, promote_amb_fraction=V7_PROMOTE_AMB_FRACTION,
            promote_conf=V7_PROMOTE_CONF, promote_margin=V7_PROMOTE_MARGIN,
            lambda_rel_sup=1.0, lambda_amb_sup=V7_LAMBDA_AMB_SUP, lambda_weak_sup=V7_LAMBDA_WEAK_SUP,
            lambda_align=V7_LAMBDA_ALIGN, lambda_ent_min=V7_LAMBDA_ENT_MIN, lambda_ent_max=V7_LAMBDA_ENT_MAX,
            lambda_cons=V7_LAMBDA_CONS, lambda_proto=V7_LAMBDA_PROTO,
            lambda_src_proto=LAMBDA_SRC_PROTO * 0.60, lambda_src_logit=LAMBDA_SRC_LOGIT * 0.60,
            lambda_u_repel=V7_LAMBDA_UNKNOWN_REPEL,
            unknown_warmup_epochs=V7_UNKNOWN_WARMUP_EPOCHS, unknown_ramp_epochs=V7_UNKNOWN_RAMP_EPOCHS,
            P=P_seven, bucket_info=seven_info),
        
        "ThreeBucketV8Adaptive_v9": dict(
            train="three_bucket_v8_versioned",
            version="v9",
            idx_gate=idx_gate, idx_rel=tbv8_versions["v9"]["idx_rel"], idx_amb=tbv8_versions["v9"]["idx_amb"], idx_weak=tbv8_versions["v9"]["idx_weak"], idx_unk=tbv8_versions["v9"]["idx_unk"],
            idx_sup_eval=tbv8_versions["v9"]["idx_sup_eval"],
            q_seed=tbv8_versions["v9"]["q_seed"], w_rel=tbv8_versions["v9"]["w_rel"], w_amb=tbv8_versions["v9"]["w_amb"], w_weak=tbv8_versions["v9"]["w_weak"],
            idx_weak_cold=tbv8_versions["v9"]["idx_weak_cold"], w_weak_cold=tbv8_versions["v9"]["w_weak_cold"],
            bucket_score=tbv8_versions["v9"]["aux"]["bucket_score"], unknown_score=tbv8_versions["v9"]["aux"]["unknown_score"],
            lambda_align=V8_LAMBDA_ALIGN, lambda_ent_min=V8_LAMBDA_ENT_MIN, lambda_ent_max=V8_LAMBDA_ENT_MAX,
            lambda_cons=V8_LAMBDA_CONS, lambda_proto=V8_LAMBDA_PROTO,
            lambda_u_repel=V8_LAMBDA_UNKNOWN_REPEL,
            P=tbv8_versions["v9"]["q_seed"], bucket_info=tbv8_versions["v9"]["info"]),
        "ThreeBucketV8Adaptive_v10": dict(
            train="three_bucket_v8_versioned",
            version="v10",
            idx_gate=idx_gate, idx_rel=tbv8_versions["v10"]["idx_rel"], idx_amb=tbv8_versions["v10"]["idx_amb"], idx_weak=tbv8_versions["v10"]["idx_weak"], idx_unk=tbv8_versions["v10"]["idx_unk"],
            idx_sup_eval=tbv8_versions["v10"]["idx_sup_eval"],
            q_seed=tbv8_versions["v10"]["q_seed"], w_rel=tbv8_versions["v10"]["w_rel"], w_amb=tbv8_versions["v10"]["w_amb"], w_weak=tbv8_versions["v10"]["w_weak"],
            idx_weak_cold=tbv8_versions["v10"]["idx_weak_cold"], w_weak_cold=tbv8_versions["v10"]["w_weak_cold"],
            bucket_score=tbv8_versions["v10"]["aux"]["bucket_score"], unknown_score=tbv8_versions["v10"]["aux"]["unknown_score"],
            lambda_align=V8_LAMBDA_ALIGN, lambda_ent_min=V8_LAMBDA_ENT_MIN, lambda_ent_max=V8_LAMBDA_ENT_MAX,
            lambda_cons=V8_LAMBDA_CONS, lambda_proto=V8_LAMBDA_PROTO,
            lambda_u_repel=V8_LAMBDA_UNKNOWN_REPEL,
            P=tbv8_versions["v10"]["q_seed"], bucket_info=tbv8_versions["v10"]["info"]),
        "ThreeBucketV8Adaptive_v11": dict(
            train="three_bucket_v8_versioned",
            version="v11",
            idx_gate=idx_gate, idx_rel=tbv8_versions["v11"]["idx_rel"], idx_amb=tbv8_versions["v11"]["idx_amb"], idx_weak=tbv8_versions["v11"]["idx_weak"], idx_unk=tbv8_versions["v11"]["idx_unk"],
            idx_sup_eval=tbv8_versions["v11"]["idx_sup_eval"],
            q_seed=tbv8_versions["v11"]["q_seed"], w_rel=tbv8_versions["v11"]["w_rel"], w_amb=tbv8_versions["v11"]["w_amb"], w_weak=tbv8_versions["v11"]["w_weak"],
            idx_weak_cold=tbv8_versions["v11"]["idx_weak_cold"], w_weak_cold=tbv8_versions["v11"]["w_weak_cold"],
            bucket_score=tbv8_versions["v11"]["aux"]["bucket_score"], unknown_score=tbv8_versions["v11"]["aux"]["unknown_score"],
            lambda_align=V8_LAMBDA_ALIGN, lambda_ent_min=V8_LAMBDA_ENT_MIN, lambda_ent_max=V8_LAMBDA_ENT_MAX,
            lambda_cons=V8_LAMBDA_CONS, lambda_proto=V8_LAMBDA_PROTO,
            lambda_u_repel=V8_LAMBDA_UNKNOWN_REPEL,
            P=tbv8_versions["v11"]["q_seed"], bucket_info=tbv8_versions["v11"]["info"]),
        "ThreeBucketV8Adaptive_v12": dict(
            train="three_bucket_v8_versioned",
            version="v12",
            idx_gate=idx_gate, idx_rel=tbv8_versions["v12"]["idx_rel"], idx_amb=tbv8_versions["v12"]["idx_amb"], idx_weak=tbv8_versions["v12"]["idx_weak"], idx_unk=tbv8_versions["v12"]["idx_unk"],
            idx_sup_eval=tbv8_versions["v12"]["idx_sup_eval"],
            q_seed=tbv8_versions["v12"]["q_seed"], w_rel=tbv8_versions["v12"]["w_rel"], w_amb=tbv8_versions["v12"]["w_amb"], w_weak=tbv8_versions["v12"]["w_weak"],
            idx_weak_cold=tbv8_versions["v12"]["idx_weak_cold"], w_weak_cold=tbv8_versions["v12"]["w_weak_cold"],
            bucket_score=tbv8_versions["v12"]["aux"]["bucket_score"], unknown_score=tbv8_versions["v12"]["aux"]["unknown_score"],
            lambda_align=V8_LAMBDA_ALIGN, lambda_ent_min=V8_LAMBDA_ENT_MIN, lambda_ent_max=V8_LAMBDA_ENT_MAX,
            lambda_cons=V8_LAMBDA_CONS, lambda_proto=V8_LAMBDA_PROTO,
            lambda_u_repel=V8_LAMBDA_UNKNOWN_REPEL,
            P=tbv8_versions["v12"]["q_seed"], bucket_info=tbv8_versions["v12"]["info"]),
        "UngatedAdapt": dict(train="hard", idx=idx_all, y=y_logit[idx_all], q=None, w=np.ones((len(idx_all)), dtype=np.float32), P=P_logit_adapt),

"M2V8RouteSplit": dict(
    train="generic", sup_mode="soft", sup_idx=idx_sup_v4, q=P_refine[idx_sup_v4], y=None,
    w=normalize_conf_weights(score_sup_v4[idx_sup_v4]) if len(idx_sup_v4) > 0 else np.ones((0), dtype=np.float32),
    align_idx=idx_align_v4, unrel_idx=idx_unk_v4,
    lambda_align=LAMBDA_ALIGN * 0.75, lambda_ent_min=LAMBDA_ENT_MIN * 0.50, lambda_ent_max=LAMBDA_ENT_MAX * 0.75,
    lambda_cons=LAMBDA_CONS * 0.80, lambda_proto=LAMBDA_PROTO * 0.90, dynamic_align=False, P=P_refine),
"M2V8ThreeBucket": dict(
    train="three_bucket",
    idx_gate=idx_align_v4, idx_rel=idx_rel_v4, idx_amb=idx_amb_v4, idx_unk=idx_unk_v4,
    idx_sup_eval=idx_sup_v4,
    q_seed=P_refine,
    w_rel=normalize_conf_weights(score_sup_v4[idx_rel_v4]) if len(idx_rel_v4) > 0 else np.ones((0), dtype=np.float32),
    w_amb=normalize_conf_weights(score_amb_v4[idx_amb_v4]) if len(idx_amb_v4) > 0 else np.ones((0), dtype=np.float32),
    use_ema=True, teacher_blend=min(0.70, EMA_TEACHER_BLEND + 0.05),
    lambda_rel_sup=1.0, lambda_amb_sup=LAMBDA_AMB_SUP * 0.90,
    lambda_align=LAMBDA_ALIGN * 0.70, lambda_ent_min=LAMBDA_ENT_MIN * 0.35,
    lambda_ent_max=LAMBDA_ENT_MAX * 0.55, lambda_cons=LAMBDA_CONS * 0.90,
    lambda_proto=LAMBDA_PROTO * 0.90, lambda_src_proto=LAMBDA_SRC_PROTO * 0.90,
    lambda_src_logit=LAMBDA_SRC_LOGIT * 0.90, lambda_u_repel=LAMBDA_UNKNOWN_REPEL * 0.60,
    unknown_warmup_epochs=UNKNOWN_WARMUP_EPOCHS, unknown_ramp_epochs=UNKNOWN_RAMP_EPOCHS,
    P=P_refine,
    bucket_info=dict(
        rel_size=int(len(idx_rel_v4)), amb_size=int(len(idx_amb_v4)), unk_size=int(len(idx_unk_v4)),
        sup_size=int(len(idx_sup_v4)),
        mean_p_known=float(np.mean(known_score_v4)) if len(known_score_v4) > 0 else float("nan"),
        mean_p_drift=float(np.mean(drift_score_v4)) if len(drift_score_v4) > 0 else float("nan"))),
"OracleGatePseudoAdapt": dict(train="hard", idx=idx_oracle, y=y_logit[idx_oracle], q=None, w=np.ones((len(idx_oracle)), dtype=np.float32), P=P_logit_adapt),
        "PredRouteOracleLabelAdapt": dict(
            train="hard",
            idx=idx_sup_v4[y_true_adapt[idx_sup_v4] >= 0] if len(idx_sup_v4) > 0 else np.zeros((0), dtype=np.int64),
            y=y_true_adapt[idx_sup_v4[y_true_adapt[idx_sup_v4] >= 0]] if len(idx_sup_v4) > 0 else np.zeros((0), dtype=np.int64),
            q=None,
            w=np.ones((int(np.sum(y_true_adapt[idx_sup_v4] >= 0))), dtype=np.float32) if len(idx_sup_v4) > 0 else np.ones((0), dtype=np.float32),
            P=onehot_from_labels(np.clip(y_true_adapt, 0, K-1), K),
            bucket_info=dict(
                pred_route_known_size=int(len(idx_sup_v4)),
                pred_route_known_precision=float((y_true_adapt[idx_sup_v4] >= 0).mean()) if len(idx_sup_v4) > 0 else float("nan"),
                pred_route_known_oracle_train_size=int(np.sum(y_true_adapt[idx_sup_v4] >= 0)) if len(idx_sup_v4) > 0 else 0)),
        "OracleLabelAdapt": dict(train="hard", idx=idx_oracle, y=y_true_adapt[idx_oracle], q=None, w=np.ones((len(idx_oracle)), dtype=np.float32), P=onehot_from_labels(np.clip(y_true_adapt, 0, K-1), K)),
    }

    # [FIX-2A] Do NOT build v22 here.
    # At this point build_pseudo_method_bank() has not yet injected DG_RAINCOAT_v19M2W / DG_RAINCOAT_v19M2W_EnergyU.
    # If we call build_weighted_entry_minimal_from_base() here, base_spec becomes None, v22 is never created,
    # and the later evaluation loop silently falls back to adapters.get(name, None) -> NoAdapt behavior.
    # The actual v22 construction is moved to run_one_split() AFTER the v19 builder block is executed.
    v22_aux_info = dict(v22_deferred=True, v22_reason="build_after_v19_in_run_one_split")

    info = {}
    total_known = max(1, int(np.sum(y_true_adapt >= 0)))
    for name, spec in methods.items():
        sel_idx = spec.get("idx", spec.get("idx_sup_eval", spec.get("sup_idx", np.zeros((0), dtype=np.int64))))
        sel_idx = np.asarray(sel_idx, dtype=np.int64)
        stats = summarize_method_on_selected(sel_idx, y_true_adapt, spec["P"])
        sel_precision = float((y_true_adapt[sel_idx] >= 0).mean()) if len(sel_idx) > 0 else float("nan")
        sel_recall = float(np.sum(y_true_adapt[sel_idx] >= 0) / total_known) if len(sel_idx) > 0 else 0.0
        sel_keep = float(len(sel_idx) / max(1, len(y_true_adapt)))

        unknown_idx = np.asarray(spec.get("idx_unk_eval", spec.get("unrel_idx", spec.get("idx_unk", np.zeros((0), dtype=np.int64)))), dtype=np.int64)
        unknown_keep = float(len(unknown_idx) / max(1, len(y_true_adapt)))
        unknown_precision = float((y_true_adapt[unknown_idx] < 0).mean()) if len(unknown_idx) > 0 else float("nan")

        info[name] = dict(
            sel_precision=sel_precision,
            sel_recall=sel_recall,
            sel_keep_ratio=sel_keep,
            sel_size=stats["sel_size"],
            pseudo_acc_selected=stats["pseudo_acc"],
            pseudo_acc=stats["pseudo_acc"],
            unknown_cand_keep=unknown_keep,
            unknown_cand_precision=unknown_precision)
        if "w_sup" in spec:
            w_sup_all = np.asarray(spec.get("w_sup", np.zeros((0), dtype=np.float32)), dtype=np.float32)
            if len(w_sup_all) == len(y_true_adapt):
                known_mask = (y_true_adapt >= 0)
                pred_all = np.argmax(spec["P"], axis=1)
                info[name].update(
                    sum_w_sup=float(np.sum(w_sup_all)),
                    num_w_sup_gt_02=int(np.sum(w_sup_all > 0.20)),
                    num_w_sup_gt_05=int(np.sum(w_sup_all > 0.50)),
                    effective_coverage=float(np.sum(w_sup_all[known_mask]) / max(1, int(np.sum(known_mask)))),
                    soft_pseudo_acc=float(np.sum(w_sup_all[known_mask] * (pred_all[known_mask] == y_true_adapt[known_mask]).astype(np.float32)) / max(1e-8, float(np.sum(w_sup_all[known_mask])))) if np.any(known_mask) else float("nan"))
        if "w_stab" in spec:
            w_stab_all = np.asarray(spec.get("w_stab", np.zeros((0), dtype=np.float32)), dtype=np.float32)
            if len(w_stab_all) == len(y_true_adapt):
                info[name].update(
                    sum_w_stab=float(np.sum(w_stab_all)),
                    effective_stable_pressure=float(np.sum(w_stab_all) / max(1, len(w_stab_all))))
        if "w_unk" in spec:
            w_unk_all = np.asarray(spec.get("w_unk", np.zeros((0), dtype=np.float32)), dtype=np.float32)
            if len(w_unk_all) == len(y_true_adapt):
                info[name].update(sum_w_unk=float(np.sum(w_unk_all)))
        if "bucket_info" in spec:
            info[name].update(spec["bucket_info"])


    for _nm in ["M2V8RouteSplit", "M2V8ThreeBucket"]:
        if _nm in info:
            info[_nm].update(
                rel_size=int(len(idx_rel_v4)),
                amb_size=int(len(idx_amb_v4)),
                unk_size=int(len(idx_unk_v4)),
                sup_size=int(len(idx_sup_v4)),
                mean_p_known=float(np.mean(known_score_v4)) if len(known_score_v4) > 0 else float("nan"),
                mean_p_drift=float(np.mean(drift_score_v4)) if len(drift_score_v4) > 0 else float("nan"))

    aux = dict(
        P_lp=P_lp, P_tri=P_tri, P_refine=P_refine, common_score=common_score, split_info=split_info,
        P_three=P_three, three_bucket=three_info, three_threshold_reliable=three_aux["threshold_reliable"],
        three_threshold_unknown=three_aux["threshold_unknown"], three_bucket_score=three_aux["bucket_score"],
        P_v6=P_six, v6_bucket=six_info, v6_threshold_reliable=six_aux["threshold_reliable"],
        v6_threshold_unknown=six_aux["threshold_unknown"], v6_bucket_score=six_aux["bucket_score"])
    return methods, info, aux




def build_dg_raincoat_partition_versioned(
    K,
    Z_adapt,
    gate_pred,
    Sid_adapt,
    Sdom_adapt,
    P_logit_adapt,
    P_proto_adapt,
    P_knn_adapt,
    protos,
    tau_conf,
    min_keep=64,
    version="v10"):
    cfg = dict(DGR_VERSION_CFG.get(version, DGR_VERSION_CFG["v10"]))
    idx_gate = np.where(gate_pred == 1)[0]
    n = len(gate_pred)
    empty = np.zeros((0), dtype=np.int64)
    if len(idx_gate) == 0:
        aux = dict(
            common_score=np.zeros((n), dtype=np.float32),
            rescue_score=np.zeros((n), dtype=np.float32),
            unknown_score=np.zeros((n), dtype=np.float32),
            P_seed=weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN]),
            w_rel=np.zeros((0), dtype=np.float32),
            w_amb=np.zeros((0), dtype=np.float32),
            support_idx=np.zeros((0), dtype=np.int64),
            support_weight=np.zeros((0), dtype=np.float32),
            rescue_idx=np.zeros((0), dtype=np.int64))
        info = dict(reliable_size=0, ambiguous_size=0, unknown_size=0, rescue_size=0, support_size=0)
        return empty, empty, empty, aux, info

    P_tri = weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN])
    y_tri = np.argmax(P_tri, axis=1)
    y_logit = np.argmax(P_logit_adapt, axis=1)
    y_proto = np.argmax(P_proto_adapt, axis=1)
    y_knn = np.argmax(P_knn_adapt, axis=1)
    conf_tri = msp_from_probs(P_tri)
    margin_tri = top2_margin(P_tri)
    _, proto_margin, proto_dist = prototype_margin_dist(Z_adapt, protos)

    agree = ((y_logit == y_proto).astype(np.float32) + (y_logit == y_knn).astype(np.float32) + (y_proto == y_knn).astype(np.float32)) / 3.0
    sid_n = robust_unit_interval(Sid_adapt, idx_gate)
    sdom_n = robust_unit_interval(Sdom_adapt, idx_gate)
    conf_n = robust_unit_interval(conf_tri, idx_gate)
    margin_n = robust_unit_interval(margin_tri, idx_gate)
    proto_margin_n = robust_unit_interval(proto_margin, idx_gate)
    proto_dist_n = robust_unit_interval(proto_dist, idx_gate)

    common_score = (
        0.32 * sid_n +
        0.18 * sdom_n +
        0.15 * conf_n +
        0.10 * margin_n +
        0.09 * agree +
        0.09 * proto_margin_n +
        0.07 * (1.0 - proto_dist_n)
    ).astype(np.float32)
    rescue_score = (
        0.24 * sid_n +
        0.22 * sdom_n +
        0.18 * conf_n +
        0.12 * margin_n +
        0.10 * agree +
        0.08 * proto_margin_n +
        0.06 * (1.0 - proto_dist_n)
    ).astype(np.float32)
    unknown_score = (
        0.42 * (1.0 - sid_n) +
        0.18 * proto_dist_n +
        0.14 * (1.0 - conf_n) +
        0.10 * (1.0 - margin_n) +
        0.08 * (1.0 - agree) +
        0.08 * (1.0 - sdom_n)
    ).astype(np.float32)

    ng = len(idx_gate)
    k_rel = min(ng, max(int(round(cfg["rel_keep_q"] * ng)), max(16, min_keep // 2)))
    k_amb = min(max(0, ng - k_rel), max(int(round(cfg["amb_keep_q"] * ng)), max(16, min_keep // 3)))
    k_unk = min(max(0, ng - k_rel - k_amb), max(int(round(cfg["unk_keep_q"] * ng)), max(8, min_keep // int(max(2, cfg["unk_divisor"])))))

    idx_rel = select_topk_class_balanced(idx_gate, y_tri, common_score, k_rel, K, min_per_class=1)
    rem1 = np.setdiff1d(idx_gate, idx_rel, assume_unique=False)

    sid_thr = float(np.quantile(sid_n[idx_gate], cfg["rescue_sid_q"])) if ng >= 10 else 0.55
    conf_thr = max(float(tau_conf) * 0.92, cfg["rescue_conf"])
    margin_thr = cfg["rescue_margin"]
    proto_thr = float(np.quantile(proto_margin_n[idx_gate], cfg["rescue_proto_q"])) if ng >= 10 else 0.35
    if cfg["rescue_mode"] == "low_sdom":
        sdom_thr = float(np.quantile(sdom_n[idx_gate], cfg["rescue_sdom_q"])) if ng >= 10 else 0.50
        sdom_cond = sdom_n <= sdom_thr
    else:
        sdom_thr = float(np.quantile(sdom_n[idx_gate], cfg["rescue_sdom_q"])) if ng >= 10 else 0.58
        sdom_cond = sdom_n >= sdom_thr

    rescue_mask = (
        (sid_n >= sid_thr) &
        (conf_tri >= conf_thr) &
        (margin_tri >= margin_thr) &
        (proto_margin_n >= proto_thr) &
        sdom_cond
    )
    rescue_candidates = np.intersect1d(rem1, idx_gate[rescue_mask[idx_gate]], assume_unique=False)

    extreme_unk_thr = float(np.quantile(unknown_score[idx_gate], DG_UNKNOWN_EXTREME_Q)) if ng >= 10 else float(np.max(unknown_score[idx_gate]))
    hard_unk_candidates = np.setdiff1d(rem1[unknown_score[rem1] >= extreme_unk_thr], rescue_candidates, assume_unique=False)
    if len(hard_unk_candidates) < k_unk:
        ranked_unk = rem1[np.argsort(-unknown_score[rem1])]
        ranked_unk = np.setdiff1d(ranked_unk, rescue_candidates, assume_unique=False)
        idx_unk = ranked_unk[:k_unk]
    else:
        idx_unk = hard_unk_candidates[np.argsort(-unknown_score[hard_unk_candidates])[:k_unk]]

    rem2 = np.setdiff1d(rem1, idx_unk, assume_unique=False)
    a, b, c = cfg["rescue_rank_blend"]
    rescue_rank = a * rescue_score + b * common_score + c * conf_tri
    rescue_pool = np.intersect1d(rem2, rescue_candidates, assume_unique=False)
    k_rescue = min(len(rem2), max(8, int(round(cfg["rescue_frac"] * ng))))
    idx_rescue = select_topk_class_balanced(
        rescue_pool, y_tri, rescue_rank, min(k_rescue, len(rescue_pool)), K, min_per_class=0) if len(rescue_pool) > 0 else np.zeros((0), dtype=np.int64)

    rem3 = np.setdiff1d(rem2, idx_rescue, assume_unique=False)
    a, b, c = cfg["common_score_blend"]
    amb_score = a * common_score + b * rescue_score + c * conf_tri
    idx_amb_main = select_topk_class_balanced(rem3, y_tri, amb_score, k_amb, K, min_per_class=1) if len(rem3) > 0 else np.zeros((0), dtype=np.int64)
    idx_amb = _concat_unique_idx(idx_rescue, idx_amb_main)

    if cfg["support_mode"] == "nonunk":
        support_idx = np.setdiff1d(idx_gate, idx_unk, assume_unique=False)
    else:
        cand_mask = (
            (common_score >= (float(np.quantile(common_score[idx_gate], cfg["support_common_q"])) if ng >= 10 else 0.50)) |
            ((conf_tri >= (float(np.quantile(conf_tri[idx_gate], cfg["support_conf_q"])) if ng >= 10 else 0.52)) &
             (unknown_score <= (float(np.quantile(unknown_score[idx_gate], cfg["support_unknown_q"])) if ng >= 10 else 0.60)))
        ) & (~np.isin(np.arange(len(gate_pred)), idx_unk))
        support_idx = idx_gate[cand_mask[idx_gate]]
        support_idx = _concat_unique_idx(idx_rel, idx_amb, support_idx)

    if len(support_idx) == 0:
        support_idx = _concat_unique_idx(idx_rel, idx_amb)
    support_idx = support_idx if len(support_idx) > 0 else idx_gate

    P_seed = refine_probs_multi_stage(Z_adapt, P_tri, protos, idx_support=support_idx, iters=max(REFINE_ITERS, 3), src_mix=cfg["support_refine_mix"])
    if len(idx_amb) > 0:
        P_seed[idx_amb] = restrict_topk_probs(P_seed[idx_amb], k=int(cfg["ambiguous_topk"]), sharpen=float(cfg["ambiguous_sharpen"]))

    support_score = np.maximum(common_score[support_idx], cfg["common_score_blend"][1] * rescue_score[support_idx]) - cfg["support_unknown_penalty"] * unknown_score[support_idx]
    med = float(np.median(support_score)) if len(support_score) > 0 else 0.0
    std = float(np.std(support_score)) if len(support_score) > 0 else 1.0
    support_weight = cfg["support_weight_floor"] + cfg["support_weight_scale"] * sigmoid_np((support_score - med) / max(1e-6, std + 1e-6))
    support_weight = support_weight.astype(np.float32)
    if len(idx_rescue) > 0:
        rescue_local = np.nonzero(np.isin(support_idx, idx_rescue))[0]
        support_weight[rescue_local] = np.maximum(support_weight[rescue_local], cfg["rescue_min_weight"])
    if len(idx_rel) > 0:
        rel_local = np.nonzero(np.isin(support_idx, idx_rel))[0]
        support_weight[rel_local] = np.maximum(support_weight[rel_local], cfg["reliable_min_weight"])

    w_rel = np.asarray([], dtype=np.float32)
    if len(idx_rel) > 0:
        rel_score = 0.70 * common_score[idx_rel] + 0.30 * conf_tri[idx_rel]
        w_rel = 0.80 + 0.20 * normalize_conf_weights(rel_score)
    w_amb = np.asarray([], dtype=np.float32)
    if len(idx_amb) > 0:
        amb_score_loc = 0.55 * np.maximum(common_score[idx_amb], rescue_score[idx_amb]) + 0.45 * conf_tri[idx_amb]
        w_amb = cfg["ambiguous_min_weight"] + 0.30 * normalize_conf_weights(amb_score_loc)

    aux = dict(
        common_score=common_score.astype(np.float32),
        rescue_score=rescue_score.astype(np.float32),
        unknown_score=unknown_score.astype(np.float32),
        P_seed=P_seed.astype(np.float32),
        w_rel=w_rel.astype(np.float32),
        w_amb=w_amb.astype(np.float32),
        support_idx=support_idx.astype(np.int64),
        support_weight=support_weight.astype(np.float32),
        rescue_idx=idx_rescue.astype(np.int64))
    info = dict(
        reliable_size=int(len(idx_rel)),
        ambiguous_size=int(len(idx_amb)),
        unknown_size=int(len(idx_unk)),
        rescue_size=int(len(idx_rescue)),
        support_size=int(len(support_idx)),
        reliable_keep=float(len(idx_rel) / max(1, len(gate_pred))),
        ambiguous_keep=float(len(idx_amb) / max(1, len(gate_pred))),
        unknown_keep=float(len(idx_unk) / max(1, len(gate_pred))),
        common_score_gate_mean=float(common_score[idx_gate].mean()),
        unknown_score_gate_mean=float(unknown_score[idx_gate].mean()))
    return idx_rel.astype(np.int64), idx_amb.astype(np.int64), idx_unk.astype(np.int64), aux, info


def adapt_dg_raincoat_lite_versioned(
    fc_layer,
    Z_replay,
    y_replay,
    Z_target,
    idx_gate,
    idx_rel,
    idx_amb,
    idx_unk,
    q_seed,
    w_rel=None,
    w_amb=None,
    protos=None,
    Sid_adapt=None,
    Sdom_adapt=None,
    bottleneck=64,
    scale=0.3,
    epochs_stage1=DG_STAGE1_EPOCHS,
    epochs_stage2=DG_STAGE2_EPOCHS,
    lr=1e-3,
    wd=1e-4,
    batch=128,
    lambda_replay_ce=1.0,
    lambda_replay_anchor=1.0,
    lambda_align_stage1=DG_LAMBDA_ALIGN_STAGE1,
    lambda_align_stage2=DG_LAMBDA_ALIGN_STAGE2,
    lambda_ent_min=DG_LAMBDA_ENT_MIN,
    lambda_ent_max=DG_LAMBDA_ENT_MAX,
    lambda_cons=DG_LAMBDA_CONS,
    lambda_proto=DG_LAMBDA_PROTO,
    lambda_u_repel=DG_LAMBDA_UNKNOWN_REPEL,
    version="v10",
    unknown_loss_mode=UNKNOWN_LOSS_DEFAULT,
    unknown_energy_margin=UNKNOWN_ENERGY_MARGIN):
    cfg = dict(DGR_VERSION_CFG.get(version, DGR_VERSION_CFG["v10"]))
    idx_gate = np.asarray(idx_gate, dtype=np.int64)
    idx_rel = np.asarray(idx_rel, dtype=np.int64)
    idx_amb = np.asarray(idx_amb, dtype=np.int64)
    idx_unk = np.asarray(idx_unk, dtype=np.int64)
    if len(idx_gate) == 0:
        return None, {}

    q_seed = normalize_rows(q_seed.astype(np.float32))
    w_rel = np.ones((len(idx_rel)), dtype=np.float32) if (w_rel is None or len(w_rel) == 0) else np.asarray(w_rel, dtype=np.float32)
    w_amb = np.ones((len(idx_amb)), dtype=np.float32) * cfg["warm_amb_init"] if (w_amb is None or len(w_amb) == 0) else np.asarray(w_amb, dtype=np.float32)

    idx_sup0 = _concat_unique_idx(idx_rel, idx_amb)
    w_sup0 = np.concatenate([cfg["warm_rel_scale"] * w_rel, cfg["warm_amb_scale"] * w_amb], axis=0) if len(idx_sup0) > 0 else np.ones((0), dtype=np.float32)

    warm = adapt_on_embeddings_generic(
        fc_layer=fc_layer,
        Z_replay=Z_replay, y_replay=y_replay,
        Z_sup=Z_target[idx_sup0] if len(idx_sup0) > 0 else None,
        q_sup=q_seed[idx_sup0] if len(idx_sup0) > 0 else None,
        w_sup=w_sup0 if len(idx_sup0) > 0 else None,
        Z_align=Z_target[idx_gate],
        Z_unrel=Z_target[idx_unk] if len(idx_unk) > 0 else None,
        proto_bank=protos,
        bottleneck=bottleneck, scale=scale, epochs=epochs_stage1,
        lr=lr, wd=wd, batch=batch,
        lambda_replay_ce=lambda_replay_ce, lambda_replay_anchor=lambda_replay_anchor,
        lambda_sup=1.0,
        lambda_align=lambda_align_stage1,
        lambda_ent_min=lambda_ent_min * 0.8,
        lambda_ent_max=lambda_ent_max * 0.35,
        lambda_cons=lambda_cons * 0.8,
        lambda_proto=lambda_proto * 0.8,
        dynamic_align=True)
    if warm is None:
        return None, {}

    Z_gate1, logits_gate1 = apply_adapter_and_fc(warm, fc_layer, Z_target[idx_gate], batch=512)
    P_logit1 = softmax_np(logits_gate1)
    P_proto1 = proto_probs_cosine(Z_gate1, protos, temp=PROTO_T, l2norm=True)
    P_mix1 = weighted_prob_fusion([P_logit1, P_proto1, q_seed[idx_gate]], [1.0, 1.0, cfg["mix_seed_blend"]])

    conf1 = msp_from_probs(P_mix1)
    margin1 = top2_margin(P_mix1)
    y1 = np.argmax(P_mix1, axis=1)
    y0 = np.argmax(q_seed[idx_gate], axis=1)
    y_logit1 = np.argmax(P_logit1, axis=1)
    y_proto1 = np.argmax(P_proto1, axis=1)
    stable_vote = (((y1 == y0).astype(np.float32) + (y1 == y_logit1).astype(np.float32) + (y1 == y_proto1).astype(np.float32)) / 3.0).astype(np.float32)
    stable_hard = (stable_vote >= float(cfg.get("stable_hard_thr", 2.0 / 3.0))).astype(np.float32)
    stable_used = stable_vote if version == "v12" else stable_hard

    sid_n = robust_unit_interval(Sid_adapt, idx_gate)[idx_gate] if Sid_adapt is not None else np.ones((len(idx_gate)), dtype=np.float32) * 0.5
    sdom_n = robust_unit_interval(Sdom_adapt, idx_gate)[idx_gate] if Sdom_adapt is not None else np.ones((len(idx_gate)), dtype=np.float32) * 0.5
    conf_n = robust_unit_interval(conf1, np.arange(len(idx_gate)))
    margin_n = robust_unit_interval(margin1, np.arange(len(idx_gate)))
    _, proto_margin1, proto_dist1 = prototype_margin_dist(Z_gate1, protos)
    proto_margin_n = robust_unit_interval(proto_margin1, np.arange(len(idx_gate)))
    proto_dist_n = robust_unit_interval(proto_dist1, np.arange(len(idx_gate)))
    Z0n = normalize_vec_rows_np(Z_target[idx_gate])
    Z1n = normalize_vec_rows_np(Z_gate1)
    move = np.sqrt(np.maximum(1e-12, np.sum((Z1n - Z0n) ** 2, axis=1)))
    move_n = robust_unit_interval(move, np.arange(len(idx_gate)))
    move_low_n = 1.0 - move_n

    common_score = (
        0.30 * sid_n + 0.18 * sdom_n + 0.14 * conf_n + 0.08 * margin_n +
        0.10 * stable_used + 0.08 * proto_margin_n + 0.08 * move_low_n + 0.04 * (1.0 - proto_dist_n)
    ).astype(np.float32)
    rescue_score = (
        0.22 * sid_n + 0.22 * sdom_n + 0.18 * conf_n + 0.10 * margin_n +
        0.12 * stable_used + 0.10 * proto_margin_n + 0.06 * move_low_n
    ).astype(np.float32)
    unknown_score = (
        0.38 * (1.0 - sid_n) + 0.18 * proto_dist_n + 0.12 * (1.0 - conf_n) + 0.08 * (1.0 - margin_n) +
        0.08 * move_n + 0.08 * (1.0 - stable_used) + 0.08 * (1.0 - sdom_n)
    ).astype(np.float32)

    ng = len(idx_gate)
    if cfg["rescue_mode"] == "low_sdom":
        sdom_thr = float(np.quantile(sdom_n, cfg["rescue_sdom_q"])) if ng >= 10 else 0.50
        sdom_cond = sdom_n <= sdom_thr
    else:
        sdom_thr = float(np.quantile(sdom_n, cfg["rescue_sdom_q"])) if ng >= 10 else 0.58
        sdom_cond = sdom_n >= sdom_thr
    rescue_mask_local = (
        (sid_n >= (float(np.quantile(sid_n, cfg["rescue_sid_q"])) if ng >= 10 else 0.55)) &
        (conf1 >= max(float(np.quantile(conf1, 0.52)) if ng >= 10 else cfg["rescue_conf"], cfg["rescue_conf"])) &
        (margin1 >= float(cfg["rescue_margin"])) &
        (proto_margin_n >= (float(np.quantile(proto_margin_n, cfg["rescue_proto_q"])) if ng >= 10 else 0.35)) &
        sdom_cond &
        (stable_hard >= 1.0)
    )

    extreme_unk_thr = float(np.quantile(unknown_score, DG_UNKNOWN_EXTREME_Q)) if ng >= 10 else float(np.max(unknown_score))
    local_all = np.arange(ng, dtype=np.int64)
    local_unk = local_all[(unknown_score >= extreme_unk_thr) & (~rescue_mask_local)]
    idx_unk1 = idx_gate[local_unk]

    if cfg["support_mode"] == "nonunk":
        support_local = np.setdiff1d(local_all, local_unk, assume_unique=False)
    else:
        candidate_mask = (
            rescue_mask_local |
            (common_score >= (float(np.quantile(common_score, cfg["support_common_q"])) if ng >= 10 else 0.50)) |
            ((stable_hard >= 1.0) & (conf1 >= (float(np.quantile(conf1, cfg["support_conf_q"])) if ng >= 10 else cfg["rescue_conf"])) &
             (unknown_score <= (float(np.quantile(unknown_score, cfg["support_unknown_q"])) if ng >= 10 else 0.60)))
        ) & (unknown_score < extreme_unk_thr)
        support_local = local_all[candidate_mask]
        if len(support_local) < max(16, ng // 8):
            support_score_fallback = np.maximum(common_score, 0.95 * rescue_score) - 0.20 * unknown_score
            k_support = min(ng, max(16, ng // int(max(2, cfg["support_fallback_div"]))))
            support_local = np.argsort(-support_score_fallback)[:k_support]
            support_local = np.setdiff1d(support_local, local_unk, assume_unique=False)
    if len(support_local) == 0:
        support_local = np.setdiff1d(local_all, local_unk, assume_unique=False)
        if len(support_local) == 0:
            support_local = local_all.copy()
            idx_unk1 = np.zeros((0), dtype=np.int64)
    support_idx = idx_gate[support_local]

    q_stage2_local = refine_probs_multi_stage(Z_gate1, P_mix1, protos, idx_support=support_local, iters=max(REFINE_ITERS, 3), src_mix=cfg["support_refine_mix"])
    q_stage2 = q_seed.copy()
    q_stage2[idx_gate] = q_stage2_local.astype(np.float32)

    support_score = np.maximum(common_score[support_local], 0.96 * rescue_score[support_local]) - cfg["support_unknown_penalty"] * unknown_score[support_local]
    med = float(np.median(support_score)) if len(support_score) > 0 else 0.0
    std = float(np.std(support_score)) if len(support_score) > 0 else 1.0
    support_weight = cfg["support_weight_floor"] + cfg["support_weight_scale"] * sigmoid_np((support_score - med) / max(1e-6, std + 1e-6))
    support_weight = support_weight.astype(np.float32)

    rescue_local = support_local[rescue_mask_local[support_local]]
    if len(rescue_local) > 0:
        rescue_pos = np.nonzero(np.isin(support_local, rescue_local))[0]
        support_weight[rescue_pos] = np.maximum(support_weight[rescue_pos], cfg["rescue_min_weight"])

    reliable_mask_local = common_score[support_local] >= max(float(np.quantile(common_score[support_local], 0.80)) if len(support_local) >= 10 else 0.72, 0.72)
    reliable_pos = np.nonzero(reliable_mask_local)[0]
    if len(reliable_pos) > 0:
        support_weight[reliable_pos] = np.maximum(support_weight[reliable_pos], cfg["reliable_min_weight"])

    ambiguous_pos = np.nonzero((~reliable_mask_local) & (support_weight < cfg["reliable_min_weight"]))[0]
    amb_near_size = 0
    amb_far_size = 0
    if len(ambiguous_pos) > 0:
        support_weight[ambiguous_pos] = np.maximum(support_weight[ambiguous_pos], cfg["ambiguous_min_weight"])
        if version == "v12":
            amb_idx_local = support_local[ambiguous_pos]
            amb_rank = (0.48 * np.maximum(common_score[amb_idx_local], rescue_score[amb_idx_local]) + 0.28 * conf1[amb_idx_local] + 0.14 * stable_used[amb_idx_local] - 0.10 * unknown_score[amb_idx_local]).astype(np.float32)
            amb_thr = float(np.quantile(amb_rank, cfg.get("amb_near_q", 0.58))) if len(amb_rank) >= 10 else float(np.median(amb_rank))
            near_mask = amb_rank >= amb_thr
            amb_near_pos = ambiguous_pos[near_mask]
            amb_far_pos = ambiguous_pos[~near_mask]
            amb_near_size = int(len(amb_near_pos))
            amb_far_size = int(len(amb_far_pos))
            if len(amb_near_pos) > 0:
                support_weight[amb_near_pos] = np.maximum(support_weight[amb_near_pos], cfg.get("amb_near_min_weight", cfg["ambiguous_min_weight"]))
            if len(amb_far_pos) > 0:
                support_weight[amb_far_pos] = np.minimum(np.maximum(support_weight[amb_far_pos], cfg.get("amb_far_min_weight", 0.04)), cfg.get("amb_far_max_weight", 0.10))
                q_stage2[support_idx[amb_far_pos]] = restrict_topk_probs(q_stage2[support_idx[amb_far_pos]], k=2, sharpen=cfg.get("amb_far_sharpen", 1.03))

    if len(support_local) > 0:
        low_weight_mask = support_weight < cfg["low_weight_cut"]
        if np.any(low_weight_mask):
            q_stage2[support_idx[low_weight_mask]] = restrict_topk_probs(q_stage2[support_idx[low_weight_mask]], k=2, sharpen=cfg["ambiguous_sharpen"])

    if cfg["align_mode"] == "safe_nonunk":
        align_mask = (common_score >= (float(np.quantile(common_score, cfg["align_common_q"])) if ng >= 10 else 0.40)) & (unknown_score <= (float(np.quantile(unknown_score, cfg["align_unknown_q"])) if ng >= 10 else 0.80))
        align_idx = idx_gate[align_mask]
        align_idx = np.setdiff1d(align_idx, idx_unk1, assume_unique=False)
        if len(align_idx) == 0:
            align_idx = np.setdiff1d(idx_gate, idx_unk1, assume_unique=False)
    else:
        align_idx = np.setdiff1d(idx_gate, idx_unk1, assume_unique=False)
    if len(align_idx) == 0:
        align_idx = support_idx

    final_adapter = adapt_on_embeddings_generic(
        fc_layer=fc_layer,
        Z_replay=Z_replay, y_replay=y_replay,
        Z_sup=Z_target[support_idx] if len(support_idx) > 0 else None,
        q_sup=q_stage2[support_idx] if len(support_idx) > 0 else None,
        w_sup=support_weight if len(support_idx) > 0 else None,
        Z_align=Z_target[align_idx],
        Z_unrel=Z_target[idx_unk1] if len(idx_unk1) > 0 else None,
        proto_bank=protos,
        bottleneck=bottleneck, scale=scale, epochs=epochs_stage2,
        lr=lr, wd=wd, batch=batch,
        lambda_replay_ce=lambda_replay_ce, lambda_replay_anchor=lambda_replay_anchor,
        lambda_sup=1.0,
        lambda_align=lambda_align_stage2,
        lambda_ent_min=lambda_ent_min,
        lambda_ent_max=lambda_ent_max * cfg["final_ent_max_scale"],
        lambda_cons=lambda_cons,
        lambda_proto=lambda_proto,
        dynamic_align=True,
        init_state=clone_state_dict_cpu(warm))

    info = dict(
        reliable_size=int(np.sum(support_weight >= cfg["reliable_min_weight"])),
        ambiguous_size=int(np.sum((support_weight >= cfg["ambiguous_min_weight"]) & (support_weight < cfg["reliable_min_weight"]))),
        unknown_size=int(len(idx_unk1)),
        rescue_size=int(len(rescue_local)),
        support_size=int(len(support_idx)),
        reliable_keep=float(np.sum(support_weight >= cfg["reliable_min_weight"]) / max(1, len(Z_target))),
        ambiguous_keep=float(np.sum((support_weight >= cfg["ambiguous_min_weight"]) & (support_weight < cfg["reliable_min_weight"])) / max(1, len(Z_target))),
        unknown_keep=float(len(idx_unk1) / max(1, len(Z_target))),
        common_score_gate_mean=float(common_score.mean()) if len(common_score) > 0 else float("nan"),
        unknown_score_gate_mean=float(unknown_score.mean()) if len(unknown_score) > 0 else float("nan"),
        move_gate_mean=float(move.mean()) if len(move) > 0 else float("nan"),
        stable_pred_ratio=float(stable_hard.mean()) if len(stable_hard) > 0 else float("nan"),
        stability_soft_mean=float(stable_used.mean()) if len(stable_used) > 0 else float("nan"),
        amb_near_size=int(amb_near_size),
        amb_far_size=int(amb_far_size),
        stage2_reliable_size=int(np.sum(support_weight >= cfg["reliable_min_weight"])),
        stage2_ambiguous_size=int(np.sum((support_weight >= cfg["ambiguous_min_weight"]) & (support_weight < cfg["reliable_min_weight"]))),
        stage2_unknown_size=int(len(idx_unk1)),
        rescued_common_size=int(len(rescue_local)))
    final_adapter = final_adapter if final_adapter is not None else warm
    final_adapter, guard_info = guard_adapter_with_fallback(final_adapter, warm, fc_layer, Z_replay, y_replay, Z_probe=Z_target[idx_gate], probe_name=f"dg_{version}")
    info.update(guard_info)
    return final_adapter, info


def build_three_bucket_partition_v8_versioned(
    K,
    Z_adapt,
    gate_pred,
    Sid_adapt,
    Sdom_adapt,
    P_logit_adapt,
    P_proto_adapt,
    P_knn_adapt,
    protos,
    tau_conf,
    min_keep=64,
    version="v10"):
    cfg = dict(TBV8_VERSION_CFG.get(version, TBV8_VERSION_CFG["v10"]))
    idx_gate = np.where(gate_pred == 1)[0]
    n = len(gate_pred)
    empty = np.zeros((0), dtype=np.int64)
    if len(idx_gate) == 0:
        aux = dict(
            bucket_score=np.zeros((n), dtype=np.float32),
            unknown_score=np.zeros((n), dtype=np.float32),
            P_seed=weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN]),
            w_rel=np.zeros((0), dtype=np.float32),
            w_amb=np.zeros((0), dtype=np.float32),
            w_weak=np.zeros((0), dtype=np.float32),
            idx_weak_cold=np.zeros((0), dtype=np.int64),
            w_weak_cold=np.zeros((0), dtype=np.float32),
            support_idx=np.zeros((0), dtype=np.int64))
        info = dict(reliable_size=0, ambiguous_size=0, weak_size=0, cold_weak_size=0, unknown_size=0)
        return empty, empty, empty, empty, aux, info

    P_tri = weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN])
    y_tri = np.argmax(P_tri, axis=1)
    y_logit = np.argmax(P_logit_adapt, axis=1)
    y_proto = np.argmax(P_proto_adapt, axis=1)
    y_knn = np.argmax(P_knn_adapt, axis=1)
    conf_tri = msp_from_probs(P_tri)
    margin_tri = top2_margin(P_tri)
    _, proto_margin, proto_dist = prototype_margin_dist(Z_adapt, protos)

    agree = ((y_logit == y_proto).astype(np.float32) + (y_logit == y_knn).astype(np.float32) + (y_proto == y_knn).astype(np.float32)) / 3.0
    sid_n = robust_unit_interval(Sid_adapt, idx_gate)
    sdom_n = robust_unit_interval(Sdom_adapt, idx_gate)
    conf_n = robust_unit_interval(conf_tri, idx_gate)
    margin_n = robust_unit_interval(margin_tri, idx_gate)
    proto_margin_n = robust_unit_interval(proto_margin, idx_gate)
    proto_dist_n = robust_unit_interval(proto_dist, idx_gate)

    bucket_score = (
        0.30 * sid_n + 0.14 * sdom_n + 0.18 * conf_n + 0.12 * agree +
        0.12 * margin_n + 0.10 * proto_margin_n + 0.04 * (1.0 - proto_dist_n)
    ).astype(np.float32)
    unknown_score = (
        0.38 * (1.0 - sid_n) + 0.18 * proto_dist_n + 0.14 * (1.0 - conf_n) +
        0.12 * (1.0 - margin_n) + 0.10 * (1.0 - agree) + 0.08 * (1.0 - sdom_n)
    ).astype(np.float32)

    ng = len(idx_gate)
    unk_gap = float(np.quantile(unknown_score[idx_gate], 0.92) - np.quantile(unknown_score[idx_gate], 0.65)) if ng >= 10 else 0.0
    unk_q = cfg["unknown_keep_q_hard"] if unk_gap >= 0.08 else cfg["unknown_keep_q"]

    rel_mask = (bucket_score >= float(np.quantile(bucket_score[idx_gate], max(0.50, 1.0 - cfg["rel_keep_q"])))) & (conf_tri >= max(float(tau_conf), 0.55))
    idx_rel = select_idx_by_mask_with_fallback(idx_gate, rel_mask[idx_gate], bucket_score, min_keep=max(16, min_keep // 2))
    rem1 = np.setdiff1d(idx_gate, idx_rel, assume_unique=False)
    k_unk = min(len(rem1), max(int(round(unk_q * ng)), max(8, min_keep // 4)))
    idx_unk = rem1[np.argsort(-unknown_score[rem1])[:k_unk]] if len(rem1) > 0 and k_unk > 0 else np.zeros((0), dtype=np.int64)

    rem2 = np.setdiff1d(rem1, idx_unk, assume_unique=False)
    amb_q = max(cfg["amb_quant_floor"], 1.0 - (cfg["rel_keep_q"] + cfg["amb_keep_q"]))
    amb_mask = (bucket_score >= float(np.quantile(bucket_score[idx_gate], amb_q))) & (conf_tri >= max(float(tau_conf) * cfg["amb_conf_scale"], cfg["amb_conf_floor"]))
    idx_amb = select_idx_by_mask_with_fallback(rem2, amb_mask[rem2] if len(rem2) > 0 else np.zeros((0), dtype=bool), bucket_score, min_keep=max(16, min_keep // 3))
    rem3 = np.setdiff1d(rem2, idx_amb, assume_unique=False)

    weak_strength = 0.60 * bucket_score[rem3] + 0.40 * conf_tri[rem3] - 0.25 * unknown_score[rem3] if len(rem3) > 0 else np.zeros((0), dtype=np.float32)
    if len(rem3) > 0:
        cold_thr = float(np.quantile(weak_strength, cfg["cold_weak_q"])) if len(rem3) >= 10 else float(np.median(weak_strength))
        idx_weak = rem3[weak_strength >= cold_thr]
        idx_weak_cold = rem3[weak_strength < cold_thr]
    else:
        idx_weak = np.zeros((0), dtype=np.int64)
        idx_weak_cold = np.zeros((0), dtype=np.int64)

    support_idx = _concat_unique_idx(idx_rel, idx_amb, idx_weak)
    support_idx = support_idx if len(support_idx) > 0 else idx_gate
    P_seed = refine_probs_multi_stage(Z_adapt, P_tri, protos, idx_support=support_idx, iters=max(REFINE_ITERS, 3), src_mix=0.82)
    if len(idx_amb) > 0:
        P_seed[idx_amb] = restrict_topk_probs(P_seed[idx_amb], k=max(2, AMBIG_TOPK), sharpen=1.01)
    if len(idx_weak) > 0:
        P_seed[idx_weak] = restrict_topk_probs(P_seed[idx_weak], k=max(2, int(cfg["semiweak_topk"])), sharpen=0.99)
    if len(idx_weak_cold) > 0:
        P_seed[idx_weak_cold] = restrict_topk_probs(P_seed[idx_weak_cold], k=2, sharpen=0.96)

    w_rel = np.asarray([], dtype=np.float32)
    if len(idx_rel) > 0:
        rel_score = 0.65 * bucket_score[idx_rel] + 0.35 * conf_tri[idx_rel]
        w_rel = V8_REL_WEIGHT_FLOOR + (1.0 - V8_REL_WEIGHT_FLOOR) * normalize_conf_weights(rel_score)
    w_amb = np.asarray([], dtype=np.float32)
    if len(idx_amb) > 0:
        amb_score = 0.60 * bucket_score[idx_amb] + 0.40 * conf_tri[idx_amb]
        w_amb = V8_AMB_WEIGHT_FLOOR + (V8_AMB_WEIGHT_SCALE + cfg["amb_weight_bonus"]) * normalize_conf_weights(amb_score)
    w_weak = np.asarray([], dtype=np.float32)
    if len(idx_weak) > 0:
        weak_score = 0.60 * bucket_score[idx_weak] + 0.40 * conf_tri[idx_weak]
        w_weak = V8_WEAK_WEIGHT_FLOOR + (V8_WEAK_WEIGHT_SCALE + cfg["weak_weight_bonus"]) * normalize_conf_weights(weak_score)
    w_weak_cold = np.asarray([], dtype=np.float32)
    if len(idx_weak_cold) > 0:
        cold_score = 0.60 * bucket_score[idx_weak_cold] + 0.40 * conf_tri[idx_weak_cold]
        w_weak_cold = V81_COLD_WEIGHT_FLOOR + V81_COLD_WEIGHT_SCALE * normalize_conf_weights(cold_score)

    aux = dict(
        bucket_score=bucket_score.astype(np.float32),
        unknown_score=unknown_score.astype(np.float32),
        P_seed=P_seed.astype(np.float32),
        w_rel=w_rel.astype(np.float32),
        w_amb=w_amb.astype(np.float32),
        w_weak=w_weak.astype(np.float32),
        idx_weak_cold=idx_weak_cold.astype(np.int64),
        w_weak_cold=w_weak_cold.astype(np.float32),
        support_idx=support_idx.astype(np.int64))
    info = dict(
        reliable_size=int(len(idx_rel)),
        ambiguous_size=int(len(idx_amb)),
        weak_size=int(len(idx_weak)),
        cold_weak_size=int(len(idx_weak_cold)),
        unknown_size=int(len(idx_unk)),
        reliable_keep=float(len(idx_rel) / max(1, len(gate_pred))),
        ambiguous_keep=float(len(idx_amb) / max(1, len(gate_pred))),
        weak_keep=float(len(idx_weak) / max(1, len(gate_pred))),
        unknown_keep=float(len(idx_unk) / max(1, len(gate_pred))),
        bucket_score_gate_mean=float(bucket_score[idx_gate].mean()),
        unknown_score_gate_mean=float(unknown_score[idx_gate].mean()))
    return idx_rel.astype(np.int64), idx_amb.astype(np.int64), idx_weak.astype(np.int64), idx_unk.astype(np.int64), aux, info


def adapt_three_bucket_v8_versioned(
    fc_layer,
    Z_replay,
    y_replay,
    Z_target,
    idx_gate,
    idx_rel,
    idx_amb,
    idx_weak,
    idx_unk,
    q_seed,
    w_rel=None,
    w_amb=None,
    w_weak=None,
    proto_bank=None,
    bucket_score=None,
    unknown_score=None,
    Sid_adapt=None,
    Sdom_adapt=None,
    idx_weak_cold=None,
    w_weak_cold=None,
    bottleneck=64,
    scale=0.3,
    stage0_epochs=V8_STAGE0_EPOCHS,
    refresh_epochs=V8_REFRESH_EPOCHS,
    refresh_rounds=V8_REFRESH_ROUNDS,
    lr=1e-3,
    wd=1e-4,
    batch=128,
    lambda_replay_ce=1.0,
    lambda_replay_anchor=1.0,
    lambda_align=V8_LAMBDA_ALIGN,
    lambda_ent_min=V8_LAMBDA_ENT_MIN,
    lambda_ent_max=V8_LAMBDA_ENT_MAX,
    lambda_cons=V8_LAMBDA_CONS,
    lambda_proto=V8_LAMBDA_PROTO,
    lambda_u_repel=V8_LAMBDA_UNKNOWN_REPEL,
    version="v10",
    unknown_loss_mode=UNKNOWN_LOSS_DEFAULT,
    unknown_energy_margin=UNKNOWN_ENERGY_MARGIN):
    cfg = dict(TBV8_VERSION_CFG.get(version, TBV8_VERSION_CFG["v10"]))
    idx_gate = np.asarray(idx_gate, dtype=np.int64)
    if len(idx_gate) == 0:
        return None, {}

    K = q_seed.shape[1]
    q_curr = normalize_rows(q_seed.astype(np.float32).copy())
    idx_rel_curr = np.asarray(idx_rel, dtype=np.int64)
    idx_amb_curr = np.asarray(idx_amb, dtype=np.int64)
    idx_weak_curr = np.asarray(idx_weak, dtype=np.int64)
    idx_unk_curr = np.asarray(idx_unk, dtype=np.int64)
    idx_weak_cold_curr = np.asarray(idx_weak_cold if idx_weak_cold is not None else np.zeros((0), dtype=np.int64), dtype=np.int64)

    if bucket_score is None:
        bucket_score = np.zeros((len(Z_target)), dtype=np.float32)
    if unknown_score is None:
        unknown_score = np.zeros((len(Z_target)), dtype=np.float32)

    sid_n_full = robust_unit_interval(Sid_adapt, idx_gate)[idx_gate] if Sid_adapt is not None else np.ones((len(idx_gate)), dtype=np.float32) * 0.5
    sdom_n_full = robust_unit_interval(Sdom_adapt, idx_gate)[idx_gate] if Sdom_adapt is not None else np.ones((len(idx_gate)), dtype=np.float32) * 0.5

    w_rel_curr = np.ones((len(idx_rel_curr)), dtype=np.float32) if (w_rel is None or len(w_rel) == 0) else np.asarray(w_rel, dtype=np.float32)
    w_amb_curr = np.ones((len(idx_amb_curr)), dtype=np.float32) * V8_AMB_WEIGHT_FLOOR if (w_amb is None or len(w_amb) == 0) else np.asarray(w_amb, dtype=np.float32)
    w_weak_curr = np.ones((len(idx_weak_curr)), dtype=np.float32) * V8_WEAK_WEIGHT_FLOOR if (w_weak is None or len(w_weak) == 0) else np.asarray(w_weak, dtype=np.float32)
    w_weak_cold_curr = np.ones((len(idx_weak_cold_curr)), dtype=np.float32) * V81_COLD_WEIGHT_FLOOR if (w_weak_cold is None or len(w_weak_cold) == 0) else np.asarray(w_weak_cold, dtype=np.float32)

    state = None
    promo_total_rel = 0
    promo_total_amb = 0
    refresh_stats = []
    prev_pred = np.full((len(idx_gate)), -1, dtype=np.int64)
    stable_count = np.zeros((len(idx_gate)), dtype=np.int64)

    def _pack_sup(rel_idx, amb_idx, weak_idx):
        sup_idx = _concat_unique_idx(rel_idx, amb_idx, weak_idx)
        if len(sup_idx) == 0:
            return sup_idx, None, None
        q_sup = q_curr[sup_idx]
        parts = []
        if len(rel_idx) > 0:
            parts.append(w_rel_curr)
        if len(amb_idx) > 0:
            parts.append(w_amb_curr)
        if len(weak_idx) > 0:
            parts.append(w_weak_curr)
        w_sup = np.concatenate(parts, axis=0) if len(parts) > 0 else None
        return sup_idx, q_sup, w_sup

    sup0_idx, sup0_q, sup0_w = _pack_sup(idx_rel_curr, idx_amb_curr, np.zeros((0), dtype=np.int64))
    warm = adapt_on_embeddings_generic(
        fc_layer=fc_layer,
        Z_replay=Z_replay, y_replay=y_replay,
        Z_sup=Z_target[sup0_idx] if len(sup0_idx) > 0 else None,
        q_sup=sup0_q if sup0_q is not None else None,
        w_sup=sup0_w if sup0_w is not None else None,
        Z_align=Z_target[idx_gate],
        Z_unrel=Z_target[idx_unk_curr] if len(idx_unk_curr) > 0 else None,
        proto_bank=proto_bank,
        bottleneck=bottleneck, scale=scale, epochs=stage0_epochs,
        lr=lr, wd=wd, batch=batch,
        lambda_replay_ce=lambda_replay_ce, lambda_replay_anchor=lambda_replay_anchor,
        lambda_sup=1.0,
        lambda_align=lambda_align,
        lambda_ent_min=lambda_ent_min,
        lambda_ent_max=lambda_ent_max * 0.45,
        lambda_cons=lambda_cons * 0.75,
        lambda_proto=lambda_proto * 0.8,
        dynamic_align=True,
        init_state=state)
    if warm is None:
        return None, {}
    state = clone_state_dict_cpu(warm)
    final_adapter = warm

    for rr in range(int(max(1, refresh_rounds))):
        sup_idx_round, sup_q_round, sup_w_round = _pack_sup(idx_rel_curr, idx_amb_curr, idx_weak_curr)
        adapter_r = adapt_on_embeddings_generic(
            fc_layer=fc_layer,
            Z_replay=Z_replay, y_replay=y_replay,
            Z_sup=Z_target[sup_idx_round] if len(sup_idx_round) > 0 else None,
            q_sup=sup_q_round if len(sup_idx_round) > 0 else None,
            w_sup=sup_w_round if len(sup_idx_round) > 0 else None,
            Z_align=Z_target[np.setdiff1d(idx_gate, idx_unk_curr, assume_unique=False)],
            Z_unrel=Z_target[idx_unk_curr] if len(idx_unk_curr) > 0 else None,
            proto_bank=proto_bank,
            bottleneck=bottleneck, scale=scale, epochs=refresh_epochs,
            lr=lr, wd=wd, batch=batch,
            lambda_replay_ce=lambda_replay_ce, lambda_replay_anchor=lambda_replay_anchor,
            lambda_sup=1.0,
            lambda_align=lambda_align,
            lambda_ent_min=lambda_ent_min,
            lambda_ent_max=lambda_ent_max,
            lambda_cons=lambda_cons,
            lambda_proto=lambda_proto,
            dynamic_align=True,
            init_state=state)
        if adapter_r is None:
            break
        final_adapter = adapter_r
        state = clone_state_dict_cpu(adapter_r)

        Zg, logits_g = apply_adapter_and_fc(adapter_r, fc_layer, Z_target[idx_gate], batch=512)
        P_logit_g = softmax_np(logits_g)
        P_proto_g = proto_probs_cosine(Zg, proto_bank, temp=PROTO_T, l2norm=True)
        P_mix_g = weighted_prob_fusion([P_logit_g, P_proto_g, q_curr[idx_gate]], [1.0, 1.0, 0.80])

        conf_g = msp_from_probs(P_mix_g)
        margin_g = top2_margin(P_mix_g)
        pred_g = np.argmax(P_mix_g, axis=1)
        stable_now = (pred_g == prev_pred)
        stable_count = np.where(stable_now, stable_count + 1, 0)
        prev_pred = pred_g.copy()

        conf_n = robust_unit_interval(conf_g, np.arange(len(idx_gate)))
        margin_n = robust_unit_interval(margin_g, np.arange(len(idx_gate)))
        _, proto_margin_g, proto_dist_g = prototype_margin_dist(Zg, proto_bank)
        proto_margin_n = robust_unit_interval(proto_margin_g, np.arange(len(idx_gate)))
        proto_dist_n = robust_unit_interval(proto_dist_g, np.arange(len(idx_gate)))
        stability_n = np.clip(stable_count.astype(np.float32) / float(max(1, V8_STABLE_K_REL + 1)), 0.0, 1.0)

        class_counts = np.bincount(pred_g, minlength=K).astype(np.float32)
        class_counts[class_counts <= 0] = 1.0
        rarity = np.sqrt(class_counts.mean() / class_counts)
        rarity = rarity / np.maximum(1.0, rarity.max())

        promote_score = (
            0.24 * sid_n_full + 0.12 * sdom_n_full + 0.20 * conf_n + 0.12 * margin_n +
            0.10 * proto_margin_n + 0.14 * stability_n + 0.08 * robust_unit_interval(bucket_score[idx_gate], np.arange(len(idx_gate)))
        ).astype(np.float32)
        rarity_boost = float(cfg.get("rarity_boost", 0.15))
        promote_score_pc = promote_score * ((1.0 - rarity_boost) + rarity_boost * rarity[pred_g])

        unknown_score_dyn = (
            0.34 * (1.0 - sid_n_full) + 0.16 * proto_dist_n + 0.14 * (1.0 - conf_n) +
            0.10 * (1.0 - margin_n) + 0.14 * (1.0 - stability_n) + 0.12 * (1.0 - sdom_n_full)
        ).astype(np.float32)

        unk_thr = float(np.quantile(unknown_score_dyn, max(0.82, 1.0 - cfg["unknown_keep_q"]))) if len(idx_gate) >= 10 else float(np.max(unknown_score_dyn))
        local_unk = np.where(unknown_score_dyn >= unk_thr)[0]
        idx_unk_curr = idx_gate[local_unk]

        local_all = np.arange(len(idx_gate), dtype=np.int64)
        local_nonunk = np.setdiff1d(local_all, local_unk, assume_unique=False)
        rel_budget = int(sum(max(1, int(np.ceil(cfg["class_rel_frac"] * c))) for c in class_counts if c > 0))
        amb_budget = int(sum(max(1, int(np.ceil(cfg["class_amb_frac"] * c))) for c in class_counts if c > 0))

        cand_rel_local = np.intersect1d(
            local_nonunk,
            np.where(
                (stable_count >= int(V8_STABLE_K_REL)) &
                (conf_g >= float(cfg["promote_conf_rel"])) &
                (margin_g >= float(cfg["promote_margin_rel"]))
            )[0],
            assume_unique=False)
        local_rel_sel = select_topk_class_balanced(
            cand_rel_local, pred_g, promote_score_pc,
            min(max(16, rel_budget), len(cand_rel_local)), K, min_per_class=1) if len(cand_rel_local) > 0 else np.zeros((0), dtype=np.int64)
        idx_rel_new = idx_gate[local_rel_sel]

        local_rem_after_rel = np.setdiff1d(local_nonunk, local_rel_sel, assume_unique=False)
        cand_amb_local = np.intersect1d(
            local_rem_after_rel,
            np.where(
                (stable_count >= int(V8_STABLE_K_AMB)) &
                (conf_g >= float(cfg["promote_conf_amb"])) &
                (margin_g >= float(cfg["promote_margin_amb"]))
            )[0],
            assume_unique=False)
        local_amb_sel = select_topk_class_balanced(
            cand_amb_local, pred_g, promote_score_pc,
            min(max(16, amb_budget), len(cand_amb_local)), K, min_per_class=1) if len(cand_amb_local) > 0 else np.zeros((0), dtype=np.int64)
        idx_amb_new = idx_gate[local_amb_sel]

        local_rem_after_amb = np.setdiff1d(local_rem_after_rel, local_amb_sel, assume_unique=False)
        if len(local_rem_after_amb) > 0:
            semiweak_candidates = local_rem_after_amb[
                (conf_g[local_rem_after_amb] >= max(0.45, float(cfg["promote_conf_amb"]) - 0.05)) &
                (margin_g[local_rem_after_amb] >= max(0.02, float(cfg["promote_margin_amb"]) - 0.02))
            ]
            extra_amb = []
            current_amb_pred = pred_g[local_amb_sel] if len(local_amb_sel) > 0 else np.zeros((0), dtype=np.int64)
            for k in range(K):
                target_k = max(1, int(np.ceil(cfg["class_amb_frac"] * class_counts[k]))) if class_counts[k] > 0 else 0
                cur_k = int(np.sum(current_amb_pred == k)) if len(current_amb_pred) > 0 else 0
                need_k = max(0, target_k - cur_k)
                if need_k <= 0:
                    continue
                cand_k = semiweak_candidates[pred_g[semiweak_candidates] == k]
                if len(cand_k) == 0:
                    continue
                score_k = promote_score_pc[cand_k]
                take_cap = int(cfg.get("extra_amb_cap", need_k))
                take_k = cand_k[np.argsort(-score_k)[:min(need_k, take_cap, len(cand_k))]]
                extra_amb.append(take_k)
            if len(extra_amb) > 0:
                extra_amb = np.concatenate(extra_amb, axis=0).astype(np.int64)
                local_amb_sel = np.unique(np.concatenate([local_amb_sel, extra_amb], axis=0)).astype(np.int64)

        local_rem_after_amb = np.setdiff1d(local_rem_after_rel, local_amb_sel, assume_unique=False)
        weak_strength = 0.56 * promote_score[local_rem_after_amb] + 0.26 * conf_n[local_rem_after_amb] + 0.18 * stability_n[local_rem_after_amb] - 0.16 * unknown_score_dyn[local_rem_after_amb] if len(local_rem_after_amb) > 0 else np.zeros((0), dtype=np.float32)
        if len(local_rem_after_amb) > 0:
            cold_thr = float(np.quantile(weak_strength, cfg["cold_weak_q"])) if len(local_rem_after_amb) >= 10 else float(np.median(weak_strength))
            local_weak_sel = local_rem_after_amb[weak_strength >= cold_thr]
            local_cold_sel = local_rem_after_amb[weak_strength < cold_thr]
        else:
            local_weak_sel = np.zeros((0), dtype=np.int64)
            local_cold_sel = np.zeros((0), dtype=np.int64)

        idx_rel_curr = idx_rel_new
        idx_amb_curr = idx_amb_new
        idx_weak_curr = idx_gate[local_weak_sel]
        idx_weak_cold_curr = idx_gate[local_cold_sel]

        support_curr = _concat_unique_idx(idx_rel_curr, idx_amb_curr, idx_weak_curr)
        local_support = np.nonzero(np.isin(idx_gate, support_curr))[0]
        if len(local_support) == 0:
            local_support = local_nonunk.copy()
            support_curr = idx_gate[local_support]

        q_local = refine_probs_multi_stage(Zg, P_mix_g, proto_bank, idx_support=local_support, iters=max(REFINE_ITERS, 3), src_mix=0.80)
        q_curr[idx_gate] = q_local.astype(np.float32)

        if len(idx_amb_curr) > 0:
            q_curr[idx_amb_curr] = restrict_topk_probs(q_curr[idx_amb_curr], k=max(2, AMBIG_TOPK), sharpen=1.01)
        if len(idx_weak_curr) > 0:
            q_curr[idx_weak_curr] = restrict_topk_probs(q_curr[idx_weak_curr], k=max(2, int(cfg["semiweak_topk"])), sharpen=0.99)
        if len(idx_weak_cold_curr) > 0:
            q_curr[idx_weak_cold_curr] = restrict_topk_probs(q_curr[idx_weak_cold_curr], k=2, sharpen=0.96)

        if len(idx_rel_curr) > 0:
            rel_local = np.nonzero(np.isin(idx_gate, idx_rel_curr))[0]
            w_rel_curr = V8_REL_WEIGHT_FLOOR + (1.0 - V8_REL_WEIGHT_FLOOR) * normalize_conf_weights(0.60 * promote_score[rel_local] + 0.40 * conf_g[rel_local])
        else:
            w_rel_curr = np.zeros((0), dtype=np.float32)
        if len(idx_amb_curr) > 0:
            amb_local = np.nonzero(np.isin(idx_gate, idx_amb_curr))[0]
            w_amb_curr = V8_AMB_WEIGHT_FLOOR + (V8_AMB_WEIGHT_SCALE + cfg["amb_weight_bonus"]) * normalize_conf_weights(0.55 * promote_score[amb_local] + 0.45 * conf_g[amb_local])
        else:
            w_amb_curr = np.zeros((0), dtype=np.float32)
        if len(idx_weak_curr) > 0:
            weak_local = np.nonzero(np.isin(idx_gate, idx_weak_curr))[0]
            w_weak_curr = V8_WEAK_WEIGHT_FLOOR + (V8_WEAK_WEIGHT_SCALE + cfg["weak_weight_bonus"]) * normalize_conf_weights(0.55 * promote_score[weak_local] + 0.25 * conf_g[weak_local] + 0.20 * stability_n[weak_local])
        else:
            w_weak_curr = np.zeros((0), dtype=np.float32)
        if len(idx_weak_cold_curr) > 0:
            cold_local = np.nonzero(np.isin(idx_gate, idx_weak_cold_curr))[0]
            w_weak_cold_curr = V81_COLD_WEIGHT_FLOOR + V81_COLD_WEIGHT_SCALE * normalize_conf_weights(0.50 * promote_score[cold_local] + 0.50 * conf_g[cold_local])
        else:
            w_weak_cold_curr = np.zeros((0), dtype=np.float32)

        promo_total_rel += int(len(np.setdiff1d(idx_rel_curr, idx_rel, assume_unique=False)))
        promo_total_amb += int(len(np.setdiff1d(idx_amb_curr, idx_amb, assume_unique=False)))
        refresh_stats.append(dict(
            round=int(rr + 1),
            reliable_size=int(len(idx_rel_curr)),
            ambiguous_size=int(len(idx_amb_curr)),
            weak_size=int(len(idx_weak_curr)),
            cold_weak_size=int(len(idx_weak_cold_curr)),
            unknown_size=int(len(idx_unk_curr)),
            promote_score_gate_mean=float(promote_score.mean()),
            unknown_score_gate_mean=float(unknown_score_dyn.mean()),
            stability_mean=float(stability_n.mean())))

    info = dict(
        reliable_size=int(len(idx_rel_curr)),
        ambiguous_size=int(len(idx_amb_curr)),
        weak_size=int(len(idx_weak_curr)),
        cold_weak_size=int(len(idx_weak_cold_curr)),
        unknown_size=int(len(idx_unk_curr)),
        reliable_keep=float(len(idx_rel_curr) / max(1, len(Z_target))),
        ambiguous_keep=float(len(idx_amb_curr) / max(1, len(Z_target))),
        weak_keep=float(len(idx_weak_curr) / max(1, len(Z_target))),
        unknown_keep=float(len(idx_unk_curr) / max(1, len(Z_target))),
        promoted_reliable=float(promo_total_rel),
        promoted_ambiguous=float(promo_total_amb),
        stage2_reliable_size=int(len(idx_rel_curr)),
        stage2_ambiguous_size=int(len(idx_amb_curr)),
        stage2_weak_size=int(len(idx_weak_curr)),
        stage2_cold_weak_size=int(len(idx_weak_cold_curr)),
        stage2_unknown_size=int(len(idx_unk_curr)),
        promote_score_gate_mean=float(refresh_stats[-1]["promote_score_gate_mean"]) if len(refresh_stats) > 0 else float("nan"),
        stability_gate_mean=float(refresh_stats[-1]["stability_mean"]) if len(refresh_stats) > 0 else float("nan"),
        refresh_rounds=float(len(refresh_stats)))
    final_adapter, guard_info = guard_adapter_with_fallback(final_adapter, warm, fc_layer, Z_replay, y_replay, Z_probe=Z_target[idx_gate], probe_name=f"tbv8_{version}")
    info.update(guard_info)
    return final_adapter, info

# =========================
# External baseline reconstructions (paper/repo-guided, embedding-space)
# These are unified-backbone reconstructions for fair comparison under the same WiSig backbone/module2 evaluation.
# =========================

def clone_linear_fc(fc_layer):
    fc = nn.Linear(fc_layer.in_features, fc_layer.out_features, bias=True)
    fc.load_state_dict(fc_layer.state_dict())
    return fc

def bce_targets_from_labels(y, K):
    return F.one_hot(y, num_classes=K).float()

def ova_loss_recon(ova_logits, y):
    # ova_logits: [B, K]
    target = bce_targets_from_labels(y, ova_logits.shape[1]).to(ova_logits.device)
    return F.binary_cross_entropy_with_logits(ova_logits, target)

def open_entropy_recon(ova_logits):
    p = torch.sigmoid(ova_logits)
    ent = -(p * torch.log(p + 1e-8) + (1.0 - p) * torch.log(1.0 - p + 1e-8))
    return ent.mean()

def entropy_from_logits(logits):
    p = F.softmax(logits, dim=1)
    return -(p * torch.log(p + 1e-8)).sum(dim=1)

def coral_loss(zs, zt):
    if zs is None or zt is None or len(zs) < 2 or len(zt) < 2:
        return torch.tensor(0.0, device=device)
    zs = zs - zs.mean(0, keepdim=True)
    zt = zt - zt.mean(0, keepdim=True)
    cs = (zs.t() @ zs) / max(len(zs) - 1, 1)
    ct = (zt.t() @ zt) / max(len(zt) - 1, 1)
    return ((cs - ct) ** 2).mean()

def make_balanced_loader(Z, y, batch):
    ds = EmbDatasetWeightedHard(Z, y)
    return DataLoader(ds, batch_size=batch, shuffle=True, drop_last=len(Z) >= batch)

def cycle_next(it_ref, loader):
    try:
        return next(it_ref)
    except StopIteration:
        it_ref = iter(loader)
        return next(it_ref), it_ref

def _safe_batch_pair(src_it, src_loader, tgt_it, tgt_loader):
    try:
        src_b = next(src_it)
    except StopIteration:
        src_it = iter(src_loader)
        src_b = next(src_it)
    try:
        tgt_b = next(tgt_it)
    except StopIteration:
        tgt_it = iter(tgt_loader)
        tgt_b = next(tgt_it)
    return src_b, tgt_b, src_it, tgt_it

def fit_ovanet_recon(fc_layer, Z_replay, y_replay, Z_adapt,
                     bottleneck=64, scale=0.3, epochs=20, lr=1e-3, wd=1e-4, batch=128,
                     lambda_open_t=0.5):
    K = fc_layer.out_features
    adapter = ResidualAdapter(fc_layer.in_features, bottleneck=bottleneck, scale=scale).to(device)
    cls_head = clone_linear_fc(fc_layer).to(device)
    ova_head = nn.Linear(fc_layer.in_features, K).to(device)
    opt = torch.optim.Adam(list(adapter.parameters()) + list(cls_head.parameters()) + list(ova_head.parameters()), lr=lr, weight_decay=wd)
    ce = nn.CrossEntropyLoss()
    src_loader = make_balanced_loader(Z_replay, y_replay, batch)
    tgt_loader = DataLoader(EmbDatasetWeightedHard(Z_adapt, np.zeros((len(Z_adapt)), dtype=np.int64)), batch_size=batch, shuffle=True, drop_last=len(Z_adapt) >= batch)
    steps = max(1, max(len(src_loader), len(tgt_loader)))
    for _ in range(epochs):
        adapter.train(); cls_head.train(); ova_head.train()
        src_it, tgt_it = iter(src_loader), iter(tgt_loader)
        for _step in range(steps):
            (zs, ys, _), (zt, _, _), src_it, tgt_it = _safe_batch_pair(src_it, src_loader, tgt_it, tgt_loader)
            zs, ys, zt = zs.to(device), ys.to(device), zt.to(device)
            opt.zero_grad()
            z2s = adapter(zs)
            z2t = adapter(zt)
            logits_s = cls_head(z2s)
            ova_s = ova_head(z2s)
            ova_t = ova_head(z2t)
            loss = ce(logits_s, ys) + 0.5 * ova_loss_recon(ova_s, ys) + lambda_open_t * open_entropy_recon(ova_t)
            loss.backward()
            opt.step()
    return {"kind": "custom_fc", "adapter": adapter.cpu(), "fc": cls_head.cpu()}

def fit_comet_recon(fc_layer, Z_replay, y_replay, Z_adapt,
                    bottleneck=64, scale=0.3, epochs=20, lr=1e-3, wd=1e-4, batch=128,
                    known_thr=0.70, unknown_thr=0.40, ema=0.99,
                    lambda_known=0.7, lambda_unknown=0.2, lambda_contrast=0.3):
    K = fc_layer.out_features
    student = ResidualAdapter(fc_layer.in_features, bottleneck=bottleneck, scale=scale).to(device)
    teacher = ResidualAdapter(fc_layer.in_features, bottleneck=bottleneck, scale=scale).to(device)
    teacher.load_state_dict(student.state_dict())
    cls_s = clone_linear_fc(fc_layer).to(device)
    cls_t = clone_linear_fc(fc_layer).to(device)
    cls_t.load_state_dict(cls_s.state_dict())
    opt = torch.optim.Adam(list(student.parameters()) + list(cls_s.parameters()), lr=lr, weight_decay=wd)
    ce = nn.CrossEntropyLoss()
    src_loader = make_balanced_loader(Z_replay, y_replay, batch)
    tgt_loader = DataLoader(EmbDatasetWeightedHard(Z_adapt, np.zeros((len(Z_adapt)), dtype=np.int64)), batch_size=batch, shuffle=True, drop_last=len(Z_adapt) >= batch)
    steps = max(1, max(len(src_loader), len(tgt_loader)))
    for _ in range(epochs):
        student.train(); cls_s.train(); teacher.eval(); cls_t.eval()
        src_it, tgt_it = iter(src_loader), iter(tgt_loader)
        for _step in range(steps):
            (zs, ys, _), (zt, _, _), src_it, tgt_it = _safe_batch_pair(src_it, src_loader, tgt_it, tgt_loader)
            zs, ys, zt = zs.to(device), ys.to(device), zt.to(device)
            with torch.no_grad():
                zt_teacher = teacher(zt)
                logits_t_teacher = cls_t(zt_teacher)
                probs_t = F.softmax(logits_t_teacher, dim=1)
                maxp, pseudo = probs_t.max(dim=1)
                known_mask = maxp >= known_thr
                unknown_mask = maxp <= unknown_thr
            opt.zero_grad()
            z2s = student(zs)
            logits_s = cls_s(z2s)
            loss = ce(logits_s, ys)
            z2t = student(zt)
            logits_t = cls_s(z2t)
            if known_mask.any():
                loss = loss + lambda_known * ce(logits_t[known_mask], pseudo[known_mask])
                with torch.no_grad():
                    proto = []
                    for k in range(K):
                        idxk = torch.where(ys == k)[0]
                        if len(idxk) == 0:
                            proto.append(z2s.mean(0))
                        else:
                            proto.append(z2s[idxk].mean(0))
                    proto = torch.stack(proto, dim=0)
                cos = 1.0 - F.cosine_similarity(z2t[known_mask], proto[pseudo[known_mask]], dim=1)
                loss = loss + lambda_contrast * cos.mean()
            if unknown_mask.any():
                ent = entropy_from_logits(logits_t[unknown_mask])
                loss = loss - lambda_unknown * ent.mean()
            loss.backward()
            opt.step()
            with torch.no_grad():
                for ps, pt in zip(student.parameters(), teacher.parameters()):
                    pt.data.mul_(ema).add_(ps.data * (1.0 - ema))
                for ps, pt in zip(cls_s.parameters(), cls_t.parameters()):
                    pt.data.mul_(ema).add_(ps.data * (1.0 - ema))
    return {"kind": "custom_fc", "adapter": student.cpu(), "fc": cls_s.cpu()}

def fit_pcpd_recon(fc_layer, Z_replay, y_replay, Z_adapt,
                   bottleneck=64, scale=0.3, epochs=20, lr=1e-3, wd=1e-4, batch=128,
                   conf_thr=0.75, lambda_pseudo=0.7, lambda_align=0.3, lambda_proto=0.3):
    K = fc_layer.out_features
    adapter = ResidualAdapter(fc_layer.in_features, bottleneck=bottleneck, scale=scale).to(device)
    cls_head = clone_linear_fc(fc_layer).to(device)
    opt = torch.optim.Adam(list(adapter.parameters()) + list(cls_head.parameters()), lr=lr, weight_decay=wd)
    ce = nn.CrossEntropyLoss()
    src_loader = make_balanced_loader(Z_replay, y_replay, batch)
    tgt_loader = DataLoader(EmbDatasetWeightedHard(Z_adapt, np.zeros((len(Z_adapt)), dtype=np.int64)), batch_size=batch, shuffle=True, drop_last=len(Z_adapt) >= batch)
    steps = max(1, max(len(src_loader), len(tgt_loader)))
    for _ in range(epochs):
        adapter.train(); cls_head.train()
        src_it, tgt_it = iter(src_loader), iter(tgt_loader)
        for _step in range(steps):
            (zs, ys, _), (zt, _, _), src_it, tgt_it = _safe_batch_pair(src_it, src_loader, tgt_it, tgt_loader)
            zs, ys, zt = zs.to(device), ys.to(device), zt.to(device)
            opt.zero_grad()
            z2s = adapter(zs)
            logits_s = cls_head(z2s)
            loss = ce(logits_s, ys)
            z2t = adapter(zt)
            logits_t = cls_head(z2t)
            with torch.no_grad():
                probs_t = F.softmax(logits_t, dim=1)
                maxp, pseudo = probs_t.max(dim=1)
                keep = maxp >= conf_thr
            if keep.any():
                loss = loss + lambda_pseudo * ce(logits_t[keep], pseudo[keep])
            loss = loss + lambda_align * coral_loss(z2s, z2t)
            with torch.no_grad():
                proto = []
                for k in range(K):
                    idxk = torch.where(ys == k)[0]
                    if len(idxk) == 0:
                        proto.append(z2s.mean(0))
                    else:
                        proto.append(z2s[idxk].mean(0))
                proto = torch.stack(proto, dim=0)
            if keep.any():
                cos = 1.0 - F.cosine_similarity(z2t[keep], proto[pseudo[keep]], dim=1)
                loss = loss + lambda_proto * cos.mean()
            loss.backward()
            opt.step()
    return {"kind": "custom_fc", "adapter": adapter.cpu(), "fc": cls_head.cpu()}

def fit_jrffpsc_recon(fc_layer, Z_replay, y_replay, Z_adapt,
                      bottleneck=64, scale=0.3, epochs=20, lr=1e-3, wd=1e-4, batch=128,
                      conf_thr=0.8, lambda_siam=0.4, margin=1.0):
    K = fc_layer.out_features
    adapter = ResidualAdapter(fc_layer.in_features, bottleneck=bottleneck, scale=scale).to(device)
    cls_head = clone_linear_fc(fc_layer).to(device)
    cmp_mlp = nn.Sequential(
        nn.Linear(fc_layer.in_features * 2, bottleneck),
        nn.ReLU(inplace=True),
        nn.Linear(bottleneck, 1)
    ).to(device)
    opt = torch.optim.Adam(list(adapter.parameters()) + list(cls_head.parameters()) + list(cmp_mlp.parameters()), lr=lr, weight_decay=wd)
    ce = nn.CrossEntropyLoss()
    src_loader = make_balanced_loader(Z_replay, y_replay, batch)
    tgt_loader = DataLoader(EmbDatasetWeightedHard(Z_adapt, np.zeros((len(Z_adapt)), dtype=np.int64)), batch_size=batch, shuffle=True, drop_last=len(Z_adapt) >= batch)
    steps = max(1, max(len(src_loader), len(tgt_loader)))
    for _ in range(epochs):
        adapter.train(); cls_head.train(); cmp_mlp.train()
        src_it, tgt_it = iter(src_loader), iter(tgt_loader)
        for _step in range(steps):
            (zs, ys, _), (zt, _, _), src_it, tgt_it = _safe_batch_pair(src_it, src_loader, tgt_it, tgt_loader)
            zs, ys, zt = zs.to(device), ys.to(device), zt.to(device)
            opt.zero_grad()
            z2s = adapter(zs)
            logits_s = cls_head(z2s)
            loss = ce(logits_s, ys)
            with torch.no_grad():
                proto = []
                for k in range(K):
                    idxk = torch.where(ys == k)[0]
                    if len(idxk) == 0:
                        proto.append(z2s.mean(0))
                    else:
                        proto.append(z2s[idxk].mean(0))
                proto = torch.stack(proto, dim=0)
            pos_score = cmp_mlp(torch.cat([z2s, proto[ys]], dim=1)).squeeze(1)
            neg_proto = proto[(ys + 1) % K]
            neg_score = cmp_mlp(torch.cat([z2s, neg_proto], dim=1)).squeeze(1)
            loss = loss + lambda_siam * (F.softplus(-pos_score).mean() + F.softplus(neg_score).mean())
            z2t = adapter(zt)
            logits_t = cls_head(z2t)
            with torch.no_grad():
                probs_t = F.softmax(logits_t, dim=1)
                maxp, pseudo = probs_t.max(dim=1)
                keep = maxp >= conf_thr
            if keep.any():
                pos_t = cmp_mlp(torch.cat([z2t[keep], proto[pseudo[keep]]], dim=1)).squeeze(1)
                loss = loss + 0.2 * F.softplus(-pos_t).mean()
            loss.backward()
            opt.step()
    return {"kind": "custom_fc", "adapter": adapter.cpu(), "fc": cls_head.cpu()}


In [11]:

def evaluate_variant(adapter, model_fc, Z_A, y_A, D_A, Z_B, y_B, D_B, Z_C, y_C, D_C, Z_U, D_U,
                     mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids,
                     tau_id, tau_dom, router_bundle=None):
    """
    Evaluate one method on the final module3 test pipeline.

    IMPORTANT EVALUATION CONVENTION
    --------------------------------
    At the module3 stage, both stable-known and drift-known samples are treated as KNOWN.
    Therefore, the final-stage errors are:
      (1) false rejection: a known sample is rejected as unknown / non-known;
      (2) accepted-but-wrong classification: the sample is accepted as known, but assigned to the wrong TX.

    We intentionally DO NOT treat stable-vs-drift confusion as a final-stage error here.
    That distinction belongs to module2 / routing analysis, while module3 is judged by retention and TX correctness.
    """
    Z_A2, logits_A2 = apply_adapter_and_fc(adapter, model_fc, Z_A, batch=512)
    Z_B2, logits_B2 = apply_adapter_and_fc(adapter, model_fc, Z_B, batch=512)
    Z_C2, logits_C2 = apply_adapter_and_fc(adapter, model_fc, Z_C, batch=512)
    Z_U2, logits_U2 = apply_adapter_and_fc(adapter, model_fc, Z_U, batch=512)

    if router_bundle is None:
        sc_A = compute_module2_scores_from_cached(Z_A2, logits_A2, D_A, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids, tau_id=tau_id, tau_dom=tau_dom)
        sc_B = compute_module2_scores_from_cached(Z_B2, logits_B2, D_B, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids, tau_id=tau_id, tau_dom=tau_dom)
        sc_C = compute_module2_scores_from_cached(Z_C2, logits_C2, D_C, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids, tau_id=tau_id, tau_dom=tau_dom)
        sc_U = compute_module2_scores_from_cached(Z_U2, logits_U2, D_U, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids, tau_id=tau_id, tau_dom=tau_dom)
        auc_score_A = -np.nan_to_num(sc_A["Sid"], nan=-1e6, posinf=1e6, neginf=-1e6)
        auc_score_B = -np.nan_to_num(sc_B["Sid"], nan=-1e6, posinf=1e6, neginf=-1e6)
        auc_score_C = -np.nan_to_num(sc_C["Sid"], nan=-1e6, posinf=1e6, neginf=-1e6)
        auc_score_U = -np.nan_to_num(sc_U["Sid"], nan=-1e6, posinf=1e6, neginf=-1e6)
    else:
        sc_A = compute_module2_v8_scores_from_cached(Z_A2, logits_A2, D_A, router_bundle)
        sc_B = compute_module2_v8_scores_from_cached(Z_B2, logits_B2, D_B, router_bundle)
        sc_C = compute_module2_v8_scores_from_cached(Z_C2, logits_C2, D_C, router_bundle)
        sc_U = compute_module2_v8_scores_from_cached(Z_U2, logits_U2, D_U, router_bundle)
        auc_score_A = np.nan_to_num(sc_A["p_unknown"], nan=0.0, posinf=1.0, neginf=0.0)
        auc_score_B = np.nan_to_num(sc_B["p_unknown"], nan=0.0, posinf=1.0, neginf=0.0)
        auc_score_C = np.nan_to_num(sc_C["p_unknown"], nan=0.0, posinf=1.0, neginf=0.0)
        auc_score_U = np.nan_to_num(sc_U["p_unknown"], nan=0.0, posinf=1.0, neginf=0.0)

    pred_A = np.argmax(logits_A2, 1)
    pred_B = np.argmax(logits_B2, 1)
    pred_C = np.argmax(logits_C2, 1)

    # gate_pred == 0 means finally accepted as KNOWN.
    accept_A = (sc_A["gate_pred"] == 0)
    accept_B = (sc_B["gate_pred"] == 0)
    accept_C = (sc_C["gate_pred"] == 0)
    accept_U = (sc_U["gate_pred"] == 0)

    # Stable metrics.
    stable_cls_acc = closed_set_acc_from_logits(logits_A2, y_A)
    stable_acc_e2e = float(np.mean(accept_A & (pred_A == y_A)))
    stable_wrongcls = float(np.mean(accept_A & (pred_A != y_A)))
    FRR_stable = float(np.mean(~accept_A))

    # Drift metrics (classification-only and end-to-end).
    drift_cls_acc_rx = closed_set_acc_from_logits(logits_B2, y_B)
    drift_cls_acc_day = closed_set_acc_from_logits(logits_C2, y_C)
    drift_cls_acc_all = float((np.sum(pred_B == y_B) + np.sum(pred_C == y_C)) / (len(y_B) + len(y_C)))

    drift_acc_rx = float(np.mean(accept_B & (pred_B == y_B)))
    drift_acc_day = float(np.mean(accept_C & (pred_C == y_C)))
    drift_acc_all = float((np.sum(accept_B & (pred_B == y_B)) + np.sum(accept_C & (pred_C == y_C))) / (len(y_B) + len(y_C)))

    drift_wrongcls_rx = float(np.mean(accept_B & (pred_B != y_B)))
    drift_wrongcls_day = float(np.mean(accept_C & (pred_C != y_C)))
    drift_wrongcls_all = float((np.sum(accept_B & (pred_B != y_B)) + np.sum(accept_C & (pred_C != y_C))) / (len(y_B) + len(y_C)))

    FRR_drift_rx = float(np.mean(~accept_B))
    FRR_drift_day = float(np.mean(~accept_C))
    FRR_drift_all = float((np.sum(~accept_B) + np.sum(~accept_C)) / (len(y_B) + len(y_C)))

    # Known aggregate metrics (stable + drift).
    n_known = len(y_A) + len(y_B) + len(y_C)
    known_cls_acc = float((np.sum(pred_A == y_A) + np.sum(pred_B == y_B) + np.sum(pred_C == y_C)) / n_known)
    known_acc_e2e = float((np.sum(accept_A & (pred_A == y_A)) + np.sum(accept_B & (pred_B == y_B)) + np.sum(accept_C & (pred_C == y_C))) / n_known)
    known_wrongcls = float((np.sum(accept_A & (pred_A != y_A)) + np.sum(accept_B & (pred_B != y_B)) + np.sum(accept_C & (pred_C != y_C))) / n_known)
    FRR_known = float((np.sum(~accept_A) + np.sum(~accept_B) + np.sum(~accept_C)) / n_known)

    # Unknown false acceptance rate.
    FAR_unknown_accept = float(np.mean(accept_U))

    # Threshold-free known-vs-unknown separability.
    # Legacy note: auc_unknown_eval is retained as the historical AUC_SU alias.
    auc_metrics = compute_known_unknown_auc_metrics(auc_score_A, auc_score_B, auc_score_C, auc_score_U)
    auc_unknown_eval = float(auc_metrics["auc_unknown_eval"])
    auc_su = float(auc_metrics["auc_su"])
    auc_du = float(auc_metrics["auc_du"])
    auc_ku = float(auc_metrics["auc_ku"])

    return dict(
        stable_cls_acc=stable_cls_acc,
        stable_acc_e2e=stable_acc_e2e,
        stable_wrongcls=stable_wrongcls,
        FRR_stable=FRR_stable,
        drift_cls_acc_rx=drift_cls_acc_rx,
        drift_cls_acc_day=drift_cls_acc_day,
        drift_cls_acc_all=drift_cls_acc_all,
        drift_acc_rx=drift_acc_rx,
        drift_acc_day=drift_acc_day,
        drift_acc_all=drift_acc_all,
        drift_wrongcls_rx=drift_wrongcls_rx,
        drift_wrongcls_day=drift_wrongcls_day,
        drift_wrongcls_all=drift_wrongcls_all,
        FRR_drift_rx=FRR_drift_rx,
        FRR_drift_day=FRR_drift_day,
        FRR_drift_all=FRR_drift_all,
        known_cls_acc=known_cls_acc,
        known_acc_e2e=known_acc_e2e,
        known_wrongcls=known_wrongcls,
        FRR_known=FRR_known,
        FAR_unknown_accept=FAR_unknown_accept,
        auc_unknown_eval=auc_unknown_eval,
        auc_su=auc_su,
        auc_du=auc_du,
        auc_ku=auc_ku)



def logsumexp_np(logits, axis=1):
    logits = np.asarray(logits, dtype=np.float32)
    m = np.max(logits, axis=axis, keepdims=True)
    out = m + np.log(np.sum(np.exp(logits - m), axis=axis, keepdims=True) + 1e-12)
    return np.squeeze(out, axis=axis).astype(np.float32)


def normalized_entropy_from_logits(logits):
    P = softmax_np(logits)
    K = max(2, P.shape[1])
    ent = -np.sum(P * np.log(np.clip(P, 1e-12, 1.0)), axis=1)
    ent = ent / np.log(float(K))
    return np.clip(ent, 0.0, 1.0).astype(np.float32)


def top1_top2_margin_from_logits(logits):
    P = softmax_np(logits)
    if P.shape[1] < 2:
        return np.ones((P.shape[0]), dtype=np.float32)
    top2 = np.partition(P, kth=P.shape[1]-2, axis=1)[:, -2:]
    margin = top2[:, 1] - top2[:, 0]
    return margin.astype(np.float32)


def max_proto_similarity_from_Z(Z, protos):
    Zx = normalize_vec_rows_np(Z)
    P = np.asarray(protos, dtype=np.float32)
    P = normalize_vec_rows_np(P)
    return np.max(Zx @ P.T, axis=1).astype(np.float32)


def _safe_quantile(x, q, fallback=0.0):
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return float(fallback)
    q = float(np.clip(q, 0.0, 1.0))
    return float(np.quantile(x, q))


def rowwise_pick(mat, cls_idx, fallback=0.0):
    mat = np.asarray(mat, dtype=np.float32)
    cls_idx = np.asarray(cls_idx, dtype=np.int64).reshape(-1)
    if mat.ndim != 2:
        return np.full((len(cls_idx)), float(fallback), dtype=np.float32)
    out = np.full((len(cls_idx)), float(fallback), dtype=np.float32)
    ok = (cls_idx >= 0) & (cls_idx < mat.shape[1])
    if np.any(ok):
        out[ok] = mat[np.arange(len(cls_idx))[ok], cls_idx[ok]]
    return out.astype(np.float32)


def l2_to_selected_prototypes(Z, cls_idx, protos, l2norm=True):
    Zx = np.asarray(Z, dtype=np.float32)
    P = np.asarray(protos, dtype=np.float32)
    cls_idx = np.asarray(cls_idx, dtype=np.int64).reshape(-1)
    if l2norm:
        Zx = normalize_vec_rows_np(Zx)
        P = normalize_vec_rows_np(P)
    out = np.full((len(cls_idx)), np.inf, dtype=np.float32)
    ok = (cls_idx >= 0) & (cls_idx < len(P))
    if np.any(ok):
        diff = Zx[ok] - P[cls_idx[ok]]
        out[ok] = np.linalg.norm(diff, axis=1).astype(np.float32)
    return out.astype(np.float32)


def nearest_proto_distance_and_pred(Z, protos, l2norm=True):
    Zx = np.asarray(Z, dtype=np.float32)
    P = np.asarray(protos, dtype=np.float32)
    if l2norm:
        Zx = normalize_vec_rows_np(Zx)
        P = normalize_vec_rows_np(P)
    dist = np.linalg.norm(Zx[:, None, :] - P[None, :, :], axis=2).astype(np.float32)
    pred = np.argmin(dist, axis=1).astype(np.int64)
    dmin = dist[np.arange(len(pred)), pred].astype(np.float32)
    return pred, dmin


def build_classwise_thresholds(values, y, K, q, fallback):
    values = np.asarray(values, dtype=np.float32).reshape(-1)
    y = np.asarray(y, dtype=np.int64).reshape(-1)
    thr = np.full((int(K)), float(fallback), dtype=np.float32)
    q = float(np.clip(q, 0.0, 1.0))
    for k in range(int(K)):
        vk = values[y == k]
        vk = vk[np.isfinite(vk)]
        if len(vk) == 0:
            continue
        thr[k] = float(np.quantile(vk, q))
    return thr.astype(np.float32)


def build_variant_eval_cache(adapter, model_fc, Z_tr, y_tr, Z_A, D_A, Z_B, D_B, Z_C, D_C, Z_U, D_U,
                             mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids,
                             tau_id, tau_dom, router_bundle=None):
    y_tr = np.asarray(y_tr, dtype=np.int64)
    Z_tr2, logits_tr2 = apply_adapter_and_fc(adapter, model_fc, Z_tr, batch=512)
    if len(y_tr) != len(Z_tr2):
        raise RuntimeError(
            f"build_variant_eval_cache label/feature length mismatch: len(y_tr)={len(y_tr)} vs len(Z_tr2)={len(Z_tr2)}. "
            "Pass the exact training labels aligned with Z_tr (normally stageA['y_tr_fold'])."
        )
    Z_A2, logits_A2 = apply_adapter_and_fc(adapter, model_fc, Z_A, batch=512)
    Z_B2, logits_B2 = apply_adapter_and_fc(adapter, model_fc, Z_B, batch=512)
    Z_C2, logits_C2 = apply_adapter_and_fc(adapter, model_fc, Z_C, batch=512)
    Z_U2, logits_U2 = apply_adapter_and_fc(adapter, model_fc, Z_U, batch=512)

    K_eval = int(model_fc.out_features)
    protos_eval = fit_class_prototypes(Z_tr2, y_tr, K=K_eval, l2norm=True)

    P_tr2 = softmax_np(logits_tr2)
    trueprob_tr2 = rowwise_pick(P_tr2, y_tr, fallback=0.0)
    energy_tr2 = logsumexp_np(logits_tr2)
    proto_true_dist_tr2 = l2_to_selected_prototypes(Z_tr2, y_tr, protos_eval, l2norm=True)
    proto_pred_tr2, proto_min_dist_tr2 = nearest_proto_distance_and_pred(Z_tr2, protos_eval, l2norm=True)

    if router_bundle is None:
        sc_tr = compute_module2_scores_from_cached(Z_tr2, logits_tr2, np.zeros((len(Z_tr2), D_A.shape[1]), dtype=np.float32), mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids, tau_id=tau_id, tau_dom=tau_dom)
        sc_A = compute_module2_scores_from_cached(Z_A2, logits_A2, D_A, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids, tau_id=tau_id, tau_dom=tau_dom)
        sc_B = compute_module2_scores_from_cached(Z_B2, logits_B2, D_B, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids, tau_id=tau_id, tau_dom=tau_dom)
        sc_C = compute_module2_scores_from_cached(Z_C2, logits_C2, D_C, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids, tau_id=tau_id, tau_dom=tau_dom)
        sc_U = compute_module2_scores_from_cached(Z_U2, logits_U2, D_U, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids, tau_id=tau_id, tau_dom=tau_dom)
        auc_score_A = -np.nan_to_num(sc_A["Sid"], nan=-1e6, posinf=1e6, neginf=-1e6)
        auc_score_B = -np.nan_to_num(sc_B["Sid"], nan=-1e6, posinf=1e6, neginf=-1e6)
        auc_score_C = -np.nan_to_num(sc_C["Sid"], nan=-1e6, posinf=1e6, neginf=-1e6)
        auc_score_U = -np.nan_to_num(sc_U["Sid"], nan=-1e6, posinf=1e6, neginf=-1e6)
    else:
        D_tr_proxy = np.repeat(np.mean(np.asarray(D_A, dtype=np.float32), axis=0, keepdims=True), repeats=len(Z_tr2), axis=0).astype(np.float32)
        sc_tr = compute_module2_v8_scores_from_cached(Z_tr2, logits_tr2, D_tr_proxy, router_bundle)
        sc_A = compute_module2_v8_scores_from_cached(Z_A2, logits_A2, D_A, router_bundle)
        sc_B = compute_module2_v8_scores_from_cached(Z_B2, logits_B2, D_B, router_bundle)
        sc_C = compute_module2_v8_scores_from_cached(Z_C2, logits_C2, D_C, router_bundle)
        sc_U = compute_module2_v8_scores_from_cached(Z_U2, logits_U2, D_U, router_bundle)
        auc_score_A = np.nan_to_num(sc_A["p_unknown"], nan=0.0, posinf=1.0, neginf=0.0)
        auc_score_B = np.nan_to_num(sc_B["p_unknown"], nan=0.0, posinf=1.0, neginf=0.0)
        auc_score_C = np.nan_to_num(sc_C["p_unknown"], nan=0.0, posinf=1.0, neginf=0.0)
        auc_score_U = np.nan_to_num(sc_U["p_unknown"], nan=0.0, posinf=1.0, neginf=0.0)

    return dict(
        Z_tr2=Z_tr2, logits_tr2=logits_tr2, y_tr=np.asarray(y_tr, dtype=np.int64), protos_eval=protos_eval,
        P_tr2=P_tr2, trueprob_tr2=trueprob_tr2, energy_tr2=energy_tr2,
        proto_true_dist_tr2=proto_true_dist_tr2,
        proto_pred_tr2=proto_pred_tr2, proto_min_dist_tr2=proto_min_dist_tr2,
        Z_A2=Z_A2, logits_A2=logits_A2,
        Z_B2=Z_B2, logits_B2=logits_B2,
        Z_C2=Z_C2, logits_C2=logits_C2,
        Z_U2=Z_U2, logits_U2=logits_U2,
        module2=dict(
            sc_tr=sc_tr, sc_A=sc_A, sc_B=sc_B, sc_C=sc_C, sc_U=sc_U,
            auc_score_A=auc_score_A, auc_score_B=auc_score_B, auc_score_C=auc_score_C, auc_score_U=auc_score_U))





def build_final_rejector_from_cache(eval_cache, final_rejector_head="module2", frr_target=0.05, base_name=None):
    head = str(final_rejector_head).lower()
    base_name = None if base_name is None else str(base_name)

    logits_A2 = np.asarray(eval_cache["logits_A2"], dtype=np.float32)
    logits_B2 = np.asarray(eval_cache["logits_B2"], dtype=np.float32)
    logits_C2 = np.asarray(eval_cache["logits_C2"], dtype=np.float32)
    logits_U2 = np.asarray(eval_cache["logits_U2"], dtype=np.float32)
    Z_A2 = np.asarray(eval_cache["Z_A2"], dtype=np.float32)
    Z_B2 = np.asarray(eval_cache["Z_B2"], dtype=np.float32)
    Z_C2 = np.asarray(eval_cache["Z_C2"], dtype=np.float32)
    Z_U2 = np.asarray(eval_cache["Z_U2"], dtype=np.float32)
    y_tr = np.asarray(eval_cache["y_tr"], dtype=np.int64)
    K_eval = int(eval_cache["protos_eval"].shape[0])

    def _pred_from_logits(logits):
        return np.argmax(logits, axis=1).astype(np.int64)

    def _predclass_proto_accept(Z, pred, thr_by_class):
        dist = l2_to_selected_prototypes(Z, pred, eval_cache["protos_eval"], l2norm=True)
        thr = thr_by_class[np.clip(pred, 0, len(thr_by_class)-1)]
        return (dist <= thr), dist, thr

    def _state_index(sc):
        if isinstance(sc, dict):
            if all(k in sc for k in ["p_stable", "p_drift", "p_unknown"]):
                P = np.stack([np.asarray(sc["p_stable"], dtype=np.float32),
                              np.asarray(sc["p_drift"], dtype=np.float32),
                              np.asarray(sc["p_unknown"], dtype=np.float32)], axis=1)
                return np.argmax(P, axis=1).astype(np.int64)
            if "gate_pred" in sc:
                return np.asarray(sc["gate_pred"], dtype=np.int64)
        return None

    if head == "native":
        family = BASELINE_NATIVE_REJECTOR_FAMILY.get(base_name, None)
        if family is None:
            head = get_default_rejector_head(base_name if base_name is not None else "")
        else:
            if family == "comet":
                score_A = normalized_entropy_from_logits(logits_A2)
                score_B = normalized_entropy_from_logits(logits_B2)
                score_C = normalized_entropy_from_logits(logits_C2)
                score_U = normalized_entropy_from_logits(logits_U2)
                thr = float(COMET_NATIVE_ENTROPY_THRESHOLD)
                return dict(
                    accept_A=score_A <= thr, accept_B=score_B <= thr, accept_C=score_C <= thr, accept_U=score_U <= thr,
                    auc_score_A=score_A, auc_score_B=score_B, auc_score_C=score_C, auc_score_U=score_U,
                    threshold=thr, score_name="comet_native_entropy", family="comet_native")

            if family == "ovanet":
                P_A = softmax_np(logits_A2); P_B = softmax_np(logits_B2); P_C = softmax_np(logits_C2); P_U = softmax_np(logits_U2)
                pred_A = _pred_from_logits(logits_A2); pred_B = _pred_from_logits(logits_B2); pred_C = _pred_from_logits(logits_C2); pred_U = _pred_from_logits(logits_U2)
                thr_by_class = build_classwise_thresholds(eval_cache["trueprob_tr2"], y_tr, K_eval, q=frr_target, fallback=0.0)
                score_A = rowwise_pick(P_A, pred_A, fallback=0.0); score_B = rowwise_pick(P_B, pred_B, fallback=0.0)
                score_C = rowwise_pick(P_C, pred_C, fallback=0.0); score_U = rowwise_pick(P_U, pred_U, fallback=0.0)
                thr_A = thr_by_class[pred_A]; thr_B = thr_by_class[pred_B]; thr_C = thr_by_class[pred_C]; thr_U = thr_by_class[pred_U]
                return dict(
                    accept_A=score_A >= thr_A, accept_B=score_B >= thr_B, accept_C=score_C >= thr_C, accept_U=score_U >= thr_U,
                    auc_score_A=(thr_A - score_A).astype(np.float32), auc_score_B=(thr_B - score_B).astype(np.float32),
                    auc_score_C=(thr_C - score_C).astype(np.float32), auc_score_U=(thr_U - score_U).astype(np.float32),
                    threshold=float(np.mean(thr_by_class)), score_name="ovanet_native_predclass_prob", family="ovanet_native")

            if family == "pcpd":
                thr_by_class = build_classwise_thresholds(eval_cache["proto_true_dist_tr2"], y_tr, K_eval, q=1.0 - frr_target, fallback=np.inf)
                pred_A, dist_A = nearest_proto_distance_and_pred(Z_A2, eval_cache["protos_eval"], l2norm=True)
                pred_B, dist_B = nearest_proto_distance_and_pred(Z_B2, eval_cache["protos_eval"], l2norm=True)
                pred_C, dist_C = nearest_proto_distance_and_pred(Z_C2, eval_cache["protos_eval"], l2norm=True)
                pred_U, dist_U = nearest_proto_distance_and_pred(Z_U2, eval_cache["protos_eval"], l2norm=True)
                thr_A = thr_by_class[pred_A]; thr_B = thr_by_class[pred_B]; thr_C = thr_by_class[pred_C]; thr_U = thr_by_class[pred_U]
                return dict(
                    accept_A=dist_A <= thr_A, accept_B=dist_B <= thr_B, accept_C=dist_C <= thr_C, accept_U=dist_U <= thr_U,
                    auc_score_A=(dist_A - thr_A).astype(np.float32), auc_score_B=(dist_B - thr_B).astype(np.float32),
                    auc_score_C=(dist_C - thr_C).astype(np.float32), auc_score_U=(dist_U - thr_U).astype(np.float32),
                    pred_A=pred_A, pred_B=pred_B, pred_C=pred_C,
                    threshold=float(np.mean(thr_by_class)), score_name="pcpd_native_proto_l2", family="pcpd_native")

            if family == "jrffp_sc":
                thr_by_class = build_classwise_thresholds(eval_cache["proto_true_dist_tr2"], y_tr, K_eval, q=1.0 - frr_target, fallback=np.inf)
                pred_A = _pred_from_logits(logits_A2); pred_B = _pred_from_logits(logits_B2); pred_C = _pred_from_logits(logits_C2); pred_U = _pred_from_logits(logits_U2)
                dist_A = l2_to_selected_prototypes(Z_A2, pred_A, eval_cache["protos_eval"], l2norm=True)
                dist_B = l2_to_selected_prototypes(Z_B2, pred_B, eval_cache["protos_eval"], l2norm=True)
                dist_C = l2_to_selected_prototypes(Z_C2, pred_C, eval_cache["protos_eval"], l2norm=True)
                dist_U = l2_to_selected_prototypes(Z_U2, pred_U, eval_cache["protos_eval"], l2norm=True)
                thr_A = thr_by_class[pred_A]; thr_B = thr_by_class[pred_B]; thr_C = thr_by_class[pred_C]; thr_U = thr_by_class[pred_U]
                return dict(
                    accept_A=dist_A <= thr_A, accept_B=dist_B <= thr_B, accept_C=dist_C <= thr_C, accept_U=dist_U <= thr_U,
                    auc_score_A=(dist_A - thr_A).astype(np.float32), auc_score_B=(dist_B - thr_B).astype(np.float32),
                    auc_score_C=(dist_C - thr_C).astype(np.float32), auc_score_U=(dist_U - thr_U).astype(np.float32),
                    pred_A=pred_A, pred_B=pred_B, pred_C=pred_C,
                    threshold=float(np.mean(thr_by_class)), score_name="jrffpsc_native_predclass_proto_l2", family="jrffp_sc_native")

    if head == "module2":
        m2 = eval_cache["module2"]
        return dict(
            accept_A=(m2["sc_A"]["gate_pred"] == 0),
            accept_B=(m2["sc_B"]["gate_pred"] == 0),
            accept_C=(m2["sc_C"]["gate_pred"] == 0),
            accept_U=(m2["sc_U"]["gate_pred"] == 0),
            auc_score_A=np.asarray(m2["auc_score_A"], dtype=np.float32),
            auc_score_B=np.asarray(m2["auc_score_B"], dtype=np.float32),
            auc_score_C=np.asarray(m2["auc_score_C"], dtype=np.float32),
            auc_score_U=np.asarray(m2["auc_score_U"], dtype=np.float32),
            threshold=float("nan"), score_name="module2_gate", family="module2")

    if head == "msp":
        score_A = msp_from_logits(logits_A2); score_B = msp_from_logits(logits_B2); score_C = msp_from_logits(logits_C2); score_U = msp_from_logits(logits_U2)
        thr = _safe_quantile(score_A, frr_target, fallback=0.0)
        return dict(accept_A=score_A >= thr, accept_B=score_B >= thr, accept_C=score_C >= thr, accept_U=score_U >= thr,
                    auc_score_A=-score_A, auc_score_B=-score_B, auc_score_C=-score_C, auc_score_U=-score_U,
                    threshold=thr, score_name="msp", family="generic_msp")

    if head == "energy":
        score_A = logsumexp_np(logits_A2); score_B = logsumexp_np(logits_B2); score_C = logsumexp_np(logits_C2); score_U = logsumexp_np(logits_U2)
        thr = _safe_quantile(score_A, frr_target, fallback=-1e6)
        return dict(accept_A=score_A >= thr, accept_B=score_B >= thr, accept_C=score_C >= thr, accept_U=score_U >= thr,
                    auc_score_A=-score_A, auc_score_B=-score_B, auc_score_C=-score_C, auc_score_U=-score_U,
                    threshold=thr, score_name="energy", family="generic_energy")

    if head == "energy_class":
        pred_A = _pred_from_logits(logits_A2); pred_B = _pred_from_logits(logits_B2); pred_C = _pred_from_logits(logits_C2); pred_U = _pred_from_logits(logits_U2)
        score_A = logsumexp_np(logits_A2); score_B = logsumexp_np(logits_B2); score_C = logsumexp_np(logits_C2); score_U = logsumexp_np(logits_U2)
        thr_global = _safe_quantile(eval_cache["energy_tr2"], frr_target, fallback=-1e6)
        thr_by_class = build_classwise_thresholds(eval_cache["energy_tr2"], y_tr, K_eval, q=frr_target, fallback=thr_global)
        thr_A = thr_by_class[pred_A]; thr_B = thr_by_class[pred_B]; thr_C = thr_by_class[pred_C]; thr_U = thr_by_class[pred_U]
        return dict(
            accept_A=score_A >= thr_A, accept_B=score_B >= thr_B, accept_C=score_C >= thr_C, accept_U=score_U >= thr_U,
            auc_score_A=(thr_A - score_A).astype(np.float32), auc_score_U=(thr_U - score_U).astype(np.float32),
            pred_A=pred_A, pred_B=pred_B, pred_C=pred_C,
            threshold=float(np.mean(thr_by_class)), score_name="energy_class", family="energy_class")

    if head == "energy_state":
        score_A = logsumexp_np(logits_A2); score_B = logsumexp_np(logits_B2); score_C = logsumexp_np(logits_C2); score_U = logsumexp_np(logits_U2)
        thr = _safe_quantile(score_A, frr_target, fallback=-1e6)
        delta = max(float(np.std(eval_cache["energy_tr2"]) * float(ENERGY_STATE_SHIFT_STD)), float(ENERGY_STATE_SHIFT_MINABS))
        scale = max(float(ENERGY_STATE_UNKNOWN_SCALE), 1e-6)
        thr_state = np.asarray([thr + delta, thr - delta, thr + delta / scale], dtype=np.float32)
        m2 = eval_cache["module2"]
        sA = _state_index(m2.get("sc_A", {})); sB = _state_index(m2.get("sc_B", {})); sC = _state_index(m2.get("sc_C", {})); sU = _state_index(m2.get("sc_U", {}))
        if sA is None:
            sA = np.zeros((len(score_A),), dtype=np.int64)
            sB = np.zeros((len(score_B),), dtype=np.int64)
            sC = np.zeros((len(score_C),), dtype=np.int64)
            sU = np.full((len(score_U),), 2, dtype=np.int64)
        sA = np.clip(sA, 0, 2); sB = np.clip(sB, 0, 2); sC = np.clip(sC, 0, 2); sU = np.clip(sU, 0, 2)
        tA = thr_state[sA]; tB = thr_state[sB]; tC = thr_state[sC]; tU = thr_state[sU]
        return dict(
            accept_A=score_A >= tA, accept_B=score_B >= tB, accept_C=score_C >= tC, accept_U=score_U >= tU,
            auc_score_A=(tA - score_A).astype(np.float32), auc_score_B=(tB - score_B).astype(np.float32),
            auc_score_C=(tC - score_C).astype(np.float32), auc_score_U=(tU - score_U).astype(np.float32),
            threshold=float(thr), threshold_state=thr_state.astype(np.float32), score_name="energy_state", family="energy_state")

    if head == "energy_proto":
        pred_A = _pred_from_logits(logits_A2); pred_B = _pred_from_logits(logits_B2); pred_C = _pred_from_logits(logits_C2); pred_U = _pred_from_logits(logits_U2)
        score_A = logsumexp_np(logits_A2); score_B = logsumexp_np(logits_B2); score_C = logsumexp_np(logits_C2); score_U = logsumexp_np(logits_U2)
        thr_energy = _safe_quantile(eval_cache["energy_tr2"], frr_target, fallback=-1e6)
        band = max(float(np.std(eval_cache["energy_tr2"]) * float(ENERGY_CASCADE_BAND_STD)), float(ENERGY_CASCADE_BAND_MINABS))
        thr_by_class = build_classwise_thresholds(eval_cache["proto_true_dist_tr2"], y_tr, K_eval, q=1.0 - frr_target, fallback=np.inf)
        proto_A, dist_A, pdthr_A = _predclass_proto_accept(Z_A2, pred_A, thr_by_class)
        proto_B, dist_B, pdthr_B = _predclass_proto_accept(Z_B2, pred_B, thr_by_class)
        proto_C, dist_C, pdthr_C = _predclass_proto_accept(Z_C2, pred_C, thr_by_class)
        proto_U, dist_U, pdthr_U = _predclass_proto_accept(Z_U2, pred_U, thr_by_class)
        en_A = score_A >= thr_energy; en_B = score_B >= thr_energy; en_C = score_C >= thr_energy; en_U = score_U >= thr_energy
        bd_A = np.abs(score_A - thr_energy) <= band; bd_B = np.abs(score_B - thr_energy) <= band; bd_C = np.abs(score_C - thr_energy) <= band; bd_U = np.abs(score_U - thr_energy) <= band
        acc_A = en_A & (~bd_A | proto_A); acc_B = en_B & (~bd_B | proto_B); acc_C = en_C & (~bd_C | proto_C); acc_U = en_U & (~bd_U | proto_U)
        return dict(
            accept_A=acc_A, accept_B=acc_B, accept_C=acc_C, accept_U=acc_U,
            auc_score_A=(thr_energy - score_A).astype(np.float32), auc_score_B=(thr_energy - score_B).astype(np.float32),
            auc_score_C=(thr_energy - score_C).astype(np.float32), auc_score_U=(thr_energy - score_U).astype(np.float32),
            pred_A=pred_A, pred_B=pred_B, pred_C=pred_C,
            threshold=float(thr_energy), band=float(band), score_name="energy_proto", family="energy_proto")

    if head == "entropy":
        score_A = normalized_entropy_from_logits(logits_A2); score_B = normalized_entropy_from_logits(logits_B2); score_C = normalized_entropy_from_logits(logits_C2); score_U = normalized_entropy_from_logits(logits_U2)
        thr = _safe_quantile(score_A, 1.0 - frr_target, fallback=1.0)
        return dict(accept_A=score_A <= thr, accept_B=score_B <= thr, accept_C=score_C <= thr, accept_U=score_U <= thr,
                    auc_score_A=score_A, auc_score_B=score_B, auc_score_C=score_C, auc_score_U=score_U,
                    threshold=thr, score_name="entropy", family="generic_entropy")

    if head == "margin":
        score_A = top1_top2_margin_from_logits(logits_A2); score_B = top1_top2_margin_from_logits(logits_B2); score_C = top1_top2_margin_from_logits(logits_C2); score_U = top1_top2_margin_from_logits(logits_U2)
        thr = _safe_quantile(score_A, frr_target, fallback=0.0)
        return dict(accept_A=score_A >= thr, accept_B=score_B >= thr, accept_C=score_C >= thr, accept_U=score_U >= thr,
                    auc_score_A=-score_A, auc_score_B=-score_B, auc_score_C=-score_C, auc_score_U=-score_U,
                    threshold=thr, score_name="margin", family="generic_margin")

    if head == "proto":
        protos_eval = eval_cache["protos_eval"]
        score_A = max_proto_similarity_from_Z(Z_A2, protos_eval); score_B = max_proto_similarity_from_Z(Z_B2, protos_eval)
        score_C = max_proto_similarity_from_Z(Z_C2, protos_eval); score_U = max_proto_similarity_from_Z(Z_U2, protos_eval)
        thr = _safe_quantile(score_A, frr_target, fallback=-1.0)
        return dict(accept_A=score_A >= thr, accept_B=score_B >= thr, accept_C=score_C >= thr, accept_U=score_U >= thr,
                    auc_score_A=-score_A, auc_score_B=-score_B, auc_score_C=-score_C, auc_score_U=-score_U,
                    threshold=thr, score_name="proto_maxcos", family="generic_proto")

    raise ValueError(f"Unsupported final_rejector_head: {final_rejector_head}")

def evaluate_variant_with_rejector(eval_cache, y_A, y_B, y_C, final_rejector_head="module2", frr_target=0.05, base_name=None):
    rejector = build_final_rejector_from_cache(
        eval_cache, final_rejector_head=final_rejector_head, frr_target=frr_target, base_name=base_name)

    logits_A2 = eval_cache["logits_A2"]
    logits_B2 = eval_cache["logits_B2"]
    logits_C2 = eval_cache["logits_C2"]

    pred_A = np.asarray(rejector.get("pred_A", np.argmax(logits_A2, 1)), dtype=np.int64)
    pred_B = np.asarray(rejector.get("pred_B", np.argmax(logits_B2, 1)), dtype=np.int64)
    pred_C = np.asarray(rejector.get("pred_C", np.argmax(logits_C2, 1)), dtype=np.int64)

    accept_A = np.asarray(rejector["accept_A"], dtype=bool)
    accept_B = np.asarray(rejector["accept_B"], dtype=bool)
    accept_C = np.asarray(rejector["accept_C"], dtype=bool)
    accept_U = np.asarray(rejector["accept_U"], dtype=bool)

    stable_cls_acc = float(np.mean(pred_A == y_A))
    stable_acc_e2e = float(np.mean(accept_A & (pred_A == y_A)))
    stable_wrongcls = float(np.mean(accept_A & (pred_A != y_A)))
    FRR_stable = float(np.mean(~accept_A))

    drift_cls_acc_rx = float(np.mean(pred_B == y_B))
    drift_cls_acc_day = float(np.mean(pred_C == y_C))
    drift_cls_acc_all = float((np.sum(pred_B == y_B) + np.sum(pred_C == y_C)) / (len(y_B) + len(y_C)))

    drift_acc_rx = float(np.mean(accept_B & (pred_B == y_B)))
    drift_acc_day = float(np.mean(accept_C & (pred_C == y_C)))
    drift_acc_all = float((np.sum(accept_B & (pred_B == y_B)) + np.sum(accept_C & (pred_C == y_C))) / (len(y_B) + len(y_C)))

    drift_wrongcls_rx = float(np.mean(accept_B & (pred_B != y_B)))
    drift_wrongcls_day = float(np.mean(accept_C & (pred_C != y_C)))
    drift_wrongcls_all = float((np.sum(accept_B & (pred_B != y_B)) + np.sum(accept_C & (pred_C != y_C))) / (len(y_B) + len(y_C)))

    FRR_drift_rx = float(np.mean(~accept_B))
    FRR_drift_day = float(np.mean(~accept_C))
    FRR_drift_all = float((np.sum(~accept_B) + np.sum(~accept_C)) / (len(y_B) + len(y_C)))

    n_known = len(y_A) + len(y_B) + len(y_C)
    known_cls_acc = float((np.sum(pred_A == y_A) + np.sum(pred_B == y_B) + np.sum(pred_C == y_C)) / n_known)
    known_acc_e2e = float((np.sum(accept_A & (pred_A == y_A)) + np.sum(accept_B & (pred_B == y_B)) + np.sum(accept_C & (pred_C == y_C))) / n_known)
    known_wrongcls = float((np.sum(accept_A & (pred_A != y_A)) + np.sum(accept_B & (pred_B != y_B)) + np.sum(accept_C & (pred_C != y_C))) / n_known)
    FRR_known = float((np.sum(~accept_A) + np.sum(~accept_B) + np.sum(~accept_C)) / n_known)

    FAR_unknown_accept = float(np.mean(accept_U))
    auc_metrics = compute_known_unknown_auc_metrics(
        np.asarray(rejector["auc_score_A"], dtype=np.float32),
        np.asarray(rejector["auc_score_B"], dtype=np.float32),
        np.asarray(rejector["auc_score_C"], dtype=np.float32),
        np.asarray(rejector["auc_score_U"], dtype=np.float32),
    )
    auc_unknown_eval = float(auc_metrics["auc_unknown_eval"])
    auc_su = float(auc_metrics["auc_su"])
    auc_du = float(auc_metrics["auc_du"])
    auc_ku = float(auc_metrics["auc_ku"])

    return dict(
        stable_cls_acc=stable_cls_acc,
        stable_acc_e2e=stable_acc_e2e,
        stable_wrongcls=stable_wrongcls,
        FRR_stable=FRR_stable,
        drift_cls_acc_rx=drift_cls_acc_rx,
        drift_cls_acc_day=drift_cls_acc_day,
        drift_cls_acc_all=drift_cls_acc_all,
        drift_acc_rx=drift_acc_rx,
        drift_acc_day=drift_acc_day,
        drift_acc_all=drift_acc_all,
        drift_wrongcls_rx=drift_wrongcls_rx,
        drift_wrongcls_day=drift_wrongcls_day,
        drift_wrongcls_all=drift_wrongcls_all,
        FRR_drift_rx=FRR_drift_rx,
        FRR_drift_day=FRR_drift_day,
        FRR_drift_all=FRR_drift_all,
        known_cls_acc=known_cls_acc,
        known_acc_e2e=known_acc_e2e,
        known_wrongcls=known_wrongcls,
        FRR_known=FRR_known,
        FAR_unknown_accept=FAR_unknown_accept,
        auc_unknown_eval=auc_unknown_eval,
        auc_su=auc_su,
        auc_du=auc_du,
        auc_ku=auc_ku,
        final_rejector_head=str(final_rejector_head).lower(),
        final_rejector_threshold=float(rejector.get("threshold", float("nan"))),
        final_rejector_score_name=str(rejector.get("score_name", "unknown")),
        final_rejector_family_used=str(rejector.get("family", "unknown")),
        final_rejector_base_method=str(base_name) if base_name is not None else "unknown")


In [12]:

# ===== v36 unknown-proxy builder + H-score wrappers =====
def entropy_from_probs_np(P):
    P = normalize_rows(np.asarray(P, dtype=np.float32))
    K = max(2, P.shape[1])
    return (-(P * np.log(P + 1e-12)).sum(axis=1) / np.log(float(K))).astype(np.float32)


def build_v36_unknown_proxy_from_base(
    base_spec,
    Z_adapt,
    p_known_adapt,
    p_drift_given_known_adapt,
    P_route_v4,
    P_target,
    tau_conf,
    base_name=None,
    overrides=None,
    method_name="DG_RAINCOAT_v36UnknownProxy"):
    if base_spec is None:
        return None, {"v36_active": False, "v36_reason": "base_spec_none", "v36_method_name": method_name}

    overrides = dict(overrides or {})
    n_target = len(Z_adapt)
    p_known = np.clip(np.asarray(p_known_adapt, dtype=np.float32), 0.0, 1.0)
    p_drift = np.clip(np.asarray(p_drift_given_known_adapt, dtype=np.float32), 0.0, 1.0)
    p_route = np.asarray(P_route_v4, dtype=np.float32) if P_route_v4 is not None else None
    if p_route is None or p_route.ndim != 2 or p_route.shape[1] < 3:
        p_unknown = np.clip(1.0 - p_known, 0.0, 1.0)
    else:
        p_unknown = np.clip(p_route[:, 2], 0.0, 1.0)

    P_full = normalize_rows(np.asarray(P_target, dtype=np.float32))
    conf_full = msp_from_probs(P_full)
    ent_full = entropy_from_probs_np(P_full)
    conf_smooth = float(overrides.get("v36_proxy_conf_smooth", 0.08))
    conf_soft = 0.5 + 0.5 * sigmoid_np((conf_full - float(tau_conf)) / max(1e-6, conf_smooth))

    support_idx = np.asarray(base_spec.get("idx_sup_eval", base_spec.get("sup_idx", np.zeros((0), dtype=np.int64))), dtype=np.int64)
    align_idx = np.asarray(base_spec.get("align_idx", np.zeros((0), dtype=np.int64)), dtype=np.int64)
    base_unrel = np.asarray(base_spec.get("unrel_idx", base_spec.get("idx_unk", np.zeros((0), dtype=np.int64))), dtype=np.int64)

    support_mask = np.zeros((n_target,), dtype=bool)
    support_mask[support_idx] = True
    base_unrel_mask = np.zeros((n_target,), dtype=bool)
    base_unrel_mask[base_unrel] = True

    entropy_mix = float(overrides.get("v36_proxy_entropy_mix", 0.50))
    proxy_score = p_unknown * ((1.0 - conf_soft) * (1.0 - entropy_mix) + ent_full * entropy_mix)
    proxy_score = proxy_score * np.clip(1.0 - 0.5 * p_known, 0.0, 1.0)

    cand_mask = (~support_mask)
    pu_min = float(overrides.get("v36_proxy_pu_min", 0.42))
    conf_max = float(overrides.get("v36_proxy_conf_max", 0.78))
    cand_mask = cand_mask & (p_unknown >= pu_min) & (conf_full <= conf_max)
    cand_idx = np.where(cand_mask)[0].astype(np.int64)

    proxy_cap = float(overrides.get("v36_proxy_cap", 0.08))
    proxy_min_keep = int(overrides.get("v36_proxy_min_keep", 24))
    max_proxy = min(max(0, len(np.where(~support_mask)[0])), max(proxy_min_keep, int(round(proxy_cap * n_target))))
    if max_proxy <= 0:
        proxy_idx = np.zeros((0,), dtype=np.int64)
    else:
        if len(cand_idx) == 0:
            fallback_idx = np.where(~support_mask)[0].astype(np.int64)
            order = np.argsort(-proxy_score[fallback_idx])
            proxy_idx = fallback_idx[order[:min(len(fallback_idx), max_proxy)]].astype(np.int64)
        else:
            q = float(overrides.get("v36_proxy_q", 0.92))
            thr = float(np.quantile(proxy_score[cand_idx], q)) if len(cand_idx) > 0 else float("inf")
            keep = cand_idx[proxy_score[cand_idx] >= thr]
            if len(keep) < min(len(cand_idx), proxy_min_keep):
                order = np.argsort(-proxy_score[cand_idx])
                keep = cand_idx[order[:min(len(cand_idx), proxy_min_keep)]]
            order = np.argsort(-proxy_score[keep])
            proxy_idx = keep[order[:min(len(keep), max_proxy)]].astype(np.int64)

    unrel_idx = np.unique(np.concatenate([base_unrel, proxy_idx])).astype(np.int64)

    spec = deepcopy(base_spec)
    spec["train"] = "lq_soft"
    spec["method_name"] = method_name
    spec["sup_idx"] = support_idx
    spec["idx_sup_eval"] = support_idx
    spec["align_idx"] = align_idx
    spec["unrel_idx"] = unrel_idx
    spec["idx_unk_eval"] = unrel_idx
    spec["P"] = P_full
    spec["fallback_name"] = base_name
    spec["unknown_loss_mode"] = spec.get("unknown_loss_mode", "energy_margin")
    spec["unknown_energy_margin"] = float(spec.get("unknown_energy_margin", UNKNOWN_ENERGY_MARGIN))
    spec["lambda_ent_max"] = float(spec.get("lambda_ent_max", 0.0)) * float(overrides.get("v36_lambda_ent_max_scale", 1.15))
    spec["bucket_info"] = {
        **dict(base_spec.get("bucket_info", {})),
        "v36_active": True,
        "v36_proxy_added": int(len(proxy_idx)),
        "v36_proxy_total_unrel": int(len(unrel_idx)),
        "v36_proxy_cap": float(proxy_cap),
        "v36_proxy_q": float(overrides.get("v36_proxy_q", 0.92)),
        "v36_proxy_mean_score": float(np.mean(proxy_score[proxy_idx])) if len(proxy_idx) > 0 else 0.0,
    }
    return spec, spec["bucket_info"]


def build_v36_method_info_from_spec(method_name, spec, y_adapt, extra_info=None, base_name=None, target_source=None):
    sel_idx = np.asarray(spec.get("idx_sup_eval", spec.get("sup_idx", np.zeros((0), dtype=np.int64))), dtype=np.int64)
    stats = summarize_method_on_selected(sel_idx, y_adapt, spec["P"])
    total_known = max(1, int(np.sum(y_adapt >= 0)))
    sel_precision = float((y_adapt[sel_idx] >= 0).mean()) if len(sel_idx) > 0 else float("nan")
    sel_recall = float(np.sum(y_adapt[sel_idx] >= 0) / total_known) if len(sel_idx) > 0 else 0.0
    sel_keep = float(len(sel_idx) / max(1, len(y_adapt)))
    unk_idx = np.asarray(spec.get("idx_unk_eval", spec.get("unrel_idx", np.zeros((0), dtype=np.int64))), dtype=np.int64)
    unknown_keep = float(len(unk_idx) / max(1, len(y_adapt)))
    unknown_precision = float((y_adapt[unk_idx] < 0).mean()) if len(unk_idx) > 0 else float("nan")
    info = dict(
        sel_precision=sel_precision,
        sel_recall=sel_recall,
        sel_keep_ratio=sel_keep,
        sel_size=stats["sel_size"],
        pseudo_acc_selected=stats["pseudo_acc"],
        pseudo_acc=stats["pseudo_acc"],
        unknown_cand_keep=unknown_keep,
        unknown_cand_precision=unknown_precision,
        v36_target_source=target_source,
        v36_base_name=base_name,
        v36_proxy_added=int(spec.get("bucket_info", {}).get("v36_proxy_added", 0)),
        v36_proxy_total_unrel=int(len(unk_idx)),
        v36_lambda_ent_max=float(spec.get("lambda_ent_max", np.nan)),
    )
    if extra_info:
        info.update(extra_info)
    return info


_evaluate_variant_with_rejector_v35 = evaluate_variant_with_rejector

def evaluate_variant_with_rejector(eval_cache, y_A, y_B, y_C, final_rejector_head="module2", frr_target=0.05, base_name=None):
    out = _evaluate_variant_with_rejector_v35(eval_cache, y_A, y_B, y_C, final_rejector_head=final_rejector_head, frr_target=frr_target, base_name=base_name)
    unknown_acc = float(np.clip(1.0 - out["FAR_unknown_accept"], 0.0, 1.0))
    out["H_KU"] = harmonic_mean_np(out["known_acc_e2e"], unknown_acc)
    out["H_DU"] = harmonic_mean_np(out["drift_acc_all"], unknown_acc)
    return out


In [13]:
from pathlib import Path
def _stagea_cache_path(fold_dir):
    return os.path.join(fold_dir, STAGEA_CACHE_FILENAME)

def _save_stagea_cache(payload, cache_path):
    ensure_parent_dir(cache_path)
    Path(_win_safe_path(cache_path)).parent.mkdir(parents=True, exist_ok=True)
    payload_to_save = dict(payload)
    payload_to_save.pop("model", None)
    with _safe_open(cache_path, "wb") as f:
        pickle.dump(payload_to_save, f, protocol=pickle.HIGHEST_PROTOCOL)

def _load_stagea_cache(cache_path):
    with _safe_open(cache_path, "rb") as f:
        return pickle.load(f)

def build_or_load_stagea_fold(
    fold_dir,
    protocol_tag,
    split_id,
    fold,
    K,
    source_rxs,
    drift_rx,
    X_src, y_src, D_src, rx_src,
    idx_train, idx_val, idx_te,
    X_B, y_B, D_B,
    X_C, y_C, D_C,
    X_D, y_D, D_D,
    X_E, y_E, D_E,
    X_F, y_F, D_F):
    _safe_makedirs(fold_dir)
    cache_path = _stagea_cache_path(fold_dir)
    cache_path = _win_safe_path(cache_path)
    ensure_parent_dir(cache_path)
    if STAGEA_CACHE_ENABLED and (not STAGEA_CACHE_FORCE_REBUILD) and os.path.exists(cache_path):
        payload = _load_stagea_cache(cache_path)
        if payload.get("cache_version", None) == STAGEA_CACHE_VERSION:
            model = ResNet18_1D(num_classes=K, in_planes=IN_PLANES, dropout=DROPOUT).to(device)
            model.load_state_dict(payload["best_state"])
            model.eval()
            payload["model"] = model
            payload["cache_hit"] = True
            log_kv(
                f"{protocol_tag}|split{split_id}|fold{fold}",
                stageA="cache_hit",
                cache=os.path.basename(cache_path),
                adapt_n=len(payload["Z_adapt"]),
                eval_B=len(payload["Z_B_ev"]),
                eval_C=len(payload["Z_C_ev"]),
                eval_U=len(payload["Z_U_ev"]))
            return payload

    t0 = time.time()
    train_loader = DataLoader(ArrayDataset(X_src[idx_train], y_src[idx_train]), batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(ArrayDataset(X_src[idx_val],   y_src[idx_val]),   batch_size=BATCH_SIZE, shuffle=False)

    model = ResNet18_1D(num_classes=K, in_planes=IN_PLANES, dropout=DROPOUT).to(device)
    opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_val = -1.0
    best_state = None
    patience = 0
    for ep in range(1, EPOCHS + 1):
        _ = run_epoch(model, train_loader, optimizer=opt)
        _, va_acc = run_epoch(model, val_loader, optimizer=None)
        if va_acc > best_val + 1e-4:
            best_val = va_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
            if patience >= PATIENCE:
                break

    model.load_state_dict(best_state)
    model.eval()
    torch.save(best_state, _win_safe_path(os.path.join(fold_dir, "best_source_model.pth")))

    logits_tr, Z_tr = infer_logits_embed(model, X_src[idx_train], batch=512)
    logits_A, Z_A = infer_logits_embed(model, X_src[idx_te], batch=512)
    D_A = D_src[idx_te]
    y_A_eval = y_src[idx_te]
    y_tr_fold = y_src[idx_train]

    mu_z_src, var_z_src = fit_emb_maha_diag(Z_tr, y_tr_fold, K)
    ref_sid_emb_A = sid_embmaha(Z_A, mu_z_src, var_z_src)
    ref_sid_en_A  = sid_energy(logits_A)
    source_rx_ids = sorted({RX_USE.index(r) for r in source_rxs})
    models_kr, fallback_k = fit_device_rx_models(D_src[idx_train], y_tr_fold, rx_src[idx_train], K, source_rx_ids, reg=1e-3, min_n=20)
    sc_A = compute_module2_scores_from_cached(
        Z_A, logits_A, D_A,
        mu_z_src, var_z_src,
        ref_sid_emb_A, ref_sid_en_A,
        models_kr, fallback_k, source_rx_ids
    )
    tau_id, tau_dom = calibrate_module2_thresholds(sc_A["Sid"], sc_A["Sdom"], FRR_TARGET, FALSE_DRIFT_TARGET)

    Sdom_tr_true = sdom_V1_mixNLL(D_src[idx_train], y_tr_fold, models_kr, fallback_k, source_rx_ids)
    dom_mu, dom_std, dom_global_mu, dom_global_std = fit_classwise_dom_stats(Sdom_tr_true, y_tr_fold, K, min_std=0.10)

    protos = fit_class_prototypes(Z_tr, y_tr_fold, K, l2norm=True)
    Zm, ym = build_knn_memory(Z_tr, y_tr_fold, per_class=KNN_MEM_PER_CLASS, seed=SEED + 31*fold)

    P_logit_A = softmax_np(logits_A)
    tau_conf = float(np.quantile(msp_from_probs(P_logit_A), CONF_STABLE_Q))

    B_sp = split_adapt_eval(X_B, y_B, D_B, frac=ADAPT_FRAC, max_adapt=MAX_ADAPT_PER_SET, max_eval=MAX_EVAL_PER_SET, seed=10000+fold)
    C_sp = split_adapt_eval(X_C, y_C, D_C, frac=ADAPT_FRAC, max_adapt=MAX_ADAPT_PER_SET, max_eval=MAX_EVAL_PER_SET, seed=11000+fold)
    D_sp = split_adapt_eval(X_D, y_D, D_D, frac=ADAPT_FRAC, max_adapt=MAX_ADAPT_PER_SET, max_eval=MAX_EVAL_PER_SET, seed=12000+fold)
    E_sp = split_adapt_eval(X_E, y_E, D_E, frac=ADAPT_FRAC, max_adapt=MAX_ADAPT_PER_SET, max_eval=MAX_EVAL_PER_SET, seed=13000+fold)
    F_sp = split_adapt_eval(X_F, y_F, D_F, frac=ADAPT_FRAC, max_adapt=MAX_ADAPT_PER_SET, max_eval=MAX_EVAL_PER_SET, seed=14000+fold)

    X_adapt, y_adapt, D_adapt = concat_sets([
        (B_sp[0], B_sp[1], B_sp[2]),
        (C_sp[0], C_sp[1], C_sp[2]),
        (D_sp[0], D_sp[1], D_sp[2]),
        (E_sp[0], E_sp[1], E_sp[2]),
        (F_sp[0], F_sp[1], F_sp[2])])

    X_B_ev, y_B_ev, D_B_ev = B_sp[3], B_sp[4], B_sp[5]
    X_C_ev, y_C_ev, D_C_ev = C_sp[3], C_sp[4], C_sp[5]
    X_U_ev, y_U_ev, D_U_ev = concat_sets([
        (D_sp[3], D_sp[4], D_sp[5]),
        (E_sp[3], E_sp[4], E_sp[5]),
        (F_sp[3], F_sp[4], F_sp[5])])

    logits_B_ad, Z_B_ad = infer_logits_embed(model, B_sp[0], batch=512)
    logits_C_ad, Z_C_ad = infer_logits_embed(model, C_sp[0], batch=512)
    logits_D_ad, Z_D_ad = infer_logits_embed(model, D_sp[0], batch=512)
    logits_E_ad, Z_E_ad = infer_logits_embed(model, E_sp[0], batch=512)
    logits_F_ad, Z_F_ad = infer_logits_embed(model, F_sp[0], batch=512)

    router_bundle_v8 = fit_module2_v8_router_from_source_calibration(
        X_stable=X_src[idx_te],
        y_stable=y_A_eval,
        Z_stable=Z_A,
        logits_stable=logits_A,
        D_stable=D_A,
        protos=protos,
        mu_z_src=mu_z_src, var_z_src=var_z_src,
        ref_sid_emb_A=ref_sid_emb_A, ref_sid_en_A=ref_sid_en_A,
        models_kr=models_kr, fallback_k=fallback_k, source_rx_ids=source_rx_ids,
        dom_mu=dom_mu, dom_std=dom_std, dom_global_mu=dom_global_mu, dom_global_std=dom_global_std,
        seed=SEED + 1000 * split_id + 100 * len(source_rxs) + 10 * fold + 7)

    logits_adapt, Z_adapt = infer_logits_embed(model, X_adapt, batch=512)
    sc_adapt = compute_module2_scores_from_cached(
        Z_adapt, logits_adapt, D_adapt,
        mu_z_src, var_z_src,
        ref_sid_emb_A, ref_sid_en_A,
        models_kr, fallback_k, source_rx_ids,
        tau_id=tau_id, tau_dom=tau_dom
    )
    sc_adapt_v8 = compute_module2_v8_scores_from_cached(Z_adapt, logits_adapt, D_adapt, router_bundle_v8)
    gate_pred_adapt = sc_adapt_v8["gate_pred"]

    P_logit_adapt = softmax_np(logits_adapt)
    P_proto_adapt = proto_probs_cosine(Z_adapt, protos, temp=PROTO_T, l2norm=True)
    P_knn_adapt   = knn_class_probs(Z_adapt, Zm, ym, K, topk=KNN_TOPK)

    logits_B_ev, Z_B_ev = infer_logits_embed(model, X_B_ev, batch=512)
    logits_C_ev, Z_C_ev = infer_logits_embed(model, X_C_ev, batch=512)
    logits_U_ev, Z_U_ev = infer_logits_embed(model, X_U_ev, batch=512)

    P_logit_B = softmax_np(logits_B_ev)
    P_proto_B = proto_probs_cosine(Z_B_ev, protos, temp=PROTO_T, l2norm=True)
    P_knn_B   = knn_class_probs(Z_B_ev, Zm, ym, K, topk=KNN_TOPK)
    P_lp_B    = weighted_prob_fusion([P_logit_B, P_proto_B], [W_LOGIT, W_PROTO])
    P_tri_B   = weighted_prob_fusion([P_logit_B, P_proto_B, P_knn_B], [W_LOGIT, W_PROTO, W_KNN])

    P_logit_C = softmax_np(logits_C_ev)
    P_proto_C = proto_probs_cosine(Z_C_ev, protos, temp=PROTO_T, l2norm=True)
    P_knn_C   = knn_class_probs(Z_C_ev, Zm, ym, K, topk=KNN_TOPK)
    P_lp_C    = weighted_prob_fusion([P_logit_C, P_proto_C], [W_LOGIT, W_PROTO])
    P_tri_C   = weighted_prob_fusion([P_logit_C, P_proto_C, P_knn_C], [W_LOGIT, W_PROTO, W_KNN])

    Z_replay, y_replay = make_replay_subset(Z_tr, y_tr_fold, per_class=REPLAY_PER_CLASS, seed=SEED+fold)

    payload = dict(
        cache_version=STAGEA_CACHE_VERSION,
        best_state=best_state,
        idx_train=np.asarray(idx_train, dtype=np.int64),
        idx_val=np.asarray(idx_val, dtype=np.int64),
        idx_te=np.asarray(idx_te, dtype=np.int64),
        y_tr_fold=np.asarray(y_tr_fold),
        logits_tr=logits_tr, Z_tr=Z_tr,
        logits_A=logits_A, Z_A=Z_A, y_A_eval=y_A_eval, D_A=D_A,
        mu_z_src=mu_z_src, var_z_src=var_z_src,
        ref_sid_emb_A=ref_sid_emb_A, ref_sid_en_A=ref_sid_en_A,
        source_rx_ids=source_rx_ids,
        models_kr=models_kr, fallback_k=fallback_k,
        tau_id=float(tau_id), tau_dom=float(tau_dom), tau_conf=float(tau_conf),
        dom_mu=dom_mu, dom_std=dom_std, dom_global_mu=dom_global_mu, dom_global_std=dom_global_std,
        protos=protos, Zm=Zm, ym=ym,
        logits_adapt=logits_adapt, Z_adapt=Z_adapt, y_adapt=y_adapt, D_adapt=D_adapt,
        sc_adapt=sc_adapt, sc_adapt_v8=sc_adapt_v8, gate_pred_adapt=gate_pred_adapt,
        router_bundle_v8=router_bundle_v8,
        P_logit_adapt=P_logit_adapt, P_proto_adapt=P_proto_adapt, P_knn_adapt=P_knn_adapt,
        logits_B_ev=logits_B_ev, Z_B_ev=Z_B_ev, y_B_ev=y_B_ev, D_B_ev=D_B_ev,
        logits_C_ev=logits_C_ev, Z_C_ev=Z_C_ev, y_C_ev=y_C_ev, D_C_ev=D_C_ev,
        logits_U_ev=logits_U_ev, Z_U_ev=Z_U_ev, y_U_ev=y_U_ev, D_U_ev=D_U_ev,
        P_logit_B=P_logit_B, P_proto_B=P_proto_B, P_knn_B=P_knn_B, P_lp_B=P_lp_B, P_tri_B=P_tri_B,
        P_logit_C=P_logit_C, P_proto_C=P_proto_C, P_knn_C=P_knn_C, P_lp_C=P_lp_C, P_tri_C=P_tri_C,
        Z_replay=Z_replay, y_replay=y_replay,
        cache_hit=False,
        stageA_seconds=float(time.time() - t0))
    if STAGEA_CACHE_ENABLED:
        _save_stagea_cache(payload, cache_path)
        log_kv(
            f"{protocol_tag}|split{split_id}|fold{fold}",
            stageA="cache_saved",
            cache=os.path.basename(cache_path),
            seconds=f"{payload['stageA_seconds']:.1f}"
        )
    payload["model"] = model
    return payload


In [14]:
def run_one_split(protocol_tag, rx_protocol_tag, tx_protocol_tag, split_id, KNOWN_TX, UNKNOWN_TX, source_rxs, drift_rx, protocol_dir, stagea_cache_protocol_dir=None):
    if "apply_soda_protocol_runtime" in globals():
        _active_soda_cfg = apply_soda_protocol_runtime(protocol_tag)
    else:
        _active_soda_cfg = {}
    split_dir = os.path.join(protocol_dir, f"txsplit_{split_id}")
    _safe_makedirs(split_dir)
    cache_protocol_dir = stagea_cache_protocol_dir if stagea_cache_protocol_dir is not None else protocol_dir
    cache_split_dir = os.path.join(cache_protocol_dir, f"txsplit_{split_id}")
    _safe_makedirs(cache_split_dir)
    with _safe_open(os.path.join(split_dir, "tx_split.json"), "w", encoding="utf-8") as f:
        json.dump({"protocol_tag": protocol_tag, "rx_protocol": rx_protocol_tag, "tx_protocol": tx_protocol_tag, "KNOWN_TX": KNOWN_TX, "UNKNOWN_TX": UNKNOWN_TX, "SOURCE_RXS": source_rxs, "DRIFT_RX": drift_rx}, f, indent=2)

    X_src, y_src, D_src, rx_src = build_source_train(compact_dataset, KNOWN_TX, source_rxs=source_rxs)
    K = len(KNOWN_TX)

    X_B, y_B, D_B = build_eval_set(compact_dataset, KNOWN_TX, KNOWN_TX, drift_rx, TEST_DATES_RX)
    X_C, y_C, D_C = build_eval_set(compact_dataset, KNOWN_TX, KNOWN_TX, source_rxs, TEST_DATES_DAY)
    X_D, y_D, D_D = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, source_rxs, TEST_DATES_RX)
    X_E, y_E, D_E = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, drift_rx, TEST_DATES_RX)
    X_F, y_F, D_F = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, source_rxs, TEST_DATES_DAY)

    log_kv(
        f"{protocol_tag}|split{split_id}",
        src_train=len(X_src), eval_known_rx=len(X_B), eval_known_day=len(X_C),
        eval_unk_src_rx=len(X_D), eval_unk_drift_rx=len(X_E), eval_unk_src_day=len(X_F)
    )

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 1000*split_id)
    rows = []
    tau_sweep_rows = []

    for fold, (idx_tr, idx_te) in enumerate(skf.split(X_src, y_src), start=1):
        if MAX_FOLDS_PER_PROTOCOL is not None and int(fold) > int(MAX_FOLDS_PER_PROTOCOL):
            break
        fold_dir = os.path.join(split_dir, f"fold_{fold}")
        cache_fold_dir = os.path.join(cache_split_dir, f"fold_{fold}")
        _safe_makedirs(fold_dir)
        _safe_makedirs(cache_fold_dir)

        rng = np.random.RandomState(SEED + 1000*split_id + fold)
        perm = rng.permutation(idx_tr)
        n_val = max(1, int(0.1 * len(perm)))
        idx_val = perm[:n_val]
        idx_train = perm[n_val:]

        stageA = build_or_load_stagea_fold(
            fold_dir=cache_fold_dir,
            protocol_tag=protocol_tag,
            split_id=split_id,
            fold=fold,
            K=K,
            source_rxs=source_rxs,
            drift_rx=drift_rx,
            X_src=X_src, y_src=y_src, D_src=D_src, rx_src=rx_src,
            idx_train=idx_train, idx_val=idx_val, idx_te=idx_te,
            X_B=X_B, y_B=y_B, D_B=D_B,
            X_C=X_C, y_C=y_C, D_C=D_C,
            X_D=X_D, y_D=y_D, D_D=D_D,
            X_E=X_E, y_E=y_E, D_E=D_E,
            X_F=X_F, y_F=y_F, D_F=D_F)

        model = stageA["model"]
        logits_tr, Z_tr = stageA["logits_tr"], stageA["Z_tr"]
        y_tr_fold = np.asarray(stageA.get("y_tr_fold", y_src[np.asarray(stageA.get("idx_train", idx_train), dtype=np.int64)]), dtype=np.int64)
        logits_A, Z_A = stageA["logits_A"], stageA["Z_A"]
        D_A = stageA["D_A"]
        mu_z_src, var_z_src = stageA["mu_z_src"], stageA["var_z_src"]
        ref_sid_emb_A, ref_sid_en_A = stageA["ref_sid_emb_A"], stageA["ref_sid_en_A"]
        source_rx_ids = stageA["source_rx_ids"]
        models_kr, fallback_k = stageA["models_kr"], stageA["fallback_k"]
        tau_id, tau_dom = stageA["tau_id"], stageA["tau_dom"]
        dom_mu, dom_std = stageA["dom_mu"], stageA["dom_std"]
        dom_global_mu, dom_global_std = stageA["dom_global_mu"], stageA["dom_global_std"]
        protos, Zm, ym = stageA["protos"], stageA["Zm"], stageA["ym"]
        tau_conf = stageA["tau_conf"]

        logits_adapt, Z_adapt = stageA["logits_adapt"], stageA["Z_adapt"]
        y_adapt, D_adapt = stageA["y_adapt"], stageA["D_adapt"]
        sc_adapt, sc_adapt_v8 = stageA["sc_adapt"], stageA["sc_adapt_v8"]
        gate_pred_adapt = stageA["gate_pred_adapt"]
        router_bundle_v8 = stageA["router_bundle_v8"]

        P_logit_adapt = stageA["P_logit_adapt"]
        P_proto_adapt = stageA["P_proto_adapt"]
        P_knn_adapt = stageA["P_knn_adapt"]

        methods, method_info, aux = build_pseudo_method_bank(
            K=K,
            Z_adapt=Z_adapt,
            gate_pred=gate_pred_adapt,
            y_true_adapt=y_adapt,
            P_logit_adapt=P_logit_adapt,
            P_proto_adapt=P_proto_adapt,
            P_knn_adapt=P_knn_adapt,
            protos=protos,
            tau_conf=tau_conf,
            Sid_adapt=sc_adapt_v8.get("Sid", sc_adapt["Sid"]),
            Sdom_adapt=sc_adapt_v8.get("Sdrift", sc_adapt["Sdom"]),
            gate_pred_v4=sc_adapt_v8.get("gate_pred", None),
            p_known_adapt=sc_adapt_v8.get("p_known", None),
            p_drift_given_known_adapt=sc_adapt_v8.get("p_drift_given_known", None),
            P_route_v4=sc_adapt_v8.get("route_probs", None),
            min_keep=PSEUDO_MIN_KEEP)
        dgr13_spec, dgr13_info = build_label_quality_method_from_base(
            methods["DG_RAINCOAT_v12"],
            mode="dgr",
            fc_layer=model.fc,
            Z_adapt=Z_adapt,
            P_logit_adapt=P_logit_adapt,
            P_proto_adapt=P_proto_adapt,
            P_knn_adapt=P_knn_adapt,
            protos=protos)
        if dgr13_spec is not None:
            methods["DG_RAINCOAT_v13LQ"] = dgr13_spec
            _base_info_dgr = dict(method_info.get("DG_RAINCOAT_v12", {}))
            method_info["DG_RAINCOAT_v13LQ"] = {**_base_info_dgr, **dgr13_spec.get("bucket_info", {}), **(dgr13_info or {})}
        dgr14_spec, dgr14_info = build_label_quality_method_from_base_v14(
            methods["DG_RAINCOAT_v12"],
            mode="dgr",
            fc_layer=model.fc,
            Z_adapt=Z_adapt,
            P_logit_adapt=P_logit_adapt,
            P_proto_adapt=P_proto_adapt,
            P_knn_adapt=P_knn_adapt,
            protos=protos,
            base_name="DG_RAINCOAT_v12")
        if dgr14_spec is not None:
            methods["DG_RAINCOAT_v14Stable"] = dgr14_spec
            _base_info_dgr14 = dict(method_info.get("DG_RAINCOAT_v12", {}))
            method_info["DG_RAINCOAT_v14Stable"] = {**_base_info_dgr14, **dgr14_spec.get("bucket_info", {}), **(dgr14_info or {})}
        dgr15_spec, dgr15_info = build_quality_aware_method_from_base_v15(
            methods.get("DG_RAINCOAT_v14Stable", methods.get("DG_RAINCOAT_v12")),
            mode="dgr",
            fc_layer=model.fc,
            Z_adapt=Z_adapt,
            P_logit_adapt=P_logit_adapt,
            P_proto_adapt=P_proto_adapt,
            P_knn_adapt=P_knn_adapt,
            protos=protos,
            base_name="DG_RAINCOAT_v14Stable")
        if dgr15_spec is not None:
            methods["DG_RAINCOAT_v15QW"] = dgr15_spec
            _base_info_dgr15 = dict(method_info.get("DG_RAINCOAT_v14Stable", method_info.get("DG_RAINCOAT_v12", {})))
            method_info["DG_RAINCOAT_v15QW"] = {**_base_info_dgr15, **dgr15_spec.get("bucket_info", {}), **(dgr15_info or {})}
        dgr16_spec, dgr16_info = build_quality_aware_method_from_base_v16(
            methods.get("DG_RAINCOAT_v15QW", methods.get("DG_RAINCOAT_v14Stable", methods.get("DG_RAINCOAT_v12"))),
            mode="dgr",
            fc_layer=model.fc,
            Z_adapt=Z_adapt,
            P_logit_adapt=P_logit_adapt,
            P_proto_adapt=P_proto_adapt,
            P_knn_adapt=P_knn_adapt,
            protos=protos,
            base_name="DG_RAINCOAT_v15QW")
        if dgr16_spec is not None:
            methods["DG_RAINCOAT_v16QW"] = dgr16_spec
            _base_info_dgr16 = dict(method_info.get("DG_RAINCOAT_v15QW", method_info.get("DG_RAINCOAT_v14Stable", method_info.get("DG_RAINCOAT_v12", {}))))
            method_info["DG_RAINCOAT_v16QW"] = {**_base_info_dgr16, **dgr16_spec.get("bucket_info", {}), **(dgr16_info or {})}
        dgr17_spec, dgr17_info = build_quality_aware_method_from_base_v17(
            methods.get("DG_RAINCOAT_v16QW", methods.get("DG_RAINCOAT_v15QW", methods.get("DG_RAINCOAT_v14Stable", methods.get("DG_RAINCOAT_v12")))),
            mode="dgr",
            fc_layer=model.fc,
            Z_adapt=Z_adapt,
            P_logit_adapt=P_logit_adapt,
            P_proto_adapt=P_proto_adapt,
            P_knn_adapt=P_knn_adapt,
            protos=protos,
            base_name="DG_RAINCOAT_v16QW")
        if dgr17_spec is not None:
            methods["DG_RAINCOAT_v17LCQ"] = dgr17_spec
            _base_info_dgr17 = dict(method_info.get("DG_RAINCOAT_v16QW", method_info.get("DG_RAINCOAT_v15QW", method_info.get("DG_RAINCOAT_v14Stable", method_info.get("DG_RAINCOAT_v12", {})))))
            method_info["DG_RAINCOAT_v17LCQ"] = {**_base_info_dgr17, **dgr17_spec.get("bucket_info", {}), **(dgr17_info or {})}
        elif "DG_RAINCOAT_v16QW" in methods:
            methods["DG_RAINCOAT_v17LCQ"] = deepcopy(methods["DG_RAINCOAT_v16QW"])
            method_info["DG_RAINCOAT_v17LCQ"] = {**dict(method_info.get("DG_RAINCOAT_v16QW", {})), "v17_lcq_active": False, "v17_lcq_reason": "builder_returned_none"}
        dgr19_spec, dgr19_info = build_m2v4_weighted_guided_method(
            methods.get("DG_RAINCOAT_v17LCQ", methods.get("DG_RAINCOAT_v16QW", methods.get("DG_RAINCOAT_v15QW", methods.get("DG_RAINCOAT_v14Stable", methods.get("DG_RAINCOAT_v12"))))),
            mode="dgr",
            Z_adapt=Z_adapt,
            p_known_adapt=sc_adapt_v8.get("p_known", np.ones((len(Z_adapt)), dtype=np.float32)),
            p_drift_given_known_adapt=sc_adapt_v8.get("p_drift_given_known", np.ones((len(Z_adapt)), dtype=np.float32) * 0.5),
            base_name="DG_RAINCOAT_v17LCQ")
        if dgr19_spec is not None:
            methods["DG_RAINCOAT_v19M2W"] = dgr19_spec
            _base_info_dgr19 = dict(method_info.get("DG_RAINCOAT_v17LCQ", method_info.get("DG_RAINCOAT_v16QW", method_info.get("DG_RAINCOAT_v15QW", {}))))
            method_info["DG_RAINCOAT_v19M2W"] = {**_base_info_dgr19, **dgr19_spec.get("bucket_info", {}), **(dgr19_info or {})}
        elif "DG_RAINCOAT_v17LCQ" in methods:
            methods["DG_RAINCOAT_v19M2W"] = deepcopy(methods["DG_RAINCOAT_v17LCQ"])
            method_info["DG_RAINCOAT_v19M2W"] = {**dict(method_info.get("DG_RAINCOAT_v17LCQ", {})), "v19_m2w_active": False, "v19_m2w_reason": "builder_returned_none"}
        # Build the single SODA-RFF mainline from the v19-style base spec.
        base_v19_name = "DG_RAINCOAT_v19M2W"
        base_v19_spec = methods.get(base_v19_name, None)
        if base_v19_spec is not None:
            if isinstance(aux, dict) and aux.get("P_refine", None) is not None:
                P_target_soda = aux.get("P_refine")
                soda_target_source = "aux.P_refine"
            elif isinstance(base_v19_spec, dict) and base_v19_spec.get("P", None) is not None:
                P_target_soda = base_v19_spec.get("P")
                soda_target_source = f"{base_v19_name}.P"
            else:
                P_target_soda = P_logit_adapt
                soda_target_source = "P_logit_adapt"
            P_target_soda = normalize_rows(np.asarray(P_target_soda, dtype=np.float32))

            soda_overrides = {}
            if "get_soda_protocol_fixed_config" in globals():
                _soda_cfg_protocol = get_soda_protocol_fixed_config(protocol_tag)
                for _kk in [
                    "v36_proxy_cap", "v36_proxy_q", "v36_proxy_pu_min", "v36_proxy_conf_max",
                    "v36_proxy_conf_smooth", "v36_proxy_entropy_mix", "v36_lambda_ent_max_scale",
                    "v36_proxy_min_keep"
                ]:
                    if _kk in _soda_cfg_protocol:
                        soda_overrides[_kk] = _soda_cfg_protocol[_kk]
            else:
                soda_overrides = dict(SODA_DEFAULT_OVERRIDES)
                soda_overrides.update(SODA_PROTOCOL_OVERRIDES.get(str(protocol_tag), {}))

            soda_spec, soda_extra = build_v36_unknown_proxy_from_base(
                base_v19_spec,
                Z_adapt=Z_adapt,
                p_known_adapt=sc_adapt_v8.get("p_known", np.ones((len(Z_adapt)), dtype=np.float32)),
                p_drift_given_known_adapt=sc_adapt_v8.get("p_drift_given_known", np.ones((len(Z_adapt)), dtype=np.float32) * 0.5),
                P_route_v4=sc_adapt_v8.get("route_probs", None),
                P_target=P_target_soda,
                tau_conf=tau_conf,
                base_name=base_v19_name,
                overrides=soda_overrides,
                method_name=SODA_METHOD_NAME
            )
            if soda_spec is None:
                method_info[SODA_METHOD_NAME] = {
                    "soda_active": False,
                    "soda_reason": "builder_returned_none",
                    "soda_method_name": SODA_METHOD_NAME,
                }
            else:
                methods[SODA_METHOD_NAME] = soda_spec
                method_info[SODA_METHOD_NAME] = build_v36_method_info_from_spec(
                    SODA_METHOD_NAME,
                    soda_spec,
                    y_adapt,
                    extra_info=soda_extra,
                    base_name=base_v19_name,
                    target_source=soda_target_source
                )
                method_info[SODA_METHOD_NAME]["framework_family"] = "SODA-RFF"
                method_info[SODA_METHOD_NAME]["framework_note"] = (
                    "State-aware open-set domain adaptation mainline with source-calibrated "
                    "known/drift/unknown responsibilities and unknown-proxy recovery."
                )
                if "get_soda_protocol_fixed_config" in globals():
                    _soda_cfg_protocol = get_soda_protocol_fixed_config(protocol_tag)
                    method_info[SODA_METHOD_NAME]["fixed_profile_tag"] = _soda_cfg_protocol.get("profile_tag", "unknown")
                    method_info[SODA_METHOD_NAME]["fixed_rejector_head"] = _soda_cfg_protocol.get("default_rejector_head", "energy")

        logits_B_ev, Z_B_ev = stageA["logits_B_ev"], stageA["Z_B_ev"]
        y_B_ev, D_B_ev = stageA["y_B_ev"], stageA["D_B_ev"]
        logits_C_ev, Z_C_ev = stageA["logits_C_ev"], stageA["Z_C_ev"]
        y_C_ev, D_C_ev = stageA["y_C_ev"], stageA["D_C_ev"]
        logits_U_ev, Z_U_ev = stageA["logits_U_ev"], stageA["Z_U_ev"]
        y_U_ev, D_U_ev = stageA["y_U_ev"], stageA["D_U_ev"]

        P_logit_B, P_proto_B, P_knn_B = stageA["P_logit_B"], stageA["P_proto_B"], stageA["P_knn_B"]
        P_lp_B, P_tri_B = stageA["P_lp_B"], stageA["P_tri_B"]
        P_logit_C, P_proto_C, P_knn_C = stageA["P_logit_C"], stageA["P_proto_C"], stageA["P_knn_C"]
        P_lp_C, P_tri_C = stageA["P_lp_C"], stageA["P_tri_C"]

        Z_replay, y_replay = stageA["Z_replay"], stageA["y_replay"]
        pseudo_eval = {
            "logit": dict(rx=pseudo_acc_from_probs(P_logit_B, y_B_ev), day=pseudo_acc_from_probs(P_logit_C, y_C_ev)),
            "lp":    dict(rx=pseudo_acc_from_probs(P_lp_B,    y_B_ev), day=pseudo_acc_from_probs(P_lp_C,    y_C_ev)),
            "tri":   dict(rx=pseudo_acc_from_probs(P_tri_B,   y_B_ev), day=pseudo_acc_from_probs(P_tri_C,   y_C_ev)),
        }
        pseudo_eval["logit"]["all"] = float((np.sum(np.argmax(P_logit_B,1) == y_B_ev) + np.sum(np.argmax(P_logit_C,1) == y_C_ev)) / (len(y_B_ev) + len(y_C_ev)))
        pseudo_eval["lp"]["all"]    = float((np.sum(np.argmax(P_lp_B,1)    == y_B_ev) + np.sum(np.argmax(P_lp_C,1)    == y_C_ev)) / (len(y_B_ev) + len(y_C_ev)))
        pseudo_eval["tri"]["all"]   = float((np.sum(np.argmax(P_tri_B,1)   == y_B_ev) + np.sum(np.argmax(P_tri_C,1)   == y_C_ev)) / (len(y_B_ev) + len(y_C_ev)))

        def _pseudo_metric(block, key, default=float("nan")):
            try:
                return float(pseudo_eval.get(block, {}).get(key, default))
            except Exception:
                return float(default)

        # Reconstructed external baselines under the unified embedding-space WiSig pipeline.
        # These are paper-guided comparison baselines, not line-by-line official reproductions.
        if EXPERIMENT_MODE == "ext_compare":
            def _row_norm_np(P):
                P = np.asarray(P, dtype=np.float32)
                P = np.clip(P, 1e-8, None)
                return (P / np.sum(P, axis=1, keepdims=True)).astype(np.float32)

            def _topk_mask(score, frac=0.25, min_n=128, largest=True):
                score = np.asarray(score, dtype=np.float32)
                n = len(score)
                mask = np.zeros((n), dtype=bool)
                if n == 0:
                    return mask
                k = int(max(min_n, round(float(frac) * n)))
                k = max(1, min(n, k))
                order = np.argsort(score)
                idx = order[-k:] if largest else order[:k]
                mask[idx] = True
                return mask

            def _safe_proto_matrix(proto_bank, K, dim):
                proto_mat = np.zeros((K, dim), dtype=np.float32)
                if isinstance(proto_bank, dict):
                    for kk, vv in proto_bank.items():
                        kk = int(kk)
                        if 0 <= kk < K:
                            proto_mat[kk] = np.asarray(vv, dtype=np.float32)
                else:
                    arr = np.asarray(proto_bank, dtype=np.float32)
                    if arr.ndim == 2:
                        upto = min(K, arr.shape[0])
                        proto_mat[:upto] = arr[:upto]
                return normalize_vec_rows_np(proto_mat + 1e-8)

            def _build_generic_spec(name, sup_mask, q=None, y=None, w_score=None, align_mask=None, unrel_mask=None, **cfg):
                sup_mask = np.asarray(sup_mask, dtype=bool)
                if sup_mask.sum() == 0:
                    fallback_score = w_score if w_score is not None else conf_logit_ext
                    sup_mask = _topk_mask(fallback_score, frac=0.15, min_n=min(192, len(Z_adapt)))
                sup_idx = np.where(sup_mask)[0].astype(np.int64)
                spec = dict(train="generic", sup_idx=sup_idx)
                if y is not None:
                    spec["y"] = np.asarray(y, dtype=np.int64)[sup_idx]
                else:
                    spec["q"] = np.asarray(q, dtype=np.float32)[sup_idx]
                if w_score is None:
                    w = np.ones((len(sup_idx)), dtype=np.float32)
                else:
                    w = np.asarray(w_score, dtype=np.float32)[sup_idx]
                    w = np.clip(w, 1e-3, None).astype(np.float32)
                spec["w"] = w
                spec["align_idx"] = np.where(np.asarray(align_mask, dtype=bool))[0].astype(np.int64) if align_mask is not None else np.zeros((0), dtype=np.int64)
                spec["unrel_idx"] = np.where(np.asarray(unrel_mask, dtype=bool))[0].astype(np.int64) if unrel_mask is not None else np.zeros((0), dtype=np.int64)
                spec.update(cfg)
                return spec

            p_known_ext = np.asarray(sc_adapt_v8.get("p_known", np.ones((len(Z_adapt)), dtype=np.float32)), dtype=np.float32)
            p_unk_ext = np.asarray(sc_adapt_v8.get("p_unknown", 1.0 - p_known_ext), dtype=np.float32)
            p_drift_ext = np.asarray(sc_adapt_v8.get("p_drift_given_known", np.full((len(Z_adapt)), 0.5, dtype=np.float32)), dtype=np.float32)
            conf_logit_ext = msp_from_probs(P_logit_adapt).astype(np.float32)
            conf_proto_ext = msp_from_probs(P_proto_adapt).astype(np.float32)
            margin_logit_ext = top2_margin(P_logit_adapt).astype(np.float32)
            pred_logit_ext = np.argmax(P_logit_adapt, axis=1).astype(np.int64)
            pred_proto_ext = np.argmax(P_proto_adapt, axis=1).astype(np.int64)
            pred_knn_ext = np.argmax(P_knn_adapt, axis=1).astype(np.int64)
            agree_lp_ext = (pred_logit_ext == pred_proto_ext).astype(np.float32)
            agree_tri_ext = ((pred_logit_ext == pred_proto_ext) & (pred_logit_ext == pred_knn_ext)).astype(np.float32)

            P_fuse_lp_ext = _row_norm_np(0.55 * P_logit_adapt + 0.45 * P_proto_adapt)
            P_fuse_tri_ext = _row_norm_np((P_logit_adapt + P_proto_adapt + P_knn_adapt) / 3.0)

            proto_mat_ext = _safe_proto_matrix(protos, K, Z_adapt.shape[1])
            Z_adapt_n = normalize_vec_rows_np(Z_adapt)
            dist_pred_ext = np.linalg.norm(Z_adapt_n - proto_mat_ext[pred_logit_ext], axis=1).astype(np.float32)
            dist_score_ext = (1.0 - robust_unit_interval(dist_pred_ext)).astype(np.float32)

            q_known50 = float(np.quantile(p_known_ext, 0.50)) if len(p_known_ext) > 0 else 0.5
            q_known60 = float(np.quantile(p_known_ext, 0.60)) if len(p_known_ext) > 0 else 0.5
            q_unk70 = float(np.quantile(p_unk_ext, 0.70)) if len(p_unk_ext) > 0 else 0.5
            q_unk75 = float(np.quantile(p_unk_ext, 0.75)) if len(p_unk_ext) > 0 else 0.5
            q_margin25 = float(np.quantile(margin_logit_ext, 0.25)) if len(margin_logit_ext) > 0 else 0.0
            q_conf35 = float(np.quantile(conf_logit_ext, 0.35)) if len(conf_logit_ext) > 0 else 0.0
            q_dist75 = float(np.quantile(dist_pred_ext, 0.75)) if len(dist_pred_ext) > 0 else 1.0

            # PCPD: prototype calibration + pseudo-labeling + alignment.
            pcpd_score = (
                0.40 * robust_unit_interval(p_known_ext) +
                0.25 * robust_unit_interval(conf_proto_ext) +
                0.20 * agree_lp_ext +
                0.15 * robust_unit_interval(1.0 - p_unk_ext)
            ).astype(np.float32)
            pcpd_sup = _topk_mask(pcpd_score, frac=0.22, min_n=min(224, len(Z_adapt)))
            pcpd_align = p_known_ext >= q_known50
            pcpd_unrel = p_unk_ext >= q_unk70
            methods["PCPD_recon"] = _build_generic_spec(
                "PCPD_recon",
                sup_mask=pcpd_sup,
                q=P_fuse_lp_ext,
                w_score=np.clip(pcpd_score, 1e-3, None),
                align_mask=pcpd_align,
                unrel_mask=pcpd_unrel,
                lambda_sup=1.0,
                lambda_align=0.05,
                lambda_ent_min=0.03,
                lambda_ent_max=0.05,
                lambda_cons=0.0,
                lambda_proto=0.35,
                dynamic_align=True,
                unknown_loss_mode=UNKNOWN_LOSS_DEFAULT,
                unknown_energy_margin=UNKNOWN_ENERGY_MARGIN)
            method_info["PCPD_recon"] = {
                "recon_family": "PCPD",
                "recon_note": "prototype-calibration + pseudo-label + domain-alignment reconstruction",
                "sel_keep": float(np.mean(pcpd_sup.astype(np.float32))),
                "unknown_keep": float(np.mean(pcpd_unrel.astype(np.float32))),
            }

            # OVANet: source-derived known/unknown boundary via one-vs-all style confidence.
            ova_score = (
                0.50 * robust_unit_interval(conf_logit_ext) +
                0.25 * robust_unit_interval(margin_logit_ext) +
                0.25 * robust_unit_interval(1.0 - p_unk_ext)
            ).astype(np.float32)
            ova_sup = _topk_mask(ova_score, frac=0.28, min_n=min(256, len(Z_adapt)))
            ova_align = (p_known_ext >= q_known60)
            ova_unrel = (p_unk_ext >= q_unk75) | ((conf_logit_ext <= q_conf35) & (~ova_sup))
            methods["OVANet_recon"] = _build_generic_spec(
                "OVANet_recon",
                sup_mask=ova_sup,
                q=P_logit_adapt,
                w_score=np.clip(ova_score, 1e-3, None),
                align_mask=ova_align,
                unrel_mask=ova_unrel,
                lambda_sup=1.0,
                lambda_align=0.0,
                lambda_ent_min=0.00,
                lambda_ent_max=0.10,
                lambda_cons=0.0,
                lambda_proto=0.0,
                dynamic_align=False,
                unknown_loss_mode=UNKNOWN_LOSS_DEFAULT,
                unknown_energy_margin=UNKNOWN_ENERGY_MARGIN)
            method_info["OVANet_recon"] = {
                "recon_family": "OVANet",
                "recon_note": "one-vs-all style known/unknown boundary reconstruction",
                "sel_keep": float(np.mean(ova_sup.astype(np.float32))),
                "unknown_keep": float(np.mean(ova_unrel.astype(np.float32))),
            }

            # COMET: mean-teacher / contrastive flavored reconstruction with consistency and target entropy shaping.
            comet_score = (
                0.35 * robust_unit_interval(conf_logit_ext) +
                0.25 * robust_unit_interval(1.0 - p_unk_ext) +
                0.20 * robust_unit_interval(margin_logit_ext) +
                0.10 * agree_lp_ext +
                0.10 * agree_tri_ext
            ).astype(np.float32)
            comet_sup = _topk_mask(comet_score, frac=0.25, min_n=min(224, len(Z_adapt)))
            comet_align = p_known_ext >= q_known60
            comet_unrel = (p_unk_ext >= q_unk70) | (margin_logit_ext <= q_margin25)
            methods["COMET_recon"] = _build_generic_spec(
                "COMET_recon",
                sup_mask=comet_sup,
                q=P_fuse_tri_ext,
                w_score=np.clip(comet_score, 1e-3, None),
                align_mask=comet_align,
                unrel_mask=comet_unrel,
                lambda_sup=1.0,
                lambda_align=0.02,
                lambda_ent_min=0.03,
                lambda_ent_max=0.08,
                lambda_cons=0.10,
                lambda_proto=0.20,
                dynamic_align=False,
                unknown_loss_mode=UNKNOWN_LOSS_DEFAULT,
                unknown_energy_margin=UNKNOWN_ENERGY_MARGIN)
            method_info["COMET_recon"] = {
                "recon_family": "COMET",
                "recon_note": "mean-teacher + contrastive reconstruction under unified embedding pipeline",
                "sel_keep": float(np.mean(comet_sup.astype(np.float32))),
                "unknown_keep": float(np.mean(comet_unrel.astype(np.float32))),
            }

            # JRFFP-SC: prediction + siamese comparison, reconstructed via predicted-class proto-distance gating.
            jsc_score = (
                0.45 * robust_unit_interval(conf_logit_ext) +
                0.35 * dist_score_ext +
                0.20 * agree_lp_ext
            ).astype(np.float32)
            jsc_sup = _topk_mask(jsc_score, frac=0.20, min_n=min(160, len(Z_adapt)))
            jsc_unrel = (dist_pred_ext >= q_dist75) | (p_unk_ext >= q_unk75)
            methods["JRFFP_SC_recon"] = _build_generic_spec(
                "JRFFP_SC_recon",
                sup_mask=jsc_sup,
                y=pred_logit_ext,
                w_score=np.clip(jsc_score, 1e-3, None),
                align_mask=np.zeros((len(Z_adapt)), dtype=bool),
                unrel_mask=jsc_unrel,
                lambda_sup=1.0,
                lambda_align=0.0,
                lambda_ent_min=0.0,
                lambda_ent_max=0.05,
                lambda_cons=0.0,
                lambda_proto=0.55,
                dynamic_align=False,
                unknown_loss_mode=UNKNOWN_LOSS_DEFAULT,
                unknown_energy_margin=UNKNOWN_ENERGY_MARGIN)
            method_info["JRFFP_SC_recon"] = {
                "recon_family": "JRFFP-SC",
                "recon_note": "joint prediction + siamese-comparison reconstruction via class-prototype distance gating",
                "sel_keep": float(np.mean(jsc_sup.astype(np.float32))),
                "unknown_keep": float(np.mean(jsc_unrel.astype(np.float32))),
            }
        # Compact compare-set baselines are expected to be instantiated directly in the ext_compare block above.
        # Do not lazily reconstruct them here: that path is fragile because it closes over many intermediate
        # quantities and can fail when notebook execution order changes. Instead, evaluation below is restricted
        # to methods that were actually instantiated in this fold.


        # Restrict internal methods to the selected representative compare set before training adapters.
        methods = {k: v for k, v in methods.items() if k in METHOD_ORDER}
        method_info = {k: v for k, v in method_info.items() if (k in methods) or (k == "SourceOnly")}

        adapters = {"SourceOnly": None}
        extra_method_info = {}
        for name, spec in methods.items():
            if spec["train"] == "hard":
                idx = spec["idx"]
                adapters[name] = adapt_on_embeddings_hard(
                    model.fc, Z_adapt[idx], spec["y"], Z_replay, y_replay, target_weights=spec["w"],
                    bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE, epochs=ADAPT_EPOCHS,
                    lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                    lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR
                )
            elif spec["train"] == "soft":
                idx = spec["idx"]
                adapters[name] = adapt_on_embeddings_soft(
                    model.fc, Z_adapt[idx], spec["q"], Z_replay, y_replay, target_weights=spec["w"],
                    bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE, epochs=ADAPT_EPOCHS,
                    lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                    lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR
                )
            elif spec["train"] == "generic":
                sup_idx = spec["sup_idx"]
                align_idx = spec.get("align_idx", np.zeros((0), dtype=np.int64))
                unrel_idx = spec.get("unrel_idx", np.zeros((0), dtype=np.int64))
                adapters[name] = adapt_on_embeddings_generic(
                    fc_layer=model.fc,
                    Z_replay=Z_replay, y_replay=y_replay,
                    Z_sup=Z_adapt[sup_idx] if len(sup_idx) > 0 else None,
                    y_sup=spec.get("y", None),
                    q_sup=spec.get("q", None),
                    w_sup=spec.get("w", None),
                    Z_align=Z_adapt[align_idx] if len(align_idx) > 0 else None,
                    Z_unrel=Z_adapt[unrel_idx] if len(unrel_idx) > 0 else None,
                    proto_bank=protos,
                    bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE, epochs=ADAPT_EPOCHS,
                    lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                    lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR,
                    lambda_align=spec.get("lambda_align", 0.0),
                    lambda_ent_min=spec.get("lambda_ent_min", 0.0),
                    lambda_ent_max=spec.get("lambda_ent_max", 0.0),
                    lambda_cons=spec.get("lambda_cons", 0.0),
                    lambda_proto=spec.get("lambda_proto", 0.0),
                    dynamic_align=spec.get("dynamic_align", False),
                    unknown_loss_mode=spec.get("unknown_loss_mode", UNKNOWN_LOSS_DEFAULT),
                    unknown_energy_margin=spec.get("unknown_energy_margin", UNKNOWN_ENERGY_MARGIN))
            elif spec["train"] == "lq_soft":
                sup_idx = np.asarray(spec.get("sup_idx", np.zeros((0), dtype=np.int64)), dtype=np.int64)
                align_idx = np.asarray(spec.get("align_idx", np.zeros((0), dtype=np.int64)), dtype=np.int64)
                unrel_idx = np.asarray(spec.get("unrel_idx", np.zeros((0), dtype=np.int64)), dtype=np.int64)
                adapters[name] = adapt_on_embeddings_generic(
                    fc_layer=model.fc,
                    Z_replay=Z_replay, y_replay=y_replay,
                    Z_sup=Z_adapt[sup_idx] if len(sup_idx) > 0 else None,
                    q_sup=spec["q"] if len(sup_idx) > 0 else None,
                    w_sup=spec.get("w", None),
                    Z_align=Z_adapt[align_idx] if len(align_idx) > 0 else None,
                    Z_unrel=Z_adapt[unrel_idx] if len(unrel_idx) > 0 else None,
                    proto_bank=protos,
                    bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE, epochs=ADAPT_EPOCHS,
                    lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                    lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR,
                    lambda_sup=spec.get("lambda_sup", 1.0),
                    lambda_align=spec.get("lambda_align", 0.0),
                    lambda_ent_min=spec.get("lambda_ent_min", 0.0),
                    lambda_ent_max=spec.get("lambda_ent_max", 0.0),
                    lambda_cons=spec.get("lambda_cons", 0.0),
                    lambda_proto=spec.get("lambda_proto", 0.0),
                    dynamic_align=spec.get("dynamic_align", True),
                    unknown_loss_mode=spec.get("unknown_loss_mode", UNKNOWN_LOSS_DEFAULT),
                    unknown_energy_margin=spec.get("unknown_energy_margin", UNKNOWN_ENERGY_MARGIN))
                fallback_name = spec.get("fallback_name", None)
                if fallback_name is not None and fallback_name in adapters:
                    adapters[name], guard_info = guard_adapter_with_fallback(
                        adapters[name], adapters[fallback_name], model.fc, Z_replay, y_replay,
                        Z_probe=Z_adapt[sup_idx] if len(sup_idx) > 0 else None, probe_name=name
                    )
                    extra_method_info[name] = {**spec.get("bucket_info", {}), **guard_info}
                else:
                    extra_method_info[name] = spec.get("bucket_info", {})
            elif spec["train"] == "weighted_open_minimal":
                sup_idx = np.asarray(spec.get("sup_idx", np.zeros((0), dtype=np.int64)), dtype=np.int64)
                stab_idx = np.asarray(spec.get("stab_idx", np.zeros((0), dtype=np.int64)), dtype=np.int64)
                unk_idx = np.asarray(spec.get("unk_idx", np.zeros((0), dtype=np.int64)), dtype=np.int64)
                adapters[name] = adapt_on_embeddings_weighted_open_minimal(
                    fc_layer=model.fc,
                    Z_replay=Z_replay, y_replay=y_replay,
                    Z_sup=Z_adapt[sup_idx] if len(sup_idx) > 0 else None,
                    q_sup=spec.get("q", None),
                    w_sup=spec.get("w_sup", None),
                    Z_stab=Z_adapt[stab_idx] if len(stab_idx) > 0 else None,
                    w_stab=spec.get("w_stab", None),
                    Z_unk=Z_adapt[unk_idx] if len(unk_idx) > 0 else None,
                    w_unk=spec.get("w_unk", None),
                    proto_bank=protos,
                    bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE, epochs=ADAPT_EPOCHS,
                    lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                    lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR,
                    lambda_sup=spec.get("lambda_sup", V22_LAMBDA_SUP),
                    lambda_proto=spec.get("lambda_proto", V22_LAMBDA_PROTO),
                    lambda_stab=spec.get("lambda_stab", V22_LAMBDA_STAB),
                    lambda_unkE=spec.get("lambda_unkE", V22_LAMBDA_UNKE),
                    unknown_energy_margin=spec.get("unknown_energy_margin", V22_UNKNOWN_ENERGY_MARGIN),
                    energy_bg_q=spec.get("energy_bg_q", V22_ENERGY_BG_Q),
                    energy_bg_ema=spec.get("energy_bg_ema", V22_ENERGY_BG_EMA))
                extra_method_info[name] = spec.get("bucket_info", {})
            elif spec["train"] == "three_bucket":
                adapters[name] = adapt_three_bucket_v5(
                    fc_layer=model.fc,
                    Z_replay=Z_replay, y_replay=y_replay,
                    Z_target=Z_adapt,
                    idx_gate=spec["idx_gate"],
                    idx_rel=spec["idx_rel"],
                    idx_amb=spec["idx_amb"],
                    idx_unk=spec["idx_unk"],
                    q_seed=spec["q_seed"],
                    w_rel=spec.get("w_rel", None),
                    w_amb=spec.get("w_amb", None),
                    proto_bank=protos,
                    bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE,
                    epochs=THREE_BUCKET_EPOCHS,
                    lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                    lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR,
                    lambda_rel_sup=spec.get("lambda_rel_sup", 1.0),
                    lambda_amb_sup=spec.get("lambda_amb_sup", LAMBDA_AMB_SUP),
                    lambda_align=spec.get("lambda_align", LAMBDA_ALIGN),
                    lambda_ent_min=spec.get("lambda_ent_min", LAMBDA_ENT_MIN),
                    lambda_ent_max=spec.get("lambda_ent_max", LAMBDA_ENT_MAX),
                    lambda_cons=spec.get("lambda_cons", LAMBDA_CONS),
                    lambda_proto=spec.get("lambda_proto", LAMBDA_PROTO),
                    lambda_src_proto=spec.get("lambda_src_proto", LAMBDA_SRC_PROTO),
                    lambda_u_repel=spec.get("lambda_u_repel", LAMBDA_UNKNOWN_REPEL),
                    lambda_src_logit=spec.get("lambda_src_logit", LAMBDA_SRC_LOGIT),
                    use_ema=spec.get("use_ema", True),
                    ema_momentum=EMA_MOMENTUM,
                    teacher_blend=spec.get("teacher_blend", EMA_TEACHER_BLEND),
                    unknown_warmup_epochs=spec.get("unknown_warmup_epochs", UNKNOWN_WARMUP_EPOCHS),
                    unknown_ramp_epochs=spec.get("unknown_ramp_epochs", UNKNOWN_RAMP_EPOCHS))
                extra_method_info[name] = spec.get("bucket_info", {})

            elif spec["train"] == "raincoat":
                adapters[name], rain_info = adapt_raincoat_lite(
                    fc_layer=model.fc,
                    Z_target=Z_adapt,
                    idx_gate=spec["idx_gate"],
                    idx_common_seed=spec["sup_idx"],
                    idx_unknown_seed=spec["unrel_idx"],
                    Z_replay=Z_replay, y_replay=y_replay,
                    protos=protos, tau_conf=tau_conf, min_keep=PSEUDO_MIN_KEEP,
                    bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE,
                    epochs_stage1=RAIN_STAGE1_EPOCHS, epochs_stage2=RAIN_STAGE2_EPOCHS,
                    lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                    lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR)
                extra_method_info[name] = rain_info
            elif spec["train"] == "dg_raincoat_versioned":
                adapters[name], dg_info = adapt_dg_raincoat_lite_versioned(
                    fc_layer=model.fc,
                    Z_replay=Z_replay, y_replay=y_replay,
                    Z_target=Z_adapt,
                    idx_gate=spec["idx_gate"], idx_rel=spec["idx_rel"], idx_amb=spec["idx_amb"], idx_unk=spec["idx_unk"],
                    q_seed=spec["q_seed"], w_rel=spec["w_rel"], w_amb=spec["w_amb"],
                    protos=protos,
                    Sid_adapt=sc_adapt["Sid"], Sdom_adapt=sc_adapt["Sdom"],
                    bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE,
                    epochs_stage1=DG_STAGE1_EPOCHS, epochs_stage2=DG_STAGE2_EPOCHS,
                    lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                    lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR,
                    lambda_align_stage1=spec.get("lambda_align_stage1", DG_LAMBDA_ALIGN_STAGE1),
                    lambda_align_stage2=spec.get("lambda_align_stage2", DG_LAMBDA_ALIGN_STAGE2),
                    lambda_ent_min=spec.get("lambda_ent_min", DG_LAMBDA_ENT_MIN),
                    lambda_ent_max=spec.get("lambda_ent_max", DG_LAMBDA_ENT_MAX),
                    lambda_cons=spec.get("lambda_cons", DG_LAMBDA_CONS),
                    lambda_proto=spec.get("lambda_proto", DG_LAMBDA_PROTO),
                    lambda_u_repel=spec.get("lambda_u_repel", DG_LAMBDA_UNKNOWN_REPEL),
                    version=spec.get("version", "v10"),
                    unknown_loss_mode=spec.get("unknown_loss_mode", UNKNOWN_LOSS_DEFAULT),
                    unknown_energy_margin=spec.get("unknown_energy_margin", UNKNOWN_ENERGY_MARGIN))
                extra_method_info[name] = {**spec.get("bucket_info", {}), **(dg_info or {})}
            elif spec["train"] == "three_bucket_v6":
                adapters[name] = adapt_three_bucket_v6(
                    fc_layer=model.fc,
                    Z_replay=Z_replay, y_replay=y_replay,
                    Z_target=Z_adapt,
                    idx_gate=spec["idx_gate"], idx_rel=spec["idx_rel"], idx_amb=spec["idx_amb"],
                    idx_weak=spec["idx_weak"], idx_unk=spec["idx_unk"],
                    q_seed=spec["q_seed"], w_rel=spec["w_rel"], w_amb=spec["w_amb"], w_weak=spec["w_weak"],
                    proto_bank=protos,
                    idx_weak_cold=spec.get("idx_weak_cold", None), w_weak_cold=spec.get("w_weak_cold", None),
                    bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE, epochs=THREE_BUCKET_EPOCHS,
                    lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                    lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR,
                    lambda_rel_sup=spec.get("lambda_rel_sup", 1.0),
                    lambda_amb_sup=spec.get("lambda_amb_sup", V6_LAMBDA_AMB_SUP),
                    lambda_weak_sup=spec.get("lambda_weak_sup", V6_LAMBDA_WEAK_SUP),
                    lambda_align=spec.get("lambda_align", V6_LAMBDA_ALIGN),
                    lambda_ent_min=spec.get("lambda_ent_min", V6_LAMBDA_ENT_MIN),
                    lambda_ent_max=spec.get("lambda_ent_max", V6_LAMBDA_ENT_MAX),
                    lambda_cons=spec.get("lambda_cons", V6_LAMBDA_CONS),
                    lambda_proto=spec.get("lambda_proto", V6_LAMBDA_PROTO),
                    lambda_src_proto=spec.get("lambda_src_proto", LAMBDA_SRC_PROTO),
                    lambda_src_logit=spec.get("lambda_src_logit", LAMBDA_SRC_LOGIT),
                    lambda_u_repel=spec.get("lambda_u_repel", V6_LAMBDA_UNKNOWN_REPEL),
                    use_ema=spec.get("use_ema", True),
                    teacher_blend=spec.get("teacher_blend", EMA_TEACHER_BLEND),
                    weak_teacher_blend=spec.get("weak_teacher_blend", V6_WEAK_TEACHER_BLEND),
                    weak_start_epoch=spec.get("weak_start_epoch", V6_WEAK_START_EPOCH),
                    unknown_warmup_epochs=spec.get("unknown_warmup_epochs", V6_UNKNOWN_WARMUP_EPOCHS),
                    unknown_ramp_epochs=spec.get("unknown_ramp_epochs", V6_UNKNOWN_RAMP_EPOCHS))
                extra_method_info[name] = spec.get("bucket_info", {})
            elif spec["train"] == "three_bucket_v7":
                adapters[name], v7_info = adapt_three_bucket_v7(
                    fc_layer=model.fc,
                    Z_replay=Z_replay, y_replay=y_replay,
                    Z_target=Z_adapt,
                    idx_gate=spec["idx_gate"], idx_rel=spec["idx_rel"], idx_amb=spec["idx_amb"],
                    idx_weak=spec["idx_weak"], idx_unk=spec["idx_unk"],
                    q_seed=spec["q_seed"], w_rel=spec["w_rel"], w_amb=spec["w_amb"], w_weak=spec["w_weak"],
                    proto_bank=protos,
                    idx_weak_cold=spec.get("idx_weak_cold", None), w_weak_cold=spec.get("w_weak_cold", None),
                    bucket_score=spec.get("bucket_score", None), unknown_score=spec.get("unknown_score", None),
                    bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE, epochs=THREE_BUCKET_EPOCHS,
                    stage1_epochs=spec.get("stage1_epochs", V7_STAGE1_EPOCHS),
                    lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                    lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR,
                    lambda_rel_sup=spec.get("lambda_rel_sup", 1.0),
                    lambda_amb_sup=spec.get("lambda_amb_sup", V7_LAMBDA_AMB_SUP),
                    lambda_weak_sup=spec.get("lambda_weak_sup", V7_LAMBDA_WEAK_SUP),
                    lambda_align=spec.get("lambda_align", V7_LAMBDA_ALIGN),
                    lambda_ent_min=spec.get("lambda_ent_min", V7_LAMBDA_ENT_MIN),
                    lambda_ent_max=spec.get("lambda_ent_max", V7_LAMBDA_ENT_MAX),
                    lambda_cons=spec.get("lambda_cons", V7_LAMBDA_CONS),
                    lambda_proto=spec.get("lambda_proto", V7_LAMBDA_PROTO),
                    lambda_src_proto=spec.get("lambda_src_proto", LAMBDA_SRC_PROTO * 0.60),
                    lambda_src_logit=spec.get("lambda_src_logit", LAMBDA_SRC_LOGIT * 0.60),
                    lambda_u_repel=spec.get("lambda_u_repel", V7_LAMBDA_UNKNOWN_REPEL),
                    use_ema=spec.get("use_ema", True),
                    teacher_blend=spec.get("teacher_blend", EMA_TEACHER_BLEND),
                    promote_blend=spec.get("promote_blend", V7_PROMOTE_BLEND),
                    promote_rel_fraction=spec.get("promote_rel_fraction", V7_PROMOTE_REL_FRACTION),
                    promote_amb_fraction=spec.get("promote_amb_fraction", V7_PROMOTE_AMB_FRACTION),
                    promote_conf=spec.get("promote_conf", V7_PROMOTE_CONF),
                    promote_margin=spec.get("promote_margin", V7_PROMOTE_MARGIN),
                    unknown_warmup_epochs=spec.get("unknown_warmup_epochs", V7_UNKNOWN_WARMUP_EPOCHS),
                    unknown_ramp_epochs=spec.get("unknown_ramp_epochs", V7_UNKNOWN_RAMP_EPOCHS))
                extra_method_info[name] = {**spec.get("bucket_info", {}), **(v7_info or {})}
            elif spec["train"] == "three_bucket_v8_versioned":
                adapters[name], v8_info = adapt_three_bucket_v8_versioned(
                    fc_layer=model.fc,
                    Z_replay=Z_replay, y_replay=y_replay,
                    Z_target=Z_adapt,
                    idx_gate=spec["idx_gate"], idx_rel=spec["idx_rel"], idx_amb=spec["idx_amb"],
                    idx_weak=spec["idx_weak"], idx_unk=spec["idx_unk"],
                    q_seed=spec["q_seed"], w_rel=spec["w_rel"], w_amb=spec["w_amb"], w_weak=spec["w_weak"],
                    proto_bank=protos,
                    idx_weak_cold=spec.get("idx_weak_cold", None), w_weak_cold=spec.get("w_weak_cold", None),
                    bucket_score=spec.get("bucket_score", None), unknown_score=spec.get("unknown_score", None),
                    Sid_adapt=sc_adapt["Sid"], Sdom_adapt=sc_adapt["Sdom"],
                    bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE,
                    stage0_epochs=V8_STAGE0_EPOCHS, refresh_epochs=V8_REFRESH_EPOCHS, refresh_rounds=V8_REFRESH_ROUNDS,
                    lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                    lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR,
                    lambda_align=spec.get("lambda_align", V8_LAMBDA_ALIGN),
                    lambda_ent_min=spec.get("lambda_ent_min", V8_LAMBDA_ENT_MIN),
                    lambda_ent_max=spec.get("lambda_ent_max", V8_LAMBDA_ENT_MAX),
                    lambda_cons=spec.get("lambda_cons", V8_LAMBDA_CONS),
                    lambda_proto=spec.get("lambda_proto", V8_LAMBDA_PROTO),
                    lambda_u_repel=spec.get("lambda_u_repel", V8_LAMBDA_UNKNOWN_REPEL),
                    version=spec.get("version", "v10"),
                    unknown_loss_mode=spec.get("unknown_loss_mode", UNKNOWN_LOSS_DEFAULT),
                    unknown_energy_margin=spec.get("unknown_energy_margin", UNKNOWN_ENERGY_MARGIN))
                extra_method_info[name] = {**spec.get("bucket_info", {}), **(v8_info or {})}
            else:
                raise ValueError(f"Unknown training mode: {spec['train']}")


        run_tau_conf_sweep = bool(globals().get("RUN_TAU_CONF_SWEEP", False))
        tau_conf_sweep_methods = list(globals().get("TAU_CONF_SWEEP_METHODS", []))
        tau_conf_sweep_quantiles = list(globals().get("TAU_CONF_SWEEP_QUANTILES", []))
        if run_tau_conf_sweep and len(tau_conf_sweep_methods) > 0:
            tau_quantiles = sorted(set([float(CONF_STABLE_Q)] + [float(v) for v in tau_conf_sweep_quantiles]))
            stable_conf_pool = msp_from_probs(P_logit_A)
            fold_tau_rows = []
            for tau_q in tau_quantiles:
                tau_conf_scan = float(np.quantile(stable_conf_pool, tau_q))
                methods_scan, method_info_scan, _ = build_pseudo_method_bank(
                    K=K,
                    Z_adapt=Z_adapt,
                    gate_pred=gate_pred_adapt,
                    y_true_adapt=y_adapt,
                    P_logit_adapt=P_logit_adapt,
                    P_proto_adapt=P_proto_adapt,
                    P_knn_adapt=P_knn_adapt,
                    protos=protos,
                    tau_conf=tau_conf_scan,
                    Sid_adapt=sc_adapt_v8.get("Sid", sc_adapt["Sid"]),
                    Sdom_adapt=sc_adapt_v8.get("Sdrift", sc_adapt["Sdom"]),
                    gate_pred_v4=sc_adapt_v8.get("gate_pred", None),
                    p_known_adapt=sc_adapt_v8.get("p_known", None),
                    p_drift_given_known_adapt=sc_adapt_v8.get("p_drift_given_known", None),
                    P_route_v4=sc_adapt_v8.get("route_probs", None),
                    min_keep=PSEUDO_MIN_KEEP)
                for sweep_name in tau_conf_sweep_methods:
                    if sweep_name not in methods_scan:
                        continue
                    adapter_scan, extra_scan = fit_single_adapter_from_spec(
                        methods_scan[sweep_name],
                        model.fc,
                        Z_replay,
                        y_replay,
                        Z_adapt,
                        protos,
                        sc_adapt=sc_adapt)
                    metrics_scan = evaluate_variant(
                        adapter_scan, model.fc,
                        Z_A, y_src[idx_te], D_A,
                        Z_B_ev, y_B_ev, D_B_ev,
                        Z_C_ev, y_C_ev, D_C_ev,
                        Z_U_ev, D_U_ev,
                        mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids,
                        tau_id, tau_dom, router_bundle=router_bundle_v8
                    )
                    info_scan = dict(method_info_scan.get(sweep_name, {}))
                    info_scan.update(extra_scan or {})
                    fold_tau_rows.append(dict(
                        protocol_tag=protocol_tag,
                        rx_protocol=rx_protocol_tag,
                        tx_protocol=tx_protocol_tag,
                        split_id=int(split_id),
                        fold_id=int(fold),
                        method=sweep_name,
                        tau_conf_q=float(tau_q),
                        tau_conf=float(tau_conf_scan),
                        sel_precision=float(info_scan.get("sel_precision", float("nan"))),
                        sel_recall=float(info_scan.get("sel_recall", float("nan"))),
                        sel_keep=float(info_scan.get("sel_keep_ratio", float("nan"))),
                        unknown_cand_keep=float(info_scan.get("unknown_cand_keep", float("nan"))),
                        unknown_cand_precision=float(info_scan.get("unknown_cand_precision", float("nan"))),
                        drift_acc_all=float(metrics_scan["drift_acc_all"]),
                        drift_cls_acc_all=float(metrics_scan["drift_cls_acc_all"]),
                        FAR_unknown_accept=float(metrics_scan["FAR_unknown_accept"]),
                        FRR_drift_all=float(metrics_scan["FRR_drift_all"]),
                        stable_acc_e2e=float(metrics_scan["stable_acc_e2e"]),
                        FRR_stable=float(metrics_scan["FRR_stable"]),
                        known_acc_e2e=float(metrics_scan["known_acc_e2e"]),
                        FRR_known=float(metrics_scan["FRR_known"])))
            tau_sweep_rows.extend(fold_tau_rows)
            with _safe_open(os.path.join(fold_dir, "tau_conf_sweep_fold.json"), "w", encoding="utf-8") as f:
                json.dump(fold_tau_rows, f, indent=2)

        fold_metrics = dict(
            split=split_id, fold=fold,
            tau_id=tau_id, tau_dom=tau_dom, tau_conf=tau_conf,
            pseudo_logit_rx=_pseudo_metric("logit", "rx"), pseudo_logit_day=_pseudo_metric("logit", "day"), pseudo_logit_all=_pseudo_metric("logit", "all"),
            pseudo_lp_rx=_pseudo_metric("lp", "rx"),       pseudo_lp_day=_pseudo_metric("lp", "day"),       pseudo_lp_all=_pseudo_metric("lp", "all"),
            pseudo_tri_rx=_pseudo_metric("tri", "rx"),     pseudo_tri_day=_pseudo_metric("tri", "day"),     pseudo_tri_all=_pseudo_metric("tri", "all"),
            stageA_cache_hit=bool(stageA.get("cache_hit", False)),
            stageA_seconds=float(stageA.get("stageA_seconds", float("nan"))),
            run_version_tag=RUN_VERSION_TAG,
            run_host_tag=RUN_HOST_TAG,
            code_file_name=CODE_FILE_NAME,
            config_source=str(globals().get("SODA_LAST_APPLIED_CONFIG_SOURCE", _active_soda_cfg.get("config_source", "manual_fixed_config"))),
            selected_profile_tag=str(globals().get("SODA_LAST_APPLIED_PROFILE_TAG", _active_soda_cfg.get("selected_profile_tag", _active_soda_cfg.get("profile_tag", "")))),
            selected_rejector_head=str(globals().get("SODA_LAST_APPLIED_REJECTOR_HEAD", _active_soda_cfg.get("selected_rejector_head", _active_soda_cfg.get("default_rejector_head", "")))),
            selected_snapshot_path=str(globals().get("SODA_LAST_APPLIED_SNAPSHOT_PATH", _active_soda_cfg.get("selected_snapshot_path", SELECTED_VARIANT_JSON_PATH if "SELECTED_VARIANT_JSON_PATH" in globals() else ""))),
            v22_sweep_plan=V22_SWEEP_PLAN)

        for name, info in method_info.items():
            fold_metrics[f"{name}_sel_precision"] = info.get("sel_precision", float("nan"))
            fold_metrics[f"{name}_sel_recall"] = info.get("sel_recall", float("nan"))
            fold_metrics[f"{name}_sel_keep"] = info.get("sel_keep_ratio", float("nan"))
            fold_metrics[f"{name}_pseudo_acc_selected"] = info.get("pseudo_acc_selected", float("nan"))
            fold_metrics[f"{name}_sel_size"] = info.get("sel_size", float("nan"))
            fold_metrics[f"{name}_unknown_cand_keep"] = info.get("unknown_cand_keep", float("nan"))
            fold_metrics[f"{name}_unknown_cand_precision"] = info.get("unknown_cand_precision", float("nan"))
            for k, v in info.items():
                if k not in {"sel_precision","sel_recall","sel_keep_ratio","pseudo_acc_selected","sel_size","unknown_cand_keep","unknown_cand_precision","pseudo_acc"}:
                    fold_metrics[f"{name}_{k}"] = v
        for name, info in extra_method_info.items():
            for k, v in info.items():
                fold_metrics[f"{name}_{k}"] = v

        # v22 variants must export their weighted-entry diagnostics; otherwise this run should not be trusted.
        v22_method_names = [nm for nm in methods.keys() if str(nm).startswith("DG_RAINCOAT_v22WeightedEntryMinimal")]
        for v22_name in v22_method_names:
            required_v22_keys = [
                f"{v22_name}_sum_w_sup",
                f"{v22_name}_sum_w_stab",
                f"{v22_name}_sum_w_unk",
                f"{v22_name}_effective_coverage",
                f"{v22_name}_soft_pseudo_acc"]
            missing_v22_stats = [k for k in required_v22_keys if k not in fold_metrics]
            if len(missing_v22_stats) > 0:
                raise RuntimeError(f"{v22_name} diagnostics missing from fold_metrics: {missing_v22_stats}")

        fold_metrics["m2v8_p_known_mean"] = float(np.mean(sc_adapt_v8.get("p_known", np.zeros((len(y_adapt)), dtype=np.float32))))
        fold_metrics["m2v8_p_drift_mean"] = float(np.mean(sc_adapt_v8.get("p_drift_given_known", np.zeros((len(y_adapt)), dtype=np.float32))))
        fold_metrics["m2v8_route_unknown_mean"] = float(np.mean(sc_adapt_v8.get("p_unknown", np.zeros((len(y_adapt)), dtype=np.float32))))

        # [FIX-3B] If a method is listed in METHOD_ORDER but never instantiated, stop here.
        missing_adapters = [nm for nm in methods.keys() if nm not in adapters]
        if len(missing_adapters) > 0:
            raise RuntimeError(f"Methods missing in adapters dict (would silently fall back to NoAdapt): {missing_adapters}")


        requested_eval_method_order = list(EVAL_METHOD_ORDER) if "EVAL_METHOD_ORDER" in globals() else list(METHOD_ORDER)
        eval_method_order = [nm for nm in requested_eval_method_order if (nm in methods) or (nm in ("NoAdapt", "SourceOnly"))]
        for base_name in eval_method_order:
            if base_name in ("NoAdapt", "SourceOnly"):
                adapter = None
            else:
                adapter = adapters[base_name]
            _router_family = str(method_info.get(base_name, {}).get("router_family", "v8"))
            _router_bundle_eval = None if (_router_family == "v5" or str(base_name).endswith("_M2V5") or str(base_name).startswith("M2V5")) else router_bundle_v8
            eval_cache = build_variant_eval_cache(
                adapter, model.fc,
                Z_tr, y_tr_fold,
                Z_A, D_A,
                Z_B_ev, D_B_ev,
                Z_C_ev, D_C_ev,
                Z_U_ev, D_U_ev,
                mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids,
                tau_id, tau_dom, router_bundle=_router_bundle_eval
            )
            for _head in get_eval_heads_for_base_method(base_name):
                _eval_name = make_eval_method_name(base_name, _head)
                fold_metrics[_eval_name] = evaluate_variant_with_rejector(
                    eval_cache,
                    y_src[idx_te], y_B_ev, y_C_ev,
                    final_rejector_head=_head,
                    frr_target=FINAL_REJECTOR_FRR_TARGET if "FINAL_REJECTOR_FRR_TARGET" in globals() else FRR_TARGET,
                    base_name=base_name
                )
        _safe_makedirs(fold_dir)
        plot_bar_compare({k: fold_metrics[k]["stable_acc_e2e"] for k in eval_method_order}, os.path.join(fold_dir, "stable_acc_e2e_compare.png"), "Stable end-to-end accuracy compare")
        plot_bar_compare({k: fold_metrics[k]["drift_acc_all"] for k in eval_method_order}, os.path.join(fold_dir, "drift_acc_all_compare.png"), "Drift end-to-end accuracy (all) compare")
        plot_bar_compare({k: fold_metrics[k]["drift_cls_acc_all"] for k in eval_method_order}, os.path.join(fold_dir, "drift_cls_acc_all_compare.png"), "Drift classification accuracy (all) compare")
        plot_bar_compare({k: fold_metrics[k]["known_acc_e2e"] for k in eval_method_order}, os.path.join(fold_dir, "known_acc_e2e_compare.png"), "Known end-to-end accuracy compare")
        plot_bar_compare({k: fold_metrics[k]["FAR_unknown_accept"] for k in eval_method_order}, os.path.join(fold_dir, "far_unknown_compare.png"), "Unknown FAR compare")
        plot_bar_compare({k: fold_metrics[k]["FRR_stable"] for k in eval_method_order}, os.path.join(fold_dir, "frr_stable_compare.png"), "Stable FRR compare")
        plot_bar_compare({k: fold_metrics[k]["FRR_drift_all"] for k in eval_method_order}, os.path.join(fold_dir, "frr_drift_compare.png"), "Drift FRR compare")
        plot_bar_compare({k: fold_metrics[k]["FRR_known"] for k in eval_method_order}, os.path.join(fold_dir, "frr_known_compare.png"), "Known FRR compare")

        fold_main_metrics = {}
        for _mn in eval_method_order:
            _m = fold_metrics.get(_mn, {})
            if isinstance(_m, dict):
                fold_main_metrics[_mn] = {
                    "stable_acc_e2e": float(_m.get("stable_acc_e2e", float("nan"))),
                    "FRR_stable": float(_m.get("FRR_stable", float("nan"))),
                    "drift_acc_all": float(_m.get("drift_acc_all", float("nan"))),
                    "drift_cls_acc_all": float(_m.get("drift_cls_acc_all", float("nan"))),
                    "FRR_drift_all": float(_m.get("FRR_drift_all", float("nan"))),
                    "known_acc_e2e": float(_m.get("known_acc_e2e", float("nan"))),
                    "FRR_known": float(_m.get("FRR_known", float("nan"))),
                    "FAR_unknown_accept": float(_m.get("FAR_unknown_accept", float("nan"))),
                    "stable_wrongcls": float(_m.get("stable_wrongcls", float("nan"))),
                    "drift_wrongcls_all": float(_m.get("drift_wrongcls_all", float("nan"))),
                    "known_wrongcls": float(_m.get("known_wrongcls", float("nan"))),
                    "auc_unknown_eval": float(_m.get("auc_unknown_eval", float("nan"))),
                    "auc_su": float(_m.get("auc_su", _m.get("auc_unknown_eval", float("nan")))),
                    "auc_du": float(_m.get("auc_du", float("nan"))),
                    "auc_ku": float(_m.get("auc_ku", float("nan"))),
                }
                with _safe_open(os.path.join(fold_dir, "metrics_module3_adaptation_suite.json"), "w", encoding="utf-8") as f:
                    json.dump(fold_metrics, f, indent=2)
                with _safe_open(os.path.join(fold_dir, "main_metrics_fold.json"), "w", encoding="utf-8") as f:
                    json.dump(fold_main_metrics, f, indent=2)

        fold_metrics["protocol_tag"] = protocol_tag
        fold_metrics["rx_protocol"] = rx_protocol_tag
        fold_metrics["tx_protocol"] = tx_protocol_tag
        fold_metrics["split_id"] = int(split_id)
        fold_metrics["fold_id"] = int(fold)
        fold_metrics["source_rxs"] = list(source_rxs)
        fold_metrics["drift_rx"] = list(drift_rx)
        fold_metrics["known_tx"] = list(KNOWN_TX)
        fold_metrics["unknown_tx"] = list(UNKNOWN_TX)

        rows.append(fold_metrics)
        log_aliases = {
            SODA_METHOD_NAME: "SODA-RFF",
            "OVANet_recon": "OVANet",
            "COMET_recon": "COMET",
            "PCPD_recon": "PCPD",
            "JRFFP_SC_recon": "JRFFP-SC",
            "SourceOnly": "SourceOnly",
        }
        for _nm in eval_method_order:
            _base_nm, _head_nm = parse_eval_method_name(_nm) if "parse_eval_method_name" in globals() else (str(_nm), "module2")
            _head_suffix = "" if str(_head_nm).lower() == get_default_rejector_head(_base_nm) else f"[{str(_head_nm).upper()}]"
            _base_tag = log_aliases.get(_base_nm, _base_nm)
            log_aliases[_nm] = f"{_base_tag}{_head_suffix}"

        # Live fold-end logging for the six main end-to-end metrics.
        # These are the primary model-selection indicators during training:
        #   FRR_stable, FRR_drift_all, FRR_known, FAR_unknown_accept,
        #   stable_acc_e2e, drift_acc_all.
        active_parts = []
        for method_name in eval_method_order:
            if method_name in fold_metrics and isinstance(fold_metrics[method_name], dict) and "drift_acc_all" in fold_metrics[method_name]:
                tag = log_aliases.get(method_name, method_name)
                _frr_stable = float(fold_metrics[method_name].get("FRR_stable", float("nan")))
                _frr_drift = float(fold_metrics[method_name].get("FRR_drift_all", float("nan")))
                _frr_known = float(fold_metrics[method_name].get("FRR_known", float("nan")))
                _far_unk = float(fold_metrics[method_name].get("FAR_unknown_accept", float("nan")))
                _stable_e2e = float(fold_metrics[method_name].get("stable_acc_e2e", float("nan")))
                _drift_e2e = float(fold_metrics[method_name].get("drift_acc_all", float("nan")))
                active_parts.append(
                    f"{tag}(FRRs={_frr_stable:.3f},FRRd={_frr_drift:.3f},FRRk={_frr_known:.3f},"
                    f"FARu={_far_unk:.3f},Stb={_stable_e2e:.3f},Drf={_drift_e2e:.3f})"
                )
        print(
            f"[{protocol_tag} | split {split_id} fold {fold}] "
            f"pLogit={pseudo_eval['logit']['all']:.3f} "
            f"pLP={pseudo_eval['lp']['all']:.3f} "
            f"pTri={pseudo_eval['tri']['all']:.3f}"
        )
        print("  " + " | ".join(active_parts))

    _safe_makedirs(split_dir)
    with _safe_open(os.path.join(split_dir, "metrics_all_folds.json"), "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2)
    with _safe_open(os.path.join(split_dir, "tau_conf_sweep_all_folds.json"), "w", encoding="utf-8") as f:
        json.dump(tau_sweep_rows, f, indent=2)
    return rows, tau_sweep_rows


def execute_default_full_suite():
    all_rows_local = []
    all_tau_local = []
    for protocol_idx, ps in enumerate(PROTOCOL_SPECS, start=1):
        protocol_tag = ps["protocol_tag"]
        protocol_dir = os.path.join(RUN_DIR, protocol_tag)
        _safe_makedirs(protocol_dir)
        with _safe_open(os.path.join(protocol_dir, "protocol_spec.json"), "w", encoding="utf-8") as f:
            json.dump({
                "protocol_tag": protocol_tag,
                "rx_protocol": ps["rx_protocol"],
                "tx_protocol": ps["tx_protocol"],
                "source_rx_num": ps["source_rx_num"],
                "drift_rx_num": ps["drift_rx_num"],
                "known_tx_num": ps["known_tx_num"],
                "unknown_tx_num": ps["unknown_tx_num"],
                "source_rxs": ps["source_rxs"],
                "drift_rx": ps["drift_rx"],
                "tx_splits": ps["tx_splits"],
            }, f, indent=2)

        log_banner(f"PROTOCOL {protocol_idx}/{len(PROTOCOL_SPECS)}: {protocol_tag}")
        print("RX protocol:", ps["rx_protocol"], "| SOURCE_RXS:", ps["source_rxs"], "| DRIFT_RX:", ps["drift_rx"])
        print("TX protocol:", ps["tx_protocol"])
        for split_pos, tx_split in enumerate(ps["tx_splits"], start=1):
            if isinstance(tx_split, dict):
                split_id = int(tx_split.get("split_id", split_pos))
                KNOWN_TX = list(tx_split["known"])
                UNKNOWN_TX = list(tx_split["unknown"])
            else:
                split_id = split_pos
                KNOWN_TX, UNKNOWN_TX = tx_split
                KNOWN_TX = list(KNOWN_TX)
                UNKNOWN_TX = list(UNKNOWN_TX)
            print("\n=== TX split", split_id, "===")
            print("KNOWN_TX:", KNOWN_TX)
            print("UNKNOWN_TX:", UNKNOWN_TX)
            _rows, _tau_rows = run_one_split(
                protocol_tag=protocol_tag,
                rx_protocol_tag=ps["rx_protocol"],
                tx_protocol_tag=ps["tx_protocol"],
                split_id=split_id,
                KNOWN_TX=KNOWN_TX,
                UNKNOWN_TX=UNKNOWN_TX,
                source_rxs=ps["source_rxs"],
                drift_rx=ps["drift_rx"],
                protocol_dir=protocol_dir)
            all_rows_local.extend(_rows)
            all_tau_local.extend(_tau_rows)
    return all_rows_local, all_tau_local



In [15]:
def save_json_safe(path, obj):
    ensure_parent_dir(path)
    with _safe_open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)

def execute_protocol_specs_minimal(protocol_specs, run_subdir="fixed_cleanwin", max_splits=None, max_folds=None):
    global MAX_FOLDS_PER_PROTOCOL
    rows_all, tau_all = [], []
    base_dir = RUN_DIR if not run_subdir else os.path.join(RUN_DIR, run_subdir)
    _safe_makedirs(base_dir)

    old_max_folds = MAX_FOLDS_PER_PROTOCOL
    MAX_FOLDS_PER_PROTOCOL = max_folds
    try:
        for ps in protocol_specs:
            protocol_dir = os.path.join(base_dir, ps["protocol_tag"])
            _safe_makedirs(protocol_dir)

            tx_splits = list(ps["tx_splits"])
            if max_splits is not None:
                tx_splits = tx_splits[:int(max_splits)]

            for split_id, (KNOWN_TX, UNKNOWN_TX) in enumerate(tx_splits, start=1):
                rows, tau_rows = run_one_split(
                    protocol_tag=ps["protocol_tag"],
                    rx_protocol_tag=ps["rx_protocol"],
                    tx_protocol_tag=ps["tx_protocol"],
                    split_id=split_id,
                    KNOWN_TX=KNOWN_TX,
                    UNKNOWN_TX=UNKNOWN_TX,
                    source_rxs=ps["source_rxs"],
                    drift_rx=ps["drift_rx"],
                    protocol_dir=protocol_dir,
                    stagea_cache_protocol_dir=os.path.join(base_dir, "_stagea_cache", ps["protocol_tag"]),
                )
                rows_all.extend(rows)
                tau_all.extend(tau_rows)
    finally:
        MAX_FOLDS_PER_PROTOCOL = old_max_folds

    return rows_all, tau_all

def summarize_rows_by_protocol(rows_subset, method_order=None):
    method_order = list(method_order or EVAL_METHOD_ORDER)
    out = {}
    if not rows_subset:
        return out

    protocol_tags = sorted({str(r.get("protocol_tag", "UNKNOWN")) for r in rows_subset})
    metric_keys = [
        "stable_acc_e2e", "drift_acc_all", "known_acc_e2e", "known_acc", "unk_rej_acc",
        "FRR_stable", "FRR_drift_all", "FRR_known", "FAR_unknown_accept",
        "auc_unknown_eval", "auc_su", "auc_du", "auc_ku", "H_KU", "H_DU",
    ]
    for ptag in protocol_tags:
        p_rows = [r for r in rows_subset if str(r.get("protocol_tag", "UNKNOWN")) == ptag]
        variants = {}
        for m in method_order:
            vals = {k: [] for k in metric_keys}
            for rr in p_rows:
                if m in rr and isinstance(rr[m], dict):
                    for k in metric_keys:
                        if k in rr[m]:
                            vals[k].append(float(rr[m][k]))
            if any(len(v) > 0 for v in vals.values()):
                variants[m] = {}
                for k, arr in vals.items():
                    if arr:
                        variants[m][k] = {
                            "mean": float(np.mean(arr)),
                            "std": float(np.std(arr)),
                        }
        out[ptag] = {"variants": variants}
    return out

def run_minimal_smoke_test(protocol_combo=("9-3", "4-2"), max_splits=1, max_folds=1):
    specs = build_protocol_specs(
        TX_USE, RX_USE,
        SELECTED_RX_PROTOCOLS, SELECTED_TX_PROTOCOLS,
        RX_PROTOCOL_LIBRARY, TX_PROTOCOL_LIBRARY,
        tx_split_repeats=TX_SPLIT_REPEATS,
        seed=SEED + 777,
        explicit_protocol_combos=[protocol_combo],
    )
    rows, tau_rows = execute_protocol_specs_minimal(
        specs,
        run_subdir="smoke_test",
        max_splits=max_splits,
        max_folds=max_folds,
    )
    summary = summarize_rows_by_protocol(rows)
    save_json_safe(os.path.join(RUN_DIR, "smoke_test_summary.json"), summary)
    return rows, tau_rows, summary

# Example:
# smoke_rows, smoke_tau, smoke_summary = run_minimal_smoke_test()


def validate_runtime_entrypoints():
    checks = {}
    checks["has_run_one_split"] = callable(globals().get("run_one_split", None))
    checks["has_build_protocol_specs"] = callable(globals().get("build_protocol_specs", None))
    checks["has_run_selected_pipeline"] = True
    checks["execution_mode_valid"] = str(EXECUTION_MODE).strip().lower() in {"smoke", "full"}
    checks["smoke_combo_valid"] = isinstance(SMOKE_PROTOCOL_COMBO, tuple) and len(SMOKE_PROTOCOL_COMBO) == 2
    missing = [k for k, ok in checks.items() if not ok]
    if missing:
        raise RuntimeError(f"Runtime entry validation failed: {missing}")
    return checks


def run_selected_pipeline(execution_mode=None):
    _entry_checks = validate_runtime_entrypoints()
    mode = str(execution_mode or EXECUTION_MODE).strip().lower()
    if mode not in {"smoke", "full"}:
        raise ValueError(f"Unsupported EXECUTION_MODE={mode!r}. Use 'smoke' or 'full'.")

    if mode == "smoke":
        rows, tau_rows, summary = run_minimal_smoke_test(
            protocol_combo=SMOKE_PROTOCOL_COMBO,
            max_splits=SMOKE_MAX_SPLITS,
            max_folds=SMOKE_MAX_FOLDS,
        )
        print("\nExecuted minimal smoke test only.")
        print("Protocol combo:", SMOKE_PROTOCOL_COMBO, "| max_splits:", SMOKE_MAX_SPLITS, "| max_folds:", SMOKE_MAX_FOLDS)
        return rows, tau_rows, summary

    # In the minimal SODA-RFF notebook, full mode always means the fixed four-protocol suite.
    # Legacy v34 protocol-tuning entrypoints are intentionally removed.
    rows, tau_rows = execute_default_full_suite()

    save_json_safe(os.path.join(RUN_DIR, "metrics_all_splits_folds.json"), rows)
    summary = summarize_rows_by_protocol(rows)
    save_json_safe(os.path.join(RUN_DIR, "summary_by_protocol.json"), summary)
    print("\nExecuted full run.")
    print("Saved all metrics to:", RUN_DIR)
    return rows, tau_rows, summary

# Legacy auto-run entry removed. The fixed-FAR bundle entrypoint is defined in the appended cells below.



## Fixed-FAR operating-point experiment

This extension adds an **equal-FAR diagnostic experiment** that keeps the trained model/adaptation parameters fixed and **only sweeps the final accept/reject threshold**.

Design choices:
- The fixed-FAR threshold is matched on the **combined unknown evaluation pool** used by the final-stage FAR metric.
- The threshold search is **post-hoc**: it does **not** retrain the backbone, router, or adapter parameters.
- The notebook exports:
  - per-fold raw fixed-FAR rows,
  - protocol-level and overall summary tables,
  - CSV / XLSX workbooks,
  - curve plots for the main metrics versus FAR target.

This is an **operating-point comparison**. It is suitable for answering whether a method remains better when all methods are aligned to the same unknown acceptance risk.


In [16]:

import os, json, math
from pathlib import Path

import pandas as pd

# ============================================================
# Fixed-FAR post-hoc operating-point sweep
# ============================================================

RUN_POINT_METRICS_BEFORE_FIXED_FAR = False
RUN_FIXED_FAR_EXPERIMENT = True
FIXED_FAR_RUN_SUBDIR = "fixed_far_bundle"

FIXED_FAR_TARGETS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
FIXED_FAR_REFERENCE_FRR_TARGET = float(globals().get("FINAL_REJECTOR_FRR_TARGET", globals().get("FRR_TARGET", 0.05)))
FIXED_FAR_TARGET_SOURCE = "combined_unknown_eval_oracle"
FIXED_FAR_METHODS = list(METHOD_ORDER)
FIXED_FAR_EXPORT_XLSX = True
FIXED_FAR_EXPORT_JSON = True
FIXED_FAR_SAVE_PROTOCOL_PLOTS = True
FIXED_FAR_SAVE_OVERALL_PLOTS = True
FIXED_FAR_SAVE_FOLD_ROWS = True
FIXED_FAR_PLOT_METRICS = [
    "drift_acc_all",
    "known_acc_e2e",
    "stable_acc_e2e",
    "H_DU",
    "H_KU",
    "FRR_known",
    "FRR_drift_all",
]
FIXED_FAR_METRIC_KEYS = [
    "stable_cls_acc", "stable_acc_e2e", "stable_wrongcls",
    "drift_cls_acc_rx", "drift_cls_acc_day", "drift_cls_acc_all",
    "drift_acc_rx", "drift_acc_day", "drift_acc_all",
    "drift_wrongcls_rx", "drift_wrongcls_day", "drift_wrongcls_all",
    "known_cls_acc", "known_acc_e2e", "known_wrongcls",
    "FRR_stable", "FRR_drift_rx", "FRR_drift_day", "FRR_drift_all", "FRR_known",
    "FAR_unknown_accept", "unk_rej_acc", "H_KU", "H_DU", "auc_unknown_eval", "auc_su", "auc_du", "auc_ku",
    "n_stable", "n_drift_rx", "n_drift_day", "n_unknown",
]
FIXED_FAR_KEY_PIVOT_METRICS = [
    "drift_acc_all", "known_acc_e2e", "stable_acc_e2e",
    "FRR_known", "FRR_drift_all", "FAR_unknown_accept", "H_KU", "H_DU", "auc_unknown_eval", "auc_su", "auc_du", "auc_ku",
]


def _ff_slug(x):
    try:
        return f"{float(x):.2f}".replace(".", "p")
    except Exception:
        return str(x).replace(".", "p")


def _safe_mean_std(values):
    arr = np.asarray([float(v) for v in values if pd.notnull(v)], dtype=np.float64)
    if arr.size == 0:
        return float("nan"), float("nan")
    return float(arr.mean()), float(arr.std())


def _fixed_far_accept_from_margin(margin, bias):
    margin = np.asarray(margin, dtype=np.float32).reshape(-1)
    return (margin <= float(bias))


def _evaluate_from_accept_and_pred(pred_A, pred_B, pred_C, accept_A, accept_B, accept_C, accept_U,
                                   margin_A, margin_B, margin_C, margin_U, y_A, y_B, y_C):
    pred_A = np.asarray(pred_A, dtype=np.int64)
    pred_B = np.asarray(pred_B, dtype=np.int64)
    pred_C = np.asarray(pred_C, dtype=np.int64)

    accept_A = np.asarray(accept_A, dtype=bool)
    accept_B = np.asarray(accept_B, dtype=bool)
    accept_C = np.asarray(accept_C, dtype=bool)
    accept_U = np.asarray(accept_U, dtype=bool)

    y_A = np.asarray(y_A, dtype=np.int64)
    y_B = np.asarray(y_B, dtype=np.int64)
    y_C = np.asarray(y_C, dtype=np.int64)

    stable_cls_acc = float(np.mean(pred_A == y_A))
    stable_acc_e2e = float(np.mean(accept_A & (pred_A == y_A)))
    stable_wrongcls = float(np.mean(accept_A & (pred_A != y_A)))
    FRR_stable = float(np.mean(~accept_A))

    drift_cls_acc_rx = float(np.mean(pred_B == y_B))
    drift_cls_acc_day = float(np.mean(pred_C == y_C))
    drift_cls_acc_all = float((np.sum(pred_B == y_B) + np.sum(pred_C == y_C)) / max(1, len(y_B) + len(y_C)))

    drift_acc_rx = float(np.mean(accept_B & (pred_B == y_B)))
    drift_acc_day = float(np.mean(accept_C & (pred_C == y_C)))
    drift_acc_all = float((np.sum(accept_B & (pred_B == y_B)) + np.sum(accept_C & (pred_C == y_C))) / max(1, len(y_B) + len(y_C)))

    drift_wrongcls_rx = float(np.mean(accept_B & (pred_B != y_B)))
    drift_wrongcls_day = float(np.mean(accept_C & (pred_C != y_C)))
    drift_wrongcls_all = float((np.sum(accept_B & (pred_B != y_B)) + np.sum(accept_C & (pred_C != y_C))) / max(1, len(y_B) + len(y_C)))

    FRR_drift_rx = float(np.mean(~accept_B))
    FRR_drift_day = float(np.mean(~accept_C))
    FRR_drift_all = float((np.sum(~accept_B) + np.sum(~accept_C)) / max(1, len(y_B) + len(y_C)))

    n_known = int(len(y_A) + len(y_B) + len(y_C))
    known_cls_acc = float((np.sum(pred_A == y_A) + np.sum(pred_B == y_B) + np.sum(pred_C == y_C)) / max(1, n_known))
    known_acc_e2e = float((np.sum(accept_A & (pred_A == y_A)) + np.sum(accept_B & (pred_B == y_B)) + np.sum(accept_C & (pred_C == y_C))) / max(1, n_known))
    known_wrongcls = float((np.sum(accept_A & (pred_A != y_A)) + np.sum(accept_B & (pred_B != y_B)) + np.sum(accept_C & (pred_C != y_C))) / max(1, n_known))
    FRR_known = float((np.sum(~accept_A) + np.sum(~accept_B) + np.sum(~accept_C)) / max(1, n_known))

    FAR_unknown_accept = float(np.mean(accept_U))
    unk_rej_acc = float(np.clip(1.0 - FAR_unknown_accept, 0.0, 1.0))

    auc_metrics = compute_known_unknown_auc_metrics(margin_A, margin_B, margin_C, margin_U)
    auc_unknown_eval = float(auc_metrics["auc_unknown_eval"])
    auc_su = float(auc_metrics["auc_su"])
    auc_du = float(auc_metrics["auc_du"])
    auc_ku = float(auc_metrics["auc_ku"])

    H_KU = harmonic_mean_np(known_acc_e2e, unk_rej_acc)
    H_DU = harmonic_mean_np(drift_acc_all, unk_rej_acc)

    return dict(
        stable_cls_acc=stable_cls_acc,
        stable_acc_e2e=stable_acc_e2e,
        stable_wrongcls=stable_wrongcls,
        drift_cls_acc_rx=drift_cls_acc_rx,
        drift_cls_acc_day=drift_cls_acc_day,
        drift_cls_acc_all=drift_cls_acc_all,
        drift_acc_rx=drift_acc_rx,
        drift_acc_day=drift_acc_day,
        drift_acc_all=drift_acc_all,
        drift_wrongcls_rx=drift_wrongcls_rx,
        drift_wrongcls_day=drift_wrongcls_day,
        drift_wrongcls_all=drift_wrongcls_all,
        known_cls_acc=known_cls_acc,
        known_acc_e2e=known_acc_e2e,
        known_wrongcls=known_wrongcls,
        FRR_stable=FRR_stable,
        FRR_drift_rx=FRR_drift_rx,
        FRR_drift_day=FRR_drift_day,
        FRR_drift_all=FRR_drift_all,
        FRR_known=FRR_known,
        FAR_unknown_accept=FAR_unknown_accept,
        unk_rej_acc=unk_rej_acc,
        H_KU=H_KU,
        H_DU=H_DU,
        auc_unknown_eval=auc_unknown_eval,
        auc_su=auc_su,
        auc_du=auc_du,
        auc_ku=auc_ku,
        n_stable=int(len(y_A)),
        n_drift_rx=int(len(y_B)),
        n_drift_day=int(len(y_C)),
        n_unknown=int(len(accept_U)),
    )

def _build_fixed_far_operating_bundle(eval_cache, final_rejector_head="energy", base_name=None, ref_frr_target=0.05):
    head = str(final_rejector_head).lower()
    base_name = str(base_name or "")

    y_tr = np.asarray(eval_cache["y_tr"], dtype=np.int64)
    K_eval = int(eval_cache["logits_tr2"].shape[1])

    logits_A2 = np.asarray(eval_cache["logits_A2"], dtype=np.float32)
    logits_B2 = np.asarray(eval_cache["logits_B2"], dtype=np.float32)
    logits_C2 = np.asarray(eval_cache["logits_C2"], dtype=np.float32)
    logits_U2 = np.asarray(eval_cache["logits_U2"], dtype=np.float32)

    pred_argmax_A = np.argmax(logits_A2, axis=1).astype(np.int64)
    pred_argmax_B = np.argmax(logits_B2, axis=1).astype(np.int64)
    pred_argmax_C = np.argmax(logits_C2, axis=1).astype(np.int64)
    pred_argmax_U = np.argmax(logits_U2, axis=1).astype(np.int64)

    def _out(mA, mB, mC, mU, pred_A=None, pred_B=None, pred_C=None, score_name=None, family=None, base_threshold=np.nan):
        return dict(
            margin_A=np.asarray(mA, dtype=np.float32),
            margin_B=np.asarray(mB, dtype=np.float32),
            margin_C=np.asarray(mC, dtype=np.float32),
            margin_U=np.asarray(mU, dtype=np.float32),
            pred_A=np.asarray(pred_A if pred_A is not None else pred_argmax_A, dtype=np.int64),
            pred_B=np.asarray(pred_B if pred_B is not None else pred_argmax_B, dtype=np.int64),
            pred_C=np.asarray(pred_C if pred_C is not None else pred_argmax_C, dtype=np.int64),
            pred_U=np.asarray(pred_argmax_U, dtype=np.int64),
            score_name=str(score_name or head),
            family=str(family or head),
            base_threshold=float(base_threshold),
        )

    if head == "native":
        family = str(BASELINE_NATIVE_REJECTOR_FAMILY.get(base_name, "")).lower()
        if family == "comet":
            score_A = normalized_entropy_from_logits(logits_A2)
            score_B = normalized_entropy_from_logits(logits_B2)
            score_C = normalized_entropy_from_logits(logits_C2)
            score_U = normalized_entropy_from_logits(logits_U2)
            thr = float(COMET_NATIVE_ENTROPY_THRESHOLD)
            return _out(score_A - thr, score_B - thr, score_C - thr, score_U - thr,
                        score_name="comet_native_entropy", family="comet_native", base_threshold=thr)

        if family == "ovanet":
            P_A = softmax_np(logits_A2); P_B = softmax_np(logits_B2); P_C = softmax_np(logits_C2); P_U = softmax_np(logits_U2)
            pred_A = pred_argmax_A; pred_B = pred_argmax_B; pred_C = pred_argmax_C; pred_U = pred_argmax_U
            thr_global = _safe_quantile(eval_cache["trueprob_tr2"], ref_frr_target, fallback=0.0)
            thr_by_class = build_classwise_thresholds(eval_cache["trueprob_tr2"], y_tr, K_eval, q=ref_frr_target, fallback=thr_global)
            score_A = rowwise_pick(P_A, pred_A, fallback=0.0); score_B = rowwise_pick(P_B, pred_B, fallback=0.0)
            score_C = rowwise_pick(P_C, pred_C, fallback=0.0); score_U = rowwise_pick(P_U, pred_U, fallback=0.0)
            mA = thr_by_class[pred_A] - score_A
            mB = thr_by_class[pred_B] - score_B
            mC = thr_by_class[pred_C] - score_C
            mU = thr_by_class[pred_U] - score_U
            return _out(mA, mB, mC, mU, pred_A=pred_A, pred_B=pred_B, pred_C=pred_C,
                        score_name="ovanet_native_predclass_prob", family="ovanet_native", base_threshold=float(np.mean(thr_by_class)))

        if family == "pcpd":
            thr_global = _safe_quantile(eval_cache["proto_true_dist_tr2"], 1.0 - ref_frr_target, fallback=np.inf)
            thr_by_class = build_classwise_thresholds(eval_cache["proto_true_dist_tr2"], y_tr, K_eval, q=1.0 - ref_frr_target, fallback=thr_global)
            pred_A, dist_A = nearest_proto_distance_and_pred(eval_cache["Z_A2"], eval_cache["protos_eval"], l2norm=True)
            pred_B, dist_B = nearest_proto_distance_and_pred(eval_cache["Z_B2"], eval_cache["protos_eval"], l2norm=True)
            pred_C, dist_C = nearest_proto_distance_and_pred(eval_cache["Z_C2"], eval_cache["protos_eval"], l2norm=True)
            pred_U, dist_U = nearest_proto_distance_and_pred(eval_cache["Z_U2"], eval_cache["protos_eval"], l2norm=True)
            mA = dist_A - thr_by_class[pred_A]
            mB = dist_B - thr_by_class[pred_B]
            mC = dist_C - thr_by_class[pred_C]
            mU = dist_U - thr_by_class[pred_U]
            return _out(mA, mB, mC, mU, pred_A=pred_A, pred_B=pred_B, pred_C=pred_C,
                        score_name="pcpd_native_proto_l2", family="pcpd_native", base_threshold=float(np.mean(thr_by_class)))

        if family == "jrffp_sc":
            thr_global = _safe_quantile(eval_cache["proto_true_dist_tr2"], 1.0 - ref_frr_target, fallback=np.inf)
            thr_by_class = build_classwise_thresholds(eval_cache["proto_true_dist_tr2"], y_tr, K_eval, q=1.0 - ref_frr_target, fallback=thr_global)
            pred_A = pred_argmax_A; pred_B = pred_argmax_B; pred_C = pred_argmax_C; pred_U = pred_argmax_U
            dist_A = l2_to_selected_prototypes(eval_cache["Z_A2"], pred_A, eval_cache["protos_eval"], l2norm=True)
            dist_B = l2_to_selected_prototypes(eval_cache["Z_B2"], pred_B, eval_cache["protos_eval"], l2norm=True)
            dist_C = l2_to_selected_prototypes(eval_cache["Z_C2"], pred_C, eval_cache["protos_eval"], l2norm=True)
            dist_U = l2_to_selected_prototypes(eval_cache["Z_U2"], pred_U, eval_cache["protos_eval"], l2norm=True)
            mA = dist_A - thr_by_class[pred_A]
            mB = dist_B - thr_by_class[pred_B]
            mC = dist_C - thr_by_class[pred_C]
            mU = dist_U - thr_by_class[pred_U]
            return _out(mA, mB, mC, mU, pred_A=pred_A, pred_B=pred_B, pred_C=pred_C,
                        score_name="jrffpsc_native_predclass_proto_l2", family="jrffp_sc_native", base_threshold=float(np.mean(thr_by_class)))

        raise ValueError(f"Unsupported native fixed-FAR family for base method {base_name!r}: {family!r}")

    if head == "energy":
        score_A = logsumexp_np(logits_A2); score_B = logsumexp_np(logits_B2); score_C = logsumexp_np(logits_C2); score_U = logsumexp_np(logits_U2)
        thr = _safe_quantile(eval_cache["energy_tr2"], ref_frr_target, fallback=-1e6)
        return _out(thr - score_A, thr - score_B, thr - score_C, thr - score_U,
                    score_name="energy", family="generic_energy", base_threshold=thr)

    if head == "energy_class":
        pred_A = pred_argmax_A; pred_B = pred_argmax_B; pred_C = pred_argmax_C; pred_U = pred_argmax_U
        score_A = logsumexp_np(logits_A2); score_B = logsumexp_np(logits_B2); score_C = logsumexp_np(logits_C2); score_U = logsumexp_np(logits_U2)
        thr_global = _safe_quantile(eval_cache["energy_tr2"], ref_frr_target, fallback=-1e6)
        thr_by_class = build_classwise_thresholds(eval_cache["energy_tr2"], y_tr, K_eval, q=ref_frr_target, fallback=thr_global)
        return _out(
            thr_by_class[pred_A] - score_A,
            thr_by_class[pred_B] - score_B,
            thr_by_class[pred_C] - score_C,
            thr_by_class[pred_U] - score_U,
            pred_A=pred_A, pred_B=pred_B, pred_C=pred_C,
            score_name="energy_class", family="energy_class", base_threshold=float(np.mean(thr_by_class))
        )

    if head == "energy_state":
        score_A = logsumexp_np(logits_A2); score_B = logsumexp_np(logits_B2); score_C = logsumexp_np(logits_C2); score_U = logsumexp_np(logits_U2)
        thr = _safe_quantile(score_A, ref_frr_target, fallback=-1e6)
        delta = max(float(np.std(eval_cache["energy_tr2"]) * float(ENERGY_STATE_SHIFT_STD)), float(ENERGY_STATE_SHIFT_MINABS))
        scale = max(float(ENERGY_STATE_UNKNOWN_SCALE), 1e-6)
        thr_state = np.asarray([thr + delta, thr - delta, thr + delta / scale], dtype=np.float32)
        m2 = eval_cache["module2"]

        def _state_index(sc_dict, n_default, fill_unknown=False):
            if not isinstance(sc_dict, dict):
                return np.full((n_default,), 2 if fill_unknown else 0, dtype=np.int64)
            if "route_probs" in sc_dict and sc_dict["route_probs"] is not None:
                return np.argmax(np.asarray(sc_dict["route_probs"], dtype=np.float32), axis=1).astype(np.int64)
            if "gate_pred" in sc_dict and sc_dict["gate_pred"] is not None:
                return np.asarray(sc_dict["gate_pred"], dtype=np.int64).reshape(-1)
            return np.full((n_default,), 2 if fill_unknown else 0, dtype=np.int64)

        sA = np.clip(_state_index(m2.get("sc_A", {}), len(score_A), False), 0, 2)
        sB = np.clip(_state_index(m2.get("sc_B", {}), len(score_B), False), 0, 2)
        sC = np.clip(_state_index(m2.get("sc_C", {}), len(score_C), False), 0, 2)
        sU = np.clip(_state_index(m2.get("sc_U", {}), len(score_U), True), 0, 2)

        return _out(
            thr_state[sA] - score_A,
            thr_state[sB] - score_B,
            thr_state[sC] - score_C,
            thr_state[sU] - score_U,
            score_name="energy_state", family="energy_state", base_threshold=thr
        )

    if head == "msp":
        score_A = msp_from_logits(logits_A2); score_B = msp_from_logits(logits_B2); score_C = msp_from_logits(logits_C2); score_U = msp_from_logits(logits_U2)
        thr = _safe_quantile(score_A, ref_frr_target, fallback=0.0)
        return _out(thr - score_A, thr - score_B, thr - score_C, thr - score_U,
                    score_name="msp", family="generic_msp", base_threshold=thr)

    if head == "entropy":
        score_A = normalized_entropy_from_logits(logits_A2); score_B = normalized_entropy_from_logits(logits_B2)
        score_C = normalized_entropy_from_logits(logits_C2); score_U = normalized_entropy_from_logits(logits_U2)
        thr = _safe_quantile(score_A, 1.0 - ref_frr_target, fallback=1.0)
        return _out(score_A - thr, score_B - thr, score_C - thr, score_U - thr,
                    score_name="entropy", family="generic_entropy", base_threshold=thr)

    if head == "margin":
        score_A = top1_top2_margin_from_logits(logits_A2); score_B = top1_top2_margin_from_logits(logits_B2)
        score_C = top1_top2_margin_from_logits(logits_C2); score_U = top1_top2_margin_from_logits(logits_U2)
        thr = _safe_quantile(score_A, ref_frr_target, fallback=0.0)
        return _out(thr - score_A, thr - score_B, thr - score_C, thr - score_U,
                    score_name="margin", family="generic_margin", base_threshold=thr)

    if head == "proto":
        score_A = max_proto_similarity_from_Z(eval_cache["Z_A2"], eval_cache["protos_eval"])
        score_B = max_proto_similarity_from_Z(eval_cache["Z_B2"], eval_cache["protos_eval"])
        score_C = max_proto_similarity_from_Z(eval_cache["Z_C2"], eval_cache["protos_eval"])
        score_U = max_proto_similarity_from_Z(eval_cache["Z_U2"], eval_cache["protos_eval"])
        thr = _safe_quantile(score_A, ref_frr_target, fallback=-1.0)
        return _out(thr - score_A, thr - score_B, thr - score_C, thr - score_U,
                    score_name="proto_maxcos", family="generic_proto", base_threshold=thr)

    raise ValueError(f"Unsupported fixed-FAR rejector head: {head}")


def evaluate_variant_at_fixed_far(eval_cache, y_A, y_B, y_C, final_rejector_head="energy", far_target=0.10, base_name=None, ref_frr_target=0.05):
    bundle = _build_fixed_far_operating_bundle(
        eval_cache,
        final_rejector_head=final_rejector_head,
        base_name=base_name,
        ref_frr_target=ref_frr_target,
    )
    far_target = float(np.clip(far_target, 0.0, 1.0))
    margin_U = np.asarray(bundle["margin_U"], dtype=np.float32)
    if margin_U.size == 0:
        bias = float("inf")
    elif far_target <= 0.0:
        bias = float(np.min(margin_U) - 1e-6)
    elif far_target >= 1.0:
        bias = float(np.max(margin_U) + 1e-6)
    else:
        bias = float(np.quantile(margin_U, far_target))

    accept_A = _fixed_far_accept_from_margin(bundle["margin_A"], bias)
    accept_B = _fixed_far_accept_from_margin(bundle["margin_B"], bias)
    accept_C = _fixed_far_accept_from_margin(bundle["margin_C"], bias)
    accept_U = _fixed_far_accept_from_margin(bundle["margin_U"], bias)

    out = _evaluate_from_accept_and_pred(
        bundle["pred_A"], bundle["pred_B"], bundle["pred_C"],
        accept_A, accept_B, accept_C, accept_U,
        bundle["margin_A"], bundle["margin_B"], bundle["margin_C"], bundle["margin_U"],
        y_A, y_B, y_C
    )
    out.update(dict(
        far_target=float(far_target),
        far_realized=float(out["FAR_unknown_accept"]),
        fixed_far_bias=float(bias),
        final_rejector_head=str(final_rejector_head).lower(),
        final_rejector_base_method=str(base_name) if base_name is not None else "unknown",
        final_rejector_score_name=str(bundle["score_name"]),
        final_rejector_family_used=str(bundle["family"]),
        final_rejector_reference_threshold=float(bundle["base_threshold"]),
        fixed_far_target_source=str(FIXED_FAR_TARGET_SOURCE),
    ))
    return out


def _rebuild_compare_methods_from_stageA(stageA, model, protocol_tag):
    K = int(model.fc.out_features)
    Z_adapt = stageA["Z_adapt"]
    y_adapt = stageA["y_adapt"]
    P_logit_adapt = stageA["P_logit_adapt"]
    P_proto_adapt = stageA["P_proto_adapt"]
    P_knn_adapt = stageA["P_knn_adapt"]
    protos = stageA["protos"]
    tau_conf = stageA["tau_conf"]
    sc_adapt = stageA["sc_adapt"]
    sc_adapt_v8 = stageA["sc_adapt_v8"]
    gate_pred_adapt = stageA["gate_pred_adapt"]

    methods, method_info, aux = build_pseudo_method_bank(
        K=K,
        Z_adapt=Z_adapt,
        gate_pred=gate_pred_adapt,
        y_true_adapt=y_adapt,
        P_logit_adapt=P_logit_adapt,
        P_proto_adapt=P_proto_adapt,
        P_knn_adapt=P_knn_adapt,
        protos=protos,
        tau_conf=tau_conf,
        Sid_adapt=sc_adapt_v8.get("Sid", sc_adapt["Sid"]),
        Sdom_adapt=sc_adapt_v8.get("Sdrift", sc_adapt["Sdom"]),
        gate_pred_v4=sc_adapt_v8.get("gate_pred", None),
        p_known_adapt=sc_adapt_v8.get("p_known", None),
        p_drift_given_known_adapt=sc_adapt_v8.get("p_drift_given_known", None),
        P_route_v4=sc_adapt_v8.get("route_probs", None),
        min_keep=PSEUDO_MIN_KEEP)

    dgr13_spec, dgr13_info = build_label_quality_method_from_base(
        methods["DG_RAINCOAT_v12"],
        mode="dgr",
        fc_layer=model.fc,
        Z_adapt=Z_adapt,
        P_logit_adapt=P_logit_adapt,
        P_proto_adapt=P_proto_adapt,
        P_knn_adapt=P_knn_adapt,
        protos=protos)
    if dgr13_spec is not None:
        methods["DG_RAINCOAT_v13LQ"] = dgr13_spec
        _base_info_dgr = dict(method_info.get("DG_RAINCOAT_v12", {}))
        method_info["DG_RAINCOAT_v13LQ"] = {**_base_info_dgr, **dgr13_spec.get("bucket_info", {}), **(dgr13_info or {})}

    dgr14_spec, dgr14_info = build_label_quality_method_from_base_v14(
        methods["DG_RAINCOAT_v12"],
        mode="dgr",
        fc_layer=model.fc,
        Z_adapt=Z_adapt,
        P_logit_adapt=P_logit_adapt,
        P_proto_adapt=P_proto_adapt,
        P_knn_adapt=P_knn_adapt,
        protos=protos,
        base_name="DG_RAINCOAT_v12")
    if dgr14_spec is not None:
        methods["DG_RAINCOAT_v14Stable"] = dgr14_spec
        _base_info_dgr14 = dict(method_info.get("DG_RAINCOAT_v12", {}))
        method_info["DG_RAINCOAT_v14Stable"] = {**_base_info_dgr14, **dgr14_spec.get("bucket_info", {}), **(dgr14_info or {})}

    dgr15_spec, dgr15_info = build_quality_aware_method_from_base_v15(
        methods.get("DG_RAINCOAT_v14Stable", methods.get("DG_RAINCOAT_v12")),
        mode="dgr",
        fc_layer=model.fc,
        Z_adapt=Z_adapt,
        P_logit_adapt=P_logit_adapt,
        P_proto_adapt=P_proto_adapt,
        P_knn_adapt=P_knn_adapt,
        protos=protos,
        base_name="DG_RAINCOAT_v14Stable")
    if dgr15_spec is not None:
        methods["DG_RAINCOAT_v15QW"] = dgr15_spec
        _base_info_dgr15 = dict(method_info.get("DG_RAINCOAT_v14Stable", method_info.get("DG_RAINCOAT_v12", {})))
        method_info["DG_RAINCOAT_v15QW"] = {**_base_info_dgr15, **dgr15_spec.get("bucket_info", {}), **(dgr15_info or {})}

    dgr16_spec, dgr16_info = build_quality_aware_method_from_base_v16(
        methods.get("DG_RAINCOAT_v15QW", methods.get("DG_RAINCOAT_v14Stable", methods.get("DG_RAINCOAT_v12"))),
        mode="dgr",
        fc_layer=model.fc,
        Z_adapt=Z_adapt,
        P_logit_adapt=P_logit_adapt,
        P_proto_adapt=P_proto_adapt,
        P_knn_adapt=P_knn_adapt,
        protos=protos,
        base_name="DG_RAINCOAT_v15QW")
    if dgr16_spec is not None:
        methods["DG_RAINCOAT_v16QW"] = dgr16_spec
        _base_info_dgr16 = dict(method_info.get("DG_RAINCOAT_v15QW", method_info.get("DG_RAINCOAT_v14Stable", method_info.get("DG_RAINCOAT_v12", {}))))
        method_info["DG_RAINCOAT_v16QW"] = {**_base_info_dgr16, **dgr16_spec.get("bucket_info", {}), **(dgr16_info or {})}

    dgr17_spec, dgr17_info = build_quality_aware_method_from_base_v17(
        methods.get("DG_RAINCOAT_v16QW", methods.get("DG_RAINCOAT_v15QW", methods.get("DG_RAINCOAT_v14Stable", methods.get("DG_RAINCOAT_v12")))),
        mode="dgr",
        fc_layer=model.fc,
        Z_adapt=Z_adapt,
        P_logit_adapt=P_logit_adapt,
        P_proto_adapt=P_proto_adapt,
        P_knn_adapt=P_knn_adapt,
        protos=protos,
        base_name="DG_RAINCOAT_v16QW")
    if dgr17_spec is not None:
        methods["DG_RAINCOAT_v17LCQ"] = dgr17_spec
        _base_info_dgr17 = dict(method_info.get("DG_RAINCOAT_v16QW", method_info.get("DG_RAINCOAT_v15QW", method_info.get("DG_RAINCOAT_v14Stable", method_info.get("DG_RAINCOAT_v12", {})))))
        method_info["DG_RAINCOAT_v17LCQ"] = {**_base_info_dgr17, **dgr17_spec.get("bucket_info", {}), **(dgr17_info or {})}
    elif "DG_RAINCOAT_v16QW" in methods:
        methods["DG_RAINCOAT_v17LCQ"] = deepcopy(methods["DG_RAINCOAT_v16QW"])
        method_info["DG_RAINCOAT_v17LCQ"] = {**dict(method_info.get("DG_RAINCOAT_v16QW", {})), "v17_lcq_active": False, "v17_lcq_reason": "builder_returned_none"}

    dgr19_spec, dgr19_info = build_m2v4_weighted_guided_method(
        methods.get("DG_RAINCOAT_v17LCQ", methods.get("DG_RAINCOAT_v16QW", methods.get("DG_RAINCOAT_v15QW", methods.get("DG_RAINCOAT_v14Stable", methods.get("DG_RAINCOAT_v12"))))),
        mode="dgr",
        Z_adapt=Z_adapt,
        p_known_adapt=sc_adapt_v8.get("p_known", np.ones((len(Z_adapt)), dtype=np.float32)),
        p_drift_given_known_adapt=sc_adapt_v8.get("p_drift_given_known", np.ones((len(Z_adapt)), dtype=np.float32) * 0.5),
        base_name="DG_RAINCOAT_v17LCQ")
    if dgr19_spec is not None:
        methods["DG_RAINCOAT_v19M2W"] = dgr19_spec
        _base_info_dgr19 = dict(method_info.get("DG_RAINCOAT_v17LCQ", method_info.get("DG_RAINCOAT_v16QW", method_info.get("DG_RAINCOAT_v15QW", {}))))
        method_info["DG_RAINCOAT_v19M2W"] = {**_base_info_dgr19, **dgr19_spec.get("bucket_info", {}), **(dgr19_info or {})}
    elif "DG_RAINCOAT_v17LCQ" in methods:
        methods["DG_RAINCOAT_v19M2W"] = deepcopy(methods["DG_RAINCOAT_v17LCQ"])
        method_info["DG_RAINCOAT_v19M2W"] = {**dict(method_info.get("DG_RAINCOAT_v17LCQ", {})), "v19_m2w_active": False, "v19_m2w_reason": "builder_returned_none"}

    base_v19_name = "DG_RAINCOAT_v19M2W"
    base_v19_spec = methods.get(base_v19_name, None)
    if base_v19_spec is not None:
        if isinstance(aux, dict) and aux.get("P_refine", None) is not None:
            P_target_soda = aux.get("P_refine")
            soda_target_source = "aux.P_refine"
        elif isinstance(base_v19_spec, dict) and base_v19_spec.get("P", None) is not None:
            P_target_soda = base_v19_spec.get("P")
            soda_target_source = f"{base_v19_name}.P"
        else:
            P_target_soda = P_logit_adapt
            soda_target_source = "P_logit_adapt"
        P_target_soda = normalize_rows(np.asarray(P_target_soda, dtype=np.float32))

        soda_overrides = {}
        if "get_soda_protocol_fixed_config" in globals():
            _soda_cfg_protocol = get_soda_protocol_fixed_config(protocol_tag)
            for _kk in [
                "v36_proxy_cap", "v36_proxy_q", "v36_proxy_pu_min", "v36_proxy_conf_max",
                "v36_proxy_conf_smooth", "v36_proxy_entropy_mix", "v36_lambda_ent_max_scale",
                "v36_proxy_min_keep"
            ]:
                if _kk in _soda_cfg_protocol:
                    soda_overrides[_kk] = _soda_cfg_protocol[_kk]
        else:
            soda_overrides = dict(SODA_DEFAULT_OVERRIDES)
            soda_overrides.update(SODA_PROTOCOL_OVERRIDES.get(str(protocol_tag), {}))

        soda_spec, soda_extra = build_v36_unknown_proxy_from_base(
            base_v19_spec,
            Z_adapt=Z_adapt,
            p_known_adapt=sc_adapt_v8.get("p_known", np.ones((len(Z_adapt)), dtype=np.float32)),
            p_drift_given_known_adapt=sc_adapt_v8.get("p_drift_given_known", np.ones((len(Z_adapt)), dtype=np.float32) * 0.5),
            P_route_v4=sc_adapt_v8.get("route_probs", None),
            P_target=P_target_soda,
            tau_conf=tau_conf,
            base_name=base_v19_name,
            overrides=soda_overrides,
            method_name=SODA_METHOD_NAME
        )
        if soda_spec is None:
            method_info[SODA_METHOD_NAME] = {
                "soda_active": False,
                "soda_reason": "builder_returned_none",
                "soda_method_name": SODA_METHOD_NAME,
            }
        else:
            methods[SODA_METHOD_NAME] = soda_spec
            method_info[SODA_METHOD_NAME] = build_v36_method_info_from_spec(
                SODA_METHOD_NAME,
                soda_spec,
                y_adapt,
                extra_info=soda_extra,
                base_name=base_v19_name,
                target_source=soda_target_source
            )
            method_info[SODA_METHOD_NAME]["framework_family"] = "SODA-RFF"
            method_info[SODA_METHOD_NAME]["framework_note"] = (
                "State-aware open-set domain adaptation mainline with source-calibrated "
                "known/drift/unknown responsibilities and unknown-proxy recovery."
            )

    if EXPERIMENT_MODE == "ext_compare":
        def _row_norm_np(P):
            P = np.asarray(P, dtype=np.float32)
            P = np.clip(P, 1e-8, None)
            return (P / np.sum(P, axis=1, keepdims=True)).astype(np.float32)

        def _topk_mask(score, frac=0.25, min_n=128, largest=True):
            score = np.asarray(score, dtype=np.float32)
            n = len(score)
            mask = np.zeros((n,), dtype=bool)
            if n == 0:
                return mask
            k = int(max(min_n, round(float(frac) * n)))
            k = max(1, min(n, k))
            order = np.argsort(score)
            idx = order[-k:] if largest else order[:k]
            mask[idx] = True
            return mask

        def _safe_proto_matrix(proto_bank, K, dim):
            proto_mat = np.zeros((K, dim), dtype=np.float32)
            if isinstance(proto_bank, dict):
                for kk, vv in proto_bank.items():
                    kk = int(kk)
                    if 0 <= kk < K:
                        proto_mat[kk] = np.asarray(vv, dtype=np.float32)
            else:
                arr = np.asarray(proto_bank, dtype=np.float32)
                if arr.ndim == 2:
                    upto = min(K, arr.shape[0])
                    proto_mat[:upto] = arr[:upto]
            return normalize_vec_rows_np(proto_mat + 1e-8)

        def _build_generic_spec(name, sup_mask, q=None, y=None, w_score=None, align_mask=None, unrel_mask=None, **cfg_local):
            sup_mask = np.asarray(sup_mask, dtype=bool)
            if sup_mask.sum() == 0:
                fallback_score = w_score if w_score is not None else conf_logit_ext
                sup_mask = _topk_mask(fallback_score, frac=0.15, min_n=min(192, len(Z_adapt)))
            sup_idx = np.where(sup_mask)[0].astype(np.int64)
            spec = dict(train="generic", sup_idx=sup_idx)
            if y is not None:
                spec["y"] = np.asarray(y, dtype=np.int64)[sup_idx]
            else:
                if q is None:
                    q = P_logit_adapt
                spec["q"] = np.asarray(q, dtype=np.float32)[sup_idx]
            if w_score is not None:
                spec["w"] = np.asarray(w_score, dtype=np.float32)[sup_idx]
            if align_mask is not None:
                spec["align_idx"] = np.where(np.asarray(align_mask, dtype=bool))[0].astype(np.int64)
            if unrel_mask is not None:
                spec["unrel_idx"] = np.where(np.asarray(unrel_mask, dtype=bool))[0].astype(np.int64)
            spec.update(cfg_local)
            return spec

        p_known_ext = np.clip(np.asarray(sc_adapt_v8.get("p_known", np.ones((len(Z_adapt)), dtype=np.float32)), dtype=np.float32), 0.0, 1.0)
        if sc_adapt_v8.get("route_probs", None) is not None and np.asarray(sc_adapt_v8["route_probs"]).ndim == 2 and np.asarray(sc_adapt_v8["route_probs"]).shape[1] >= 3:
            p_unk_ext = np.clip(np.asarray(sc_adapt_v8["route_probs"], dtype=np.float32)[:, 2], 0.0, 1.0)
        else:
            p_unk_ext = np.clip(1.0 - p_known_ext, 0.0, 1.0)

        pred_logit_ext = np.argmax(P_logit_adapt, axis=1)
        conf_logit_ext = np.max(P_logit_adapt, axis=1).astype(np.float32)
        margin_logit_ext = top1_top2_margin_from_logits(np.log(np.clip(P_logit_adapt, 1e-8, 1.0)))
        pred_proto_ext = np.argmax(P_proto_adapt, axis=1)
        conf_proto_ext = np.max(P_proto_adapt, axis=1).astype(np.float32)
        pred_knn_ext = np.argmax(P_knn_adapt, axis=1)
        agree_lp_ext = ((pred_logit_ext == pred_proto_ext)).astype(np.float32)
        agree_tri_ext = ((pred_logit_ext == pred_proto_ext) & (pred_logit_ext == pred_knn_ext)).astype(np.float32)

        P_fuse_lp_ext = _row_norm_np(0.55 * P_logit_adapt + 0.45 * P_proto_adapt)
        P_fuse_tri_ext = _row_norm_np((P_logit_adapt + P_proto_adapt + P_knn_adapt) / 3.0)

        proto_mat_ext = _safe_proto_matrix(protos, K, Z_adapt.shape[1])
        Z_adapt_n = normalize_vec_rows_np(Z_adapt)
        dist_pred_ext = np.linalg.norm(Z_adapt_n - proto_mat_ext[pred_logit_ext], axis=1).astype(np.float32)
        dist_score_ext = (1.0 - robust_unit_interval(dist_pred_ext)).astype(np.float32)

        q_known50 = float(np.quantile(p_known_ext, 0.50)) if len(p_known_ext) > 0 else 0.5
        q_known60 = float(np.quantile(p_known_ext, 0.60)) if len(p_known_ext) > 0 else 0.5
        q_unk70 = float(np.quantile(p_unk_ext, 0.70)) if len(p_unk_ext) > 0 else 0.5
        q_unk75 = float(np.quantile(p_unk_ext, 0.75)) if len(p_unk_ext) > 0 else 0.5
        q_margin25 = float(np.quantile(margin_logit_ext, 0.25)) if len(margin_logit_ext) > 0 else 0.0
        q_conf35 = float(np.quantile(conf_logit_ext, 0.35)) if len(conf_logit_ext) > 0 else 0.0
        q_dist75 = float(np.quantile(dist_pred_ext, 0.75)) if len(dist_pred_ext) > 0 else 1.0

        pcpd_score = (
            0.40 * robust_unit_interval(p_known_ext) +
            0.25 * robust_unit_interval(conf_proto_ext) +
            0.20 * agree_lp_ext +
            0.15 * robust_unit_interval(1.0 - p_unk_ext)
        ).astype(np.float32)
        pcpd_sup = _topk_mask(pcpd_score, frac=0.22, min_n=min(224, len(Z_adapt)))
        pcpd_align = p_known_ext >= q_known50
        pcpd_unrel = p_unk_ext >= q_unk70
        methods["PCPD_recon"] = _build_generic_spec(
            "PCPD_recon",
            sup_mask=pcpd_sup,
            q=P_fuse_lp_ext,
            w_score=np.clip(pcpd_score, 1e-3, None),
            align_mask=pcpd_align,
            unrel_mask=pcpd_unrel,
            lambda_sup=1.0,
            lambda_align=0.05,
            lambda_ent_min=0.03,
            lambda_ent_max=0.05,
            lambda_cons=0.0,
            lambda_proto=0.35,
            dynamic_align=True,
            unknown_loss_mode=UNKNOWN_LOSS_DEFAULT,
            unknown_energy_margin=UNKNOWN_ENERGY_MARGIN)
        method_info["PCPD_recon"] = {
            "recon_family": "PCPD",
            "recon_note": "prototype-calibration + pseudo-label + domain-alignment reconstruction",
        }

        ova_score = (
            0.50 * robust_unit_interval(conf_logit_ext) +
            0.25 * robust_unit_interval(margin_logit_ext) +
            0.25 * robust_unit_interval(1.0 - p_unk_ext)
        ).astype(np.float32)
        ova_sup = _topk_mask(ova_score, frac=0.28, min_n=min(256, len(Z_adapt)))
        ova_align = (p_known_ext >= q_known60)
        ova_unrel = (p_unk_ext >= q_unk75) | ((conf_logit_ext <= q_conf35) & (~ova_sup))
        methods["OVANet_recon"] = _build_generic_spec(
            "OVANet_recon",
            sup_mask=ova_sup,
            q=P_logit_adapt,
            w_score=np.clip(ova_score, 1e-3, None),
            align_mask=ova_align,
            unrel_mask=ova_unrel,
            lambda_sup=1.0,
            lambda_align=0.0,
            lambda_ent_min=0.00,
            lambda_ent_max=0.10,
            lambda_cons=0.0,
            lambda_proto=0.0,
            dynamic_align=False,
            unknown_loss_mode=UNKNOWN_LOSS_DEFAULT,
            unknown_energy_margin=UNKNOWN_ENERGY_MARGIN)
        method_info["OVANet_recon"] = {
            "recon_family": "OVANet",
            "recon_note": "one-vs-all style known/unknown boundary reconstruction",
        }

        comet_score = (
            0.35 * robust_unit_interval(conf_logit_ext) +
            0.25 * robust_unit_interval(1.0 - p_unk_ext) +
            0.20 * robust_unit_interval(margin_logit_ext) +
            0.10 * agree_lp_ext +
            0.10 * agree_tri_ext
        ).astype(np.float32)
        comet_sup = _topk_mask(comet_score, frac=0.25, min_n=min(224, len(Z_adapt)))
        comet_align = p_known_ext >= q_known60
        comet_unrel = (p_unk_ext >= q_unk70) | (margin_logit_ext <= q_margin25)
        methods["COMET_recon"] = _build_generic_spec(
            "COMET_recon",
            sup_mask=comet_sup,
            q=P_fuse_tri_ext,
            w_score=np.clip(comet_score, 1e-3, None),
            align_mask=comet_align,
            unrel_mask=comet_unrel,
            lambda_sup=1.0,
            lambda_align=0.02,
            lambda_ent_min=0.03,
            lambda_ent_max=0.08,
            lambda_cons=0.10,
            lambda_proto=0.20,
            dynamic_align=False,
            unknown_loss_mode=UNKNOWN_LOSS_DEFAULT,
            unknown_energy_margin=UNKNOWN_ENERGY_MARGIN)
        method_info["COMET_recon"] = {
            "recon_family": "COMET",
            "recon_note": "mean-teacher + contrastive reconstruction under unified embedding pipeline",
        }

        jsc_score = (
            0.45 * robust_unit_interval(conf_logit_ext) +
            0.35 * dist_score_ext +
            0.20 * agree_lp_ext
        ).astype(np.float32)
        jsc_sup = _topk_mask(jsc_score, frac=0.20, min_n=min(160, len(Z_adapt)))
        jsc_unrel = (dist_pred_ext >= q_dist75) | (p_unk_ext >= q_unk75)
        methods["JRFFP_SC_recon"] = _build_generic_spec(
            "JRFFP_SC_recon",
            sup_mask=jsc_sup,
            y=pred_logit_ext,
            w_score=np.clip(jsc_score, 1e-3, None),
            align_mask=np.zeros((len(Z_adapt)), dtype=bool),
            unrel_mask=jsc_unrel,
            lambda_sup=1.0,
            lambda_align=0.0,
            lambda_ent_min=0.0,
            lambda_ent_max=0.05,
            lambda_cons=0.0,
            lambda_proto=0.55,
            dynamic_align=False,
            unknown_loss_mode=UNKNOWN_LOSS_DEFAULT,
            unknown_energy_margin=UNKNOWN_ENERGY_MARGIN)
        method_info["JRFFP_SC_recon"] = {
            "recon_family": "JRFFP-SC",
            "recon_note": "joint prediction + siamese-comparison reconstruction via class-prototype distance gating",
        }

    methods = {k: v for k, v in methods.items() if k in METHOD_ORDER}
    method_info = {k: v for k, v in method_info.items() if (k in methods) or (k == "SourceOnly")}
    if "SourceOnly" not in method_info:
        method_info["SourceOnly"] = {"framework_family": "SourceOnly", "framework_note": "No target adaptation."}
    return methods, method_info


def _build_fixed_far_fold_context(protocol_tag, split_id, KNOWN_TX, UNKNOWN_TX, source_rxs, drift_rx,
                                  protocol_dir, stagea_cache_protocol_dir=None):
    split_dir = os.path.join(protocol_dir, f"txsplit_{split_id}")
    _safe_makedirs(split_dir)

    cache_protocol_dir = stagea_cache_protocol_dir if stagea_cache_protocol_dir is not None else protocol_dir
    cache_split_dir = os.path.join(cache_protocol_dir, f"txsplit_{split_id}")
    _safe_makedirs(cache_split_dir)

    X_src, y_src, D_src, rx_src = build_source_train(compact_dataset, KNOWN_TX, source_rxs=source_rxs)
    K = len(KNOWN_TX)

    X_B, y_B, D_B = build_eval_set(compact_dataset, KNOWN_TX, KNOWN_TX, drift_rx, TEST_DATES_RX)
    X_C, y_C, D_C = build_eval_set(compact_dataset, KNOWN_TX, KNOWN_TX, source_rxs, TEST_DATES_DAY)
    X_D, y_D, D_D = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, source_rxs, TEST_DATES_RX)
    X_E, y_E, D_E = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, drift_rx, TEST_DATES_RX)
    X_F, y_F, D_F = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, source_rxs, TEST_DATES_DAY)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 1000 * split_id)
    return dict(
        split_dir=split_dir,
        cache_split_dir=cache_split_dir,
        X_src=X_src, y_src=y_src, D_src=D_src, rx_src=rx_src,
        X_B=X_B, y_B=y_B, D_B=D_B,
        X_C=X_C, y_C=y_C, D_C=D_C,
        X_D=X_D, y_D=y_D, D_D=D_D,
        X_E=X_E, y_E=y_E, D_E=D_E,
        X_F=X_F, y_F=y_F, D_F=D_F,
        skf=skf,
        K=K,
    )


def run_fixed_far_one_split(protocol_tag, rx_protocol_tag, tx_protocol_tag, split_id,
                            KNOWN_TX, UNKNOWN_TX, source_rxs, drift_rx,
                            protocol_dir, stagea_cache_protocol_dir=None):
    if "apply_soda_protocol_runtime" in globals():
        _ = apply_soda_protocol_runtime(protocol_tag)

    ctx = _build_fixed_far_fold_context(
        protocol_tag=protocol_tag,
        split_id=split_id,
        KNOWN_TX=KNOWN_TX,
        UNKNOWN_TX=UNKNOWN_TX,
        source_rxs=source_rxs,
        drift_rx=drift_rx,
        protocol_dir=protocol_dir,
        stagea_cache_protocol_dir=stagea_cache_protocol_dir,
    )

    rows = []
    for fold, (idx_tr, idx_te) in enumerate(ctx["skf"].split(ctx["X_src"], ctx["y_src"]), start=1):
        if MAX_FOLDS_PER_PROTOCOL is not None and int(fold) > int(MAX_FOLDS_PER_PROTOCOL):
            break
        fold_dir = os.path.join(ctx["split_dir"], f"fold_{fold}")
        cache_fold_dir = os.path.join(ctx["cache_split_dir"], f"fold_{fold}")
        _safe_makedirs(fold_dir)
        _safe_makedirs(cache_fold_dir)

        rng = np.random.RandomState(SEED + 1000 * split_id + fold)
        perm = rng.permutation(idx_tr)
        n_val = max(1, int(0.1 * len(perm)))
        idx_val = perm[:n_val]
        idx_train = perm[n_val:]

        stageA = build_or_load_stagea_fold(
            fold_dir=cache_fold_dir,
            protocol_tag=protocol_tag,
            split_id=split_id,
            fold=fold,
            K=ctx["K"],
            source_rxs=source_rxs,
            drift_rx=drift_rx,
            X_src=ctx["X_src"], y_src=ctx["y_src"], D_src=ctx["D_src"], rx_src=ctx["rx_src"],
            idx_train=idx_train, idx_val=idx_val, idx_te=idx_te,
            X_B=ctx["X_B"], y_B=ctx["y_B"], D_B=ctx["D_B"],
            X_C=ctx["X_C"], y_C=ctx["y_C"], D_C=ctx["D_C"],
            X_D=ctx["X_D"], y_D=ctx["y_D"], D_D=ctx["D_D"],
            X_E=ctx["X_E"], y_E=ctx["y_E"], D_E=ctx["D_E"],
            X_F=ctx["X_F"], y_F=ctx["y_F"], D_F=ctx["D_F"]
        )

        model = stageA["model"]
        methods, method_info = _rebuild_compare_methods_from_stageA(stageA, model, protocol_tag=protocol_tag)

        y_tr_fold = np.asarray(stageA.get("y_tr_fold", ctx["y_src"][np.asarray(stageA.get("idx_train", idx_train), dtype=np.int64)]), dtype=np.int64)
        Z_tr = stageA["Z_tr"]
        Z_A, D_A = stageA["Z_A"], stageA["D_A"]
        Z_B_ev, y_B_ev, D_B_ev = stageA["Z_B_ev"], stageA["y_B_ev"], stageA["D_B_ev"]
        Z_C_ev, y_C_ev, D_C_ev = stageA["Z_C_ev"], stageA["y_C_ev"], stageA["D_C_ev"]
        Z_U_ev, D_U_ev = stageA["Z_U_ev"], stageA["D_U_ev"]

        mu_z_src, var_z_src = stageA["mu_z_src"], stageA["var_z_src"]
        ref_sid_emb_A, ref_sid_en_A = stageA["ref_sid_emb_A"], stageA["ref_sid_en_A"]
        source_rx_ids = stageA["source_rx_ids"]
        models_kr, fallback_k = stageA["models_kr"], stageA["fallback_k"]
        tau_id, tau_dom = stageA["tau_id"], stageA["tau_dom"]
        Z_replay, y_replay = stageA["Z_replay"], stageA["y_replay"]
        protos = stageA["protos"]
        sc_adapt = stageA["sc_adapt"]
        router_bundle_v8 = stageA["router_bundle_v8"]

        active_methods = [m for m in FIXED_FAR_METHODS if (m == "SourceOnly") or (m in methods)]
        fold_rows = []
        for base_name in active_methods:
            adapter = None
            if base_name != "SourceOnly":
                adapter, _extra = fit_single_adapter_from_spec(
                    methods[base_name],
                    model.fc,
                    Z_replay,
                    y_replay,
                    stageA["Z_adapt"],
                    protos,
                    sc_adapt=sc_adapt
                )
            _router_family = str(method_info.get(base_name, {}).get("router_family", "v8"))
            _router_bundle_eval = None if (_router_family == "v5" or str(base_name).endswith("_M2V5") or str(base_name).startswith("M2V5")) else router_bundle_v8

            eval_cache = build_variant_eval_cache(
                adapter, model.fc,
                Z_tr, y_tr_fold,
                Z_A, D_A,
                Z_B_ev, D_B_ev,
                Z_C_ev, D_C_ev,
                Z_U_ev, D_U_ev,
                mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids,
                tau_id, tau_dom, router_bundle=_router_bundle_eval
            )

            for head in get_eval_heads_for_base_method(base_name):
                eval_name = make_eval_method_name(base_name, head)
                for far_target in FIXED_FAR_TARGETS:
                    metric_row = evaluate_variant_at_fixed_far(
                        eval_cache,
                        ctx["y_src"][idx_te], y_B_ev, y_C_ev,
                        final_rejector_head=head,
                        far_target=far_target,
                        base_name=base_name,
                        ref_frr_target=FIXED_FAR_REFERENCE_FRR_TARGET,
                    )
                    fold_rows.append({
                        "protocol_tag": protocol_tag,
                        "rx_protocol": rx_protocol_tag,
                        "tx_protocol": tx_protocol_tag,
                        "split_id": int(split_id),
                        "fold_id": int(fold),
                        "method": eval_name,
                        "base_method": str(base_name),
                        **metric_row,
                    })

        if FIXED_FAR_SAVE_FOLD_ROWS:
            with _safe_open(os.path.join(fold_dir, "fixed_far_rows.json"), "w", encoding="utf-8") as f:
                json.dump(fold_rows, f, indent=2)
        rows.extend(fold_rows)

    split_path = os.path.join(ctx["split_dir"], "fixed_far_rows_all_folds.json")
    with _safe_open(split_path, "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2)
    return rows


def summarize_fixed_far_rows(rows, metric_keys=None):
    metric_keys = list(metric_keys or FIXED_FAR_METRIC_KEYS)
    if not rows:
        return pd.DataFrame(), pd.DataFrame()

    raw_df = pd.DataFrame(rows)
    group_cols = ["protocol_tag", "method", "far_target"]
    overall_cols = ["method", "far_target"]

    protocol_records = []
    for key, g in raw_df.groupby(group_cols, dropna=False):
        rec = {c: v for c, v in zip(group_cols, key)}
        rec["n_rows"] = int(len(g))
        for mk in metric_keys:
            mean_v, std_v = _safe_mean_std(g[mk].tolist() if mk in g.columns else [])
            rec[f"{mk}_mean"] = mean_v
            rec[f"{mk}_std"] = std_v
        protocol_records.append(rec)

    overall_records = []
    for key, g in raw_df.groupby(overall_cols, dropna=False):
        rec = {c: v for c, v in zip(overall_cols, key)}
        rec["n_rows"] = int(len(g))
        for mk in metric_keys:
            mean_v, std_v = _safe_mean_std(g[mk].tolist() if mk in g.columns else [])
            rec[f"{mk}_mean"] = mean_v
            rec[f"{mk}_std"] = std_v
        overall_records.append(rec)

    protocol_df = pd.DataFrame(protocol_records).sort_values(["protocol_tag", "far_target", "method"]).reset_index(drop=True)
    overall_df = pd.DataFrame(overall_records).sort_values(["far_target", "method"]).reset_index(drop=True)
    return protocol_df, overall_df


def _write_fixed_far_workbook(raw_df, protocol_df, overall_df, out_path):
    with pd.ExcelWriter(_win_safe_path(out_path), engine="openpyxl") as writer:
        raw_df.to_excel(writer, sheet_name="raw_rows", index=False)
        protocol_df.to_excel(writer, sheet_name="protocol_summary", index=False)
        overall_df.to_excel(writer, sheet_name="overall_summary", index=False)
        for metric in FIXED_FAR_KEY_PIVOT_METRICS:
            pcol = f"{metric}_mean"
            if pcol in protocol_df.columns:
                piv = protocol_df.pivot_table(index=["protocol_tag", "far_target"], columns="method", values=pcol)
                piv.to_excel(writer, sheet_name=f"pvt_{metric[:24]}")
        cfg_df = pd.DataFrame([{
            "RUN_POINT_METRICS_BEFORE_FIXED_FAR": RUN_POINT_METRICS_BEFORE_FIXED_FAR,
            "RUN_FIXED_FAR_EXPERIMENT": RUN_FIXED_FAR_EXPERIMENT,
            "FIXED_FAR_REFERENCE_FRR_TARGET": FIXED_FAR_REFERENCE_FRR_TARGET,
            "FIXED_FAR_TARGET_SOURCE": FIXED_FAR_TARGET_SOURCE,
            "FIXED_FAR_TARGETS": json.dumps(FIXED_FAR_TARGETS),
            "FIXED_FAR_METHODS": json.dumps(FIXED_FAR_METHODS),
        }])
        cfg_df.to_excel(writer, sheet_name="fixed_far_config", index=False)


def _plot_fixed_far_curve_df(summary_df, out_path, metric, title, method_order=None):
    method_order = list(method_order or FIXED_FAR_METHODS)
    mean_col = f"{metric}_mean"
    std_col = f"{metric}_std"
    if summary_df.empty or mean_col not in summary_df.columns:
        return

    fig, ax = plt.subplots(figsize=(8, 5))
    plotted = False
    for method in method_order:
        g = summary_df[summary_df["method"] == method].sort_values("far_target")
        if g.empty:
            continue
        x = g["far_target"].to_numpy(dtype=float)
        y = g[mean_col].to_numpy(dtype=float)
        ax.plot(x, y, marker="o", linewidth=1.8, label=str(method))
        if std_col in g.columns:
            s = g[std_col].fillna(0.0).to_numpy(dtype=float)
            ax.fill_between(x, y - s, y + s, alpha=0.15)
        plotted = True
    if not plotted:
        plt.close(fig)
        return
    ax.set_xlabel("Target FAR")
    ax.set_ylabel(metric)
    ax.set_title(title)
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best", fontsize=8)
    fig.tight_layout()
    ensure_parent_dir(out_path)
    fig.savefig(_win_safe_path(out_path), dpi=180, bbox_inches="tight")
    plt.close(fig)


def export_fixed_far_artifacts(rows, base_dir, method_order=None):
    method_order = list(method_order or FIXED_FAR_METHODS)
    raw_df = pd.DataFrame(rows).sort_values(["protocol_tag", "split_id", "fold_id", "far_target", "method"]).reset_index(drop=True)
    protocol_df, overall_df = summarize_fixed_far_rows(rows)

    raw_csv = os.path.join(base_dir, "fixed_far_raw_rows.csv")
    protocol_csv = os.path.join(base_dir, "fixed_far_protocol_summary.csv")
    overall_csv = os.path.join(base_dir, "fixed_far_overall_summary.csv")
    raw_df.to_csv(_win_safe_path(raw_csv), index=False)
    protocol_df.to_csv(_win_safe_path(protocol_csv), index=False)
    overall_df.to_csv(_win_safe_path(overall_csv), index=False)

    xlsx_path = None
    if FIXED_FAR_EXPORT_XLSX:
        xlsx_path = os.path.join(base_dir, "fixed_far_summary_workbook.xlsx")
        _write_fixed_far_workbook(raw_df, protocol_df, overall_df, xlsx_path)

    if FIXED_FAR_EXPORT_JSON:
        with _safe_open(os.path.join(base_dir, "fixed_far_raw_rows.json"), "w", encoding="utf-8") as f:
            json.dump(rows, f, indent=2)
        with _safe_open(os.path.join(base_dir, "fixed_far_protocol_summary.json"), "w", encoding="utf-8") as f:
            json.dump(protocol_df.to_dict(orient="records"), f, indent=2)
        with _safe_open(os.path.join(base_dir, "fixed_far_overall_summary.json"), "w", encoding="utf-8") as f:
            json.dump(overall_df.to_dict(orient="records"), f, indent=2)

    if FIXED_FAR_SAVE_OVERALL_PLOTS:
        for metric in FIXED_FAR_PLOT_METRICS:
            _plot_fixed_far_curve_df(
                overall_df,
                os.path.join(base_dir, f"fixed_far_overall__{metric}.png"),
                metric=metric,
                title=f"Fixed-FAR overall curve: {metric}",
                method_order=method_order,
            )

    if FIXED_FAR_SAVE_PROTOCOL_PLOTS and (not protocol_df.empty):
        for protocol_tag in sorted(protocol_df["protocol_tag"].unique()):
            g = protocol_df[protocol_df["protocol_tag"] == protocol_tag].copy()
            pdir = os.path.join(base_dir, protocol_tag)
            _safe_makedirs(pdir)
            for metric in FIXED_FAR_PLOT_METRICS:
                _plot_fixed_far_curve_df(
                    g,
                    os.path.join(pdir, f"fixed_far__{protocol_tag}__{metric}.png"),
                    metric=metric,
                    title=f"{protocol_tag} fixed-FAR curve: {metric}",
                    method_order=method_order,
                )

    return {
        "raw_df": raw_df,
        "protocol_df": protocol_df,
        "overall_df": overall_df,
        "raw_csv": raw_csv,
        "protocol_csv": protocol_csv,
        "overall_csv": overall_csv,
        "xlsx_path": xlsx_path,
    }


def _select_specs_for_execution_mode(execution_mode=None):
    mode = str(execution_mode or EXECUTION_MODE).strip().lower()
    if mode not in {"smoke", "full"}:
        raise ValueError(f"Unsupported EXECUTION_MODE={mode!r}. Use 'smoke' or 'full'.")
    if mode == "smoke":
        specs = build_protocol_specs(
            TX_USE, RX_USE,
            SELECTED_RX_PROTOCOLS, SELECTED_TX_PROTOCOLS,
            RX_PROTOCOL_LIBRARY, TX_PROTOCOL_LIBRARY,
            tx_split_repeats=TX_SPLIT_REPEATS,
            seed=SEED + 777,
            explicit_protocol_combos=[SMOKE_PROTOCOL_COMBO],
        )
        return mode, specs, SMOKE_MAX_SPLITS, SMOKE_MAX_FOLDS
    return mode, list(PROTOCOL_SPECS), None, None


def run_fixed_far_protocol_specs(protocol_specs, base_dir, max_splits=None, max_folds=None):
    global MAX_FOLDS_PER_PROTOCOL
    old_max = MAX_FOLDS_PER_PROTOCOL
    MAX_FOLDS_PER_PROTOCOL = max_folds
    rows = []
    try:
        for ps in protocol_specs:
            protocol_dir = os.path.join(base_dir, ps["protocol_tag"])
            _safe_makedirs(protocol_dir)
            _strict_protocol_package_log(ps["protocol_tag"])
            tx_splits = list(ps["tx_splits"])
            if max_splits is not None:
                tx_splits = tx_splits[:int(max_splits)]

            for split_id, (KNOWN_TX, UNKNOWN_TX) in enumerate(tx_splits, start=1):
                split_rows = run_fixed_far_one_split(
                    protocol_tag=ps["protocol_tag"],
                    rx_protocol_tag=ps["rx_protocol"],
                    tx_protocol_tag=ps["tx_protocol"],
                    split_id=split_id,
                    KNOWN_TX=list(KNOWN_TX),
                    UNKNOWN_TX=list(UNKNOWN_TX),
                    source_rxs=ps["source_rxs"],
                    drift_rx=ps["drift_rx"],
                    protocol_dir=protocol_dir,
                    stagea_cache_protocol_dir=os.path.join(base_dir, "_stagea_cache", ps["protocol_tag"]),
                )
                rows.extend(split_rows)
    finally:
        MAX_FOLDS_PER_PROTOCOL = old_max
    return rows


def run_fixed_far_bundle(execution_mode=None):
    mode, specs, max_splits, max_folds = _select_specs_for_execution_mode(execution_mode=execution_mode)
    base_dir = os.path.join(RUN_DIR, FIXED_FAR_RUN_SUBDIR)
    _safe_makedirs(base_dir)

    point_rows = []
    point_tau_rows = []
    point_summary = {}
    if RUN_POINT_METRICS_BEFORE_FIXED_FAR:
        point_rows, point_tau_rows = execute_protocol_specs_minimal(
            specs,
            run_subdir=FIXED_FAR_RUN_SUBDIR,
            max_splits=max_splits,
            max_folds=max_folds,
        )
        point_summary = summarize_rows_by_protocol(point_rows)

    fixed_far_rows = []
    fixed_far_artifacts = {}
    if RUN_FIXED_FAR_EXPERIMENT:
        fixed_far_rows = run_fixed_far_protocol_specs(
            specs,
            base_dir=base_dir,
            max_splits=max_splits,
            max_folds=max_folds,
        )
        fixed_far_artifacts = export_fixed_far_artifacts(
            fixed_far_rows,
            base_dir=base_dir,
            method_order=FIXED_FAR_METHODS,
        )

    bundle = {
        "execution_mode": mode,
        "base_dir": base_dir,
        "point_rows": point_rows,
        "point_tau_rows": point_tau_rows,
        "point_summary": point_summary,
        "fixed_far_rows": fixed_far_rows,
        "fixed_far_artifacts": {
            k: (v if not isinstance(v, pd.DataFrame) else v.head(20))
            for k, v in fixed_far_artifacts.items()
        },
    }

    manifest = {
        "execution_mode": mode,
        "base_dir": base_dir,
        "run_point_metrics_before_fixed_far": RUN_POINT_METRICS_BEFORE_FIXED_FAR,
        "run_fixed_far_experiment": RUN_FIXED_FAR_EXPERIMENT,
        "fixed_far_targets": FIXED_FAR_TARGETS,
        "fixed_far_methods": FIXED_FAR_METHODS,
        "fixed_far_reference_frr_target": FIXED_FAR_REFERENCE_FRR_TARGET,
        "fixed_far_target_source": FIXED_FAR_TARGET_SOURCE,
        "n_fixed_far_rows": int(len(fixed_far_rows)),
    }
    with _safe_open(os.path.join(base_dir, "fixed_far_manifest.json"), "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    print("Fixed-FAR bundle finished.")
    print("Mode:", mode)
    print("Base dir:", base_dir)
    print("Fixed-FAR rows:", len(fixed_far_rows))
    return bundle


In [17]:
# ===== Strict paper-reproduction baseline patch =====
import contextlib

# This cell replaces the earlier "inspired/reconstruction" baseline bank with
# paper-faithful adapted implementations under the same RFF benchmark.
#
# Scope:
#   PCPD_strict      : prototype loss + pseudo-label calibration + DANN/GRL.
#   OVANet_strict    : closed classifier + one-vs-all open-set head + HNCS + OEM.
#   COMET_strict     : mean-teacher test-time adaptation + entropy pseudo labels
#                      + prototype contrast/repel losses.
#   JRFFP_SC_strict  : prediction network + learned Siamese comparison gate.
#
# These are still dataset-adapted reproductions: the common ResNet18_1D feature
# extractor and IQ input protocol are intentionally shared for fair RFF testing.


# ===== Direct soft-route core-buffer SODA diagnostic variant =====
def _direct_json_dumps(obj):
    try:
        return json.dumps(_json_safe_v2(obj), ensure_ascii=False, sort_keys=True)
    except Exception:
        return str(obj)


def _direct_norm01(x):
    x = np.asarray(x, dtype=np.float32)
    out = np.zeros_like(x, dtype=np.float32)
    finite = np.isfinite(x)
    if not np.any(finite):
        return out
    lo, hi = float(np.min(x[finite])), float(np.max(x[finite]))
    if abs(hi - lo) < 1e-12:
        out[finite] = 1.0
    else:
        out[finite] = (x[finite] - lo) / (hi - lo + 1e-12)
    return np.clip(out, 0.0, 1.0).astype(np.float32)


def _direct_class_balanced_topk(score, eligible, pseudo_y, keep_frac, min_core_size):
    score = np.asarray(score, dtype=np.float32).reshape(-1)
    eligible = np.asarray(eligible, dtype=bool).reshape(-1)
    pseudo_y = np.asarray(pseudo_y, dtype=np.int64).reshape(-1)
    n = int(len(score))
    idx_eligible = np.where(eligible & np.isfinite(score))[0]
    if len(idx_eligible) == 0:
        return np.zeros((0,), dtype=np.int64)
    k_total = int(max(int(min_core_size), round(float(keep_frac) * max(1, n))))
    k_total = max(1, min(k_total, len(idx_eligible)))
    classes = sorted(np.unique(pseudo_y[idx_eligible]).tolist())
    selected = []
    if classes:
        per_class = int(np.ceil(k_total / max(1, len(classes))))
        for cls in classes:
            idx_c = idx_eligible[pseudo_y[idx_eligible] == int(cls)]
            idx_c = idx_c[np.argsort(score[idx_c])[::-1]]
            selected.extend(idx_c[:per_class].tolist())
    selected = list(dict.fromkeys([int(i) for i in selected]))
    if len(selected) < k_total:
        order = idx_eligible[np.argsort(score[idx_eligible])[::-1]]
        for i in order:
            if int(i) not in selected:
                selected.append(int(i))
            if len(selected) >= k_total:
                break
    return np.asarray(selected[:k_total], dtype=np.int64)


def _direct_global_topk(score, eligible, keep_frac, min_core_size):
    score = np.asarray(score, dtype=np.float32).reshape(-1)
    eligible = np.asarray(eligible, dtype=bool).reshape(-1)
    n = int(len(score))
    idx_eligible = np.where(eligible & np.isfinite(score))[0]
    if len(idx_eligible) == 0:
        return np.zeros((0,), dtype=np.int64)
    k_total = int(max(int(min_core_size), round(float(keep_frac) * max(1, n))))
    k_total = max(1, min(k_total, len(idx_eligible)))
    order = idx_eligible[np.argsort(score[idx_eligible])[::-1]]
    return np.asarray(order[:k_total], dtype=np.int64)


def compute_direct_corebuffer_scores(stageA, P_target_soda=None):
    """Compute direct soft-route core-buffer scores for every raw target adaptation sample."""
    eps = 1e-8
    n = int(len(stageA["Z_adapt"]))
    sc = dict(stageA.get("sc_adapt_v8", {}) or {})
    P_route = sc.get("route_probs", None)
    route_source = "sc_adapt_v8.route_probs"
    if P_route is not None:
        P_route = normalize_rows(np.asarray(P_route, dtype=np.float32))
    if P_route is None or P_route.ndim != 2 or P_route.shape[0] != n or P_route.shape[1] < 3:
        p_known = np.clip(np.asarray(sc.get("p_known", np.ones((n,), dtype=np.float32) * 0.5), dtype=np.float32), 0.0, 1.0)
        p_drift_given_known = np.clip(np.asarray(sc.get("p_drift_given_known", np.ones((n,), dtype=np.float32) * 0.5), dtype=np.float32), 0.0, 1.0)
        pi_s = np.clip(p_known * (1.0 - p_drift_given_known), eps, None)
        pi_d = np.clip(p_known * p_drift_given_known, eps, None)
        pi_u = np.clip(1.0 - p_known, eps, None)
        P_route = normalize_rows(np.stack([pi_s, pi_d, pi_u], axis=1).astype(np.float32))
        route_source = "reconstructed_from_p_known_and_p_drift_given_known"
    pi_s = np.clip(P_route[:, 0], 0.0, 1.0).astype(np.float32)
    pi_d = np.clip(P_route[:, 1], 0.0, 1.0).astype(np.float32)
    pi_u = np.clip(P_route[:, 2], 0.0, 1.0).astype(np.float32)
    p_known = np.clip(pi_s + pi_d, eps, 1.0).astype(np.float32)
    p_drift_given_known = np.clip(pi_d / np.clip(p_known, eps, None), 0.0, 1.0).astype(np.float32)

    if P_target_soda is None:
        parts, weights = [], []
        for key, w in [("P_logit_adapt", 0.50), ("P_proto_adapt", 0.30), ("P_knn_adapt", 0.20)]:
            if key in stageA and stageA[key] is not None:
                parts.append(normalize_rows(np.asarray(stageA[key], dtype=np.float32)))
                weights.append(float(w))
        P_target_soda = weighted_prob_fusion(parts, weights) if parts else softmax_np(np.asarray(stageA["logits_adapt"], dtype=np.float32))
    q = normalize_rows(np.asarray(P_target_soda, dtype=np.float32))
    conf = np.max(q, axis=1).astype(np.float32)
    H_norm = entropy_from_probs_np(q).astype(np.float32)
    q_sorted = np.sort(q, axis=1)
    margin = (q_sorted[:, -1] - q_sorted[:, -2]).astype(np.float32) if q.shape[1] >= 2 else np.ones((n,), dtype=np.float32)
    margin = np.clip(margin, 0.0, 1.0).astype(np.float32)
    pred_parts = []
    for key in ["P_logit_adapt", "P_proto_adapt", "P_knn_adapt"]:
        if key in stageA and stageA[key] is not None:
            P = normalize_rows(np.asarray(stageA[key], dtype=np.float32))
            if P.shape == q.shape:
                pred_parts.append(np.argmax(P, axis=1).astype(np.int64))
    if len(pred_parts) >= 2:
        arr = np.stack(pred_parts, axis=1)
        pseudo = np.argmax(q, axis=1).astype(np.int64)
        agreement = np.mean(arr == pseudo[:, None], axis=1).astype(np.float32)
    else:
        agreement = np.ones((n,), dtype=np.float32)
    agreement = np.clip(agreement, 0.0, 1.0).astype(np.float32)
    s_stable = np.clip(pi_s * conf * margin * agreement * (1.0 - H_norm), 0.0, 1.0).astype(np.float32)
    s_shift = np.clip(pi_d * p_known * conf * margin * agreement * (1.0 - H_norm), 0.0, 1.0).astype(np.float32)
    s_unk = np.clip(pi_u * H_norm * (1.0 - conf) * (1.0 - p_known), 0.0, 1.0).astype(np.float32)
    return dict(
        q=q, pseudo_y=np.argmax(q, axis=1).astype(np.int64), route_source=route_source,
        pi_s=pi_s, pi_d=pi_d, pi_u=pi_u, p_known=p_known, p_drift_given_known=p_drift_given_known,
        conf=conf, H_norm=H_norm, margin=margin, agreement=agreement,
        s_stable=s_stable, s_shift=s_shift, s_unk=s_unk,
        m_stable=(pi_s - np.maximum(pi_d, pi_u)).astype(np.float32),
        m_shift=(pi_d - np.maximum(pi_s, pi_u)).astype(np.float32),
        m_unk=(pi_u - np.maximum(pi_s, pi_d)).astype(np.float32),
    )


def _direct_cap_by_score(idx, score, max_frac, n):
    idx = np.asarray(idx, dtype=np.int64)
    if not bool(globals().get("DIRECT_COREBUFFER_USE_MAX_CAP", False)):
        return idx, False
    cap = int(max(0, round(float(max_frac) * max(1, int(n)))))
    if cap <= 0 or len(idx) <= cap:
        return idx, False
    score = np.asarray(score, dtype=np.float32)
    order = idx[np.argsort(score[idx])[::-1]]
    return np.asarray(order[:cap], dtype=np.int64), True


def _direct_threshold_masks(scores):
    stable_mask = (
        (scores["s_stable"] >= float(DIRECT_COREBUFFER_STABLE_SCORE_MIN)) &
        (scores["m_stable"] >= float(DIRECT_COREBUFFER_STABLE_MARGIN_MIN)) &
        (scores["conf"] >= float(DIRECT_COREBUFFER_STABLE_CONF_MIN)) &
        (scores["H_norm"] <= float(DIRECT_COREBUFFER_STABLE_ENTROPY_MAX))
    )
    shift_mask = (
        (scores["s_shift"] >= float(DIRECT_COREBUFFER_SHIFT_SCORE_MIN)) &
        (scores["m_shift"] >= float(DIRECT_COREBUFFER_SHIFT_MARGIN_MIN)) &
        (scores["p_known"] >= float(DIRECT_COREBUFFER_SHIFT_KNOWN_MIN)) &
        (scores["conf"] >= float(DIRECT_COREBUFFER_SHIFT_CONF_MIN)) &
        (scores["H_norm"] <= float(DIRECT_COREBUFFER_SHIFT_ENTROPY_MAX))
    )
    unknown_mask = (
        (scores["s_unk"] >= float(DIRECT_COREBUFFER_UNK_SCORE_MIN)) &
        (scores["m_unk"] >= float(DIRECT_COREBUFFER_UNK_MARGIN_MIN)) &
        (scores["p_known"] <= float(DIRECT_COREBUFFER_UNK_KNOWN_MAX)) &
        (scores["conf"] <= float(DIRECT_COREBUFFER_UNK_CONF_MAX)) &
        (scores["H_norm"] >= float(DIRECT_COREBUFFER_UNK_ENTROPY_MIN))
    )
    if bool(DIRECT_COREBUFFER_USE_AGREEMENT_MIN):
        stable_mask = stable_mask & (scores["agreement"] >= float(DIRECT_COREBUFFER_STABLE_AGREE_MIN))
        shift_mask = shift_mask & (scores["agreement"] >= float(DIRECT_COREBUFFER_SHIFT_AGREE_MIN))
    return stable_mask.astype(bool), shift_mask.astype(bool), unknown_mask.astype(bool)


def _direct_relax_thresholds_for_smoke(scores):
    relaxed = dict(
        stable_score_min=min(float(DIRECT_COREBUFFER_STABLE_SCORE_MIN), 0.005),
        shift_score_min=min(float(DIRECT_COREBUFFER_SHIFT_SCORE_MIN), 0.005),
        unk_score_min=min(float(DIRECT_COREBUFFER_UNK_SCORE_MIN), 0.005),
        stable_margin_min=min(float(DIRECT_COREBUFFER_STABLE_MARGIN_MIN), -0.05),
        shift_margin_min=min(float(DIRECT_COREBUFFER_SHIFT_MARGIN_MIN), -0.05),
        unk_margin_min=min(float(DIRECT_COREBUFFER_UNK_MARGIN_MIN), -0.05),
        stable_conf_min=min(float(DIRECT_COREBUFFER_STABLE_CONF_MIN), 0.40),
        shift_conf_min=min(float(DIRECT_COREBUFFER_SHIFT_CONF_MIN), 0.35),
        unk_conf_max=max(float(DIRECT_COREBUFFER_UNK_CONF_MAX), 0.80),
        stable_entropy_max=max(float(DIRECT_COREBUFFER_STABLE_ENTROPY_MAX), 0.95),
        shift_entropy_max=max(float(DIRECT_COREBUFFER_SHIFT_ENTROPY_MAX), 0.95),
        unk_entropy_min=min(float(DIRECT_COREBUFFER_UNK_ENTROPY_MIN), 0.10),
        shift_known_min=min(float(DIRECT_COREBUFFER_SHIFT_KNOWN_MIN), 0.20),
        unk_known_max=max(float(DIRECT_COREBUFFER_UNK_KNOWN_MAX), 0.90),
    )
    stable_mask = (
        (scores["s_stable"] >= relaxed["stable_score_min"]) &
        (scores["m_stable"] >= relaxed["stable_margin_min"]) &
        (scores["conf"] >= relaxed["stable_conf_min"]) &
        (scores["H_norm"] <= relaxed["stable_entropy_max"])
    )
    shift_mask = (
        (scores["s_shift"] >= relaxed["shift_score_min"]) &
        (scores["m_shift"] >= relaxed["shift_margin_min"]) &
        (scores["p_known"] >= relaxed["shift_known_min"]) &
        (scores["conf"] >= relaxed["shift_conf_min"]) &
        (scores["H_norm"] <= relaxed["shift_entropy_max"])
    )
    unknown_mask = (
        (scores["s_unk"] >= relaxed["unk_score_min"]) &
        (scores["m_unk"] >= relaxed["unk_margin_min"]) &
        (scores["p_known"] <= relaxed["unk_known_max"]) &
        (scores["conf"] <= relaxed["unk_conf_max"]) &
        (scores["H_norm"] >= relaxed["unk_entropy_min"])
    )
    return stable_mask.astype(bool), shift_mask.astype(bool), unknown_mask.astype(bool), relaxed


def select_direct_corebuffer_subsets(scores):
    n = int(len(scores["s_stable"]))
    mode = str(globals().get("DIRECT_COREBUFFER_SELECTION_MODE", "threshold_first")).strip().lower()
    if mode == "topk_legacy":
        all_mask = np.ones((n,), dtype=bool)
        pseudo_y = scores["pseudo_y"]
        fallback = {"selection_mode": "topk_legacy"}
        def select_one(name, score_key, margin_key, frac, margin_min, allowed, class_balanced=False):
            score = scores[score_key]
            margin = scores[margin_key]
            eligible = np.asarray(allowed, dtype=bool) & (margin >= float(margin_min))
            idx = _direct_class_balanced_topk(score, eligible, pseudo_y, frac, DIRECT_COREBUFFER_MIN_CORE_SIZE) if class_balanced else _direct_global_topk(score, eligible, frac, DIRECT_COREBUFFER_MIN_CORE_SIZE)
            fallback[f"{name}_topk_legacy_used"] = True
            return np.asarray(idx, dtype=np.int64)
        unk_idx = select_one("unknown", "s_unk", "m_unk", DIRECT_COREBUFFER_UNK_KEEP_FRAC, DIRECT_COREBUFFER_UNK_MARGIN_MIN, all_mask, False)
        remain = all_mask.copy(); remain[unk_idx] = False
        shift_idx = select_one("shift", "s_shift", "m_shift", DIRECT_COREBUFFER_SHIFT_KEEP_FRAC, DIRECT_COREBUFFER_SHIFT_MARGIN_MIN, remain, bool(DIRECT_COREBUFFER_CLASS_BALANCED_KNOWN))
        remain[shift_idx] = False
        stable_idx = select_one("stable", "s_stable", "m_stable", DIRECT_COREBUFFER_STABLE_KEEP_FRAC, DIRECT_COREBUFFER_STABLE_MARGIN_MIN, remain, bool(DIRECT_COREBUFFER_CLASS_BALANCED_KNOWN))
        used = np.zeros((n,), dtype=bool); used[unk_idx] = True; used[shift_idx] = True; used[stable_idx] = True
        return dict(stable_idx=np.sort(stable_idx), shift_idx=np.sort(shift_idx), unknown_idx=np.sort(unk_idx), buffer_idx=np.where(~used)[0].astype(np.int64), fallback=fallback)
    if mode != "threshold_first":
        raise ValueError(f"Unsupported DIRECT_COREBUFFER_SELECTION_MODE={mode!r}")

    stable_mask, shift_mask, unknown_mask = _direct_threshold_masks(scores)
    thresholds_relaxed = False
    relaxed_thresholds = {}
    if bool(globals().get("RUN_DIRECT_COREBUFFER_SMOKE", False)) and bool(globals().get("DIRECT_COREBUFFER_SMOKE_RELAX_THRESHOLDS", False)):
        if not (np.any(stable_mask) and np.any(shift_mask) and np.any(unknown_mask)):
            stable_mask, shift_mask, unknown_mask, relaxed_thresholds = _direct_relax_thresholds_for_smoke(scores)
            thresholds_relaxed = True

    overlap_stable_shift = stable_mask & shift_mask
    overlap_stable_unknown = stable_mask & unknown_mask
    overlap_shift_unknown = shift_mask & unknown_mask
    overlap_all_three = stable_mask & shift_mask & unknown_mask
    policy = str(DIRECT_COREBUFFER_CONFLICT_POLICY).strip().lower()
    if policy == "priority_unknown_shift_stable":
        unknown_idx = np.where(unknown_mask)[0].astype(np.int64)
        shift_idx = np.where(shift_mask & ~unknown_mask)[0].astype(np.int64)
        stable_idx = np.where(stable_mask & ~unknown_mask & ~shift_mask)[0].astype(np.int64)
    elif policy == "max_normalized_score":
        norm_s = _direct_norm01(scores["s_stable"])
        norm_d = _direct_norm01(scores["s_shift"])
        norm_u = _direct_norm01(scores["s_unk"])
        stable_idx, shift_idx, unknown_idx = [], [], []
        for i in range(n):
            candidates = []
            if stable_mask[i]: candidates.append(("stable", norm_s[i]))
            if shift_mask[i]: candidates.append(("shift", norm_d[i]))
            if unknown_mask[i]: candidates.append(("unknown", norm_u[i]))
            if not candidates:
                continue
            best = sorted(candidates, key=lambda x: (-float(x[1]), {"unknown": 0, "shift": 1, "stable": 2}[x[0]]))[0][0]
            if best == "unknown": unknown_idx.append(i)
            elif best == "shift": shift_idx.append(i)
            else: stable_idx.append(i)
        stable_idx = np.asarray(stable_idx, dtype=np.int64)
        shift_idx = np.asarray(shift_idx, dtype=np.int64)
        unknown_idx = np.asarray(unknown_idx, dtype=np.int64)
    else:
        raise ValueError(f"Unsupported DIRECT_COREBUFFER_CONFLICT_POLICY={policy!r}")

    stable_idx, stable_capped = _direct_cap_by_score(stable_idx, scores["s_stable"], DIRECT_COREBUFFER_STABLE_MAX_FRAC, n)
    shift_idx, shift_capped = _direct_cap_by_score(shift_idx, scores["s_shift"], DIRECT_COREBUFFER_SHIFT_MAX_FRAC, n)
    unknown_idx, unknown_capped = _direct_cap_by_score(unknown_idx, scores["s_unk"], DIRECT_COREBUFFER_UNK_MAX_FRAC, n)
    used = np.zeros((n,), dtype=bool)
    used[stable_idx] = True; used[shift_idx] = True; used[unknown_idx] = True
    buffer_idx = np.where(~used)[0].astype(np.int64)
    if not bool(globals().get("RUN_DIRECT_COREBUFFER_SMOKE", False)):
        assert not thresholds_relaxed, "Threshold relaxation is forbidden outside smoke mode."
    fallback = dict(
        selection_mode="threshold_first",
        conflict_policy=policy,
        raw_stable_mask_count=int(np.sum(stable_mask)),
        raw_shift_mask_count=int(np.sum(shift_mask)),
        raw_unknown_mask_count=int(np.sum(unknown_mask)),
        overlap_stable_shift_count=int(np.sum(overlap_stable_shift)),
        overlap_stable_unknown_count=int(np.sum(overlap_stable_unknown)),
        overlap_shift_unknown_count=int(np.sum(overlap_shift_unknown)),
        overlap_all_three_count=int(np.sum(overlap_all_three)),
        conflict_assigned_by_priority_count=int(np.sum(overlap_stable_shift | overlap_stable_unknown | overlap_shift_unknown)),
        thresholds_relaxed_for_smoke=bool(thresholds_relaxed),
        relaxed_thresholds_json=_direct_json_dumps(relaxed_thresholds),
        topk_fallback_used=False,
        forced_min_core_size_used=False,
        max_cap_enabled=bool(DIRECT_COREBUFFER_USE_MAX_CAP),
        stable_max_cap_applied=bool(stable_capped),
        shift_max_cap_applied=bool(shift_capped),
        unknown_max_cap_applied=bool(unknown_capped),
        stable_core_empty=bool(len(stable_idx) == 0),
        shift_core_empty=bool(len(shift_idx) == 0),
        unknown_core_empty=bool(len(unknown_idx) == 0),
    )
    return dict(stable_idx=np.sort(stable_idx).astype(np.int64), shift_idx=np.sort(shift_idx).astype(np.int64), unknown_idx=np.sort(unknown_idx).astype(np.int64), buffer_idx=buffer_idx, fallback=fallback)


def _direct_subset_stats(idx, scores, y_adapt=None):
    idx = np.asarray(idx, dtype=np.int64)
    out = {"count": int(len(idx))}
    for key in ["pi_s", "pi_d", "pi_u", "p_known", "p_drift_given_known", "conf", "H_norm", "margin", "s_stable", "s_shift", "s_unk"]:
        vals = np.asarray(scores[key], dtype=np.float32)[idx] if len(idx) else np.asarray([], dtype=np.float32)
        vals = vals[np.isfinite(vals)]
        out[f"{key}_mean"] = float(np.mean(vals)) if vals.size else float("nan")
        out[f"{key}_std"] = float(np.std(vals)) if vals.size else float("nan")
    pseudo_y = np.asarray(scores["pseudo_y"], dtype=np.int64)[idx] if len(idx) else np.asarray([], dtype=np.int64)
    out["pseudo_class_distribution_json"] = _direct_json_dumps({int(k): int(np.sum(pseudo_y == k)) for k in sorted(np.unique(pseudo_y).tolist())}) if len(pseudo_y) else "{}"
    # Oracle target labels are used only for post-hoc diagnostics, never for adaptation, selection, thresholding, or tuning.
    if y_adapt is not None and len(idx):
        y = np.asarray(y_adapt, dtype=np.int64)[idx]
        out["oracle_known_frac"] = float(np.mean(y >= 0))
        known = y >= 0
        out["oracle_pseudo_acc_known"] = float(np.mean(pseudo_y[known] == y[known])) if np.any(known) else float("nan")
        out["oracle_unknown_frac"] = float(np.mean(y < 0))
    else:
        out["oracle_known_frac"] = float("nan")
        out["oracle_pseudo_acc_known"] = float("nan")
        out["oracle_unknown_frac"] = float("nan")
    return out


def make_soda_direct_corebuffer_spec(stageA, protocol_tag=None, base_soda_spec=None, base_soda_info=None):
    if not bool(globals().get("DIRECT_COREBUFFER_ENABLE", True)):
        return None, {"direct_corebuffer_active": False, "direct_corebuffer_reason": "disabled"}
    if base_soda_spec is None:
        return None, {"direct_corebuffer_active": False, "direct_corebuffer_reason": "base_soda_spec_missing"}
    p_source = str(globals().get("DIRECT_COREBUFFER_P_SOURCE", "base_soda")).strip().lower()
    if p_source == "base_soda":
        P_target_soda = np.asarray(base_soda_spec.get("P", None), dtype=np.float32) if base_soda_spec.get("P", None) is not None else None
    elif p_source == "direct_fusion":
        P_target_soda = normalize_rows(0.50 * normalize_rows(stageA["P_logit_adapt"]) + 0.30 * normalize_rows(stageA["P_proto_adapt"]) + 0.20 * normalize_rows(stageA["P_knn_adapt"]))
    elif p_source == "logit_only":
        P_target_soda = normalize_rows(stageA["P_logit_adapt"])
    else:
        raise ValueError(f"Unsupported DIRECT_COREBUFFER_P_SOURCE={p_source!r}")
    scores = compute_direct_corebuffer_scores(stageA, P_target_soda=P_target_soda)
    subsets = select_direct_corebuffer_subsets(scores)
    stable_idx = subsets["stable_idx"]
    shift_idx = subsets["shift_idx"]
    unknown_idx = subsets["unknown_idx"]
    buffer_idx = subsets["buffer_idx"]
    q = scores["q"]
    if str(DIRECT_COREBUFFER_SUP_WEIGHT_MODE).strip().lower() == "score" and len(stable_idx):
        w = np.clip(0.6 + 0.4 * _direct_norm01(scores["s_stable"][stable_idx]), 0.6, 1.0).astype(np.float32)
    else:
        w = np.ones((len(stable_idx),), dtype=np.float32)
    spec = deepcopy(base_soda_spec)
    spec["train"] = "lq_soft"
    spec["method_name"] = str(DIRECT_COREBUFFER_METHOD_NAME)
    spec["sup_idx"] = stable_idx
    spec["idx_sup_eval"] = stable_idx
    spec["align_idx"] = shift_idx
    spec["unrel_idx"] = unknown_idx
    spec["idx_unk_eval"] = unknown_idx
    spec["q"] = q[stable_idx]
    spec["w"] = w
    spec["P"] = q
    spec.pop("fallback_name", None)
    n = int(len(scores["s_stable"]))
    sets = {"stable": set(stable_idx.tolist()), "shift": set(shift_idx.tolist()), "unknown": set(unknown_idx.tolist()), "buffer": set(buffer_idx.tolist())}
    disjoint = dict(
        stable_shift_empty=len(sets["stable"] & sets["shift"]) == 0,
        stable_unknown_empty=len(sets["stable"] & sets["unknown"]) == 0,
        shift_unknown_empty=len(sets["shift"] & sets["unknown"]) == 0,
        buffer_disjoint=len(sets["buffer"] & (sets["stable"] | sets["shift"] | sets["unknown"])) == 0,
    )
    diag = dict(
        direct_corebuffer_active=True,
        direct_corebuffer_method=str(DIRECT_COREBUFFER_METHOD_NAME),
        direct_corebuffer_protocol_tag=str(protocol_tag),
        n_target_adapt=int(n),
        n_stable_core=int(len(stable_idx)),
        n_shift_core=int(len(shift_idx)),
        n_unknown_core=int(len(unknown_idx)),
        n_buffer=int(len(buffer_idx)),
        frac_stable_core=float(len(stable_idx) / max(1, n)),
        frac_shift_core=float(len(shift_idx) / max(1, n)),
        frac_unknown_core=float(len(unknown_idx) / max(1, n)),
        frac_buffer=float(len(buffer_idx) / max(1, n)),
        selection_mode=str(DIRECT_COREBUFFER_SELECTION_MODE),
        conflict_policy=str(DIRECT_COREBUFFER_CONFLICT_POLICY),
        p_source=str(DIRECT_COREBUFFER_P_SOURCE),
        stable_keep_frac=float(DIRECT_COREBUFFER_STABLE_KEEP_FRAC),
        shift_keep_frac=float(DIRECT_COREBUFFER_SHIFT_KEEP_FRAC),
        unknown_keep_frac=float(DIRECT_COREBUFFER_UNK_KEEP_FRAC),
        stable_score_min=float(DIRECT_COREBUFFER_STABLE_SCORE_MIN),
        shift_score_min=float(DIRECT_COREBUFFER_SHIFT_SCORE_MIN),
        unknown_score_min=float(DIRECT_COREBUFFER_UNK_SCORE_MIN),
        stable_margin_min=float(DIRECT_COREBUFFER_STABLE_MARGIN_MIN),
        shift_margin_min=float(DIRECT_COREBUFFER_SHIFT_MARGIN_MIN),
        unknown_margin_min=float(DIRECT_COREBUFFER_UNK_MARGIN_MIN),
        stable_conf_min=float(DIRECT_COREBUFFER_STABLE_CONF_MIN),
        shift_conf_min=float(DIRECT_COREBUFFER_SHIFT_CONF_MIN),
        unknown_conf_max=float(DIRECT_COREBUFFER_UNK_CONF_MAX),
        stable_entropy_max=float(DIRECT_COREBUFFER_STABLE_ENTROPY_MAX),
        shift_entropy_max=float(DIRECT_COREBUFFER_SHIFT_ENTROPY_MAX),
        unknown_entropy_min=float(DIRECT_COREBUFFER_UNK_ENTROPY_MIN),
        shift_known_min=float(DIRECT_COREBUFFER_SHIFT_KNOWN_MIN),
        unknown_known_max=float(DIRECT_COREBUFFER_UNK_KNOWN_MAX),
        use_agreement_min=bool(DIRECT_COREBUFFER_USE_AGREEMENT_MIN),
        stable_agree_min=float(DIRECT_COREBUFFER_STABLE_AGREE_MIN),
        shift_agree_min=float(DIRECT_COREBUFFER_SHIFT_AGREE_MIN),
        use_max_cap=bool(DIRECT_COREBUFFER_USE_MAX_CAP),
        class_balanced_known=bool(DIRECT_COREBUFFER_CLASS_BALANCED_KNOWN),
        buffer_mode=str(DIRECT_COREBUFFER_BUFFER_MODE),
        sup_weight_mode=str(DIRECT_COREBUFFER_SUP_WEIGHT_MODE),
        min_core_size=int(DIRECT_COREBUFFER_MIN_CORE_SIZE),
        route_probability_source=str(scores.get("route_source", "unknown")),
        buffer_skipped_from_training=True,
        buffer_in_sup_idx=False,
        buffer_in_align_idx=False,
        buffer_in_unrel_idx=False,
        buffer_in_idx_unk_eval=False,
        core_sets_mutually_exclusive=bool(all(disjoint.values())),
        core_buffer_union_exhaustive=bool(len(sets["stable"] | sets["shift"] | sets["unknown"] | sets["buffer"]) == n),
        **disjoint,
        **{k: v for k, v in dict(subsets.get("fallback", {})).items() if k not in {"selection_mode", "conflict_policy"}},
        **{f"fallback_{k}": v for k, v in dict(subsets.get("fallback", {})).items()},
    )
    for name, idx in [("stable", stable_idx), ("shift", shift_idx), ("unknown", unknown_idx), ("buffer", buffer_idx)]:
        stats = _direct_subset_stats(idx, scores, y_adapt=stageA.get("y_adapt", None))
        for k, v in stats.items():
            diag[f"{name}_{k}"] = v
    bucket = dict(spec.get("bucket_info", {}))
    bucket.update(diag)
    bucket.update(dict(
        ablation_variant=str(DIRECT_COREBUFFER_METHOD_NAME),
        ablation_note="Direct soft-route core-buffer construction from all raw target adaptation samples; historical hard-gate partitions are not used for subset selection.",
        direct_corebuffer_loss_mapping="stable_core->Lsup/Lproto; shift_core->alignment/entropy/consistency; unknown_core->Lunk; buffer->skipped",
    ))
    spec["bucket_info"] = bucket
    info = dict(base_soda_info or {})
    info.update(bucket)
    info["framework_family"] = "SODA-RFF"
    info["framework_note"] = "Direct soft-route core-buffer SODA diagnostic variant."
    return spec, info


def _direct_corebuffer_row_fields(method_info):
    keys = [
        "direct_corebuffer_active", "direct_corebuffer_method", "direct_corebuffer_protocol_tag",
        "n_target_adapt", "n_stable_core", "n_shift_core", "n_unknown_core", "n_buffer",
        "frac_stable_core", "frac_shift_core", "frac_unknown_core", "frac_buffer",
        "selection_mode", "conflict_policy", "p_source",
        "stable_keep_frac", "shift_keep_frac", "unknown_keep_frac",
        "stable_score_min", "shift_score_min", "unknown_score_min",
        "stable_margin_min", "shift_margin_min", "unknown_margin_min",
        "stable_conf_min", "shift_conf_min", "unknown_conf_max",
        "stable_entropy_max", "shift_entropy_max", "unknown_entropy_min",
        "shift_known_min", "unknown_known_max",
        "use_agreement_min", "stable_agree_min", "shift_agree_min", "use_max_cap",
        "raw_stable_mask_count", "raw_shift_mask_count", "raw_unknown_mask_count",
        "overlap_stable_shift_count", "overlap_stable_unknown_count", "overlap_shift_unknown_count", "overlap_all_three_count",
        "conflict_assigned_by_priority_count", "thresholds_relaxed_for_smoke", "relaxed_thresholds_json",
        "topk_fallback_used", "forced_min_core_size_used", "max_cap_enabled",
        "stable_max_cap_applied", "shift_max_cap_applied", "unknown_max_cap_applied",
        "stable_core_empty", "shift_core_empty", "unknown_core_empty",
        "core_buffer_union_exhaustive",
        "stable_margin_min", "shift_margin_min", "unknown_margin_min",
        "class_balanced_known", "buffer_mode", "sup_weight_mode", "min_core_size",
        "route_probability_source", "buffer_skipped_from_training", "buffer_in_sup_idx",
        "buffer_in_align_idx", "buffer_in_unrel_idx", "buffer_in_idx_unk_eval",
        "core_sets_mutually_exclusive", "stable_shift_empty", "stable_unknown_empty", "shift_unknown_empty", "buffer_disjoint",
        "fallback_unknown_relaxed_margin", "fallback_unknown_topk_fallback",
        "fallback_shift_relaxed_margin", "fallback_shift_topk_fallback",
        "fallback_stable_relaxed_margin", "fallback_stable_topk_fallback",
    ]
    subset_prefixes = ["stable", "shift", "unknown", "buffer"]
    stat_suffixes = [
        "count", "pi_s_mean", "pi_s_std", "pi_d_mean", "pi_d_std", "pi_u_mean", "pi_u_std",
        "p_known_mean", "p_known_std", "p_drift_given_known_mean", "p_drift_given_known_std",
        "conf_mean", "conf_std", "H_norm_mean", "H_norm_std", "margin_mean", "margin_std",
        "s_stable_mean", "s_stable_std", "s_shift_mean", "s_shift_std", "s_unk_mean", "s_unk_std",
        "pseudo_class_distribution_json", "oracle_known_frac", "oracle_unknown_frac", "oracle_pseudo_acc_known",
    ]
    for pref in subset_prefixes:
        keys.extend([f"{pref}_{suf}" for suf in stat_suffixes])
    return {k: method_info.get(k, "") for k in keys if k in method_info}

STRICT_REPRO_VERSION = "strict_repro_v1"
STRICT_BASELINE_METHODS = ["PCPD_strict", "OVANet_strict", "COMET_strict", "JRFFP_SC_strict"]
METHOD_ORDER = [SODA_METHOD_NAME, DIRECT_COREBUFFER_METHOD_NAME, "SourceOnly"] + list(STRICT_BASELINE_METHODS)
FIXED_FAR_METHODS = list(METHOD_ORDER)
SOURCE_CAL_METHODS = list(METHOD_ORDER)

BASELINE_NATIVE_REJECTOR_FAMILY.update({
    "PCPD_strict": "pcpd_strict",
    "OVANet_strict": "ovanet_strict",
    "COMET_strict": "comet_strict",
    "JRFFP_SC_strict": "jrffp_sc_strict",
})

SOURCE_CAL_RUN_SUBDIR = "source_calibrated_bundle"
SOURCE_CAL_FRR_TARGETS = [0.01, 0.05, 0.10]
SOURCE_CAL_EXPORT_XLSX = True
SOURCE_CAL_EXPORT_JSON = True

STRICT_OVA_LAMBDA_OEM = 0.10
STRICT_OVA_EPOCHS = ADAPT_EPOCHS
STRICT_PCPD_TAU = 0.90
STRICT_PCPD_LAMBDA_PROTO = 0.35
STRICT_PCPD_LAMBDA_DANN = 0.10
STRICT_PCPD_LAMBDA_PL = 1.00
STRICT_COMET_DELTA_LOW = 0.40
STRICT_COMET_DELTA_HIGH = 0.60
STRICT_COMET_EMA = 0.99
STRICT_COMET_LAMBDA_CONTRAST = 0.40
STRICT_COMET_LAMBDA_ENT = 0.20
STRICT_COMET_UNKNOWN_MARGIN = 0.25
STRICT_JRFFP_LAMBDA_SIA = 1.00
STRICT_JRFFP_LAMBDA_CE = 1.00

STRICT_TUNABLE_CONSTANTS = [
    "STRICT_OVA_LAMBDA_OEM",
    "STRICT_OVA_EPOCHS",
    "STRICT_PCPD_TAU",
    "STRICT_PCPD_LAMBDA_PROTO",
    "STRICT_PCPD_LAMBDA_DANN",
    "STRICT_PCPD_LAMBDA_PL",
    "STRICT_COMET_DELTA_LOW",
    "STRICT_COMET_DELTA_HIGH",
    "STRICT_COMET_EMA",
    "STRICT_COMET_LAMBDA_CONTRAST",
    "STRICT_COMET_LAMBDA_ENT",
    "STRICT_COMET_UNKNOWN_MARGIN",
    "STRICT_JRFFP_LAMBDA_SIA",
    "STRICT_JRFFP_LAMBDA_CE",
]
STRICT_BUDGET_KEYS = [
    "SODA_METHOD_NAME", "IN_PLANES", "DROPOUT", "EPOCHS", "LR", "WEIGHT_DECAY", "PATIENCE",
    "ADAPT_FRAC", "MAX_ADAPT_PER_SET", "ADAPT_EPOCHS", "ADAPT_BATCH", "ADAPT_LR", "ADAPT_WEIGHT_DECAY",
    "TX_SPLIT_REPEATS", "SOURCE_CAL_FRR_TARGETS", "FIXED_FAR_TARGETS", "ROUTER_CAL_FRAC", "ROUTER_MIN_CAL_PER_POOL",
]


def _strict_jsonable(value):
    if isinstance(value, (str, int, float, bool)) or value is None:
        if isinstance(value, float) and (not np.isfinite(value)):
            return None
        return value
    if isinstance(value, dict):
        return {str(k): _strict_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_strict_jsonable(v) for v in value]
    try:
        return value.item()
    except Exception:
        return str(value)


def get_strict_budget_manifest(env=None):
    env = globals() if env is None else env
    adapted_counts = env.get("STRICT_SEARCH_ADAPTED_CANDIDATES_PER_METHOD", None)
    return {
        "same_budget_version": "strict_protocol_level_search_v1",
        "budget_keys": {k: _strict_jsonable(env.get(k)) for k in STRICT_BUDGET_KEYS if k in env},
        "same_backbone": "ResNet18_1D shared by SODA-RFF, SourceOnly, and strict baselines",
        "same_source_training": "Stage-A source model and source split cache are reused across candidates",
        "same_target_unlabeled_pool": "Each candidate consumes the same Z_adapt pool for the protocol/split/fold",
        "same_adaptation_epochs": _strict_jsonable(env.get("ADAPT_EPOCHS")),
        "same_candidate_budget_adapted_methods": _strict_jsonable(adapted_counts),
        "source_only_budget_note": "SourceOnly has no target-adaptation hyperparameters; it is evaluated once on the same FAR grid.",
        "selection_rule": "SODA-aligned FAR grid: mean over FAR points of 0.5*H_KU + 0.5*H_DU, then drift/known/stable/AUC tie-breakers.",
    }


def load_strict_protocol_param_package(path):
    if not path:
        return {}
    if not os.path.exists(path):
        return {}
    with _safe_open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict) and "selected" in data:
        return data
    if isinstance(data, dict):
        return {"selected": data}
    return {}


def _strict_selected_entry(package, protocol_tag, base_method):
    selected = package.get("selected", {}) if isinstance(package, dict) else {}
    proto = selected.get(str(protocol_tag), {}) if isinstance(selected, dict) else {}
    entry = proto.get(str(base_method), {}) if isinstance(proto, dict) else {}
    return dict(entry) if isinstance(entry, dict) else {}


def _strict_selected_updates(package, protocol_tag, base_method):
    entry = _strict_selected_entry(package, protocol_tag, base_method)
    updates = dict(entry.get("updates", {})) if isinstance(entry.get("updates", {}), dict) else {}
    for key in STRICT_TUNABLE_CONSTANTS:
        if key in entry:
            updates[key] = entry[key]
    return updates


def selected_eval_heads(package, protocol_tag, base_method, default_heads):
    entry = _strict_selected_entry(package, protocol_tag, base_method)
    head = entry.get("final_eval_head", None)
    if head is None:
        return list(default_heads)
    return [str(head)]


def selected_source_cal_frr_targets(package, protocol_tag, base_method, default_targets):
    entry = _strict_selected_entry(package, protocol_tag, base_method)
    target = entry.get("final_source_cal_frr_target", None)
    if target is None:
        return [float(v) for v in default_targets]
    return [float(target)]


@contextlib.contextmanager
def strict_protocol_method_context(env, protocol_tag, base_method):
    package = env.get("STRICT_PROTOCOL_PARAM_PACKAGE", {})
    updates = _strict_selected_updates(package, protocol_tag, base_method)
    old_values = {k: env.get(k) for k in updates}
    try:
        env.update(updates)
        yield updates
    finally:
        for key, value in old_values.items():
            env[key] = value


STRICT_PROTOCOL_PARAM_PACKAGE_PATH = globals().get("STRICT_PROTOCOL_PARAM_PACKAGE_PATH", "strict_protocol_best_params_v2_20cands.json")
STRICT_PROTOCOL_PARAM_PACKAGE = load_strict_protocol_param_package(STRICT_PROTOCOL_PARAM_PACKAGE_PATH)


def _strict_protocol_package_active():
    return isinstance(STRICT_PROTOCOL_PARAM_PACKAGE, dict) and bool(STRICT_PROTOCOL_PARAM_PACKAGE.get("selected", {}))


def _strict_protocol_eval_heads(protocol_tag, base_method, default_heads):
    if _strict_protocol_package_active():
        return selected_eval_heads(STRICT_PROTOCOL_PARAM_PACKAGE, protocol_tag, base_method, default_heads)
    return list(default_heads)


def _strict_protocol_source_cal_frr_targets(protocol_tag, base_method, default_targets):
    if _strict_protocol_package_active():
        return selected_source_cal_frr_targets(STRICT_PROTOCOL_PARAM_PACKAGE, protocol_tag, base_method, default_targets)
    return [float(v) for v in default_targets]


def _strict_protocol_package_log(protocol_tag):
    package_path = str(globals().get("STRICT_PROTOCOL_PARAM_PACKAGE_PATH", "") or "")
    package_file = os.path.basename(package_path) if package_path else ""
    active = bool(_strict_protocol_package_active())
    package = globals().get("STRICT_PROTOCOL_PARAM_PACKAGE", {})
    selected = package.get("selected", {}) if isinstance(package, dict) else {}
    protocol_selected = selected.get(str(protocol_tag), {}) if isinstance(selected, dict) else {}
    selected_methods = []
    if isinstance(protocol_selected, dict):
        selected_methods = sorted([
            str(k) for k, v in protocol_selected.items()
            if isinstance(v, dict) and (bool(v.get("updates")) or len(v) > 0)
        ])
    print(
        f"[STRICT-BASELINE-CONFIG] protocol={protocol_tag} package_file={package_file or 'none'} "
        f"package_path={package_path or 'none'} active={active} "
        f"selected_methods={selected_methods if selected_methods else 'none'}"
    )


@contextlib.contextmanager
def _strict_protocol_method_runtime(protocol_tag, base_method):
    if _strict_protocol_package_active():
        with strict_protocol_method_context(globals(), protocol_tag, base_method) as updates:
            yield updates
    else:
        yield {}


RUN_METADATA["strict_repro_version"] = STRICT_REPRO_VERSION
RUN_METADATA["strict_baseline_methods"] = list(STRICT_BASELINE_METHODS)
RUN_METADATA["strict_protocol_param_package_path"] = STRICT_PROTOCOL_PARAM_PACKAGE_PATH
RUN_METADATA["strict_protocol_param_package_active"] = _strict_protocol_package_active()
RUN_METADATA["strict_same_budget_manifest"] = get_strict_budget_manifest(globals())


def get_default_rejector_head(base_name):
    base_name = str(base_name)
    if base_name == SODA_METHOD_NAME or base_name == globals().get("DIRECT_COREBUFFER_METHOD_NAME", "SODA_direct_corebuffer"):
        return str(get_soda_protocol_fixed_config(globals().get("SODA_ACTIVE_PROTOCOL_TAG", SODA_DEFAULT_PROTOCOL)).get("default_rejector_head", "energy"))
    if base_name in STRICT_BASELINE_METHODS:
        return "native"
    return str(DEFAULT_FINAL_REJECTOR_HEAD)


def get_supported_eval_heads_for_base_method(base_name):
    return [get_default_rejector_head(base_name)]


def get_eval_heads_for_base_method(base_name):
    return [get_default_rejector_head(base_name)]


def resolve_eval_method_order(base_method_order):
    out = []
    for _base in base_method_order:
        for _head in get_eval_heads_for_base_method(_base):
            _name = make_eval_method_name(_base, _head)
            if _name not in out:
                out.append(_name)
    return out


EVAL_METHOD_ORDER = resolve_eval_method_order(METHOD_ORDER)
EVAL_METHOD_BASES = [parse_eval_method_name(v)[0] for v in EVAL_METHOD_ORDER]


class _GradReverseFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = float(lam)
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lam * grad_output, None


def grad_reverse(x, lam=1.0):
    return _GradReverseFn.apply(x, lam)


class DomainDiscriminator(nn.Module):
    def __init__(self, dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(hidden, 2),
        )

    def forward(self, z, grl_lambda=1.0):
        return self.net(grad_reverse(z, grl_lambda))


class OVAHead(nn.Module):
    def __init__(self, dim, num_classes):
        super().__init__()
        self.num_classes = int(num_classes)
        self.fc = nn.Linear(dim, 2 * int(num_classes))

    def forward(self, z):
        return self.fc(z).view(z.shape[0], self.num_classes, 2)


class SiameseComparator(nn.Module):
    def __init__(self, dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 * dim, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(inplace=True),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, z, ref):
        h = torch.cat([torch.abs(z - ref), z * ref], dim=1)
        return self.net(h).squeeze(1)


def _clone_fc_trainable(fc_layer):
    fc = nn.Linear(fc_layer.in_features, fc_layer.out_features, bias=True).to(device)
    fc.load_state_dict(fc_layer.state_dict())
    fc.train()
    for p in fc.parameters():
        p.requires_grad = True
    return fc


def _module_to_eval_cpu_state(module):
    return {k: v.detach().cpu().clone() for k, v in module.state_dict().items()}


def _torch_proto_from_np(protos):
    return torch.tensor(np.asarray(protos, dtype=np.float32), dtype=torch.float32, device=device)


def _source_prototypes_np(Z, y, K):
    return fit_class_prototypes(np.asarray(Z, dtype=np.float32), np.asarray(y, dtype=np.int64), K=int(K), l2norm=True)


def _proto_loss_torch(z, y, protos, margin=0.50):
    if z.numel() == 0:
        return z.sum() * 0.0
    z_n = F.normalize(z, dim=1)
    p_n = F.normalize(protos, dim=1)
    d = torch.cdist(z_n, p_n, p=2.0).pow(2)
    own = d[torch.arange(z.shape[0], device=z.device), y]
    mask = F.one_hot(y, num_classes=p_n.shape[0]).bool()
    neg = d.masked_fill(mask, float("inf")).min(dim=1).values
    return own.mean() + F.relu(float(margin) + own - neg).mean()


def _unknown_proto_repel_loss(z, protos, margin=0.25):
    if z.numel() == 0:
        return z.sum() * 0.0
    sim = F.normalize(z, dim=1) @ F.normalize(protos, dim=1).T
    return F.relu(sim.max(dim=1).values - float(margin)).mean()


def _binary_entropy_from_ova_logits(ova_logits):
    p = torch.softmax(ova_logits, dim=2)
    return -(p * torch.log(p + 1e-12)).sum(dim=2).mean()


def _select_pseudo_indices(P, tau=0.90, min_keep=64):
    P = normalize_rows(np.asarray(P, dtype=np.float32))
    conf = np.max(P, axis=1)
    y = np.argmax(P, axis=1).astype(np.int64)
    keep = np.where(conf >= float(tau))[0].astype(np.int64)
    if len(keep) < min(min_keep, len(conf)):
        order = np.argsort(-conf)
        keep = order[:min(len(conf), max(int(min_keep), len(keep)))].astype(np.int64)
    w = normalize_conf_weights(conf[keep]) if len(keep) > 0 else np.zeros((0,), dtype=np.float32)
    return keep, y[keep], w.astype(np.float32)


def _strict_loader(Z, y=None, w=None, batch=None, shuffle=True):
    batch = int(batch or ADAPT_BATCH)
    if y is None:
        ds = EmbOnlyDataset(Z)
    elif w is None:
        ds = EmbDatasetWeightedHard(Z, y, None)
    else:
        ds = EmbDatasetWeightedHard(Z, y, w)
    return DataLoader(ds, batch_size=batch, shuffle=shuffle, drop_last=False)


def fit_strict_ovanet_bundle(fc_layer, Z_replay, y_replay, Z_adapt, **kwargs):
    K = int(fc_layer.out_features)
    adapter = ResidualAdapter(dim=Z_replay.shape[1], bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE).to(device)
    fc = _clone_fc_trainable(fc_layer)
    ova = OVAHead(Z_replay.shape[1], K).to(device)
    opt = optim.Adam(list(adapter.parameters()) + list(fc.parameters()) + list(ova.parameters()), lr=ADAPT_LR, weight_decay=ADAPT_WEIGHT_DECAY)
    src_loader = _strict_loader(Z_replay, y_replay, batch=ADAPT_BATCH, shuffle=True)
    tgt_loader = _strict_loader(Z_adapt, None, batch=ADAPT_BATCH, shuffle=True)
    it_src, it_tgt = cycle_loader(src_loader), cycle_loader(tgt_loader)
    steps = max(len(src_loader), len(tgt_loader), 1)
    for _ep in range(max(1, int(STRICT_OVA_EPOCHS))):
        adapter.train(); fc.train(); ova.train()
        for _ in range(steps):
            zs, ys, _ = next(it_src)
            zt = next(it_tgt)
            zs, ys, zt = zs.to(device), ys.to(device), zt.to(device)
            opt.zero_grad()
            zs2 = adapter(zs)
            logits_s = fc(zs2)
            ova_s = ova(zs2)
            loss_cls = F.cross_entropy(logits_s, ys)
            with torch.no_grad():
                masked = logits_s.detach().clone()
                masked[torch.arange(masked.shape[0], device=device), ys] = -1e9
                hard_neg = masked.argmax(dim=1)
            pos_target = torch.ones_like(ys)
            neg_target = torch.zeros_like(ys)
            loss_ova = F.cross_entropy(ova_s[torch.arange(zs2.shape[0], device=device), ys], pos_target)
            loss_ova = loss_ova + F.cross_entropy(ova_s[torch.arange(zs2.shape[0], device=device), hard_neg], neg_target)
            zt2 = adapter(zt)
            loss_oem = _binary_entropy_from_ova_logits(ova(zt2))
            loss = loss_cls + loss_ova + float(STRICT_OVA_LAMBDA_OEM) * loss_oem
            loss.backward()
            torch.nn.utils.clip_grad_norm_(list(adapter.parameters()) + list(fc.parameters()) + list(ova.parameters()), ADAPT_GRAD_CLIP)
            opt.step()
    adapter.eval(); fc.eval(); ova.eval()
    return {"adapter": adapter, "fc": fc, "ova_head": ova, "strict_family": "ovanet_strict"}


def fit_strict_pcpd_bundle(fc_layer, Z_replay, y_replay, Z_adapt, P_seed=None, **kwargs):
    K = int(fc_layer.out_features)
    adapter = ResidualAdapter(dim=Z_replay.shape[1], bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE).to(device)
    fc = _clone_fc_trainable(fc_layer)
    disc = DomainDiscriminator(Z_replay.shape[1]).to(device)
    if P_seed is None:
        P_seed = source_head_probs_from_embeddings(fc_layer, Z_adapt, batch=512)
    idx_pl, y_pl, w_pl = _select_pseudo_indices(P_seed, tau=STRICT_PCPD_TAU, min_keep=PSEUDO_MIN_KEEP)
    proto_np = _source_prototypes_np(Z_replay, y_replay, K)
    proto_t = _torch_proto_from_np(proto_np)
    opt = optim.Adam(list(adapter.parameters()) + list(fc.parameters()) + list(disc.parameters()), lr=ADAPT_LR, weight_decay=ADAPT_WEIGHT_DECAY)
    src_loader = _strict_loader(Z_replay, y_replay, batch=ADAPT_BATCH, shuffle=True)
    pl_loader = _strict_loader(Z_adapt[idx_pl], y_pl, w_pl, batch=ADAPT_BATCH, shuffle=True) if len(idx_pl) > 0 else None
    tgt_loader = _strict_loader(Z_adapt, None, batch=ADAPT_BATCH, shuffle=True)
    it_src, it_tgt = cycle_loader(src_loader), cycle_loader(tgt_loader)
    it_pl = cycle_loader(pl_loader) if pl_loader is not None else None
    steps = max(len(src_loader), len(tgt_loader), len(pl_loader) if pl_loader is not None else 0, 1)
    for ep in range(max(1, int(ADAPT_EPOCHS))):
        adapter.train(); fc.train(); disc.train()
        grl_lam = min(1.0, float(ep + 1) / max(1.0, float(ADAPT_EPOCHS)))
        for _ in range(steps):
            zs, ys, _ = next(it_src)
            zt = next(it_tgt)
            zs, ys, zt = zs.to(device), ys.to(device), zt.to(device)
            opt.zero_grad()
            zs2 = adapter(zs)
            logits_s = fc(zs2)
            loss = F.cross_entropy(logits_s, ys)
            loss = loss + float(STRICT_PCPD_LAMBDA_PROTO) * _proto_loss_torch(zs2, ys, proto_t)
            zt2 = adapter(zt)
            dom_z = torch.cat([zs2, zt2], dim=0)
            dom_y = torch.cat([
                torch.zeros(zs2.shape[0], dtype=torch.long, device=device),
                torch.ones(zt2.shape[0], dtype=torch.long, device=device),
            ], dim=0)
            loss = loss + float(STRICT_PCPD_LAMBDA_DANN) * F.cross_entropy(disc(dom_z, grl_lambda=grl_lam), dom_y)
            if it_pl is not None:
                zp, yp, wp = next(it_pl)
                zp, yp, wp = zp.to(device), yp.to(device), wp.to(device)
                zp2 = adapter(zp)
                logits_p = fc(zp2)
                ce_p = F.cross_entropy(logits_p, yp, reduction="none")
                loss = loss + float(STRICT_PCPD_LAMBDA_PL) * normalized_weighted_mean_torch(ce_p, wp)
                loss = loss + float(STRICT_PCPD_LAMBDA_PROTO) * _proto_loss_torch(zp2, yp, proto_t)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(list(adapter.parameters()) + list(fc.parameters()) + list(disc.parameters()), ADAPT_GRAD_CLIP)
            opt.step()
    adapter.eval(); fc.eval(); disc.eval()
    return {"adapter": adapter, "fc": fc, "domain_disc": disc, "strict_family": "pcpd_strict", "pcpd_pseudo_keep": int(len(idx_pl))}


def fit_strict_comet_bundle(fc_layer, Z_replay, y_replay, Z_adapt, **kwargs):
    K = int(fc_layer.out_features)
    student = ResidualAdapter(dim=Z_replay.shape[1], bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE).to(device)
    teacher = ResidualAdapter(dim=Z_replay.shape[1], bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE).to(device)
    teacher.load_state_dict(student.state_dict())
    fc = _clone_fc_trainable(fc_layer)
    teacher_fc = _clone_fc_trainable(fc_layer)
    teacher_fc.load_state_dict(fc.state_dict())
    for p in teacher.parameters():
        p.requires_grad = False
    for p in teacher_fc.parameters():
        p.requires_grad = False
    proto_t = _torch_proto_from_np(_source_prototypes_np(Z_replay, y_replay, K))
    opt = optim.Adam(list(student.parameters()) + list(fc.parameters()), lr=ADAPT_LR, weight_decay=ADAPT_WEIGHT_DECAY)
    tgt_loader = _strict_loader(Z_adapt, None, batch=ADAPT_BATCH, shuffle=True)
    for _ep in range(max(1, int(ADAPT_EPOCHS))):
        student.train(); fc.train(); teacher.eval(); teacher_fc.eval()
        for zt in tgt_loader:
            zt = zt.to(device)
            opt.zero_grad()
            with torch.no_grad():
                z_teacher = teacher(zt)
                logits_teacher = teacher_fc(z_teacher)
                p_teacher = torch.softmax(logits_teacher, dim=1)
                ent_teacher = -(p_teacher * torch.log(p_teacher + 1e-12)).sum(dim=1) / math.log(float(max(2, K)))
                pseudo = p_teacher.argmax(dim=1)
                known_mask = ent_teacher <= float(STRICT_COMET_DELTA_LOW)
                unknown_mask = ent_teacher >= float(STRICT_COMET_DELTA_HIGH)
            z_student = student(zt)
            logits_student = fc(z_student)
            loss = logits_student.sum() * 0.0
            if bool(known_mask.any()):
                zk = z_student[known_mask]
                yk = pseudo[known_mask]
                loss = loss + F.cross_entropy(logits_student[known_mask], yk)
                loss = loss + float(STRICT_COMET_LAMBDA_CONTRAST) * _proto_loss_torch(zk, yk, proto_t, margin=0.50)
                loss = loss + float(STRICT_COMET_LAMBDA_ENT) * entropy_from_logits_torch(logits_student[known_mask]).mean()
            if bool(unknown_mask.any()):
                zu = z_student[unknown_mask]
                loss = loss + float(STRICT_COMET_LAMBDA_CONTRAST) * _unknown_proto_repel_loss(zu, proto_t, margin=STRICT_COMET_UNKNOWN_MARGIN)
                loss = loss + float(STRICT_COMET_LAMBDA_ENT) * unknown_regularization_from_logits(logits_student[unknown_mask], mode="uniform_kl")
            if (not bool(known_mask.any())) and (not bool(unknown_mask.any())):
                loss = entropy_from_logits_torch(logits_student).mean() * 0.0
            loss.backward()
            torch.nn.utils.clip_grad_norm_(list(student.parameters()) + list(fc.parameters()), ADAPT_GRAD_CLIP)
            opt.step()
            with torch.no_grad():
                for tp, sp in zip(teacher.parameters(), student.parameters()):
                    tp.data.mul_(float(STRICT_COMET_EMA)).add_(sp.data, alpha=1.0 - float(STRICT_COMET_EMA))
                for tp, sp in zip(teacher_fc.parameters(), fc.parameters()):
                    tp.data.mul_(float(STRICT_COMET_EMA)).add_(sp.data, alpha=1.0 - float(STRICT_COMET_EMA))
    student.eval(); fc.eval(); teacher.eval(); teacher_fc.eval()
    return {"adapter": student, "fc": fc, "teacher_adapter": teacher, "teacher_fc": teacher_fc, "strict_family": "comet_strict"}


def fit_strict_jrffp_sc_bundle(fc_layer, Z_replay, y_replay, **kwargs):
    K = int(fc_layer.out_features)
    adapter = ResidualAdapter(dim=Z_replay.shape[1], bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE).to(device)
    fc = _clone_fc_trainable(fc_layer)
    comp = SiameseComparator(Z_replay.shape[1]).to(device)
    proto_t = _torch_proto_from_np(_source_prototypes_np(Z_replay, y_replay, K))
    opt = optim.Adam(list(adapter.parameters()) + list(fc.parameters()) + list(comp.parameters()), lr=ADAPT_LR, weight_decay=ADAPT_WEIGHT_DECAY)
    src_loader = _strict_loader(Z_replay, y_replay, batch=ADAPT_BATCH, shuffle=True)
    for _ep in range(max(1, int(ADAPT_EPOCHS))):
        adapter.train(); fc.train(); comp.train()
        for zs, ys, _ in src_loader:
            zs, ys = zs.to(device), ys.to(device)
            opt.zero_grad()
            z2 = adapter(zs)
            logits = fc(z2)
            with torch.no_grad():
                sim = F.normalize(z2.detach(), dim=1) @ F.normalize(proto_t, dim=1).T
                sim[torch.arange(sim.shape[0], device=device), ys] = -1e9
                neg = sim.argmax(dim=1)
            pos_logit = comp(z2, proto_t[ys])
            neg_logit = comp(z2, proto_t[neg])
            sia_logits = torch.cat([pos_logit, neg_logit], dim=0)
            sia_y = torch.cat([torch.ones_like(pos_logit), torch.zeros_like(neg_logit)], dim=0)
            loss = float(STRICT_JRFFP_LAMBDA_CE) * F.cross_entropy(logits, ys)
            loss = loss + float(STRICT_JRFFP_LAMBDA_SIA) * F.binary_cross_entropy_with_logits(sia_logits, sia_y)
            loss = loss + LAMBDA_REPLAY_ANCHOR * F.mse_loss(z2, zs)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(list(adapter.parameters()) + list(fc.parameters()) + list(comp.parameters()), ADAPT_GRAD_CLIP)
            opt.step()
    adapter.eval(); fc.eval(); comp.eval()
    return {"adapter": adapter, "fc": fc, "siamese": comp, "strict_family": "jrffp_sc_strict"}


_orig_fit_single_adapter_from_spec_strict = fit_single_adapter_from_spec


def fit_single_adapter_from_spec(spec, fc_layer, Z_replay, y_replay, Z_adapt, protos, sc_adapt=None):
    train_mode = str(spec.get("train", ""))
    if train_mode == "strict_ovanet":
        return fit_strict_ovanet_bundle(fc_layer, Z_replay, y_replay, Z_adapt), {"strict_train_mode": train_mode}
    if train_mode == "strict_pcpd":
        bundle = fit_strict_pcpd_bundle(fc_layer, Z_replay, y_replay, Z_adapt, P_seed=spec.get("P_seed", None))
        return bundle, {"strict_train_mode": train_mode, "pcpd_pseudo_keep": bundle.get("pcpd_pseudo_keep", 0)}
    if train_mode == "strict_comet":
        return fit_strict_comet_bundle(fc_layer, Z_replay, y_replay, Z_adapt), {"strict_train_mode": train_mode}
    if train_mode == "strict_jrffp_sc":
        return fit_strict_jrffp_sc_bundle(fc_layer, Z_replay, y_replay), {"strict_train_mode": train_mode}
    return _orig_fit_single_adapter_from_spec_strict(spec, fc_layer, Z_replay, y_replay, Z_adapt, protos, sc_adapt=sc_adapt)


def _eval_ova_pos_np(bundle, Z):
    ova = bundle["ova_head"].to(device).eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(Z), 512):
            zb = torch.tensor(Z[i:i+512], dtype=torch.float32, device=device)
            logits = ova(zb)
            out.append(torch.softmax(logits, dim=2)[:, :, 1].detach().cpu().numpy().astype(np.float32))
    return np.concatenate(out, axis=0) if out else np.zeros((0, ova.num_classes), dtype=np.float32)


def _eval_siamese_score_np(bundle, Z, pred, protos):
    comp = bundle["siamese"].to(device).eval()
    protos_t = torch.tensor(np.asarray(protos, dtype=np.float32), dtype=torch.float32, device=device)
    pred = np.asarray(pred, dtype=np.int64)
    out = []
    with torch.no_grad():
        for i in range(0, len(Z), 512):
            zb = torch.tensor(Z[i:i+512], dtype=torch.float32, device=device)
            pb = torch.tensor(pred[i:i+512], dtype=torch.long, device=device)
            score = torch.sigmoid(comp(zb, protos_t[pb]))
            out.append(score.detach().cpu().numpy().astype(np.float32))
    return np.concatenate(out, axis=0) if out else np.zeros((0,), dtype=np.float32)


def build_strict_eval_aux(bundle, eval_cache):
    family = str(bundle.get("strict_family", ""))
    y_tr = np.asarray(eval_cache["y_tr"], dtype=np.int64)
    K = int(eval_cache["logits_tr2"].shape[1])
    protos_eval = fit_class_prototypes(eval_cache["Z_tr2"], y_tr, K=K, l2norm=True)
    pred_A = np.argmax(eval_cache["logits_A2"], axis=1).astype(np.int64)
    pred_B = np.argmax(eval_cache["logits_B2"], axis=1).astype(np.int64)
    pred_C = np.argmax(eval_cache["logits_C2"], axis=1).astype(np.int64)
    pred_U = np.argmax(eval_cache["logits_U2"], axis=1).astype(np.int64)
    if family == "ovanet_strict":
        pos_tr = _eval_ova_pos_np(bundle, eval_cache["Z_tr2"])
        pos_A = _eval_ova_pos_np(bundle, eval_cache["Z_A2"])
        pos_B = _eval_ova_pos_np(bundle, eval_cache["Z_B2"])
        pos_C = _eval_ova_pos_np(bundle, eval_cache["Z_C2"])
        pos_U = _eval_ova_pos_np(bundle, eval_cache["Z_U2"])
        return dict(
            family=family,
            source_score=rowwise_pick(pos_tr, y_tr, fallback=0.0),
            score_A=rowwise_pick(pos_A, pred_A, fallback=0.0),
            score_B=rowwise_pick(pos_B, pred_B, fallback=0.0),
            score_C=rowwise_pick(pos_C, pred_C, fallback=0.0),
            score_U=rowwise_pick(pos_U, pred_U, fallback=0.0),
            pred_A=pred_A, pred_B=pred_B, pred_C=pred_C, pred_U=pred_U,
            source_y=y_tr,
            accept_high=True,
            score_name="ovanet_ova_predclass_posprob")
    if family == "pcpd_strict":
        pred_Ap, dist_A = nearest_proto_distance_and_pred(eval_cache["Z_A2"], protos_eval, l2norm=True)
        pred_Bp, dist_B = nearest_proto_distance_and_pred(eval_cache["Z_B2"], protos_eval, l2norm=True)
        pred_Cp, dist_C = nearest_proto_distance_and_pred(eval_cache["Z_C2"], protos_eval, l2norm=True)
        pred_Up, dist_U = nearest_proto_distance_and_pred(eval_cache["Z_U2"], protos_eval, l2norm=True)
        source_dist = l2_to_selected_prototypes(eval_cache["Z_tr2"], y_tr, protos_eval, l2norm=True)
        return dict(
            family=family,
            source_score=source_dist,
            score_A=dist_A, score_B=dist_B, score_C=dist_C, score_U=dist_U,
            pred_A=pred_Ap, pred_B=pred_Bp, pred_C=pred_Cp, pred_U=pred_Up,
            source_y=y_tr,
            accept_high=False,
            score_name="pcpd_proto_l2_calibrated")
    if family == "comet_strict":
        ent_tr = normalized_entropy_from_logits(eval_cache["logits_tr2"])
        ent_A = normalized_entropy_from_logits(eval_cache["logits_A2"])
        ent_B = normalized_entropy_from_logits(eval_cache["logits_B2"])
        ent_C = normalized_entropy_from_logits(eval_cache["logits_C2"])
        ent_U = normalized_entropy_from_logits(eval_cache["logits_U2"])
        return dict(
            family=family,
            source_score=ent_tr,
            score_A=ent_A, score_B=ent_B, score_C=ent_C, score_U=ent_U,
            pred_A=pred_A, pred_B=pred_B, pred_C=pred_C, pred_U=pred_U,
            source_y=y_tr,
            accept_high=False,
            score_name="comet_normalized_entropy")
    if family == "jrffp_sc_strict":
        source_score = _eval_siamese_score_np(bundle, eval_cache["Z_tr2"], y_tr, protos_eval)
        score_A = _eval_siamese_score_np(bundle, eval_cache["Z_A2"], pred_A, protos_eval)
        score_B = _eval_siamese_score_np(bundle, eval_cache["Z_B2"], pred_B, protos_eval)
        score_C = _eval_siamese_score_np(bundle, eval_cache["Z_C2"], pred_C, protos_eval)
        score_U = _eval_siamese_score_np(bundle, eval_cache["Z_U2"], pred_U, protos_eval)
        return dict(
            family=family,
            source_score=source_score,
            score_A=score_A, score_B=score_B, score_C=score_C, score_U=score_U,
            pred_A=pred_A, pred_B=pred_B, pred_C=pred_C, pred_U=pred_U,
            source_y=y_tr,
            accept_high=True,
            score_name="jrffpsc_siamese_similarity")
    return {}


_orig_build_variant_eval_cache_strict = build_variant_eval_cache


def build_variant_eval_cache(adapter, model_fc, Z_tr, y_tr, Z_A, D_A, Z_B, D_B, Z_C, D_C, Z_U, D_U,
                             mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids,
                             tau_id, tau_dom, router_bundle=None):
    cache = _orig_build_variant_eval_cache_strict(
        adapter, model_fc, Z_tr, y_tr, Z_A, D_A, Z_B, D_B, Z_C, D_C, Z_U, D_U,
        mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids,
        tau_id, tau_dom, router_bundle=router_bundle)
    if isinstance(adapter, dict) and adapter.get("strict_family", None) is not None:
        cache["strict_aux"] = build_strict_eval_aux(adapter, cache)
    return cache


def _strict_thresholds_from_source(aux, frr_target):
    source_score = np.asarray(aux["source_score"], dtype=np.float32)
    source_y = np.asarray(aux.get("source_y", np.zeros_like(source_score, dtype=np.int64)), dtype=np.int64)
    K = int(np.max(source_y)) + 1 if len(source_y) else 1
    accept_high = bool(aux.get("accept_high", True))
    q = float(frr_target) if accept_high else float(1.0 - frr_target)
    fallback = _safe_quantile(source_score, q, fallback=0.0)
    return build_classwise_thresholds(source_score, source_y, K, q=q, fallback=fallback), accept_high


def _strict_native_rejector(eval_cache, frr_target=0.05):
    aux = eval_cache.get("strict_aux", {})
    if not aux:
        return None
    thr_by_class, accept_high = _strict_thresholds_from_source(aux, frr_target)

    def _pack(split):
        pred = np.asarray(aux[f"pred_{split}"], dtype=np.int64)
        score = np.asarray(aux[f"score_{split}"], dtype=np.float32)
        thr = thr_by_class[np.clip(pred, 0, len(thr_by_class) - 1)]
        accept = score >= thr if accept_high else score <= thr
        auc_score = (thr - score).astype(np.float32) if accept_high else (score - thr).astype(np.float32)
        margin = auc_score
        return pred, score, thr, accept, auc_score, margin

    pred_A, _, _, acc_A, auc_A, _ = _pack("A")
    pred_B, _, _, acc_B, auc_B, _ = _pack("B")
    pred_C, _, _, acc_C, auc_C, _ = _pack("C")
    pred_U, _, _, acc_U, auc_U, _ = _pack("U")
    return dict(
        accept_A=acc_A, accept_B=acc_B, accept_C=acc_C, accept_U=acc_U,
        auc_score_A=auc_A, auc_score_B=auc_B, auc_score_C=auc_C, auc_score_U=auc_U,
        pred_A=pred_A, pred_B=pred_B, pred_C=pred_C, pred_U=pred_U,
        threshold=float(np.mean(thr_by_class)),
        threshold_by_class=thr_by_class.astype(np.float32),
        score_name=str(aux.get("score_name", "strict_native")),
        family=str(aux.get("family", "strict_native")))


_orig_build_final_rejector_from_cache_strict = build_final_rejector_from_cache


def build_final_rejector_from_cache(eval_cache, final_rejector_head="module2", frr_target=0.05, base_name=None):
    if str(final_rejector_head).lower() == "native" and isinstance(eval_cache, dict) and "strict_aux" in eval_cache:
        out = _strict_native_rejector(eval_cache, frr_target=frr_target)
        if out is not None:
            return out
    return _orig_build_final_rejector_from_cache_strict(eval_cache, final_rejector_head=final_rejector_head, frr_target=frr_target, base_name=base_name)


_orig_build_fixed_far_operating_bundle_strict = _build_fixed_far_operating_bundle


def _build_fixed_far_operating_bundle(eval_cache, final_rejector_head="energy", base_name=None, ref_frr_target=0.05):
    if str(final_rejector_head).lower() == "native" and isinstance(eval_cache, dict) and "strict_aux" in eval_cache:
        aux = eval_cache.get("strict_aux", {})
        thr_by_class, accept_high = _strict_thresholds_from_source(aux, ref_frr_target)
        out = {}
        for split in ["A", "B", "C", "U"]:
            pred = np.asarray(aux[f"pred_{split}"], dtype=np.int64)
            score = np.asarray(aux[f"score_{split}"], dtype=np.float32)
            thr = thr_by_class[np.clip(pred, 0, len(thr_by_class) - 1)]
            margin = (thr - score).astype(np.float32) if accept_high else (score - thr).astype(np.float32)
            out[f"margin_{split}"] = margin
            out[f"pred_{split}"] = pred
        return dict(
            margin_A=out["margin_A"], margin_B=out["margin_B"], margin_C=out["margin_C"], margin_U=out["margin_U"],
            pred_A=out["pred_A"], pred_B=out["pred_B"], pred_C=out["pred_C"], pred_U=out["pred_U"],
            score_name=str(aux.get("score_name", "strict_native")),
            family=str(aux.get("family", "strict_native")),
            base_threshold=float(np.mean(thr_by_class)))
    return _orig_build_fixed_far_operating_bundle_strict(eval_cache, final_rejector_head=final_rejector_head, base_name=base_name, ref_frr_target=ref_frr_target)


_orig_rebuild_compare_methods_from_stageA_strict = _rebuild_compare_methods_from_stageA


def _rebuild_compare_methods_from_stageA(stageA, model, protocol_tag):
    base_methods, base_info = _orig_rebuild_compare_methods_from_stageA_strict(stageA, model, protocol_tag=protocol_tag)
    methods = {}
    method_info = {}
    if SODA_METHOD_NAME in base_methods:
        methods[SODA_METHOD_NAME] = base_methods[SODA_METHOD_NAME]
        method_info[SODA_METHOD_NAME] = dict(base_info.get(SODA_METHOD_NAME, {}))
        if bool(globals().get("DIRECT_COREBUFFER_ENABLE", True)):
            direct_spec, direct_info = make_soda_direct_corebuffer_spec(
                stageA, protocol_tag=protocol_tag,
                base_soda_spec=base_methods[SODA_METHOD_NAME],
                base_soda_info=method_info[SODA_METHOD_NAME])
            if direct_spec is not None:
                methods[DIRECT_COREBUFFER_METHOD_NAME] = direct_spec
                method_info[DIRECT_COREBUFFER_METHOD_NAME] = dict(direct_info or {})
    P_logit = normalize_rows(np.asarray(stageA["P_logit_adapt"], dtype=np.float32))
    P_proto = normalize_rows(np.asarray(stageA["P_proto_adapt"], dtype=np.float32))
    P_knn = normalize_rows(np.asarray(stageA["P_knn_adapt"], dtype=np.float32))
    P_fuse = normalize_rows(0.50 * P_logit + 0.30 * P_proto + 0.20 * P_knn)
    methods["PCPD_strict"] = {"train": "strict_pcpd", "P_seed": P_fuse}
    methods["OVANet_strict"] = {"train": "strict_ovanet"}
    methods["COMET_strict"] = {"train": "strict_comet"}
    methods["JRFFP_SC_strict"] = {"train": "strict_jrffp_sc"}
    method_info.update({
        "PCPD_strict": {"framework_family": "PCPD", "repro_level": "strict_adapted", "core_components": "prototype loss + pseudo-label calibration + DANN/GRL"},
        "OVANet_strict": {"framework_family": "OVANet", "repro_level": "strict_adapted", "core_components": "closed classifier + one-vs-all head + HNCS + OEM"},
        "COMET_strict": {"framework_family": "COMET", "repro_level": "strict_adapted", "core_components": "mean teacher + entropy pseudo-labeling + contrastive/prototype losses"},
        "JRFFP_SC_strict": {"framework_family": "JRFFP-SC", "repro_level": "strict_adapted", "core_components": "prediction + learned Siamese comparison"},
        "SourceOnly": {"framework_family": "SourceOnly", "framework_note": "No target adaptation."},
    })
    methods = {k: v for k, v in methods.items() if k in METHOD_ORDER}
    method_info = {k: v for k, v in method_info.items() if (k in methods) or (k == "SourceOnly")}
    return methods, method_info


def _build_fold_eval_cache_for_method(base_name, method_spec, method_info, model, stageA, ctx, idx_te, y_tr_fold, router_bundle_v8, protocol_tag=None):
    adapter = None
    if base_name != "SourceOnly":
        with _strict_protocol_method_runtime(protocol_tag, base_name):
            adapter, _extra = fit_single_adapter_from_spec(
                method_spec,
                model.fc,
                stageA["Z_replay"],
                stageA["y_replay"],
                stageA["Z_adapt"],
                stageA["protos"],
                sc_adapt=stageA["sc_adapt"])
    _router_family = str(method_info.get(base_name, {}).get("router_family", "v8"))
    _router_bundle_eval = None if (_router_family == "v5" or str(base_name).endswith("_M2V5") or str(base_name).startswith("M2V5")) else router_bundle_v8
    return build_variant_eval_cache(
        adapter, model.fc,
        stageA["Z_tr"], y_tr_fold,
        stageA["Z_A"], stageA["D_A"],
        stageA["Z_B_ev"], stageA["D_B_ev"],
        stageA["Z_C_ev"], stageA["D_C_ev"],
        stageA["Z_U_ev"], stageA["D_U_ev"],
        stageA["mu_z_src"], stageA["var_z_src"], stageA["ref_sid_emb_A"], stageA["ref_sid_en_A"],
        stageA["models_kr"], stageA["fallback_k"], stageA["source_rx_ids"],
        stageA["tau_id"], stageA["tau_dom"], router_bundle=_router_bundle_eval)


def run_source_calibrated_one_split(protocol_tag, rx_protocol_tag, tx_protocol_tag, split_id,
                                    KNOWN_TX, UNKNOWN_TX, source_rxs, drift_rx,
                                    protocol_dir, stagea_cache_protocol_dir=None):
    if "apply_soda_protocol_runtime" in globals():
        _ = apply_soda_protocol_runtime(protocol_tag)
    ctx = _build_fixed_far_fold_context(
        protocol_tag=protocol_tag,
        split_id=split_id,
        KNOWN_TX=KNOWN_TX,
        UNKNOWN_TX=UNKNOWN_TX,
        source_rxs=source_rxs,
        drift_rx=drift_rx,
        protocol_dir=protocol_dir,
        stagea_cache_protocol_dir=stagea_cache_protocol_dir)
    rows = []
    for fold, (idx_tr, idx_te) in enumerate(ctx["skf"].split(ctx["X_src"], ctx["y_src"]), start=1):
        if MAX_FOLDS_PER_PROTOCOL is not None and int(fold) > int(MAX_FOLDS_PER_PROTOCOL):
            break
        fold_dir = os.path.join(ctx["split_dir"], f"fold_{fold}")
        cache_fold_dir = os.path.join(ctx["cache_split_dir"], f"fold_{fold}")
        _safe_makedirs(fold_dir)
        _safe_makedirs(cache_fold_dir)
        rng = np.random.RandomState(SEED + 1000 * split_id + fold)
        perm = rng.permutation(idx_tr)
        n_val = max(1, int(0.1 * len(perm)))
        idx_val = perm[:n_val]
        idx_train = perm[n_val:]
        stageA = build_or_load_stagea_fold(
            fold_dir=cache_fold_dir,
            protocol_tag=protocol_tag,
            split_id=split_id,
            fold=fold,
            K=ctx["K"],
            source_rxs=source_rxs,
            drift_rx=drift_rx,
            X_src=ctx["X_src"], y_src=ctx["y_src"], D_src=ctx["D_src"], rx_src=ctx["rx_src"],
            idx_train=idx_train, idx_val=idx_val, idx_te=idx_te,
            X_B=ctx["X_B"], y_B=ctx["y_B"], D_B=ctx["D_B"],
            X_C=ctx["X_C"], y_C=ctx["y_C"], D_C=ctx["D_C"],
            X_D=ctx["X_D"], y_D=ctx["y_D"], D_D=ctx["D_D"],
            X_E=ctx["X_E"], y_E=ctx["y_E"], D_E=ctx["D_E"],
            X_F=ctx["X_F"], y_F=ctx["y_F"], D_F=ctx["D_F"])
        model = stageA["model"]
        methods, method_info = _rebuild_compare_methods_from_stageA(stageA, model, protocol_tag=protocol_tag)
        y_tr_fold = np.asarray(stageA.get("y_tr_fold", ctx["y_src"][np.asarray(stageA.get("idx_train", idx_train), dtype=np.int64)]), dtype=np.int64)
        active_methods = [m for m in SOURCE_CAL_METHODS if (m == "SourceOnly") or (m in methods)]
        fold_rows = []
        for base_name in active_methods:
            spec = methods.get(base_name, None)
            eval_cache = _build_fold_eval_cache_for_method(
                base_name, spec, method_info, model, stageA, ctx, idx_te, y_tr_fold, stageA["router_bundle_v8"], protocol_tag=protocol_tag)
            for head in _strict_protocol_eval_heads(protocol_tag, base_name, get_eval_heads_for_base_method(base_name)):
                eval_name = make_eval_method_name(base_name, head)
                for frr_target in _strict_protocol_source_cal_frr_targets(protocol_tag, base_name, SOURCE_CAL_FRR_TARGETS):
                    metric_row = evaluate_variant_with_rejector(
                        eval_cache,
                        ctx["y_src"][idx_te], stageA["y_B_ev"], stageA["y_C_ev"],
                        final_rejector_head=head,
                        frr_target=float(frr_target),
                        base_name=base_name)
                    fold_rows.append({
                        "protocol_tag": protocol_tag,
                        "rx_protocol": rx_protocol_tag,
                        "tx_protocol": tx_protocol_tag,
                        "split_id": int(split_id),
                        "fold_id": int(fold),
                        "method": eval_name,
                        "base_method": str(base_name),
                        "frr_target": float(frr_target),
                        "threshold_protocol": "source_calibrated",
                        **metric_row,
                    })
        with _safe_open(os.path.join(fold_dir, "source_calibrated_rows.json"), "w", encoding="utf-8") as f:
            json.dump(fold_rows, f, indent=2)
        rows.extend(fold_rows)
    split_path = os.path.join(ctx["split_dir"], "source_calibrated_rows_all_folds.json")
    with _safe_open(split_path, "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2)
    return rows


def run_source_calibrated_protocol_specs(protocol_specs, base_dir, max_splits=None, max_folds=None):
    global MAX_FOLDS_PER_PROTOCOL
    old_max = MAX_FOLDS_PER_PROTOCOL
    MAX_FOLDS_PER_PROTOCOL = max_folds
    rows = []
    try:
        for ps in protocol_specs:
            protocol_dir = os.path.join(base_dir, ps["protocol_tag"])
            _safe_makedirs(protocol_dir)
            _strict_protocol_package_log(ps["protocol_tag"])
            tx_splits = list(ps["tx_splits"])
            if max_splits is not None:
                tx_splits = tx_splits[:int(max_splits)]
            for split_id, (KNOWN_TX, UNKNOWN_TX) in enumerate(tx_splits, start=1):
                rows.extend(run_source_calibrated_one_split(
                    protocol_tag=ps["protocol_tag"],
                    rx_protocol_tag=ps["rx_protocol"],
                    tx_protocol_tag=ps["tx_protocol"],
                    split_id=split_id,
                    KNOWN_TX=list(KNOWN_TX),
                    UNKNOWN_TX=list(UNKNOWN_TX),
                    source_rxs=ps["source_rxs"],
                    drift_rx=ps["drift_rx"],
                    protocol_dir=protocol_dir,
                    stagea_cache_protocol_dir=os.path.join(base_dir, "_stagea_cache", ps["protocol_tag"])))
    finally:
        MAX_FOLDS_PER_PROTOCOL = old_max
    return rows


def summarize_source_calibrated_rows(rows, metric_keys=None):
    metric_keys = list(metric_keys or FIXED_FAR_METRIC_KEYS)
    if not rows:
        return pd.DataFrame(), pd.DataFrame()
    raw_df = pd.DataFrame(rows)
    group_cols = ["protocol_tag", "method", "frr_target"]
    overall_cols = ["method", "frr_target"]
    protocol_records = []
    for key, g in raw_df.groupby(group_cols, dropna=False):
        rec = {c: v for c, v in zip(group_cols, key)}
        rec["n_rows"] = int(len(g))
        for mk in metric_keys:
            mean_v, std_v = _safe_mean_std(g[mk].tolist() if mk in g.columns else [])
            rec[f"{mk}_mean"] = mean_v
            rec[f"{mk}_std"] = std_v
        protocol_records.append(rec)
    overall_records = []
    for key, g in raw_df.groupby(overall_cols, dropna=False):
        rec = {c: v for c, v in zip(overall_cols, key)}
        rec["n_rows"] = int(len(g))
        for mk in metric_keys:
            mean_v, std_v = _safe_mean_std(g[mk].tolist() if mk in g.columns else [])
            rec[f"{mk}_mean"] = mean_v
            rec[f"{mk}_std"] = std_v
        overall_records.append(rec)
    protocol_df = pd.DataFrame(protocol_records).sort_values(["protocol_tag", "frr_target", "method"]).reset_index(drop=True)
    overall_df = pd.DataFrame(overall_records).sort_values(["frr_target", "method"]).reset_index(drop=True)
    return protocol_df, overall_df


def export_source_calibrated_artifacts(rows, base_dir):
    raw_df = pd.DataFrame(rows).sort_values(["protocol_tag", "split_id", "fold_id", "frr_target", "method"]).reset_index(drop=True)
    protocol_df, overall_df = summarize_source_calibrated_rows(rows)
    raw_csv = os.path.join(base_dir, "source_calibrated_raw_rows.csv")
    protocol_csv = os.path.join(base_dir, "source_calibrated_protocol_summary.csv")
    overall_csv = os.path.join(base_dir, "source_calibrated_overall_summary.csv")
    raw_df.to_csv(_win_safe_path(raw_csv), index=False)
    protocol_df.to_csv(_win_safe_path(protocol_csv), index=False)
    overall_df.to_csv(_win_safe_path(overall_csv), index=False)
    xlsx_path = None
    if SOURCE_CAL_EXPORT_XLSX:
        xlsx_path = os.path.join(base_dir, "source_calibrated_summary_workbook.xlsx")
        with pd.ExcelWriter(_win_safe_path(xlsx_path), engine="openpyxl") as writer:
            raw_df.to_excel(writer, sheet_name="raw_rows", index=False)
            protocol_df.to_excel(writer, sheet_name="protocol_summary", index=False)
            overall_df.to_excel(writer, sheet_name="overall_summary", index=False)
    if SOURCE_CAL_EXPORT_JSON:
        with _safe_open(os.path.join(base_dir, "source_calibrated_raw_rows.json"), "w", encoding="utf-8") as f:
            json.dump(rows, f, indent=2)
        with _safe_open(os.path.join(base_dir, "source_calibrated_protocol_summary.json"), "w", encoding="utf-8") as f:
            json.dump(protocol_df.to_dict(orient="records"), f, indent=2)
        with _safe_open(os.path.join(base_dir, "source_calibrated_overall_summary.json"), "w", encoding="utf-8") as f:
            json.dump(overall_df.to_dict(orient="records"), f, indent=2)
    return {
        "raw_df": raw_df,
        "protocol_df": protocol_df,
        "overall_df": overall_df,
        "raw_csv": raw_csv,
        "protocol_csv": protocol_csv,
        "overall_csv": overall_csv,
        "xlsx_path": xlsx_path,
    }


def run_strict_threshold_one_split(protocol_tag, rx_protocol_tag, tx_protocol_tag, split_id,
                                   KNOWN_TX, UNKNOWN_TX, source_rxs, drift_rx,
                                   protocol_dir, stagea_cache_protocol_dir=None):
    if "apply_soda_protocol_runtime" in globals():
        _ = apply_soda_protocol_runtime(protocol_tag)
    ctx = _build_fixed_far_fold_context(
        protocol_tag=protocol_tag,
        split_id=split_id,
        KNOWN_TX=KNOWN_TX,
        UNKNOWN_TX=UNKNOWN_TX,
        source_rxs=source_rxs,
        drift_rx=drift_rx,
        protocol_dir=protocol_dir,
        stagea_cache_protocol_dir=stagea_cache_protocol_dir)
    fixed_rows, source_rows = [], []
    for fold, (idx_tr, idx_te) in enumerate(ctx["skf"].split(ctx["X_src"], ctx["y_src"]), start=1):
        if MAX_FOLDS_PER_PROTOCOL is not None and int(fold) > int(MAX_FOLDS_PER_PROTOCOL):
            break
        fold_dir = os.path.join(ctx["split_dir"], f"fold_{fold}")
        cache_fold_dir = os.path.join(ctx["cache_split_dir"], f"fold_{fold}")
        _safe_makedirs(fold_dir)
        _safe_makedirs(cache_fold_dir)
        rng = np.random.RandomState(SEED + 1000 * split_id + fold)
        perm = rng.permutation(idx_tr)
        n_val = max(1, int(0.1 * len(perm)))
        idx_val = perm[:n_val]
        idx_train = perm[n_val:]
        stageA = build_or_load_stagea_fold(
            fold_dir=cache_fold_dir,
            protocol_tag=protocol_tag,
            split_id=split_id,
            fold=fold,
            K=ctx["K"],
            source_rxs=source_rxs,
            drift_rx=drift_rx,
            X_src=ctx["X_src"], y_src=ctx["y_src"], D_src=ctx["D_src"], rx_src=ctx["rx_src"],
            idx_train=idx_train, idx_val=idx_val, idx_te=idx_te,
            X_B=ctx["X_B"], y_B=ctx["y_B"], D_B=ctx["D_B"],
            X_C=ctx["X_C"], y_C=ctx["y_C"], D_C=ctx["D_C"],
            X_D=ctx["X_D"], y_D=ctx["y_D"], D_D=ctx["D_D"],
            X_E=ctx["X_E"], y_E=ctx["y_E"], D_E=ctx["D_E"],
            X_F=ctx["X_F"], y_F=ctx["y_F"], D_F=ctx["D_F"])
        model = stageA["model"]
        methods, method_info = _rebuild_compare_methods_from_stageA(stageA, model, protocol_tag=protocol_tag)
        y_tr_fold = np.asarray(stageA.get("y_tr_fold", ctx["y_src"][np.asarray(stageA.get("idx_train", idx_train), dtype=np.int64)]), dtype=np.int64)
        active_methods = [m for m in METHOD_ORDER if (m == "SourceOnly") or (m in methods)]
        fixed_fold_rows, source_fold_rows = [], []
        for base_name in active_methods:
            spec = methods.get(base_name, None)
            eval_cache = _build_fold_eval_cache_for_method(
                base_name, spec, method_info, model, stageA, ctx, idx_te, y_tr_fold, stageA["router_bundle_v8"], protocol_tag=protocol_tag)
            for head in _strict_protocol_eval_heads(protocol_tag, base_name, get_eval_heads_for_base_method(base_name)):
                eval_name = make_eval_method_name(base_name, head)
                if RUN_FIXED_FAR_EXPERIMENT:
                    for far_target in FIXED_FAR_TARGETS:
                        metric_row = evaluate_variant_at_fixed_far(
                            eval_cache,
                            ctx["y_src"][idx_te], stageA["y_B_ev"], stageA["y_C_ev"],
                            final_rejector_head=head,
                            far_target=far_target,
                            base_name=base_name,
                            ref_frr_target=FIXED_FAR_REFERENCE_FRR_TARGET)
                        fixed_fold_rows.append({
                            "protocol_tag": protocol_tag,
                            "rx_protocol": rx_protocol_tag,
                            "tx_protocol": tx_protocol_tag,
                            "split_id": int(split_id),
                            "fold_id": int(fold),
                            "method": eval_name,
                            "base_method": str(base_name),
                            "threshold_protocol": "fixed_far",
                            **metric_row,
                        })
                for frr_target in _strict_protocol_source_cal_frr_targets(protocol_tag, base_name, SOURCE_CAL_FRR_TARGETS):
                    metric_row = evaluate_variant_with_rejector(
                        eval_cache,
                        ctx["y_src"][idx_te], stageA["y_B_ev"], stageA["y_C_ev"],
                        final_rejector_head=head,
                        frr_target=float(frr_target),
                        base_name=base_name)
                    source_fold_rows.append({
                        "protocol_tag": protocol_tag,
                        "rx_protocol": rx_protocol_tag,
                        "tx_protocol": tx_protocol_tag,
                        "split_id": int(split_id),
                        "fold_id": int(fold),
                        "method": eval_name,
                        "base_method": str(base_name),
                        "frr_target": float(frr_target),
                        "threshold_protocol": "source_calibrated",
                        **metric_row,
                    })
        if FIXED_FAR_SAVE_FOLD_ROWS:
            with _safe_open(os.path.join(fold_dir, "strict_fixed_far_rows.json"), "w", encoding="utf-8") as f:
                json.dump(fixed_fold_rows, f, indent=2)
            with _safe_open(os.path.join(fold_dir, "strict_source_calibrated_rows.json"), "w", encoding="utf-8") as f:
                json.dump(source_fold_rows, f, indent=2)
        fixed_rows.extend(fixed_fold_rows)
        source_rows.extend(source_fold_rows)
    with _safe_open(os.path.join(ctx["split_dir"], "strict_fixed_far_rows_all_folds.json"), "w", encoding="utf-8") as f:
        json.dump(fixed_rows, f, indent=2)
    with _safe_open(os.path.join(ctx["split_dir"], "strict_source_calibrated_rows_all_folds.json"), "w", encoding="utf-8") as f:
        json.dump(source_rows, f, indent=2)
    return fixed_rows, source_rows


def run_strict_threshold_protocol_specs(protocol_specs, base_dir, max_splits=None, max_folds=None):
    global MAX_FOLDS_PER_PROTOCOL
    old_max = MAX_FOLDS_PER_PROTOCOL
    MAX_FOLDS_PER_PROTOCOL = max_folds
    fixed_rows, source_rows = [], []
    try:
        for ps in protocol_specs:
            protocol_dir = os.path.join(base_dir, ps["protocol_tag"])
            _safe_makedirs(protocol_dir)
            _strict_protocol_package_log(ps["protocol_tag"])
            tx_splits = list(ps["tx_splits"])
            if max_splits is not None:
                tx_splits = tx_splits[:int(max_splits)]
            for split_id, (KNOWN_TX, UNKNOWN_TX) in enumerate(tx_splits, start=1):
                fr, sr = run_strict_threshold_one_split(
                    protocol_tag=ps["protocol_tag"],
                    rx_protocol_tag=ps["rx_protocol"],
                    tx_protocol_tag=ps["tx_protocol"],
                    split_id=split_id,
                    KNOWN_TX=list(KNOWN_TX),
                    UNKNOWN_TX=list(UNKNOWN_TX),
                    source_rxs=ps["source_rxs"],
                    drift_rx=ps["drift_rx"],
                    protocol_dir=protocol_dir,
                    stagea_cache_protocol_dir=os.path.join(base_dir, "_stagea_cache", ps["protocol_tag"]))
                fixed_rows.extend(fr)
                source_rows.extend(sr)
    finally:
        MAX_FOLDS_PER_PROTOCOL = old_max
    return fixed_rows, source_rows


def run_strict_threshold_comparison_bundle(execution_mode=None):
    mode, specs, max_splits, max_folds = _select_specs_for_execution_mode(execution_mode=execution_mode)
    base_dir = os.path.join(RUN_DIR, "strict_threshold_comparison")
    fixed_dir = os.path.join(base_dir, FIXED_FAR_RUN_SUBDIR)
    source_dir = os.path.join(base_dir, SOURCE_CAL_RUN_SUBDIR)
    _safe_makedirs(fixed_dir)
    _safe_makedirs(source_dir)

    combined_rows_dir = os.path.join(base_dir, "fold_rows")
    _safe_makedirs(combined_rows_dir)
    fixed_far_rows, source_cal_rows = run_strict_threshold_protocol_specs(
        specs,
        base_dir=combined_rows_dir,
        max_splits=max_splits,
        max_folds=max_folds)
    fixed_far_artifacts = export_fixed_far_artifacts(
        fixed_far_rows,
        base_dir=fixed_dir,
        method_order=FIXED_FAR_METHODS) if fixed_far_rows else {}
    source_cal_artifacts = export_source_calibrated_artifacts(source_cal_rows, source_dir)

    bundle = {
        "execution_mode": mode,
        "base_dir": base_dir,
        "strict_repro_version": STRICT_REPRO_VERSION,
        "fixed_far_rows": fixed_far_rows,
        "source_calibrated_rows": source_cal_rows,
        "fixed_far_artifacts": {k: v for k, v in fixed_far_artifacts.items() if not isinstance(v, pd.DataFrame)},
        "source_calibrated_artifacts": {k: v for k, v in source_cal_artifacts.items() if not isinstance(v, pd.DataFrame)},
        "fixed_far_targets": FIXED_FAR_TARGETS,
        "source_cal_frr_targets": SOURCE_CAL_FRR_TARGETS,
        "methods": METHOD_ORDER,
        "strict_protocol_param_package_path": STRICT_PROTOCOL_PARAM_PACKAGE_PATH,
        "strict_protocol_param_package_active": _strict_protocol_package_active(),
        "strict_protocol_param_package": STRICT_PROTOCOL_PARAM_PACKAGE if _strict_protocol_package_active() else {},
    }
    with _safe_open(os.path.join(base_dir, "strict_threshold_manifest.json"), "w", encoding="utf-8") as f:
        json.dump(bundle, f, indent=2)
    print("Strict threshold comparison finished.")
    print("Mode:", mode)
    print("Base dir:", base_dir)
    print("Fixed-FAR rows:", len(fixed_far_rows))
    print("Source-calibrated rows:", len(source_cal_rows))
    return {
        **bundle,
        "fixed_far_artifacts_full": fixed_far_artifacts,
        "source_calibrated_artifacts_full": source_cal_artifacts,
    }

print("Strict reproduction patch active.")
print("METHOD_ORDER:", METHOD_ORDER)
print("EVAL_METHOD_ORDER:", EVAL_METHOD_ORDER)
print("SOURCE_CAL_FRR_TARGETS:", SOURCE_CAL_FRR_TARGETS)

Strict reproduction patch active.
METHOD_ORDER: ['SODA-RFF', 'SODA_direct_corebuffer', 'SourceOnly', 'PCPD_strict', 'OVANet_strict', 'COMET_strict', 'JRFFP_SC_strict']
EVAL_METHOD_ORDER: ['SODA-RFF', 'SODA_direct_corebuffer', 'SourceOnly', 'PCPD_strict', 'OVANet_strict', 'COMET_strict', 'JRFFP_SC_strict']
SOURCE_CAL_FRR_TARGETS: [0.01, 0.05, 0.1]


In [18]:

# ===== Strict final v2: native score bundles + explicit operating-point policies =====

STRICT_FINAL_VERSION = "strict_final_v2_score_policy_split"
PAPER_NATIVE_RUN_SUBDIR = "paper_native_bundle"
FIXED_FAR_NATIVE_SCORE_RUN_SUBDIR = "fixed_far_native_score_bundle"
FIXED_FRR_DRIFT_NATIVE_SCORE_RUN_SUBDIR = "fixed_frr_drift_native_score_bundle"

PAPER_NATIVE_SOURCE_FRR_TARGET = 0.05
OVANET_NATIVE_POS_THRESHOLD = 0.5
PCPD_NATIVE_RADIUS_Q = 0.993
COMET_NATIVE_ENTROPY_THRESHOLD_FINAL = 0.5
TARGET_CAL_MIN_COUNT_PER_CLASS = 32
TARGET_CAL_MIN_UNKNOWN_PER_CLASS = 24
FIXED_FRR_DRIFT_TARGETS = list(FIXED_FAR_TARGETS)
if bool(globals().get("RUN_DIRECT_COREBUFFER_SMOKE", False)):
    FIXED_FAR_TARGETS = [0.10]
    FIXED_FRR_DRIFT_TARGETS = [0.10]


OPERATING_POINT_METRIC_KEYS = [
    "drift_acc_all", "FRR_drift_all", "known_acc_e2e", "stable_acc_e2e", "FRR_known",
    "unk_rej_acc", "FAR_unknown_accept", "H_KU", "H_DU", "auc_unknown_eval", "auc_su", "auc_du", "auc_ku",
]
OPERATING_POINT_PIVOT_METRICS = [
    "H_KU", "H_DU", "drift_acc_all", "known_acc_e2e", "FRR_drift_all", "FAR_unknown_accept", "unk_rej_acc",
]
FIXED_FAR_NATIVE_SCORE_PLOT_METRICS = [
    "drift_acc_all", "FRR_drift_all", "known_acc_e2e", "stable_acc_e2e", "FRR_known",
    "unk_rej_acc", "FAR_unknown_accept", "H_KU", "H_DU",
]
FIXED_FRR_DRIFT_PLOT_METRICS = [
    "unk_rej_acc", "FAR_unknown_accept", "drift_acc_all", "known_acc_e2e", "stable_acc_e2e", "FRR_known", "H_KU", "H_DU",
]
PAPER_NATIVE_PLOT_METRICS = ["H_KU", "H_DU", "drift_acc_all", "known_acc_e2e", "FAR_unknown_accept", "FRR_drift_all"]


def _json_safe_v2(obj):
    if isinstance(obj, (str, int, bool)) or obj is None:
        return obj
    if isinstance(obj, float):
        return float(obj) if np.isfinite(obj) else None
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        x = float(obj)
        return x if np.isfinite(x) else None
    if isinstance(obj, np.ndarray):
        return [_json_safe_v2(x) for x in obj.tolist()]
    if isinstance(obj, dict):
        return {str(k): _json_safe_v2(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_json_safe_v2(x) for x in obj]
    try:
        return obj.item()
    except Exception:
        return str(obj)


def _write_json_v2(path, obj):
    ensure_parent_dir(path)
    with _safe_open(path, "w", encoding="utf-8") as f:
        json.dump(_json_safe_v2(obj), f, indent=2)
    return path


def _finite_array_v2(x):
    arr = np.asarray(x, dtype=np.float32).reshape(-1)
    return arr[np.isfinite(arr)]


def _threshold_scalar_v2(threshold):
    arr = _finite_array_v2(threshold)
    if arr.size == 0:
        return float("nan")
    return float(np.mean(arr))


def _score_unknown_margin_v2(score, accept_high):
    score = np.asarray(score, dtype=np.float32)
    return (-score).astype(np.float32) if bool(accept_high) else score.astype(np.float32)


def _argmax_logits_v2(eval_cache, split):
    return np.argmax(np.asarray(eval_cache[f"logits_{split}2"], dtype=np.float32), axis=1).astype(np.int64)


def _score_bundle_base(method_name, family, score_name, accept_high, is_class_conditional, eval_cache,
                       score_source, score_A, score_B, score_C, score_U,
                       pred_source=None, pred_A=None, pred_B=None, pred_C=None, pred_U=None,
                       native_threshold_kind="", native_threshold_meta=None, extra_meta=None):
    y_tr = np.asarray(eval_cache["y_tr"], dtype=np.int64)
    if pred_source is None:
        pred_source = np.argmax(np.asarray(eval_cache["logits_tr2"], dtype=np.float32), axis=1).astype(np.int64)
    if pred_A is None:
        pred_A = _argmax_logits_v2(eval_cache, "A")
    if pred_B is None:
        pred_B = _argmax_logits_v2(eval_cache, "B")
    if pred_C is None:
        pred_C = _argmax_logits_v2(eval_cache, "C")
    if pred_U is None:
        pred_U = _argmax_logits_v2(eval_cache, "U")
    return dict(
        method_name=str(method_name), family=str(family), score_name=str(score_name),
        accept_high=bool(accept_high), is_class_conditional=bool(is_class_conditional),
        pred_source=np.asarray(pred_source, dtype=np.int64),
        pred_A=np.asarray(pred_A, dtype=np.int64), pred_B=np.asarray(pred_B, dtype=np.int64),
        pred_C=np.asarray(pred_C, dtype=np.int64), pred_U=np.asarray(pred_U, dtype=np.int64),
        score_source=np.asarray(score_source, dtype=np.float32),
        score_A=np.asarray(score_A, dtype=np.float32), score_B=np.asarray(score_B, dtype=np.float32),
        score_C=np.asarray(score_C, dtype=np.float32), score_U=np.asarray(score_U, dtype=np.float32),
        source_y=y_tr,
        native_threshold_kind=str(native_threshold_kind),
        native_threshold_meta=dict(native_threshold_meta or {}),
        extra_meta=dict(extra_meta or {}),
    )


def _jrffp_siamese_score_against_refs(adapter_bundle, Z, pred, enrolled_refs):
    if not isinstance(adapter_bundle, dict) or "siamese" not in adapter_bundle:
        raise RuntimeError("JRFFP_SC_strict score bundle requires the trained Siamese comparator bundle.")
    return _eval_siamese_score_np(adapter_bundle, np.asarray(Z, dtype=np.float32), np.asarray(pred, dtype=np.int64), enrolled_refs)


def _eer_threshold_from_distance_scores(genuine_D, impostor_D):
    genuine_D = _finite_array_v2(genuine_D)
    impostor_D = _finite_array_v2(impostor_D)
    if genuine_D.size == 0 or impostor_D.size == 0:
        pooled = np.concatenate([genuine_D, impostor_D]) if (genuine_D.size + impostor_D.size) else np.asarray([0.5], dtype=np.float32)
        return float(np.quantile(pooled, 0.5)), dict(eer=float("nan"), n_genuine=int(genuine_D.size), n_impostor=int(impostor_D.size))
    candidates = np.unique(np.concatenate([genuine_D, impostor_D])).astype(np.float32)
    best = None
    for thr in candidates:
        fnr = float(np.mean(genuine_D > thr))
        fpr = float(np.mean(impostor_D <= thr))
        key = (abs(fnr - fpr), 0.5 * (fnr + fpr), float(thr))
        if best is None or key < best[0]:
            best = (key, float(thr), fnr, fpr)
    _, thr, fnr, fpr = best
    return float(thr), dict(eer=float(0.5 * (fnr + fpr)), fnr=float(fnr), fpr=float(fpr), n_genuine=int(genuine_D.size), n_impostor=int(impostor_D.size))


def build_method_native_score_bundle(method_name, eval_cache, adapter_bundle=None, method_info=None):
    method_name = str(method_name)
    y_tr = np.asarray(eval_cache["y_tr"], dtype=np.int64)
    K = int(eval_cache["logits_tr2"].shape[1])
    pred_tr = np.argmax(np.asarray(eval_cache["logits_tr2"], dtype=np.float32), axis=1).astype(np.int64)
    pred_A = _argmax_logits_v2(eval_cache, "A")
    pred_B = _argmax_logits_v2(eval_cache, "B")
    pred_C = _argmax_logits_v2(eval_cache, "C")
    pred_U = _argmax_logits_v2(eval_cache, "U")

    if method_name in {str(SODA_METHOD_NAME), str(globals().get("DIRECT_COREBUFFER_METHOD_NAME", "SODA_direct_corebuffer")), "SourceOnly"}:
        return _score_bundle_base(
            method_name=method_name, family="energy", score_name="energy_logsumexp", accept_high=True, is_class_conditional=False,
            eval_cache=eval_cache, score_source=np.asarray(eval_cache["energy_tr2"], dtype=np.float32),
            score_A=logsumexp_np(eval_cache["logits_A2"]), score_B=logsumexp_np(eval_cache["logits_B2"]),
            score_C=logsumexp_np(eval_cache["logits_C2"]), score_U=logsumexp_np(eval_cache["logits_U2"]),
            pred_source=pred_tr, pred_A=pred_A, pred_B=pred_B, pred_C=pred_C, pred_U=pred_U,
            native_threshold_kind="source_quantile",
            native_threshold_meta=dict(frr_target=float(PAPER_NATIVE_SOURCE_FRR_TARGET), note="Current SODA/SourceOnly design uses source-calibrated energy threshold."))

    if method_name == "OVANet_strict":
        if isinstance(adapter_bundle, dict) and "ova_head" in adapter_bundle:
            pos_tr = _eval_ova_pos_np(adapter_bundle, eval_cache["Z_tr2"])
            pos_A = _eval_ova_pos_np(adapter_bundle, eval_cache["Z_A2"])
            pos_B = _eval_ova_pos_np(adapter_bundle, eval_cache["Z_B2"])
            pos_C = _eval_ova_pos_np(adapter_bundle, eval_cache["Z_C2"])
            pos_U = _eval_ova_pos_np(adapter_bundle, eval_cache["Z_U2"])
            score_source = rowwise_pick(pos_tr, y_tr, fallback=0.0)
            score_A = rowwise_pick(pos_A, pred_A, fallback=0.0)
            score_B = rowwise_pick(pos_B, pred_B, fallback=0.0)
            score_C = rowwise_pick(pos_C, pred_C, fallback=0.0)
            score_U = rowwise_pick(pos_U, pred_U, fallback=0.0)
        else:
            aux = eval_cache.get("strict_aux", {})
            score_source = aux["source_score"]; score_A = aux["score_A"]; score_B = aux["score_B"]; score_C = aux["score_C"]; score_U = aux["score_U"]
        return _score_bundle_base(
            method_name, "ovanet_predclass_ova", "ovanet_predclass_ova_posprob", True, True, eval_cache,
            score_source, score_A, score_B, score_C, score_U,
            pred_source=y_tr, pred_A=pred_A, pred_B=pred_B, pred_C=pred_C, pred_U=pred_U,
            native_threshold_kind="fixed_predclass_ova_probability",
            native_threshold_meta=dict(threshold=float(OVANET_NATIVE_POS_THRESHOLD), note="Paper-native OVANet policy: predicted-class OVA positive probability >= 0.5."))

    if method_name == "PCPD_strict":
        protos = np.asarray(eval_cache["protos_eval"], dtype=np.float32)
        score_source = l2_to_selected_prototypes(eval_cache["Z_tr2"], y_tr, protos, l2norm=True)
        score_A = l2_to_selected_prototypes(eval_cache["Z_A2"], pred_A, protos, l2norm=True)
        score_B = l2_to_selected_prototypes(eval_cache["Z_B2"], pred_B, protos, l2norm=True)
        score_C = l2_to_selected_prototypes(eval_cache["Z_C2"], pred_C, protos, l2norm=True)
        score_U = l2_to_selected_prototypes(eval_cache["Z_U2"], pred_U, protos, l2norm=True)
        return _score_bundle_base(
            method_name, "pcpd_predclass_proto_distance", "pcpd_predclass_proto_l2", False, True, eval_cache,
            score_source, score_A, score_B, score_C, score_U,
            pred_source=y_tr, pred_A=pred_A, pred_B=pred_B, pred_C=pred_C, pred_U=pred_U,
            native_threshold_kind="source_classwise_radius",
            native_threshold_meta=dict(q=float(PCPD_NATIVE_RADIUS_Q), note="Paper-like PCPD source-native classwise prototype radius."))

    if method_name == "COMET_strict":
        return _score_bundle_base(
            method_name, "comet_entropy", "comet_normalized_entropy", False, False, eval_cache,
            normalized_entropy_from_logits(eval_cache["logits_tr2"]),
            normalized_entropy_from_logits(eval_cache["logits_A2"]),
            normalized_entropy_from_logits(eval_cache["logits_B2"]),
            normalized_entropy_from_logits(eval_cache["logits_C2"]),
            normalized_entropy_from_logits(eval_cache["logits_U2"]),
            pred_source=pred_tr, pred_A=pred_A, pred_B=pred_B, pred_C=pred_C, pred_U=pred_U,
            native_threshold_kind="fixed_entropy_delta",
            native_threshold_meta=dict(threshold=float(COMET_NATIVE_ENTROPY_THRESHOLD_FINAL), note="Paper-native COMET policy: normalized entropy <= 0.5."))

    if method_name == "JRFFP_SC_strict":
        enrolled_refs = _source_prototypes_np(eval_cache["Z_tr2"], y_tr, K)
        source_similarity = _jrffp_siamese_score_against_refs(adapter_bundle, eval_cache["Z_tr2"], y_tr, enrolled_refs)
        score_A_similarity = _jrffp_siamese_score_against_refs(adapter_bundle, eval_cache["Z_A2"], pred_A, enrolled_refs)
        score_B_similarity = _jrffp_siamese_score_against_refs(adapter_bundle, eval_cache["Z_B2"], pred_B, enrolled_refs)
        score_C_similarity = _jrffp_siamese_score_against_refs(adapter_bundle, eval_cache["Z_C2"], pred_C, enrolled_refs)
        score_U_similarity = _jrffp_siamese_score_against_refs(adapter_bundle, eval_cache["Z_U2"], pred_U, enrolled_refs)
        impostor_pred_source = ((y_tr + 1) % max(1, K)).astype(np.int64)
        impostor_similarity = _jrffp_siamese_score_against_refs(adapter_bundle, eval_cache["Z_tr2"], impostor_pred_source, enrolled_refs)
        genuine_D = (1.0 - source_similarity).astype(np.float32)
        impostor_D = (1.0 - impostor_similarity).astype(np.float32)
        eer_thr, eer_meta = _eer_threshold_from_distance_scores(genuine_D, impostor_D)
        return _score_bundle_base(
            method_name, "jrffp_sc_verification_distance", "jrffp_sc_D_1_minus_smatch", False, True, eval_cache,
            genuine_D, (1.0 - score_A_similarity).astype(np.float32), (1.0 - score_B_similarity).astype(np.float32),
            (1.0 - score_C_similarity).astype(np.float32), (1.0 - score_U_similarity).astype(np.float32),
            pred_source=y_tr, pred_A=pred_A, pred_B=pred_B, pred_C=pred_C, pred_U=pred_U,
            native_threshold_kind="source_validation_eer_distance",
            native_threshold_meta=dict(lambda_native=float(eer_thr), calibration_details=eer_meta,
                                       note="Unified JRFFP-SC: p_hat identity plus Siamese D=1-s_match to enrolled source reference."),
            extra_meta=dict(enrolled_reference="source class mean adapted embeddings", impostor_pair_rule="deterministic next-class source pairs"))

    raise ValueError(f"Unsupported method for native score bundle: {method_name}")


def _make_acceptor(threshold_policy, threshold_value, accept_high, is_class_conditional=False, threshold_by_class=None, meta=None):
    meta = dict(meta or {})
    if threshold_by_class is not None:
        threshold_by_class = np.asarray(threshold_by_class, dtype=np.float32)
        threshold_value = _threshold_scalar_v2(threshold_by_class)
    return dict(
        threshold_policy=str(threshold_policy),
        threshold_value=float(threshold_value) if np.isfinite(float(threshold_value)) else float("nan"),
        threshold_by_class=threshold_by_class,
        accept_high=bool(accept_high),
        classwise_mode=bool(is_class_conditional and threshold_by_class is not None),
        meta=meta,
    )


def _source_quantile_acceptor(score_bundle, frr_target, threshold_policy="source_calibrated_quantile", note=""):
    accept_high = bool(score_bundle["accept_high"])
    q = float(frr_target) if accept_high else float(1.0 - frr_target)
    thr = _safe_quantile(score_bundle["score_source"], q, fallback=0.0)
    return _make_acceptor(threshold_policy, thr, accept_high, False, None,
                          dict(calibration_source="source", frr_target=float(frr_target), q=float(q), paper_native_note=str(note)))


def build_native_paper_acceptor(score_bundle, method_name, native_cfg=None):
    method_name = str(method_name)
    native_cfg = dict(native_cfg or {})
    if method_name in {str(SODA_METHOD_NAME), str(globals().get("DIRECT_COREBUFFER_METHOD_NAME", "SODA_direct_corebuffer")), "SourceOnly"}:
        return _source_quantile_acceptor(
            score_bundle, float(native_cfg.get("source_frr_target", PAPER_NATIVE_SOURCE_FRR_TARGET)),
            threshold_policy="paper_native_source_quantile",
            note="Kept by definition for SODA-RFF/SourceOnly in this unified benchmark.")
    if method_name == "OVANet_strict":
        return _make_acceptor("paper_native_fixed_ovanet_ova_0p5", OVANET_NATIVE_POS_THRESHOLD, True, False, None,
                              dict(calibration_source="paper_fixed", paper_native_note="Predicted-class OVA positive probability >= 0.5."))
    if method_name == "PCPD_strict":
        y = np.asarray(score_bundle["source_y"], dtype=np.int64)
        vals = np.asarray(score_bundle["score_source"], dtype=np.float32)
        K = int(np.max(y)) + 1 if len(y) else 1
        q = float(native_cfg.get("pcpd_radius_q", PCPD_NATIVE_RADIUS_Q))
        fallback = _safe_quantile(vals, q, fallback=float("inf"))
        thr = build_classwise_thresholds(vals, y, K, q=q, fallback=fallback)
        fallback_classes = [int(k) for k in range(K) if int(np.sum(y == k)) == 0]
        return _make_acceptor("paper_native_pcpd_source_classwise_radius", _threshold_scalar_v2(thr), False, True, thr,
                              dict(calibration_source="source_true_class_distances", q=q, fallback_threshold=float(fallback),
                                   fallback_classes=fallback_classes, paper_native_note="Classwise source intra-class prototype-distance radius."))
    if method_name == "COMET_strict":
        return _make_acceptor("paper_native_fixed_comet_entropy_0p5", COMET_NATIVE_ENTROPY_THRESHOLD_FINAL, False, False, None,
                              dict(calibration_source="paper_fixed", paper_native_note="Normalized entropy <= 0.5."))
    if method_name == "JRFFP_SC_strict":
        meta = dict(score_bundle.get("native_threshold_meta", {}))
        thr = float(meta.get("lambda_native", 0.5))
        meta.update(dict(calibration_source="source_validation_genuine_impostor_pairs", paper_native_note="EER lambda on D=1-s_match."))
        return _make_acceptor("paper_native_jrffp_sc_source_validation_eer", thr, False, False, None, meta)
    raise ValueError(f"Unsupported paper-native acceptor method: {method_name}")


def _target_global_threshold(scores, accept_high, objective, target_value):
    scores = _finite_array_v2(scores)
    if scores.size == 0:
        return float("nan")
    target_value = float(np.clip(target_value, 0.0, 1.0))
    if objective == "fixed_far_unknown":
        q = 1.0 - target_value if bool(accept_high) else target_value
    elif objective == "fixed_frr_drift_all":
        q = target_value if bool(accept_high) else 1.0 - target_value
    else:
        raise ValueError(f"Unknown target-calibration objective: {objective}")
    return _safe_quantile(scores, q, fallback=float(np.nanmean(scores)))


def _target_classwise_thresholds(score_bundle, objective, target_value, global_thr):
    if objective == "fixed_far_unknown":
        scores = np.asarray(score_bundle["score_U"], dtype=np.float32)
        pred = np.asarray(score_bundle["pred_U"], dtype=np.int64)
        min_count = int(TARGET_CAL_MIN_UNKNOWN_PER_CLASS)
    else:
        scores = np.concatenate([np.asarray(score_bundle["score_B"], dtype=np.float32), np.asarray(score_bundle["score_C"], dtype=np.float32)])
        pred = np.concatenate([np.asarray(score_bundle["pred_B"], dtype=np.int64), np.asarray(score_bundle["pred_C"], dtype=np.int64)])
        min_count = int(TARGET_CAL_MIN_COUNT_PER_CLASS)
    K = int(np.max(np.concatenate([score_bundle["source_y"], pred]))) + 1 if len(pred) else int(np.max(score_bundle["source_y"])) + 1
    thr = np.full((K,), float(global_thr), dtype=np.float32)
    fallback_classes = []
    q = (1.0 - float(target_value) if bool(score_bundle["accept_high"]) else float(target_value)) if objective == "fixed_far_unknown" else (float(target_value) if bool(score_bundle["accept_high"]) else 1.0 - float(target_value))
    for k in range(K):
        vals = scores[pred == k]
        vals = vals[np.isfinite(vals)]
        if len(vals) < min_count:
            fallback_classes.append(int(k))
            continue
        thr[k] = float(np.quantile(vals, np.clip(q, 0.0, 1.0)))
    return thr, fallback_classes, min_count


def build_target_calibrated_acceptor(score_bundle, objective, target_value, classwise_if_supported=True):
    objective = str(objective)
    target_value = float(target_value)
    accept_high = bool(score_bundle["accept_high"])
    if objective == "fixed_far_unknown":
        calib_scores = score_bundle["score_U"]
        calibration_source = "target_unknown_U"
    elif objective == "fixed_frr_drift_all":
        calib_scores = np.concatenate([np.asarray(score_bundle["score_B"], dtype=np.float32), np.asarray(score_bundle["score_C"], dtype=np.float32)])
        calibration_source = "target_drift_B_union_C"
    else:
        raise ValueError(f"Unsupported objective: {objective}")
    global_thr = _target_global_threshold(calib_scores, accept_high, objective, target_value)
    threshold_by_class = None
    fallback_classes = []
    min_count = None
    if bool(classwise_if_supported) and bool(score_bundle.get("is_class_conditional", False)):
        threshold_by_class, fallback_classes, min_count = _target_classwise_thresholds(score_bundle, objective, target_value, global_thr)
    return _make_acceptor(
        f"target_calibrated_{objective}", global_thr, accept_high,
        bool(score_bundle.get("is_class_conditional", False)), threshold_by_class,
        dict(objective=objective, target_value=float(target_value), calibration_source=calibration_source,
             global_threshold=float(global_thr), fallback_classes=fallback_classes, min_count_per_class=min_count,
             classwise_if_supported=bool(classwise_if_supported)))


def _threshold_for_split(acceptor, pred):
    if acceptor.get("threshold_by_class", None) is None:
        return np.full((len(pred),), float(acceptor["threshold_value"]), dtype=np.float32)
    thr = np.asarray(acceptor["threshold_by_class"], dtype=np.float32)
    pred = np.asarray(pred, dtype=np.int64)
    return thr[np.clip(pred, 0, len(thr) - 1)]


def _accept_from_score(score, pred, acceptor):
    score = np.asarray(score, dtype=np.float32)
    thr = _threshold_for_split(acceptor, pred)
    return score >= thr if bool(acceptor["accept_high"]) else score <= thr


def apply_score_acceptor(score_bundle, acceptor, y_A, y_B, y_C):
    accept_A = _accept_from_score(score_bundle["score_A"], score_bundle["pred_A"], acceptor)
    accept_B = _accept_from_score(score_bundle["score_B"], score_bundle["pred_B"], acceptor)
    accept_C = _accept_from_score(score_bundle["score_C"], score_bundle["pred_C"], acceptor)
    accept_U = _accept_from_score(score_bundle["score_U"], score_bundle["pred_U"], acceptor)
    margin_A = _score_unknown_margin_v2(score_bundle["score_A"], score_bundle["accept_high"])
    margin_B = _score_unknown_margin_v2(score_bundle["score_B"], score_bundle["accept_high"])
    margin_C = _score_unknown_margin_v2(score_bundle["score_C"], score_bundle["accept_high"])
    margin_U = _score_unknown_margin_v2(score_bundle["score_U"], score_bundle["accept_high"])
    out = _evaluate_from_accept_and_pred(
        score_bundle["pred_A"], score_bundle["pred_B"], score_bundle["pred_C"],
        accept_A, accept_B, accept_C, accept_U,
        margin_A, margin_B, margin_C, margin_U, y_A, y_B, y_C)
    out["score_HKU_HDU"] = float(0.5 * out.get("H_KU", 0.0) + 0.5 * out.get("H_DU", 0.0))
    out.update(dict(
        score_family=str(score_bundle["family"]), score_name=str(score_bundle["score_name"]),
        accept_high=bool(score_bundle["accept_high"]), is_class_conditional=bool(score_bundle["is_class_conditional"]),
        threshold_policy=str(acceptor["threshold_policy"]),
        threshold_value=float(acceptor.get("threshold_value", float("nan"))),
        classwise_mode=bool(acceptor.get("classwise_mode", False)),
        threshold_by_class_json=json.dumps(_json_safe_v2(acceptor.get("threshold_by_class"))) if acceptor.get("threshold_by_class") is not None else "",
        calibration_source=str(acceptor.get("meta", {}).get("calibration_source", "")),
        fallback_classes_json=json.dumps(_json_safe_v2(acceptor.get("meta", {}).get("fallback_classes", []))),
    ))
    return out


def build_native_threshold_registry_row(score_bundle, acceptor, row_base):
    meta = dict(acceptor.get("meta", {}))
    return dict(
        protocol_tag=row_base["protocol_tag"], split_id=int(row_base["split_id"]), fold_id=int(row_base["fold_id"]),
        method=row_base["method"], base_method=row_base["base_method"],
        score_family=str(score_bundle["family"]), score_name=str(score_bundle["score_name"]), accept_high=bool(score_bundle["accept_high"]),
        threshold_policy=str(acceptor["threshold_policy"]), threshold_value=float(acceptor.get("threshold_value", float("nan"))),
        classwise_mode=bool(acceptor.get("classwise_mode", False)), calibration_source=str(meta.get("calibration_source", "")),
        paper_native_note=str(meta.get("paper_native_note", score_bundle.get("native_threshold_meta", {}).get("note", ""))),
        threshold_by_class=_json_safe_v2(acceptor.get("threshold_by_class")),
        threshold_meta=_json_safe_v2(meta),
    )


def _build_fold_eval_cache_and_adapter_v2(base_name, method_spec, method_info, model, stageA, ctx, idx_te, y_tr_fold, router_bundle_v8, protocol_tag=None):
    adapter = None
    extra = {}
    if base_name != "SourceOnly":
        with _strict_protocol_method_runtime(protocol_tag, str(SODA_METHOD_NAME) if str(base_name) == str(globals().get("DIRECT_COREBUFFER_METHOD_NAME", "SODA_direct_corebuffer")) else base_name):
            adapter, extra = fit_single_adapter_from_spec(
                method_spec, model.fc, stageA["Z_replay"], stageA["y_replay"], stageA["Z_adapt"], stageA["protos"], sc_adapt=stageA["sc_adapt"])
    _router_family = str(method_info.get(base_name, {}).get("router_family", "v8"))
    _router_bundle_eval = None if (_router_family == "v5" or str(base_name).endswith("_M2V5") or str(base_name).startswith("M2V5")) else router_bundle_v8
    cache = build_variant_eval_cache(
        adapter, model.fc, stageA["Z_tr"], y_tr_fold,
        stageA["Z_A"], stageA["D_A"], stageA["Z_B_ev"], stageA["D_B_ev"], stageA["Z_C_ev"], stageA["D_C_ev"], stageA["Z_U_ev"], stageA["D_U_ev"],
        stageA["mu_z_src"], stageA["var_z_src"], stageA["ref_sid_emb_A"], stageA["ref_sid_en_A"], stageA["models_kr"], stageA["fallback_k"], stageA["source_rx_ids"],
        stageA["tau_id"], stageA["tau_dom"], router_bundle=_router_bundle_eval)
    return cache, adapter, extra


def _safe_mean_std_v2(values):
    arr = np.asarray([float(v) for v in values if pd.notnull(v)], dtype=np.float64)
    if arr.size == 0:
        return float("nan"), float("nan")
    return float(arr.mean()), float(arr.std(ddof=0))


def summarize_operating_point_rows(rows, metric_keys=None):
    metric_keys = list(metric_keys or OPERATING_POINT_METRIC_KEYS + ["score_HKU_HDU"])
    if not rows:
        return pd.DataFrame(), pd.DataFrame()
    raw_df = pd.DataFrame(rows)
    group_cols = ["protocol_tag", "method", "target_value"]
    overall_cols = ["method", "target_value"]
    protocol_records = []
    for key, g in raw_df.groupby(group_cols, dropna=False):
        rec = {c: v for c, v in zip(group_cols, key)}
        rec["target_label"] = str(g.get("target_label", pd.Series([rec["target_value"]])).iloc[0])
        rec["n_rows"] = int(len(g))
        for mk in metric_keys:
            mean_v, std_v = _safe_mean_std_v2(g[mk].tolist() if mk in g.columns else [])
            rec[mk] = mean_v
            rec[f"{mk}_mean"] = mean_v
            rec[f"{mk}_std"] = std_v
        protocol_records.append(rec)
    overall_records = []
    for key, g in raw_df.groupby(overall_cols, dropna=False):
        rec = {c: v for c, v in zip(overall_cols, key)}
        rec["target_label"] = str(g.get("target_label", pd.Series([rec["target_value"]])).iloc[0])
        rec["n_rows"] = int(len(g))
        for mk in metric_keys:
            mean_v, std_v = _safe_mean_std_v2(g[mk].tolist() if mk in g.columns else [])
            rec[mk] = mean_v
            rec[f"{mk}_mean"] = mean_v
            rec[f"{mk}_std"] = std_v
        overall_records.append(rec)
    protocol_df = pd.DataFrame(protocol_records).sort_values(["protocol_tag", "target_value", "method"]).reset_index(drop=True)
    overall_df = pd.DataFrame(overall_records).sort_values(["target_value", "method"]).reset_index(drop=True)
    return protocol_df, overall_df


def build_curves_from_summary(summary_df, include_protocol=True):
    out = {}
    if summary_df.empty:
        return out
    group_cols = ["protocol_tag", "method"] if include_protocol and "protocol_tag" in summary_df.columns else ["method"]
    for key, g in summary_df.groupby(group_cols, dropna=False):
        if include_protocol:
            proto, method = key
            out.setdefault(str(proto), {})[str(method)] = g.sort_values("target_value").to_dict(orient="records")
        else:
            method = key if isinstance(key, str) else key[0]
            out[str(method)] = g.sort_values("target_value").to_dict(orient="records")
    return _json_safe_v2(out)


def build_best_method_by_protocol_target(protocol_df, metric_primary="score_HKU_HDU_mean"):
    rows = []
    if protocol_df.empty or metric_primary not in protocol_df.columns:
        return rows
    for (protocol_tag, target_value), g in protocol_df.groupby(["protocol_tag", "target_value"], dropna=False):
        gg = g.sort_values(metric_primary, ascending=False).reset_index(drop=True)
        best = gg.iloc[0]
        runner = gg.iloc[1] if len(gg) > 1 else None
        best_value = float(best[metric_primary]) if pd.notnull(best[metric_primary]) else float("nan")
        runner_value = float(runner[metric_primary]) if runner is not None and pd.notnull(runner[metric_primary]) else float("nan")
        rows.append(dict(
            protocol_tag=str(protocol_tag), target_value=float(target_value) if pd.notnull(target_value) else float("nan"),
            metric_primary=str(metric_primary), best_method=str(best["method"]), best_value=best_value,
            runner_up_method=str(runner["method"]) if runner is not None else "", runner_up_value=runner_value,
            margin=float(best_value - runner_value) if np.isfinite(best_value) and np.isfinite(runner_value) else float("nan"),
        ))
    return rows


def _plot_operating_point_metric(summary_df, out_path, metric, title, policy_cfg, method_order=None):
    mean_col = f"{metric}_mean"
    std_col = f"{metric}_std"
    if summary_df.empty or mean_col not in summary_df.columns:
        return
    method_order = list(method_order or METHOD_ORDER)
    policy_name = str(policy_cfg.get("policy_name", ""))
    fig, ax = plt.subplots(figsize=(8, 5))
    plotted = False
    if policy_name == "paper_native":
        vals, labels = [], []
        for method in method_order:
            g = summary_df[summary_df["method"] == method]
            if g.empty:
                continue
            vals.append(float(g[mean_col].mean()))
            labels.append(str(method))
        if vals:
            ax.bar(np.arange(len(vals)), vals)
            ax.set_xticks(np.arange(len(vals)))
            ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
            plotted = True
    else:
        for method in method_order:
            g = summary_df[summary_df["method"] == method].sort_values("target_value")
            if g.empty:
                continue
            x = g["target_value"].to_numpy(dtype=float)
            y = g[mean_col].to_numpy(dtype=float)
            ax.plot(x, y, marker="o", linewidth=1.8, label=str(method))
            if std_col in g.columns:
                s = g[std_col].fillna(0.0).to_numpy(dtype=float)
                ax.fill_between(x, y - s, y + s, alpha=0.12)
            plotted = True
        ax.set_xlabel(str(policy_cfg.get("x_label", "target_value")))
        ax.legend(loc="best", fontsize=8)
    if not plotted:
        plt.close(fig)
        return
    ax.set_ylabel(metric)
    ax.set_title(title)
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    ensure_parent_dir(out_path)
    fig.savefig(_win_safe_path(out_path), dpi=180, bbox_inches="tight")
    plt.close(fig)


def _operating_point_pivot_sheet_name(metric):
    explicit = {
        "H_KU": "pivot_HKU",
        "H_DU": "pivot_HDU",
    }
    return explicit.get(str(metric), f"pivot_{metric}")[:31]


def export_operating_point_artifacts(rows, base_dir, policy_cfg, threshold_registry=None):
    _safe_makedirs(base_dir)
    prefix = str(policy_cfg["file_prefix"])
    raw_df = pd.DataFrame(rows)
    if not raw_df.empty:
        raw_df = raw_df.sort_values(["protocol_tag", "split_id", "fold_id", "target_value", "method"]).reset_index(drop=True)
    protocol_df, overall_df = summarize_operating_point_rows(rows)
    curves_by_protocol = build_curves_from_summary(protocol_df, include_protocol=True)
    curves_overall = build_curves_from_summary(overall_df, include_protocol=False)
    best_rows = build_best_method_by_protocol_target(protocol_df, metric_primary="score_HKU_HDU_mean")
    rank_df = pd.DataFrame(best_rows)

    raw_csv = os.path.join(base_dir, f"{prefix}_raw_rows.csv")
    raw_json = os.path.join(base_dir, f"{prefix}_raw_rows.json")
    protocol_csv = os.path.join(base_dir, f"{prefix}_protocol_summary.csv")
    protocol_json = os.path.join(base_dir, f"{prefix}_protocol_summary.json")
    overall_csv = os.path.join(base_dir, f"{prefix}_overall_summary.csv")
    overall_json = os.path.join(base_dir, f"{prefix}_overall_summary.json")
    curves_protocol_json = os.path.join(base_dir, f"{prefix}_curves_by_protocol.json")
    curves_overall_json = os.path.join(base_dir, f"{prefix}_curves_overall.json")
    best_csv = os.path.join(base_dir, f"{prefix}_best_method_by_protocol_target.csv")
    best_json = os.path.join(base_dir, f"{prefix}_best_method_by_protocol_target.json")
    workbook_path = os.path.join(base_dir, f"{prefix}_summary_workbook.xlsx")
    manifest_path = os.path.join(base_dir, f"{prefix}_manifest.json")

    raw_df.to_csv(_win_safe_path(raw_csv), index=False)
    protocol_df.to_csv(_win_safe_path(protocol_csv), index=False)
    overall_df.to_csv(_win_safe_path(overall_csv), index=False)
    pd.DataFrame(best_rows).to_csv(_win_safe_path(best_csv), index=False)
    _write_json_v2(raw_json, rows)
    _write_json_v2(protocol_json, protocol_df.to_dict(orient="records"))
    _write_json_v2(overall_json, overall_df.to_dict(orient="records"))
    _write_json_v2(curves_protocol_json, curves_by_protocol)
    _write_json_v2(curves_overall_json, curves_overall)
    _write_json_v2(best_json, best_rows)

    with pd.ExcelWriter(_win_safe_path(workbook_path), engine="openpyxl") as writer:
        raw_df.to_excel(writer, sheet_name="raw_rows", index=False)
        protocol_df.to_excel(writer, sheet_name="protocol_summary", index=False)
        overall_df.to_excel(writer, sheet_name="overall_summary", index=False)
        pd.DataFrame([_json_safe_v2(policy_cfg)]).to_excel(writer, sheet_name="config", index=False)
        for metric in OPERATING_POINT_PIVOT_METRICS:
            col = f"{metric}_mean"
            sheet_name = _operating_point_pivot_sheet_name(metric)
            if col in protocol_df.columns and not protocol_df.empty:
                piv = protocol_df.pivot_table(index=["protocol_tag", "target_value"], columns="method", values=col)
            else:
                piv = pd.DataFrame()
            piv.to_excel(writer, sheet_name=sheet_name)
        rank_df.to_excel(writer, sheet_name="rank_by_protocol_target", index=False)

    plot_metrics = list(policy_cfg.get("plot_metrics", []))
    if not overall_df.empty:
        for metric in plot_metrics:
            _plot_operating_point_metric(overall_df, os.path.join(base_dir, f"{prefix}_overall__{metric}.png"), metric, f"{policy_cfg['policy_name']} overall: {metric}", policy_cfg)
    if not protocol_df.empty:
        for protocol_tag in sorted(protocol_df["protocol_tag"].unique()):
            pdir = os.path.join(base_dir, str(protocol_tag))
            _safe_makedirs(pdir)
            g = protocol_df[protocol_df["protocol_tag"] == protocol_tag].copy()
            for metric in plot_metrics:
                _plot_operating_point_metric(g, os.path.join(pdir, f"{prefix}__{protocol_tag}__{metric}.png"), metric, f"{protocol_tag} {policy_cfg['policy_name']}: {metric}", policy_cfg)

    native_registry_path = None
    if threshold_registry is not None:
        native_registry_path = os.path.join(base_dir, "native_threshold_registry.json")
        _write_json_v2(native_registry_path, threshold_registry)

    manifest = dict(
        policy_cfg=_json_safe_v2(policy_cfg), n_rows=int(len(rows)),
        raw_csv=raw_csv, raw_json=raw_json, protocol_csv=protocol_csv, protocol_json=protocol_json,
        overall_csv=overall_csv, overall_json=overall_json, curves_by_protocol_json=curves_protocol_json,
        curves_overall_json=curves_overall_json, workbook_path=workbook_path, best_csv=best_csv,
        best_json=best_json, native_threshold_registry_json=native_registry_path,
    )
    _write_json_v2(manifest_path, manifest)
    return dict(raw_df=raw_df, protocol_df=protocol_df, overall_df=overall_df, best_df=rank_df,
                manifest=manifest, manifest_path=manifest_path, raw_csv=raw_csv,
                protocol_csv=protocol_csv, overall_csv=overall_csv, workbook_path=workbook_path)


def _append_policy_row(rows, score_bundle, acceptor, y_A, y_B, y_C, row_base, policy_name, target_value, target_label):
    metric_row = apply_score_acceptor(score_bundle, acceptor, y_A, y_B, y_C)
    target_axis = {
        "paper_native": "paper_native",
        "fixed_far_unknown": "target_far_unknown",
        "fixed_frr_drift_all": "target_frr_drift_all",
    }.get(str(policy_name), "target_value")
    rows.append({
        **row_base, "policy_name": str(policy_name), "threshold_protocol": str(policy_name),
        "target_axis": target_axis, "target_value": float(target_value), "target_label": str(target_label), **metric_row,
    })


def run_strict_threshold_one_split_v2(protocol_tag, rx_protocol_tag, tx_protocol_tag, split_id,
                                      KNOWN_TX, UNKNOWN_TX, source_rxs, drift_rx,
                                      protocol_dir, stagea_cache_protocol_dir=None):
    if "apply_soda_protocol_runtime" in globals():
        _ = apply_soda_protocol_runtime(protocol_tag)
    ctx = _build_fixed_far_fold_context(
        protocol_tag=protocol_tag, split_id=split_id, KNOWN_TX=KNOWN_TX, UNKNOWN_TX=UNKNOWN_TX,
        source_rxs=source_rxs, drift_rx=drift_rx, protocol_dir=protocol_dir, stagea_cache_protocol_dir=stagea_cache_protocol_dir)
    paper_rows, fixed_far_rows, fixed_frr_rows, native_registry = [], [], [], []
    for fold, (idx_tr, idx_te) in enumerate(ctx["skf"].split(ctx["X_src"], ctx["y_src"]), start=1):
        if MAX_FOLDS_PER_PROTOCOL is not None and int(fold) > int(MAX_FOLDS_PER_PROTOCOL):
            break
        fold_dir = os.path.join(ctx["split_dir"], f"fold_{fold}")
        cache_fold_dir = os.path.join(ctx["cache_split_dir"], f"fold_{fold}")
        _safe_makedirs(fold_dir); _safe_makedirs(cache_fold_dir)
        rng = np.random.RandomState(SEED + 1000 * split_id + fold)
        perm = rng.permutation(idx_tr)
        n_val = max(1, int(0.1 * len(perm)))
        idx_val = perm[:n_val]
        idx_train = perm[n_val:]
        stageA = build_or_load_stagea_fold(
            fold_dir=cache_fold_dir, protocol_tag=protocol_tag, split_id=split_id, fold=fold, K=ctx["K"],
            source_rxs=source_rxs, drift_rx=drift_rx,
            X_src=ctx["X_src"], y_src=ctx["y_src"], D_src=ctx["D_src"], rx_src=ctx["rx_src"],
            idx_train=idx_train, idx_val=idx_val, idx_te=idx_te,
            X_B=ctx["X_B"], y_B=ctx["y_B"], D_B=ctx["D_B"],
            X_C=ctx["X_C"], y_C=ctx["y_C"], D_C=ctx["D_C"],
            X_D=ctx["X_D"], y_D=ctx["y_D"], D_D=ctx["D_D"],
            X_E=ctx["X_E"], y_E=ctx["y_E"], D_E=ctx["D_E"],
            X_F=ctx["X_F"], y_F=ctx["y_F"], D_F=ctx["D_F"])
        model = stageA["model"]
        methods, method_info = _rebuild_compare_methods_from_stageA(stageA, model, protocol_tag=protocol_tag)
        y_tr_fold = np.asarray(stageA.get("y_tr_fold", ctx["y_src"][np.asarray(stageA.get("idx_train", idx_train), dtype=np.int64)]), dtype=np.int64)
        active_methods = [m for m in METHOD_ORDER if (m == "SourceOnly") or (m in methods)]
        for base_name in active_methods:
            spec = methods.get(base_name, None)
            eval_cache, adapter_bundle, _extra = _build_fold_eval_cache_and_adapter_v2(
                base_name, spec, method_info, model, stageA, ctx, idx_te, y_tr_fold, stageA["router_bundle_v8"], protocol_tag=protocol_tag)
            score_bundle = build_method_native_score_bundle(base_name, eval_cache, adapter_bundle=adapter_bundle, method_info=method_info.get(base_name, {}))
            row_base = dict(protocol_tag=str(protocol_tag), rx_protocol=str(rx_protocol_tag), tx_protocol=str(tx_protocol_tag),
                            split_id=int(split_id), fold_id=int(fold), method=str(base_name), base_method=str(base_name),
                            strict_final_version=STRICT_FINAL_VERSION)
            if str(base_name) == str(globals().get("DIRECT_COREBUFFER_METHOD_NAME", "SODA_direct_corebuffer")):
                row_base.update(_direct_corebuffer_row_fields(method_info.get(base_name, {})))
            paper_acceptor = build_native_paper_acceptor(score_bundle, base_name, native_cfg={})
            _append_policy_row(paper_rows, score_bundle, paper_acceptor, ctx["y_src"][idx_te], stageA["y_B_ev"], stageA["y_C_ev"], row_base, "paper_native", 0.0, "paper_native")
            native_registry.append(build_native_threshold_registry_row(score_bundle, paper_acceptor, row_base))
            for far_target in FIXED_FAR_TARGETS:
                acceptor = build_target_calibrated_acceptor(score_bundle, "fixed_far_unknown", float(far_target))
                _append_policy_row(fixed_far_rows, score_bundle, acceptor, ctx["y_src"][idx_te], stageA["y_B_ev"], stageA["y_C_ev"],
                                   {**row_base, "target_far_unknown": float(far_target)}, "fixed_far_unknown", float(far_target), f"FAR={float(far_target):.3f}")
            for frr_target in FIXED_FRR_DRIFT_TARGETS:
                acceptor = build_target_calibrated_acceptor(score_bundle, "fixed_frr_drift_all", float(frr_target))
                _append_policy_row(fixed_frr_rows, score_bundle, acceptor, ctx["y_src"][idx_te], stageA["y_B_ev"], stageA["y_C_ev"],
                                   {**row_base, "target_frr_drift_all": float(frr_target)}, "fixed_frr_drift_all", float(frr_target), f"FRR_drift={float(frr_target):.3f}")
    _write_json_v2(os.path.join(ctx["split_dir"], "strict_v2_paper_native_rows.json"), paper_rows)
    _write_json_v2(os.path.join(ctx["split_dir"], "strict_v2_fixed_far_unknown_rows.json"), fixed_far_rows)
    _write_json_v2(os.path.join(ctx["split_dir"], "strict_v2_fixed_frr_drift_all_rows.json"), fixed_frr_rows)
    return paper_rows, fixed_far_rows, fixed_frr_rows, native_registry


def run_strict_threshold_protocol_specs_v2(protocol_specs, base_dir, max_splits=None, max_folds=None):
    global MAX_FOLDS_PER_PROTOCOL
    old_max = MAX_FOLDS_PER_PROTOCOL
    MAX_FOLDS_PER_PROTOCOL = max_folds
    paper_rows, fixed_far_rows, fixed_frr_rows, native_registry = [], [], [], []
    try:
        for ps in protocol_specs:
            protocol_dir = os.path.join(base_dir, ps["protocol_tag"])
            _safe_makedirs(protocol_dir)
            _strict_protocol_package_log(ps["protocol_tag"])
            tx_splits = list(ps["tx_splits"])
            if max_splits is not None:
                tx_splits = tx_splits[:int(max_splits)]
            for split_id, (KNOWN_TX, UNKNOWN_TX) in enumerate(tx_splits, start=1):
                pr, fr, dr, nr = run_strict_threshold_one_split_v2(
                    protocol_tag=ps["protocol_tag"], rx_protocol_tag=ps["rx_protocol"], tx_protocol_tag=ps["tx_protocol"],
                    split_id=split_id, KNOWN_TX=list(KNOWN_TX), UNKNOWN_TX=list(UNKNOWN_TX),
                    source_rxs=ps["source_rxs"], drift_rx=ps["drift_rx"], protocol_dir=protocol_dir,
                    stagea_cache_protocol_dir=os.path.join(base_dir, "_stagea_cache", ps["protocol_tag"]))
                paper_rows.extend(pr); fixed_far_rows.extend(fr); fixed_frr_rows.extend(dr); native_registry.extend(nr)
    finally:
        MAX_FOLDS_PER_PROTOCOL = old_max
    return paper_rows, fixed_far_rows, fixed_frr_rows, native_registry



def export_direct_corebuffer_diagnostics(all_rows, base_dir):
    _safe_makedirs(base_dir)
    rows = [dict(r) for r in list(all_rows or []) if str(r.get("method")) == str(globals().get("DIRECT_COREBUFFER_METHOD_NAME", "SODA_direct_corebuffer"))]
    all_path = os.path.join(base_dir, "direct_corebuffer_all_splits_folds.csv")
    diag_path = os.path.join(base_dir, "direct_corebuffer_subset_diagnostics.csv")
    protocol_json = os.path.join(base_dir, "direct_corebuffer_protocol_summary.json")
    overall_json = os.path.join(base_dir, "direct_corebuffer_overall_summary.json")
    manifest_path = os.path.join(base_dir, "direct_corebuffer_manifest.json")
    pd.DataFrame(rows).to_csv(_win_safe_path(all_path), index=False)
    diag_seen, diag_rows = set(), []
    diag_prefixes = ["n_", "frac_", "stable_", "shift_", "unknown_", "buffer_", "fallback_", "core_sets_", "core_buffer_", "raw_", "overlap_", "conflict_", "thresholds_", "relaxed_", "topk_", "forced_", "max_cap_", "use_", "selection_", "p_source", "route_probability_source", "class_balanced_known", "sup_weight_mode", "min_core_size"]
    for r in rows:
        key = (str(r.get("protocol_tag")), int(r.get("split_id", -1)), int(r.get("fold_id", -1)))
        if key in diag_seen:
            continue
        diag_seen.add(key)
        d = {"protocol_tag": key[0], "split_id": key[1], "fold_id": key[2], "method": str(r.get("method"))}
        for k, v in r.items():
            if any(str(k).startswith(pref) for pref in diag_prefixes) or str(k) in {
                "direct_corebuffer_active", "direct_corebuffer_method", "direct_corebuffer_protocol_tag",
                "selection_mode", "conflict_policy", "p_source",
                "buffer_skipped_from_training", "buffer_in_sup_idx", "buffer_in_align_idx",
                "buffer_in_unrel_idx", "buffer_in_idx_unk_eval",
            }:
                d[k] = v
        diag_rows.append(d)
    pd.DataFrame(diag_rows).to_csv(_win_safe_path(diag_path), index=False)

    def _mean_metric(vals):
        vals = np.asarray([float(v) for v in vals if pd.notnull(v)], dtype=np.float64)
        return float(np.mean(vals)) if vals.size else float("nan")
    protocol_summary, overall_summary = [], []
    df = pd.DataFrame(rows)
    if not df.empty:
        for (ptag, policy), g in df.groupby(["protocol_tag", "policy_name"], dropna=False):
            rec = {"protocol_tag": str(ptag), "policy_name": str(policy), "method": str(globals().get("DIRECT_COREBUFFER_METHOD_NAME", "SODA_direct_corebuffer")), "n_rows": int(len(g))}
            for mk in OPERATING_POINT_METRIC_KEYS + ["score_HKU_HDU"]:
                if mk in g.columns:
                    rec[mk] = _mean_metric(g[mk].tolist())
            protocol_summary.append(rec)
        for policy, g in df.groupby(["policy_name"], dropna=False):
            rec = {"policy_name": str(policy), "method": str(globals().get("DIRECT_COREBUFFER_METHOD_NAME", "SODA_direct_corebuffer")), "n_rows": int(len(g))}
            for mk in OPERATING_POINT_METRIC_KEYS + ["score_HKU_HDU"]:
                if mk in g.columns:
                    rec[mk] = _mean_metric(g[mk].tolist())
            overall_summary.append(rec)
    _write_json_v2(protocol_json, protocol_summary)
    _write_json_v2(overall_json, overall_summary)
    warnings = []
    if diag_rows and all(int(d.get("n_buffer", 0) or 0) <= 0 for d in diag_rows):
        warnings.append("all_samples_consumed_by_cores_no_buffer")
    manifest = dict(
        method=str(globals().get("DIRECT_COREBUFFER_METHOD_NAME", "SODA_direct_corebuffer")),
        n_rows=int(len(rows)), n_diagnostic_rows=int(len(diag_rows)),
        output_files=dict(all_splits_folds_csv=all_path, subset_diagnostics_csv=diag_path, protocol_summary_json=protocol_json, overall_summary_json=overall_json),
        config=dict(
            selection_mode=str(DIRECT_COREBUFFER_SELECTION_MODE),
            conflict_policy=str(DIRECT_COREBUFFER_CONFLICT_POLICY),
            p_source=str(DIRECT_COREBUFFER_P_SOURCE),
            stable_score_min=float(DIRECT_COREBUFFER_STABLE_SCORE_MIN),
            shift_score_min=float(DIRECT_COREBUFFER_SHIFT_SCORE_MIN),
            unknown_score_min=float(DIRECT_COREBUFFER_UNK_SCORE_MIN),
            stable_margin_min=float(DIRECT_COREBUFFER_STABLE_MARGIN_MIN),
            shift_margin_min=float(DIRECT_COREBUFFER_SHIFT_MARGIN_MIN),
            unknown_margin_min=float(DIRECT_COREBUFFER_UNK_MARGIN_MIN),
            stable_conf_min=float(DIRECT_COREBUFFER_STABLE_CONF_MIN),
            shift_conf_min=float(DIRECT_COREBUFFER_SHIFT_CONF_MIN),
            unknown_conf_max=float(DIRECT_COREBUFFER_UNK_CONF_MAX),
            stable_entropy_max=float(DIRECT_COREBUFFER_STABLE_ENTROPY_MAX),
            shift_entropy_max=float(DIRECT_COREBUFFER_SHIFT_ENTROPY_MAX),
            unknown_entropy_min=float(DIRECT_COREBUFFER_UNK_ENTROPY_MIN),
            shift_known_min=float(DIRECT_COREBUFFER_SHIFT_KNOWN_MIN),
            unknown_known_max=float(DIRECT_COREBUFFER_UNK_KNOWN_MAX),
            use_agreement_min=bool(DIRECT_COREBUFFER_USE_AGREEMENT_MIN),
            use_max_cap=bool(DIRECT_COREBUFFER_USE_MAX_CAP),
            smoke_relax_thresholds=bool(DIRECT_COREBUFFER_SMOKE_RELAX_THRESHOLDS),
            class_balanced_known=bool(DIRECT_COREBUFFER_CLASS_BALANCED_KNOWN),
            buffer_mode=str(DIRECT_COREBUFFER_BUFFER_MODE),
            sup_weight_mode=str(DIRECT_COREBUFFER_SUP_WEIGHT_MODE),
            min_core_size=int(DIRECT_COREBUFFER_MIN_CORE_SIZE)),
        warnings=warnings)
    _write_json_v2(manifest_path, manifest)
    return dict(all_splits_folds_csv=all_path, subset_diagnostics_csv=diag_path, protocol_summary_json=protocol_json, overall_summary_json=overall_json, manifest_json=manifest_path, manifest=manifest)


def validate_direct_corebuffer_smoke_outputs(bundle):
    base_dir = bundle.get("base_dir", "") if isinstance(bundle, dict) else ""
    diag_dir = os.path.join(base_dir, "direct_corebuffer_diagnostics")
    required = [
        "direct_corebuffer_all_splits_folds.csv",
        "direct_corebuffer_subset_diagnostics.csv",
        "direct_corebuffer_protocol_summary.json",
        "direct_corebuffer_overall_summary.json",
        "direct_corebuffer_manifest.json",
    ]
    missing = [p for p in required if not os.path.exists(_win_safe_path(os.path.join(diag_dir, p)))]
    if missing:
        raise RuntimeError(f"SODA_direct_corebuffer smoke missing diagnostics: {missing}")
    df = pd.read_csv(_win_safe_path(os.path.join(diag_dir, "direct_corebuffer_all_splits_folds.csv")))
    if df.empty:
        raise RuntimeError("SODA_direct_corebuffer smoke produced no metric rows.")
    policies = set(df.get("policy_name", pd.Series([], dtype=str)).astype(str).unique())
    needed = {"paper_native", "fixed_far_unknown", "fixed_frr_drift_all"}
    if not needed.issubset(policies):
        raise RuntimeError(f"SODA_direct_corebuffer smoke missing policies: {sorted(needed - policies)}")
    diag = pd.read_csv(_win_safe_path(os.path.join(diag_dir, "direct_corebuffer_subset_diagnostics.csv")))
    if diag.empty:
        raise RuntimeError("SODA_direct_corebuffer diagnostics are empty.")
    required_diag_cols = ["selection_mode", "topk_fallback_used", "forced_min_core_size_used", "core_buffer_union_exhaustive"]
    missing_diag_cols = [c for c in required_diag_cols if c not in diag.columns]
    if missing_diag_cols:
        raise RuntimeError(f"SODA_direct_corebuffer diagnostics missing required threshold-first columns: {missing_diag_cols}")
    if "selection_mode" in diag.columns and not (diag["selection_mode"].astype(str) == "threshold_first").all():
        raise RuntimeError("SODA_direct_corebuffer selection_mode must be threshold_first.")
    for col in ["stable_shift_empty", "stable_unknown_empty", "shift_unknown_empty", "buffer_disjoint", "buffer_skipped_from_training", "core_buffer_union_exhaustive"]:
        if col in diag.columns and not diag[col].astype(str).str.lower().isin(["true", "1"]).all():
            raise RuntimeError(f"SODA_direct_corebuffer diagnostic check failed: {col}")
    if "topk_fallback_used" in diag.columns and diag["topk_fallback_used"].astype(str).str.lower().isin(["true", "1"]).any():
        raise RuntimeError("Threshold-first smoke unexpectedly used top-k fallback.")
    if "forced_min_core_size_used" in diag.columns and diag["forced_min_core_size_used"].astype(str).str.lower().isin(["true", "1"]).any():
        raise RuntimeError("Threshold-first smoke unexpectedly used forced minimum core-size fallback.")
    for col in ["buffer_in_sup_idx", "buffer_in_align_idx", "buffer_in_unrel_idx", "buffer_in_idx_unk_eval"]:
        if col in diag.columns and not diag[col].astype(str).str.lower().isin(["false", "0"]).all():
            raise RuntimeError(f"SODA_direct_corebuffer buffer leaked into training set: {col}")
    print("SODA_DIRECT_COREBUFFER SMOKE TEST PASSED")
    return dict(out_dir=diag_dir, result_rows=int(len(df)), diagnostic_rows=int(len(diag)))


def run_strict_threshold_comparison_bundle_v2(execution_mode=None):
    mode, specs, max_splits, max_folds = _select_specs_for_execution_mode(execution_mode=execution_mode)
    base_dir = os.path.join(RUN_DIR, "strict_threshold_comparison")
    fold_rows_dir = os.path.join(base_dir, "fold_rows_v2")
    _safe_makedirs(fold_rows_dir)
    paper_rows, fixed_far_rows, fixed_frr_rows, native_registry = run_strict_threshold_protocol_specs_v2(
        specs, base_dir=fold_rows_dir, max_splits=max_splits, max_folds=max_folds)

    paper_cfg = dict(policy_name="paper_native", file_prefix="paper_native", x_label="paper_native", plot_metrics=PAPER_NATIVE_PLOT_METRICS,
                     source_frr_target=PAPER_NATIVE_SOURCE_FRR_TARGET, strict_final_version=STRICT_FINAL_VERSION)
    far_cfg = dict(policy_name="fixed_far_unknown", file_prefix="fixed_far_unknown", x_label="target FAR_unknown", plot_metrics=FIXED_FAR_NATIVE_SCORE_PLOT_METRICS,
                   targets=[float(x) for x in FIXED_FAR_TARGETS], strict_final_version=STRICT_FINAL_VERSION)
    frr_cfg = dict(policy_name="fixed_frr_drift_all", file_prefix="fixed_frr_drift_all", x_label="target FRR_drift_all", plot_metrics=FIXED_FRR_DRIFT_PLOT_METRICS,
                   targets=[float(x) for x in FIXED_FRR_DRIFT_TARGETS], strict_final_version=STRICT_FINAL_VERSION)

    paper_artifacts = export_operating_point_artifacts(paper_rows, os.path.join(base_dir, PAPER_NATIVE_RUN_SUBDIR), paper_cfg, threshold_registry=native_registry)
    far_artifacts = export_operating_point_artifacts(fixed_far_rows, os.path.join(base_dir, FIXED_FAR_NATIVE_SCORE_RUN_SUBDIR), far_cfg)
    frr_artifacts = export_operating_point_artifacts(fixed_frr_rows, os.path.join(base_dir, FIXED_FRR_DRIFT_NATIVE_SCORE_RUN_SUBDIR), frr_cfg)
    direct_corebuffer_artifacts = export_direct_corebuffer_diagnostics(
        paper_rows + fixed_far_rows + fixed_frr_rows,
        os.path.join(base_dir, "direct_corebuffer_diagnostics"))

    manifest = dict(
        execution_mode=mode, base_dir=base_dir, strict_final_version=STRICT_FINAL_VERSION,
        methods=list(METHOD_ORDER), protocol_tags=[str(ps["protocol_tag"]) for ps in specs],
        strict_protocol_param_package_path=STRICT_PROTOCOL_PARAM_PACKAGE_PATH,
        strict_protocol_param_package_active=_strict_protocol_package_active(),
        bundles=dict(paper_native=paper_artifacts["manifest"], fixed_far_unknown=far_artifacts["manifest"], fixed_frr_drift_all=frr_artifacts["manifest"], direct_corebuffer=direct_corebuffer_artifacts.get("manifest", {})),
        rows=dict(paper_native=len(paper_rows), fixed_far_unknown=len(fixed_far_rows), fixed_frr_drift_all=len(fixed_frr_rows)),
    )
    _write_json_v2(os.path.join(base_dir, "strict_threshold_manifest.json"), manifest)
    print("Strict threshold comparison v2 finished.")
    print("Mode:", mode)
    print("Base dir:", base_dir)
    print("Paper-native rows:", len(paper_rows))
    print("Fixed-FAR rows:", len(fixed_far_rows))
    print("Fixed-FRR-drift rows:", len(fixed_frr_rows))
    _return_bundle = dict(
        execution_mode=mode, base_dir=base_dir, strict_final_version=STRICT_FINAL_VERSION,
        paper_native_rows=paper_rows, fixed_far_unknown_rows=fixed_far_rows, fixed_frr_drift_all_rows=fixed_frr_rows,
        native_threshold_registry=native_registry,
        paper_native_artifacts_full=paper_artifacts, fixed_far_unknown_artifacts_full=far_artifacts,
        fixed_frr_drift_all_artifacts_full=frr_artifacts, direct_corebuffer_artifacts=direct_corebuffer_artifacts, manifest=manifest)
    if bool(globals().get("RUN_DIRECT_COREBUFFER_SMOKE", False)) and str(mode) == "smoke":
        _return_bundle["direct_corebuffer_smoke_validation"] = validate_direct_corebuffer_smoke_outputs(_return_bundle)
    return _return_bundle


print("Strict final v2 score/policy refactor active.")
print("STRICT_FINAL_VERSION:", STRICT_FINAL_VERSION)


Strict final v2 score/policy refactor active.
STRICT_FINAL_VERSION: strict_final_v2_score_policy_split


In [19]:
# Strict final v2 comparison entry cell
strict_threshold_bundle = run_strict_threshold_comparison_bundle_v2(execution_mode=EXECUTION_MODE)

# Convenient aliases for inspection
run_summary = strict_threshold_bundle
paper_native_overall_summary = None
fixed_far_unknown_overall_summary = None
fixed_frr_drift_all_overall_summary = None

_pn_art = strict_threshold_bundle.get("paper_native_artifacts_full", {})
_ff_art = strict_threshold_bundle.get("fixed_far_unknown_artifacts_full", {})
_fd_art = strict_threshold_bundle.get("fixed_frr_drift_all_artifacts_full", {})
if isinstance(_pn_art.get("overall_df", None), pd.DataFrame):
    paper_native_overall_summary = _pn_art["overall_df"]
if isinstance(_ff_art.get("overall_df", None), pd.DataFrame):
    fixed_far_unknown_overall_summary = _ff_art["overall_df"]
if isinstance(_fd_art.get("overall_df", None), pd.DataFrame):
    fixed_frr_drift_all_overall_summary = _fd_art["overall_df"]

print("Strict final v2 run_summary keys:", sorted(run_summary.keys()))


[STRICT-BASELINE-CONFIG] protocol=RX9-3_TX4-2 package_file=strict_protocol_best_params_v2_20cands.json package_path=strict_protocol_best_params_v2_20cands.json active=True selected_methods=['COMET_strict', 'JRFFP_SC_strict', 'OVANet_strict', 'PCPD_strict', 'SourceOnly']
[SODA-CONFIG] protocol=RX9-3_TX4-2 | source=selected_snapshot_json
[SODA-CONFIG] loaded selected snapshot for RX9-3_TX4-2
[SODA-CONFIG] selected_profile_tag=u93_42_repel_plus_p06_10 | selected_rejector_head=energy | json_path=./resolved_selected_variants.json
[SODA-CONFIG] snapshot/runtime keys = {'gamma_notunk': 1.0192, 'lambda_unkE': 0.021, 'lambda_stab': 0.0, 'v19_lambda_align': 0.03515, 'v19_lambda_ent_min': 0.0057, 'v19_lambda_ent_max': 0.0143, 'v19_lambda_proto': 0.13775, 'v19_unknown_energy_margin': 0.04752, 'v36_proxy_cap': 0.06, 'v36_proxy_q': 0.92, 'v36_proxy_pu_min': 0.38, 'v36_proxy_conf_max': 0.8, 'v36_proxy_conf_smooth': 0.08, 'v36_proxy_entropy_mix': 0.35, 'v36_lambda_ent_max_scale': 1.0, 'v36_proxy_min_k